<a href="https://colab.research.google.com/github/kahaos/Auto-GPT/blob/master/Another_copy_of_Untitled0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# SILVER AGENT — V1
# First test: connect to OANDA Practice and retrieve live XAG/USD data

!pip -q install requests pandas numpy

import requests
import pandas as pd
import numpy as np
from datetime import datetime, timezone

print("Silver Agent starting...")
print("Libraries installed successfully.")

Silver Agent starting...
Libraries installed successfully.


In [10]:
# SILVER AGENT — V2
# Download recent XAG/USD market history

import pandas as pd
from datetime import datetime, timezone

print("Downloading recent silver market history...")

url = f"{OANDA_URL}/v3/instruments/XAG_USD/candles"

params = {
    "count": 200,
    "granularity": "H1",
    "price": "M"
}

response = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=15
)

if response.status_code == 200:

    candles = response.json()["candles"]

    rows = []

    for candle in candles:
        if candle["complete"]:
            rows.append({
                "time": candle["time"],
                "open": float(candle["mid"]["o"]),
                "high": float(candle["mid"]["h"]),
                "low": float(candle["mid"]["l"]),
                "close": float(candle["mid"]["c"]),
                "volume": int(candle["volume"])
            })

    silver = pd.DataFrame(rows)

    silver["time"] = pd.to_datetime(silver["time"])

    print("\n✅ HISTORICAL DATA RECEIVED")
    print("--------------------------------")
    print(f"Candles loaded: {len(silver)}")
    print(f"Timeframe: 1 hour")
    print(f"Oldest: {silver['time'].iloc[0]}")
    print(f"Newest: {silver['time'].iloc[-1]}")
    print("--------------------------------")

    display(silver.tail(10))

else:
    print("\n❌ DATA DOWNLOAD FAILED")
    print(f"HTTP status: {response.status_code}")
    print(response.text)


✅ HISTORICAL DATA RECEIVED
--------------------------------
Candles loaded: 199
Timeframe: 1 hour
Oldest: 2026-09-06 23:00:00+00:00
Newest: 2026-09-17 15:00:00+00:00
--------------------------------


,time,open,high,low,close,volume
189,2026-09-17 06:00:00+00:00,63.7695,63.9235,63.71050,63.8215,13001
190,2026-09-17 07:00:00+00:00,63.8215,64.4565,63.78250,64.3385,13728
191,2026-09-17 08:00:00+00:00,64.3390,64.3480,63.79465,63.8610,12394
192,2026-09-17 09:00:00+00:00,63.8585,64.0550,63.85600,63.9215,10578
193,2026-09-17 10:00:00+00:00,63.9225,64.1575,63.86200,64.1115,9815
194,2026-09-17 11:00:00+00:00,64.1125,64.7770,64.08250,64.7685,14671
195,2026-09-17 12:00:00+00:00,64.7690,65.5950,64.59000,65.4945,29322
196,2026-09-17 13:00:00+00:00,65.4955,65.7695,65.25800,65.7385,33646
197,2026-09-17 14:00:00+00:00,65.7410,66.1595,65.50350,65.5545,25768
198,2026-09-17 15:00:00+00:00,65.5510,65.9145,65.54050,65.9050,18133


In [11]:
# SILVER AGENT — V3
# First opportunity detector

import numpy as np
import pandas as pd

df = silver.copy()

# --- Technical measurements ---

# Returns
df["return_1h"] = df["close"].pct_change()

# Moving averages
df["MA_10"] = df["close"].rolling(10).mean()
df["MA_30"] = df["close"].rolling(30).mean()

# Momentum
df["momentum_5h"] = df["close"].pct_change(5)
df["momentum_10h"] = df["close"].pct_change(10)

# Volatility
df["volatility"] = df["return_1h"].rolling(20).std()

# Average volume
df["avg_volume_20"] = df["volume"].rolling(20).mean()

# Relative volume
df["relative_volume"] = df["volume"] / df["avg_volume_20"]

# Current candle range
df["range"] = df["high"] - df["low"]

# --- Latest observation ---

latest = df.iloc[-1]

score = 0
signals = []

# 1. Trend
if latest["MA_10"] > latest["MA_30"]:
    score += 20
    signals.append("Short-term trend is bullish")
else:
    signals.append("Short-term trend is bearish")

# 2. Momentum
if latest["momentum_5h"] > 0:
    score += 15
    signals.append("5-hour momentum is positive")

if latest["momentum_10h"] > 0:
    score += 15
    signals.append("10-hour momentum is positive")

# 3. Volume
if latest["relative_volume"] > 1.2:
    score += 20
    signals.append("Trading volume is above normal")

# 4. Volatility
if latest["volatility"] > df["volatility"].median():
    score += 15
    signals.append("Volatility is elevated")

# 5. Price above short-term average
if latest["close"] > latest["MA_10"]:
    score += 15
    signals.append("Price is above the 10-hour average")

# --- Classification ---

if score >= 75:
    rating = "STRONG OPPORTUNITY"
elif score >= 55:
    rating = "INTERESTING"
elif score >= 35:
    rating = "WATCH"
else:
    rating = "NO CLEAR OPPORTUNITY"

# --- Agent report ---

print("╔══════════════════════════════════════╗")
print("║       SILVER OPPORTUNITY AGENT       ║")
print("╚══════════════════════════════════════╝")

print(f"\nCurrent price: {latest['close']:.4f}")
print(f"Opportunity score: {score}/100")
print(f"Assessment: {rating}")

print("\nSignals detected:")
for signal in signals:
    print(f" • {signal}")

print("\nMarket measurements:")
print(f" • 5h momentum:  {latest['momentum_5h'] * 100:.2f}%")
print(f" • 10h momentum: {latest['momentum_10h'] * 100:.2f}%")
print(f" • Relative volume: {latest['relative_volume']:.2f}x")
print(f" • Volatility: {latest['volatility']:.5f}")

print("\n⚠️ This is an experimental signal, not financial advice.")

╔══════════════════════════════════════╗
║       SILVER OPPORTUNITY AGENT       ║
╚══════════════════════════════════════╝

Current price: 65.9050
Opportunity score: 65/100
Assessment: INTERESTING

Signals detected:
 • Short-term trend is bullish
 • 5-hour momentum is positive
 • 10-hour momentum is positive
 • Price is above the 10-hour average

Market measurements:
 • 5h momentum:  2.80%
 • 10h momentum: 3.34%
 • Relative volume: 1.17x
 • Volatility: 0.00497

⚠️ This is an experimental signal, not financial advice.


In [12]:
# SILVER AGENT — V4
# Historical signal backtest

test = df.copy()

# Calculate future returns
test["future_6h"] = test["close"].shift(-6) / test["close"] - 1
test["future_12h"] = test["close"].shift(-12) / test["close"] - 1
test["future_24h"] = test["close"].shift(-24) / test["close"] - 1

def calculate_score(row):

    score = 0

    if row["MA_10"] > row["MA_30"]:
        score += 20

    if row["momentum_5h"] > 0:
        score += 15

    if row["momentum_10h"] > 0:
        score += 15

    if row["relative_volume"] > 1.2:
        score += 20

    if row["volatility"] > test["volatility"].median():
        score += 15

    if row["close"] > row["MA_10"]:
        score += 15

    return score


test["score"] = test.apply(calculate_score, axis=1)

# Remove rows where there isn't enough future data
test = test.dropna(subset=["future_6h", "future_12h", "future_24h"])

# Only investigate stronger signals
signals = test[test["score"] >= 55].copy()

print("╔══════════════════════════════════════╗")
print("║       SILVER AGENT BACKTEST          ║")
print("╚══════════════════════════════════════╝")

print(f"\nHistorical candles tested: {len(test)}")
print(f"Signals ≥55/100: {len(signals)}")

if len(signals) > 0:

    print("\nAverage future returns after signals:")
    print(f" • 6 hours :  {signals['future_6h'].mean()*100:+.3f}%")
    print(f" • 12 hours:  {signals['future_12h'].mean()*100:+.3f}%")
    print(f" • 24 hours:  {signals['future_24h'].mean()*100:+.3f}%")

    print("\nProbability of positive return:")

    print(
        f" • 6 hours :  "
        f"{(signals['future_6h'] > 0).mean()*100:.1f}%"
    )

    print(
        f" • 12 hours:  "
        f"{(signals['future_12h'] > 0).mean()*100:.1f}%"
    )

    print(
        f" • 24 hours:  "
        f"{(signals['future_24h'] > 0).mean()*100:.1f}%"
    )

else:
    print("\nNot enough strong signals yet.")

print("\n⚠️ Historical results do not guarantee future performance.")

╔══════════════════════════════════════╗
║       SILVER AGENT BACKTEST          ║
╚══════════════════════════════════════╝

Historical candles tested: 175
Signals ≥55/100: 53

Average future returns after signals:
 • 6 hours :  -0.374%
 • 12 hours:  -0.803%
 • 24 hours:  -1.783%

Probability of positive return:
 • 6 hours :  35.8%
 • 12 hours:  35.8%
 • 24 hours:  22.6%

⚠️ Historical results do not guarantee future performance.


In [13]:
# SILVER AGENT — V5
# Download a much larger historical dataset

print("Downloading extended silver history...")

url = f"{OANDA_URL}/v3/instruments/XAG_USD/candles"

params = {
    "count": 2000,
    "granularity": "H1",
    "price": "M"
}

response = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=20
)

if response.status_code == 200:

    candles = response.json()["candles"]

    rows = []

    for candle in candles:
        if candle["complete"]:
            rows.append({
                "time": candle["time"],
                "open": float(candle["mid"]["o"]),
                "high": float(candle["mid"]["h"]),
                "low": float(candle["mid"]["l"]),
                "close": float(candle["mid"]["c"]),
                "volume": int(candle["volume"])
            })

    silver = pd.DataFrame(rows)
    silver["time"] = pd.to_datetime(silver["time"])

    print("\n✅ EXTENDED HISTORY LOADED")
    print("--------------------------------")
    print(f"Candles: {len(silver)}")
    print(f"From: {silver['time'].iloc[0]}")
    print(f"To:   {silver['time'].iloc[-1]}")
    print("--------------------------------")

else:
    print("❌ DOWNLOAD FAILED")
    print(response.status_code)
    print(response.text)


✅ EXTENDED HISTORY LOADED
--------------------------------
Candles: 1999
From: 2026-05-19 06:00:00+00:00
To:   2026-09-17 15:00:00+00:00
--------------------------------


In [14]:
# SILVER AGENT — V6
# Re-run baseline strategy on extended history

df = silver.copy()

df["return_1h"] = df["close"].pct_change()
df["MA_10"] = df["close"].rolling(10).mean()
df["MA_30"] = df["close"].rolling(30).mean()

df["momentum_5h"] = df["close"].pct_change(5)
df["momentum_10h"] = df["close"].pct_change(10)

df["volatility"] = df["return_1h"].rolling(20).std()
df["avg_volume_20"] = df["volume"].rolling(20).mean()
df["relative_volume"] = df["volume"] / df["avg_volume_20"]

df["future_6h"] = df["close"].shift(-6) / df["close"] - 1
df["future_12h"] = df["close"].shift(-12) / df["close"] - 1
df["future_24h"] = df["close"].shift(-24) / df["close"] - 1

vol_median = df["volatility"].median()

def score_signal(row):
    score = 0

    if row["MA_10"] > row["MA_30"]:
        score += 20

    if row["momentum_5h"] > 0:
        score += 15

    if row["momentum_10h"] > 0:
        score += 15

    if row["relative_volume"] > 1.2:
        score += 20

    if row["volatility"] > vol_median:
        score += 15

    if row["close"] > row["MA_10"]:
        score += 15

    return score

df["score"] = df.apply(score_signal, axis=1)

test = df.dropna(
    subset=["score", "future_6h", "future_12h", "future_24h"]
)

signals = test[test["score"] >= 55]

print("╔══════════════════════════════════════╗")
print("║   EXTENDED SILVER BASELINE TEST      ║")
print("╚══════════════════════════════════════╝")

print(f"\nCandles tested: {len(test)}")
print(f"Signals ≥55: {len(signals)}")

if len(signals) > 0:

    print("\nAverage return after signal:")
    print(f"6h  : {signals['future_6h'].mean()*100:+.3f}%")
    print(f"12h : {signals['future_12h'].mean()*100:+.3f}%")
    print(f"24h : {signals['future_24h'].mean()*100:+.3f}%")

    print("\nPositive outcome rate:")
    print(f"6h  : {(signals['future_6h'] > 0).mean()*100:.1f}%")
    print(f"12h : {(signals['future_12h'] > 0).mean()*100:.1f}%")
    print(f"24h : {(signals['future_24h'] > 0).mean()*100:.1f}%")

print("\n⚠️ Baseline only — no trading.")

╔══════════════════════════════════════╗
║   EXTENDED SILVER BASELINE TEST      ║
╚══════════════════════════════════════╝

Candles tested: 1975
Signals ≥55: 732

Average return after signal:
6h  : -0.156%
12h : -0.182%
24h : -0.391%

Positive outcome rate:
6h  : 41.9%
12h : 44.5%
24h : 43.7%

⚠️ Baseline only — no trading.


In [15]:
# ============================================================
# SILVER AGENT — V7
# STRATEGY DISCOVERY ENGINE
# ============================================================

import itertools
import numpy as np
import pandas as pd

print("==========================================")
print("     SILVER AGENT — V7 DISCOVERY ENGINE")
print("==========================================")

# ------------------------------------------------------------
# 1. BUILD FEATURES
# ------------------------------------------------------------

data = silver.copy().reset_index(drop=True)

data["return_1h"] = data["close"].pct_change()

# Moving averages
data["MA_5"] = data["close"].rolling(5).mean()
data["MA_10"] = data["close"].rolling(10).mean()
data["MA_20"] = data["close"].rolling(20).mean()
data["MA_50"] = data["close"].rolling(50).mean()

# Momentum
data["mom_3"] = data["close"].pct_change(3)
data["mom_6"] = data["close"].pct_change(6)
data["mom_12"] = data["close"].pct_change(12)
data["mom_24"] = data["close"].pct_change(24)

# Volatility
data["vol_10"] = data["return_1h"].rolling(10).std()
data["vol_20"] = data["return_1h"].rolling(20).std()

# Candle behaviour
data["candle_body"] = (
    data["close"] - data["open"]
) / data["open"]

data["candle_range"] = (
    data["high"] - data["low"]
) / data["open"]

# Recent highs/lows
data["high_20"] = data["high"].rolling(20).max().shift(1)
data["low_20"] = data["low"].rolling(20).min().shift(1)

# Volume
data["volume_ma20"] = data["volume"].rolling(20).mean()
data["relative_volume"] = (
    data["volume"] / data["volume_ma20"]
)

# Future returns
data["future_6h"] = (
    data["close"].shift(-6) / data["close"] - 1
)

data["future_12h"] = (
    data["close"].shift(-12) / data["close"] - 1
)

data["future_24h"] = (
    data["close"].shift(-24) / data["close"] - 1
)

# ------------------------------------------------------------
# 2. CREATE MARKET CONDITIONS
# ------------------------------------------------------------

conditions = {

    # Trend
    "trend_up":
    data["MA_10"] > data["MA_20"],

    "trend_down":
        data["MA_10"] < data["MA_20"],

    "price_above_MA20":
        data["close"] > data["MA_20"],

    "price_below_MA20":
        data["close"] < data["MA_20"],

    # Momentum
    "mom3_up":
        data["mom_3"] > 0,

    "mom3_down":
        data["mom_3"] < 0,

    "mom6_up":
        data["mom_6"] > 0,

    "mom6_down":
        data["mom_6"] < 0,

    "mom12_up":
        data["mom_12"] > 0,

    "mom12_down":
        data["mom_12"] < 0,

    # Candle behaviour
    "strong_bull_candle":
        data["candle_body"] > 0.001,

    "strong_bear_candle":
        data["candle_body"] < -0.001,

    # Volume
    "high_volume":
        data["relative_volume"] > 1.2,

    "low_volume":
        data["relative_volume"] < 0.8,

    # Volatility
    "high_volatility":
        data["vol_20"] > data["vol_20"].rolling(100).median(),

    "low_volatility":
        data["vol_20"] < data["vol_20"].rolling(100).median(),

    # Breakouts
    "breakout_up":
        data["close"] > data["high_20"],

    "breakout_down":
        data["close"] < data["low_20"],
}

# ------------------------------------------------------------
# 3. REMOVE INVALID ROWS
# ------------------------------------------------------------

required = [
    "future_6h",
    "future_12h",
    "future_24h"
]

data = data.dropna(subset=required).copy()

# ------------------------------------------------------------
# 4. TRAIN / TEST SPLIT
# ------------------------------------------------------------

split = int(len(data) * 0.70)

train = data.iloc[:split].copy()
test = data.iloc[split:].copy()

print(f"\nTotal usable candles: {len(data)}")
print(f"Training candles:     {len(train)}")
print(f"Unseen test candles:  {len(test)}")

print("\nThe test period remains hidden from strategy selection.")

# ------------------------------------------------------------
# 5. TRANSACTION COST ASSUMPTION
# ------------------------------------------------------------

# Conservative estimated cost per completed trade.
# This is deliberately included so tiny historical advantages
# don't look artificially attractive.

COST = 0.0005

# ------------------------------------------------------------
# 6. TEST A STRATEGY
# ------------------------------------------------------------

def evaluate_strategy(frame, selected_conditions, direction):

    mask = pd.Series(True, index=frame.index)

    for name in selected_conditions:

        condition = conditions[name]

        # Align condition to the frame
        condition = condition.reindex(frame.index)

        mask &= condition.fillna(False)

    signals = frame[mask].copy()

    if len(signals) == 0:
        return None

    if direction == "LONG":

        returns = signals["future_12h"] - COST

    else:

        returns = -signals["future_12h"] - COST

    return {
        "signals": len(signals),
        "average": returns.mean(),
        "median": returns.median(),
        "win_rate": (returns > 0).mean(),
        "total": returns.sum()
    }

# ------------------------------------------------------------
# 7. DISCOVER STRATEGIES
# ------------------------------------------------------------

print("\nSearching strategy combinations...")

condition_names = list(conditions.keys())

results = []

# Test combinations of 2 and 3 conditions.
# This keeps the search broad without creating enormous
# numbers of fragile combinations.

for combination_size in [2, 3]:

    for combo in itertools.combinations(
        condition_names,
        combination_size
    ):

        for direction in ["LONG", "SHORT"]:

            result = evaluate_strategy(
                train,
                combo,
                direction
            )

            if result is None:
                continue

            # Require a minimum number of observations.
            if result["signals"] < 15:
                continue

            results.append({
                "strategy": " + ".join(combo),
                "conditions": combo,
                "direction": direction,
                "signals": result["signals"],
                "average": result["average"],
                "median": result["median"],
                "win_rate": result["win_rate"],
                "total": result["total"]
            })

results_df = pd.DataFrame(results)

# ------------------------------------------------------------
# 8. RANK DISCOVERED STRATEGIES
# ------------------------------------------------------------

# We favour average return but penalise strategies with
# weak win rates and very few observations.

results_df["quality"] = (
    results_df["average"] *
    np.sqrt(results_df["signals"])
)

results_df = results_df.sort_values(
    "quality",
    ascending=False
)

print(f"\nStrategies tested: {len(results_df)}")

print("\nTOP 10 STRATEGIES FOUND IN TRAINING DATA")
print("------------------------------------------")

display(
    results_df[
        [
            "direction",
            "strategy",
            "signals",
            "average",
            "win_rate",
            "quality"
        ]
    ].head(10)
)

# ------------------------------------------------------------
# 9. TEST THE BEST STRATEGIES ON UNSEEN DATA
# ------------------------------------------------------------

top = results_df.head(10).copy()

unseen_results = []

for _, row in top.iterrows():

    result = evaluate_strategy(
        test,
        row["conditions"],
        row["direction"]
    )

    if result is None:
        continue

    unseen_results.append({
        "direction": row["direction"],
        "strategy": row["strategy"],
        "training_signals": row["signals"],
        "training_avg": row["average"],
        "test_signals": result["signals"],
        "test_avg": result["average"],
        "test_win_rate": result["win_rate"],
        "test_total": result["total"]
    })

unseen_df = pd.DataFrame(unseen_results)

print("\n==========================================")
print("       UNSEEN TEST RESULTS")
print("==========================================")

if len(unseen_df) > 0:

    display(
        unseen_df[
            [
                "direction",
                "strategy",
                "training_signals",
                "training_avg",
                "test_signals",
                "test_avg",
                "test_win_rate"
            ]
        ]
    )

else:

    print("No usable strategies survived into the test period.")

# ------------------------------------------------------------
# 10. SUMMARY
# ------------------------------------------------------------

if len(unseen_df) > 0:

    best = unseen_df.sort_values(
        "test_avg",
        ascending=False
    ).iloc[0]

    print("\n==========================================")
    print("             AGENT CONCLUSION")
    print("==========================================")

    print(f"\nBest unseen-test direction: {best['direction']}")
    print(f"Conditions: {best['strategy']}")
    print(f"Test signals: {best['test_signals']}")
    print(
        f"Average 12h result: "
        f"{best['test_avg'] * 100:+.3f}%"
    )
    print(
        f"Win rate: "
        f"{best['test_win_rate'] * 100:.1f}%"
    )

    if best["test_avg"] > 0:
        print("\n🟢 The best discovered idea was positive on unseen data.")
        print("This is interesting — but NOT proof of a profitable strategy.")

    else:
        print("\n🔴 The discovered strategies did not survive the unseen test.")
        print("The agent needs better features or a different hypothesis.")

print("\n⚠️ Research only. No trades have been placed.")

     SILVER AGENT — V7 DISCOVERY ENGINE

Total usable candles: 1975
Training candles:     1382
Unseen test candles:  593

The test period remains hidden from strategy selection.

Searching strategy combinations...

Strategies tested: 1298

TOP 10 STRATEGIES FOUND IN TRAINING DATA
------------------------------------------


,direction,strategy,signals,average,win_rate,quality
509,SHORT,trend_down + mom3_up + low_volatility,159,0.004737,0.559748,0.059726
877,SHORT,price_below_MA20 + high_volume + high_volatility,97,0.005970,0.618557,0.058800
241,SHORT,high_volume + high_volatility,187,0.004187,0.577540,0.057259
779,SHORT,price_below_MA20 + mom3_up + low_volatility,95,0.005802,0.568421,0.056554
773,SHORT,price_below_MA20 + mom3_up + high_volume,48,0.007548,0.687500,0.052291
611,SHORT,trend_down + high_volume + high_volatility,100,0.005059,0.590000,0.050586
1121,SHORT,mom6_up + high_volume + high_volatility,81,0.005510,0.592593,0.049589
1261,SHORT,strong_bull_candle + high_volume + high_volati...,75,0.005600,0.573333,0.048500
1247,SHORT,mom12_down + high_volume + high_volatility,101,0.004780,0.594059,0.048041
363,SHORT,trend_up + mom6_up + high_volatility,183,0.003464,0.639344,0.046866



       UNSEEN TEST RESULTS


,direction,strategy,training_signals,training_avg,test_signals,test_avg,test_win_rate
0,SHORT,trend_down + mom3_up + low_volatility,159,0.004737,93,-0.005304,0.387097
1,SHORT,price_below_MA20 + high_volume + high_volatility,97,0.005970,37,0.003568,0.648649
2,SHORT,high_volume + high_volatility,187,0.004187,73,0.006071,0.602740
3,SHORT,price_below_MA20 + mom3_up + low_volatility,95,0.005802,52,-0.007568,0.346154
4,SHORT,price_below_MA20 + mom3_up + high_volume,48,0.007548,19,0.005458,0.631579
5,SHORT,trend_down + high_volume + high_volatility,100,0.005059,35,0.004074,0.628571
6,SHORT,mom6_up + high_volume + high_volatility,81,0.005510,37,0.005000,0.540541
7,SHORT,strong_bull_candle + high_volume + high_volati...,75,0.005600,35,0.005490,0.571429
8,SHORT,mom12_down + high_volume + high_volatility,101,0.004780,37,0.003867,0.648649
9,SHORT,trend_up + mom6_up + high_volatility,183,0.003464,87,0.005438,0.517241



             AGENT CONCLUSION

Best unseen-test direction: SHORT
Conditions: high_volume + high_volatility
Test signals: 73
Average 12h result: +0.607%
Win rate: 60.3%

🟢 The best discovered idea was positive on unseen data.
This is interesting — but NOT proof of a profitable strategy.

⚠️ Research only. No trades have been placed.


In [16]:
# ============================================================
# SILVER AGENT — V8a
# EXTENDED HISTORICAL DATA COLLECTOR
# ============================================================

import requests
import pandas as pd
import time

print("==========================================")
print("   SILVER AGENT — EXTENDED HISTORY")
print("==========================================")

TARGET_CANDLES = 20000
CHUNK_SIZE = 5000

all_candles = []
end_time = None

while len(all_candles) < TARGET_CANDLES:

    params = {
        "granularity": "H1",
        "price": "M",
        "count": CHUNK_SIZE
    }

    # After the first request, work backwards in time.
    if end_time is not None:
        params["to"] = end_time
        params["includeFirst"] = False

    url = f"{OANDA_URL}/v3/instruments/XAG_USD/candles"

    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=30
    )

    if response.status_code != 200:
        print("\n❌ OANDA REQUEST FAILED")
        print("HTTP status:", response.status_code)
        print(response.text)
        break

    batch = response.json().get("candles", [])

    if not batch:
        print("\nNo more historical candles available.")
        break

    # Keep completed candles only
    completed = [
        candle for candle in batch
        if candle.get("complete", False)
    ]

    if not completed:
        print("\nNo completed candles returned.")
        break

    all_candles.extend(completed)

    # Oldest candle becomes the endpoint for the next request
    end_time = completed[0]["time"]

    print(
        f"Downloaded {len(completed):,} candles | "
        f"Total: {len(all_candles):,} | "
        f"Oldest: {end_time}"
    )

    # Small pause to be polite to the API
    time.sleep(0.25)

# ------------------------------------------------------------
# Convert to DataFrame
# ------------------------------------------------------------

rows = []

for candle in all_candles:

    rows.append({
        "time": candle["time"],
        "open": float(candle["mid"]["o"]),
        "high": float(candle["mid"]["h"]),
        "low": float(candle["mid"]["l"]),
        "close": float(candle["mid"]["c"]),
        "volume": int(candle["volume"])
    })

silver_extended = pd.DataFrame(rows)

# Remove duplicates
silver_extended = silver_extended.drop_duplicates(
    subset=["time"]
)

# Convert time
silver_extended["time"] = pd.to_datetime(
    silver_extended["time"],
    utc=True
)

# Sort chronologically
silver_extended = silver_extended.sort_values(
    "time"
).reset_index(drop=True)

print("\n==========================================")
print("       EXTENDED HISTORY COMPLETE")
print("==========================================")

print(f"\nTotal unique candles: {len(silver_extended):,}")

print(
    f"Oldest: {silver_extended['time'].iloc[0]}"
)

print(
    f"Newest: {silver_extended['time'].iloc[-1]}"
)

print("\n✅ Historical dataset ready.")

# Replace our working dataset
silver = silver_extended.copy()

display(silver.tail(10))

   SILVER AGENT — EXTENDED HISTORY
Downloaded 4,999 candles | Total: 4,999 | Oldest: 2025-11-12T11:00:00.000000000Z

❌ OANDA REQUEST FAILED
HTTP status: 400
{"errorMessage":"The 'includeFirst' flag has no meaning unless the 'from' parameter is specified. 'includeFirst' is used to avoid seeing the same candle twice when the 'from' parameter is not on a candle boundary."}

       EXTENDED HISTORY COMPLETE

Total unique candles: 4,999
Oldest: 2025-11-12 11:00:00+00:00
Newest: 2026-09-17 15:00:00+00:00

✅ Historical dataset ready.


,time,open,high,low,close,volume
4989,2026-09-17 06:00:00+00:00,63.7695,63.9235,63.71050,63.8215,13001
4990,2026-09-17 07:00:00+00:00,63.8215,64.4565,63.78250,64.3385,13728
4991,2026-09-17 08:00:00+00:00,64.3390,64.3480,63.79465,63.8610,12394
4992,2026-09-17 09:00:00+00:00,63.8585,64.0550,63.85600,63.9215,10578
4993,2026-09-17 10:00:00+00:00,63.9225,64.1575,63.86200,64.1115,9815
4994,2026-09-17 11:00:00+00:00,64.1125,64.7770,64.08250,64.7685,14671
4995,2026-09-17 12:00:00+00:00,64.7690,65.5950,64.59000,65.4945,29322
4996,2026-09-17 13:00:00+00:00,65.4955,65.7695,65.25800,65.7385,33646
4997,2026-09-17 14:00:00+00:00,65.7410,66.1595,65.50350,65.5545,25768
4998,2026-09-17 15:00:00+00:00,65.5510,65.9145,65.54050,65.9050,18133


In [17]:
# ============================================================
# SILVER AGENT — V8b
# DATA QUALITY + FEATURE ENGINEERING
# ============================================================

import pandas as pd
import numpy as np

print("==========================================")
print("   SILVER AGENT — V8b")
print("   DATA QUALITY CHECK")
print("==========================================")

# ------------------------------------------------------------
# 1. Make sure data is correctly sorted
# ------------------------------------------------------------

silver = silver.sort_values("time").reset_index(drop=True)

# ------------------------------------------------------------
# 2. Check duplicates
# ------------------------------------------------------------

duplicates = silver["time"].duplicated().sum()

print(f"\nDuplicate timestamps: {duplicates}")

# ------------------------------------------------------------
# 3. Check hourly gaps
# ------------------------------------------------------------

time_diff = silver["time"].diff()

expected_gap = pd.Timedelta(hours=1)

gaps = silver.loc[
    time_diff > expected_gap,
    ["time"]
].copy()

gaps["gap"] = time_diff[time_diff > expected_gap].values

print(f"Hourly gaps found: {len(gaps)}")

if len(gaps) > 0:
    print("\nLargest gaps:")
    print(
        gaps.sort_values(
            "gap",
            ascending=False
        ).head(10).to_string(index=False)
    )

# ------------------------------------------------------------
# 4. Basic data sanity checks
# ------------------------------------------------------------

bad_prices = (
    (silver["high"] < silver["low"]) |
    (silver["high"] < silver["open"]) |
    (silver["high"] < silver["close"]) |
    (silver["low"] > silver["open"]) |
    (silver["low"] > silver["close"])
).sum()

bad_volume = (silver["volume"] <= 0).sum()

print(f"\nInvalid price candles: {bad_prices}")
print(f"Invalid volume candles: {bad_volume}")

# ------------------------------------------------------------
# 5. Build technical features
# ------------------------------------------------------------

data = silver.copy()

data["MA_5"] = data["close"].rolling(5).mean()
data["MA_10"] = data["close"].rolling(10).mean()
data["MA_20"] = data["close"].rolling(20).mean()
data["MA_50"] = data["close"].rolling(50).mean()

# Momentum
data["mom_3"] = data["close"].pct_change(3)
data["mom_6"] = data["close"].pct_change(6)
data["mom_12"] = data["close"].pct_change(12)
data["mom_24"] = data["close"].pct_change(24)

# Volatility
data["vol_10"] = data["close"].pct_change().rolling(10).std()
data["vol_20"] = data["close"].pct_change().rolling(20).std()

# Candle structure
data["candle_body"] = (
    data["close"] - data["open"]
) / data["open"]

data["candle_range"] = (
    data["high"] - data["low"]
) / data["open"]

# Recent highs/lows
data["high_20"] = data["high"].rolling(20).max()
data["low_20"] = data["low"].rolling(20).min()

# Relative volume
data["avg_volume_20"] = (
    data["volume"].rolling(20).mean()
)

data["relative_volume"] = (
    data["volume"] /
    data["avg_volume_20"]
)

# Trend conditions
data["trend_up"] = (
    data["MA_10"] > data["MA_20"]
)

data["trend_down"] = (
    data["MA_10"] < data["MA_20"]
)

data["price_above_MA20"] = (
    data["close"] > data["MA_20"]
)

data["price_below_MA20"] = (
    data["close"] < data["MA_20"]
)

# Momentum conditions
data["mom3_up"] = data["mom_3"] > 0
data["mom3_down"] = data["mom_3"] < 0

data["mom6_up"] = data["mom_6"] > 0
data["mom6_down"] = data["mom_6"] < 0

data["mom12_up"] = data["mom_12"] > 0
data["mom12_down"] = data["mom_12"] < 0

# ------------------------------------------------------------
# 6. Future returns
# ------------------------------------------------------------

data["future_6h"] = (
    data["close"].shift(-6) /
    data["close"] - 1
)

data["future_12h"] = (
    data["close"].shift(-12) /
    data["close"] - 1
)

data["future_24h"] = (
    data["close"].shift(-24) /
    data["close"] - 1
)

# ------------------------------------------------------------
# 7. Remove rows where indicators aren't ready
# ------------------------------------------------------------

data = data.dropna().reset_index(drop=True)

print("\n==========================================")
print("       FEATURE ENGINEERING COMPLETE")
print("==========================================")

print(f"\nUsable rows: {len(data):,}")

print(
    f"Period: {data['time'].iloc[0]} "
    f"→ {data['time'].iloc[-1]}"
)

print("\nLatest market data:")

display(
    data[
        [
            "time",
            "close",
            "MA_10",
            "MA_20",
            "MA_50",
            "mom_6",
            "mom_12",
            "vol_20"
        ]
    ].tail(10)
)

print("\n✅ V8b complete.")

   SILVER AGENT — V8b
   DATA QUALITY CHECK

Duplicate timestamps: 0
Hourly gaps found: 218

Largest gaps:
                     time             gap
2026-04-05 22:00:00+00:00 3 days 02:00:00
2026-06-21 22:00:00+00:00 2 days 06:00:00
2026-07-05 22:00:00+00:00 2 days 06:00:00
2025-11-30 23:00:00+00:00 2 days 04:00:00
2026-02-01 23:00:00+00:00 2 days 02:00:00
2026-07-12 22:00:00+00:00 2 days 02:00:00
2026-02-15 23:00:00+00:00 2 days 02:00:00
2026-02-08 23:00:00+00:00 2 days 02:00:00
2026-07-19 22:00:00+00:00 2 days 02:00:00
2026-04-19 22:00:00+00:00 2 days 02:00:00

Invalid price candles: 0
Invalid volume candles: 0

       FEATURE ENGINEERING COMPLETE

Usable rows: 4,926
Period: 2025-11-14 14:00:00+00:00 → 2026-09-16 14:00:00+00:00

Latest market data:


,time,close,MA_10,MA_20,MA_50,mom_6,mom_12,vol_20
4916,2026-09-16 05:00:00+00:00,64.5850,64.02705,63.731200,63.46139,0.014435,0.017800,0.005401
4917,2026-09-16 06:00:00+00:00,64.7275,64.13175,63.814575,63.47765,0.019917,0.019363,0.005362
4918,2026-09-16 07:00:00+00:00,64.5140,64.21470,63.881650,63.48974,0.014674,0.010241,0.005459
4919,2026-09-16 08:00:00+00:00,64.5370,64.29405,63.944025,63.50147,-0.002974,0.013450,0.005458
4920,2026-09-16 09:00:00+00:00,64.5110,64.37855,63.980325,63.51128,-0.001107,0.012978,0.005220
4921,2026-09-16 10:00:00+00:00,64.7465,64.50685,64.027075,63.53339,0.002974,0.015735,0.005265
4922,2026-09-16 11:00:00+00:00,64.7330,64.62205,64.106225,63.56172,0.002292,0.016759,0.004582
4923,2026-09-16 12:00:00+00:00,64.7700,64.62610,64.181150,63.59205,0.000657,0.020587,0.004582
4924,2026-09-16 13:00:00+00:00,64.4220,64.61005,64.229475,63.62784,-0.001426,0.013227,0.004787
4925,2026-09-16 14:00:00+00:00,64.7115,64.62575,64.290150,63.66648,0.002704,-0.000278,0.004859



✅ V8b complete.


In [ ]:
# ============================================================
# SILVER AGENT — V8c
# WALK-FORWARD STRATEGY TEST
# ============================================================

import pandas as pd
import numpy as np
from itertools import combinations

print("==========================================")
print("   SILVER AGENT — V8c")
print("   WALK-FORWARD TEST")
print("==========================================")

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

TRAIN_SIZE = 1200
TEST_SIZE = 400
STEP_SIZE = 400

COST = 0.0005

HORIZONS = {
    "6h": "future_6h",
    "12h": "future_12h",
    "24h": "future_24h"
}

# ------------------------------------------------------------
# CONDITIONS
# ------------------------------------------------------------

conditions = {

    "trend_up":
        data["trend_up"],

    "trend_down":
        data["trend_down"],

    "price_above_MA20":
        data["price_above_MA20"],

    "price_below_MA20":
        data["price_below_MA20"],

    "mom3_up":
        data["mom3_up"],

    "mom3_down":
        data["mom3_down"],

    "mom6_up":
        data["mom6_up"],

    "mom6_down":
        data["mom6_down"],

    "mom12_up":
        data["mom12_up"],

    "mom12_down":
        data["mom12_down"]
}

condition_names = list(conditions.keys())

# ------------------------------------------------------------
# STRATEGY DEFINITIONS
# ------------------------------------------------------------

strategies = []

for n in [2, 3]:

    for combo in combinations(condition_names, n):

        for direction in ["LONG", "SHORT"]:

            strategies.append({
                "direction": direction,
                "conditions": combo
            })

print(f"\nStrategies tested: {len(strategies):,}")

# ------------------------------------------------------------
# WALK-FORWARD ENGINE
# ------------------------------------------------------------

results = []

start = TRAIN_SIZE

window_number = 0

while start + TEST_SIZE <= len(data):

    window_number += 1

    train_start = start - TRAIN_SIZE
    train_end = start

    test_start = start
    test_end = start + TEST_SIZE

    train = data.iloc[
        train_start:train_end
    ].copy()

    test = data.iloc[
        test_start:test_end
    ].copy()

    # --------------------------------------------------------
    # Evaluate every strategy on training data
    # --------------------------------------------------------

    train_scores = []

    for strategy in strategies:

        mask = pd.Series(
            True,
            index=train.index
        )

        for condition in strategy["conditions"]:
            mask &= conditions[condition].iloc[
                train.index
            ]

        selected = train.loc[mask]

        if len(selected) < 30:
            continue

        if strategy["direction"] == "LONG":

            returns = (
                selected["future_12h"] - COST
            )

        else:

            returns = (
                -selected["future_12h"] - COST
            )

        train_scores.append({
            "direction":
                strategy["direction"],

            "conditions":
                strategy["conditions"],

            "train_signals":
                len(selected),

            "train_avg":
                returns.mean(),

            "train_win":
                (returns > 0).mean()
        })

    if not train_scores:
        start += STEP_SIZE
        continue

    # --------------------------------------------------------
    # Pick the best strategies from training ONLY
    # --------------------------------------------------------

    train_scores_df = pd.DataFrame(
        train_scores
    )

    train_scores_df["quality"] = (
        train_scores_df["train_avg"] *
        np.sqrt(train_scores_df["train_signals"])
    )

    train_scores_df = train_scores_df.sort_values(
        "quality",
        ascending=False
    )

    # Take top 10 training strategies
    selected_strategies = train_scores_df.head(10)

    # --------------------------------------------------------
    # Test selected strategies on unseen data
    # --------------------------------------------------------

    for _, strategy in selected_strategies.iterrows():

        mask = pd.Series(
            True,
            index=test.index
        )

        for condition in strategy["conditions"]:
            mask &= conditions[condition].iloc[
                test.index
            ]

        selected_test = test.loc[mask]

        if len(selected_test) < 5:
            continue

        if strategy["direction"] == "LONG":

            returns_12h = (
                selected_test["future_12h"] - COST
            )

            returns_6h = (
                selected_test["future_6h"] - COST
            )

            returns_24h = (
                selected_test["future_24h"] - COST
            )

        else:

            returns_12h = (
                -selected_test["future_12h"] - COST
            )

            returns_6h = (
                -selected_test["future_6h"] - COST
            )

            returns_24h = (
                -selected_test["future_24h"] - COST
            )

        results.append({

            "window":
                window_number,

            "test_start":
                test["time"].iloc[0],

            "test_end":
                test["time"].iloc[-1],

            "direction":
                strategy["direction"],

            "conditions":
                " + ".join(strategy["conditions"]),

            "signals":
                len(selected_test),

            "avg_6h":
                returns_6h.mean(),

            "avg_12h":
                returns_12h.mean(),

            "avg_24h":
                returns_24h.mean(),

            "win_12h":
                (returns_12h > 0).mean()
        })

    print(
        f"Window {window_number}: "
        f"{train['time'].iloc[0].date()} → "
        f"{test['time'].iloc[-1].date()} | "
        f"tested {len(selected_strategies)} candidates"
    )

    start += STEP_SIZE

# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

wf = pd.DataFrame(results)

print("\n==========================================")
print("       WALK-FORWARD TEST COMPLETE")
print("==========================================")

print(f"\nWalk-forward windows: {window_number}")
print(f"Test observations: {len(wf):,}")

if len(wf) > 0:

    summary = (
        wf
        .groupby(
            ["direction", "conditions"]
        )
        .agg(
            appearances=("window", "count"),
            total_signals=("signals", "sum"),
            avg_6h=("avg_6h", "mean"),
            avg_12h=("avg_12h", "mean"),
            avg_24h=("avg_24h", "mean"),
            avg_win=("win_12h", "mean")
        )
        .reset_index()
    )

    # Require repeated appearances
    summary = summary[
        summary["appearances"] >= 2
    ]

    summary = summary.sort_values(
        [
            "avg_12h",
            "avg_win"
        ],
        ascending=False
    )

    print("\nTOP WALK-FORWARD STRATEGIES:")
    display(
        summary.head(20)
    )

else:

    print("\nNo valid walk-forward results found.")

print("\n✅ V8c complete.")

   SILVER AGENT — V8c
   WALK-FORWARD TEST

Strategies tested: 330
Window 1: 2025-11-14 → 2026-02-24 | tested 10 candidates
Window 2: 2025-12-10 → 2026-03-19 | tested 10 candidates


In [ ]:
# ============================================================
# SILVER AGENT — V8d
# CANDIDATE STRATEGY STRESS TEST
# ============================================================

print("==========================================")
print("   SILVER AGENT — V8d")
print("   CANDIDATE STRESS TEST")
print("==========================================")

# ------------------------------------------------------------
# Candidate discovered by V8c
# ------------------------------------------------------------

CANDIDATE_DIRECTION = "SHORT"

CANDIDATE_CONDITIONS = [
    "trend_up",
    "price_above_MA20",
    "mom12_up"
]

TRAIN_SIZE = 1200
TEST_SIZE = 400
STEP_SIZE = 400

COST = 0.0005

candidate_results = []

start = TRAIN_SIZE
window_number = 0

while start + TEST_SIZE <= len(data):

    window_number += 1

    test = data.iloc[
        start:start + TEST_SIZE
    ].copy()

    # --------------------------------------------------------
    # Apply candidate WITHOUT retraining
    # --------------------------------------------------------

    mask = pd.Series(
        True,
        index=test.index
    )

    for condition in CANDIDATE_CONDITIONS:
        mask &= conditions[condition].iloc[test.index]

    selected = test.loc[mask]

    if len(selected) > 0:

        returns_6h = (
            -selected["future_6h"] - COST
        )

        returns_12h = (
            -selected["future_12h"] - COST
        )

        returns_24h = (
            -selected["future_24h"] - COST
        )

        candidate_results.append({

            "window":
                window_number,

            "test_start":
                test["time"].iloc[0],

            "test_end":
                test["time"].iloc[-1],

            "signals":
                len(selected),

            "avg_6h":
                returns_6h.mean(),

            "avg_12h":
                returns_12h.mean(),

            "avg_24h":
                returns_24h.mean(),

            "win_12h":
                (returns_12h > 0).mean()
        })

    start += STEP_SIZE

candidate_df = pd.DataFrame(candidate_results)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\n==========================================")
print("       STRESS TEST COMPLETE")
print("==========================================")

print("\nCandidate:")
print(
    "SHORT | trend_up + price_above_MA20 + mom12_up"
)

print(
    f"\nWalk-forward windows tested: {window_number}"
)

print(
    f"Windows with signals: {len(candidate_df)}"
)

if len(candidate_df) > 0:

    print("\nRESULT BY WINDOW:")

    display(candidate_df)

    print("\n------------------------------------------")
    print("AGGREGATE RESULTS")
    print("------------------------------------------")

    total_signals = candidate_df["signals"].sum()

    weighted_6h = np.average(
        candidate_df["avg_6h"],
        weights=candidate_df["signals"]
    )

    weighted_12h = np.average(
        candidate_df["avg_12h"],
        weights=candidate_df["signals"]
    )

    weighted_24h = np.average(
        candidate_df["avg_24h"],
        weights=candidate_df["signals"]
    )

    weighted_win = np.average(
        candidate_df["win_12h"],
        weights=candidate_df["signals"]
    )

    profitable_windows = (
        candidate_df["avg_12h"] > 0
    ).sum()

    print(
        f"\nTotal signals: {total_signals:,}"
    )

    print(
        f"Weighted average 6h: "
        f"{weighted_6h:.4%}"
    )

    print(
        f"Weighted average 12h: "
        f"{weighted_12h:.4%}"
    )

    print(
        f"Weighted average 24h: "
        f"{weighted_24h:.4%}"
    )

    print(
        f"Weighted 12h win rate: "
        f"{weighted_win:.2%}"
    )

    print(
        f"Profitable windows: "
        f"{profitable_windows}/{len(candidate_df)}"
    )

    print("\n✅ Candidate stress test complete.")

else:

    print("\nNo signals found.")

In [ ]:
# ============================================================
# SILVER AGENT — V8e
# STRATEGY vs MARKET BENCHMARK
# ============================================================

print("==========================================")
print("   SILVER AGENT — V8e")
print("   STRATEGY vs MARKET")
print("==========================================")

# ------------------------------------------------------------
# Candidate
# ------------------------------------------------------------

CANDIDATE_CONDITIONS = [
    "trend_up",
    "price_above_MA20",
    "mom12_up"
]

COST = 0.0005

TRAIN_SIZE = 1200
TEST_SIZE = 400
STEP_SIZE = 400

candidate_returns = []
market_returns = []

start = TRAIN_SIZE
window_number = 0

while start + TEST_SIZE <= len(data):

    window_number += 1

    test = data.iloc[
        start:start + TEST_SIZE
    ].copy()

    # --------------------------------------------------------
    # Candidate signal
    # --------------------------------------------------------

    mask = pd.Series(
        True,
        index=test.index
    )

    for condition in CANDIDATE_CONDITIONS:
        mask &= conditions[condition].iloc[test.index]

    selected = test.loc[mask]

    if len(selected) > 0:

        # Actual silver 24h return
        actual_24h = selected["future_24h"]

        # Candidate is SHORT
        short_return = (
            -actual_24h - COST
        )

        for value in actual_24h:
            market_returns.append({
                "window": window_number,
                "market_24h": value
            })

        for value in short_return:
            candidate_returns.append({
                "window": window_number,
                "short_24h": value
            })

    start += STEP_SIZE

candidate_df = pd.DataFrame(candidate_returns)
market_df = pd.DataFrame(market_returns)

# ------------------------------------------------------------
# Merge
# ------------------------------------------------------------

comparison = pd.concat(
    [
        candidate_df.reset_index(drop=True),
        market_df["market_24h"].reset_index(drop=True)
    ],
    axis=1
)

comparison["long_return"] = (
    comparison["market_24h"] - COST
)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\n==========================================")
print("          BENCHMARK RESULTS")
print("==========================================")

if len(comparison) > 0:

    print(
        f"\nSignals analysed: "
        f"{len(comparison):,}"
    )

    print("\nAverage 24h return:")

    print(
        f"Market / LONG: "
        f"{comparison['long_return'].mean():.4%}"
    )

    print(
        f"Candidate SHORT: "
        f"{comparison['short_24h'].mean():.4%}"
    )

    print(
        f"Difference: "
        f"{comparison['short_24h'].mean() - comparison['long_return'].mean():.4%}"
    )

    print("\nWin rates:")

    print(
        f"LONG: "
        f"{(comparison['long_return'] > 0).mean():.2%}"
    )

    print(
        f"SHORT: "
        f"{(comparison['short_24h'] > 0).mean():.2%}"
    )

    print("\nMedian 24h return:")

    print(
        f"LONG: "
        f"{comparison['long_return'].median():.4%}"
    )

    print(
        f"SHORT: "
        f"{comparison['short_24h'].median():.4%}"
    )

    # --------------------------------------------------------
    # Per-window comparison
    # --------------------------------------------------------

    window_summary = (
        comparison
        .assign(
            candidate_win=
                comparison["short_24h"] > 0,

            market_win=
                comparison["long_return"] > 0
        )
        .groupby("window")
        .agg(
            signals=("short_24h", "count"),

            market_avg=
                ("long_return", "mean"),

            short_avg=
                ("short_24h", "mean"),

            market_win_rate=
                ("market_win", "mean"),

            short_win_rate=
                ("candidate_win", "mean")
        )
        .reset_index()
    )

    window_summary["short_advantage"] = (
        window_summary["short_avg"] -
        window_summary["market_avg"]
    )

    print("\nRESULT BY WALK-FORWARD WINDOW:")

    display(window_summary)

    print("\n==========================================")
    print("             FINAL VERDICT")
    print("==========================================")

    advantage = (
        comparison["short_24h"].mean()
        -
        comparison["long_return"].mean()
    )

    if advantage > 0:
        print(
            "\n🟢 SHORT candidate beat the "
            "LONG benchmark on average."
        )
    else:
        print(
            "\n🔴 SHORT candidate did NOT beat "
            "the LONG benchmark."
        )

else:

    print("\nNo candidate signals found.")

print("\n✅ V8e complete.")

In [ ]:
# ============================================================
# SILVER AGENT — V9a
# MARKET REGIME DETECTION
# ============================================================

print("==========================================")
print("   SILVER AGENT — V9a")
print("   MARKET REGIME DETECTION")
print("==========================================")

regime = data.copy()

# ------------------------------------------------------------
# 1. Volatility regime
# ------------------------------------------------------------

vol_median = regime["vol_20"].median()

regime["volatility_regime"] = np.where(
    regime["vol_20"] >= vol_median,
    "HIGH_VOL",
    "LOW_VOL"
)

# ------------------------------------------------------------
# 2. Trend strength
# ------------------------------------------------------------

regime["trend_strength"] = (
    abs(regime["MA_10"] - regime["MA_20"])
    / regime["close"]
)

trend_median = regime["trend_strength"].median()

regime["trend_regime"] = np.where(
    regime["trend_strength"] >= trend_median,
    "STRONG_TREND",
    "WEAK_TREND"
)

# ------------------------------------------------------------
# 3. Momentum regime
# ------------------------------------------------------------

momentum_median = regime["mom_12"].median()

regime["momentum_regime"] = np.where(
    regime["mom_12"] >= momentum_median,
    "POSITIVE_MOMENTUM",
    "NEGATIVE_MOMENTUM"
)

# ------------------------------------------------------------
# 4. Price location
# ------------------------------------------------------------

regime["price_regime"] = np.where(
    regime["close"] >= regime["MA_20"],
    "ABOVE_MA20",
    "BELOW_MA20"
)

# ------------------------------------------------------------
# 5. Range regime
# ------------------------------------------------------------

range_median = regime["candle_range"].median()

regime["range_regime"] = np.where(
    regime["candle_range"] >= range_median,
    "LARGE_RANGE",
    "SMALL_RANGE"
)

# ------------------------------------------------------------
# 6. Combined regime
# ------------------------------------------------------------

regime["market_regime"] = (
    regime["volatility_regime"]
    + " | "
    + regime["trend_regime"]
    + " | "
    + regime["momentum_regime"]
)

# ------------------------------------------------------------
# 7. Candidate signal
# ------------------------------------------------------------

candidate_mask = (
    regime["trend_up"]
    &
    regime["price_above_MA20"]
    &
    regime["mom12_up"]
)

regime["candidate_signal"] = candidate_mask

# ------------------------------------------------------------
# 8. Candidate SHORT return
# ------------------------------------------------------------

regime["candidate_return_24h"] = np.where(
    regime["candidate_signal"],
    -regime["future_24h"] - COST,
    np.nan
)

# ------------------------------------------------------------
# 9. Regime statistics
# ------------------------------------------------------------

regime_summary = (
    regime
    .groupby("market_regime")
    .agg(
        observations=("close", "count"),
        signals=("candidate_signal", "sum"),
        avg_short_return=("candidate_return_24h", "mean"),
        win_rate=(
            "candidate_return_24h",
            lambda x: (x > 0).mean()
        )
    )
    .reset_index()
)

regime_summary = regime_summary[
    regime_summary["signals"] >= 20
].copy()

regime_summary = regime_summary.sort_values(
    "avg_short_return",
    ascending=False
)

print("\n==========================================")
print("          REGIME RESULTS")
print("==========================================")

display(regime_summary)

# ------------------------------------------------------------
# 10. Detailed regime combinations
# ------------------------------------------------------------

detailed_summary = (
    regime[
        regime["candidate_signal"]
    ]
    .groupby(
        [
            "volatility_regime",
            "trend_regime",
            "momentum_regime"
        ]
    )
    .agg(
        signals=("candidate_signal", "count"),
        avg_24h=("candidate_return_24h", "mean"),
        median_24h=(
            "candidate_return_24h",
            "median"
        ),
        win_rate=(
            "candidate_return_24h",
            lambda x: (x > 0).mean()
        )
    )
    .reset_index()
)

detailed_summary = detailed_summary.sort_values(
    "avg_24h",
    ascending=False
)

print("\n==========================================")
print("       DETAILED CANDIDATE REGIMES")
print("==========================================")

display(detailed_summary)

print("\n✅ V9a complete.")

In [ ]:
# ============================================================
# SILVER AGENT — V9b
# REGIME ROBUSTNESS + DRAWDOWN TEST
# ============================================================

print("==========================================")
print("   SILVER AGENT — V9b")
print("   REGIME ROBUSTNESS TEST")
print("==========================================")

# ------------------------------------------------------------
# Target regime
# ------------------------------------------------------------

target_mask = (
    regime["candidate_signal"]
    &
    (regime["volatility_regime"] == "HIGH_VOL")
    &
    (regime["trend_regime"] == "STRONG_TREND")
    &
    (regime["momentum_regime"] == "POSITIVE_MOMENTUM")
)

target = regime.loc[target_mask].copy()

# SHORT return
target["strategy_return"] = (
    -target["future_24h"] - COST
)

returns = target["strategy_return"].dropna()

print("\n==========================================")
print("          TARGET REGIME")
print("==========================================")

print(
    "\nHIGH_VOL + STRONG_TREND + "
    "POSITIVE_MOMENTUM"
)

print(f"\nSignals: {len(returns):,}")

# ------------------------------------------------------------
# Distribution
# ------------------------------------------------------------

print("\n------------------------------------------")
print("RETURN DISTRIBUTION")
print("------------------------------------------")

print(
    f"Mean:       {returns.mean():.4%}"
)

print(
    f"Median:     {returns.median():.4%}"
)

print(
    f"25th pct:   {returns.quantile(0.25):.4%}"
)

print(
    f"75th pct:   {returns.quantile(0.75):.4%}"
)

print(
    f"Minimum:    {returns.min():.4%}"
)

print(
    f"Maximum:    {returns.max():.4%}"
)

print(
    f"Win rate:   {(returns > 0).mean():.2%}"
)

# ------------------------------------------------------------
# Remove largest winners
# ------------------------------------------------------------

sorted_returns = returns.sort_values(
    ascending=False
)

for percentage in [1, 5, 10]:

    n_remove = max(
        1,
        int(len(sorted_returns) * percentage / 100)
    )

    trimmed = sorted_returns.iloc[n_remove:]

    print(
        f"\nMean after removing top "
        f"{percentage}% winners: "
        f"{trimmed.mean():.4%}"
    )

# ------------------------------------------------------------
# Equal-weight cumulative return
# ------------------------------------------------------------

equity = (1 + returns.reset_index(drop=True)).cumprod()

peak = equity.cummax()

drawdown = (
    equity / peak
) - 1

max_drawdown = drawdown.min()

print("\n------------------------------------------")
print("EQUITY / DRAWDOWN")
print("------------------------------------------")

print(
    f"Final cumulative return: "
    f"{equity.iloc[-1] - 1:.2%}"
)

print(
    f"Maximum drawdown: "
    f"{max_drawdown:.2%}"
)

# ------------------------------------------------------------
# Walk-forward regime results
# ------------------------------------------------------------

target["wf_window"] = (
    ((target.index - TRAIN_SIZE) // STEP_SIZE) + 1
)

window_results = (
    target
    .groupby("wf_window")
    .agg(
        signals=("strategy_return", "count"),
        avg_return=("strategy_return", "mean"),
        median_return=("strategy_return", "median"),
        win_rate=(
            "strategy_return",
            lambda x: (x > 0).mean()
        )
    )
    .reset_index()
)

print("\n==========================================")
print("       REGIME BY WALK-FORWARD")
print("==========================================")

display(window_results)

# ------------------------------------------------------------
# Positive windows
# ------------------------------------------------------------

positive_windows = (
    window_results["avg_return"] > 0
).sum()

print(
    f"\nProfitable windows: "
    f"{positive_windows}/"
    f"{len(window_results)}"
)

print("\n==========================================")
print("             V9b COMPLETE")
print("==========================================")

In [ ]:
# ============================================================
# SILVER AGENT — V10a
# SEQUENTIAL TRADE SIMULATOR
# ============================================================

print("==========================================")
print("   SILVER AGENT — V10a")
print("   SEQUENTIAL TRADE SIMULATOR")
print("==========================================")

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

STOP_MULTIPLIER = 1.5
TARGET_MULTIPLIER = 2.0
MAX_HOLD = 24

COST = 0.0005

INITIAL_CAPITAL = 1000.0

# ------------------------------------------------------------
# PROMISING REGIME
# ------------------------------------------------------------

signal = (
    regime["trend_up"]
    &
    regime["price_above_MA20"]
    &
    regime["mom12_up"]
    &
    (regime["volatility_regime"] == "HIGH_VOL")
    &
    (regime["trend_regime"] == "STRONG_TREND")
    &
    (regime["momentum_regime"] == "POSITIVE_MOMENTUM")
)

# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------

sim = regime.copy().reset_index(drop=True)

sim["signal"] = signal.reset_index(drop=True)

# Volatility expressed as percentage move
sim["atr_proxy"] = (
    sim["candle_range"]
    .rolling(20)
    .mean()
)

# ------------------------------------------------------------
# Sequential simulation
# ------------------------------------------------------------

capital = INITIAL_CAPITAL

equity_curve = []

trades = []

i = 20

while i < len(sim) - MAX_HOLD - 1:

    # --------------------------------------------------------
    # Look for entry
    # --------------------------------------------------------

    if not sim.loc[i, "signal"]:
        equity_curve.append({
            "time": sim.loc[i, "time"],
            "capital": capital
        })

        i += 1
        continue

    entry_price = sim.loc[i, "close"]

    volatility = sim.loc[i, "atr_proxy"]

    if pd.isna(volatility) or volatility <= 0:
        i += 1
        continue

    # --------------------------------------------------------
    # SHORT trade levels
    # --------------------------------------------------------

    stop_price = (
        entry_price *
        (1 + STOP_MULTIPLIER * volatility)
    )

    target_price = (
        entry_price *
        (1 - TARGET_MULTIPLIER * volatility)
    )

    exit_price = None
    exit_reason = None
    exit_index = None

    # --------------------------------------------------------
    # Manage trade
    # --------------------------------------------------------

    for j in range(
        i + 1,
        min(
            i + MAX_HOLD + 1,
            len(sim)
        )
    ):

        high = sim.loc[j, "high"]
        low = sim.loc[j, "low"]

        # ----------------------------------------------------
        # SHORT stop
        # ----------------------------------------------------

        if high >= stop_price:

            exit_price = stop_price
            exit_reason = "STOP"
            exit_index = j
            break

        # ----------------------------------------------------
        # SHORT target
        # ----------------------------------------------------

        if low <= target_price:

            exit_price = target_price
            exit_reason = "TARGET"
            exit_index = j
            break

    # --------------------------------------------------------
    # Maximum holding period
    # --------------------------------------------------------

    if exit_price is None:

        exit_index = min(
            i + MAX_HOLD,
            len(sim) - 1
        )

        exit_price = sim.loc[
            exit_index,
            "close"
        ]

        exit_reason = "TIME"

    # --------------------------------------------------------
    # Calculate SHORT return
    # --------------------------------------------------------

    gross_return = (
        entry_price / exit_price
    ) - 1

    net_return = gross_return - COST

    capital_before = capital

    capital = capital * (
        1 + net_return
    )

    trades.append({

        "entry_time":
            sim.loc[i, "time"],

        "exit_time":
            sim.loc[exit_index, "time"],

        "entry":
            entry_price,

        "exit":
            exit_price,

        "stop":
            stop_price,

        "target":
            target_price,

        "reason":
            exit_reason,

        "return":
            net_return,

        "capital_before":
            capital_before,

        "capital_after":
            capital
    })

    equity_curve.append({
        "time":
            sim.loc[exit_index, "time"],

        "capital":
            capital
    })

    # --------------------------------------------------------
    # IMPORTANT:
    # Skip forward until trade is finished.
    # This prevents overlapping positions.
    # --------------------------------------------------------

    i = exit_index + 1

# ------------------------------------------------------------
# Convert results
# ------------------------------------------------------------

trades_df = pd.DataFrame(trades)

equity_df = pd.DataFrame(
    equity_curve
)

print("\n==========================================")
print("       SIMULATION COMPLETE")
print("==========================================")

if len(trades_df) > 0:

    returns = trades_df["return"]

    # --------------------------------------------------------
    # Drawdown
    # --------------------------------------------------------

    equity = trades_df[
        "capital_after"
    ]

    running_peak = equity.cummax()

    drawdown = (
        equity / running_peak
    ) - 1

    max_drawdown = drawdown.min()

    # --------------------------------------------------------
    # Profit factor
    # --------------------------------------------------------

    gross_profit = returns[
        returns > 0
    ].sum()

    gross_loss = abs(
        returns[returns < 0].sum()
    )

    if gross_loss > 0:
        profit_factor = (
            gross_profit / gross_loss
        )
    else:
        profit_factor = np.inf

    # --------------------------------------------------------
    # Results
    # --------------------------------------------------------

    print(
        f"\nInitial capital: "
        f"£{INITIAL_CAPITAL:,.2f}"
    )

    print(
        f"Final simulated capital: "
        f"£{capital:,.2f}"
    )

    print(
        f"Total return: "
        f"{(capital / INITIAL_CAPITAL - 1):.2%}"
    )

    print(
        f"Trades: "
        f"{len(trades_df):,}"
    )

    print(
        f"Win rate: "
        f"{(returns > 0).mean():.2%}"
    )

    print(
        f"Average trade: "
        f"{returns.mean():.4%}"
    )

    print(
        f"Median trade: "
        f"{returns.median():.4%}"
    )

    print(
        f"Best trade: "
        f"{returns.max():.2%}"
    )

    print(
        f"Worst trade: "
        f"{returns.min():.2%}"
    )

    print(
        f"Profit factor: "
        f"{profit_factor:.2f}"
    )

    print(
        f"Maximum drawdown: "
        f"{max_drawdown:.2%}"
    )

    # --------------------------------------------------------
    # Exit breakdown
    # --------------------------------------------------------

    print("\nEXIT REASONS:")

    exit_summary = (
        trades_df
        .groupby("reason")
        .agg(
            trades=("return", "count"),
            avg_return=("return", "mean"),
            win_rate=(
                "return",
                lambda x: (x > 0).mean()
            )
        )
        .reset_index()
    )

    display(exit_summary)

    print("\nLAST 20 TRADES:")

    display(
        trades_df.tail(20)
    )

    print("\n==========================================")
    print("             V10a COMPLETE")
    print("==========================================")

else:

    print("\nNo trades generated.")

    print("\n==========================================")
    print("             V10a COMPLETE")
    print("==========================================")

In [ ]:
# ============================================================
# SILVER AGENT — V10b
# HONEST ATR-BASED BACKTESTER
# ============================================================

print("==========================================")
print("   SILVER AGENT — V10b")
print("   HONEST ATR BACKTESTER")
print("==========================================")

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

INITIAL_CAPITAL = 1000.0
RISK_PER_TRADE = 0.01
MAX_HOLD = 24
COST = 0.0005

PARAMETERS = [
    (1.0, 1.5),
    (1.0, 2.0),
    (1.5, 2.0),
    (2.0, 3.0)
]

# ------------------------------------------------------------
# Build proper True Range and ATR
# ------------------------------------------------------------

bt = regime.copy().reset_index(drop=True)

bt["previous_close"] = bt["close"].shift(1)

bt["TR_1"] = (
    bt["high"] - bt["low"]
)

bt["TR_2"] = (
    abs(bt["high"] - bt["previous_close"])
)

bt["TR_3"] = (
    abs(bt["low"] - bt["previous_close"])
)

bt["true_range"] = bt[
    ["TR_1", "TR_2", "TR_3"]
].max(axis=1)

bt["ATR_20"] = (
    bt["true_range"]
    .rolling(20)
    .mean()
)

# ------------------------------------------------------------
# Candidate regime
# ------------------------------------------------------------

bt["signal"] = (
    bt["trend_up"]
    &
    bt["price_above_MA20"]
    &
    bt["mom12_up"]
    &
    (bt["volatility_regime"] == "HIGH_VOL")
    &
    (bt["trend_regime"] == "STRONG_TREND")
    &
    (bt["momentum_regime"] == "POSITIVE_MOMENTUM")
)

# ------------------------------------------------------------
# Backtest function
# ------------------------------------------------------------

def run_backtest(
    stop_atr,
    target_atr
):

    capital = INITIAL_CAPITAL

    trades = []

    i = 20

    while i < len(bt) - MAX_HOLD - 1:

        if not bt.loc[i, "signal"]:
            i += 1
            continue

        entry_price = bt.loc[i, "close"]

        atr = bt.loc[i, "ATR_20"]

        if pd.isna(atr) or atr <= 0:
            i += 1
            continue

        # ----------------------------------------------------
        # SHORT position
        # ----------------------------------------------------

        stop_distance = (
            stop_atr * atr
        )

        target_distance = (
            target_atr * atr
        )

        stop_price = (
            entry_price + stop_distance
        )

        target_price = (
            entry_price - target_distance
        )

        # ----------------------------------------------------
        # Risk-based position sizing
        # ----------------------------------------------------

        risk_amount = (
            capital * RISK_PER_TRADE
        )

        # Position size in price units
        position_size = (
            risk_amount / stop_distance
        )

        exit_price = None
        exit_reason = None
        exit_index = None

        # ----------------------------------------------------
        # Manage position
        # ----------------------------------------------------

        for j in range(
            i + 1,
            min(
                i + MAX_HOLD + 1,
                len(bt)
            )
        ):

            high = bt.loc[j, "high"]
            low = bt.loc[j, "low"]

            hit_stop = (
                high >= stop_price
            )

            hit_target = (
                low <= target_price
            )

            # ------------------------------------------------
            # Conservative ambiguity rule:
            # If both are hit in same candle,
            # assume STOP happened first.
            # ------------------------------------------------

            if hit_stop and hit_target:

                exit_price = stop_price
                exit_reason = "AMBIGUOUS_STOP"
                exit_index = j
                break

            elif hit_stop:

                exit_price = stop_price
                exit_reason = "STOP"
                exit_index = j
                break

            elif hit_target:

                exit_price = target_price
                exit_reason = "TARGET"
                exit_index = j
                break

        # ----------------------------------------------------
        # Time exit
        # ----------------------------------------------------

        if exit_price is None:

            exit_index = min(
                i + MAX_HOLD,
                len(bt) - 1
            )

            exit_price = bt.loc[
                exit_index,
                "close"
            ]

            exit_reason = "TIME"

        # ----------------------------------------------------
        # SHORT P&L
        # ----------------------------------------------------

        price_return = (
            entry_price - exit_price
        ) / entry_price

        # Transaction cost
        price_return -= COST

        pnl = (
            position_size
            * (entry_price - exit_price)
        )

        # Apply proportional transaction cost
        pnl -= (
            abs(position_size * entry_price)
            * COST
        )

        # Cap loss at risk amount
        if pnl < -risk_amount:
            pnl = -risk_amount

        capital_before = capital

        capital = max(
            0.0,
            capital + pnl
        )

        trades.append({

            "entry_time":
                bt.loc[i, "time"],

            "exit_time":
                bt.loc[exit_index, "time"],

            "entry":
                entry_price,

            "exit":
                exit_price,

            "stop":
                stop_price,

            "target":
                target_price,

            "reason":
                exit_reason,

            "return":
                price_return,

            "pnl":
                pnl,

            "capital_before":
                capital_before,

            "capital_after":
                capital
        })

        # No overlapping trades
        i = exit_index + 1

    return pd.DataFrame(trades)

# ------------------------------------------------------------
# Run all parameter combinations
# ------------------------------------------------------------

all_results = []

for stop_atr, target_atr in PARAMETERS:

    trades = run_backtest(
        stop_atr,
        target_atr
    )

    if len(trades) == 0:
        continue

    returns = trades["return"]
    pnl = trades["pnl"]

    wins = pnl[pnl > 0]
    losses = pnl[pnl < 0]

    gross_profit = wins.sum()
    gross_loss = abs(losses.sum())

    profit_factor = (
        gross_profit / gross_loss
        if gross_loss > 0
        else np.inf
    )

    equity = trades[
        "capital_after"
    ]

    peak = equity.cummax()

    drawdown = (
        equity / peak
    ) - 1

    all_results.append({

        "stop_ATR":
            stop_atr,

        "target_ATR":
            target_atr,

        "trades":
            len(trades),

        "final_capital":
            trades["capital_after"].iloc[-1],

        "total_return":
            (
                trades["capital_after"].iloc[-1]
                / INITIAL_CAPITAL
            ) - 1,

        "win_rate":
            (pnl > 0).mean(),

        "avg_trade":
            pnl.mean(),

        "profit_factor":
            profit_factor,

        "max_drawdown":
            drawdown.min(),

        "best_trade":
            pnl.max(),

        "worst_trade":
            pnl.min()
    })

results_df = pd.DataFrame(
    all_results
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\n==========================================")
print("       ATR BACKTEST RESULTS")
print("==========================================")

display(
    results_df.sort_values(
        "profit_factor",
        ascending=False
    )
)

print("\n==========================================")
print("             V10b COMPLETE")
print("==========================================")

print(
    "\n⚠️ These are historical simulations only."
)

In [ ]:
# ============================================================
# SILVER AGENT — V11
# LOCKED OUT-OF-SAMPLE TEST
# ============================================================

print("==========================================")
print("   SILVER AGENT — V11")
print("   LOCKED OUT-OF-SAMPLE TEST")
print("==========================================")

# ------------------------------------------------------------
# LOCKED PARAMETERS
# ------------------------------------------------------------

INITIAL_CAPITAL = 1000.0
RISK_PER_TRADE = 0.01
MAX_HOLD = 24
COST = 0.0005

STOP_ATR = 1.0
TARGET_ATR = 1.5

# ------------------------------------------------------------
# Create proper ATR
# ------------------------------------------------------------

oos = regime.copy().reset_index(drop=True)

oos["previous_close"] = oos["close"].shift(1)

oos["TR_1"] = (
    oos["high"] - oos["low"]
)

oos["TR_2"] = (
    abs(oos["high"] - oos["previous_close"])
)

oos["TR_3"] = (
    abs(oos["low"] - oos["previous_close"])
)

oos["true_range"] = oos[
    ["TR_1", "TR_2", "TR_3"]
].max(axis=1)

oos["ATR_20"] = (
    oos["true_range"]
    .rolling(20)
    .mean()
)

# ------------------------------------------------------------
# Locked signal
# ------------------------------------------------------------

oos["signal"] = (
    oos["trend_up"]
    &
    oos["price_above_MA20"]
    &
    oos["mom12_up"]
    &
    (oos["volatility_regime"] == "HIGH_VOL")
    &
    (oos["trend_regime"] == "STRONG_TREND")
    &
    (oos["momentum_regime"] == "POSITIVE_MOMENTUM")
)

# ------------------------------------------------------------
# IMPORTANT:
# Latest 25% is completely untouched.
# ------------------------------------------------------------

split_point = int(
    len(oos) * 0.75
)

oos_test = oos.iloc[
    split_point:
].copy().reset_index(drop=True)

print("\n------------------------------------------")
print("OUT-OF-SAMPLE PERIOD")
print("------------------------------------------")

print(
    f"Start: {oos_test['time'].iloc[0]}"
)

print(
    f"End:   {oos_test['time'].iloc[-1]}"
)

print(
    f"Candles: {len(oos_test):,}"
)

# ------------------------------------------------------------
# Sequential backtest
# ------------------------------------------------------------

capital = INITIAL_CAPITAL

trades = []

i = 20

while i < len(oos_test) - MAX_HOLD - 1:

    if not oos_test.loc[i, "signal"]:
        i += 1
        continue

    entry_price = oos_test.loc[
        i, "close"
    ]

    atr = oos_test.loc[
        i, "ATR_20"
    ]

    if pd.isna(atr) or atr <= 0:
        i += 1
        continue

    stop_distance = (
        STOP_ATR * atr
    )

    target_distance = (
        TARGET_ATR * atr
    )

    stop_price = (
        entry_price + stop_distance
    )

    target_price = (
        entry_price - target_distance
    )

    # Risk 1% of current capital
    risk_amount = (
        capital * RISK_PER_TRADE
    )

    position_size = (
        risk_amount / stop_distance
    )

    exit_price = None
    exit_reason = None
    exit_index = None

    for j in range(
        i + 1,
        min(
            i + MAX_HOLD + 1,
            len(oos_test)
        )
    ):

        high = oos_test.loc[
            j, "high"
        ]

        low = oos_test.loc[
            j, "low"
        ]

        hit_stop = (
            high >= stop_price
        )

        hit_target = (
            low <= target_price
        )

        # Conservative rule:
        # if both occur in the same candle,
        # assume the stop happened first.

        if hit_stop and hit_target:

            exit_price = stop_price
            exit_reason = "AMBIGUOUS_STOP"
            exit_index = j
            break

        elif hit_stop:

            exit_price = stop_price
            exit_reason = "STOP"
            exit_index = j
            break

        elif hit_target:

            exit_price = target_price
            exit_reason = "TARGET"
            exit_index = j
            break

    # --------------------------------------------------------
    # Time exit
    # --------------------------------------------------------

    if exit_price is None:

        exit_index = min(
            i + MAX_HOLD,
            len(oos_test) - 1
        )

        exit_price = oos_test.loc[
            exit_index,
            "close"
        ]

        exit_reason = "TIME"

    # --------------------------------------------------------
    # P&L
    # --------------------------------------------------------

    pnl = (
        position_size
        * (entry_price - exit_price)
    )

    # Transaction cost
    pnl -= (
        abs(position_size * entry_price)
        * COST
    )

    # Never allow loss greater than planned risk
    pnl = max(
        pnl,
        -risk_amount
    )

    capital_before = capital

    capital = max(
        0,
        capital + pnl
    )

    trades.append({

        "entry_time":
            oos_test.loc[i, "time"],

        "exit_time":
            oos_test.loc[exit_index, "time"],

        "entry":
            entry_price,

        "exit":
            exit_price,

        "reason":
            exit_reason,

        "pnl":
            pnl,

        "return_on_risk":
            pnl / risk_amount,

        "capital_before":
            capital_before,

        "capital_after":
            capital
    })

    # No overlapping trades
    i = exit_index + 1

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

oos_trades = pd.DataFrame(
    trades
)

print("\n==========================================")
print("       OUT-OF-SAMPLE RESULTS")
print("==========================================")

if len(oos_trades) == 0:

    print("\n❌ No trades generated.")

else:

    pnl = oos_trades["pnl"]

    wins = pnl[pnl > 0]
    losses = pnl[pnl < 0]

    gross_profit = wins.sum()
    gross_loss = abs(
        losses.sum()
    )

    profit_factor = (
        gross_profit / gross_loss
        if gross_loss > 0
        else np.inf
    )

    equity = oos_trades[
        "capital_after"
    ]

    peak = equity.cummax()

    drawdown = (
        equity / peak
    ) - 1

    total_return = (
        capital / INITIAL_CAPITAL
    ) - 1

    print(
        f"\nInitial capital: "
        f"£{INITIAL_CAPITAL:,.2f}"
    )

    print(
        f"Final capital: "
        f"£{capital:,.2f}"
    )

    print(
        f"Total return: "
        f"{total_return:.2%}"
    )

    print(
        f"Trades: "
        f"{len(oos_trades):,}"
    )

    print(
        f"Win rate: "
        f"{(pnl > 0).mean():.2%}"
    )

    print(
        f"Average P&L/trade: "
        f"£{pnl.mean():.2f}"
    )

    print(
        f"Profit factor: "
        f"{profit_factor:.2f}"
    )

    print(
        f"Maximum drawdown: "
        f"{drawdown.min():.2%}"
    )

    print(
        f"Best trade: "
        f"£{pnl.max():.2f}"
    )

    print(
        f"Worst trade: "
        f"£{pnl.min():.2f}"
    )

    print("\nEXIT BREAKDOWN:")

    exit_summary = (
        oos_trades
        .groupby("reason")
        .agg(
            trades=("pnl", "count"),
            avg_pnl=("pnl", "mean"),
            total_pnl=("pnl", "sum"),
            win_rate=(
                "pnl",
                lambda x: (x > 0).mean()
            )
        )
        .reset_index()
    )

    display(exit_summary)

    print("\nLAST 15 OUT-OF-SAMPLE TRADES:")

    display(
        oos_trades.tail(15)
    )

print("\n==========================================")
print("             V11 COMPLETE")
print("==========================================")

print(
    "\n⚠️ Parameters were locked before this test."
)

print(
    "⚠️ No real trades were placed."
)

In [ ]:

# ============================================================
# V12a — MARKET STATE DATASET
# ============================================================

import numpy as np
import pandas as pd

df = silver_extended.copy()

# ------------------------------------------------------------
# 1. Basic price features
# ------------------------------------------------------------

df["return_1h"]  = df["close"].pct_change(1)
df["return_3h"]  = df["close"].pct_change(3)
df["return_6h"]  = df["close"].pct_change(6)
df["return_12h"] = df["close"].pct_change(12)
df["return_24h"] = df["close"].pct_change(24)

# ------------------------------------------------------------
# 2. Moving averages / trend
# ------------------------------------------------------------

df["ma5"]  = df["close"].rolling(5).mean()
df["ma10"] = df["close"].rolling(10).mean()
df["ma20"] = df["close"].rolling(20).mean()
df["ma50"] = df["close"].rolling(50).mean()

df["trend_strength"] = (
    (df["ma10"] - df["ma20"]) / df["close"]
)

df["trend_longer"] = (
    (df["ma20"] - df["ma50"]) / df["close"]
)

# ------------------------------------------------------------
# 3. Momentum
# ------------------------------------------------------------

df["mom3"]  = df["close"].pct_change(3)
df["mom6"]  = df["close"].pct_change(6)
df["mom12"] = df["close"].pct_change(12)
df["mom24"] = df["close"].pct_change(24)

# ------------------------------------------------------------
# 4. Volatility
# ------------------------------------------------------------

df["vol10"] = df["return_1h"].rolling(10).std()
df["vol20"] = df["return_1h"].rolling(20).std()

prev_close = df["close"].shift(1)

tr1 = df["high"] - df["low"]
tr2 = (df["high"] - prev_close).abs()
tr3 = (df["low"] - prev_close).abs()

df["true_range"] = pd.concat(
    [tr1, tr2, tr3],
    axis=1
).max(axis=1)

df["atr20"] = df["true_range"].rolling(20).mean()
df["atr_percent"] = df["atr20"] / df["close"]

# ------------------------------------------------------------
# 5. Candle behaviour
# ------------------------------------------------------------

df["candle_range"] = df["high"] - df["low"]

df["body"] = (
    df["close"] - df["open"]
).abs()

df["body_ratio"] = (
    df["body"] /
    df["candle_range"].replace(0, np.nan)
)

# ------------------------------------------------------------
# 6. Recent trading range
# ------------------------------------------------------------

df["high20"] = df["high"].rolling(20).max()
df["low20"] = df["low"].rolling(20).min()

df["range_position20"] = (
    (df["close"] - df["low20"]) /
    (df["high20"] - df["low20"]).replace(0, np.nan)
)

df["distance_high20"] = (
    (df["high20"] - df["close"]) /
    df["close"]
)

df["distance_low20"] = (
    (df["close"] - df["low20"]) /
    df["close"]
)

# ------------------------------------------------------------
# 7. Activity / volume
# ------------------------------------------------------------

df["volume20"] = df["volume"].rolling(20).mean()

df["relative_volume"] = (
    df["volume"] /
    df["volume20"].replace(0, np.nan)
)

# ------------------------------------------------------------
# 8. Multi-timeframe agreement
# ------------------------------------------------------------

df["momentum_agreement"] = (
    (df["mom3"] > 0).astype(int) +
    (df["mom6"] > 0).astype(int) +
    (df["mom12"] > 0).astype(int) +
    (df["mom24"] > 0).astype(int)
)

df["trend_agreement"] = (
    (df["close"] > df["ma10"]).astype(int) +
    (df["close"] > df["ma20"]).astype(int) +
    (df["close"] > df["ma50"]).astype(int)
)

# ------------------------------------------------------------
# 9. FUTURE OUTCOMES
# These are targets, NOT inputs.
# ------------------------------------------------------------

df["future_6h"] = (
    df["close"].shift(-6) / df["close"] - 1
)

df["future_12h"] = (
    df["close"].shift(-12) / df["close"] - 1
)

df["future_24h"] = (
    df["close"].shift(-24) / df["close"] - 1
)

df["future_abs_24h"] = (
    df["future_24h"].abs()
)

# ------------------------------------------------------------
# 10. Clean dataset
# ------------------------------------------------------------

feature_columns = [
    "return_1h",
    "return_3h",
    "return_6h",
    "return_12h",
    "return_24h",
    "trend_strength",
    "trend_longer",
    "mom3",
    "mom6",
    "mom12",
    "mom24",
    "vol10",
    "vol20",
    "atr_percent",
    "body_ratio",
    "range_position20",
    "distance_high20",
    "distance_low20",
    "relative_volume",
    "momentum_agreement",
    "trend_agreement"
]

v12 = df.dropna(
    subset=feature_columns + ["future_abs_24h"]
).copy()

print("=" * 60)
print("V12a — MARKET STATE DATASET")
print("=" * 60)

print(f"Rows available: {len(v12):,}")
print(
    f"Period: {v12.index.min()} -> {v12.index.max()}"
)

print()
print("FEATURES:")

for col in feature_columns:
    print(f"  ✓ {col}")

print()
print("FUTURE OPPORTUNITY SIZE:")

print(
    f"Average 24h movement: "
    f"{v12['future_abs_24h'].mean() * 100:.3f}%"
)

print(
    f"Median 24h movement: "
    f"{v12['future_abs_24h'].median() * 100:.3f}%"
)

print(
    f"75th percentile: "
    f"{v12['future_abs_24h'].quantile(0.75) * 100:.3f}%"
)

print(
    f"90th percentile: "
    f"{v12['future_abs_24h'].quantile(0.90) * 100:.3f}%"
)

print()
print("LATEST MARKET STATE:")

latest = v12.iloc[-1]

print(f"  Silver price:       {latest['close']:.4f}")
print(f"  Trend strength:     {latest['trend_strength'] * 100:.3f}%")
print(f"  Momentum 12h:       {latest['mom12'] * 100:.3f}%")
print(f"  Volatility 20h:     {latest['vol20'] * 100:.3f}%")
print(f"  ATR:                {latest['atr20']:.4f}")
print(f"  ATR %:              {latest['atr_percent'] * 100:.3f}%")
print(f"  Range position:     {latest['range_position20'] * 100:.1f}%")
print(f"  Relative volume:    {latest['relative_volume']:.2f}x")
print(f"  Momentum agreement: {latest['momentum_agreement']}/4")
print(f"  Trend agreement:    {latest['trend_agreement']}/3")

print()
print("=" * 60)
print("V12a COMPLETE")
print("=" * 60)

In [ ]:
# ============================================================
# V12b — OPPORTUNITY STATE ANALYSIS
#
# Goal:
# Find market states associated with unusually large
# future 24-hour moves WITHOUT using future test data.
# ============================================================

import numpy as np
import pandas as pd

data = v12.copy()

# ------------------------------------------------------------
# 1. Train / test split
# ------------------------------------------------------------

split = int(len(data) * 0.70)

train = data.iloc[:split].copy()
test = data.iloc[split:].copy()

print("=" * 65)
print("V12b — OPPORTUNITY STATE ANALYSIS")
print("=" * 65)

print(f"Total observations : {len(data):,}")
print(f"Training           : {len(train):,}")
print(f"Unseen test        : {len(test):,}")

# ------------------------------------------------------------
# 2. Establish the opportunity baseline
#
# IMPORTANT:
# The threshold is calculated ONLY from training data.
# ------------------------------------------------------------

opportunity_threshold = train["future_abs_24h"].quantile(0.75)

train["opportunity"] = (
    train["future_abs_24h"] >= opportunity_threshold
)

test["opportunity"] = (
    test["future_abs_24h"] >= opportunity_threshold
)

print()
print("TRAINING OPPORTUNITY THRESHOLD")
print(
    f"75th percentile future move: "
    f"{opportunity_threshold * 100:.3f}%"
)

print(
    f"Training opportunity rate: "
    f"{train['opportunity'].mean() * 100:.2f}%"
)

print(
    f"Test opportunity rate: "
    f"{test['opportunity'].mean() * 100:.2f}%"
)

# ------------------------------------------------------------
# 3. Create market-state categories
#
# Thresholds are calculated from TRAINING data only.
# ------------------------------------------------------------

thresholds = {}

state_features = [
    "trend_strength",
    "mom12",
    "vol20",
    "atr_percent",
    "relative_volume",
    "range_position20",
]

for feature in state_features:
    thresholds[feature] = {
        "low": train[feature].quantile(0.25),
        "high": train[feature].quantile(0.75),
    }

# ------------------------------------------------------------
# 4. Function to classify market states
# ------------------------------------------------------------

def classify_states(df, thresholds):

    x = df.copy()

    # Trend
    x["trend_state"] = np.select(
        [
            x["trend_strength"] <= thresholds["trend_strength"]["low"],
            x["trend_strength"] >= thresholds["trend_strength"]["high"],
        ],
        [
            "STRONG_DOWN",
            "STRONG_UP",
        ],
        default="NEUTRAL"
    )

    # Momentum
    x["momentum_state"] = np.select(
        [
            x["mom12"] <= thresholds["mom12"]["low"],
            x["mom12"] >= thresholds["mom12"]["high"],
        ],
        [
            "NEGATIVE",
            "POSITIVE",
        ],
        default="NEUTRAL"
    )

    # Volatility
    x["volatility_state"] = np.select(
        [
            x["vol20"] <= thresholds["vol20"]["low"],
            x["vol20"] >= thresholds["vol20"]["high"],
        ],
        [
            "LOW",
            "HIGH",
        ],
        default="NORMAL"
    )

    # ATR
    x["atr_state"] = np.select(
        [
            x["atr_percent"] <= thresholds["atr_percent"]["low"],
            x["atr_percent"] >= thresholds["atr_percent"]["high"],
        ],
        [
            "LOW",
            "HIGH",
        ],
        default="NORMAL"
    )

    # Volume
    x["activity_state"] = np.select(
        [
            x["relative_volume"] <= thresholds["relative_volume"]["low"],
            x["relative_volume"] >= thresholds["relative_volume"]["high"],
        ],
        [
            "QUIET",
            "HIGH",
        ],
        default="NORMAL"
    )

    # Position in recent range
    x["range_state"] = np.select(
        [
            x["range_position20"] <= thresholds["range_position20"]["low"],
            x["range_position20"] >= thresholds["range_position20"]["high"],
        ],
        [
            "NEAR_LOW",
            "NEAR_HIGH",
        ],
        default="MIDDLE"
    )

    return x


train = classify_states(train, thresholds)
test = classify_states(test, thresholds)

# ------------------------------------------------------------
# 5. Calculate opportunity statistics for individual states
# ------------------------------------------------------------

def state_analysis(df, feature):

    result = (
        df.groupby(feature)
        .agg(
            observations=("future_abs_24h", "size"),
            avg_move=("future_abs_24h", "mean"),
            median_move=("future_abs_24h", "median"),
            opportunity_rate=("opportunity", "mean")
        )
        .sort_values("avg_move", ascending=False)
    )

    return result


print()
print("=" * 65)
print("TRAINING STATE ANALYSIS")
print("=" * 65)

for feature in [
    "trend_state",
    "momentum_state",
    "volatility_state",
    "atr_state",
    "activity_state",
    "range_state",
]:

    print()
    print(f"--- {feature.upper()} ---")

    result = state_analysis(train, feature)

    display(
        result.style.format({
            "observations": "{:,.0f}",
            "avg_move": "{:.3%}",
            "median_move": "{:.3%}",
            "opportunity_rate": "{:.2%}"
        })
    )

# ------------------------------------------------------------
# 6. Find combinations of states
# ------------------------------------------------------------

train["market_state"] = (
    train["trend_state"] + " | " +
    train["momentum_state"] + " | " +
    train["volatility_state"] + " | " +
    train["range_state"]
)

test["market_state"] = (
    test["trend_state"] + " | " +
    test["momentum_state"] + " | " +
    test["volatility_state"] + " | " +
    test["range_state"]
)

combo_train = (
    train.groupby("market_state")
    .agg(
        observations=("future_abs_24h", "size"),
        avg_move=("future_abs_24h", "mean"),
        median_move=("future_abs_24h", "median"),
        opportunity_rate=("opportunity", "mean")
    )
)

# Require enough observations to avoid tiny lucky samples
combo_train = combo_train[
    combo_train["observations"] >= 25
].copy()

combo_train = combo_train.sort_values(
    "avg_move",
    ascending=False
)

print()
print("=" * 65)
print("TOP TRAINING MARKET STATES")
print("=" * 65)

display(
    combo_train.head(15).style.format({
        "observations": "{:,.0f}",
        "avg_move": "{:.3%}",
        "median_move": "{:.3%}",
        "opportunity_rate": "{:.2%}"
    })
)

# ------------------------------------------------------------
# 7. Select promising states from TRAINING ONLY
#
# We deliberately don't optimize heavily.
# We simply select states with:
# - >= 25 observations
# - above the training median opportunity size
# ------------------------------------------------------------

training_median = train["future_abs_24h"].median()

selected_states = combo_train[
    combo_train["avg_move"] > training_median
].index.tolist()

print()
print("=" * 65)
print("SELECTED TRAINING STATES")
print("=" * 65)

print(
    f"Training median move: "
    f"{training_median * 100:.3f}%"
)

print(
    f"States selected: {len(selected_states)}"
)

for state in selected_states:
    print("  ", state)

# ------------------------------------------------------------
# 8. Test selected states on COMPLETELY UNSEEN DATA
# ------------------------------------------------------------

test["selected_state"] = (
    test["market_state"].isin(selected_states)
)

selected_test = test[
    test["selected_state"]
].copy()

print()
print("=" * 65)
print("UNSEEN TEST RESULTS")
print("=" * 65)

print(
    f"Test observations: {len(test):,}"
)

print(
    f"Selected opportunities: "
    f"{len(selected_test):,}"
)

if len(selected_test) > 0:

    print(
        f"Average future 24h move: "
        f"{selected_test['future_abs_24h'].mean() * 100:.3f}%"
    )

    print(
        f"Median future 24h move: "
        f"{selected_test['future_abs_24h'].median() * 100:.3f}%"
    )

    print(
        f"Opportunity rate: "
        f"{selected_test['opportunity'].mean() * 100:.2f}%"
    )

    print(
        f"Baseline test average: "
        f"{test['future_abs_24h'].mean() * 100:.3f}%"
    )

    print(
        f"Baseline test opportunity rate: "
        f"{test['opportunity'].mean() * 100:.2f}%"
    )

    improvement = (
        selected_test["future_abs_24h"].mean()
        / test["future_abs_24h"].mean()
        - 1
    )

    print(
        f"Average-move improvement: "
        f"{improvement * 100:.2f}%"
    )

else:

    print("No selected states appeared in the unseen test period.")

print()
print("=" * 65)
print("V12b COMPLETE")
print("=" * 65)

In [ ]:
# ============================================================
# V12c — TOP-STATE STRESS TEST
#
# Goal:
# Test whether the strongest market states discovered in
# training remain useful on completely unseen data.
# ============================================================

import numpy as np
import pandas as pd

print("=" * 70)
print("V12c — TOP-STATE STRESS TEST")
print("=" * 70)

# ------------------------------------------------------------
# 1. Rank training states
# ------------------------------------------------------------

ranked_states = combo_train.copy()

# We want a balance between:
# - size of future movement
# - frequency of large moves
#
# Opportunity rate is important because a state producing
# large moves only occasionally may not be useful.

ranked_states["quality_score"] = (
    ranked_states["avg_move"] *
    (0.5 + ranked_states["opportunity_rate"])
)

ranked_states = ranked_states.sort_values(
    "quality_score",
    ascending=False
)

# ------------------------------------------------------------
# 2. Test several levels of selectivity
#
# These are chosen BEFORE looking at test performance.
# ------------------------------------------------------------

TOP_LEVELS = [3, 5, 10, 15]

print()
print("TOP TRAINING STATES")
print("-" * 70)

display(
    ranked_states.head(15).style.format({
        "observations": "{:,.0f}",
        "avg_move": "{:.3%}",
        "median_move": "{:.3%}",
        "opportunity_rate": "{:.2%}",
        "quality_score": "{:.5f}"
    })
)

# ------------------------------------------------------------
# 3. Baseline test performance
# ------------------------------------------------------------

baseline_avg = test["future_abs_24h"].mean()
baseline_median = test["future_abs_24h"].median()
baseline_opp = test["opportunity"].mean()

print()
print("UNSEEN TEST BASELINE")
print("-" * 70)

print(f"Observations:        {len(test):,}")
print(f"Average 24h move:    {baseline_avg * 100:.3f}%")
print(f"Median 24h move:     {baseline_median * 100:.3f}%")
print(f"Opportunity rate:    {baseline_opp * 100:.2f}%")

# ------------------------------------------------------------
# 4. Test frozen state selections
# ------------------------------------------------------------

results = []

for top_n in TOP_LEVELS:

    selected = ranked_states.head(top_n).index.tolist()

    test["top_state_signal"] = (
        test["market_state"].isin(selected)
    )

    signals = test[
        test["top_state_signal"]
    ].copy()

    if len(signals) == 0:
        results.append({
            "top_n": top_n,
            "signals": 0,
            "coverage": 0,
            "avg_move": np.nan,
            "median_move": np.nan,
            "opportunity_rate": np.nan,
            "improvement": np.nan
        })
        continue

    avg_move = signals["future_abs_24h"].mean()
    median_move = signals["future_abs_24h"].median()
    opportunity_rate = signals["opportunity"].mean()

    improvement = (
        avg_move / baseline_avg - 1
    )

    results.append({
        "top_n": top_n,
        "signals": len(signals),
        "coverage": len(signals) / len(test),
        "avg_move": avg_move,
        "median_move": median_move,
        "opportunity_rate": opportunity_rate,
        "improvement": improvement
    })

# ------------------------------------------------------------
# 5. Results
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

print()
print("=" * 70)
print("FROZEN TOP-STATE RESULTS — UNSEEN DATA")
print("=" * 70)

display(
    results_df.style.format({
        "top_n": "{:.0f}",
        "signals": "{:,.0f}",
        "coverage": "{:.2%}",
        "avg_move": "{:.3%}",
        "median_move": "{:.3%}",
        "opportunity_rate": "{:.2%}",
        "improvement": "{:+.2%}"
    })
)

# ------------------------------------------------------------
# 6. Examine the most selective version
# ------------------------------------------------------------

best_n = 3

selected_top3 = ranked_states.head(best_n).index.tolist()

test["top3_signal"] = (
    test["market_state"].isin(selected_top3)
)

top3 = test[
    test["top3_signal"]
].copy()

print()
print("=" * 70)
print(f"TOP {best_n} STATE DETAILS")
print("=" * 70)

for i, state in enumerate(selected_top3, 1):
    print(f"{i}. {state}")

print()

if len(top3) > 0:

    print(f"Signals:             {len(top3):,}")
    print(
        f"Coverage:            "
        f"{len(top3) / len(test) * 100:.2f}%"
    )

    print(
        f"Average 24h move:    "
        f"{top3['future_abs_24h'].mean() * 100:.3f}%"
    )

    print(
        f"Median 24h move:     "
        f"{top3['future_abs_24h'].median() * 100:.3f}%"
    )

    print(
        f"Opportunity rate:    "
        f"{top3['opportunity'].mean() * 100:.2f}%"
    )

    print(
        f"Baseline average:    "
        f"{baseline_avg * 100:.3f}%"
    )

    print(
        f"Improvement:         "
        f"{(top3['future_abs_24h'].mean() / baseline_avg - 1) * 100:+.2f}%"
    )

else:

    print("None of the top states appeared in the unseen period.")

# ------------------------------------------------------------
# 7. Important interpretation
# ------------------------------------------------------------

print()
print("=" * 70)
print("V12c COMPLETE")
print("=" * 70)

print("""
IMPORTANT:
The test period has NOT been used to choose the states.

If the top 3 / 5 / 10 states perform well on unseen data,
that is evidence that the market-state concept may contain
real information.

If performance collapses, we reject the idea rather than
optimising against the test period.
""")

In [ ]:
# ============================================================
# V12d — ROLLING WALK-FORWARD OPPORTUNITY VALIDATION
#
# Goal:
# Repeatedly discover the strongest 3 market states using
# ONLY the past, then test them on the next unseen period.
#
# No peeking into the future.
# No optimisation against test results.
# ============================================================

import numpy as np
import pandas as pd

print("=" * 70)
print("V12d — ROLLING WALK-FORWARD VALIDATION")
print("=" * 70)

data = v12.copy()

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

TRAIN_SIZE = 1400
TEST_SIZE = 400
STEP_SIZE = 400

TOP_N = 3
MIN_OBSERVATIONS = 25

# ------------------------------------------------------------
# Helper: create training thresholds
# ------------------------------------------------------------

def make_thresholds(train_df):

    threshold_features = [
        "trend_strength",
        "mom12",
        "vol20",
        "atr_percent",
        "relative_volume",
        "range_position20",
    ]

    thresholds = {}

    for feature in threshold_features:

        thresholds[feature] = {
            "low": train_df[feature].quantile(0.25),
            "high": train_df[feature].quantile(0.75),
        }

    return thresholds


# ------------------------------------------------------------
# Helper: classify market state
# ------------------------------------------------------------

def classify_market_states(df, thresholds):

    x = df.copy()

    x["trend_state"] = np.select(
        [
            x["trend_strength"] <= thresholds["trend_strength"]["low"],
            x["trend_strength"] >= thresholds["trend_strength"]["high"],
        ],
        [
            "STRONG_DOWN",
            "STRONG_UP",
        ],
        default="NEUTRAL"
    )

    x["momentum_state"] = np.select(
        [
            x["mom12"] <= thresholds["mom12"]["low"],
            x["mom12"] >= thresholds["mom12"]["high"],
        ],
        [
            "NEGATIVE",
            "POSITIVE",
        ],
        default="NEUTRAL"
    )

    x["volatility_state"] = np.select(
        [
            x["vol20"] <= thresholds["vol20"]["low"],
            x["vol20"] >= thresholds["vol20"]["high"],
        ],
        [
            "LOW",
            "HIGH",
        ],
        default="NORMAL"
    )

    x["range_state"] = np.select(
        [
            x["range_position20"] <= thresholds["range_position20"]["low"],
            x["range_position20"] >= thresholds["range_position20"]["high"],
        ],
        [
            "NEAR_LOW",
            "NEAR_HIGH",
        ],
        default="MIDDLE"
    )

    x["market_state"] = (
        x["trend_state"] + " | " +
        x["momentum_state"] + " | " +
        x["volatility_state"] + " | " +
        x["range_state"]
    )

    return x


# ------------------------------------------------------------
# Walk-forward loop
# ------------------------------------------------------------

results = []
all_test_signals = []

window_number = 0
train_start = 0

while True:

    train_end = train_start + TRAIN_SIZE
    test_end = train_end + TEST_SIZE

    if test_end > len(data):
        break

    window_number += 1

    train = data.iloc[
        train_start:train_end
    ].copy()

    test = data.iloc[
        train_end:test_end
    ].copy()

    # --------------------------------------------------------
    # Opportunity threshold from TRAINING ONLY
    # --------------------------------------------------------

    opportunity_threshold = (
        train["future_abs_24h"].quantile(0.75)
    )

    train["opportunity"] = (
        train["future_abs_24h"] >= opportunity_threshold
    )

    test["opportunity"] = (
        test["future_abs_24h"] >= opportunity_threshold
    )

    # --------------------------------------------------------
    # Build states using TRAINING thresholds only
    # --------------------------------------------------------

    thresholds = make_thresholds(train)

    train = classify_market_states(
        train,
        thresholds
    )

    test = classify_market_states(
        test,
        thresholds
    )

    # --------------------------------------------------------
    # Rank states using TRAINING DATA ONLY
    # --------------------------------------------------------

    ranked = (
        train.groupby("market_state")
        .agg(
            observations=("future_abs_24h", "size"),
            avg_move=("future_abs_24h", "mean"),
            median_move=("future_abs_24h", "median"),
            opportunity_rate=("opportunity", "mean")
        )
    )

    ranked = ranked[
        ranked["observations"] >= MIN_OBSERVATIONS
    ].copy()

    if len(ranked) == 0:

        train_start += STEP_SIZE
        continue

    # Same quality concept used in V12c
    ranked["quality_score"] = (
        ranked["avg_move"] *
        (0.5 + ranked["opportunity_rate"])
    )

    ranked = ranked.sort_values(
        "quality_score",
        ascending=False
    )

    selected_states = ranked.head(
        TOP_N
    ).index.tolist()

    # --------------------------------------------------------
    # Test frozen states
    # --------------------------------------------------------

    test["signal"] = (
        test["market_state"].isin(
            selected_states
        )
    )

    signals = test[
        test["signal"]
    ].copy()

    if len(signals) > 0:

        avg_move = signals[
            "future_abs_24h"
        ].mean()

        median_move = signals[
            "future_abs_24h"
        ].median()

        opportunity_rate = signals[
            "opportunity"
        ].mean()

        coverage = (
            len(signals) / len(test)
        )

        baseline_avg = (
            test["future_abs_24h"].mean()
        )

        improvement = (
            avg_move / baseline_avg - 1
        )

        result = {
            "window": window_number,
            "train_start": train.index.min(),
            "train_end": train.index.max(),
            "test_start": test.index.min(),
            "test_end": test.index.max(),
            "signals": len(signals),
            "coverage": coverage,
            "avg_move": avg_move,
            "median_move": median_move,
            "opportunity_rate": opportunity_rate,
            "baseline_avg": baseline_avg,
            "improvement": improvement,
            "states": " || ".join(
                selected_states
            )
        }

        results.append(result)

        signals["window"] = window_number
        all_test_signals.append(signals)

    train_start += STEP_SIZE


# ------------------------------------------------------------
# Results dataframe
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

print()
print("=" * 70)
print("WALK-FORWARD RESULTS")
print("=" * 70)

if len(results_df) > 0:

    display(
        results_df[
            [
                "window",
                "signals",
                "coverage",
                "avg_move",
                "median_move",
                "opportunity_rate",
                "baseline_avg",
                "improvement"
            ]
        ].style.format({
            "window": "{:.0f}",
            "signals": "{:,.0f}",
            "coverage": "{:.2%}",
            "avg_move": "{:.3%}",
            "median_move": "{:.3%}",
            "opportunity_rate": "{:.2%}",
            "baseline_avg": "{:.3%}",
            "improvement": "{:+.2%}"
        })
    )

    # --------------------------------------------------------
    # Aggregate statistics
    # --------------------------------------------------------

    profitable_windows = (
        results_df["improvement"] > 0
    ).sum()

    total_windows = len(results_df)

    total_signals = (
        results_df["signals"].sum()
    )

    weighted_avg_move = (
        np.average(
            results_df["avg_move"],
            weights=results_df["signals"]
        )
    )

    weighted_baseline = (
        np.average(
            results_df["baseline_avg"],
            weights=results_df["signals"]
        )
    )

    overall_improvement = (
        weighted_avg_move /
        weighted_baseline - 1
    )

    weighted_opportunity = (
        np.average(
            results_df["opportunity_rate"],
            weights=results_df["signals"]
        )
    )

    weighted_coverage = (
        np.average(
            results_df["coverage"],
            weights=np.ones(len(results_df))
        )
    )

    print()
    print("=" * 70)
    print("WALK-FORWARD SUMMARY")
    print("=" * 70)

    print(
        f"Windows tested:              {total_windows}"
    )

    print(
        f"Profitable windows:           "
        f"{profitable_windows}/{total_windows} "
        f"({profitable_windows / total_windows * 100:.1f}%)"
    )

    print(
        f"Total signals:                "
        f"{total_signals:,}"
    )

    print(
        f"Weighted signal avg move:     "
        f"{weighted_avg_move * 100:.3f}%"
    )

    print(
        f"Weighted baseline move:       "
        f"{weighted_baseline * 100:.3f}%"
    )

    print(
        f"Overall improvement:          "
        f"{overall_improvement * 100:+.2f}%"
    )

    print(
        f"Weighted opportunity rate:    "
        f"{weighted_opportunity * 100:.2f}%"
    )

    print(
        f"Average signal coverage:      "
        f"{weighted_coverage * 100:.2f}%"
    )

    # --------------------------------------------------------
    # Strongest / weakest windows
    # --------------------------------------------------------

    print()
    print("=" * 70)
    print("STRONGEST WINDOWS")
    print("=" * 70)

    display(
        results_df.sort_values(
            "improvement",
            ascending=False
        ).head(5)[
            [
                "window",
                "signals",
                "coverage",
                "avg_move",
                "opportunity_rate",
                "improvement",
                "states"
            ]
        ]
    )

    print()
    print("=" * 70)
    print("WEAKEST WINDOWS")
    print("=" * 70)

    display(
        results_df.sort_values(
            "improvement",
            ascending=True
        ).head(5)[
            [
                "window",
                "signals",
                "coverage",
                "avg_move",
                "opportunity_rate",
                "improvement",
                "states"
            ]
        ]
    )

else:

    print("No valid walk-forward windows were produced.")

print()
print("=" * 70)
print("V12d COMPLETE")
print("=" * 70)

print("""
IMPORTANT:
Every window selected its states using ONLY information
available before that window's test period.

The test periods were not used to select the states.
""")

In [ ]:
# ============================================================
# V13a — LARGE HISTORICAL DATA COLLECTOR
#
# Goal:
# Collect up to 20,000 completed H1 XAG/USD candles.
#
# Uses fixed time windows rather than relying on candle counts
# while moving backwards through history.
# ============================================================

import requests
import pandas as pd
import numpy as np
import time

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

TARGET_CANDLES = 20000
GRANULARITY = "H1"
PRICE = "M"

# OANDA allows a maximum of 5000 candles per request.
MAX_PER_REQUEST = 5000

# ------------------------------------------------------------
# Make sure credentials already exist
# ------------------------------------------------------------

if "OANDA_URL" not in globals():
    OANDA_URL = "https://api-fxpractice.oanda.com"

if "ACCOUNT_ID" not in globals():
    raise RuntimeError(
        "ACCOUNT_ID is missing. Run your credential cell first."
    )

if "API_TOKEN" not in globals():
    raise RuntimeError(
        "API_TOKEN is missing. Run your credential cell first."
    )

headers = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}

# ------------------------------------------------------------
# Helper function
# ------------------------------------------------------------

def fetch_candles(from_time=None, to_time=None, count=5000):

    params = {
        "granularity": GRANULARITY,
        "price": PRICE,
        "count": min(count, MAX_PER_REQUEST)
    }

    if from_time is not None:
        params["from"] = from_time

    if to_time is not None:
        params["to"] = to_time

    url = (
        f"{OANDA_URL}/v3/instruments/"
        f"XAG_USD/candles"
    )

    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=30
    )

    if response.status_code != 200:
        print("OANDA ERROR:")
        print(response.status_code)
        print(response.text[:1000])
        response.raise_for_status()

    return response.json()["candles"]


# ------------------------------------------------------------
# First request:
# Get the most recent 5000 completed candles.
# ------------------------------------------------------------

print("=" * 70)
print("V13a — LARGE HISTORICAL DATA COLLECTOR")
print("=" * 70)

print()
print("Downloading most recent candles...")

candles = fetch_candles(
    count=MAX_PER_REQUEST
)

print(
    f"First batch received: {len(candles):,}"
)

# ------------------------------------------------------------
# Convert to dataframe
# ------------------------------------------------------------

def candles_to_df(candles):

    rows = []

    for c in candles:

        if not c.get("complete", False):
            continue

        mid = c.get("mid")

        if mid is None:
            continue

        rows.append({
            "time": c["time"],
            "open": float(mid["o"]),
            "high": float(mid["h"]),
            "low": float(mid["l"]),
            "close": float(mid["c"]),
            "volume": float(c["volume"])
        })

    result = pd.DataFrame(rows)

    if len(result) == 0:
        return result

    result["time"] = pd.to_datetime(
        result["time"],
        utc=True
    )

    result = (
        result
        .drop_duplicates("time")
        .sort_values("time")
        .set_index("time")
    )

    return result


first_df = candles_to_df(candles)

print(
    f"Completed candles: {len(first_df):,}"
)

# ------------------------------------------------------------
# Find the oldest candle currently available
# ------------------------------------------------------------

if len(first_df) == 0:
    raise RuntimeError(
        "No completed candles were returned."
    )

oldest_time = first_df.index.min()

print(
    f"Newest candle: {first_df.index.max()}"
)

print(
    f"Oldest candle: {oldest_time}"
)

# ------------------------------------------------------------
# Download older fixed windows
#
# We use the oldest timestamp as the end of the next request.
# includeFirst=False prevents duplication of the boundary.
# ------------------------------------------------------------

all_batches = [first_df]

while sum(len(x) for x in all_batches) < TARGET_CANDLES:

    current_count = sum(
        len(x) for x in all_batches
    )

    remaining = TARGET_CANDLES - current_count

    request_count = min(
        MAX_PER_REQUEST,
        remaining + 10
    )

    end_time = oldest_time.isoformat()

    print()
    print(
        f"Requesting older data | "
        f"currently {current_count:,} candles"
    )

    try:

        batch = fetch_candles(
            to_time=end_time,
            count=request_count
        )

    except Exception as e:

        print("Request failed:", e)
        break

    batch_df = candles_to_df(batch)

    if len(batch_df) == 0:
        print(
            "No more completed candles returned."
        )
        break

    # Remove anything we already have
    existing_times = set(
        all_batches[0].index
    )

    # More efficient global deduplication happens below,
    # but first remove the obvious boundary.
    batch_df = batch_df[
        ~batch_df.index.isin(existing_times)
    ]

    if len(batch_df) == 0:
        print(
            "No new candles received. "
            "Stopping to avoid an infinite loop."
        )
        break

    all_batches.append(batch_df)

    new_oldest = batch_df.index.min()

    print(
        f"New candles: {len(batch_df):,}"
    )

    print(
        f"New oldest: {new_oldest}"
    )

    if new_oldest >= oldest_time:
        print(
            "Pagination did not move backwards. "
            "Stopping safely."
        )
        break

    oldest_time = new_oldest

    time.sleep(0.2)


# ------------------------------------------------------------
# Combine everything
# ------------------------------------------------------------

silver_v13 = pd.concat(
    all_batches,
    axis=0
)

silver_v13 = (
    silver_v13
    .sort_index()
    .loc[~silver_v13.index.duplicated(
        keep="first"
    )]
)

# Keep target size
if len(silver_v13) > TARGET_CANDLES:
    silver_v13 = silver_v13.iloc[
        -TARGET_CANDLES:
    ]

# ------------------------------------------------------------
# Validate
# ------------------------------------------------------------

print()
print("=" * 70)
print("V13a DATASET RESULT")
print("=" * 70)

print(
    f"Total candles:      {len(silver_v13):,}"
)

print(
    f"Start:              {silver_v13.index.min()}"
)

print(
    f"End:                {silver_v13.index.max()}"
)

print(
    f"Duplicate times:    "
    f"{silver_v13.index.duplicated().sum()}"
)

print(
    f"Missing OHLC rows:  "
    f"{silver_v13[['open','high','low','close']].isna().any(axis=1).sum()}"
)

print(
    f"Invalid prices:     "
    f"{(silver_v13[['open','high','low','close']] <= 0).any(axis=1).sum()}"
)

# ------------------------------------------------------------
# Check chronological gaps
# ------------------------------------------------------------

time_diff = (
    silver_v13.index.to_series()
    .diff()
    .dropna()
)

expected_gap = pd.Timedelta(hours=1)

gap_count = (
    time_diff > expected_gap
).sum()

print(
    f"Hourly gaps:        {gap_count:,}"
)

print()
print("Latest 5 candles:")
display(
    silver_v13.tail()
)

print()
print("=" * 70)
print("V13a COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# V13b — LARGE-HISTORY FEATURE ENGINE
#
# Builds the research dataset from the new 20,000-candle
# silver_v13 history.
# ============================================================

import numpy as np
import pandas as pd

print("=" * 70)
print("V13b — LARGE-HISTORY FEATURE ENGINE")
print("=" * 70)

# ------------------------------------------------------------
# 1. Start with V13 history
# ------------------------------------------------------------

df = silver_v13.copy()

# Make absolutely sure the data is chronological
df = df.sort_index()

# ------------------------------------------------------------
# 2. Data-quality diagnostics
# ------------------------------------------------------------

time_diff = df.index.to_series().diff()

# Actual elapsed hours between observations
df["elapsed_hours"] = (
    time_diff.dt.total_seconds() / 3600
)

# A normal consecutive H1 candle
df["consecutive_h1"] = (
    df["elapsed_hours"] == 1
)

print()
print("DATA QUALITY")
print("-" * 70)

print(f"Candles:              {len(df):,}")
print(f"Start:                {df.index.min()}")
print(f"End:                  {df.index.max()}")
print(
    f"Duplicate timestamps: "
    f"{df.index.duplicated().sum()}"
)

print(
    f"OHLC missing rows:     "
    f"{df[['open','high','low','close']].isna().any(axis=1).sum()}"
)

print(
    f"Non-consecutive gaps:  "
    f"{(~df['consecutive_h1'].fillna(True)).sum():,}"
)

# ------------------------------------------------------------
# 3. Returns
#
# These are observation-to-observation returns.
# We also calculate them only when observations are reasonably
# close together.
# ------------------------------------------------------------

df["return_1obs"] = (
    df["close"].pct_change()
)

# ------------------------------------------------------------
# 4. Moving averages
# ------------------------------------------------------------

df["ma5"] = (
    df["close"].rolling(5).mean()
)

df["ma10"] = (
    df["close"].rolling(10).mean()
)

df["ma20"] = (
    df["close"].rolling(20).mean()
)

df["ma50"] = (
    df["close"].rolling(50).mean()
)

# Trend strength
df["trend_strength"] = (
    (df["ma10"] - df["ma20"])
    / df["close"]
)

df["trend_longer"] = (
    (df["ma20"] - df["ma50"])
    / df["close"]
)

# ------------------------------------------------------------
# 5. Observation-based momentum
# ------------------------------------------------------------

df["mom3"] = (
    df["close"].pct_change(3)
)

df["mom6"] = (
    df["close"].pct_change(6)
)

df["mom12"] = (
    df["close"].pct_change(12)
)

df["mom24"] = (
    df["close"].pct_change(24)
)

# ------------------------------------------------------------
# 6. Volatility
# ------------------------------------------------------------

df["vol10"] = (
    df["return_1obs"].rolling(10).std()
)

df["vol20"] = (
    df["return_1obs"].rolling(20).std()
)

# ------------------------------------------------------------
# 7. True Range / ATR
# ------------------------------------------------------------

prev_close = df["close"].shift(1)

tr1 = (
    df["high"] - df["low"]
)

tr2 = (
    df["high"] - prev_close
).abs()

tr3 = (
    df["low"] - prev_close
).abs()

df["true_range"] = pd.concat(
    [tr1, tr2, tr3],
    axis=1
).max(axis=1)

df["atr20"] = (
    df["true_range"].rolling(20).mean()
)

df["atr_percent"] = (
    df["atr20"] / df["close"]
)

# ------------------------------------------------------------
# 8. Candle behaviour
# ------------------------------------------------------------

df["candle_range"] = (
    df["high"] - df["low"]
)

df["body"] = (
    df["close"] - df["open"]
).abs()

df["body_ratio"] = (
    df["body"]
    / df["candle_range"].replace(0, np.nan)
)

# ------------------------------------------------------------
# 9. Recent range
# ------------------------------------------------------------

df["high20"] = (
    df["high"].rolling(20).max()
)

df["low20"] = (
    df["low"].rolling(20).min()
)

range_size = (
    df["high20"] - df["low20"]
).replace(0, np.nan)

df["range_position20"] = (
    (df["close"] - df["low20"])
    / range_size
)

df["distance_high20"] = (
    (df["high20"] - df["close"])
    / df["close"]
)

df["distance_low20"] = (
    (df["close"] - df["low20"])
    / df["close"]
)

# ------------------------------------------------------------
# 10. Volume / activity
# ------------------------------------------------------------

df["volume20"] = (
    df["volume"].rolling(20).mean()
)

df["relative_volume"] = (
    df["volume"]
    / df["volume20"].replace(0, np.nan)
)

# ------------------------------------------------------------
# 11. Multi-timeframe agreement
# ------------------------------------------------------------

df["momentum_agreement"] = (
    (df["mom3"] > 0).astype(int) +
    (df["mom6"] > 0).astype(int) +
    (df["mom12"] > 0).astype(int) +
    (df["mom24"] > 0).astype(int)
)

df["trend_agreement"] = (
    (df["close"] > df["ma10"]).astype(int) +
    (df["close"] > df["ma20"]).astype(int) +
    (df["close"] > df["ma50"]).astype(int)
)

# ------------------------------------------------------------
# 12. Future returns
#
# IMPORTANT:
# These are targets only.
# They must NEVER be used as model inputs.
# ------------------------------------------------------------

df["future_6obs"] = (
    df["close"].shift(-6)
    / df["close"] - 1
)

df["future_12obs"] = (
    df["close"].shift(-12)
    / df["close"] - 1
)

df["future_24obs"] = (
    df["close"].shift(-24)
    / df["close"] - 1
)

df["future_abs_24obs"] = (
    df["future_24obs"].abs()
)

# ------------------------------------------------------------
# 13. Valid feature list
# ------------------------------------------------------------

feature_columns = [
    "return_1obs",
    "mom3",
    "mom6",
    "mom12",
    "mom24",
    "trend_strength",
    "trend_longer",
    "vol10",
    "vol20",
    "atr_percent",
    "body_ratio",
    "range_position20",
    "distance_high20",
    "distance_low20",
    "relative_volume",
    "momentum_agreement",
    "trend_agreement"
]

# ------------------------------------------------------------
# 14. Remove rows without sufficient history
# ------------------------------------------------------------

v13 = df.dropna(
    subset=feature_columns
    + [
        "future_6obs",
        "future_12obs",
        "future_24obs",
        "future_abs_24obs"
    ]
).copy()

# ------------------------------------------------------------
# 15. Remove observations where the future window crosses
# a very large data gap.
#
# We don't want a "24 observation" return spanning a weekend
# or another long market closure.
# ------------------------------------------------------------

future_gap_24 = (
    df["elapsed_hours"]
    .rolling(24)
    .sum()
)

# This measures the elapsed time represented by the previous
# 24 observation intervals.

v13["future_elapsed_hours"] = (
    v13.index.to_series()
    .shift(-24)
)

# Instead of using that complicated intermediate directly,
# calculate future timestamp difference cleanly.

future_times = (
    v13.index.to_series()
    .shift(-24)
)

v13["future_hours"] = (
    future_times.values
    - v13.index.values
) / np.timedelta64(1, "h")

# Keep 24-observation outcomes only when they represent
# a reasonably normal market interval.
#
# 72 hours allows for weekends/market closures while still
# rejecting extreme timestamp corruption.

v13["valid_future_window"] = (
    v13["future_hours"] <= 72
)

v13 = v13[
    v13["valid_future_window"]
].copy()

# ------------------------------------------------------------
# 16. Final diagnostics
# ------------------------------------------------------------

print()
print("=" * 70)
print("V13b FEATURE DATASET")
print("=" * 70)

print(
    f"Raw candles:          {len(df):,}"
)

print(
    f"Usable feature rows:  {len(v13):,}"
)

print(
    f"Feature loss:         "
    f"{(1 - len(v13) / len(df)) * 100:.2f}%"
)

print(
    f"Period:               "
    f"{v13.index.min()} -> {v13.index.max()}"
)

print()
print("FUTURE 24-OBSERVATION MOVEMENT")
print("-" * 70)

print(
    f"Mean:                 "
    f"{v13['future_abs_24obs'].mean() * 100:.3f}%"
)

print(
    f"Median:               "
    f"{v13['future_abs_24obs'].median() * 100:.3f}%"
)

print(
    f"75th percentile:      "
    f"{v13['future_abs_24obs'].quantile(.75) * 100:.3f}%"
)

print(
    f"90th percentile:      "
    f"{v13['future_abs_24obs'].quantile(.90) * 100:.3f}%"
)

print()
print("LATEST MARKET STATE")
print("-" * 70)

latest = v13.iloc[-1]

print(
    f"Silver price:         {latest['close']:.4f}"
)

print(
    f"Trend strength:       "
    f"{latest['trend_strength'] * 100:.3f}%"
)

print(
    f"12-observation mom:   "
    f"{latest['mom12'] * 100:.3f}%"
)

print(
    f"Volatility:           "
    f"{latest['vol20'] * 100:.3f}%"
)

print(
    f"ATR:                  "
    f"{latest['atr20']:.4f}"
)

print(
    f"ATR %:                "
    f"{latest['atr_percent'] * 100:.3f}%"
)

print(
    f"Range position:       "
    f"{latest['range_position20'] * 100:.1f}%"
)

print(
    f"Relative volume:      "
    f"{latest['relative_volume']:.2f}x"
)

print(
    f"Momentum agreement:   "
    f"{latest['momentum_agreement']}/4"
)

print(
    f"Trend agreement:      "
    f"{latest['trend_agreement']}/3"
)

print()
print("=" * 70)
print("V13b COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# V13c — FEATURE PREDICTIVENESS ANALYSIS
#
# Goal:
# Determine which market features are associated with larger
# future movements, WITHOUT building a trading strategy.
#
# IMPORTANT:
# This is analysis only. No feature is being selected for
# trading yet.
# ============================================================

import numpy as np
import pandas as pd

print("=" * 70)
print("V13c — FEATURE PREDICTIVENESS ANALYSIS")
print("=" * 70)

data = v13.copy()

# ------------------------------------------------------------
# 1. Chronological train / test split
# ------------------------------------------------------------

split = int(len(data) * 0.70)

train = data.iloc[:split].copy()
test = data.iloc[split:].copy()

print()
print("DATA SPLIT")
print("-" * 70)

print(f"Total observations: {len(data):,}")
print(f"Training:          {len(train):,}")
print(f"Unseen test:       {len(test):,}")

# ------------------------------------------------------------
# 2. Features to investigate
# ------------------------------------------------------------

features = [
    "return_1obs",
    "mom3",
    "mom6",
    "mom12",
    "mom24",
    "trend_strength",
    "trend_longer",
    "vol10",
    "vol20",
    "atr_percent",
    "body_ratio",
    "range_position20",
    "distance_high20",
    "distance_low20",
    "relative_volume",
    "momentum_agreement",
    "trend_agreement"
]

# ------------------------------------------------------------
# 3. Create quartile boundaries using TRAINING DATA ONLY
# ------------------------------------------------------------

quantile_edges = {}

for feature in features:

    edges = train[feature].quantile(
        [0.0, 0.25, 0.50, 0.75, 1.0]
    ).values

    # Ensure unique boundaries
    edges = np.unique(edges)

    quantile_edges[feature] = edges


# ------------------------------------------------------------
# 4. Analyse each feature
#
# For each feature we compare:
# Q1 = lowest 25%
# Q2 = 25-50%
# Q3 = 50-75%
# Q4 = highest 25%
#
# The bins are created from TRAINING data only.
# ------------------------------------------------------------

def analyse_feature(df, feature, edges):

    x = df[[feature, "future_abs_24obs"]].copy()

    # pd.cut requires enough unique boundaries
    if len(edges) < 5:
        return pd.DataFrame()

    x["quartile"] = pd.cut(
        x[feature],
        bins=edges,
        labels=[
            "Q1_LOW",
            "Q2",
            "Q3",
            "Q4_HIGH"
        ],
        include_lowest=True,
        duplicates="drop"
    )

    result = (
        x.groupby(
            "quartile",
            observed=False
        )
        .agg(
            observations=("future_abs_24obs", "size"),
            avg_future_move=("future_abs_24obs", "mean"),
            median_future_move=("future_abs_24obs", "median"),
            p90_future_move=(
                "future_abs_24obs",
                lambda x: x.quantile(0.90)
            )
        )
    )

    return result


# ------------------------------------------------------------
# 5. TRAINING analysis
# ------------------------------------------------------------

print()
print("=" * 70)
print("TRAINING FEATURE ANALYSIS")
print("=" * 70)

training_summary = []

for feature in features:

    result = analyse_feature(
        train,
        feature,
        quantile_edges[feature]
    )

    if result.empty:
        continue

    q1 = result.iloc[0]["avg_future_move"]
    q4 = result.iloc[-1]["avg_future_move"]

    spread = q4 - q1

    training_summary.append({
        "feature": feature,
        "Q1_avg": q1,
        "Q4_avg": q4,
        "Q4_minus_Q1": spread
    })

training_summary = pd.DataFrame(
    training_summary
)

training_summary = training_summary.sort_values(
    "Q4_minus_Q1",
    ascending=False
)

display(
    training_summary.style.format({
        "Q1_avg": "{:.3%}",
        "Q4_avg": "{:.3%}",
        "Q4_minus_Q1": "{:+.3%}"
    })
)

# ------------------------------------------------------------
# 6. Test the same frozen quartile boundaries
# ------------------------------------------------------------

print()
print("=" * 70)
print("UNSEEN TEST FEATURE ANALYSIS")
print("=" * 70)

test_summary = []

for feature in features:

    result = analyse_feature(
        test,
        feature,
        quantile_edges[feature]
    )

    if result.empty:
        continue

    q1 = result.iloc[0]["avg_future_move"]
    q4 = result.iloc[-1]["avg_future_move"]

    spread = q4 - q1

    test_summary.append({
        "feature": feature,
        "Q1_avg": q1,
        "Q4_avg": q4,
        "Q4_minus_Q1": spread
    })

test_summary = pd.DataFrame(
    test_summary
)

test_summary = test_summary.sort_values(
    "Q4_minus_Q1",
    ascending=False
)

display(
    test_summary.style.format({
        "Q1_avg": "{:.3%}",
        "Q4_avg": "{:.3%}",
        "Q4_minus_Q1": "{:+.3%}"
    })
)

# ------------------------------------------------------------
# 7. Compare TRAIN vs TEST
#
# This is the important part.
#
# A feature that looks strong in training but reverses or
# disappears in test is probably unstable.
# ------------------------------------------------------------

comparison = (
    training_summary[
        [
            "feature",
            "Q4_minus_Q1"
        ]
    ]
    .rename(
        columns={
            "Q4_minus_Q1":
            "train_spread"
        }
    )
    .merge(
        test_summary[
            [
                "feature",
                "Q4_minus_Q1"
            ]
        ].rename(
            columns={
                "Q4_minus_Q1":
                "test_spread"
            }
        ),
        on="feature",
        how="inner"
    )
)

comparison["same_direction"] = (
    np.sign(
        comparison["train_spread"]
    )
    ==
    np.sign(
        comparison["test_spread"]
    )
)

comparison["test_strength_vs_train"] = (
    comparison["test_spread"].abs()
    /
    comparison["train_spread"].abs()
    .replace(0, np.nan)
)

comparison = comparison.sort_values(
    "test_spread",
    ascending=False
)

print()
print("=" * 70)
print("TRAIN vs UNSEEN TEST")
print("=" * 70)

display(
    comparison.style.format({
        "train_spread": "{:+.3%}",
        "test_spread": "{:+.3%}",
        "test_strength_vs_train": "{:.2f}x"
    })
)

# ------------------------------------------------------------
# 8. Count stable features
# ------------------------------------------------------------

stable = comparison[
    comparison["same_direction"]
].copy()

print()
print("=" * 70)
print("STABILITY SUMMARY")
print("=" * 70)

print(
    f"Features tested:          {len(comparison)}"
)

print(
    f"Same direction in test:   {len(stable)}"
)

print(
    f"Stability rate:           "
    f"{len(stable) / len(comparison) * 100:.1f}%"
)

# ------------------------------------------------------------
# 9. Strongest candidates that survived
# ------------------------------------------------------------

survivors = comparison[
    (comparison["same_direction"]) &
    (comparison["test_spread"] > 0)
].copy()

survivors = survivors.sort_values(
    "test_spread",
    ascending=False
)

print()
print("=" * 70)
print("FEATURES THAT SURVIVED UNSEEN TEST")
print("=" * 70)

if len(survivors) > 0:

    display(
        survivors.style.format({
            "train_spread": "{:+.3%}",
            "test_spread": "{:+.3%}",
            "test_strength_vs_train": "{:.2f}x"
        })
    )

else:

    print(
        "No individual features produced a "
        "positive Q4-vs-Q1 relationship in "
        "the unseen period."
    )

# ------------------------------------------------------------
# 10. Final reminder
# ------------------------------------------------------------

print()
print("=" * 70)
print("V13c COMPLETE")
print("=" * 70)

print("""
This test does NOT create a trading strategy.

We are looking for features whose relationship with
future movement survives from training into unseen data.

A feature that looks excellent in training but fails
in the unseen test will NOT be used simply because it
looked good historically.
""")

In [ ]:

# V13d — Rebuild V13 features cleanly
# Then prepare the Opportunity Score dataset

df = silver_v13.copy()
df = df.sort_index()

print("Raw candles:", len(df))
print("Date range:", df.index.min(), "->", df.index.max())

# ---------------------------------------------------------
# BASIC FEATURES
# ---------------------------------------------------------

df["return_1obs"] = df["close"].pct_change()

df["MA5"] = df["close"].rolling(5).mean()
df["MA10"] = df["close"].rolling(10).mean()
df["MA20"] = df["close"].rolling(20).mean()
df["MA50"] = df["close"].rolling(50).mean()

df["trend_strength"] = (
    (df["MA10"] - df["MA20"]) / df["close"]
)

df["trend_longer"] = (
    (df["MA20"] - df["MA50"]) / df["close"]
)

# Momentum
df["mom3"] = df["close"].pct_change(3)
df["mom6"] = df["close"].pct_change(6)
df["mom12"] = df["close"].pct_change(12)
df["mom24"] = df["close"].pct_change(24)

# Volatility
df["vol10"] = df["return_1obs"].rolling(10).std()
df["vol20"] = df["return_1obs"].rolling(20).std()

# ---------------------------------------------------------
# TRUE RANGE / ATR
# ---------------------------------------------------------

previous_close = df["close"].shift(1)

tr1 = df["high"] - df["low"]
tr2 = (df["high"] - previous_close).abs()
tr3 = (df["low"] - previous_close).abs()

df["true_range"] = pd.concat(
    [tr1, tr2, tr3],
    axis=1
).max(axis=1)

df["ATR20"] = df["true_range"].rolling(20).mean()

df["atr_percent"] = (
    df["ATR20"] / df["close"]
)

# ---------------------------------------------------------
# CANDLE STRUCTURE
# ---------------------------------------------------------

df["candle_range"] = (
    df["high"] - df["low"]
)

df["candle_body"] = (
    df["close"] - df["open"]
).abs()

df["body_ratio"] = (
    df["candle_body"] /
    df["candle_range"].replace(0, np.nan)
)

# ---------------------------------------------------------
# 20-CANDLE PRICE LOCATION
# ---------------------------------------------------------

df["high20"] = df["high"].rolling(20).max()
df["low20"] = df["low"].rolling(20).min()

df["range_position20"] = (
    (df["close"] - df["low20"]) /
    (df["high20"] - df["low20"]).replace(0, np.nan)
)

df["distance_high20"] = (
    (df["high20"] - df["close"]) /
    df["close"]
)

df["distance_low20"] = (
    (df["close"] - df["low20"]) /
    df["close"]
)

# ---------------------------------------------------------
# VOLUME
# ---------------------------------------------------------

df["volume20"] = df["volume"].rolling(20).mean()

df["relative_volume"] = (
    df["volume"] /
    df["volume20"].replace(0, np.nan)
)

# ---------------------------------------------------------
# AGREEMENT FEATURES
# ---------------------------------------------------------

df["momentum_agreement"] = (
    (df["mom3"] > 0).astype(int) +
    (df["mom6"] > 0).astype(int) +
    (df["mom12"] > 0).astype(int) +
    (df["mom24"] > 0).astype(int)
)

df["trend_agreement"] = (
    (df["close"] > df["MA10"]).astype(int) +
    (df["close"] > df["MA20"]).astype(int) +
    (df["close"] > df["MA50"]).astype(int)
)

# ---------------------------------------------------------
# FUTURE MOVEMENT
# ---------------------------------------------------------

df["future_6obs"] = (
    df["close"].shift(-6) / df["close"] - 1
)

df["future_12obs"] = (
    df["close"].shift(-12) / df["close"] - 1
)

df["future_24obs"] = (
    df["close"].shift(-24) / df["close"] - 1
)

df["future_abs_24obs"] = (
    df["future_24obs"].abs()
)

# ---------------------------------------------------------
# CLEAN DATA
# ---------------------------------------------------------

features = [
    "vol20",
    "atr_percent",
    "distance_high20",
    "relative_volume"
]

target = "future_abs_24obs"

data = df[features + [target]].dropna().copy()

print("\n==============================")
print("V13d FEATURE DATASET")
print("==============================")

print("Usable rows:", len(data))
print("Date range:", data.index.min(), "->", data.index.max())

print("\nFuture 24-observation movement:")
print(
    "Mean:",
    round(data[target].mean() * 100, 3),
    "%"
)

print(
    "Median:",
    round(data[target].median() * 100, 3),
    "%"
)

print(
    "75th percentile:",
    round(data[target].quantile(.75) * 100, 3),
    "%"
)

print(
    "90th percentile:",
    round(data[target].quantile(.90) * 100, 3),
    "%"
)

print("\nFeatures ready:")
for feature in features:
    print("✓", feature)

In [ ]:
# V13d — Opportunity Score
# Train on first 70%, then test completely unseen final 30%

# ---------------------------------------------------------
# 1. Chronological train/test split
# ---------------------------------------------------------

split = int(len(data) * 0.70)

train = data.iloc[:split].copy()
test = data.iloc[split:].copy()

print("==============================")
print("TRAIN / TEST SPLIT")
print("==============================")

print(
    "Train:",
    len(train),
    train.index.min(),
    "->",
    train.index.max()
)

print(
    "Test:",
    len(test),
    test.index.min(),
    "->",
    test.index.max()
)

# ---------------------------------------------------------
# 2. Convert each feature into a TRAINING percentile
# ---------------------------------------------------------
# IMPORTANT:
# Test data is scored against the TRAINING distribution.
# Nothing from the test period influences the score.

features = [
    "vol20",
    "atr_percent",
    "distance_high20",
    "relative_volume"
]

def percentile_score(train_series, values):

    sorted_values = np.sort(
        train_series.dropna().values
    )

    return (
        np.searchsorted(
            sorted_values,
            values,
            side="right"
        )
        / len(sorted_values)
        * 100
    )

# ---------------------------------------------------------
# Training scores
# ---------------------------------------------------------

for feature in features:

    train[feature + "_score"] = percentile_score(
        train[feature],
        train[feature].values
    )

# ---------------------------------------------------------
# Unseen test scores
# ---------------------------------------------------------

for feature in features:

    test[feature + "_score"] = percentile_score(
        train[feature],
        test[feature].values
    )

score_columns = [
    feature + "_score"
    for feature in features
]

# Equal weighting
train["opportunity_score"] = (
    train[score_columns].mean(axis=1)
)

test["opportunity_score"] = (
    test[score_columns].mean(axis=1)
)

# ---------------------------------------------------------
# 3. Define what counts as a BIG future move
# ---------------------------------------------------------
# This threshold is also learned from TRAIN only.

big_move_threshold = train["future_abs_24obs"].quantile(0.75)

print("\nTraining big-move threshold:")
print(
    round(big_move_threshold * 100, 3),
    "%"
)

# ---------------------------------------------------------
# 4. Analyse training score buckets
# ---------------------------------------------------------

train["score_bucket"] = pd.qcut(
    train["opportunity_score"],
    10,
    labels=False,
    duplicates="drop"
) + 1

train_bucket = train.groupby(
    "score_bucket"
)[
    "future_abs_24obs"
].agg(
    ["count", "mean", "median"]
)

train_bucket["big_move_rate"] = (
    train.groupby("score_bucket")[
        "future_abs_24obs"
    ].apply(
        lambda x: (
            x >= big_move_threshold
        ).mean()
    ).values
)

print("\n==============================")
print("TRAINING SCORE BUCKETS")
print("==============================")

print(
    train_bucket.to_string(
        formatters={
            "mean": lambda x: f"{x*100:.3f}%",
            "median": lambda x: f"{x*100:.3f}%",
            "big_move_rate": lambda x: f"{x*100:.2f}%"
        }
    )
)

# ---------------------------------------------------------
# 5. Freeze the top-10% threshold
# ---------------------------------------------------------

score_threshold = train[
    "opportunity_score"
].quantile(0.90)

print("\nFrozen high-opportunity score threshold:")
print(round(score_threshold, 2), "/ 100")

# ---------------------------------------------------------
# 6. Apply frozen threshold to UNSEEN TEST
# ---------------------------------------------------------

test["high_opportunity"] = (
    test["opportunity_score"]
    >= score_threshold
)

selected = test[
    test["high_opportunity"]
].copy()

# ---------------------------------------------------------
# 7. Compare against the entire unseen test set
# ---------------------------------------------------------

baseline_mean = (
    test["future_abs_24obs"].mean()
)

selected_mean = (
    selected["future_abs_24obs"].mean()
)

baseline_median = (
    test["future_abs_24obs"].median()
)

selected_median = (
    selected["future_abs_24obs"].median()
)

baseline_big_move_rate = (
    test["future_abs_24obs"]
    >= big_move_threshold
).mean()

selected_big_move_rate = (
    selected["future_abs_24obs"]
    >= big_move_threshold
).mean()

print("\n==============================")
print("UNSEEN TEST RESULT")
print("==============================")

print("Test observations:", len(test))

print(
    "High-opportunity observations:",
    len(selected)
)

print(
    "Coverage:",
    round(
        len(selected) / len(test) * 100,
        2
    ),
    "%"
)

print("\n--- Baseline ---")

print(
    "Mean future move:",
    round(
        baseline_mean * 100,
        3
    ),
    "%"
)

print(
    "Median future move:",
    round(
        baseline_median * 100,
        3
    ),
    "%"
)

print(
    "Big-move rate:",
    round(
        baseline_big_move_rate * 100,
        2
    ),
    "%"
)

print("\n--- HIGH OPPORTUNITY ---")

print(
    "Mean future move:",
    round(
        selected_mean * 100,
        3
    ),
    "%"
)

print(
    "Median future move:",
    round(
        selected_median * 100,
        3
    ),
    "%"
)

print(
    "Big-move rate:",
    round(
        selected_big_move_rate * 100,
        2
    ),
    "%"
)

print("\n--- IMPROVEMENT ---")

print(
    "Mean improvement:",
    round(
        (selected_mean - baseline_mean) * 100,
        3
    ),
    "percentage points"
)

print(
    "Big-move improvement:",
    round(
        (selected_big_move_rate -
         baseline_big_move_rate) * 100,
        2
    ),
    "percentage points"
)

In [ ]:
# V13e — Strict Opportunity Score Stress Test
#
# Tests the frozen Opportunity Score at:
# Top 5%
# Top 10%
# Top 20%
# Top 30%
#
# IMPORTANT:
# The score itself is still created from TRAINING data only.
# We then rank the completely unseen TEST period.

# ---------------------------------------------------------
# 1. Rank the UNSEEN test observations
# ---------------------------------------------------------

test = test.copy()

test["score_rank"] = (
    test["opportunity_score"]
    .rank(
        method="first",
        ascending=False
    )
)

test["score_percentile"] = (
    test["score_rank"] / len(test)
)

# ---------------------------------------------------------
# 2. Baseline
# ---------------------------------------------------------

baseline_mean = test["future_abs_24obs"].mean()
baseline_median = test["future_abs_24obs"].median()

big_move_threshold = train[
    "future_abs_24obs"
].quantile(0.75)

baseline_big_move_rate = (
    test["future_abs_24obs"]
    >= big_move_threshold
).mean()

print("==============================")
print("V13e — STRICT UNSEEN TEST")
print("==============================")

print("Test observations:", len(test))

print(
    "Baseline mean move:",
    round(baseline_mean * 100, 3),
    "%"
)

print(
    "Baseline median move:",
    round(baseline_median * 100, 3),
    "%"
)

print(
    "Baseline big-move rate:",
    round(baseline_big_move_rate * 100, 2),
    "%"
)

# ---------------------------------------------------------
# 3. Test different opportunity concentrations
# ---------------------------------------------------------

results = []

for top_pct in [0.05, 0.10, 0.20, 0.30]:

    cutoff = int(len(test) * top_pct)

    selected = test.iloc[:0].copy()

    selected = test.sort_values(
        "opportunity_score",
        ascending=False
    ).head(cutoff)

    mean_move = (
        selected["future_abs_24obs"].mean()
    )

    median_move = (
        selected["future_abs_24obs"].median()
    )

    big_move_rate = (
        selected["future_abs_24obs"]
        >= big_move_threshold
    ).mean()

    mean_improvement = (
        mean_move - baseline_mean
    )

    big_move_improvement = (
        big_move_rate -
        baseline_big_move_rate
    )

    results.append({
        "top_percent": top_pct * 100,
        "observations": len(selected),
        "mean_move": mean_move,
        "median_move": median_move,
        "big_move_rate": big_move_rate,
        "mean_improvement": mean_improvement,
        "big_move_improvement": big_move_improvement
    })

results_df = pd.DataFrame(results)

# ---------------------------------------------------------
# 4. Display results
# ---------------------------------------------------------

print("\n==============================")
print("OPPORTUNITY CONCENTRATION")
print("==============================")

display_df = results_df.copy()

display_df["mean_move"] = (
    display_df["mean_move"] * 100
)

display_df["median_move"] = (
    display_df["median_move"] * 100
)

display_df["big_move_rate"] = (
    display_df["big_move_rate"] * 100
)

display_df["mean_improvement"] = (
    display_df["mean_improvement"] * 100
)

display_df["big_move_improvement"] = (
    display_df["big_move_improvement"] * 100
)

display_df.columns = [
    "Top %",
    "Observations",
    "Mean Move %",
    "Median Move %",
    "Big Move %",
    "Mean Improvement pp",
    "Big Move Improvement pp"
]

print(
    display_df.to_string(
        index=False,
        formatters={
            "Mean Move %": lambda x: f"{x:.3f}",
            "Median Move %": lambda x: f"{x:.3f}",
            "Big Move %": lambda x: f"{x:.2f}",
            "Mean Improvement pp": lambda x: f"{x:.3f}",
            "Big Move Improvement pp": lambda x: f"{x:.2f}"
        }
    )
)

# ---------------------------------------------------------
# 5. Find the best concentration level
# ---------------------------------------------------------

best = results_df.loc[
    results_df["mean_move"].idxmax()
]

print("\n==============================")
print("BEST CONCENTRATION")
print("==============================")

print(
    "Top:",
    round(best["top_percent"], 1),
    "%"
)

print(
    "Mean future move:",
    round(best["mean_move"] * 100, 3),
    "%"
)

print(
    "Median future move:",
    round(best["median_move"] * 100, 3),
    "%"
)

print(
    "Big-move rate:",
    round(best["big_move_rate"] * 100, 2),
    "%"
)

print(
    "Mean improvement:",
    round(
        best["mean_improvement"] * 100,
        3
    ),
    "percentage points"
)

In [ ]:
# =========================================================
# V13f — WALK-FORWARD OPPORTUNITY DETECTOR
# =========================================================
#
# Each window:
#   1. Train only on historical data
#   2. Build the opportunity score
#   3. Freeze the scoring system
#   4. Test on the next unseen block
#
# We test the TOP 10% of scores in each unseen block.
# =========================================================

features = [
    "vol20",
    "atr_percent",
    "distance_high20",
    "relative_volume"
]

target = "future_abs_24obs"

# ---------------------------------------------------------
# Settings
# ---------------------------------------------------------

TRAIN_SIZE = 5000
TEST_SIZE = 1000
STEP_SIZE = 1000
TOP_PCT = 0.10

print("==============================")
print("V13f SETTINGS")
print("==============================")

print("Training observations:", TRAIN_SIZE)
print("Test observations:", TEST_SIZE)
print("Step size:", STEP_SIZE)
print("Target selection:", f"Top {TOP_PCT*100:.0f}%")

# ---------------------------------------------------------
# Percentile scoring function
# ---------------------------------------------------------

def percentile_score(train_series, values):

    sorted_values = np.sort(
        train_series.dropna().values
    )

    return (
        np.searchsorted(
            sorted_values,
            values,
            side="right"
        )
        / len(sorted_values)
        * 100
    )

# ---------------------------------------------------------
# Walk-forward loop
# ---------------------------------------------------------

walk_results = []

window_number = 0

start = 0

while start + TRAIN_SIZE + TEST_SIZE <= len(data):

    window_number += 1

    train_wf = data.iloc[
        start:start + TRAIN_SIZE
    ].copy()

    test_wf = data.iloc[
        start + TRAIN_SIZE:
        start + TRAIN_SIZE + TEST_SIZE
    ].copy()

    # -----------------------------------------------------
    # Create frozen scores from TRAIN only
    # -----------------------------------------------------

    for feature in features:

        train_wf[
            feature + "_score"
        ] = percentile_score(
            train_wf[feature],
            train_wf[feature].values
        )

        test_wf[
            feature + "_score"
        ] = percentile_score(
            train_wf[feature],
            test_wf[feature].values
        )

    score_columns = [
        feature + "_score"
        for feature in features
    ]

    train_wf["opportunity_score"] = (
        train_wf[score_columns].mean(axis=1)
    )

    test_wf["opportunity_score"] = (
        test_wf[score_columns].mean(axis=1)
    )

    # -----------------------------------------------------
    # Select EXACT top 10% of unseen observations
    # -----------------------------------------------------

    n_select = int(
        len(test_wf) * TOP_PCT
    )

    selected = (
        test_wf
        .sort_values(
            "opportunity_score",
            ascending=False
        )
        .head(n_select)
    )

    # -----------------------------------------------------
    # Baseline
    # -----------------------------------------------------

    baseline_mean = (
        test_wf[target].mean()
    )

    baseline_median = (
        test_wf[target].median()
    )

    # -----------------------------------------------------
    # Selected results
    # -----------------------------------------------------

    selected_mean = (
        selected[target].mean()
    )

    selected_median = (
        selected[target].median()
    )

    # Big-move threshold learned ONLY from training
    big_move_threshold = (
        train_wf[target].quantile(0.75)
    )

    baseline_big_move = (
        test_wf[target] >=
        big_move_threshold
    ).mean()

    selected_big_move = (
        selected[target] >=
        big_move_threshold
    ).mean()

    # -----------------------------------------------------
    # Store results
    # -----------------------------------------------------

    walk_results.append({

        "window": window_number,

        "train_start":
            train_wf.index.min(),

        "train_end":
            train_wf.index.max(),

        "test_start":
            test_wf.index.min(),

        "test_end":
            test_wf.index.max(),

        "signals":
            len(selected),

        "baseline_mean":
            baseline_mean,

        "selected_mean":
            selected_mean,

        "mean_improvement":
            selected_mean -
            baseline_mean,

        "baseline_median":
            baseline_median,

        "selected_median":
            selected_median,

        "baseline_big_move":
            baseline_big_move,

        "selected_big_move":
            selected_big_move,

        "big_move_improvement":
            selected_big_move -
            baseline_big_move
    })

    start += STEP_SIZE


# ---------------------------------------------------------
# Convert to dataframe
# ---------------------------------------------------------

walk_df = pd.DataFrame(
    walk_results
)

print("\n==============================")
print("WALK-FORWARD RESULTS")
print("==============================")

display_df = walk_df.copy()

display_df["baseline_mean"] *= 100
display_df["selected_mean"] *= 100
display_df["mean_improvement"] *= 100

display_df["baseline_median"] *= 100
display_df["selected_median"] *= 100

display_df["baseline_big_move"] *= 100
display_df["selected_big_move"] *= 100
display_df["big_move_improvement"] *= 100

print(
    display_df[
        [
            "window",
            "test_start",
            "test_end",
            "signals",
            "baseline_mean",
            "selected_mean",
            "mean_improvement",
            "baseline_big_move",
            "selected_big_move",
            "big_move_improvement"
        ]
    ].to_string(
        index=False,
        formatters={
            "baseline_mean":
                lambda x: f"{x:.3f}%",
            "selected_mean":
                lambda x: f"{x:.3f}%",
            "mean_improvement":
                lambda x: f"{x:.3f} pp",
            "baseline_big_move":
                lambda x: f"{x:.2f}%",
            "selected_big_move":
                lambda x: f"{x:.2f}%",
            "big_move_improvement":
                lambda x: f"{x:.2f} pp"
        }
    )
)

# ---------------------------------------------------------
# Overall results
# ---------------------------------------------------------

total_signals = walk_df["signals"].sum()

weighted_selected_mean = (
    sum(
        walk_df["selected_mean"] *
        walk_df["signals"]
    )
    / total_signals
)

weighted_baseline_mean = (
    sum(
        walk_df["baseline_mean"] *
        walk_df["signals"]
    )
    / total_signals
)

weighted_selected_big = (
    sum(
        walk_df["selected_big_move"] *
        walk_df["signals"]
    )
    / total_signals
)

weighted_baseline_big = (
    sum(
        walk_df["baseline_big_move"] *
        walk_df["signals"]
    )
    / total_signals
)

profitable_windows = (
    walk_df["mean_improvement"] > 0
).sum()

print("\n==============================")
print("OVERALL WALK-FORWARD RESULT")
print("==============================")

print(
    "Windows tested:",
    len(walk_df)
)

print(
    "Total selected observations:",
    total_signals
)

print(
    "Weighted baseline mean:",
    round(
        weighted_baseline_mean * 100,
        3
    ),
    "%"
)

print(
    "Weighted selected mean:",
    round(
        weighted_selected_mean * 100,
        3
    ),
    "%"
)

print(
    "Weighted mean improvement:",
    round(
        (weighted_selected_mean -
         weighted_baseline_mean) * 100,
        3
    ),
    "percentage points"
)

print(
    "\nWeighted baseline big-move rate:",
    round(
        weighted_baseline_big * 100,
        2
    ),
    "%"
)

print(
    "Weighted selected big-move rate:",
    round(
        weighted_selected_big * 100,
        2
    ),
    "%"
)

print(
    "Weighted big-move improvement:",
    round(
        (weighted_selected_big -
         weighted_baseline_big) * 100,
        2
    ),
    "percentage points"
)

print(
    "\nProfitable windows:",
    profitable_windows,
    "/",
    len(walk_df)
)

print(
    "Window success rate:",
    round(
        profitable_windows /
        len(walk_df) *
        100,
        1
    ),
    "%"
)

In [ ]:
# =========================================================
# V13g — VOLATILITY REGIME ANALYSIS
# =========================================================

regime_data = data.copy()

regime_results = []

for row in walk_results:

    test_start = row["test_start"]
    test_end = row["test_end"]

    train_start = row["train_start"]
    train_end = row["train_end"]

    train_regime = regime_data.loc[
        train_start:train_end
    ].copy()

    test_regime = regime_data.loc[
        test_start:test_end
    ].copy()

    # -----------------------------------------------------
    # Training-only volatility thresholds
    # -----------------------------------------------------

    low_threshold = train_regime["vol20"].quantile(0.33)
    high_threshold = train_regime["vol20"].quantile(0.67)

    def assign_regime(value):
        if value <= low_threshold:
            return "LOW"
        elif value >= high_threshold:
            return "HIGH"
        else:
            return "MEDIUM"

    test_regime["vol_regime"] = (
        test_regime["vol20"].apply(assign_regime)
    )

    # -----------------------------------------------------
    # Recreate frozen Opportunity Score
    # -----------------------------------------------------

    for feature in features:

        test_regime[feature + "_score"] = (
            percentile_score(
                train_regime[feature],
                test_regime[feature].values
            )
        )

    score_columns = [
        feature + "_score"
        for feature in features
    ]

    test_regime["opportunity_score"] = (
        test_regime[score_columns].mean(axis=1)
    )

    # -----------------------------------------------------
    # Select exact top 10%
    # -----------------------------------------------------

    n_select = int(len(test_regime) * TOP_PCT)

    selected_index = (
        test_regime
        .sort_values(
            "opportunity_score",
            ascending=False
        )
        .head(n_select)
        .index
    )

    test_regime["selected"] = (
        test_regime.index.isin(selected_index)
    )

    # -----------------------------------------------------
    # Analyse each regime
    # -----------------------------------------------------

    for regime in ["LOW", "MEDIUM", "HIGH"]:

        regime_all = test_regime[
            test_regime["vol_regime"] == regime
        ]

        regime_selected = regime_all[
            regime_all["selected"]
        ]

        if len(regime_all) == 0:
            continue

        training_big_move_threshold = (
            train_regime[target].quantile(0.75)
        )

        baseline_big = (
            regime_all[target] >=
            training_big_move_threshold
        ).mean()

        if len(regime_selected) > 0:

            selected_mean = (
                regime_selected[target].mean()
            )

            selected_big = (
                regime_selected[target] >=
                training_big_move_threshold
            ).mean()

        else:

            selected_mean = np.nan
            selected_big = np.nan

        regime_results.append({
            "window": row["window"],
            "regime": regime,
            "observations": len(regime_all),
            "selected": len(regime_selected),
            "baseline_mean": regime_all[target].mean(),
            "selected_mean": selected_mean,
            "baseline_big_move": baseline_big,
            "selected_big_move": selected_big
        })


# ---------------------------------------------------------
# Results dataframe
# ---------------------------------------------------------

regime_df = pd.DataFrame(regime_results)

regime_df["mean_improvement"] = (
    regime_df["selected_mean"] -
    regime_df["baseline_mean"]
)

regime_df["big_move_improvement"] = (
    regime_df["selected_big_move"] -
    regime_df["baseline_big_move"]
)

print("==============================")
print("V13g — VOLATILITY REGIMES")
print("==============================")

display_df = regime_df.copy()

for column in [
    "baseline_mean",
    "selected_mean",
    "mean_improvement"
]:
    display_df[column] *= 100

for column in [
    "baseline_big_move",
    "selected_big_move",
    "big_move_improvement"
]:
    display_df[column] *= 100

print(
    display_df[
        [
            "window",
            "regime",
            "observations",
            "selected",
            "baseline_mean",
            "selected_mean",
            "mean_improvement",
            "baseline_big_move",
            "selected_big_move",
            "big_move_improvement"
        ]
    ].to_string(
        index=False,
        formatters={
            "baseline_mean": lambda x: f"{x:.3f}%",
            "selected_mean": lambda x: f"{x:.3f}%",
            "mean_improvement": lambda x: f"{x:.3f} pp",
            "baseline_big_move": lambda x: f"{x:.2f}%",
            "selected_big_move": lambda x: f"{x:.2f}%",
            "big_move_improvement": lambda x: f"{x:.2f} pp"
        }
    )
)

# ---------------------------------------------------------
# Aggregate by regime
# ---------------------------------------------------------

print("\n==============================")
print("AGGREGATE BY REGIME")
print("==============================")

for regime in ["LOW", "MEDIUM", "HIGH"]:

    subset = regime_df[
        regime_df["regime"] == regime
    ].dropna(
        subset=["selected_mean"]
    )

    if len(subset) == 0:
        continue

    print("\n" + regime)

    print(
        "Windows:",
        len(subset)
    )

    print(
        "Average baseline:",
        round(
            subset["baseline_mean"].mean() * 100,
            3
        ),
        "%"
    )

    print(
        "Average selected:",
        round(
            subset["selected_mean"].mean() * 100,
            3
        ),
        "%"
    )

    print(
        "Average improvement:",
        round(
            subset["mean_improvement"].mean() * 100,
            3
        ),
        "percentage points"
    )

    print(
        "Average big-move improvement:",
        round(
            subset["big_move_improvement"].mean() * 100,
            2
        ),
        "percentage points"
    )

In [ ]:
# =========================================================
# V14 — DIRECTION ENGINE
# Cell 1: Build directional features
# =========================================================

direction_data = df.copy()

# ---------------------------------------------------------
# Future SIGNED movement
# ---------------------------------------------------------
# Unlike V13:
#   future_abs_24obs = how BIG?
#
# Here:
#   future_24obs = UP or DOWN?

direction_data["future_direction"] = (
    direction_data["close"].shift(-24)
    / direction_data["close"]
    - 1
)

# ---------------------------------------------------------
# Direction features
# ---------------------------------------------------------

# Momentum
direction_data["mom3"] = (
    direction_data["close"].pct_change(3)
)

direction_data["mom6"] = (
    direction_data["close"].pct_change(6)
)

direction_data["mom12"] = (
    direction_data["close"].pct_change(12)
)

direction_data["mom24"] = (
    direction_data["close"].pct_change(24)
)

# Moving averages
direction_data["MA5"] = (
    direction_data["close"].rolling(5).mean()
)

direction_data["MA10"] = (
    direction_data["close"].rolling(10).mean()
)

direction_data["MA20"] = (
    direction_data["close"].rolling(20).mean()
)

direction_data["MA50"] = (
    direction_data["close"].rolling(50).mean()
)

# Price relative to moving averages
direction_data["price_vs_MA10"] = (
    direction_data["close"]
    / direction_data["MA10"]
    - 1
)

direction_data["price_vs_MA20"] = (
    direction_data["close"]
    / direction_data["MA20"]
    - 1
)

direction_data["price_vs_MA50"] = (
    direction_data["close"]
    / direction_data["MA50"]
    - 1
)

# MA structure
direction_data["MA10_vs_MA20"] = (
    direction_data["MA10"]
    / direction_data["MA20"]
    - 1
)

direction_data["MA20_vs_MA50"] = (
    direction_data["MA20"]
    / direction_data["MA50"]
    - 1
)

# Recent price location
direction_data["high20"] = (
    direction_data["high"].rolling(20).max()
)

direction_data["low20"] = (
    direction_data["low"].rolling(20).min()
)

direction_data["range_position20"] = (
    (
        direction_data["close"]
        - direction_data["low20"]
    )
    /
    (
        direction_data["high20"]
        - direction_data["low20"]
    ).replace(0, np.nan)
)

# Distance from recent extremes
direction_data["distance_high20"] = (
    (
        direction_data["high20"]
        - direction_data["close"]
    )
    /
    direction_data["close"]
)

direction_data["distance_low20"] = (
    (
        direction_data["close"]
        - direction_data["low20"]
    )
    /
    direction_data["close"]
)

# Candle direction
direction_data["candle_return"] = (
    direction_data["close"]
    / direction_data["open"]
    - 1
)

# Short-term trend agreement
direction_data["trend_agreement"] = (
    (direction_data["close"] > direction_data["MA10"]).astype(int)
    +
    (direction_data["close"] > direction_data["MA20"]).astype(int)
    +
    (direction_data["close"] > direction_data["MA50"]).astype(int)
)

# Momentum agreement
direction_data["momentum_agreement"] = (
    (direction_data["mom3"] > 0).astype(int)
    +
    (direction_data["mom6"] > 0).astype(int)
    +
    (direction_data["mom12"] > 0).astype(int)
    +
    (direction_data["mom24"] > 0).astype(int)
)

# ---------------------------------------------------------
# Keep relevant columns
# ---------------------------------------------------------

direction_features = [
    "mom3",
    "mom6",
    "mom12",
    "mom24",
    "price_vs_MA10",
    "price_vs_MA20",
    "price_vs_MA50",
    "MA10_vs_MA20",
    "MA20_vs_MA50",
    "range_position20",
    "distance_high20",
    "distance_low20",
    "candle_return",
    "trend_agreement",
    "momentum_agreement"
]

direction_data = direction_data[
    direction_features +
    [
        "vol20",
        "atr_percent",
        "relative_volume",
        "future_abs_24obs",
        "future_direction"
    ]
].dropna()

print("==============================")
print("V14 — DIRECTION DATASET")
print("==============================")

print(
    "Rows:",
    len(direction_data)
)

print(
    "Date range:",
    direction_data.index.min(),
    "->",
    direction_data.index.max()
)

print("\nFuture 24-observation direction:")

print(
    "Mean:",
    round(
        direction_data["future_direction"].mean() * 100,
        3
    ),
    "%"
)

print(
    "Median:",
    round(
        direction_data["future_direction"].median() * 100,
        3
    ),
    "%"
)

print(
    "Positive:",
    round(
        (
            direction_data["future_direction"] > 0
        ).mean() * 100,
        2
    ),
    "%"
)

print(
    "Negative:",
    round(
        (
            direction_data["future_direction"] < 0
        ).mean() * 100,
        2
    ),
    "%"
)

print("\nDirection features:", len(direction_features))

for feature in direction_features:
    print("✓", feature)

In [ ]:
# =========================================================
# V14 — DIRECTION ENGINE
# Cell 2: Feature predictiveness
# =========================================================

# ---------------------------------------------------------
# Chronological split
# ---------------------------------------------------------

split = int(len(direction_data) * 0.70)

train_dir = direction_data.iloc[:split].copy()
test_dir = direction_data.iloc[split:].copy()

print("==============================")
print("V14 TRAIN / TEST")
print("==============================")

print(
    "Train:",
    len(train_dir),
    train_dir.index.min(),
    "->",
    train_dir.index.max()
)

print(
    "Test:",
    len(test_dir),
    test_dir.index.min(),
    "->",
    test_dir.index.max()
)

# ---------------------------------------------------------
# Identify high-opportunity observations
# ---------------------------------------------------------
# Recreate V13 opportunity score using TRAIN ONLY.

opportunity_features = [
    "vol20",
    "atr_percent",
    "distance_high20",
    "relative_volume"
]

for feature in opportunity_features:

    train_dir[
        feature + "_opp_score"
    ] = percentile_score(
        train_dir[feature],
        train_dir[feature].values
    )

    test_dir[
        feature + "_opp_score"
    ] = percentile_score(
        train_dir[feature],
        test_dir[feature].values
    )

opp_score_columns = [
    feature + "_opp_score"
    for feature in opportunity_features
]

train_dir["opportunity_score"] = (
    train_dir[opp_score_columns].mean(axis=1)
)

test_dir["opportunity_score"] = (
    test_dir[opp_score_columns].mean(axis=1)
)

# ---------------------------------------------------------
# Select TOP 10% opportunity observations
# ---------------------------------------------------------

train_cutoff = train_dir[
    "opportunity_score"
].quantile(0.90)

test_cutoff = test_dir[
    "opportunity_score"
].quantile(0.90)

train_high = train_dir[
    train_dir["opportunity_score"] >= train_cutoff
].copy()

test_high = test_dir[
    test_dir["opportunity_score"] >= test_cutoff
].copy()

print("\n==============================")
print("HIGH OPPORTUNITY DATA")
print("==============================")

print(
    "Training high-opportunity:",
    len(train_high)
)

print(
    "Test high-opportunity:",
    len(test_high)
)

print(
    "Train positive direction:",
    round(
        (
            train_high["future_direction"] > 0
        ).mean() * 100,
        2
    ),
    "%"
)

print(
    "Test positive direction:",
    round(
        (
            test_high["future_direction"] > 0
        ).mean() * 100,
        2
    ),
    "%"
)

# ---------------------------------------------------------
# Direction feature analysis
# ---------------------------------------------------------
#
# For every directional feature:
#
#   Q1 = lowest 25% of training values
#   Q4 = highest 25%
#
# We calculate:
#   average future direction
#   probability of UP
#
# Then see whether the relationship survives
# into the unseen test data.

direction_results = []

for feature in direction_features:

    # Training quartiles
    q1 = train_high[feature].quantile(0.25)
    q4 = train_high[feature].quantile(0.75)

    train_low = train_high[
        train_high[feature] <= q1
    ]

    train_high_feature = train_high[
        train_high[feature] >= q4
    ]

    test_low = test_high[
        test_high[feature] <= q1
    ]

    test_high_feature = test_high[
        test_high[feature] >= q4
    ]

    # Need enough observations
    if (
        len(train_low) < 20 or
        len(train_high_feature) < 20 or
        len(test_low) < 10 or
        len(test_high_feature) < 10
    ):
        continue

    train_low_mean = (
        train_low["future_direction"].mean()
    )

    train_high_mean = (
        train_high_feature["future_direction"].mean()
    )

    test_low_mean = (
        test_low["future_direction"].mean()
    )

    test_high_mean = (
        test_high_feature["future_direction"].mean()
    )

    train_spread = (
        train_high_mean -
        train_low_mean
    )

    test_spread = (
        test_high_mean -
        test_low_mean
    )

    train_low_up = (
        train_low["future_direction"] > 0
    ).mean()

    train_high_up = (
        train_high_feature["future_direction"] > 0
    ).mean()

    test_low_up = (
        test_low["future_direction"] > 0
    ).mean()

    test_high_up = (
        test_high_feature["future_direction"] > 0
    ).mean()

    # Does the direction of the relationship survive?
    survives = (
        np.sign(train_spread) ==
        np.sign(test_spread)
    )

    direction_results.append({

        "feature": feature,

        "train_low_mean":
            train_low_mean,

        "train_high_mean":
            train_high_mean,

        "train_spread":
            train_spread,

        "test_low_mean":
            test_low_mean,

        "test_high_mean":
            test_high_mean,

        "test_spread":
            test_spread,

        "train_low_up":
            train_low_up,

        "train_high_up":
            train_high_up,

        "test_low_up":
            test_low_up,

        "test_high_up":
            test_high_up,

        "survives":
            survives
    })

direction_results_df = pd.DataFrame(
    direction_results
)

# ---------------------------------------------------------
# Sort by unseen test strength
# ---------------------------------------------------------

direction_results_df = (
    direction_results_df
    .sort_values(
        "test_spread",
        ascending=False
    )
)

# ---------------------------------------------------------
# Display
# ---------------------------------------------------------

print("\n==============================")
print("DIRECTION FEATURE RESULTS")
print("==============================")

display_df = direction_results_df.copy()

for column in [
    "train_low_mean",
    "train_high_mean",
    "train_spread",
    "test_low_mean",
    "test_high_mean",
    "test_spread"
]:
    display_df[column] *= 100

for column in [
    "train_low_up",
    "train_high_up",
    "test_low_up",
    "test_high_up"
]:
    display_df[column] *= 100

print(
    display_df[
        [
            "feature",
            "train_spread",
            "test_spread",
            "train_low_up",
            "train_high_up",
            "test_low_up",
            "test_high_up",
            "survives"
        ]
    ].to_string(
        index=False,
        formatters={
            "train_spread":
                lambda x: f"{x:.3f} pp",

            "test_spread":
                lambda x: f"{x:.3f} pp",

            "train_low_up":
                lambda x: f"{x:.1f}%",

            "train_high_up":
                lambda x: f"{x:.1f}%",

            "test_low_up":
                lambda x: f"{x:.1f}%",

            "test_high_up":
                lambda x: f"{x:.1f}%"
        }
    )
)

# ---------------------------------------------------------
# Survival count
# ---------------------------------------------------------

survivors = direction_results_df[
    direction_results_df["survives"]
]

print("\n==============================")
print("SURVIVAL")
print("==============================")

print(
    "Features tested:",
    len(direction_results_df)
)

print(
    "Direction survived:",
    len(survivors),
    "/",
    len(direction_results_df)
)

print(
    "Survival rate:",
    round(
        len(survivors) /
        len(direction_results_df) *
        100,
        1
    ),
    "%"
)

In [ ]:

# =========================================================
# V14c — DIRECTION SCORE
# Corrected version
# =========================================================

surviving_direction_features = [
    "mom6",
    "MA10_vs_MA20",
    "mom3",
    "mom12",
    "candle_return"
]

# ---------------------------------------------------------
# 1. Determine direction of each feature
#    using TRAINING data only
# ---------------------------------------------------------

direction_signs = {}

print("==============================")
print("V14c — TRAINING DIRECTION SIGNS")
print("==============================")

for feature in surviving_direction_features:

    q1 = train_high[feature].quantile(0.25)
    q4 = train_high[feature].quantile(0.75)

    low_group = train_high[
        train_high[feature] <= q1
    ]

    high_group = train_high[
        train_high[feature] >= q4
    ]

    low_mean = (
        low_group["future_direction"].mean()
    )

    high_mean = (
        high_group["future_direction"].mean()
    )

    spread = high_mean - low_mean

    if spread >= 0:
        direction_signs[feature] = 1
        direction = "BULLISH"
    else:
        direction_signs[feature] = -1
        direction = "BEARISH"

    print(
        feature,
        "->",
        direction,
        "| spread:",
        round(spread * 100, 3),
        "pp"
    )

# ---------------------------------------------------------
# 2. Create scores ONLY for high-opportunity data
# ---------------------------------------------------------

train_high_scored = train_high.copy()
test_high_scored = test_high.copy()

for feature in surviving_direction_features:

    # Training percentile distribution
    train_high_scored[
        feature + "_dir_score"
    ] = percentile_score(
        train_high[feature],
        train_high[feature].values
    )

    # Test values scored against TRAIN distribution
    test_high_scored[
        feature + "_dir_score"
    ] = percentile_score(
        train_high[feature],
        test_high[feature].values
    )

    # Reverse bearish features
    if direction_signs[feature] == -1:

        train_high_scored[
            feature + "_dir_score"
        ] = (
            100 -
            train_high_scored[
                feature + "_dir_score"
            ]
        )

        test_high_scored[
            feature + "_dir_score"
        ] = (
            100 -
            test_high_scored[
                feature + "_dir_score"
            ]
        )

# ---------------------------------------------------------
# 3. Composite Direction Score
# ---------------------------------------------------------

direction_score_columns = [
    feature + "_dir_score"
    for feature in surviving_direction_features
]

train_high_scored["direction_score"] = (
    train_high_scored[
        direction_score_columns
    ].mean(axis=1)
)

test_high_scored["direction_score"] = (
    test_high_scored[
        direction_score_columns
    ].mean(axis=1)
)

# ---------------------------------------------------------
# 4. Direction groups
# ---------------------------------------------------------

bins = [
    -np.inf,
    20,
    40,
    60,
    80,
    np.inf
]

labels = [
    "STRONG_DOWN",
    "DOWN",
    "NEUTRAL",
    "UP",
    "STRONG_UP"
]

train_high_scored["direction_group"] = pd.cut(
    train_high_scored["direction_score"],
    bins=bins,
    labels=labels
)

test_high_scored["direction_group"] = pd.cut(
    test_high_scored["direction_score"],
    bins=bins,
    labels=labels
)

# ---------------------------------------------------------
# 5. TRAINING RESULTS
# ---------------------------------------------------------

print("\n==============================")
print("TRAINING DIRECTION GROUPS")
print("==============================")

train_groups = (
    train_high_scored
    .groupby(
        "direction_group",
        observed=True
    )["future_direction"]
    .agg(
        ["count", "mean", "median"]
    )
)

train_groups["up_rate"] = (
    train_high_scored
    .groupby(
        "direction_group",
        observed=True
    )["future_direction"]
    .apply(
        lambda x: (x > 0).mean()
    )
    .values
)

print(
    train_groups.to_string(
        formatters={
            "mean":
                lambda x: f"{x*100:.3f}%",
            "median":
                lambda x: f"{x*100:.3f}%",
            "up_rate":
                lambda x: f"{x*100:.2f}%"
        }
    )
)

# ---------------------------------------------------------
# 6. UNSEEN TEST RESULTS
# ---------------------------------------------------------

print("\n==============================")
print("UNSEEN TEST DIRECTION GROUPS")
print("==============================")

test_groups = (
    test_high_scored
    .groupby(
        "direction_group",
        observed=True
    )["future_direction"]
    .agg(
        ["count", "mean", "median"]
    )
)

test_groups["up_rate"] = (
    test_high_scored
    .groupby(
        "direction_group",
        observed=True
    )["future_direction"]
    .apply(
        lambda x: (x > 0).mean()
    )
    .values
)

print(
    test_groups.to_string(
        formatters={
            "mean":
                lambda x: f"{x*100:.3f}%",
            "median":
                lambda x: f"{x*100:.3f}%",
            "up_rate":
                lambda x: f"{x*100:.2f}%"
        }
    )
)

# ---------------------------------------------------------
# 7. Simple classifier summary
# ---------------------------------------------------------

print("\n==============================")
print("DIRECTION CLASSIFIER")
print("==============================")

groups_to_check = [
    ("STRONG DOWN", 0, 20),
    ("DOWN", 20, 40),
    ("UP", 60, 80),
    ("STRONG UP", 80, 100)
]

for name, lower, upper in groups_to_check:

    group = test_high_scored[
        (
            test_high_scored["direction_score"]
            >= lower
        )
        &
        (
            test_high_scored["direction_score"]
            < upper
        )
    ]

    if len(group) == 0:
        continue

    print(
        f"{name}:",
        len(group),
        "signals | UP rate:",
        round(
            (
                group["future_direction"] > 0
            ).mean() * 100,
            2
        ),
        "% | Avg future move:",
        round(
            group["future_direction"].mean() * 100,
            3
        ),
        "%"
    )

In [ ]:
# =========================================================
# V14d — SIMPLE DIRECTION RULES
# =========================================================
#
# We deliberately keep this simple.
#
# LONG:
#   mom6 > 0
#   MA10_vs_MA20 > 0
#
# SHORT:
#   mom6 < 0
#   MA10_vs_MA20 < 0
#
# Only evaluate these inside HIGH-OPPORTUNITY conditions.
#
# Then test them walk-forward.
# =========================================================

direction_rules = {
    "LONG_mom6": {
        "side": "LONG",
        "condition": (
            lambda x:
            x["mom6"] > 0
        )
    },

    "SHORT_mom6": {
        "side": "SHORT",
        "condition": (
            lambda x:
            x["mom6"] < 0
        )
    },

    "LONG_MA10_MA20": {
        "side": "LONG",
        "condition": (
            lambda x:
            x["MA10_vs_MA20"] > 0
        )
    },

    "SHORT_MA10_MA20": {
        "side": "SHORT",
        "condition": (
            lambda x:
            x["MA10_vs_MA20"] < 0
        )
    },

    "LONG_BOTH": {
        "side": "LONG",
        "condition": (
            lambda x:
            (x["mom6"] > 0) &
            (x["MA10_vs_MA20"] > 0)
        )
    },

    "SHORT_BOTH": {
        "side": "SHORT",
        "condition": (
            lambda x:
            (x["mom6"] < 0) &
            (x["MA10_vs_MA20"] < 0)
        )
    }
}

# ---------------------------------------------------------
# Walk-forward settings
# ---------------------------------------------------------

TRAIN_SIZE = 5000
TEST_SIZE = 1000
STEP_SIZE = 1000

rule_results = []

window_number = 0
start = 0

while (
    start + TRAIN_SIZE + TEST_SIZE
    <= len(direction_data)
):

    window_number += 1

    train_wf = direction_data.iloc[
        start:
        start + TRAIN_SIZE
    ].copy()

    test_wf = direction_data.iloc[
        start + TRAIN_SIZE:
        start + TRAIN_SIZE + TEST_SIZE
    ].copy()

    # -----------------------------------------------------
    # Build Opportunity Score using TRAIN only
    # -----------------------------------------------------

    for feature in opportunity_features:

        train_wf[
            feature + "_opp_score"
        ] = percentile_score(
            train_wf[feature],
            train_wf[feature].values
        )

        test_wf[
            feature + "_opp_score"
        ] = percentile_score(
            train_wf[feature],
            test_wf[feature].values
        )

    opp_columns = [
        feature + "_opp_score"
        for feature in opportunity_features
    ]

    train_wf["opportunity_score"] = (
        train_wf[opp_columns].mean(axis=1)
    )

    test_wf["opportunity_score"] = (
        test_wf[opp_columns].mean(axis=1)
    )

    # -----------------------------------------------------
    # Top 10% opportunity threshold
    # -----------------------------------------------------

    opportunity_cutoff = (
        train_wf["opportunity_score"]
        .quantile(0.90)
    )

    high_opportunity = test_wf[
        test_wf["opportunity_score"]
        >= opportunity_cutoff
    ].copy()

    # -----------------------------------------------------
    # Test every simple directional rule
    # -----------------------------------------------------

    for rule_name, rule in direction_rules.items():

        mask = rule["condition"](
            high_opportunity
        )

        selected = high_opportunity[
            mask
        ].copy()

        if len(selected) < 10:
            continue

        future = selected[
            "future_direction"
        ]

        # LONG return
        if rule["side"] == "LONG":

            directional_return = future

        # SHORT return
        else:

            directional_return = -future

        rule_results.append({

            "window": window_number,

            "rule": rule_name,

            "side": rule["side"],

            "signals": len(selected),

            "avg_return":
                directional_return.mean(),

            "median_return":
                directional_return.median(),

            "win_rate":
                (directional_return > 0).mean(),

            "best":
                directional_return.max(),

            "worst":
                directional_return.min()
        })

    start += STEP_SIZE


# ---------------------------------------------------------
# Results
# ---------------------------------------------------------

rules_df = pd.DataFrame(
    rule_results
)

print("==============================")
print("V14d — WALK-FORWARD RULES")
print("==============================")

display_df = rules_df.copy()

display_df["avg_return"] *= 100
display_df["median_return"] *= 100
display_df["win_rate"] *= 100
display_df["best"] *= 100
display_df["worst"] *= 100

print(
    display_df.to_string(
        index=False,
        formatters={
            "avg_return":
                lambda x: f"{x:.3f}%",
            "median_return":
                lambda x: f"{x:.3f}%",
            "win_rate":
                lambda x: f"{x:.1f}%",
            "best":
                lambda x: f"{x:.2f}%",
            "worst":
                lambda x: f"{x:.2f}%"
        }
    )
)

# ---------------------------------------------------------
# Aggregate by rule
# ---------------------------------------------------------

print("\n==============================")
print("AGGREGATE RULE PERFORMANCE")
print("==============================")

for rule_name in direction_rules.keys():

    subset = rules_df[
        rules_df["rule"] == rule_name
    ].copy()

    if len(subset) == 0:
        continue

    total_signals = subset[
        "signals"
    ].sum()

    weighted_return = (
        (
            subset["avg_return"]
            * subset["signals"]
        ).sum()
        / total_signals
    )

    weighted_win = (
        (
            subset["win_rate"]
            * subset["signals"]
        ).sum()
        / total_signals
    )

    positive_windows = (
        subset["avg_return"] > 0
    ).sum()

    print(
        "\n",
        rule_name
    )

    print(
        "Signals:",
        total_signals
    )

    print(
        "Weighted avg return:",
        round(
            weighted_return * 100,
            3
        ),
        "%"
    )

    print(
        "Weighted win rate:",
        round(
            weighted_win * 100,
            2
        ),
        "%"
    )

    print(
        "Positive windows:",
        positive_windows,
        "/",
        len(subset)
    )

In [ ]:
from google.colab import userdata

OANDA_ACCOUNT_ID = userdata.get("OANDA_ACCOUNT_ID")
OANDA_API_TOKEN = userdata.get("OANDA_API_TOKEN")

OANDA_URL = "https://api-fxpractice.oanda.com"

print("✅ Credentials loaded securely")
print("🔐 Credentials not displayed")

In [ ]:
# =========================================================
# SILVER AGENT — STEP 5
# DATABASE HEALTH CHECK
# =========================================================

import pandas as pd
import numpy as np
import os

DATABASE_PATH = (
    "/content/drive/MyDrive/"
    "SilverAgent/database/silver_h1.parquet"
)

print("=" * 60)
print("SILVER DATABASE HEALTH CHECK")
print("=" * 60)

# ---------------------------------------------------------
# Load database
# ---------------------------------------------------------

db = pd.read_parquet(DATABASE_PATH)
db = db.sort_index()

# ---------------------------------------------------------
# Basic information
# ---------------------------------------------------------

print(f"\nCandles: {len(db):,}")
print(f"Start:   {db.index.min()}")
print(f"End:     {db.index.max()}")

# ---------------------------------------------------------
# Duplicate timestamps
# ---------------------------------------------------------

duplicates = db.index.duplicated().sum()

print(f"\nDuplicate timestamps: {duplicates}")

# ---------------------------------------------------------
# Missing values
# ---------------------------------------------------------

required_columns = [
    "open",
    "high",
    "low",
    "close",
    "volume"
]

missing = db[required_columns].isna().sum()

print("\nMissing values:")
print(missing)

# ---------------------------------------------------------
# Price validation
# ---------------------------------------------------------

invalid_prices = (
    (db["open"] <= 0) |
    (db["high"] <= 0) |
    (db["low"] <= 0) |
    (db["close"] <= 0) |
    (db["high"] < db["low"]) |
    (db["high"] < db["open"]) |
    (db["high"] < db["close"]) |
    (db["low"] > db["open"]) |
    (db["low"] > db["close"])
).sum()

print(f"\nInvalid price rows: {invalid_prices}")

# ---------------------------------------------------------
# Time gaps
# ---------------------------------------------------------

time_diff = db.index.to_series().diff()

expected = pd.Timedelta(hours=1)

gaps = (time_diff > expected).sum()

print(f"Hourly gaps: {gaps}")

# ---------------------------------------------------------
# File size
# ---------------------------------------------------------

file_size = os.path.getsize(DATABASE_PATH)

print(
    f"Database size: "
    f"{file_size / (1024 ** 2):.2f} MB"
)

# ---------------------------------------------------------
# Latest market data
# ---------------------------------------------------------

latest = db.iloc[-1]

print("\nLatest candle:")
print(f"Time:   {db.index[-1]}")
print(f"Open:   {latest['open']}")
print(f"High:   {latest['high']}")
print(f"Low:    {latest['low']}")
print(f"Close:  {latest['close']}")
print(f"Volume: {latest['volume']}")

# ---------------------------------------------------------
# Overall verdict
# ---------------------------------------------------------

healthy = (
    duplicates == 0
    and missing.sum() == 0
    and invalid_prices == 0
    and db.index.is_monotonic_increasing
)

print("\n" + "=" * 60)

if healthy:
    print("🟢 DATABASE HEALTH: GOOD")
else:
    print("🔴 DATABASE HEALTH: PROBLEM DETECTED")

print("=" * 60)

In [ ]:
# =========================================================
# SILVER AGENT — STEP 5
# DATABASE HEALTH CHECK + AUTO DRIVE MOUNT
# =========================================================

import os
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. Make sure Google Drive is mounted
# ---------------------------------------------------------

DRIVE_ROOT = "/content/drive"

if not os.path.exists("/content/drive/MyDrive"):
    print("🔄 Google Drive not mounted — mounting now...")

    from google.colab import drive
    drive.mount("/content/drive")

print("✅ Google Drive available")

# ---------------------------------------------------------
# 2. Database path
# ---------------------------------------------------------

DATABASE_PATH = (
    "/content/drive/MyDrive/"
    "SilverAgent/database/silver_h1.parquet"
)

# ---------------------------------------------------------
# 3. Check database exists
# ---------------------------------------------------------

if not os.path.exists(DATABASE_PATH):
    raise FileNotFoundError(
        f"\n❌ Database not found:\n{DATABASE_PATH}\n\n"
        "Check Google Drive → SilverAgent → database."
    )

print("✅ Database file found")

# ---------------------------------------------------------
# 4. Load database
# ---------------------------------------------------------

db = pd.read_parquet(DATABASE_PATH)
db = db.sort_index()

print("\n" + "=" * 60)
print("SILVER DATABASE HEALTH CHECK")
print("=" * 60)

# ---------------------------------------------------------
# 5. Basic information
# ---------------------------------------------------------

print(f"\nCandles: {len(db):,}")
print(f"Start:   {db.index.min()}")
print(f"End:     {db.index.max()}")

# ---------------------------------------------------------
# 6. Duplicate timestamps
# ---------------------------------------------------------

duplicates = db.index.duplicated().sum()

print(f"\nDuplicate timestamps: {duplicates}")

# ---------------------------------------------------------
# 7. Missing values
# ---------------------------------------------------------

required_columns = [
    "open",
    "high",
    "low",
    "close",
    "volume"
]

missing = db[required_columns].isna().sum()

print("\nMissing values:")
print(missing)

# ---------------------------------------------------------
# 8. Price validation
# ---------------------------------------------------------

invalid_prices = (
    (db["open"] <= 0) |
    (db["high"] <= 0) |
    (db["low"] <= 0) |
    (db["close"] <= 0) |
    (db["high"] < db["low"]) |
    (db["high"] < db["open"]) |
    (db["high"] < db["close"]) |
    (db["low"] > db["open"]) |
    (db["low"] > db["close"])
).sum()

print(f"\nInvalid price rows: {invalid_prices}")

# ---------------------------------------------------------
# 9. Time gaps
# ---------------------------------------------------------

time_diff = db.index.to_series().diff()

expected = pd.Timedelta(hours=1)

gaps = (time_diff > expected).sum()

print(f"Hourly gaps: {gaps}")

# ---------------------------------------------------------
# 10. Chronological order
# ---------------------------------------------------------

ordered = db.index.is_monotonic_increasing

print(f"Chronological order: {ordered}")

# ---------------------------------------------------------
# 11. File size
# ---------------------------------------------------------

file_size = os.path.getsize(DATABASE_PATH)

print(
    f"Database size: "
    f"{file_size / (1024 ** 2):.2f} MB"
)

# ---------------------------------------------------------
# 12. Latest candle
# ---------------------------------------------------------

latest = db.iloc[-1]

print("\nLatest candle:")
print(f"Time:   {db.index[-1]}")
print(f"Open:   {latest['open']}")
print(f"High:   {latest['high']}")
print(f"Low:    {latest['low']}")
print(f"Close:  {latest['close']}")
print(f"Volume: {latest['volume']}")

# ---------------------------------------------------------
# 13. Overall health
# ---------------------------------------------------------

healthy = (
    duplicates == 0
    and missing.sum() == 0
    and invalid_prices == 0
    and ordered
)

print("\n" + "=" * 60)

if healthy:
    print("🟢 DATABASE HEALTH: GOOD")
else:
    print("🔴 DATABASE HEALTH: PROBLEM DETECTED")

print("=" * 60)

In [ ]:
# =========================================================
# SILVER AGENT — STEP 6
# FEATURE ENGINE V1
# =========================================================

import os
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Make sure Drive is available
# ---------------------------------------------------------

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

# ---------------------------------------------------------
# 2. Database
# ---------------------------------------------------------

DATABASE_PATH = (
    "/content/drive/MyDrive/"
    "SilverAgent/database/silver_h1.parquet"
)

if not os.path.exists(DATABASE_PATH):
    raise FileNotFoundError("Silver database not found.")

raw = pd.read_parquet(DATABASE_PATH).sort_index()

print("=" * 60)
print("SILVER FEATURE ENGINE V1")
print("=" * 60)

print(f"Raw candles: {len(raw):,}")

# ---------------------------------------------------------
# 3. Gap-aware observation numbering
# ---------------------------------------------------------

data = raw.copy()

time_diff = data.index.to_series().diff()

# A normal consecutive H1 candle
data["consecutive"] = time_diff == pd.Timedelta(hours=1)

# ---------------------------------------------------------
# 4. Price returns
# ---------------------------------------------------------

data["return_1h"] = (
    data["close"] / data["close"].shift(1) - 1
)

data["return_3h"] = (
    data["close"] / data["close"].shift(3) - 1
)

data["return_6h"] = (
    data["close"] / data["close"].shift(6) - 1
)

data["return_12h"] = (
    data["close"] / data["close"].shift(12) - 1
)

data["return_24h"] = (
    data["close"] / data["close"].shift(24) - 1
)

# ---------------------------------------------------------
# 5. Moving averages
# ---------------------------------------------------------

data["MA5"] = data["close"].rolling(5).mean()
data["MA10"] = data["close"].rolling(10).mean()
data["MA20"] = data["close"].rolling(20).mean()
data["MA50"] = data["close"].rolling(50).mean()

data["price_vs_MA10"] = (
    data["close"] / data["MA10"] - 1
)

data["price_vs_MA20"] = (
    data["close"] / data["MA20"] - 1
)

data["price_vs_MA50"] = (
    data["close"] / data["MA50"] - 1
)

data["MA10_vs_MA20"] = (
    data["MA10"] / data["MA20"] - 1
)

data["MA20_vs_MA50"] = (
    data["MA20"] / data["MA50"] - 1
)

# ---------------------------------------------------------
# 6. Momentum
# ---------------------------------------------------------

data["mom3"] = data["return_3h"]
data["mom6"] = data["return_6h"]
data["mom12"] = data["return_12h"]
data["mom24"] = data["return_24h"]

# ---------------------------------------------------------
# 7. Volatility
# ---------------------------------------------------------

data["vol10"] = (
    data["return_1h"]
    .rolling(10)
    .std()
)

data["vol20"] = (
    data["return_1h"]
    .rolling(20)
    .std()
)

# ---------------------------------------------------------
# 8. True Range / ATR
# ---------------------------------------------------------

previous_close = data["close"].shift(1)

tr1 = data["high"] - data["low"]
tr2 = (data["high"] - previous_close).abs()
tr3 = (data["low"] - previous_close).abs()

data["true_range"] = pd.concat(
    [tr1, tr2, tr3],
    axis=1
).max(axis=1)

data["ATR20"] = (
    data["true_range"]
    .rolling(20)
    .mean()
)

data["atr_percent"] = (
    data["ATR20"] / data["close"]
)

# ---------------------------------------------------------
# 9. Candle structure
# ---------------------------------------------------------

data["candle_range"] = (
    data["high"] - data["low"]
)

data["candle_body"] = (
    data["close"] - data["open"]
)

data["body_abs"] = (
    data["candle_body"].abs()
)

data["body_ratio"] = np.where(
    data["candle_range"] > 0,
    data["body_abs"] / data["candle_range"],
    0
)

# ---------------------------------------------------------
# 10. 20-candle price location
# ---------------------------------------------------------

data["high20"] = (
    data["high"].rolling(20).max()
)

data["low20"] = (
    data["low"].rolling(20).min()
)

data["distance_high20"] = (
    data["high20"] / data["close"] - 1
)

data["distance_low20"] = (
    data["close"] / data["low20"] - 1
)

data["range_position20"] = np.where(
    (data["high20"] - data["low20"]) > 0,
    (data["close"] - data["low20"]) /
    (data["high20"] - data["low20"]),
    0.5
)

# ---------------------------------------------------------
# 11. Volume
# ---------------------------------------------------------

data["volume20"] = (
    data["volume"].rolling(20).mean()
)

data["relative_volume"] = np.where(
    data["volume20"] > 0,
    data["volume"] / data["volume20"],
    0
)

# ---------------------------------------------------------
# 12. Agreement signals
# ---------------------------------------------------------

data["momentum_agreement"] = (
    (data["mom3"] > 0).astype(int) +
    (data["mom6"] > 0).astype(int) +
    (data["mom12"] > 0).astype(int) +
    (data["mom24"] > 0).astype(int)
)

data["trend_agreement"] = (
    (data["price_vs_MA10"] > 0).astype(int) +
    (data["price_vs_MA20"] > 0).astype(int) +
    (data["price_vs_MA50"] > 0).astype(int)
)

# ---------------------------------------------------------
# 13. Future research targets
# ---------------------------------------------------------
# These are ONLY for historical research/backtesting.
# They are never used as live inputs.

data["future_6h"] = (
    data["close"].shift(-6) /
    data["close"] - 1
)

data["future_12h"] = (
    data["close"].shift(-12) /
    data["close"] - 1
)

data["future_24h"] = (
    data["close"].shift(-24) /
    data["close"] - 1
)

data["future_abs_24h"] = (
    data["future_24h"].abs()
)

# ---------------------------------------------------------
# 14. Remove unusable rows
# ---------------------------------------------------------

feature_columns = [
    "MA5",
    "MA10",
    "MA20",
    "MA50",
    "mom3",
    "mom6",
    "mom12",
    "mom24",
    "vol10",
    "vol20",
    "ATR20",
    "atr_percent",
    "high20",
    "low20",
    "volume20"
]

features = data.dropna(
    subset=feature_columns
).copy()

# ---------------------------------------------------------
# 15. Latest market state
# ---------------------------------------------------------

latest = features.iloc[-1]

print("\nFeature rows:", len(features))
print(
    "Feature period:",
    features.index.min(),
    "->",
    features.index.max()
)

print("\n" + "=" * 60)
print("LATEST SILVER MARKET STATE")
print("=" * 60)

print(f"Price:              {latest['close']:.4f}")
print(f"1h return:           {latest['return_1h'] * 100:.3f}%")
print(f"6h momentum:         {latest['mom6'] * 100:.3f}%")
print(f"12h momentum:        {latest['mom12'] * 100:.3f}%")
print(f"24h momentum:        {latest['mom24'] * 100:.3f}%")
print(f"MA10 vs MA20:        {latest['MA10_vs_MA20'] * 100:.3f}%")
print(f"Volatility 20h:      {latest['vol20'] * 100:.3f}%")
print(f"ATR:                 {latest['ATR20']:.4f}")
print(f"ATR %:               {latest['atr_percent'] * 100:.3f}%")
print(f"Relative volume:     {latest['relative_volume']:.2f}x")
print(f"Range position:      {latest['range_position20'] * 100:.1f}%")
print(f"Momentum agreement:  {latest['momentum_agreement']}/4")
print(f"Trend agreement:     {latest['trend_agreement']}/3")

print("\n🟢 FEATURE ENGINE COMPLETE")

In [ ]:
# =========================================================
# SILVER AGENT — STEP 7
# OPPORTUNITY ENGINE V1
# =========================================================

import os
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Make sure Drive is mounted
# ---------------------------------------------------------

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

DATABASE_PATH = (
    "/content/drive/MyDrive/"
    "SilverAgent/database/silver_h1.parquet"
)

RESULTS_DIR = (
    "/content/drive/MyDrive/"
    "SilverAgent/results"
)

os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------------------------------------------------------
# 2. Load database
# ---------------------------------------------------------

raw = pd.read_parquet(DATABASE_PATH).sort_index()

# ---------------------------------------------------------
# 3. Build the features needed by Opportunity Engine
# ---------------------------------------------------------

data = raw.copy()

data["return_1h"] = (
    data["close"] / data["close"].shift(1) - 1
)

data["vol10"] = (
    data["return_1h"]
    .rolling(10)
    .std()
)

data["vol20"] = (
    data["return_1h"]
    .rolling(20)
    .std()
)

previous_close = data["close"].shift(1)

tr1 = data["high"] - data["low"]
tr2 = (data["high"] - previous_close).abs()
tr3 = (data["low"] - previous_close).abs()

data["true_range"] = pd.concat(
    [tr1, tr2, tr3],
    axis=1
).max(axis=1)

data["ATR20"] = (
    data["true_range"]
    .rolling(20)
    .mean()
)

data["atr_percent"] = (
    data["ATR20"] / data["close"]
)

data["high20"] = (
    data["high"].rolling(20).max()
)

data["distance_high20"] = (
    data["high20"] / data["close"] - 1
)

data["volume20"] = (
    data["volume"].rolling(20).mean()
)

data["relative_volume"] = np.where(
    data["volume20"] > 0,
    data["volume"] / data["volume20"],
    0
)

# ---------------------------------------------------------
# 4. Remove rows without complete features
# ---------------------------------------------------------

opportunity_features = [
    "vol10",
    "vol20",
    "atr_percent",
    "distance_high20",
    "relative_volume"
]

data = data.dropna(
    subset=opportunity_features
).copy()

# ---------------------------------------------------------
# 5. Historical reference distribution
# ---------------------------------------------------------
# We deliberately use ONLY information available before
# the current candle when calculating today's score.
#
# Expanding percentile rank = current value compared with
# historical values seen up to that point.

for feature in opportunity_features:

    data[f"{feature}_rank"] = (
        data[feature]
        .expanding(min_periods=500)
        .rank(pct=True)
    )

# ---------------------------------------------------------
# 6. Opportunity score
# ---------------------------------------------------------
#
# Volatility / ATR are intentionally important because V13
# showed the strongest evidence for predicting MOVE SIZE.
#
# We use five components:
#
# volatility 10h
# volatility 20h
# ATR %
# distance from 20h high
# relative volume

data["opportunity_score"] = (
    data["vol10_rank"] +
    data["vol20_rank"] +
    data["atr_percent_rank"] +
    data["distance_high20_rank"] +
    data["relative_volume_rank"]
) / 5

# ---------------------------------------------------------
# 7. Convert score to 0–100
# ---------------------------------------------------------

data["opportunity_score_100"] = (
    data["opportunity_score"] * 100
)

# ---------------------------------------------------------
# 8. Opportunity zones
# ---------------------------------------------------------

data["opportunity_zone"] = pd.cut(
    data["opportunity_score_100"],
    bins=[
        -np.inf,
        20,
        40,
        60,
        80,
        np.inf
    ],
    labels=[
        "VERY_LOW",
        "LOW",
        "MEDIUM",
        "HIGH",
        "VERY_HIGH"
    ]
)

# ---------------------------------------------------------
# 9. Current state
# ---------------------------------------------------------

latest = data.iloc[-1]

print("=" * 60)
print("SILVER OPPORTUNITY ENGINE V1")
print("=" * 60)

print(f"\nCurrent price:       {latest['close']:.4f}")

print(
    f"Volatility 10h:      "
    f"{latest['vol10'] * 100:.3f}%"
)

print(
    f"Volatility 20h:      "
    f"{latest['vol20'] * 100:.3f}%"
)

print(
    f"ATR %:               "
    f"{latest['atr_percent'] * 100:.3f}%"
)

print(
    f"Distance high 20h:   "
    f"{latest['distance_high20'] * 100:.3f}%"
)

print(
    f"Relative volume:     "
    f"{latest['relative_volume']:.2f}x"
)

print("\n" + "-" * 60)

print(
    f"OPPORTUNITY SCORE:   "
    f"{latest['opportunity_score_100']:.1f}/100"
)

print(
    f"OPPORTUNITY ZONE:    "
    f"{latest['opportunity_zone']}"
)

# ---------------------------------------------------------
# 10. Individual component scores
# ---------------------------------------------------------

print("\nComponent scores:")

for feature in opportunity_features:

    print(
        f"  {feature:20s}"
        f"{latest[f'{feature}_rank'] * 100:6.1f}/100"
    )

# ---------------------------------------------------------
# 11. Simple interpretation
# ---------------------------------------------------------

score = latest["opportunity_score_100"]

print("\n" + "-" * 60)

if score >= 80:
    interpretation = (
        "VERY HIGH — unusually energetic market. "
        "Worth investigating direction."
    )

elif score >= 60:
    interpretation = (
        "HIGH — market conditions are interesting. "
        "Direction layer should investigate."
    )

elif score >= 40:
    interpretation = (
        "MEDIUM — some activity, but not exceptional."
    )

elif score >= 20:
    interpretation = (
        "LOW — limited evidence of an unusually large move."
    )

else:
    interpretation = (
        "VERY LOW — quiet conditions."
    )

print("Agent interpretation:")
print(interpretation)

# ---------------------------------------------------------
# 12. Save opportunity history
# ---------------------------------------------------------

history_path = (
    RESULTS_DIR +
    "/opportunity_history.parquet"
)

save_columns = [
    "close",
    "vol10",
    "vol20",
    "atr_percent",
    "distance_high20",
    "relative_volume",
    "opportunity_score_100",
    "opportunity_zone"
]

data[save_columns].to_parquet(
    history_path,
    engine="pyarrow",
    index=True
)

print("\n" + "=" * 60)
print("✅ OPPORTUNITY ENGINE COMPLETE")
print(f"✅ History saved to:")
print(history_path)
print("=" * 60)

In [ ]:
# =========================================================
# SILVER AGENT — STEP 8
# DIRECTION ENGINE V1
# =========================================================

import os
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Make sure Drive is mounted
# ---------------------------------------------------------

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

DATABASE_PATH = (
    "/content/drive/MyDrive/"
    "SilverAgent/database/silver_h1.parquet"
)

RESULTS_DIR = (
    "/content/drive/MyDrive/"
    "SilverAgent/results"
)

os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------------------------------------------------------
# 2. Load database
# ---------------------------------------------------------

data = pd.read_parquet(DATABASE_PATH).sort_index()

# ---------------------------------------------------------
# 3. Moving averages
# ---------------------------------------------------------

data["MA10"] = (
    data["close"].rolling(10).mean()
)

data["MA20"] = (
    data["close"].rolling(20).mean()
)

data["MA50"] = (
    data["close"].rolling(50).mean()
)

# ---------------------------------------------------------
# 4. Momentum
# ---------------------------------------------------------

data["mom3"] = (
    data["close"] /
    data["close"].shift(3) - 1
)

data["mom6"] = (
    data["close"] /
    data["close"].shift(6) - 1
)

data["mom12"] = (
    data["close"] /
    data["close"].shift(12) - 1
)

data["mom24"] = (
    data["close"] /
    data["close"].shift(24) - 1
)

# ---------------------------------------------------------
# 5. Relative position
# ---------------------------------------------------------

data["price_vs_MA10"] = (
    data["close"] / data["MA10"] - 1
)

data["price_vs_MA20"] = (
    data["close"] / data["MA20"] - 1
)

data["price_vs_MA50"] = (
    data["close"] / data["MA50"] - 1
)

data["MA10_vs_MA20"] = (
    data["MA10"] / data["MA20"] - 1
)

data["MA20_vs_MA50"] = (
    data["MA20"] / data["MA50"] - 1
)

# ---------------------------------------------------------
# 6. Agreement measures
# ---------------------------------------------------------

data["momentum_agreement"] = (
    (data["mom3"] > 0).astype(int) +
    (data["mom6"] > 0).astype(int) +
    (data["mom12"] > 0).astype(int) +
    (data["mom24"] > 0).astype(int)
)

data["trend_agreement"] = (
    (data["price_vs_MA10"] > 0).astype(int) +
    (data["price_vs_MA20"] > 0).astype(int) +
    (data["price_vs_MA50"] > 0).astype(int)
)

# ---------------------------------------------------------
# 7. Direction score
# ---------------------------------------------------------
#
# Evidence weights:
#
# MA10 > MA20       strongest tested directional signal
# MA20 > MA50       longer trend confirmation
# mom6               useful short-term direction
# mom12              medium-term direction
# mom24              longer momentum
# price vs MA20      trend confirmation
#
# This is an evidence score, NOT a probability.

data["direction_score_raw"] = (
      3.0 * np.sign(data["MA10_vs_MA20"])
    + 1.5 * np.sign(data["MA20_vs_MA50"])
    + 1.5 * np.sign(data["mom6"])
    + 1.0 * np.sign(data["mom12"])
    + 0.75 * np.sign(data["mom24"])
    + 1.0 * np.sign(data["price_vs_MA20"])
)

# Maximum absolute score
MAX_SCORE = (
    3.0 +
    1.5 +
    1.5 +
    1.0 +
    0.75 +
    1.0
)

# Convert to -100 → +100
data["direction_score"] = (
    data["direction_score_raw"] /
    MAX_SCORE
) * 100

# ---------------------------------------------------------
# 8. Bias
# ---------------------------------------------------------

data["direction_bias"] = np.select(
    [
        data["direction_score"] >= 35,
        data["direction_score"] <= -35
    ],
    [
        "LONG",
        "SHORT"
    ],
    default="NEUTRAL"
)

# ---------------------------------------------------------
# 9. Current state
# ---------------------------------------------------------

latest = data.dropna(
    subset=[
        "MA10",
        "MA20",
        "MA50",
        "mom6",
        "mom12",
        "mom24"
    ]
).iloc[-1]

score = latest["direction_score"]

print("=" * 60)
print("SILVER DIRECTION ENGINE V1")
print("=" * 60)

print(f"\nCurrent price:       {latest['close']:.4f}")

print("\nTrend:")
print(
    f"  MA10 vs MA20:      "
    f"{latest['MA10_vs_MA20'] * 100:+.3f}%"
)

print(
    f"  MA20 vs MA50:      "
    f"{latest['MA20_vs_MA50'] * 100:+.3f}%"
)

print("\nMomentum:")
print(
    f"  6h:                "
    f"{latest['mom6'] * 100:+.3f}%"
)

print(
    f"  12h:               "
    f"{latest['mom12'] * 100:+.3f}%"
)

print(
    f"  24h:               "
    f"{latest['mom24'] * 100:+.3f}%"
)

print("\nAgreement:")
print(
    f"  Momentum:          "
    f"{int(latest['momentum_agreement'])}/4"
)

print(
    f"  Trend:             "
    f"{int(latest['trend_agreement'])}/3"
)

print("\n" + "-" * 60)

print(
    f"DIRECTION SCORE:     "
    f"{score:+.1f}/100"
)

print(
    f"DIRECTION BIAS:      "
    f"{latest['direction_bias']}"
)

# ---------------------------------------------------------
# 10. Evidence interpretation
# ---------------------------------------------------------

if score >= 60:

    interpretation = (
        "Strong bullish evidence."
    )

elif score >= 35:

    interpretation = (
        "Moderate bullish evidence."
    )

elif score <= -60:

    interpretation = (
        "Strong bearish evidence."
    )

elif score <= -35:

    interpretation = (
        "Moderate bearish evidence."
    )

else:

    interpretation = (
        "Conflicting or insufficient directional evidence."
    )

print(
    "\nAgent interpretation:"
)

print(interpretation)

# ---------------------------------------------------------
# 11. Save direction history
# ---------------------------------------------------------

direction_path = (
    RESULTS_DIR +
    "/direction_history.parquet"
)

save_columns = [
    "close",
    "MA10_vs_MA20",
    "MA20_vs_MA50",
    "mom6",
    "mom12",
    "mom24",
    "momentum_agreement",
    "trend_agreement",
    "direction_score",
    "direction_bias"
]

data[save_columns].to_parquet(
    direction_path,
    engine="pyarrow",
    index=True
)

print("\n" + "=" * 60)
print("✅ DIRECTION ENGINE COMPLETE")
print(f"✅ History saved to:")
print(direction_path)
print("=" * 60)

In [ ]:
# =========================================================
# SILVER AGENT — STEP 9
# SETUP VALIDATOR V1
# =========================================================

import os
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Mount Drive if necessary
# ---------------------------------------------------------

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

DATABASE_PATH = (
    "/content/drive/MyDrive/"
    "SilverAgent/database/silver_h1.parquet"
)

RESULTS_DIR = (
    "/content/drive/MyDrive/"
    "SilverAgent/results"
)

os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------------------------------------------------------
# 2. Load database
# ---------------------------------------------------------

data = pd.read_parquet(DATABASE_PATH).sort_index()

# ---------------------------------------------------------
# 3. Build features
# ---------------------------------------------------------

data["return_1h"] = (
    data["close"] / data["close"].shift(1) - 1
)

data["vol10"] = (
    data["return_1h"].rolling(10).std()
)

data["vol20"] = (
    data["return_1h"].rolling(20).std()
)

previous_close = data["close"].shift(1)

tr1 = data["high"] - data["low"]
tr2 = (data["high"] - previous_close).abs()
tr3 = (data["low"] - previous_close).abs()

data["true_range"] = pd.concat(
    [tr1, tr2, tr3],
    axis=1
).max(axis=1)

data["ATR20"] = (
    data["true_range"].rolling(20).mean()
)

data["atr_percent"] = (
    data["ATR20"] / data["close"]
)

data["high20"] = (
    data["high"].rolling(20).max()
)

data["distance_high20"] = (
    data["high20"] / data["close"] - 1
)

data["volume20"] = (
    data["volume"].rolling(20).mean()
)

data["relative_volume"] = np.where(
    data["volume20"] > 0,
    data["volume"] / data["volume20"],
    0
)

# ---------------------------------------------------------
# Direction features
# ---------------------------------------------------------

data["MA10"] = data["close"].rolling(10).mean()
data["MA20"] = data["close"].rolling(20).mean()
data["MA50"] = data["close"].rolling(50).mean()

data["MA10_vs_MA20"] = (
    data["MA10"] / data["MA20"] - 1
)

data["MA20_vs_MA50"] = (
    data["MA20"] / data["MA50"] - 1
)

data["mom6"] = (
    data["close"] / data["close"].shift(6) - 1
)

data["mom12"] = (
    data["close"] / data["close"].shift(12) - 1
)

data["mom24"] = (
    data["close"] / data["close"].shift(24) - 1
)

data["price_vs_MA20"] = (
    data["close"] / data["MA20"] - 1
)

data["momentum_agreement"] = (
    (data["close"].pct_change(3) > 0).astype(int) +
    (data["mom6"] > 0).astype(int) +
    (data["mom12"] > 0).astype(int) +
    (data["mom24"] > 0).astype(int)
)

data["trend_agreement"] = (
    (data["close"] > data["MA10"]).astype(int) +
    (data["close"] > data["MA20"]).astype(int) +
    (data["close"] > data["MA50"]).astype(int)
)

# ---------------------------------------------------------
# Future outcomes
# ---------------------------------------------------------

data["future_6h"] = (
    data["close"].shift(-6) /
    data["close"] - 1
)

data["future_12h"] = (
    data["close"].shift(-12) /
    data["close"] - 1
)

data["future_24h"] = (
    data["close"].shift(-24) /
    data["close"] - 1
)

# ---------------------------------------------------------
# Remove incomplete rows
# ---------------------------------------------------------

required = [
    "vol10",
    "vol20",
    "atr_percent",
    "distance_high20",
    "relative_volume",
    "MA10",
    "MA20",
    "MA50",
    "mom6",
    "mom12",
    "mom24",
    "future_6h",
    "future_12h",
    "future_24h"
]

data = data.dropna(
    subset=required
).copy()

# ---------------------------------------------------------
# 4. Opportunity score
# ---------------------------------------------------------
#
# IMPORTANT:
# We calculate percentile ranks chronologically.
# Current observations are compared with information
# available up to that point.

opportunity_features = [
    "vol10",
    "vol20",
    "atr_percent",
    "distance_high20",
    "relative_volume"
]

for feature in opportunity_features:

    data[f"{feature}_rank"] = (
        data[feature]
        .expanding(min_periods=500)
        .rank(pct=True)
    )

data["opportunity_score"] = (
    data["vol10_rank"] +
    data["vol20_rank"] +
    data["atr_percent_rank"] +
    data["distance_high20_rank"] +
    data["relative_volume_rank"]
) / 5

data["opportunity_score_100"] = (
    data["opportunity_score"] * 100
)

# ---------------------------------------------------------
# 5. Define setup conditions
# ---------------------------------------------------------

HIGH_OPPORTUNITY = (
    data["opportunity_score_100"] >= 60
)

BULLISH_SETUP = (
    data["MA10_vs_MA20"] > 0
)

BEARISH_SETUP = (
    data["MA10_vs_MA20"] < 0
)

STRONG_BULLISH = (
    (data["MA10_vs_MA20"] > 0) &
    (data["MA20_vs_MA50"] > 0) &
    (data["mom6"] > 0)
)

STRONG_BEARISH = (
    (data["MA10_vs_MA20"] < 0) &
    (data["MA20_vs_MA50"] < 0) &
    (data["mom6"] < 0)
)

MIXED_BEARISH = (
    (data["MA10_vs_MA20"] < 0) &
    (data["MA20_vs_MA50"] < 0) &
    (data["mom6"] > 0)
)

MIXED_BULLISH = (
    (data["MA10_vs_MA20"] > 0) &
    (data["MA20_vs_MA50"] > 0) &
    (data["mom6"] < 0)
)

# ---------------------------------------------------------
# 6. Validator function
# ---------------------------------------------------------

def evaluate_setup(name, mask):

    subset = data[
        HIGH_OPPORTUNITY & mask
    ].copy()

    if len(subset) == 0:
        return {
            "setup": name,
            "signals": 0,
            "avg_6h": np.nan,
            "avg_12h": np.nan,
            "avg_24h": np.nan,
            "median_24h": np.nan,
            "win_rate_24h": np.nan,
            "big_move_rate": np.nan
        }

    return {
        "setup": name,
        "signals": len(subset),
        "avg_6h": subset["future_6h"].mean(),
        "avg_12h": subset["future_12h"].mean(),
        "avg_24h": subset["future_24h"].mean(),
        "median_24h": subset["future_24h"].median(),
        "win_rate_24h": (
            subset["future_24h"] > 0
        ).mean(),
        "big_move_rate": (
            subset["future_24h"].abs() >=
            data["future_24h"].abs().quantile(0.75)
        ).mean()
    }

# ---------------------------------------------------------
# 7. Evaluate historical setups
# ---------------------------------------------------------

setups = [
    ("BULLISH", BULLISH_SETUP),
    ("BEARISH", BEARISH_SETUP),
    ("STRONG_BULLISH", STRONG_BULLISH),
    ("STRONG_BEARISH", STRONG_BEARISH),
    ("MIXED_BEARISH", MIXED_BEARISH),
    ("MIXED_BULLISH", MIXED_BULLISH),
]

validation_results = []

for name, mask in setups:

    validation_results.append(
        evaluate_setup(name, mask)
    )

validation_df = pd.DataFrame(
    validation_results
)

# ---------------------------------------------------------
# 8. Display results
# ---------------------------------------------------------

print("=" * 60)
print("SETUP VALIDATOR V1")
print("=" * 60)

print(
    "\nHistorical setups restricted to "
    "Opportunity Score >= 60."
)

display(
    validation_df.style.format({
        "avg_6h": "{:.3%}",
        "avg_12h": "{:.3%}",
        "avg_24h": "{:.3%}",
        "median_24h": "{:.3%}",
        "win_rate_24h": "{:.1%}",
        "big_move_rate": "{:.1%}"
    })
)

# ---------------------------------------------------------
# 9. Current setup classification
# ---------------------------------------------------------

latest = data.iloc[-1]

current_score = latest["opportunity_score_100"]

if (
    latest["MA10_vs_MA20"] < 0
    and latest["MA20_vs_MA50"] < 0
    and latest["mom6"] > 0
):
    current_setup = "MIXED_BEARISH"

elif (
    latest["MA10_vs_MA20"] < 0
    and latest["MA20_vs_MA50"] < 0
    and latest["mom6"] < 0
):
    current_setup = "STRONG_BEARISH"

elif (
    latest["MA10_vs_MA20"] > 0
    and latest["MA20_vs_MA50"] > 0
    and latest["mom6"] > 0
):
    current_setup = "STRONG_BULLISH"

elif latest["MA10_vs_MA20"] > 0:
    current_setup = "BULLISH"

else:
    current_setup = "BEARISH"

print("\n" + "-" * 60)

print("CURRENT SETUP")
print("-" * 60)

print(
    f"Opportunity score: {current_score:.1f}/100"
)

print(
    f"Current setup:     {current_setup}"
)

print(
    f"MA10 vs MA20:      "
    f"{latest['MA10_vs_MA20'] * 100:+.3f}%"
)

print(
    f"MA20 vs MA50:      "
    f"{latest['MA20_vs_MA50'] * 100:+.3f}%"
)

print(
    f"6h momentum:       "
    f"{latest['mom6'] * 100:+.3f}%"
)

# ---------------------------------------------------------
# 10. Find matching historical evidence
# ---------------------------------------------------------

matching = validation_df[
    validation_df["setup"] == current_setup
]

if len(matching):

    row = matching.iloc[0]

    print("\nHistorical evidence for this setup:")

    print(
        f"  Signals:          "
        f"{int(row['signals'])}"
    )

    print(
        f"  Avg 24h outcome:   "
        f"{row['avg_24h']:.3%}"
    )

    print(
        f"  Median 24h:        "
        f"{row['median_24h']:.3%}"
    )

    print(
        f"  Win rate:          "
        f"{row['win_rate_24h']:.1%}"
    )

# ---------------------------------------------------------
# 11. Save validation results
# ---------------------------------------------------------

validation_path = (
    RESULTS_DIR +
    "/setup_validation.parquet"
)

validation_df.to_parquet(
    validation_path,
    engine="pyarrow",
    index=False
)

print("\n" + "=" * 60)
print("✅ SETUP VALIDATOR COMPLETE")
print(f"✅ Results saved to:")
print(validation_path)
print("=" * 60)

In [ ]:
# =========================================================
# SILVER AGENT — STEP 10
# DIRECTION ENGINE V2
# Evidence-based LONG / SHORT / WAIT
# =========================================================

import os
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Mount Drive
# ---------------------------------------------------------

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

DATABASE_PATH = (
    "/content/drive/MyDrive/"
    "SilverAgent/database/silver_h1.parquet"
)

RESULTS_DIR = (
    "/content/drive/MyDrive/"
    "SilverAgent/results"
)

os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------------------------------------------------------
# 2. Load persistent database
# ---------------------------------------------------------

df = pd.read_parquet(DATABASE_PATH).sort_index()

# ---------------------------------------------------------
# 3. Build directional features
# ---------------------------------------------------------

df["MA5"] = df["close"].rolling(5).mean()
df["MA10"] = df["close"].rolling(10).mean()
df["MA20"] = df["close"].rolling(20).mean()
df["MA50"] = df["close"].rolling(50).mean()

df["MA10_vs_MA20"] = (
    df["MA10"] / df["MA20"] - 1
)

df["MA20_vs_MA50"] = (
    df["MA20"] / df["MA50"] - 1
)

df["mom3"] = (
    df["close"] / df["close"].shift(3) - 1
)

df["mom6"] = (
    df["close"] / df["close"].shift(6) - 1
)

df["mom12"] = (
    df["close"] / df["close"].shift(12) - 1
)

df["mom24"] = (
    df["close"] / df["close"].shift(24) - 1
)

df["price_vs_MA20"] = (
    df["close"] / df["MA20"] - 1
)

# ---------------------------------------------------------
# 4. Build individual evidence components
# ---------------------------------------------------------
#
# V14d showed:
#
# LONG MA10 > MA20:
#   avg +0.2708%
#   win 57.17%
#
# SHORT MA10 < MA20:
#   avg -0.1951%
#   win 39.61%
#
# Therefore the model is intentionally ASYMMETRIC.
# A bearish MA10/20 relationship is not automatically
# converted into a strong short signal.

df["long_ma_signal"] = np.where(
    df["MA10_vs_MA20"] > 0,
    1,
    0
)

df["short_ma_signal"] = np.where(
    df["MA10_vs_MA20"] < 0,
    1,
    0
)

# Momentum evidence

df["long_mom6"] = np.where(
    df["mom6"] > 0,
    1,
    0
)

df["short_mom6"] = np.where(
    df["mom6"] < 0,
    1,
    0
)

df["long_mom12"] = np.where(
    df["mom12"] > 0,
    1,
    0
)

df["short_mom12"] = np.where(
    df["mom12"] < 0,
    1,
    0
)

# Longer-term trend confirmation

df["long_trend"] = np.where(
    df["MA20_vs_MA50"] > 0,
    1,
    0
)

df["short_trend"] = np.where(
    df["MA20_vs_MA50"] < 0,
    1,
    0
)

# Price location

df["long_price"] = np.where(
    df["price_vs_MA20"] > 0,
    1,
    0
)

df["short_price"] = np.where(
    df["price_vs_MA20"] < 0,
    1,
    0
)

# ---------------------------------------------------------
# 5. Evidence-weighted scores
# ---------------------------------------------------------
#
# MA10/20 receives the largest weight because it has the
# strongest evidence from our walk-forward direction tests.
#
# Short-side evidence is deliberately weaker.
#
# Maximum score = 100.

df["long_raw"] = (
    df["long_ma_signal"] * 40 +
    df["long_mom6"] * 20 +
    df["long_mom12"] * 10 +
    df["long_trend"] * 15 +
    df["long_price"] * 15
)

df["short_raw"] = (
    df["short_ma_signal"] * 25 +
    df["short_mom6"] * 20 +
    df["short_mom12"] * 10 +
    df["short_trend"] * 15 +
    df["short_price"] * 10
)

# ---------------------------------------------------------
# 6. Convert to scores
# ---------------------------------------------------------

df["long_score"] = df["long_raw"]
df["short_score"] = df["short_raw"]

# ---------------------------------------------------------
# 7. Direction decision
# ---------------------------------------------------------
#
# Require meaningful evidence before issuing a direction.
# Otherwise WAIT.

def direction_decision(row):

    long_score = row["long_score"]
    short_score = row["short_score"]

    difference = long_score - short_score

    # Strong LONG
    if (
        long_score >= 65
        and difference >= 20
    ):
        return "LONG"

    # Strong SHORT
    elif (
        short_score >= 60
        and difference <= -20
    ):
        return "SHORT"

    # Everything else
    else:
        return "WAIT"

df["direction"] = df.apply(
    direction_decision,
    axis=1
)

# ---------------------------------------------------------
# 8. Confidence
# ---------------------------------------------------------

def confidence(row):

    direction = row["direction"]

    if direction == "LONG":
        return min(
            100,
            row["long_score"]
        )

    elif direction == "SHORT":
        return min(
            100,
            row["short_score"]
        )

    return max(
        row["long_score"],
        row["short_score"]
    )

df["direction_confidence"] = df.apply(
    confidence,
    axis=1
)

# ---------------------------------------------------------
# 9. Current state
# ---------------------------------------------------------

latest = df.dropna(
    subset=[
        "MA10_vs_MA20",
        "MA20_vs_MA50",
        "mom6",
        "mom12",
        "price_vs_MA20"
    ]
).iloc[-1]

# ---------------------------------------------------------
# 10. Save history
# ---------------------------------------------------------

direction_history = df[
    [
        "long_score",
        "short_score",
        "direction",
        "direction_confidence",
        "MA10_vs_MA20",
        "MA20_vs_MA50",
        "mom6",
        "mom12",
        "mom24",
        "price_vs_MA20"
    ]
].copy()

direction_path = (
    RESULTS_DIR +
    "/direction_v2_history.parquet"
)

direction_history.to_parquet(
    direction_path,
    engine="pyarrow"
)

# ---------------------------------------------------------
# 11. Display current assessment
# ---------------------------------------------------------

print("=" * 60)
print("DIRECTION ENGINE V2")
print("=" * 60)

print("\nCURRENT MARKET")
print("-" * 60)

print(
    f"Price:              "
    f"{latest['close']:.4f}"
)

print(
    f"MA10 vs MA20:       "
    f"{latest['MA10_vs_MA20'] * 100:+.3f}%"
)

print(
    f"MA20 vs MA50:       "
    f"{latest['MA20_vs_MA50'] * 100:+.3f}%"
)

print(
    f"6h momentum:        "
    f"{latest['mom6'] * 100:+.3f}%"
)

print(
    f"12h momentum:       "
    f"{latest['mom12'] * 100:+.3f}%"
)

print(
    f"24h momentum:       "
    f"{latest['mom24'] * 100:+.3f}%"
)

print(
    f"Price vs MA20:      "
    f"{latest['price_vs_MA20'] * 100:+.3f}%"
)

print("\nEVIDENCE SCORES")
print("-" * 60)

print(
    f"LONG score:         "
    f"{latest['long_score']:.1f}/100"
)

print(
    f"SHORT score:        "
    f"{latest['short_score']:.1f}/100"
)

print(
    f"Score difference:   "
    f"{latest['long_score'] - latest['short_score']:+.1f}"
)

print("\nDECISION")
print("-" * 60)

print(
    f"Direction:          "
    f"{latest['direction']}"
)

print(
    f"Confidence:         "
    f"{latest['direction_confidence']:.1f}/100"
)

# ---------------------------------------------------------
# 12. Historical distribution
# ---------------------------------------------------------

print("\nHISTORICAL DECISIONS")
print("-" * 60)

decision_counts = (
    df["direction"]
    .value_counts()
)

total_decisions = (
    decision_counts.sum()
)

for decision in ["LONG", "SHORT", "WAIT"]:

    count = decision_counts.get(
        decision,
        0
    )

    pct = (
        count / total_decisions * 100
    )

    print(
        f"{decision:6s}: "
        f"{count:6d} "
        f"({pct:5.1f}%)"
    )

print("\n" + "=" * 60)
print("✅ DIRECTION ENGINE V2 COMPLETE")
print(f"✅ Saved to:")
print(direction_path)
print("=" * 60)

In [ ]:
# =========================================================
# SILVER AGENT — STEP 11
# SIGNAL AGREEMENT ENGINE V1
# Tests Opportunity + Direction together
# =========================================================

import os
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Mount Drive
# ---------------------------------------------------------

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

DATABASE_PATH = (
    "/content/drive/MyDrive/"
    "SilverAgent/database/silver_h1.parquet"
)

RESULTS_DIR = (
    "/content/drive/MyDrive/"
    "SilverAgent/results"
)

os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------------------------------------------------------
# 2. Load database
# ---------------------------------------------------------

df = pd.read_parquet(
    DATABASE_PATH
).sort_index()

# ---------------------------------------------------------
# 3. Rebuild required features
# ---------------------------------------------------------

df["MA10"] = df["close"].rolling(10).mean()
df["MA20"] = df["close"].rolling(20).mean()
df["MA50"] = df["close"].rolling(50).mean()

df["MA10_vs_MA20"] = (
    df["MA10"] / df["MA20"] - 1
)

df["MA20_vs_MA50"] = (
    df["MA20"] / df["MA50"] - 1
)

df["mom3"] = (
    df["close"] / df["close"].shift(3) - 1
)

df["mom6"] = (
    df["close"] / df["close"].shift(6) - 1
)

df["mom12"] = (
    df["close"] / df["close"].shift(12) - 1
)

df["mom24"] = (
    df["close"] / df["close"].shift(24) - 1
)

df["price_vs_MA20"] = (
    df["close"] / df["MA20"] - 1
)

# ---------------------------------------------------------
# 4. Opportunity features
# ---------------------------------------------------------

df["return_1h"] = (
    df["close"] / df["close"].shift(1) - 1
)

df["vol10"] = (
    df["return_1h"].rolling(10).std()
)

df["vol20"] = (
    df["return_1h"].rolling(20).std()
)

previous_close = df["close"].shift(1)

tr1 = df["high"] - df["low"]
tr2 = (df["high"] - previous_close).abs()
tr3 = (df["low"] - previous_close).abs()

df["true_range"] = pd.concat(
    [tr1, tr2, tr3],
    axis=1
).max(axis=1)

df["ATR20"] = (
    df["true_range"].rolling(20).mean()
)

df["atr_percent"] = (
    df["ATR20"] / df["close"]
)

df["high20"] = (
    df["high"].rolling(20).max()
)

df["distance_high20"] = (
    df["high20"] / df["close"] - 1
)

df["volume20"] = (
    df["volume"].rolling(20).mean()
)

df["relative_volume"] = np.where(
    df["volume20"] > 0,
    df["volume"] / df["volume20"],
    0
)

# ---------------------------------------------------------
# 5. Future outcomes
# ---------------------------------------------------------

df["future_6h"] = (
    df["close"].shift(-6) /
    df["close"] - 1
)

df["future_12h"] = (
    df["close"].shift(-12) /
    df["close"] - 1
)

df["future_24h"] = (
    df["close"].shift(-24) /
    df["close"] - 1
)

# ---------------------------------------------------------
# 6. Opportunity score
# ---------------------------------------------------------

opportunity_features = [
    "vol10",
    "vol20",
    "atr_percent",
    "distance_high20",
    "relative_volume"
]

for feature in opportunity_features:

    df[f"{feature}_rank"] = (
        df[feature]
        .expanding(min_periods=500)
        .rank(pct=True)
    )

df["opportunity_score"] = (
    df["vol10_rank"] +
    df["vol20_rank"] +
    df["atr_percent_rank"] +
    df["distance_high20_rank"] +
    df["relative_volume_rank"]
) / 5 * 100

# ---------------------------------------------------------
# 7. Direction V2
# ---------------------------------------------------------

df["long_score"] = (
    (df["MA10_vs_MA20"] > 0).astype(int) * 40 +
    (df["mom6"] > 0).astype(int) * 20 +
    (df["mom12"] > 0).astype(int) * 10 +
    (df["MA20_vs_MA50"] > 0).astype(int) * 15 +
    (df["price_vs_MA20"] > 0).astype(int) * 15
)

df["short_score"] = (
    (df["MA10_vs_MA20"] < 0).astype(int) * 25 +
    (df["mom6"] < 0).astype(int) * 20 +
    (df["mom12"] < 0).astype(int) * 10 +
    (df["MA20_vs_MA50"] < 0).astype(int) * 15 +
    (df["price_vs_MA20"] < 0).astype(int) * 10
)

df["direction"] = "WAIT"

long_condition = (
    (df["long_score"] >= 65) &
    ((df["long_score"] - df["short_score"]) >= 20)
)

short_condition = (
    (df["short_score"] >= 60) &
    ((df["short_score"] - df["long_score"]) >= 20)
)

df.loc[long_condition, "direction"] = "LONG"
df.loc[short_condition, "direction"] = "SHORT"

# ---------------------------------------------------------
# 8. Setup classification
# ---------------------------------------------------------

df["setup"] = "OTHER"

df.loc[
    (df["MA10_vs_MA20"] > 0),
    "setup"
] = "BULLISH"

df.loc[
    (df["MA10_vs_MA20"] < 0),
    "setup"
] = "BEARISH"

df.loc[
    (df["MA10_vs_MA20"] < 0) &
    (df["MA20_vs_MA50"] < 0) &
    (df["mom6"] > 0),
    "setup"
] = "MIXED_BEARISH"

df.loc[
    (df["MA10_vs_MA20"] > 0) &
    (df["MA20_vs_MA50"] > 0) &
    (df["mom6"] < 0),
    "setup"
] = "MIXED_BULLISH"

df.loc[
    (df["MA10_vs_MA20"] < 0) &
    (df["MA20_vs_MA50"] < 0) &
    (df["mom6"] < 0),
    "setup"
] = "STRONG_BEARISH"

df.loc[
    (df["MA10_vs_MA20"] > 0) &
    (df["MA20_vs_MA50"] > 0) &
    (df["mom6"] > 0),
    "setup"
] = "STRONG_BULLISH"

# ---------------------------------------------------------
# 9. Define exact signal combinations
# ---------------------------------------------------------

high_opportunity = (
    df["opportunity_score"] >= 60
)

signals = {

    "HIGH_OPP_LONG":
        high_opportunity &
        (df["direction"] == "LONG"),

    "HIGH_OPP_SHORT":
        high_opportunity &
        (df["direction"] == "SHORT"),

    "MIXED_BEARISH_SHORT":
        high_opportunity &
        (df["direction"] == "SHORT") &
        (df["setup"] == "MIXED_BEARISH"),

    "MIXED_BULLISH_LONG":
        high_opportunity &
        (df["direction"] == "LONG") &
        (df["setup"] == "MIXED_BULLISH"),

    "STRONG_BEARISH_SHORT":
        high_opportunity &
        (df["direction"] == "SHORT") &
        (df["setup"] == "STRONG_BEARISH"),

    "STRONG_BULLISH_LONG":
        high_opportunity &
        (df["direction"] == "LONG") &
        (df["setup"] == "STRONG_BULLISH")
}

# ---------------------------------------------------------
# 10. Evaluate signals
# ---------------------------------------------------------

def evaluate_signal(name, mask):

    subset = df.loc[
        mask &
        df["future_24h"].notna()
    ].copy()

    if len(subset) == 0:
        return {
            "signal": name,
            "signals": 0,
            "avg_6h": np.nan,
            "avg_12h": np.nan,
            "avg_24h": np.nan,
            "median_24h": np.nan,
            "win_rate": np.nan,
            "best": np.nan,
            "worst": np.nan
        }

    return {
        "signal": name,
        "signals": len(subset),

        "avg_6h":
            subset["future_6h"].mean(),

        "avg_12h":
            subset["future_12h"].mean(),

        "avg_24h":
            subset["future_24h"].mean(),

        "median_24h":
            subset["future_24h"].median(),

        "win_rate":
            (subset["future_24h"] > 0).mean(),

        "best":
            subset["future_24h"].max(),

        "worst":
            subset["future_24h"].min()
    }

results = []

for name, mask in signals.items():

    results.append(
        evaluate_signal(
            name,
            mask
        )
    )

results_df = pd.DataFrame(results)

# ---------------------------------------------------------
# 11. For SHORT signals, calculate SHORT profitability
# ---------------------------------------------------------
#
# If price falls, a SHORT benefits.
#
# This is only an analytical transformation.
# No trades are being placed.

for idx, row in results_df.iterrows():

    name = row["signal"]

    if "SHORT" in name:

        mask = signals[name]

        subset = df.loc[
            mask &
            df["future_24h"].notna()
        ]

        if len(subset):

            short_returns = (
                -subset["future_24h"]
            )

            results_df.loc[
                idx,
                "short_avg_24h"
            ] = short_returns.mean()

            results_df.loc[
                idx,
                "short_win_rate"
            ] = (
                short_returns > 0
            ).mean()

# ---------------------------------------------------------
# 12. Display
# ---------------------------------------------------------

print("=" * 70)
print("SIGNAL AGREEMENT ENGINE V1")
print("=" * 70)

print(
    "\nTesting combinations of:"
)

print("  Opportunity >= 60")
print("  Direction V2")
print("  Market setup")

display(
    results_df.style.format({
        "avg_6h": "{:.3%}",
        "avg_12h": "{:.3%}",
        "avg_24h": "{:.3%}",
        "median_24h": "{:.3%}",
        "win_rate": "{:.1%}",
        "best": "{:.2%}",
        "worst": "{:.2%}",
        "short_avg_24h": "{:.3%}",
        "short_win_rate": "{:.1%}"
    })
)

# ---------------------------------------------------------
# 13. Current machine state
# ---------------------------------------------------------

latest = df.dropna(
    subset=[
        "opportunity_score",
        "long_score",
        "short_score"
    ]
).iloc[-1]

print("\n" + "=" * 70)
print("CURRENT MACHINE STATE")
print("=" * 70)

print(
    f"\nOpportunity: "
    f"{latest['opportunity_score']:.1f}/100"
)

print(
    f"LONG score: "
    f"{latest['long_score']:.1f}/100"
)

print(
    f"SHORT score: "
    f"{latest['short_score']:.1f}/100"
)

print(
    f"Direction: "
    f"{latest['direction']}"
)

print(
    f"Setup: "
    f"{latest['setup']}"
)

# ---------------------------------------------------------
# 14. Exact current signal
# ---------------------------------------------------------

current_signal = None

if (
    latest["opportunity_score"] >= 60 and
    latest["direction"] == "SHORT" and
    latest["setup"] == "MIXED_BEARISH"
):

    current_signal = "MIXED_BEARISH_SHORT"

elif (
    latest["opportunity_score"] >= 60 and
    latest["direction"] == "LONG" and
    latest["setup"] == "MIXED_BULLISH"
):

    current_signal = "MIXED_BULLISH_LONG"

elif (
    latest["opportunity_score"] >= 60 and
    latest["direction"] == "SHORT"
):

    current_signal = "HIGH_OPP_SHORT"

elif (
    latest["opportunity_score"] >= 60 and
    latest["direction"] == "LONG"
):

    current_signal = "HIGH_OPP_LONG"

else:

    current_signal = "NO_HIGH_OPPORTUNITY_SIGNAL"

print(
    f"\nExact current signal:"
    f" {current_signal}"
)

# ---------------------------------------------------------
# 15. Historical evidence for current signal
# ---------------------------------------------------------

current_result = results_df[
    results_df["signal"] == current_signal
]

if len(current_result):

    r = current_result.iloc[0]

    print("\nHistorical evidence:")

    print(
        f"  Historical signals: "
        f"{int(r['signals'])}"
    )

    print(
        f"  Average 24h:        "
        f"{r['avg_24h']:.3%}"
    )

    print(
        f"  Median 24h:         "
        f"{r['median_24h']:.3%}"
    )

    print(
        f"  Long win rate:      "
        f"{r['win_rate']:.1%}"
    )

    if "SHORT" in current_signal:

        print(
            f"  SHORT avg 24h:      "
            f"{r['short_avg_24h']:.3%}"
        )

        print(
            f"  SHORT win rate:     "
            f"{r['short_win_rate']:.1%}"
        )

# ---------------------------------------------------------
# 16. Save
# ---------------------------------------------------------

output_path = (
    RESULTS_DIR +
    "/signal_agreement_v1.parquet"
)

results_df.to_parquet(
    output_path,
    engine="pyarrow",
    index=False
)

print("\n" + "=" * 70)
print("✅ SIGNAL AGREEMENT ENGINE COMPLETE")
print(f"✅ Saved to:")
print(output_path)
print("=" * 70)

In [ ]:
# =========================================================
# SILVER AGENT — STEP 12
# WALK-FORWARD SIGNAL VALIDATION V1
# =========================================================

import os
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Mount Drive
# ---------------------------------------------------------

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

DATABASE_PATH = (
    "/content/drive/MyDrive/"
    "SilverAgent/database/silver_h1.parquet"
)

RESULTS_DIR = (
    "/content/drive/MyDrive/"
    "SilverAgent/results"
)

os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------------------------------------------------------
# 2. Load database
# ---------------------------------------------------------

df = pd.read_parquet(
    DATABASE_PATH
).sort_index()

# ---------------------------------------------------------
# 3. Features
# ---------------------------------------------------------

df["MA10"] = df["close"].rolling(10).mean()
df["MA20"] = df["close"].rolling(20).mean()
df["MA50"] = df["close"].rolling(50).mean()

df["MA10_vs_MA20"] = (
    df["MA10"] / df["MA20"] - 1
)

df["MA20_vs_MA50"] = (
    df["MA20"] / df["MA50"] - 1
)

df["mom6"] = (
    df["close"] / df["close"].shift(6) - 1
)

df["mom12"] = (
    df["close"] / df["close"].shift(12) - 1
)

df["mom24"] = (
    df["close"] / df["close"].shift(24) - 1
)

df["price_vs_MA20"] = (
    df["close"] / df["MA20"] - 1
)

df["return_1h"] = (
    df["close"] / df["close"].shift(1) - 1
)

df["vol10"] = (
    df["return_1h"].rolling(10).std()
)

df["vol20"] = (
    df["return_1h"].rolling(20).std()
)

previous_close = df["close"].shift(1)

tr1 = df["high"] - df["low"]
tr2 = (df["high"] - previous_close).abs()
tr3 = (df["low"] - previous_close).abs()

df["true_range"] = pd.concat(
    [tr1, tr2, tr3],
    axis=1
).max(axis=1)

df["ATR20"] = (
    df["true_range"].rolling(20).mean()
)

df["atr_percent"] = (
    df["ATR20"] / df["close"]
)

df["high20"] = (
    df["high"].rolling(20).max()
)

df["distance_high20"] = (
    df["high20"] / df["close"] - 1
)

df["volume20"] = (
    df["volume"].rolling(20).mean()
)

df["relative_volume"] = np.where(
    df["volume20"] > 0,
    df["volume"] / df["volume20"],
    0
)

# ---------------------------------------------------------
# 4. Future return
# ---------------------------------------------------------

df["future_24h"] = (
    df["close"].shift(-24) /
    df["close"] - 1
)

# ---------------------------------------------------------
# 5. Opportunity score
# ---------------------------------------------------------

opp_features = [
    "vol10",
    "vol20",
    "atr_percent",
    "distance_high20",
    "relative_volume"
]

for feature in opp_features:

    df[f"{feature}_rank"] = (
        df[feature]
        .expanding(min_periods=500)
        .rank(pct=True)
    )

df["opportunity_score"] = (
    df["vol10_rank"] +
    df["vol20_rank"] +
    df["atr_percent_rank"] +
    df["distance_high20_rank"] +
    df["relative_volume_rank"]
) / 5 * 100

# ---------------------------------------------------------
# 6. Direction V2
# ---------------------------------------------------------

df["long_score"] = (
    (df["MA10_vs_MA20"] > 0).astype(int) * 40 +
    (df["mom6"] > 0).astype(int) * 20 +
    (df["mom12"] > 0).astype(int) * 10 +
    (df["MA20_vs_MA50"] > 0).astype(int) * 15 +
    (df["price_vs_MA20"] > 0).astype(int) * 15
)

df["short_score"] = (
    (df["MA10_vs_MA20"] < 0).astype(int) * 25 +
    (df["mom6"] < 0).astype(int) * 20 +
    (df["mom12"] < 0).astype(int) * 10 +
    (df["MA20_vs_MA50"] < 0).astype(int) * 15 +
    (df["price_vs_MA20"] < 0).astype(int) * 10
)

df["direction"] = "WAIT"

df.loc[
    (df["long_score"] >= 65) &
    ((df["long_score"] - df["short_score"]) >= 20),
    "direction"
] = "LONG"

df.loc[
    (df["short_score"] >= 60) &
    ((df["short_score"] - df["long_score"]) >= 20),
    "direction"
] = "SHORT"

# ---------------------------------------------------------
# 7. Setup classification
# ---------------------------------------------------------

df["setup"] = "OTHER"

df.loc[
    (df["MA10_vs_MA20"] > 0),
    "setup"
] = "BULLISH"

df.loc[
    (df["MA10_vs_MA20"] < 0),
    "setup"
] = "BEARISH"

df.loc[
    (df["MA10_vs_MA20"] < 0) &
    (df["MA20_vs_MA50"] < 0) &
    (df["mom6"] > 0),
    "setup"
] = "MIXED_BEARISH"

df.loc[
    (df["MA10_vs_MA20"] > 0) &
    (df["MA20_vs_MA50"] > 0) &
    (df["mom6"] < 0),
    "setup"
] = "MIXED_BULLISH"

df.loc[
    (df["MA10_vs_MA20"] < 0) &
    (df["MA20_vs_MA50"] < 0) &
    (df["mom6"] < 0),
    "setup"
] = "STRONG_BEARISH"

df.loc[
    (df["MA10_vs_MA20"] > 0) &
    (df["MA20_vs_MA50"] > 0) &
    (df["mom6"] > 0),
    "setup"
] = "STRONG_BULLISH"

# ---------------------------------------------------------
# 8. Signal definitions
# ---------------------------------------------------------

high_opp = (
    df["opportunity_score"] >= 60
)

signals = {

    "HIGH_OPP_LONG":
        high_opp &
        (df["direction"] == "LONG"),

    "HIGH_OPP_SHORT":
        high_opp &
        (df["direction"] == "SHORT"),

    "MIXED_BEARISH_SHORT":
        high_opp &
        (df["direction"] == "SHORT") &
        (df["setup"] == "MIXED_BEARISH"),

    "MIXED_BULLISH_LONG":
        high_opp &
        (df["direction"] == "LONG") &
        (df["setup"] == "MIXED_BULLISH"),

    "STRONG_BEARISH_SHORT":
        high_opp &
        (df["direction"] == "SHORT") &
        (df["setup"] == "STRONG_BEARISH"),

    "STRONG_BULLISH_LONG":
        high_opp &
        (df["direction"] == "LONG") &
        (df["setup"] == "STRONG_BULLISH")
}

# ---------------------------------------------------------
# 9. Walk-forward configuration
# ---------------------------------------------------------
#
# Fixed windows chosen BEFORE seeing results.
#
# Train: 4000 hours
# Test: 1000 hours
# Step: 1000 hours
#
# We do not optimise these numbers after seeing results.

TRAIN_SIZE = 4000
TEST_SIZE = 1000
STEP_SIZE = 1000

# ---------------------------------------------------------
# 10. Walk-forward testing
# ---------------------------------------------------------

all_window_results = []

n = len(df)

window_number = 0

for start in range(
    0,
    n - TRAIN_SIZE - TEST_SIZE + 1,
    STEP_SIZE
):

    train_start = start
    train_end = start + TRAIN_SIZE

    test_start = train_end
    test_end = train_end + TEST_SIZE

    test = df.iloc[
        test_start:test_end
    ].copy()

    window_number += 1

    for signal_name, mask in signals.items():

        # Apply signal to TEST only.
        #
        # The rules are frozen.
        # No information from the test window
        # is used to change them.

        test_signal = test.loc[
            mask.iloc[test_start:test_end].values
        ].copy()

        test_signal = test_signal[
            test_signal["future_24h"].notna()
        ]

        count = len(test_signal)

        if count == 0:
            continue

        raw_return = (
            test_signal["future_24h"]
        )

        # Direction-adjusted return
        if "SHORT" in signal_name:

            strategy_return = -raw_return

        else:

            strategy_return = raw_return

        all_window_results.append({

            "window": window_number,

            "test_start":
                test.index.min(),

            "test_end":
                test.index.max(),

            "signal":
                signal_name,

            "signals":
                count,

            "avg_return":
                strategy_return.mean(),

            "median_return":
                strategy_return.median(),

            "win_rate":
                (strategy_return > 0).mean(),

            "best":
                strategy_return.max(),

            "worst":
                strategy_return.min(),

            "total_return":
                strategy_return.sum()

        })

# ---------------------------------------------------------
# 11. Results dataframe
# ---------------------------------------------------------

wf = pd.DataFrame(
    all_window_results
)

# ---------------------------------------------------------
# 12. Aggregate results
# ---------------------------------------------------------

summary = []

for signal_name in signals.keys():

    subset = wf[
        wf["signal"] == signal_name
    ]

    if len(subset) == 0:
        continue

    summary.append({

        "signal":
            signal_name,

        "windows":
            len(subset),

        "total_signals":
            subset["signals"].sum(),

        "avg_window_return":
            subset["avg_return"].mean(),

        "weighted_avg_return":
            np.average(
                subset["avg_return"],
                weights=subset["signals"]
            ),

        "median_window_return":
            subset["avg_return"].median(),

        "avg_win_rate":
            np.average(
                subset["win_rate"],
                weights=subset["signals"]
            ),

        "positive_windows":
            (subset["avg_return"] > 0).sum(),

        "positive_window_rate":
            (subset["avg_return"] > 0).mean(),

        "worst_window":
            subset["avg_return"].min(),

        "best_window":
            subset["avg_return"].max()
    })

summary_df = pd.DataFrame(
    summary
)

# ---------------------------------------------------------
# 13. Display
# ---------------------------------------------------------

print("=" * 75)
print("WALK-FORWARD SIGNAL VALIDATION V1")
print("=" * 75)

print(
    "\nFixed protocol:"
)

print(
    f"Training window: "
    f"{TRAIN_SIZE:,} H1 candles"
)

print(
    f"Test window:     "
    f"{TEST_SIZE:,} H1 candles"
)

print(
    f"Step:             "
    f"{STEP_SIZE:,} H1 candles"
)

print(
    "\nEvery reported result below comes "
    "from unseen test windows."
)

display(
    summary_df.style.format({

        "avg_window_return":
            "{:.3%}",

        "weighted_avg_return":
            "{:.3%}",

        "median_window_return":
            "{:.3%}",

        "avg_win_rate":
            "{:.1%}",

        "positive_window_rate":
            "{:.1%}",

        "worst_window":
            "{:.3%}",

        "best_window":
            "{:.3%}"
    })
)

# ---------------------------------------------------------
# 14. Current signal
# ---------------------------------------------------------

latest = df.dropna(
    subset=[
        "opportunity_score",
        "future_24h"
    ]
).iloc[-25]

current = df.dropna(
    subset=[
        "opportunity_score"
    ]
).iloc[-1]

print("\n" + "=" * 75)
print("CURRENT SIGNAL")
print("=" * 75)

print(
    f"\nOpportunity: "
    f"{current['opportunity_score']:.1f}/100"
)

print(
    f"Direction: "
    f"{current['direction']}"
)

print(
    f"Setup: "
    f"{current['setup']}"
)

# ---------------------------------------------------------
# 15. Save results
# ---------------------------------------------------------

wf_path = (
    RESULTS_DIR +
    "/walk_forward_signal_results.parquet"
)

summary_path = (
    RESULTS_DIR +
    "/walk_forward_signal_summary.parquet"
)

wf.to_parquet(
    wf_path,
    engine="pyarrow",
    index=False
)

summary_df.to_parquet(
    summary_path,
    engine="pyarrow",
    index=False
)

print("\n" + "=" * 75)
print("✅ WALK-FORWARD VALIDATION COMPLETE")
print("=" * 75)

print(
    f"\nDetailed results:"
)

print(wf_path)

print(
    "\nSummary:"
)

print(summary_path)

In [ ]:
# =========================================================
# SILVER AGENT — STEP 13
# LONG OPPORTUNITY DETECTOR V1
# Fixed hypotheses — no optimisation
# =========================================================

import os
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Mount Drive
# ---------------------------------------------------------

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

DATABASE_PATH = (
    "/content/drive/MyDrive/"
    "SilverAgent/database/silver_h1.parquet"
)

RESULTS_DIR = (
    "/content/drive/MyDrive/"
    "SilverAgent/results"
)

os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------------------------------------------------------
# 2. Load database
# ---------------------------------------------------------

df = pd.read_parquet(
    DATABASE_PATH
).sort_index()

# ---------------------------------------------------------
# 3. Features
# ---------------------------------------------------------

df["MA10"] = df["close"].rolling(10).mean()
df["MA20"] = df["close"].rolling(20).mean()
df["MA50"] = df["close"].rolling(50).mean()

df["MA10_vs_MA20"] = (
    df["MA10"] / df["MA20"] - 1
)

df["MA20_vs_MA50"] = (
    df["MA20"] / df["MA50"] - 1
)

df["mom6"] = (
    df["close"] / df["close"].shift(6) - 1
)

df["mom12"] = (
    df["close"] / df["close"].shift(12) - 1
)

df["mom24"] = (
    df["close"] / df["close"].shift(24) - 1
)

df["price_vs_MA20"] = (
    df["close"] / df["MA20"] - 1
)

df["return_1h"] = (
    df["close"] / df["close"].shift(1) - 1
)

df["vol10"] = (
    df["return_1h"].rolling(10).std()
)

df["vol20"] = (
    df["return_1h"].rolling(20).std()
)

previous_close = df["close"].shift(1)

tr1 = df["high"] - df["low"]
tr2 = (df["high"] - previous_close).abs()
tr3 = (df["low"] - previous_close).abs()

df["true_range"] = pd.concat(
    [tr1, tr2, tr3],
    axis=1
).max(axis=1)

df["ATR20"] = (
    df["true_range"].rolling(20).mean()
)

df["atr_percent"] = (
    df["ATR20"] / df["close"]
)

df["high20"] = (
    df["high"].rolling(20).max()
)

df["distance_high20"] = (
    df["high20"] / df["close"] - 1
)

df["volume20"] = (
    df["volume"].rolling(20).mean()
)

df["relative_volume"] = np.where(
    df["volume20"] > 0,
    df["volume"] / df["volume20"],
    0
)

# ---------------------------------------------------------
# 4. Future outcomes
# ---------------------------------------------------------

df["future_6h"] = (
    df["close"].shift(-6) /
    df["close"] - 1
)

df["future_12h"] = (
    df["close"].shift(-12) /
    df["close"] - 1
)

df["future_24h"] = (
    df["close"].shift(-24) /
    df["close"] - 1
)

# ---------------------------------------------------------
# 5. Opportunity score
# ---------------------------------------------------------

opp_features = [
    "vol10",
    "vol20",
    "atr_percent",
    "distance_high20",
    "relative_volume"
]

for feature in opp_features:

    df[f"{feature}_rank"] = (
        df[feature]
        .expanding(min_periods=500)
        .rank(pct=True)
    )

df["opportunity_score"] = (
    df["vol10_rank"] +
    df["vol20_rank"] +
    df["atr_percent_rank"] +
    df["distance_high20_rank"] +
    df["relative_volume_rank"]
) / 5 * 100

# ---------------------------------------------------------
# 6. Fixed HIGH opportunity definition
# ---------------------------------------------------------

high_opportunity = (
    df["opportunity_score"] >= 60
)

# ---------------------------------------------------------
# 7. Define FIXED hypotheses
# ---------------------------------------------------------
#
# These are specified BEFORE looking at the new results.
# We do not optimise thresholds here.

signals = {

    # Hypothesis A
    "LONG_MA10_MA20":
        high_opportunity &
        (df["MA10_vs_MA20"] > 0),

    # Hypothesis B
    "LONG_MA10_MA20_MOM6":
        high_opportunity &
        (df["MA10_vs_MA20"] > 0) &
        (df["mom6"] > 0),

    # Hypothesis C
    "LONG_MA10_MA20_MOM6_MOM12":
        high_opportunity &
        (df["MA10_vs_MA20"] > 0) &
        (df["mom6"] > 0) &
        (df["mom12"] > 0),

    # Hypothesis D
    "LONG_MIXED_BULLISH":
        high_opportunity &
        (df["MA10_vs_MA20"] > 0) &
        (df["MA20_vs_MA50"] > 0) &
        (df["mom6"] < 0)
}

# ---------------------------------------------------------
# 8. Fixed walk-forward protocol
# ---------------------------------------------------------

TRAIN_SIZE = 4000
TEST_SIZE = 1000
STEP_SIZE = 1000

wf_results = []

n = len(df)

window_number = 0

# ---------------------------------------------------------
# 9. Walk-forward testing
# ---------------------------------------------------------

for start in range(
    0,
    n - TRAIN_SIZE - TEST_SIZE + 1,
    STEP_SIZE
):

    train_start = start
    train_end = start + TRAIN_SIZE

    test_start = train_end
    test_end = train_end + TEST_SIZE

    test = df.iloc[
        test_start:test_end
    ].copy()

    window_number += 1

    for signal_name, signal_mask in signals.items():

        # IMPORTANT:
        # Only the unseen TEST window is evaluated.

        test_mask = signal_mask.iloc[
            test_start:test_end
        ]

        subset = test.loc[
            test_mask.values
        ].copy()

        subset = subset[
            subset["future_24h"].notna()
        ]

        if len(subset) == 0:
            continue

        returns = subset["future_24h"]

        wf_results.append({

            "window":
                window_number,

            "signal":
                signal_name,

            "test_start":
                subset.index.min(),

            "test_end":
                subset.index.max(),

            "signals":
                len(subset),

            "avg_6h":
                subset["future_6h"].mean(),

            "avg_12h":
                subset["future_12h"].mean(),

            "avg_24h":
                returns.mean(),

            "median_24h":
                returns.median(),

            "win_rate":
                (returns > 0).mean(),

            "best":
                returns.max(),

            "worst":
                returns.min()
        })

# ---------------------------------------------------------
# 10. Results
# ---------------------------------------------------------

wf = pd.DataFrame(
    wf_results
)

summary = []

for signal_name in signals.keys():

    subset = wf[
        wf["signal"] == signal_name
    ]

    if len(subset) == 0:
        continue

    summary.append({

        "signal":
            signal_name,

        "windows":
            len(subset),

        "signals":
            subset["signals"].sum(),

        "weighted_avg_24h":
            np.average(
                subset["avg_24h"],
                weights=subset["signals"]
            ),

        "median_window_avg":
            subset["avg_24h"].median(),

        "weighted_win_rate":
            np.average(
                subset["win_rate"],
                weights=subset["signals"]
            ),

        "positive_windows":
            (
                subset["avg_24h"] > 0
            ).sum(),

        "positive_window_rate":
            (
                subset["avg_24h"] > 0
            ).mean(),

        "worst_window":
            subset["avg_24h"].min(),

        "best_window":
            subset["avg_24h"].max()
    })

summary_df = pd.DataFrame(
    summary
)

# ---------------------------------------------------------
# 11. Display
# ---------------------------------------------------------

print("=" * 75)
print("LONG OPPORTUNITY DETECTOR V1")
print("=" * 75)

print(
    "\nFIXED HYPOTHESES"
)

print(
    "No parameters were optimised using these test results."
)

print(
    "\nWalk-forward protocol:"
)

print(
    f"  Train: {TRAIN_SIZE:,} H1 candles"
)

print(
    f"  Test:  {TEST_SIZE:,} H1 candles"
)

print(
    f"  Step:  {STEP_SIZE:,} H1 candles"
)

display(
    summary_df.style.format({

        "weighted_avg_24h":
            "{:.3%}",

        "median_window_avg":
            "{:.3%}",

        "weighted_win_rate":
            "{:.1%}",

        "positive_window_rate":
            "{:.1%}",

        "worst_window":
            "{:.3%}",

        "best_window":
            "{:.3%}"
    })
)

# ---------------------------------------------------------
# 12. Window-by-window results
# ---------------------------------------------------------

print("\n" + "=" * 75)
print("WINDOW-BY-WINDOW RESULTS")
print("=" * 75)

display(
    wf.sort_values(
        ["signal", "window"]
    ).style.format({

        "avg_6h":
            "{:.3%}",

        "avg_12h":
            "{:.3%}",

        "avg_24h":
            "{:.3%}",

        "median_24h":
            "{:.3%}",

        "win_rate":
            "{:.1%}",

        "best":
            "{:.2%}",

        "worst":
            "{:.2%}"
    })
)

# ---------------------------------------------------------
# 13. Current state
# ---------------------------------------------------------

latest = df.dropna(
    subset=[
        "opportunity_score",
        "MA10_vs_MA20",
        "MA20_vs_MA50",
        "mom6",
        "mom12"
    ]
).iloc[-1]

print("\n" + "=" * 75)
print("CURRENT MARKET")
print("=" * 75)

print(
    f"\nPrice: "
    f"{latest['close']:.4f}"
)

print(
    f"Opportunity: "
    f"{latest['opportunity_score']:.1f}/100"
)

print(
    f"MA10 vs MA20: "
    f"{latest['MA10_vs_MA20'] * 100:+.3f}%"
)

print(
    f"MA20 vs MA50: "
    f"{latest['MA20_vs_MA50'] * 100:+.3f}%"
)

print(
    f"6h momentum: "
    f"{latest['mom6'] * 100:+.3f}%"
)

print(
    f"12h momentum: "
    f"{latest['mom12'] * 100:+.3f}%"
)

# ---------------------------------------------------------
# 14. Current candidate status
# ---------------------------------------------------------

current_candidates = []

for name, mask in signals.items():

    if bool(mask.iloc[-1]):
        current_candidates.append(name)

print(
    "\nCurrent LONG candidates:"
)

if current_candidates:

    for candidate in current_candidates:
        print(
            f"  🟢 {candidate}"
        )

else:

    print(
        "  None"
    )

# ---------------------------------------------------------
# 15. Save
# ---------------------------------------------------------

detail_path = (
    RESULTS_DIR +
    "/long_detector_v1_windows.parquet"
)

summary_path = (
    RESULTS_DIR +
    "/long_detector_v1_summary.parquet"
)

wf.to_parquet(
    detail_path,
    engine="pyarrow",
    index=False
)

summary_df.to_parquet(
    summary_path,
    engine="pyarrow",
    index=False
)

print("\n" + "=" * 75)
print("✅ LONG OPPORTUNITY DETECTOR COMPLETE")
print("=" * 75)

print(
    "\nSaved detailed results:"
)

print(detail_path)

print(
    "\nSaved summary:"
)

print(summary_path)

In [ ]:

# =========================================================
# SILVER AGENT — STEP 14
# GAP-AWARE FEATURE ENGINE V2
# =========================================================

import os
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Mount Drive
# ---------------------------------------------------------

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

DATABASE_PATH = (
    "/content/drive/MyDrive/"
    "SilverAgent/database/silver_h1.parquet"
)

RESULTS_DIR = (
    "/content/drive/MyDrive/"
    "SilverAgent/results"
)

os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------------------------------------------------------
# 2. Load raw database
# ---------------------------------------------------------

raw = pd.read_parquet(
    DATABASE_PATH
).sort_index()

print("=" * 70)
print("GAP-AWARE FEATURE ENGINE V2")
print("=" * 70)

print(
    f"\nRaw candles: {len(raw):,}"
)

print(
    f"Start: {raw.index.min()}"
)

print(
    f"End:   {raw.index.max()}"
)

# ---------------------------------------------------------
# 3. Check timestamp spacing
# ---------------------------------------------------------

time_diff = raw.index.to_series().diff()

# A normal H1 candle should be exactly 1 hour apart.
normal_gap = pd.Timedelta(hours=1)

raw["gap_hours"] = (
    time_diff.dt.total_seconds() / 3600
)

raw["is_normal_spacing"] = (
    raw["gap_hours"] == 1
)

# ---------------------------------------------------------
# 4. Create a continuous-hour segment ID
# ---------------------------------------------------------
#
# Every time there is a gap, a new segment begins.
#
# This prevents rolling/future calculations from crossing
# a missing-data boundary.

raw["segment_id"] = (
    (~raw["is_normal_spacing"])
    .cumsum()
)

# First candle belongs to first segment.
raw.loc[
    raw.index == raw.index.min(),
    "segment_id"
] = 0

# ---------------------------------------------------------
# 5. Basic features
# ---------------------------------------------------------

raw["return_1h"] = (
    raw["close"] /
    raw["close"].shift(1) - 1
)

# A return across a data gap is invalid.
raw.loc[
    ~raw["is_normal_spacing"],
    "return_1h"
] = np.nan

# ---------------------------------------------------------
# 6. Segment-aware rolling features
# ---------------------------------------------------------

def rolling_segment(
    series,
    window
):
    return (
        series
        .groupby(raw["segment_id"])
        .rolling(window)
        .mean()
        .reset_index(
            level=0,
            drop=True
        )
    )

def std_segment(
    series,
    window
):
    return (
        series
        .groupby(raw["segment_id"])
        .rolling(window)
        .std()
        .reset_index(
            level=0,
            drop=True
        )
    )

# Moving averages

raw["MA5"] = rolling_segment(
    raw["close"],
    5
)

raw["MA10"] = rolling_segment(
    raw["close"],
    10
)

raw["MA20"] = rolling_segment(
    raw["close"],
    20
)

raw["MA50"] = rolling_segment(
    raw["close"],
    50
)

# Volatility

raw["vol10"] = std_segment(
    raw["return_1h"],
    10
)

raw["vol20"] = std_segment(
    raw["return_1h"],
    20
)

# ---------------------------------------------------------
# 7. Gap-aware momentum
# ---------------------------------------------------------
#
# We only accept a momentum value when all candles between
# the current candle and the lookback candle are present.

def gap_aware_return(
    close,
    hours
):

    shifted = close.shift(hours)

    shifted_segment = (
        raw["segment_id"]
        .shift(hours)
    )

    valid_segment = (
        raw["segment_id"] ==
        shifted_segment
    )

    result = (
        close / shifted - 1
    )

    result.loc[
        ~valid_segment
    ] = np.nan

    return result

raw["mom3"] = gap_aware_return(
    raw["close"],
    3
)

raw["mom6"] = gap_aware_return(
    raw["close"],
    6
)

raw["mom12"] = gap_aware_return(
    raw["close"],
    12
)

raw["mom24"] = gap_aware_return(
    raw["close"],
    24
)

# ---------------------------------------------------------
# 8. Relative position to moving averages
# ---------------------------------------------------------

raw["MA10_vs_MA20"] = (
    raw["MA10"] /
    raw["MA20"] - 1
)

raw["MA20_vs_MA50"] = (
    raw["MA20"] /
    raw["MA50"] - 1
)

raw["price_vs_MA10"] = (
    raw["close"] /
    raw["MA10"] - 1
)

raw["price_vs_MA20"] = (
    raw["close"] /
    raw["MA20"] - 1
)

raw["price_vs_MA50"] = (
    raw["close"] /
    raw["MA50"] - 1
)

# ---------------------------------------------------------
# 9. True Range / ATR
# ---------------------------------------------------------

previous_close = (
    raw["close"].shift(1)
)

tr1 = (
    raw["high"] -
    raw["low"]
)

tr2 = (
    raw["high"] -
    previous_close
).abs()

tr3 = (
    raw["low"] -
    previous_close
).abs()

raw["true_range"] = pd.concat(
    [tr1, tr2, tr3],
    axis=1
).max(axis=1)

# Do not allow true range across a gap.
raw.loc[
    ~raw["is_normal_spacing"],
    "true_range"
] = np.nan

raw["ATR20"] = (
    raw["true_range"]
    .groupby(raw["segment_id"])
    .rolling(20)
    .mean()
    .reset_index(
        level=0,
        drop=True
    )
)

raw["atr_percent"] = (
    raw["ATR20"] /
    raw["close"]
)

# ---------------------------------------------------------
# 10. High/low range features
# ---------------------------------------------------------

raw["high20"] = (
    raw["high"]
    .groupby(raw["segment_id"])
    .rolling(20)
    .max()
    .reset_index(
        level=0,
        drop=True
    )
)

raw["low20"] = (
    raw["low"]
    .groupby(raw["segment_id"])
    .rolling(20)
    .min()
    .reset_index(
        level=0,
        drop=True
    )
)

raw["distance_high20"] = (
    raw["high20"] /
    raw["close"] - 1
)

raw["distance_low20"] = (
    raw["close"] /
    raw["low20"] - 1
)

range_size = (
    raw["high20"] -
    raw["low20"]
)

raw["range_position20"] = np.where(
    range_size > 0,
    (
        raw["close"] -
        raw["low20"]
    ) / range_size,
    np.nan
)

# ---------------------------------------------------------
# 11. Volume
# ---------------------------------------------------------

raw["volume20"] = (
    raw["volume"]
    .groupby(raw["segment_id"])
    .rolling(20)
    .mean()
    .reset_index(
        level=0,
        drop=True
    )
)

raw["relative_volume"] = np.where(
    raw["volume20"] > 0,
    raw["volume"] /
    raw["volume20"],
    np.nan
)

# ---------------------------------------------------------
# 12. Candle structure
# ---------------------------------------------------------

raw["candle_range"] = (
    raw["high"] -
    raw["low"]
)

raw["candle_body"] = (
    raw["close"] -
    raw["open"]
).abs()

raw["body_ratio"] = np.where(
    raw["candle_range"] > 0,
    raw["candle_body"] /
    raw["candle_range"],
    np.nan
)

# ---------------------------------------------------------
# 13. Gap-aware FUTURE labels
# ---------------------------------------------------------
#
# Critical:
# A future 6/12/24 observation is only valid if the
# destination candle belongs to the SAME continuous segment.
#
# Therefore a 24-row shift cannot accidentally become
# "24 hours" across a multi-hour data gap.

def gap_aware_future(
    close,
    hours
):

    future_close = (
        close.shift(-hours)
    )

    future_segment = (
        raw["segment_id"]
        .shift(-hours)
    )

    valid_segment = (
        raw["segment_id"] ==
        future_segment
    )

    result = (
        future_close /
        close - 1
    )

    result.loc[
        ~valid_segment
    ] = np.nan

    return result

raw["future_6h"] = gap_aware_future(
    raw["close"],
    6
)

raw["future_12h"] = gap_aware_future(
    raw["close"],
    12
)

raw["future_24h"] = gap_aware_future(
    raw["close"],
    24
)

raw["future_abs_24h"] = (
    raw["future_24h"].abs()
)

# ---------------------------------------------------------
# 14. Momentum / trend agreement
# ---------------------------------------------------------

raw["momentum_agreement"] = (
    (raw["mom3"] > 0).astype(int) +
    (raw["mom6"] > 0).astype(int) +
    (raw["mom12"] > 0).astype(int) +
    (raw["mom24"] > 0).astype(int)
)

raw["trend_agreement"] = (
    (raw["close"] > raw["MA10"]).astype(int) +
    (raw["close"] > raw["MA20"]).astype(int) +
    (raw["close"] > raw["MA50"]).astype(int)
)

# ---------------------------------------------------------
# 15. Select feature dataset
# ---------------------------------------------------------

feature_columns = [

    "open",
    "high",
    "low",
    "close",
    "volume",

    "segment_id",

    "return_1h",

    "MA5",
    "MA10",
    "MA20",
    "MA50",

    "MA10_vs_MA20",
    "MA20_vs_MA50",

    "price_vs_MA10",
    "price_vs_MA20",
    "price_vs_MA50",

    "mom3",
    "mom6",
    "mom12",
    "mom24",

    "vol10",
    "vol20",

    "true_range",
    "ATR20",
    "atr_percent",

    "high20",
    "low20",

    "distance_high20",
    "distance_low20",
    "range_position20",

    "volume20",
    "relative_volume",

    "candle_range",
    "candle_body",
    "body_ratio",

    "momentum_agreement",
    "trend_agreement",

    "future_6h",
    "future_12h",
    "future_24h",
    "future_abs_24h"
]

features = raw[
    feature_columns
].copy()

# ---------------------------------------------------------
# 16. Save
# ---------------------------------------------------------

FEATURE_PATH = (
    DATABASE_PATH.replace(
        "silver_h1.parquet",
        "silver_h1_features_v2.parquet"
    )
)

features.to_parquet(
    FEATURE_PATH,
    engine="pyarrow"
)

# ---------------------------------------------------------
# 17. Quality report
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("DATA QUALITY REPORT")
print("=" * 70)

print(
    f"\nFeature rows: "
    f"{len(features):,}"
)

print(
    f"Continuous segments: "
    f"{features['segment_id'].nunique():,}"
)

print(
    f"Hourly gaps: "
    f"{(raw['gap_hours'] > 1).sum():,}"
)

print(
    f"Rows with valid 6h future: "
    f"{features['future_6h'].notna().sum():,}"
)

print(
    f"Rows with valid 12h future: "
    f"{features['future_12h'].notna().sum():,}"
)

print(
    f"Rows with valid 24h future: "
    f"{features['future_24h'].notna().sum():,}"
)

print(
    f"\nSaved to:"
)

print(FEATURE_PATH)

# ---------------------------------------------------------
# 18. Latest state
# ---------------------------------------------------------

latest = features.iloc[-1]

print("\n" + "=" * 70)
print("LATEST MARKET STATE")
print("=" * 70)

print(
    f"\nTimestamp: "
    f"{features.index[-1]}"
)

print(
    f"Price: "
    f"{latest['close']:.4f}"
)

print(
    f"MA10 vs MA20: "
    f"{latest['MA10_vs_MA20'] * 100:+.3f}%"
)

print(
    f"MA20 vs MA50: "
    f"{latest['MA20_vs_MA50'] * 100:+.3f}%"
)

print(
    f"6h momentum: "
    f"{latest['mom6'] * 100:+.3f}%"
)

print(
    f"12h momentum: "
    f"{latest['mom12'] * 100:+.3f}%"
)

print(
    f"24h momentum: "
    f"{latest['mom24'] * 100:+.3f}%"
)

print(
    f"ATR%: "
    f"{latest['atr_percent'] * 100:.3f}%"
)

print(
    f"Relative volume: "
    f"{latest['relative_volume']:.2f}x"
)

print("\n" + "=" * 70)
print("✅ GAP-AWARE FEATURE ENGINE V2 COMPLETE")
print("=" * 70)

In [ ]:
# =========================================================
# SILVER AGENT — STEP 14B
# GAP-AWARE FEATURE ENGINE V2.1
# TIME-BASED / NO FABRICATED CANDLES
# =========================================================

import os
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Mount Drive
# ---------------------------------------------------------

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

DATABASE_PATH = (
    "/content/drive/MyDrive/"
    "SilverAgent/database/silver_h1.parquet"
)

RESULTS_PATH = (
    "/content/drive/MyDrive/"
    "SilverAgent/database/"
    "silver_h1_features_v2.parquet"
)

# ---------------------------------------------------------
# 2. Load raw database
# ---------------------------------------------------------

df = pd.read_parquet(
    DATABASE_PATH
).sort_index()

print("=" * 70)
print("GAP-AWARE FEATURE ENGINE V2.1")
print("=" * 70)

print(f"\nRaw candles: {len(df):,}")
print(f"Start: {df.index.min()}")
print(f"End:   {df.index.max()}")

# ---------------------------------------------------------
# 3. Timestamp diagnostics
# ---------------------------------------------------------

time_diff = df.index.to_series().diff()

df["gap_hours"] = (
    time_diff.dt.total_seconds() / 3600
)

df["is_normal_spacing"] = (
    df["gap_hours"] == 1
)

print(
    f"Hourly gaps: "
    f"{(df['gap_hours'] > 1).sum():,}"
)

# ---------------------------------------------------------
# 4. Helper: find nearest REAL candle to a target time
# ---------------------------------------------------------
#
# No candles are created.
#
# We simply find the closest real observation.
#
# Maximum allowed distance = 90 minutes.
#
# This allows a normal one-hour market interruption
# without accepting a weekend-sized gap.

idx_ns = df.index.view("int64")
close_values = df["close"].to_numpy()

TOLERANCE_NS = int(
    pd.Timedelta(minutes=90).value
)

def nearest_price(
    target_times
):

    target_ns = target_times.view("int64")

    positions = np.searchsorted(
        idx_ns,
        target_ns,
        side="left"
    )

    positions = np.clip(
        positions,
        1,
        len(idx_ns) - 1
    )

    left = positions - 1
    right = positions

    left_distance = np.abs(
        idx_ns[left] -
        target_ns
    )

    right_distance = np.abs(
        idx_ns[right] -
        target_ns
    )

    use_right = (
        right_distance <
        left_distance
    )

    chosen = np.where(
        use_right,
        right,
        left
    )

    distance = np.minimum(
        left_distance,
        right_distance
    )

    result = np.full(
        len(target_times),
        np.nan
    )

    valid = (
        distance <= TOLERANCE_NS
    )

    result[valid] = (
        close_values[
            chosen[valid]
        ]
    )

    return result, distance / 3.6e12

# ---------------------------------------------------------
# 5. Time-based returns
# ---------------------------------------------------------

def historical_return(
    hours
):

    target = (
        df.index -
        pd.Timedelta(hours=hours)
    )

    previous_price, distance = (
        nearest_price(target)
    )

    result = (
        df["close"].to_numpy() /
        previous_price - 1
    )

    result[
        np.isnan(previous_price)
    ] = np.nan

    return result

def future_return(
    hours
):

    target = (
        df.index +
        pd.Timedelta(hours=hours)
    )

    future_price, distance = (
        nearest_price(target)
    )

    result = (
        future_price /
        df["close"].to_numpy() - 1
    )

    result[
        np.isnan(future_price)
    ] = np.nan

    return result

# ---------------------------------------------------------
# 6. Momentum
# ---------------------------------------------------------

df["return_1h"] = historical_return(1)
df["mom3"] = historical_return(3)
df["mom6"] = historical_return(6)
df["mom12"] = historical_return(12)
df["mom24"] = historical_return(24)

# ---------------------------------------------------------
# 7. Moving averages
# ---------------------------------------------------------
#
# These are observation-based but we explicitly record
# how much real elapsed time the window covers.
#
# We do NOT fill missing candles.

for n in [5, 10, 20, 50]:

    df[f"MA{n}"] = (
        df["close"]
        .rolling(
            n,
            min_periods=n
        )
        .mean()
    )

    df[f"MA{n}_span_hours"] = (
        (
            df.index.to_series()
            -
            df.index.to_series()
            .shift(n - 1)
        )
        .dt.total_seconds()
        / 3600
    )

# ---------------------------------------------------------
# 8. Volatility
# ---------------------------------------------------------

df["vol10"] = (
    df["return_1h"]
    .rolling(
        10,
        min_periods=8
    )
    .std()
)

df["vol20"] = (
    df["return_1h"]
    .rolling(
        20,
        min_periods=16
    )
    .std()
)

# ---------------------------------------------------------
# 9. Moving-average relationships
# ---------------------------------------------------------

df["MA10_vs_MA20"] = (
    df["MA10"] /
    df["MA20"] - 1
)

df["MA20_vs_MA50"] = (
    df["MA20"] /
    df["MA50"] - 1
)

df["price_vs_MA10"] = (
    df["close"] /
    df["MA10"] - 1
)

df["price_vs_MA20"] = (
    df["close"] /
    df["MA20"] - 1
)

df["price_vs_MA50"] = (
    df["close"] /
    df["MA50"] - 1
)

# ---------------------------------------------------------
# 10. True Range
# ---------------------------------------------------------

previous_close = (
    df["close"].shift(1)
)

tr1 = (
    df["high"] -
    df["low"]
)

tr2 = (
    df["high"] -
    previous_close
).abs()

tr3 = (
    df["low"] -
    previous_close
).abs()

df["true_range"] = pd.concat(
    [tr1, tr2, tr3],
    axis=1
).max(axis=1)

# Do not calculate TR across a major gap.
df.loc[
    df["gap_hours"] > 2,
    "true_range"
] = np.nan

df["ATR20"] = (
    df["true_range"]
    .rolling(
        20,
        min_periods=16
    )
    .mean()
)

df["atr_percent"] = (
    df["ATR20"] /
    df["close"]
)

# ---------------------------------------------------------
# 11. 20-candle range
# ---------------------------------------------------------

df["high20"] = (
    df["high"]
    .rolling(
        20,
        min_periods=16
    )
    .max()
)

df["low20"] = (
    df["low"]
    .rolling(
        20,
        min_periods=16
    )
    .min()
)

df["distance_high20"] = (
    df["high20"] /
    df["close"] - 1
)

df["distance_low20"] = (
    df["close"] /
    df["low20"] - 1
)

range_size = (
    df["high20"] -
    df["low20"]
)

df["range_position20"] = np.where(
    range_size > 0,
    (
        df["close"] -
        df["low20"]
    ) / range_size,
    np.nan
)

# ---------------------------------------------------------
# 12. Volume
# ---------------------------------------------------------

df["volume20"] = (
    df["volume"]
    .rolling(
        20,
        min_periods=16
    )
    .mean()
)

df["relative_volume"] = (
    df["volume"] /
    df["volume20"]
)

# ---------------------------------------------------------
# 13. Candle structure
# ---------------------------------------------------------

df["candle_range"] = (
    df["high"] -
    df["low"]
)

df["candle_body"] = (
    df["close"] -
    df["open"]
).abs()

df["body_ratio"] = np.where(
    df["candle_range"] > 0,
    df["candle_body"] /
    df["candle_range"],
    np.nan
)

# ---------------------------------------------------------
# 14. Future opportunity labels
# ---------------------------------------------------------

df["future_6h"] = future_return(6)
df["future_12h"] = future_return(12)
df["future_24h"] = future_return(24)

df["future_abs_24h"] = (
    df["future_24h"].abs()
)

# ---------------------------------------------------------
# 15. Agreement features
# ---------------------------------------------------------

df["momentum_agreement"] = (
    (df["mom3"] > 0).astype(int) +
    (df["mom6"] > 0).astype(int) +
    (df["mom12"] > 0).astype(int) +
    (df["mom24"] > 0).astype(int)
)

df["trend_agreement"] = (
    (df["close"] > df["MA10"]).astype(int) +
    (df["close"] > df["MA20"]).astype(int) +
    (df["close"] > df["MA50"]).astype(int)
)

# ---------------------------------------------------------
# 16. Keep useful columns
# ---------------------------------------------------------

feature_columns = [

    "open",
    "high",
    "low",
    "close",
    "volume",

    "gap_hours",

    "return_1h",

    "MA5",
    "MA10",
    "MA20",
    "MA50",

    "MA10_span_hours",
    "MA20_span_hours",
    "MA50_span_hours",

    "MA10_vs_MA20",
    "MA20_vs_MA50",

    "price_vs_MA10",
    "price_vs_MA20",
    "price_vs_MA50",

    "mom3",
    "mom6",
    "mom12",
    "mom24",

    "vol10",
    "vol20",

    "true_range",
    "ATR20",
    "atr_percent",

    "high20",
    "low20",
    "distance_high20",
    "distance_low20",
    "range_position20",

    "volume20",
    "relative_volume",

    "candle_range",
    "candle_body",
    "body_ratio",

    "momentum_agreement",
    "trend_agreement",

    "future_6h",
    "future_12h",
    "future_24h",
    "future_abs_24h"
]

features = df[
    feature_columns
].copy()

# ---------------------------------------------------------
# 17. Save V2.1
# ---------------------------------------------------------

features.to_parquet(
    RESULTS_PATH,
    engine="pyarrow"
)

# ---------------------------------------------------------
# 18. Report
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("V2.1 DATA QUALITY REPORT")
print("=" * 70)

print(
    f"\nFeature rows: "
    f"{len(features):,}"
)

print(
    f"Valid 6h future: "
    f"{features['future_6h'].notna().sum():,}"
)

print(
    f"Valid 12h future: "
    f"{features['future_12h'].notna().sum():,}"
)

print(
    f"Valid 24h future: "
    f"{features['future_24h'].notna().sum():,}"
)

print(
    f"\nSaved to:"
)

print(RESULTS_PATH)

# ---------------------------------------------------------
# 19. Latest state
# ---------------------------------------------------------

latest = features.iloc[-1]

print("\n" + "=" * 70)
print("LATEST MARKET STATE")
print("=" * 70)

print(
    f"\nTimestamp: {features.index[-1]}"
)

print(
    f"Price: {latest['close']:.4f}"
)

print(
    f"MA10 vs MA20: "
    f"{latest['MA10_vs_MA20'] * 100:+.3f}%"
)

print(
    f"MA20 vs MA50: "
    f"{latest['MA20_vs_MA50'] * 100:+.3f}%"
)

print(
    f"6h momentum: "
    f"{latest['mom6'] * 100:+.3f}%"
)

print(
    f"12h momentum: "
    f"{latest['mom12'] * 100:+.3f}%"
)

print(
    f"24h momentum: "
    f"{latest['mom24'] * 100:+.3f}%"
)

print(
    f"ATR%: "
    f"{latest['atr_percent'] * 100:.3f}%"
)

print(
    f"Relative volume: "
    f"{latest['relative_volume']:.2f}x"
)

print("\n" + "=" * 70)
print("✅ GAP-AWARE FEATURE ENGINE V2.1 COMPLETE")
print("=" * 70)

In [ ]:
# =========================================================
# SILVER AGENT — STEP 15
# V2.1 VALIDATION AUDIT
# =========================================================

import os
import numpy as np
import pandas as pd

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

PATH = (
    "/content/drive/MyDrive/"
    "SilverAgent/database/"
    "silver_h1_features_v2.parquet"
)

df = pd.read_parquet(PATH).sort_index()

print("=" * 70)
print("V2.1 VALIDATION AUDIT")
print("=" * 70)

# ---------------------------------------------------------
# 1. Basic future-move statistics
# ---------------------------------------------------------

future = df["future_abs_24h"].dropna()

print("\n24H FUTURE MOVE DISTRIBUTION")
print("-" * 50)

print(
    f"Observations: {len(future):,}"
)

print(
    f"Mean:   {future.mean() * 100:.3f}%"
)

print(
    f"Median: {future.median() * 100:.3f}%"
)

print(
    f"75th:   {future.quantile(.75) * 100:.3f}%"
)

print(
    f"90th:   {future.quantile(.90) * 100:.3f}%"
)

print(
    f"95th:   {future.quantile(.95) * 100:.3f}%"
)

# ---------------------------------------------------------
# 2. Define "large opportunity"
# ---------------------------------------------------------

threshold = future.quantile(.75)

df["large_move"] = (
    df["future_abs_24h"] >= threshold
)

print(
    f"\nLarge-move threshold: "
    f"{threshold * 100:.3f}%"
)

# ---------------------------------------------------------
# 3. Test MA10 > MA20
# ---------------------------------------------------------

valid = df[
    df["future_24h"].notna() &
    df["MA10_vs_MA20"].notna()
].copy()

valid["long_ma_signal"] = (
    valid["MA10_vs_MA20"] > 0
)

longs = valid[
    valid["long_ma_signal"]
]

shorts = valid[
    ~valid["long_ma_signal"]
]

print("\n" + "=" * 70)
print("MA10 / MA20 DIRECTION AUDIT")
print("=" * 70)

print(
    f"\nLONG condition "
    f"(MA10 > MA20): "
    f"{len(longs):,} observations"
)

print(
    f"Average 24h return: "
    f"{longs['future_24h'].mean() * 100:+.3f}%"
)

print(
    f"Median 24h return: "
    f"{longs['future_24h'].median() * 100:+.3f}%"
)

print(
    f"Positive 24h rate: "
    f"{(longs['future_24h'] > 0).mean() * 100:.2f}%"
)

print(
    f"Large-move rate: "
    f"{longs['large_move'].mean() * 100:.2f}%"
)

print(
    f"\nSHORT condition "
    f"(MA10 <= MA20): "
    f"{len(shorts):,} observations"
)

print(
    f"Average 24h return: "
    f"{shorts['future_24h'].mean() * 100:+.3f}%"
)

print(
    f"Median 24h return: "
    f"{shorts['future_24h'].median() * 100:+.3f}%"
)

print(
    f"Positive 24h rate: "
    f"{(shorts['future_24h'] > 0).mean() * 100:.2f}%"
)

print(
    f"Large-move rate: "
    f"{shorts['large_move'].mean() * 100:.2f}%"
)

# ---------------------------------------------------------
# 4. Test MA10 > MA20 only during high-opportunity
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("HIGH-OPPORTUNITY DIRECTION AUDIT")
print("=" * 70)

# Opportunity proxies based only on current information.
#
# Higher volatility / ATR / distance from recent high indicate
# more energetic market conditions.

for col in [
    "vol20",
    "atr_percent",
    "distance_high20"
]:
    df[col + "_pct"] = (
        df[col]
        .rank(pct=True)
    )

df["simple_opportunity"] = (
    df[
        [
            "vol20_pct",
            "atr_percent_pct",
            "distance_high20_pct"
        ]
    ]
    .mean(axis=1)
)

high_opp = df[
    df["simple_opportunity"] >= 0.80
].copy()

high_opp = high_opp[
    high_opp["future_24h"].notna() &
    high_opp["MA10_vs_MA20"].notna()
]

high_long = high_opp[
    high_opp["MA10_vs_MA20"] > 0
]

high_short = high_opp[
    high_opp["MA10_vs_MA20"] <= 0
]

print(
    f"\nHigh-opportunity observations: "
    f"{len(high_opp):,}"
)

print(
    f"\nHigh-opportunity LONG:"
)

print(
    f"Count: {len(high_long):,}"
)

print(
    f"Average 24h: "
    f"{high_long['future_24h'].mean() * 100:+.3f}%"
)

print(
    f"Positive rate: "
    f"{(high_long['future_24h'] > 0).mean() * 100:.2f}%"
)

print(
    f"Large-move rate: "
    f"{high_long['large_move'].mean() * 100:.2f}%"
)

print(
    f"\nHigh-opportunity SHORT:"
)

print(
    f"Count: {len(high_short):,}"
)

print(
    f"Average 24h: "
    f"{high_short['future_24h'].mean() * 100:+.3f}%"
)

print(
    f"Positive rate: "
    f"{(high_short['future_24h'] > 0).mean() * 100:.2f}%"
)

print(
    f"Large-move rate: "
    f"{high_short['large_move'].mean() * 100:.2f}%"
)

# ---------------------------------------------------------
# 5. Latest market state
# ---------------------------------------------------------

latest = df.iloc[-1]

print("\n" + "=" * 70)
print("CURRENT MARKET")
print("=" * 70)

print(
    f"\nPrice: {latest['close']:.4f}"
)

print(
    f"MA10 vs MA20: "
    f"{latest['MA10_vs_MA20'] * 100:+.3f}%"
)

print(
    f"MA20 vs MA50: "
    f"{latest['MA20_vs_MA50'] * 100:+.3f}%"
)

print(
    f"6h momentum: "
    f"{latest['mom6'] * 100:+.3f}%"
)

print(
    f"ATR%: "
    f"{latest['atr_percent'] * 100:.3f}%"
)

print(
    f"Relative volume: "
    f"{latest['relative_volume']:.2f}x"
)

print("\n" + "=" * 70)
print("✅ V2.1 AUDIT COMPLETE")
print("=" * 70)

In [ ]:
# =========================================================
# OPPORTUNITY ENGINE V2 — FAST PERCENTILE CALCULATION
# =========================================================

import os
import numpy as np
import pandas as pd

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

FEATURE_PATH = (
    "/content/drive/MyDrive/"
    "SilverAgent/database/"
    "silver_h1_features_v2.parquet"
)

df = pd.read_parquet(
    FEATURE_PATH
).sort_index()

print("=" * 70)
print("OPPORTUNITY ENGINE V2 — FAST VERSION")
print("=" * 70)

print(f"\nRows: {len(df):,}")

# ---------------------------------------------------------
# Opportunity features
# ---------------------------------------------------------

opp_features = [
    "vol10",
    "vol20",
    "atr_percent",
    "distance_high20",
    "relative_volume"
]

# ---------------------------------------------------------
# FAST EXPANDING PERCENTILES
# ---------------------------------------------------------
#
# We use pandas' native expanding rank.
#
# The current observation is ranked against information
# available up to that observation.
#
# Shift(1) ensures the current value itself isn't included
# in its historical reference distribution.
# ---------------------------------------------------------

print("\nCalculating historical percentiles...")

for col in opp_features:

    history = (
        df[col]
        .shift(1)
    )

    df[col + "_pct"] = (
        history
        .expanding(
            min_periods=500
        )
        .rank(
            pct=True
        )
    )

    print(
        f"  ✓ {col}"
    )

# ---------------------------------------------------------
# Opportunity score
# ---------------------------------------------------------

pct_columns = [
    col + "_pct"
    for col in opp_features
]

df["opportunity_score"] = (
    df[pct_columns]
    .mean(axis=1)
    * 100
)

# ---------------------------------------------------------
# Opportunity zones
# ---------------------------------------------------------

def zone(score):

    if pd.isna(score):
        return "INSUFFICIENT_DATA"

    if score < 20:
        return "VERY_LOW"

    if score < 40:
        return "LOW"

    if score < 60:
        return "MEDIUM"

    if score < 80:
        return "HIGH"

    return "VERY_HIGH"

df["opportunity_zone"] = (
    df["opportunity_score"]
    .apply(zone)
)

# ---------------------------------------------------------
# Walk-forward validation
# ---------------------------------------------------------

TRAIN_SIZE = 4000
TEST_SIZE = 1000
STEP = 1000

windows = []

start = 0

while (
    start + TRAIN_SIZE + TEST_SIZE
    <= len(df)
):

    train = df.iloc[
        start:
        start + TRAIN_SIZE
    ]

    test = df.iloc[
        start + TRAIN_SIZE:
        start + TRAIN_SIZE + TEST_SIZE
    ].copy()

    # Training determines what counts as a large move.
    train_future = (
        train["future_abs_24h"]
        .dropna()
    )

    if len(train_future) < 500:
        start += STEP
        continue

    large_threshold = (
        train_future.quantile(0.75)
    )

    # Valid test data only
    test = test[
        test["future_abs_24h"].notna() &
        test["opportunity_score"].notna()
    ]

    if len(test) == 0:
        start += STEP
        continue

    # Baseline
    baseline_mean = (
        test["future_abs_24h"].mean()
    )

    baseline_large = (
        test["future_abs_24h"]
        >= large_threshold
    ).mean()

    # High opportunity
    high = test[
        test["opportunity_score"] >= 60
    ]

    # Very high opportunity
    very_high = test[
        test["opportunity_score"] >= 80
    ]

    windows.append({

        "test_rows":
            len(test),

        "baseline_mean":
            baseline_mean,

        "baseline_large":
            baseline_large,

        "high_count":
            len(high),

        "high_mean":
            high["future_abs_24h"].mean()
            if len(high)
            else np.nan,

        "high_large":
            (
                high["future_abs_24h"]
                >= large_threshold
            ).mean()
            if len(high)
            else np.nan,

        "very_high_count":
            len(very_high),

        "very_high_mean":
            very_high["future_abs_24h"].mean()
            if len(very_high)
            else np.nan,

        "very_high_large":
            (
                very_high["future_abs_24h"]
                >= large_threshold
            ).mean()
            if len(very_high)
            else np.nan
    })

    start += STEP

results = pd.DataFrame(
    windows
)

# ---------------------------------------------------------
# Weighted averages
# ---------------------------------------------------------

def weighted(
    values,
    weights
):

    values = np.asarray(
        values,
        dtype=float
    )

    weights = np.asarray(
        weights,
        dtype=float
    )

    valid = (
        np.isfinite(values) &
        np.isfinite(weights) &
        (weights > 0)
    )

    return np.average(
        values[valid],
        weights=weights[valid]
    )

baseline_mean = weighted(
    results["baseline_mean"],
    results["test_rows"]
)

high_mean = weighted(
    results["high_mean"],
    results["high_count"]
)

very_high_mean = weighted(
    results["very_high_mean"],
    results["very_high_count"]
)

baseline_large = weighted(
    results["baseline_large"],
    results["test_rows"]
)

high_large = weighted(
    results["high_large"],
    results["high_count"]
)

very_high_large = weighted(
    results["very_high_large"],
    results["very_high_count"]
)

# ---------------------------------------------------------
# Print results
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("WALK-FORWARD RESULTS")
print("=" * 70)

print(
    f"\nWindows tested: "
    f"{len(results)}"
)

print("\nBASELINE")
print(
    f"Mean 24h movement: "
    f"{baseline_mean * 100:.3f}%"
)

print(
    f"Large-move rate: "
    f"{baseline_large * 100:.2f}%"
)

print("\nHIGH OPPORTUNITY >= 60")
print(
    f"Mean 24h movement: "
    f"{high_mean * 100:.3f}%"
)

print(
    f"Improvement: "
    f"{(high_mean - baseline_mean) * 100:+.3f} pp"
)

print(
    f"Large-move rate: "
    f"{high_large * 100:.2f}%"
)

print(
    f"Improvement: "
    f"{(high_large - baseline_large) * 100:+.2f} pp"
)

print("\nVERY HIGH OPPORTUNITY >= 80")
print(
    f"Mean 24h movement: "
    f"{very_high_mean * 100:.3f}%"
)

print(
    f"Large-move rate: "
    f"{very_high_large * 100:.2f}%"
)

print(
    f"Improvement: "
    f"{(very_high_large - baseline_large) * 100:+.2f} pp"
)

# ---------------------------------------------------------
# Current state
# ---------------------------------------------------------

latest = df.iloc[-1]

print("\n" + "=" * 70)
print("CURRENT MARKET")
print("=" * 70)

print(
    f"\nPrice: "
    f"{latest['close']:.4f}"
)

print(
    f"Opportunity score: "
    f"{latest['opportunity_score']:.1f}/100"
)

print(
    f"Zone: "
    f"{latest['opportunity_zone']}"
)

print(
    f"\nvol10: "
    f"{latest['vol10'] * 100:.3f}%"
)

print(
    f"vol20: "
    f"{latest['vol20'] * 100:.3f}%"
)

print(
    f"ATR%: "
    f"{latest['atr_percent'] * 100:.3f}%"
)

print(
    f"Distance high20: "
    f"{latest['distance_high20'] * 100:.3f}%"
)

print(
    f"Relative volume: "
    f"{latest['relative_volume']:.2f}x"
)

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

RESULTS_DIR = (
    "/content/drive/MyDrive/"
    "SilverAgent/results"
)

os.makedirs(
    RESULTS_DIR,
    exist_ok=True
)

df.to_parquet(
    RESULTS_DIR +
    "/opportunity_v2_history.parquet"
)

results.to_parquet(
    RESULTS_DIR +
    "/opportunity_v2_walk_forward.parquet",
    index=False
)

print("\n" + "=" * 70)
print("✅ OPPORTUNITY ENGINE V2 COMPLETE")
print("=" * 70)

In [ ]:
# =========================================================
# SILVER AGENT — STEP 17
# DIRECTION ENGINE V3
# OPPORTUNITY + DIRECTION WALK-FORWARD TEST
# =========================================================

import os
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Drive + paths
# ---------------------------------------------------------

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

FEATURE_PATH = (
    "/content/drive/MyDrive/"
    "SilverAgent/database/"
    "silver_h1_features_v2.parquet"
)

RESULTS_DIR = (
    "/content/drive/MyDrive/"
    "SilverAgent/results"
)

os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------------------------------------------------------
# 2. Load feature data
# ---------------------------------------------------------

df = pd.read_parquet(
    FEATURE_PATH
).sort_index()

print("=" * 70)
print("DIRECTION ENGINE V3")
print("OPPORTUNITY + DIRECTION WALK-FORWARD TEST")
print("=" * 70)

print(
    f"\nRows loaded: {len(df):,}"
)

# ---------------------------------------------------------
# 3. Rebuild Opportunity Score
# ---------------------------------------------------------
#
# Must use historical information only.
# ---------------------------------------------------------

opp_features = [
    "vol10",
    "vol20",
    "atr_percent",
    "distance_high20",
    "relative_volume"
]

def expanding_percentile(
    series,
    min_periods=500
):

    history = (
        series
        .shift(1)
        .expanding(
            min_periods=min_periods
        )
        .rank(
            pct=True
        )
    )

    return history

for col in opp_features:

    df[col + "_pct"] = (
        expanding_percentile(
            df[col]
        )
    )

pct_columns = [
    col + "_pct"
    for col in opp_features
]

df["opportunity_score"] = (
    df[pct_columns]
    .mean(axis=1)
    * 100
)

# ---------------------------------------------------------
# 4. Direction evidence
# ---------------------------------------------------------
#
# We deliberately keep this simple.
#
# LONG evidence:
#   MA10 > MA20
#   MA20 > MA50
#   momentum positive
#   price above MA20
#
# SHORT evidence:
#   MA10 < MA20
#   MA20 < MA50
#   momentum negative
#   price below MA20
#
# The engine can also say WAIT.
# ---------------------------------------------------------

df["long_points"] = (
    (df["MA10_vs_MA20"] > 0).astype(int) +
    (df["MA20_vs_MA50"] > 0).astype(int) +
    (df["mom6"] > 0).astype(int) +
    (df["mom12"] > 0).astype(int) +
    (df["price_vs_MA20"] > 0).astype(int)
)

df["short_points"] = (
    (df["MA10_vs_MA20"] < 0).astype(int) +
    (df["MA20_vs_MA50"] < 0).astype(int) +
    (df["mom6"] < 0).astype(int) +
    (df["mom12"] < 0).astype(int) +
    (df["price_vs_MA20"] < 0).astype(int)
)

# ---------------------------------------------------------
# 5. Direction classification
# ---------------------------------------------------------
#
# We require at least 4/5 pieces of evidence.
#
# Otherwise the machine says WAIT.
# ---------------------------------------------------------

df["direction"] = "WAIT"

long_condition = (
    (df["long_points"] >= 4) &
    (
        df["long_points"] >
        df["short_points"]
    )
)

short_condition = (
    (df["short_points"] >= 4) &
    (
        df["short_points"] >
        df["long_points"]
    )
)

df.loc[
    long_condition,
    "direction"
] = "LONG"

df.loc[
    short_condition,
    "direction"
] = "SHORT"

# ---------------------------------------------------------
# 6. Direction confidence
# ---------------------------------------------------------

df["direction_confidence"] = (
    np.maximum(
        df["long_points"],
        df["short_points"]
    ) / 5 * 100
)

# ---------------------------------------------------------
# 7. Walk-forward validation
# ---------------------------------------------------------

TRAIN_SIZE = 4000
TEST_SIZE = 1000
STEP = 1000

windows = []

start = 0

while (
    start +
    TRAIN_SIZE +
    TEST_SIZE
    <= len(df)
):

    train = df.iloc[
        start:
        start + TRAIN_SIZE
    ]

    test = df.iloc[
        start + TRAIN_SIZE:
        start + TRAIN_SIZE + TEST_SIZE
    ].copy()

    # -----------------------------------------------------
    # Training threshold for large moves
    # -----------------------------------------------------

    train_future = (
        train["future_abs_24h"]
        .dropna()
    )

    if len(train_future) < 500:
        start += STEP
        continue

    large_threshold = (
        train_future.quantile(0.75)
    )

    # -----------------------------------------------------
    # Valid test data
    # -----------------------------------------------------

    test = test[
        test["future_24h"].notna() &
        test["future_abs_24h"].notna() &
        test["opportunity_score"].notna()
    ]

    if len(test) == 0:
        start += STEP
        continue

    # -----------------------------------------------------
    # Baseline
    # -----------------------------------------------------

    baseline = test[
        "future_24h"
    ].mean()

    baseline_abs = test[
        "future_abs_24h"
    ].mean()

    # -----------------------------------------------------
    # HIGH OPPORTUNITY
    # -----------------------------------------------------

    high = test[
        test["opportunity_score"] >= 60
    ]

    very_high = test[
        test["opportunity_score"] >= 80
    ]

    # -----------------------------------------------------
    # HIGH + LONG
    # -----------------------------------------------------

    high_long = test[
        (test["opportunity_score"] >= 60) &
        (test["direction"] == "LONG")
    ]

    # -----------------------------------------------------
    # HIGH + SHORT
    # -----------------------------------------------------

    high_short = test[
        (test["opportunity_score"] >= 60) &
        (test["direction"] == "SHORT")
    ]

    # -----------------------------------------------------
    # VERY HIGH + LONG
    # -----------------------------------------------------

    very_high_long = test[
        (test["opportunity_score"] >= 80) &
        (test["direction"] == "LONG")
    ]

    # -----------------------------------------------------
    # VERY HIGH + SHORT
    # -----------------------------------------------------

    very_high_short = test[
        (test["opportunity_score"] >= 80) &
        (test["direction"] == "SHORT")
    ]

    # -----------------------------------------------------
    # Helper for directional results
    # -----------------------------------------------------

    def directional_stats(
        subset,
        direction
    ):

        if len(subset) == 0:

            return {
                "count": 0,
                "return": np.nan,
                "win": np.nan,
                "large": np.nan
            }

        future = subset[
            "future_24h"
        ]

        # Direction-adjusted return.
        #
        # LONG benefits from positive movement.
        # SHORT benefits from negative movement.

        if direction == "LONG":

            adjusted = future

        elif direction == "SHORT":

            adjusted = -future

        else:

            adjusted = future

        return {

            "count":
                len(subset),

            "return":
                adjusted.mean(),

            "win":
                (adjusted > 0).mean(),

            "large":
                (
                    subset["future_abs_24h"]
                    >= large_threshold
                ).mean()
        }

    hl = directional_stats(
        high_long,
        "LONG"
    )

    hs = directional_stats(
        high_short,
        "SHORT"
    )

    vhl = directional_stats(
        very_high_long,
        "LONG"
    )

    vhs = directional_stats(
        very_high_short,
        "SHORT"
    )

    windows.append({

        "test_rows":
            len(test),

        "baseline_return":
            baseline,

        "baseline_abs":
            baseline_abs,

        "high_abs":
            high[
                "future_abs_24h"
            ].mean(),

        "very_high_abs":
            very_high[
                "future_abs_24h"
            ].mean(),

        "high_long_count":
            hl["count"],

        "high_long_return":
            hl["return"],

        "high_long_win":
            hl["win"],

        "high_long_large":
            hl["large"],

        "high_short_count":
            hs["count"],

        "high_short_return":
            hs["return"],

        "high_short_win":
            hs["win"],

        "high_short_large":
            hs["large"],

        "very_high_long_count":
            vhl["count"],

        "very_high_long_return":
            vhl["return"],

        "very_high_long_win":
            vhl["win"],

        "very_high_long_large":
            vhl["large"],

        "very_high_short_count":
            vhs["count"],

        "very_high_short_return":
            vhs["return"],

        "very_high_short_win":
            vhs["win"],

        "very_high_short_large":
            vhs["large"]
    })

    start += STEP

# ---------------------------------------------------------
# 8. Results
# ---------------------------------------------------------

results = pd.DataFrame(
    windows
)

print(
    f"\nWalk-forward windows: "
    f"{len(results)}"
)

# ---------------------------------------------------------
# 9. Weighted helper
# ---------------------------------------------------------

def weighted(
    values,
    weights
):

    values = np.asarray(
        values,
        dtype=float
    )

    weights = np.asarray(
        weights,
        dtype=float
    )

    valid = (
        np.isfinite(values) &
        np.isfinite(weights) &
        (weights > 0)
    )

    if valid.sum() == 0:
        return np.nan

    return np.average(
        values[valid],
        weights=weights[valid]
    )

# ---------------------------------------------------------
# 10. Aggregate results
# ---------------------------------------------------------

hl_return = weighted(
    results["high_long_return"],
    results["high_long_count"]
)

hs_return = weighted(
    results["high_short_return"],
    results["high_short_count"]
)

vhl_return = weighted(
    results["very_high_long_return"],
    results["very_high_long_count"]
)

vhs_return = weighted(
    results["very_high_short_return"],
    results["very_high_short_count"]
)

hl_win = weighted(
    results["high_long_win"],
    results["high_long_count"]
)

hs_win = weighted(
    results["high_short_win"],
    results["high_short_count"]
)

vhl_win = weighted(
    results["very_high_long_win"],
    results["very_high_long_count"]
)

vhs_win = weighted(
    results["very_high_short_win"],
    results["very_high_short_count"]
)

# ---------------------------------------------------------
# 11. Print results
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("DIRECTION RESULTS")
print("=" * 70)

print("\nHIGH OPPORTUNITY + LONG")
print("-" * 50)

print(
    f"Signals: "
    f"{results['high_long_count'].sum():,.0f}"
)

print(
    f"Direction-adjusted avg 24h: "
    f"{hl_return * 100:+.3f}%"
)

print(
    f"Win rate: "
    f"{hl_win * 100:.2f}%"
)

print("\nHIGH OPPORTUNITY + SHORT")
print("-" * 50)

print(
    f"Signals: "
    f"{results['high_short_count'].sum():,.0f}"
)

print(
    f"Direction-adjusted avg 24h: "
    f"{hs_return * 100:+.3f}%"
)

print(
    f"Win rate: "
    f"{hs_win * 100:.2f}%"
)

print("\nVERY HIGH OPPORTUNITY + LONG")
print("-" * 50)

print(
    f"Signals: "
    f"{results['very_high_long_count'].sum():,.0f}"
)

print(
    f"Direction-adjusted avg 24h: "
    f"{vhl_return * 100:+.3f}%"
)

print(
    f"Win rate: "
    f"{vhl_win * 100:.2f}%"
)

print("\nVERY HIGH OPPORTUNITY + SHORT")
print("-" * 50)

print(
    f"Signals: "
    f"{results['very_high_short_count'].sum():,.0f}"
)

print(
    f"Direction-adjusted avg 24h: "
    f"{vhs_return * 100:+.3f}%"
)

print(
    f"Win rate: "
    f"{vhs_win * 100:.2f}%"
)

# ---------------------------------------------------------
# 12. Current market
# ---------------------------------------------------------

latest = df.iloc[-1]

print("\n" + "=" * 70)
print("CURRENT MACHINE STATE")
print("=" * 70)

print(
    f"\nPrice: "
    f"{latest['close']:.4f}"
)

print(
    f"Opportunity: "
    f"{latest['opportunity_score']:.1f}/100"
)

print(
    f"Direction: "
    f"{latest['direction']}"
)

print(
    f"Direction confidence: "
    f"{latest['direction_confidence']:.0f}%"
)

print(
    f"\nLong evidence: "
    f"{latest['long_points']}/5"
)

print(
    f"Short evidence: "
    f"{latest['short_points']}/5"
)

print(
    f"\nMA10 vs MA20: "
    f"{latest['MA10_vs_MA20'] * 100:+.3f}%"
)

print(
    f"MA20 vs MA50: "
    f"{latest['MA20_vs_MA50'] * 100:+.3f}%"
)

print(
    f"6h momentum: "
    f"{latest['mom6'] * 100:+.3f}%"
)

print(
    f"12h momentum: "
    f"{latest['mom12'] * 100:+.3f}%"
)

# ---------------------------------------------------------
# 13. Save
# ---------------------------------------------------------

HISTORY_PATH = (
    RESULTS_DIR +
    "/direction_v3_history.parquet"
)

WINDOW_PATH = (
    RESULTS_DIR +
    "/direction_v3_walk_forward.parquet"
)

df.to_parquet(
    HISTORY_PATH
)

results.to_parquet(
    WINDOW_PATH,
    index=False
)

print("\n" + "=" * 70)
print("✅ DIRECTION ENGINE V3 COMPLETE")
print("=" * 70)

print(
    f"\nHistory saved:"
)

print(HISTORY_PATH)

print(
    f"\nWalk-forward results saved:"
)

print(WINDOW_PATH)

In [ ]:
# =========================================================
# SILVER AGENT — STEP 18
# OPPORTUNITY TYPE DISCOVERY V1 — CLEAN VERSION
# =========================================================

import os
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Paths
# ---------------------------------------------------------

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

FEATURE_PATH = (
    "/content/drive/MyDrive/"
    "SilverAgent/database/"
    "silver_h1_features_v2.parquet"
)

RESULTS_DIR = (
    "/content/drive/MyDrive/"
    "SilverAgent/results"
)

os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------------------------------------------------------
# 2. Load data
# ---------------------------------------------------------

df = pd.read_parquet(
    FEATURE_PATH
).sort_index()

print("=" * 70)
print("OPPORTUNITY TYPE DISCOVERY V1")
print("=" * 70)

print(f"\nRows loaded: {len(df):,}")

# ---------------------------------------------------------
# 3. Opportunity score
# ---------------------------------------------------------

opp_features = [
    "vol10",
    "vol20",
    "atr_percent",
    "distance_high20",
    "relative_volume"
]

for col in opp_features:

    historical = (
        df[col]
        .shift(1)
        .expanding(min_periods=500)
        .rank(pct=True)
    )

    df[col + "_pct"] = historical

pct_columns = [
    col + "_pct"
    for col in opp_features
]

df["opportunity_score"] = (
    df[pct_columns]
    .mean(axis=1)
    * 100
)

# ---------------------------------------------------------
# 4. Behaviour features
# ---------------------------------------------------------

df["range_expansion"] = (
    df["vol10"] /
    df["vol20"]
)

df["near_high"] = (
    df["distance_high20"] <= 0.005
)

df["near_low"] = (
    df["distance_low20"] <= 0.005
)

# ---------------------------------------------------------
# 5. Opportunity type
# ---------------------------------------------------------

df["opportunity_type"] = "OTHER"

# Breakout UP

condition = (
    (df["opportunity_score"] >= 80) &
    df["near_high"] &
    (df["mom6"] > 0)
)

df.loc[
    condition,
    "opportunity_type"
] = "BREAKOUT_UP"

# Breakout DOWN

condition = (
    (df["opportunity_score"] >= 80) &
    df["near_low"] &
    (df["mom6"] < 0)
)

df.loc[
    condition &
    (df["opportunity_type"] == "OTHER"),
    "opportunity_type"
] = "BREAKOUT_DOWN"

# Momentum UP

condition = (
    (df["opportunity_score"] >= 80) &
    (df["mom6"] > 0) &
    (df["mom12"] > 0)
)

df.loc[
    condition &
    (df["opportunity_type"] == "OTHER"),
    "opportunity_type"
] = "MOMENTUM_UP"

# Momentum DOWN

condition = (
    (df["opportunity_score"] >= 80) &
    (df["mom6"] < 0) &
    (df["mom12"] < 0)
)

df.loc[
    condition &
    (df["opportunity_type"] == "OTHER"),
    "opportunity_type"
] = "MOMENTUM_DOWN"

# Volatility expansion

condition = (
    (df["opportunity_score"] >= 80) &
    (df["range_expansion"] > 1.25)
)

df.loc[
    condition &
    (df["opportunity_type"] == "OTHER"),
    "opportunity_type"
] = "VOLATILITY_EXPANSION"

# Mean reversion

condition = (
    (df["opportunity_score"] >= 80) &
    (
        (
            (df["distance_high20"] > 0.02) &
            (df["mom6"] < 0)
        )
        |
        (
            (df["distance_low20"] > 0.02) &
            (df["mom6"] > 0)
        )
    )
)

df.loc[
    condition &
    (df["opportunity_type"] == "OTHER"),
    "opportunity_type"
] = "MEAN_REVERSION"

# ---------------------------------------------------------
# 6. Type distribution
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("OPPORTUNITY TYPE DISTRIBUTION")
print("=" * 70)

print(
    df["opportunity_type"]
    .value_counts()
)

# ---------------------------------------------------------
# 7. Walk-forward test
# ---------------------------------------------------------

TRAIN_SIZE = 4000
TEST_SIZE = 1000
STEP = 1000

types = [
    "BREAKOUT_UP",
    "BREAKOUT_DOWN",
    "MOMENTUM_UP",
    "MOMENTUM_DOWN",
    "VOLATILITY_EXPANSION",
    "MEAN_REVERSION"
]

windows = []

start = 0

while (
    start +
    TRAIN_SIZE +
    TEST_SIZE
    <= len(df)
):

    train = df.iloc[
        start:
        start + TRAIN_SIZE
    ]

    test = df.iloc[
        start + TRAIN_SIZE:
        start + TRAIN_SIZE + TEST_SIZE
    ].copy()

    train_future = (
        train["future_abs_24h"]
        .dropna()
    )

    if len(train_future) < 500:
        start += STEP
        continue

    large_threshold = (
        train_future.quantile(0.75)
    )

    test = test[
        test["future_abs_24h"].notna()
    ]

    if len(test) == 0:
        start += STEP
        continue

    row = {}

    row["test_rows"] = len(test)

    row["baseline_mean"] = (
        test["future_abs_24h"].mean()
    )

    row["baseline_large"] = (
        test["future_abs_24h"]
        >= large_threshold
    ).mean()

    for typ in types:

        subset = test[
            test["opportunity_type"] == typ
        ]

        count = len(subset)

        row[typ + "_count"] = count

        if count > 0:

            row[typ + "_mean"] = (
                subset["future_abs_24h"]
                .mean()
            )

            row[typ + "_large"] = (
                subset["future_abs_24h"]
                >= large_threshold
            ).mean()

        else:

            row[typ + "_mean"] = np.nan
            row[typ + "_large"] = np.nan

    windows.append(row)

    start += STEP

results = pd.DataFrame(windows)

print(
    f"\nWalk-forward windows: "
    f"{len(results)}"
)

# ---------------------------------------------------------
# 8. Weighted average
# ---------------------------------------------------------

def weighted_mean(
    values,
    weights
):

    values = np.asarray(
        values,
        dtype=float
    )

    weights = np.asarray(
        weights,
        dtype=float
    )

    valid = (
        np.isfinite(values) &
        np.isfinite(weights) &
        (weights > 0)
    )

    if valid.sum() == 0:
        return np.nan

    return np.average(
        values[valid],
        weights=weights[valid]
    )

# ---------------------------------------------------------
# 9. Overall baseline
# ---------------------------------------------------------

baseline_mean = weighted_mean(
    results["baseline_mean"],
    results["test_rows"]
)

baseline_large = weighted_mean(
    results["baseline_large"],
    results["test_rows"]
)

# ---------------------------------------------------------
# 10. Type results
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("WALK-FORWARD TYPE RESULTS")
print("=" * 70)

for typ in types:

    count_column = (
        typ + "_count"
    )

    mean_column = (
        typ + "_mean"
    )

    large_column = (
        typ + "_large"
    )

    count = (
        results[count_column]
        .sum()
    )

    mean = weighted_mean(
        results[mean_column],
        results[count_column]
    )

    large = weighted_mean(
        results[large_column],
        results[count_column]
    )

    print("\n" + typ)
    print("-" * 50)

    print(
        f"Signals: {count:,.0f}"
    )

    if np.isfinite(mean):

        improvement = (
            mean -
            baseline_mean
        )

        print(
            f"Avg 24h movement: "
            f"{mean * 100:.3f}%"
        )

        print(
            f"Vs baseline: "
            f"{improvement * 100:+.3f} pp"
        )

        print(
            f"Large-move rate: "
            f"{large * 100:.2f}%"
        )

        large_improvement = (
            large -
            baseline_large
        )

        print(
            f"Large-move improvement: "
            f"{large_improvement * 100:+.2f} pp"
        )

    else:

        print(
            "Insufficient observations."
        )

# ---------------------------------------------------------
# 11. Current market
# ---------------------------------------------------------

latest = df.iloc[-1]

print("\n" + "=" * 70)
print("CURRENT MARKET")
print("=" * 70)

print(
    f"\nPrice: "
    f"{latest['close']:.4f}"
)

print(
    f"Opportunity score: "
    f"{latest['opportunity_score']:.1f}/100"
)

print(
    f"Opportunity type: "
    f"{latest['opportunity_type']}"
)

print(
    f"\n6h momentum: "
    f"{latest['mom6'] * 100:+.3f}%"
)

print(
    f"12h momentum: "
    f"{latest['mom12'] * 100:+.3f}%"
)

print(
    f"24h momentum: "
    f"{latest['mom24'] * 100:+.3f}%"
)

print(
    f"Volatility expansion: "
    f"{latest['range_expansion']:.2f}x"
)

print(
    f"Distance high20: "
    f"{latest['distance_high20'] * 100:.3f}%"
)

print(
    f"Distance low20: "
    f"{latest['distance_low20'] * 100:.3f}%"
)

# ---------------------------------------------------------
# 12. Save
# ---------------------------------------------------------

HISTORY_PATH = (
    RESULTS_DIR +
    "/opportunity_type_v1_history.parquet"
)

WINDOW_PATH = (
    RESULTS_DIR +
    "/opportunity_type_v1_walk_forward.parquet"
)

df.to_parquet(
    HISTORY_PATH
)

results.to_parquet(
    WINDOW_PATH,
    index=False
)

print("\n" + "=" * 70)
print("✅ OPPORTUNITY TYPE DISCOVERY COMPLETE")
print("=" * 70)

print(
    f"\nHistory saved:"
)

print(HISTORY_PATH)

print(
    f"\nWalk-forward results saved:"
)

print(WINDOW_PATH)

In [ ]:
# =========================================================
# SILVER AGENT — STEP 19
# CONTINUOUS OPPORTUNITY FEATURES V2
# =========================================================

import os
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Paths
# ---------------------------------------------------------

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

FEATURE_PATH = (
    "/content/drive/MyDrive/"
    "SilverAgent/database/"
    "silver_h1_features_v2.parquet"
)

RESULTS_DIR = (
    "/content/drive/MyDrive/"
    "SilverAgent/results"
)

os.makedirs(
    RESULTS_DIR,
    exist_ok=True
)

# ---------------------------------------------------------
# 2. Load data
# ---------------------------------------------------------

df = pd.read_parquet(
    FEATURE_PATH
).sort_index()

print("=" * 70)
print("CONTINUOUS OPPORTUNITY FEATURES V2")
print("=" * 70)

print(
    f"\nRows loaded: {len(df):,}"
)

# ---------------------------------------------------------
# 3. Historical percentile helper
# ---------------------------------------------------------
#
# Every score uses ONLY information available before
# the current observation.
# ---------------------------------------------------------

def historical_percentile(
    series,
    min_periods=500
):

    return (
        series
        .shift(1)
        .expanding(
            min_periods=min_periods
        )
        .rank(
            pct=True
        )
    )

# ---------------------------------------------------------
# 4. Base opportunity features
# ---------------------------------------------------------

base_features = [
    "vol10",
    "vol20",
    "atr_percent",
    "distance_high20",
    "distance_low20",
    "relative_volume"
]

print(
    "\nCalculating historical percentiles..."
)

for col in base_features:

    df[
        col + "_pct"
    ] = historical_percentile(
        df[col]
    )

    print(
        f"  ✓ {col}"
    )

# ---------------------------------------------------------
# 5. Momentum intensity
# ---------------------------------------------------------
#
# This measures how strong recent movement is without
# deciding whether it is bullish or bearish.
# ---------------------------------------------------------

df["momentum_intensity"] = (
    (
        df["mom3"].abs() +
        df["mom6"].abs() +
        df["mom12"].abs()
    ) / 3
)

df[
    "momentum_intensity_pct"
] = historical_percentile(
    df["momentum_intensity"]
)

# ---------------------------------------------------------
# 6. Range expansion
# ---------------------------------------------------------

df["range_expansion"] = (
    df["vol10"] /
    df["vol20"]
)

df[
    "range_expansion_pct"
] = historical_percentile(
    df["range_expansion"]
)

# ---------------------------------------------------------
# 7. Price displacement
# ---------------------------------------------------------
#
# How far price is from its medium-term average.
# ---------------------------------------------------------

df["price_displacement"] = (
    df["price_vs_MA20"].abs()
)

df[
    "price_displacement_pct"
] = historical_percentile(
    df["price_displacement"]
)

# ---------------------------------------------------------
# 8. ATR relative to recent volatility
# ---------------------------------------------------------

df["atr_vol_ratio"] = (
    df["atr_percent"] /
    df["vol20"]
)

df[
    "atr_vol_ratio_pct"
] = historical_percentile(
    df["atr_vol_ratio"]
)

# ---------------------------------------------------------
# 9. Continuous component scores
# ---------------------------------------------------------

df["volatility_score"] = (
    df[
        [
            "vol10_pct",
            "vol20_pct",
            "atr_percent_pct"
        ]
    ]
    .mean(axis=1)
    * 100
)

df["momentum_energy_score"] = (
    (
        df["momentum_intensity_pct"] +
        df["range_expansion_pct"]
    ) / 2
    * 100
)

df["volume_score"] = (
    df["relative_volume_pct"] *
    100
)

df["displacement_score"] = (
    df[
        [
            "price_displacement_pct",
            "distance_high20_pct",
            "distance_low20_pct"
        ]
    ]
    .mean(axis=1)
    * 100
)

# ---------------------------------------------------------
# 10. Overall continuous opportunity score
# ---------------------------------------------------------
#
# This is deliberately NOT based on direction.
#
# It asks:
# "How unusual / energetic is the current market?"
# ---------------------------------------------------------

component_columns = [
    "volatility_score",
    "momentum_energy_score",
    "volume_score",
    "displacement_score"
]

df["continuous_opportunity"] = (
    df[component_columns]
    .mean(axis=1)
)

# ---------------------------------------------------------
# 11. Opportunity intensity bands
# ---------------------------------------------------------

df["opportunity_band"] = pd.cut(
    df["continuous_opportunity"],
    bins=[
        -np.inf,
        20,
        40,
        60,
        80,
        np.inf
    ],
    labels=[
        "VERY_LOW",
        "LOW",
        "MEDIUM",
        "HIGH",
        "VERY_HIGH"
    ]
)

# ---------------------------------------------------------
# 12. Direction-neutral future target
# ---------------------------------------------------------

df["future_move_magnitude"] = (
    df["future_abs_24h"]
)

# ---------------------------------------------------------
# 13. Walk-forward validation
# ---------------------------------------------------------

TRAIN_SIZE = 4000
TEST_SIZE = 1000
STEP = 1000

windows = []

start = 0

while (
    start +
    TRAIN_SIZE +
    TEST_SIZE
    <= len(df)
):

    train = df.iloc[
        start:
        start + TRAIN_SIZE
    ]

    test = df.iloc[
        start + TRAIN_SIZE:
        start + TRAIN_SIZE + TEST_SIZE
    ].copy()

    # -----------------------------------------------------
    # Training-only large move threshold
    # -----------------------------------------------------

    train_future = (
        train["future_move_magnitude"]
        .dropna()
    )

    if len(train_future) < 500:
        start += STEP
        continue

    large_threshold = (
        train_future.quantile(0.75)
    )

    # -----------------------------------------------------
    # Valid test rows
    # -----------------------------------------------------

    test = test[
        test["future_move_magnitude"].notna() &
        test["continuous_opportunity"].notna()
    ]

    if len(test) == 0:
        start += STEP
        continue

    # -----------------------------------------------------
    # Baseline
    # -----------------------------------------------------

    baseline_mean = (
        test["future_move_magnitude"]
        .mean()
    )

    baseline_large = (
        test["future_move_magnitude"]
        >= large_threshold
    ).mean()

    # -----------------------------------------------------
    # Score buckets
    # -----------------------------------------------------

    bands = {
        "LOW": (
            test["continuous_opportunity"] < 40
        ),

        "MEDIUM": (
            (test["continuous_opportunity"] >= 40) &
            (test["continuous_opportunity"] < 60)
        ),

        "HIGH": (
            (test["continuous_opportunity"] >= 60) &
            (test["continuous_opportunity"] < 80)
        ),

        "VERY_HIGH": (
            test["continuous_opportunity"] >= 80
        )
    }

    row = {

        "test_rows":
            len(test),

        "baseline_mean":
            baseline_mean,

        "baseline_large":
            baseline_large
    }

    for name, condition in bands.items():

        subset = test[condition]

        row[name + "_count"] = len(
            subset
        )

        if len(subset) > 0:

            row[name + "_mean"] = (
                subset[
                    "future_move_magnitude"
                ].mean()
            )

            row[name + "_large"] = (
                subset[
                    "future_move_magnitude"
                ] >=
                large_threshold
            ).mean()

        else:

            row[name + "_mean"] = np.nan
            row[name + "_large"] = np.nan

    windows.append(row)

    start += STEP

results = pd.DataFrame(
    windows
)

print(
    f"\nWalk-forward windows: "
    f"{len(results)}"
)

# ---------------------------------------------------------
# 14. Weighted average helper
# ---------------------------------------------------------

def weighted_mean(
    values,
    weights
):

    values = np.asarray(
        values,
        dtype=float
    )

    weights = np.asarray(
        weights,
        dtype=float
    )

    valid = (
        np.isfinite(values) &
        np.isfinite(weights) &
        (weights > 0)
    )

    if valid.sum() == 0:
        return np.nan

    return np.average(
        values[valid],
        weights=weights[valid]
    )

# ---------------------------------------------------------
# 15. Overall baseline
# ---------------------------------------------------------

baseline_mean = weighted_mean(
    results["baseline_mean"],
    results["test_rows"]
)

baseline_large = weighted_mean(
    results["baseline_large"],
    results["test_rows"]
)

# ---------------------------------------------------------
# 16. Print validation
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("CONTINUOUS OPPORTUNITY RESULTS")
print("=" * 70)

print("\nBASELINE")
print("-" * 50)

print(
    f"Average 24h movement: "
    f"{baseline_mean * 100:.3f}%"
)

print(
    f"Large-move rate: "
    f"{baseline_large * 100:.2f}%"
)

for band in [
    "LOW",
    "MEDIUM",
    "HIGH",
    "VERY_HIGH"
]:

    count = (
        results[
            band + "_count"
        ].sum()
    )

    mean = weighted_mean(
        results[
            band + "_mean"
        ],
        results[
            band + "_count"
        ]
    )

    large = weighted_mean(
        results[
            band + "_large"
        ],
        results[
            band + "_count"
        ]
    )

    print(
        f"\n{band}"
    )

    print("-" * 50)

    print(
        f"Signals: "
        f"{count:,.0f}"
    )

    if np.isfinite(mean):

        print(
            f"Average 24h movement: "
            f"{mean * 100:.3f}%"
        )

        print(
            f"Vs baseline: "
            f"{(mean - baseline_mean) * 100:+.3f} pp"
        )

        print(
            f"Large-move rate: "
            f"{large * 100:.2f}%"
        )

        print(
            f"Large-move improvement: "
            f"{(large - baseline_large) * 100:+.2f} pp"
        )

# ---------------------------------------------------------
# 17. Current machine state
# ---------------------------------------------------------

latest = df.iloc[-1]

print("\n" + "=" * 70)
print("CURRENT CONTINUOUS MARKET STATE")
print("=" * 70)

print(
    f"\nTimestamp: "
    f"{df.index[-1]}"
)

print(
    f"Price: "
    f"{latest['close']:.4f}"
)

print(
    f"\nContinuous opportunity: "
    f"{latest['continuous_opportunity']:.1f}/100"
)

print(
    f"Band: "
    f"{latest['opportunity_band']}"
)

print(
    f"\nVolatility score: "
    f"{latest['volatility_score']:.1f}"
)

print(
    f"Momentum energy: "
    f"{latest['momentum_energy_score']:.1f}"
)

print(
    f"Volume score: "
    f"{latest['volume_score']:.1f}"
)

print(
    f"Displacement score: "
    f"{latest['displacement_score']:.1f}"
)

print(
    f"\nVolatility expansion: "
    f"{latest['range_expansion']:.2f}x"
)

print(
    f"Momentum intensity: "
    f"{latest['momentum_intensity'] * 100:.3f}%"
)

print(
    f"Price displacement: "
    f"{latest['price_displacement'] * 100:.3f}%"
)

# ---------------------------------------------------------
# 18. Save
# ---------------------------------------------------------

HISTORY_PATH = (
    RESULTS_DIR +
    "/continuous_opportunity_v2_history.parquet"
)

WINDOW_PATH = (
    RESULTS_DIR +
    "/continuous_opportunity_v2_walk_forward.parquet"
)

df.to_parquet(
    HISTORY_PATH
)

results.to_parquet(
    WINDOW_PATH,
    index=False
)

print("\n" + "=" * 70)
print("✅ CONTINUOUS OPPORTUNITY V2 COMPLETE")
print("=" * 70)

print(
    f"\nHistory saved:"
)

print(HISTORY_PATH)

print(
    f"\nWalk-forward results saved:"
)

print(WINDOW_PATH)

In [ ]:
# =========================================================
# SILVER AGENT — STEP 20
# FIRST MACHINE-LEARNING OPPORTUNITY MODEL
# =========================================================

import os
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor

# ---------------------------------------------------------
# 1. Paths
# ---------------------------------------------------------

FEATURE_PATH = (
    "/content/drive/MyDrive/"
    "SilverAgent/database/"
    "silver_h1_features_v2.parquet"
)

CONTINUOUS_PATH = (
    "/content/drive/MyDrive/"
    "SilverAgent/results/"
    "continuous_opportunity_v2_history.parquet"
)

RESULTS_DIR = (
    "/content/drive/MyDrive/"
    "SilverAgent/results"
)

os.makedirs(
    RESULTS_DIR,
    exist_ok=True
)

# ---------------------------------------------------------
# 2. Load data
# ---------------------------------------------------------

features = pd.read_parquet(
    FEATURE_PATH
).sort_index()

continuous = pd.read_parquet(
    CONTINUOUS_PATH
).sort_index()

print("=" * 70)
print("STEP 20 — MACHINE LEARNING OPPORTUNITY MODEL")
print("=" * 70)

print(
    f"\nFeature rows: {len(features):,}"
)

print(
    f"Continuous rows: {len(continuous):,}"
)

# ---------------------------------------------------------
# 3. Combine
# ---------------------------------------------------------

df = features.join(
    continuous[
        [
            "continuous_opportunity",
            "volatility_score",
            "momentum_energy_score",
            "volume_score",
            "displacement_score"
        ]
    ],
    how="inner"
)

# ---------------------------------------------------------
# 4. Target
# ---------------------------------------------------------
#
# Predict the magnitude of the next 24h move.
#
# Direction is deliberately ignored.
# ---------------------------------------------------------

target = "future_abs_24h"

# ---------------------------------------------------------
# 5. Model features
# ---------------------------------------------------------

model_features = [
    "vol10",
    "vol20",
    "atr_percent",
    "relative_volume",
    "distance_high20",
    "distance_low20",
    "price_vs_MA20",
    "mom3",
    "mom6",
    "mom12",
    "momentum_intensity",
    "range_expansion",
    "price_displacement",
    "atr_vol_ratio",
    "volatility_score",
    "momentum_energy_score",
    "volume_score",
    "displacement_score"
]

available_features = [
    col
    for col in model_features
    if col in df.columns
]

print(
    f"\nModel features available: "
    f"{len(available_features)}"
)

print(
    available_features
)

# ---------------------------------------------------------
# 6. Clean model dataset
# ---------------------------------------------------------

model_df = df[
    available_features + [target]
].copy()

model_df = model_df.replace(
    [np.inf, -np.inf],
    np.nan
)

model_df = model_df.dropna(
    subset=available_features + [target]
)

print(
    f"\nUsable ML rows: "
    f"{len(model_df):,}"
)

# ---------------------------------------------------------
# 7. Walk-forward configuration
# ---------------------------------------------------------

TRAIN_SIZE = 4000
TEST_SIZE = 1000
STEP = 1000

windows = []

start = 0

# ---------------------------------------------------------
# 8. Walk-forward ML testing
# ---------------------------------------------------------

while (
    start +
    TRAIN_SIZE +
    TEST_SIZE
    <= len(model_df)
):

    train = model_df.iloc[
        start:
        start + TRAIN_SIZE
    ]

    test = model_df.iloc[
        start + TRAIN_SIZE:
        start + TRAIN_SIZE + TEST_SIZE
    ]

    X_train = train[
        available_features
    ]

    y_train = train[
        target
    ]

    X_test = test[
        available_features
    ]

    y_test = test[
        target
    ]

    # -----------------------------------------------------
    # Train model
    # -----------------------------------------------------

    model = RandomForestRegressor(
        n_estimators=150,
        max_depth=8,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    # -----------------------------------------------------
    # Predict future movement magnitude
    # -----------------------------------------------------

    predictions = model.predict(
        X_test
    )

    test_result = test.copy()

    test_result[
        "ml_prediction"
    ] = predictions

    # -----------------------------------------------------
    # Training-only thresholds
    # -----------------------------------------------------

    train_prediction_target = (
        model.predict(X_train)
    )

    prediction_threshold = np.quantile(
        train_prediction_target,
        0.80
    )

    actual_large_threshold = np.quantile(
        y_train,
        0.75
    )

    # -----------------------------------------------------
    # ML selected opportunities
    # -----------------------------------------------------

    selected = test_result[
        test_result["ml_prediction"]
        >= prediction_threshold
    ]

    # -----------------------------------------------------
    # Baseline
    # -----------------------------------------------------

    baseline_mean = (
        y_test.mean()
    )

    baseline_large = (
        y_test >= actual_large_threshold
    ).mean()

    # -----------------------------------------------------
    # ML metrics
    # -----------------------------------------------------

    if len(selected) > 0:

        selected_mean = (
            selected[target].mean()
        )

        selected_large = (
            selected[target]
            >= actual_large_threshold
        ).mean()

    else:

        selected_mean = np.nan
        selected_large = np.nan

    # -----------------------------------------------------
    # Correlation
    # -----------------------------------------------------

    if (
        np.std(predictions) > 0 and
        np.std(y_test.values) > 0
    ):

        correlation = np.corrcoef(
            predictions,
            y_test.values
        )[0, 1]

    else:

        correlation = np.nan

    # -----------------------------------------------------
    # Feature importance
    # -----------------------------------------------------

    importance = pd.Series(
        model.feature_importances_,
        index=available_features
    )

    top_features = (
        importance
        .sort_values(
            ascending=False
        )
        .head(5)
    )

    windows.append(
        {
            "window_start":
                start,

            "train_rows":
                len(train),

            "test_rows":
                len(test),

            "baseline_mean":
                baseline_mean,

            "baseline_large":
                baseline_large,

            "selected_count":
                len(selected),

            "selected_mean":
                selected_mean,

            "selected_large":
                selected_large,

            "mean_improvement":
                selected_mean -
                baseline_mean,

            "large_improvement":
                selected_large -
                baseline_large,

            "prediction_correlation":
                correlation,

            "top_feature_1":
                top_features.index[0],

            "top_feature_2":
                top_features.index[1],

            "top_feature_3":
                top_features.index[2],

            "top_feature_4":
                top_features.index[3],

            "top_feature_5":
                top_features.index[4]
        }
    )

    print(
        f"Window {len(windows):2d} | "
        f"baseline "
        f"{baseline_mean * 100:.3f}% | "
        f"ML selected "
        f"{selected_mean * 100:.3f}% | "
        f"signals "
        f"{len(selected):4d}"
    )

    start += STEP

# ---------------------------------------------------------
# 9. Results dataframe
# ---------------------------------------------------------

results = pd.DataFrame(
    windows
)

# ---------------------------------------------------------
# 10. Weighted summary
# ---------------------------------------------------------

def weighted_average(
    values,
    weights
):

    values = np.asarray(
        values,
        dtype=float
    )

    weights = np.asarray(
        weights,
        dtype=float
    )

    valid = (
        np.isfinite(values) &
        np.isfinite(weights) &
        (weights > 0)
    )

    if valid.sum() == 0:
        return np.nan

    return np.average(
        values[valid],
        weights=weights[valid]
    )

baseline_mean = weighted_average(
    results["baseline_mean"],
    results["test_rows"]
)

selected_mean = weighted_average(
    results["selected_mean"],
    results["selected_count"]
)

baseline_large = weighted_average(
    results["baseline_large"],
    results["test_rows"]
)

selected_large = weighted_average(
    results["selected_large"],
    results["selected_count"]
)

correlation = weighted_average(
    results["prediction_correlation"],
    results["test_rows"]
)

positive_windows = (
    results["mean_improvement"] > 0
).sum()

total_windows = len(
    results
)

# ---------------------------------------------------------
# 11. Final report
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("MACHINE LEARNING RESULTS")
print("=" * 70)

print(
    f"\nWalk-forward windows: "
    f"{total_windows}"
)

print(
    f"Windows beating baseline: "
    f"{positive_windows}/"
    f"{total_windows}"
)

print(
    f"\nBaseline average 24h move: "
    f"{baseline_mean * 100:.3f}%"
)

print(
    f"ML-selected average 24h move: "
    f"{selected_mean * 100:.3f}%"
)

print(
    f"Improvement: "
    f"{(selected_mean - baseline_mean) * 100:+.3f} pp"
)

print(
    f"\nBaseline large-move rate: "
    f"{baseline_large * 100:.2f}%"
)

print(
    f"ML-selected large-move rate: "
    f"{selected_large * 100:.2f}%"
)

print(
    f"Improvement: "
    f"{(selected_large - baseline_large) * 100:+.2f} pp"
)

print(
    f"\nPrediction correlation: "
    f"{correlation:.3f}"
)

# ---------------------------------------------------------
# 12. Current prediction
# ---------------------------------------------------------

latest = model_df.iloc[
    [-1]
]

latest_features = latest[
    available_features
]

# Retrain on all currently available history
final_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

final_model.fit(
    model_df[
        available_features
    ],
    model_df[
        target
    ]
)

current_prediction = (
    final_model.predict(
        latest_features
    )[0]
)

# Historical prediction distribution
all_training_predictions = (
    final_model.predict(
        model_df[
            available_features
        ]
    )
)

current_percentile = (
    (
        all_training_predictions
        <= current_prediction
    ).mean()
    * 100
)

print("\n" + "=" * 70)
print("CURRENT ML MARKET STATE")
print("=" * 70)

print(
    f"\nTimestamp: "
    f"{model_df.index[-1]}"
)

print(
    f"Price: "
    f"{df.loc[model_df.index[-1], 'close']:.4f}"
)

print(
    f"\nML predicted 24h movement: "
    f"{current_prediction * 100:.3f}%"
)

print(
    f"Historical prediction percentile: "
    f"{current_percentile:.1f}%"
)

if current_percentile >= 80:

    current_zone = "VERY HIGH"

elif current_percentile >= 60:

    current_zone = "HIGH"

elif current_percentile >= 40:

    current_zone = "MEDIUM"

elif current_percentile >= 20:

    current_zone = "LOW"

else:

    current_zone = "VERY LOW"

print(
    f"ML opportunity zone: "
    f"{current_zone}"
)

# ---------------------------------------------------------
# 13. Feature importance
# ---------------------------------------------------------

importance = pd.Series(
    final_model.feature_importances_,
    index=available_features
)

importance = (
    importance
    .sort_values(
        ascending=False
    )
)

print("\nTop learned features:")
print("-" * 50)

for feature, value in (
    importance.head(10).items()
):

    print(
        f"{feature:<28} "
        f"{value:.4f}"
    )

# ---------------------------------------------------------
# 14. Save
# ---------------------------------------------------------

RESULTS_PATH = (
    RESULTS_DIR +
    "/opportunity_ml_v1_walk_forward.parquet"
)

IMPORTANCE_PATH = (
    RESULTS_DIR +
    "/opportunity_ml_v1_feature_importance.parquet"
)

results.to_parquet(
    RESULTS_PATH,
    index=False
)

importance.to_frame(
    "importance"
).to_parquet(
    IMPORTANCE_PATH
)

print("\n" + "=" * 70)
print("✅ STEP 20 COMPLETE")
print("=" * 70)

print(
    f"\nWalk-forward results saved:"
)

print(
    RESULTS_PATH
)

print(
    f"\nFeature importance saved:"
)

print(
    IMPORTANCE_PATH
)

In [ ]:

from google.colab import drive
import os

MOUNT_POINT = "/content/gdrive"

# Create the mount point only if it doesn't exist
if not os.path.exists(MOUNT_POINT):
    os.makedirs(MOUNT_POINT)

print("Mount point ready:", MOUNT_POINT)

drive.mount(
    MOUNT_POINT
)

In [ ]:
import os

RESULTS_DIR = (
    "/content/gdrive/MyDrive/"
    "SilverAgent/results"
)

DATABASE_DIR = (
    "/content/gdrive/MyDrive/"
    "SilverAgent/database"
)

print("=" * 70)
print("SILVER AGENT — DRIVE RECOVERY")
print("=" * 70)

print("\nRESULTS:")

if os.path.exists(RESULTS_DIR):

    for f in sorted(
        os.listdir(RESULTS_DIR)
    ):
        print("✓", f)

else:

    print("❌ Results folder not found")

print("\nDATABASE:")

if os.path.exists(DATABASE_DIR):

    for f in sorted(
        os.listdir(DATABASE_DIR)
    ):
        print("✓", f)

else:

    print("❌ Database folder not found")

In [ ]:
# =========================================================
# SILVER AGENT — STEP 23
# ENGINE SCORE ROBUSTNESS GRID
# =========================================================

import os
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor

# ---------------------------------------------------------
# 1. Paths
# ---------------------------------------------------------

FEATURE_PATH = (
    "/content/gdrive/MyDrive/"
    "SilverAgent/database/"
    "silver_h1_features_v2.parquet"
)

CONTINUOUS_PATH = (
    "/content/gdrive/MyDrive/"
    "SilverAgent/results/"
    "continuous_opportunity_v2_history.parquet"
)

RESULTS_DIR = (
    "/content/gdrive/MyDrive/"
    "SilverAgent/results"
)

os.makedirs(
    RESULTS_DIR,
    exist_ok=True
)

# ---------------------------------------------------------
# 2. Load data
# ---------------------------------------------------------

features = pd.read_parquet(
    FEATURE_PATH
).sort_index()

continuous = pd.read_parquet(
    CONTINUOUS_PATH
).sort_index()

df = features.join(
    continuous[
        [
            "volatility_score",
            "momentum_energy_score",
            "volume_score",
            "displacement_score"
        ]
    ],
    how="inner"
)

print("=" * 70)
print("STEP 23 — ENGINE SCORE ROBUSTNESS GRID")
print("=" * 70)

print(
    f"\nRows loaded: {len(df):,}"
)

# ---------------------------------------------------------
# 3. Engine features
# ---------------------------------------------------------

engine_features = [
    "volatility_score",
    "momentum_energy_score",
    "volume_score",
    "displacement_score"
]

target = "future_abs_24h"

work = df[
    engine_features + [target]
].replace(
    [np.inf, -np.inf],
    np.nan
).dropna()

print(
    f"Usable rows: {len(work):,}"
)

# ---------------------------------------------------------
# 4. Configurations
# ---------------------------------------------------------
#
# selection_level:
#
# 0.70 = top 30%
# 0.80 = top 20%
# 0.90 = top 10%
#
# We deliberately test several values rather than
# choosing one because it looked best previously.
# ---------------------------------------------------------

TRAIN_SIZES = [
    2000,
    4000,
    6000,
    8000
]

SELECTION_LEVELS = [
    0.70,
    0.80,
    0.90
]

TEST_SIZE = 1000
STEP = 1000

all_results = []

# ---------------------------------------------------------
# 5. Run robustness grid
# ---------------------------------------------------------

for train_size in TRAIN_SIZES:

    for selection_level in SELECTION_LEVELS:

        top_percent = int(
            (1 - selection_level) * 100
        )

        print("\n" + "=" * 70)

        print(
            f"TRAIN={train_size} | "
            f"TOP={top_percent}%"
        )

        print("=" * 70)

        start = 0
        window = 0

        while (
            start +
            train_size +
            TEST_SIZE
            <= len(work)
        ):

            train = work.iloc[
                start:
                start + train_size
            ]

            test = work.iloc[
                start + train_size:
                start + train_size + TEST_SIZE
            ]

            # -------------------------------------------------
            # Train model
            # -------------------------------------------------

            model = RandomForestRegressor(
                n_estimators=120,
                max_depth=8,
                min_samples_leaf=20,
                random_state=42,
                n_jobs=-1
            )

            model.fit(
                train[engine_features],
                train[target]
            )

            # -------------------------------------------------
            # Predictions
            # -------------------------------------------------

            train_prediction = model.predict(
                train[engine_features]
            )

            test_prediction = model.predict(
                test[engine_features]
            )

            # -------------------------------------------------
            # Training-only selection threshold
            # -------------------------------------------------

            prediction_threshold = np.quantile(
                train_prediction,
                selection_level
            )

            # -------------------------------------------------
            # Training-only large-move threshold
            # -------------------------------------------------

            large_threshold = np.quantile(
                train[target],
                0.75
            )

            # -------------------------------------------------
            # Select opportunities
            # -------------------------------------------------

            selected = test[
                test_prediction >=
                prediction_threshold
            ]

            # -------------------------------------------------
            # Baseline
            # -------------------------------------------------

            baseline_mean = (
                test[target].mean()
            )

            baseline_large = (
                test[target]
                >= large_threshold
            ).mean()

            # -------------------------------------------------
            # Selected metrics
            # -------------------------------------------------

            if len(selected) > 0:

                selected_mean = (
                    selected[target].mean()
                )

                selected_large = (
                    selected[target]
                    >= large_threshold
                ).mean()

            else:

                selected_mean = np.nan
                selected_large = np.nan

            # -------------------------------------------------
            # Store result
            # -------------------------------------------------

            all_results.append(
                {
                    "train_size":
                        train_size,

                    "selection_level":
                        selection_level,

                    "window":
                        window + 1,

                    "test_rows":
                        len(test),

                    "signals":
                        len(selected),

                    "baseline_mean":
                        baseline_mean,

                    "selected_mean":
                        selected_mean,

                    "mean_improvement":
                        selected_mean -
                        baseline_mean,

                    "baseline_large":
                        baseline_large,

                    "selected_large":
                        selected_large,

                    "large_improvement":
                        selected_large -
                        baseline_large
                }
            )

            window += 1
            start += STEP

# ---------------------------------------------------------
# 6. Results dataframe
# ---------------------------------------------------------

results = pd.DataFrame(
    all_results
)

# ---------------------------------------------------------
# 7. Weighted-average helper
# ---------------------------------------------------------

def weighted_average(
    values,
    weights
):

    values = np.asarray(
        values,
        dtype=float
    )

    weights = np.asarray(
        weights,
        dtype=float
    )

    valid = (
        np.isfinite(values) &
        np.isfinite(weights) &
        (weights > 0)
    )

    if valid.sum() == 0:
        return np.nan

    return np.average(
        values[valid],
        weights=weights[valid]
    )

# ---------------------------------------------------------
# 8. Configuration summary
# ---------------------------------------------------------

summary_rows = []

for (
    train_size,
    selection_level
), group in results.groupby(
    [
        "train_size",
        "selection_level"
    ]
):

    selected_mean = weighted_average(
        group["selected_mean"],
        group["signals"]
    )

    baseline_mean = weighted_average(
        group["baseline_mean"],
        group["test_rows"]
    )

    selected_large = weighted_average(
        group["selected_large"],
        group["signals"]
    )

    baseline_large = weighted_average(
        group["baseline_large"],
        group["test_rows"]
    )

    positive_windows = (
        group["mean_improvement"] > 0
    ).sum()

    total_windows = len(
        group
    )

    summary_rows.append(
        {
            "train_size":
                train_size,

            "selection_level":
                selection_level,

            "signals":
                group["signals"].sum(),

            "selected_mean":
                selected_mean,

            "baseline_mean":
                baseline_mean,

            "mean_improvement":
                selected_mean -
                baseline_mean,

            "selected_large":
                selected_large,

            "baseline_large":
                baseline_large,

            "large_improvement":
                selected_large -
                baseline_large,

            "positive_windows":
                positive_windows,

            "total_windows":
                total_windows,

            "consistency":
                positive_windows /
                total_windows
        }
    )

summary = pd.DataFrame(
    summary_rows
)

# ---------------------------------------------------------
# 9. Print results
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("ROBUSTNESS GRID RESULTS")
print("=" * 70)

for _, row in summary.iterrows():

    top_percent = int(
        (1 - row["selection_level"]) * 100
    )

    print(
        f"\nTrain {int(row['train_size']):5d} | "
        f"Top {top_percent:2d}%"
    )

    print(
        f"  Signals: "
        f"{int(row['signals']):,}"
    )

    print(
        f"  Selected move: "
        f"{row['selected_mean'] * 100:.3f}%"
    )

    print(
        f"  Improvement: "
        f"{row['mean_improvement'] * 100:+.3f} pp"
    )

    print(
        f"  Large moves: "
        f"{row['selected_large'] * 100:.2f}%"
    )

    print(
        f"  Large improvement: "
        f"{row['large_improvement'] * 100:+.2f} pp"
    )

    print(
        f"  Positive windows: "
        f"{int(row['positive_windows'])}/"
        f"{int(row['total_windows'])}"
    )

# ---------------------------------------------------------
# 10. Rank configurations
# ---------------------------------------------------------

ranking = summary.sort_values(
    [
        "mean_improvement",
        "large_improvement"
    ],
    ascending=False
)

print("\n" + "=" * 70)
print("TOP CONFIGURATIONS")
print("=" * 70)

for i, (_, row) in enumerate(
    ranking.head(10).iterrows(),
    start=1
):

    top_percent = int(
        (1 - row["selection_level"]) * 100
    )

    print(
        f"{i}. "
        f"Train {int(row['train_size'])} | "
        f"Top {top_percent}% | "
        f"Move "
        f"{row['mean_improvement'] * 100:+.3f} pp | "
        f"Large "
        f"{row['large_improvement'] * 100:+.2f} pp | "
        f"Consistency "
        f"{int(row['positive_windows'])}/"
        f"{int(row['total_windows'])}"
    )

# ---------------------------------------------------------
# 11. Stability check
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("STABILITY CHECK")
print("=" * 70)

for train_size in TRAIN_SIZES:

    subset = summary[
        summary["train_size"] ==
        train_size
    ]

    print(
        f"\nTraining size {train_size}:"
    )

    for _, row in subset.iterrows():

        top_percent = int(
            (1 - row["selection_level"]) * 100
        )

        print(
            f"  Top {top_percent}%: "
            f"{row['mean_improvement'] * 100:+.3f} pp | "
            f"{row['large_improvement'] * 100:+.2f} pp"
        )

# ---------------------------------------------------------
# 12. Save
# ---------------------------------------------------------

DETAIL_PATH = (
    RESULTS_DIR +
    "/engine_score_robustness_v1.parquet"
)

SUMMARY_PATH = (
    RESULTS_DIR +
    "/engine_score_robustness_v1_summary.parquet"
)

results.to_parquet(
    DETAIL_PATH,
    index=False
)

summary.to_parquet(
    SUMMARY_PATH,
    index=False
)

print("\n" + "=" * 70)
print("STEP 23 COMPLETE")
print("=" * 70)

print(
    "\nDetailed results saved:"
)

print(
    DETAIL_PATH
)

print(
    "\nSummary saved:"
)

print(
    SUMMARY_PATH
)

In [ ]:
# =========================================================
# SILVER AGENT — STEP 24
# OPPORTUNITY HORIZON ANALYSIS
# =========================================================

import os
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor

# ---------------------------------------------------------
# 1. Paths
# ---------------------------------------------------------

FEATURE_PATH = (
    "/content/gdrive/MyDrive/"
    "SilverAgent/database/"
    "silver_h1_features_v2.parquet"
)

CONTINUOUS_PATH = (
    "/content/gdrive/MyDrive/"
    "SilverAgent/results/"
    "continuous_opportunity_v2_history.parquet"
)

RESULTS_DIR = (
    "/content/gdrive/MyDrive/"
    "SilverAgent/results"
)

os.makedirs(
    RESULTS_DIR,
    exist_ok=True
)

# ---------------------------------------------------------
# 2. Load data
# ---------------------------------------------------------

features = pd.read_parquet(
    FEATURE_PATH
).sort_index()

continuous = pd.read_parquet(
    CONTINUOUS_PATH
).sort_index()

df = features.join(
    continuous[
        [
            "volatility_score",
            "momentum_energy_score",
            "volume_score",
            "displacement_score"
        ]
    ],
    how="inner"
)

print("=" * 70)
print("STEP 24 — OPPORTUNITY HORIZON ANALYSIS")
print("=" * 70)

print(
    f"\nRows loaded: {len(df):,}"
)

# ---------------------------------------------------------
# 3. Engine features
# ---------------------------------------------------------

engine_features = [
    "volatility_score",
    "momentum_energy_score",
    "volume_score",
    "displacement_score"
]

# ---------------------------------------------------------
# 4. Horizon targets
# ---------------------------------------------------------
#
# These are already calculated by the gap-aware feature
# engine using real candles.
#
# We use absolute movement because this is an opportunity
# detector, not a directional model.
# ---------------------------------------------------------

horizon_targets = {
    "6H": "future_abs_6h",
    "12H": "future_abs_12h",
    "24H": "future_abs_24h",
    "48H": "future_abs_48h"
}

# ---------------------------------------------------------
# 5. Check available targets
# ---------------------------------------------------------

print("\nChecking horizon targets...")

for label, column in horizon_targets.items():

    if column in df.columns:

        valid = df[column].notna().sum()

        print(
            f"  ✓ {label}: "
            f"{valid:,} valid rows"
        )

    else:

        print(
            f"  ❌ {label}: "
            f"{column} NOT FOUND"
        )

# ---------------------------------------------------------
# 6. Walk-forward configuration
# ---------------------------------------------------------

TRAIN_SIZE = 4000
TEST_SIZE = 1000
STEP = 1000

# We deliberately use the same top 20% opportunity
# selection used in the earlier fair comparison.
#
# This is NOT being optimized here.
SELECTION_LEVEL = 0.80

all_results = []

# ---------------------------------------------------------
# 7. Test each horizon independently
# ---------------------------------------------------------

for horizon, target in horizon_targets.items():

    print("\n" + "=" * 70)
    print(
        f"TESTING HORIZON: {horizon}"
    )
    print("=" * 70)

    required = (
        engine_features +
        [target]
    )

    work = df[
        required
    ].replace(
        [np.inf, -np.inf],
        np.nan
    ).dropna()

    print(
        f"Usable rows: {len(work):,}"
    )

    start = 0
    window = 0

    while (
        start +
        TRAIN_SIZE +
        TEST_SIZE
        <= len(work)
    ):

        train = work.iloc[
            start:
            start + TRAIN_SIZE
        ]

        test = work.iloc[
            start + TRAIN_SIZE:
            start + TRAIN_SIZE + TEST_SIZE
        ]

        # -------------------------------------------------
        # Train model
        # -------------------------------------------------

        model = RandomForestRegressor(
            n_estimators=120,
            max_depth=8,
            min_samples_leaf=20,
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            train[engine_features],
            train[target]
        )

        # -------------------------------------------------
        # Predictions
        # -------------------------------------------------

        train_prediction = model.predict(
            train[engine_features]
        )

        test_prediction = model.predict(
            test[engine_features]
        )

        # -------------------------------------------------
        # Training-only opportunity threshold
        # -------------------------------------------------

        prediction_threshold = np.quantile(
            train_prediction,
            SELECTION_LEVEL
        )

        # -------------------------------------------------
        # Training-only large-move threshold
        # -------------------------------------------------

        large_threshold = np.quantile(
            train[target],
            0.75
        )

        # -------------------------------------------------
        # Select top 20%
        # -------------------------------------------------

        selected = test[
            test_prediction >=
            prediction_threshold
        ]

        # -------------------------------------------------
        # Baseline
        # -------------------------------------------------

        baseline_mean = (
            test[target].mean()
        )

        baseline_median = (
            test[target].median()
        )

        baseline_large = (
            test[target]
            >= large_threshold
        ).mean()

        # -------------------------------------------------
        # Selected opportunity metrics
        # -------------------------------------------------

        if len(selected) > 0:

            selected_mean = (
                selected[target].mean()
            )

            selected_median = (
                selected[target].median()
            )

            selected_large = (
                selected[target]
                >= large_threshold
            ).mean()

        else:

            selected_mean = np.nan
            selected_median = np.nan
            selected_large = np.nan

        all_results.append(
            {
                "horizon":
                    horizon,

                "window":
                    window + 1,

                "train_size":
                    TRAIN_SIZE,

                "test_rows":
                    len(test),

                "signals":
                    len(selected),

                "baseline_mean":
                    baseline_mean,

                "selected_mean":
                    selected_mean,

                "mean_improvement":
                    selected_mean -
                    baseline_mean,

                "baseline_median":
                    baseline_median,

                "selected_median":
                    selected_median,

                "baseline_large":
                    baseline_large,

                "selected_large":
                    selected_large,

                "large_improvement":
                    selected_large -
                    baseline_large
            }
        )

        print(
            f"Window {window + 1:2d} | "
            f"Base "
            f"{baseline_mean * 100:.3f}% | "
            f"Selected "
            f"{selected_mean * 100:.3f}% | "
            f"Signals "
            f"{len(selected):4d}"
        )

        window += 1
        start += STEP

# ---------------------------------------------------------
# 8. Results dataframe
# ---------------------------------------------------------

results = pd.DataFrame(
    all_results
)

# ---------------------------------------------------------
# 9. Weighted-average helper
# ---------------------------------------------------------

def weighted_average(
    values,
    weights
):

    values = np.asarray(
        values,
        dtype=float
    )

    weights = np.asarray(
        weights,
        dtype=float
    )

    valid = (
        np.isfinite(values) &
        np.isfinite(weights) &
        (weights > 0)
    )

    if valid.sum() == 0:
        return np.nan

    return np.average(
        values[valid],
        weights=weights[valid]
    )

# ---------------------------------------------------------
# 10. Horizon summary
# ---------------------------------------------------------

summary_rows = []

for horizon in horizon_targets.keys():

    group = results[
        results["horizon"] ==
        horizon
    ]

    baseline_mean = weighted_average(
        group["baseline_mean"],
        group["test_rows"]
    )

    selected_mean = weighted_average(
        group["selected_mean"],
        group["signals"]
    )

    baseline_median = weighted_average(
        group["baseline_median"],
        group["test_rows"]
    )

    selected_median = weighted_average(
        group["selected_median"],
        group["signals"]
    )

    baseline_large = weighted_average(
        group["baseline_large"],
        group["test_rows"]
    )

    selected_large = weighted_average(
        group["selected_large"],
        group["signals"]
    )

    positive_windows = (
        group["mean_improvement"] > 0
    ).sum()

    total_windows = len(
        group
    )

    summary_rows.append(
        {
            "horizon":
                horizon,

            "signals":
                group["signals"].sum(),

            "baseline_mean":
                baseline_mean,

            "selected_mean":
                selected_mean,

            "mean_improvement":
                selected_mean -
                baseline_mean,

            "baseline_median":
                baseline_median,

            "selected_median":
                selected_median,

            "baseline_large":
                baseline_large,

            "selected_large":
                selected_large,

            "large_improvement":
                selected_large -
                baseline_large,

            "positive_windows":
                positive_windows,

            "total_windows":
                total_windows,

            "consistency":
                positive_windows /
                total_windows
        }
    )

summary = pd.DataFrame(
    summary_rows
)

# ---------------------------------------------------------
# 11. Final comparison
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("OPPORTUNITY HORIZON RESULTS")
print("=" * 70)

for _, row in summary.iterrows():

    print(
        f"\n{row['horizon']}"
    )

    print("-" * 50)

    print(
        f"Signals: "
        f"{int(row['signals']):,}"
    )

    print(
        f"Baseline mean movement: "
        f"{row['baseline_mean'] * 100:.3f}%"
    )

    print(
        f"Selected mean movement: "
        f"{row['selected_mean'] * 100:.3f}%"
    )

    print(
        f"Mean improvement: "
        f"{row['mean_improvement'] * 100:+.3f} pp"
    )

    print(
        f"Baseline median: "
        f"{row['baseline_median'] * 100:.3f}%"
    )

    print(
        f"Selected median: "
        f"{row['selected_median'] * 100:.3f}%"
    )

    print(
        f"Baseline large-move rate: "
        f"{row['baseline_large'] * 100:.2f}%"
    )

    print(
        f"Selected large-move rate: "
        f"{row['selected_large'] * 100:.2f}%"
    )

    print(
        f"Large-move improvement: "
        f"{row['large_improvement'] * 100:+.2f} pp"
    )

    print(
        f"Positive windows: "
        f"{int(row['positive_windows'])}/"
        f"{int(row['total_windows'])}"
    )

# ---------------------------------------------------------
# 12. Find strongest horizon
# ---------------------------------------------------------

best_mean = summary.loc[
    summary["mean_improvement"].idxmax()
]

best_large = summary.loc[
    summary["large_improvement"].idxmax()
]

print("\n" + "=" * 70)
print("STRONGEST HORIZONS")
print("=" * 70)

print(
    f"\nBest average-movement horizon: "
    f"{best_mean['horizon']}"
)

print(
    f"Improvement: "
    f"{best_mean['mean_improvement'] * 100:+.3f} pp"
)

print(
    f"\nBest large-move horizon: "
    f"{best_large['horizon']}"
)

print(
    f"Improvement: "
    f"{best_large['large_improvement'] * 100:+.2f} pp"
)

# ---------------------------------------------------------
# 13. Save
# ---------------------------------------------------------

DETAIL_PATH = (
    RESULTS_DIR +
    "/opportunity_horizon_v1.parquet"
)

SUMMARY_PATH = (
    RESULTS_DIR +
    "/opportunity_horizon_v1_summary.parquet"
)

results.to_parquet(
    DETAIL_PATH,
    index=False
)

summary.to_parquet(
    SUMMARY_PATH,
    index=False
)

print("\n" + "=" * 70)
print("STEP 24 COMPLETE")
print("=" * 70)

print(
    "\nDetailed results saved:"
)

print(
    DETAIL_PATH
)

print(
    "\nSummary saved:"
)

print(
    SUMMARY_PATH
)

In [ ]:
# =========================================================
# STEP 24A — GAP-AWARE FUTURE HORIZON TARGETS
# =========================================================

import numpy as np
import pandas as pd
import os

FEATURE_PATH = (
    "/content/gdrive/MyDrive/"
    "SilverAgent/database/"
    "silver_h1_features_v2.parquet"
)

RAW_PATH = (
    "/content/gdrive/MyDrive/"
    "SilverAgent/database/"
    "silver_h1.parquet"
)

OUTPUT_PATH = (
    "/content/gdrive/MyDrive/"
    "SilverAgent/database/"
    "silver_h1_features_v2_horizons.parquet"
)

print("=" * 70)
print("STEP 24A — GAP-AWARE FUTURE HORIZONS")
print("=" * 70)

# ---------------------------------------------------------
# Load data
# ---------------------------------------------------------

features = pd.read_parquet(
    FEATURE_PATH
).sort_index()

raw = pd.read_parquet(
    RAW_PATH
).sort_index()

print(
    f"\nFeature rows: {len(features):,}"
)

print(
    f"Raw candles: {len(raw):,}"
)

# ---------------------------------------------------------
# Make sure index is datetime
# ---------------------------------------------------------

features.index = pd.to_datetime(
    features.index,
    utc=True
)

raw.index = pd.to_datetime(
    raw.index,
    utc=True
)

# ---------------------------------------------------------
# Use raw close prices as the authoritative future price
# ---------------------------------------------------------

future_close = raw["close"].copy()

# Remove duplicate timestamps just in case
future_close = future_close[
    ~future_close.index.duplicated(
        keep="last"
    )
].sort_index()

# ---------------------------------------------------------
# Gap-aware target builder
# ---------------------------------------------------------
#
# For each current candle:
#
# target time = current time + horizon
#
# Find the nearest REAL candle to that target time.
#
# Maximum tolerance = 90 minutes.
#
# This means we never invent a candle.
# ---------------------------------------------------------

def make_future_abs_move(
    current_index,
    horizon_hours,
    tolerance_minutes=90
):

    target_times = (
        current_index +
        pd.to_timedelta(
            horizon_hours,
            unit="h"
        )
    )

    target_frame = pd.DataFrame(
        {
            "target_time":
                target_times
        },
        index=current_index
    )

    future_frame = (
        future_close
        .rename("future_close")
        .reset_index()
        .rename(
            columns={
                future_close.index.name or "index":
                    "future_time"
            }
        )
    )

    target_frame = (
        target_frame
        .reset_index()
        .rename(
            columns={
                current_index.name or "index":
                    "current_time"
            }
        )
    )

    target_frame = target_frame.sort_values(
        "target_time"
    )

    future_frame = future_frame.sort_values(
        "future_time"
    )

    merged = pd.merge_asof(
        target_frame,
        future_frame,
        left_on="target_time",
        right_on="future_time",
        direction="nearest",
        tolerance=pd.Timedelta(
            minutes=tolerance_minutes
        )
    )

    merged = merged.set_index(
        "current_time"
    )

    current_price = (
        raw["close"]
        .reindex(
            merged.index
        )
    )

    future_price = (
        merged["future_close"]
    )

    movement = (
        (future_price / current_price - 1)
        .abs()
    )

    return movement

# ---------------------------------------------------------
# Generate horizons
# ---------------------------------------------------------

horizons = {
    "6h": 6,
    "12h": 12,
    "24h": 24,
    "48h": 48
}

for label, hours in horizons.items():

    print(
        f"\nBuilding {label} target..."
    )

    features[
        f"future_abs_{label}"
    ] = make_future_abs_move(
        features.index,
        hours
    )

    valid = features[
        f"future_abs_{label}"
    ].notna().sum()

    print(
        f"  Valid rows: {valid:,}"
    )

# ---------------------------------------------------------
# Diagnostics
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("HORIZON SUMMARY")
print("=" * 70)

for label in horizons:

    column = f"future_abs_{label}"

    series = features[
        column
    ].dropna()

    print(
        f"\n{label.upper()}"
    )

    print(
        f"  Valid: "
        f"{len(series):,}"
    )

    print(
        f"  Mean: "
        f"{series.mean() * 100:.3f}%"
    )

    print(
        f"  Median: "
        f"{series.median() * 100:.3f}%"
    )

    print(
        f"  75th percentile: "
        f"{series.quantile(.75) * 100:.3f}%"
    )

    print(
        f"  90th percentile: "
        f"{series.quantile(.90) * 100:.3f}%"
    )

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

features.to_parquet(
    OUTPUT_PATH,
    index=True
)

print("\n" + "=" * 70)
print("STEP 24A COMPLETE")
print("=" * 70)

print(
    "\nSaved:"
)

print(
    OUTPUT_PATH
)

In [ ]:
# =========================================================
# STEP 24B — OPPORTUNITY HORIZON WALK-FORWARD TEST
# =========================================================

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

FEATURE_PATH = (
    "/content/gdrive/MyDrive/"
    "SilverAgent/database/"
    "silver_h1_features_v2_horizons.parquet"
)

RESULTS_DIR = (
    "/content/gdrive/MyDrive/"
    "SilverAgent/results"
)

df = pd.read_parquet(
    FEATURE_PATH
).sort_index()

print("=" * 70)
print("STEP 24B — OPPORTUNITY HORIZON WALK-FORWARD TEST")
print("=" * 70)

print(
    f"\nRows loaded: {len(df):,}"
)

# ---------------------------------------------------------
# Engine features
# ---------------------------------------------------------

ENGINE_FEATURES = [
    "volatility_score",
    "momentum_energy_score",
    "volume_score",
    "displacement_score"
]

# ---------------------------------------------------------
# Horizon targets
# ---------------------------------------------------------

HORIZONS = {
    "6H": "future_abs_6h",
    "12H": "future_abs_12h",
    "24H": "future_abs_24h",
    "48H": "future_abs_48h"
}

# ---------------------------------------------------------
# Walk-forward settings
# ---------------------------------------------------------

TRAIN_SIZE = 4000
TEST_SIZE = 1000
STEP = 1000

# Fixed top-20% selection.
# We are NOT optimizing this here.
SELECTION_QUANTILE = 0.80

results = []

# ---------------------------------------------------------
# Run every horizon
# ---------------------------------------------------------

for horizon, target in HORIZONS.items():

    print("\n" + "=" * 70)
    print(f"HORIZON: {horizon}")
    print("=" * 70)

    columns = ENGINE_FEATURES + [target]

    work = (
        df[columns]
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .dropna()
    )

    print(
        f"Usable rows: {len(work):,}"
    )

    start = 0
    window = 0

    while (
        start +
        TRAIN_SIZE +
        TEST_SIZE
        <= len(work)
    ):

        train = work.iloc[
            start:
            start + TRAIN_SIZE
        ]

        test = work.iloc[
            start + TRAIN_SIZE:
            start + TRAIN_SIZE + TEST_SIZE
        ]

        # -------------------------------------------------
        # Train
        # -------------------------------------------------

        model = RandomForestRegressor(
            n_estimators=120,
            max_depth=8,
            min_samples_leaf=20,
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            train[ENGINE_FEATURES],
            train[target]
        )

        # -------------------------------------------------
        # Predict
        # -------------------------------------------------

        train_pred = model.predict(
            train[ENGINE_FEATURES]
        )

        test_pred = model.predict(
            test[ENGINE_FEATURES]
        )

        # -------------------------------------------------
        # Training-only threshold
        # -------------------------------------------------

        threshold = np.quantile(
            train_pred,
            SELECTION_QUANTILE
        )

        # Training-only large-move threshold
        large_threshold = np.quantile(
            train[target],
            0.75
        )

        # -------------------------------------------------
        # Select top 20%
        # -------------------------------------------------

        selected = test[
            test_pred >= threshold
        ]

        # -------------------------------------------------
        # Metrics
        # -------------------------------------------------

        baseline_mean = test[target].mean()
        selected_mean = selected[target].mean()

        baseline_median = test[target].median()
        selected_median = selected[target].median()

        baseline_large = (
            test[target] >= large_threshold
        ).mean()

        selected_large = (
            selected[target] >= large_threshold
        ).mean()

        results.append(
            {
                "horizon": horizon,
                "window": window + 1,
                "test_rows": len(test),
                "signals": len(selected),

                "baseline_mean":
                    baseline_mean,

                "selected_mean":
                    selected_mean,

                "mean_improvement":
                    selected_mean -
                    baseline_mean,

                "baseline_median":
                    baseline_median,

                "selected_median":
                    selected_median,

                "baseline_large":
                    baseline_large,

                "selected_large":
                    selected_large,

                "large_improvement":
                    selected_large -
                    baseline_large
            }
        )

        print(
            f"Window {window + 1:2d} | "
            f"Base "
            f"{baseline_mean * 100:.3f}% | "
            f"Selected "
            f"{selected_mean * 100:.3f}% | "
            f"Gain "
            f"{(selected_mean - baseline_mean) * 100:+.3f} pp | "
            f"Signals "
            f"{len(selected):4d}"
        )

        window += 1
        start += STEP

# ---------------------------------------------------------
# Results dataframe
# ---------------------------------------------------------

results_df = pd.DataFrame(
    results
)

# ---------------------------------------------------------
# Weighted summary
# ---------------------------------------------------------

summary_rows = []

for horizon in HORIZONS:

    group = results_df[
        results_df["horizon"] ==
        horizon
    ]

    baseline_mean = np.average(
        group["baseline_mean"],
        weights=group["test_rows"]
    )

    selected_mean = np.average(
        group["selected_mean"],
        weights=group["signals"]
    )

    baseline_median = np.average(
        group["baseline_median"],
        weights=group["test_rows"]
    )

    selected_median = np.average(
        group["selected_median"],
        weights=group["signals"]
    )

    baseline_large = np.average(
        group["baseline_large"],
        weights=group["test_rows"]
    )

    selected_large = np.average(
        group["selected_large"],
        weights=group["signals"]
    )

    improvement = (
        selected_mean -
        baseline_mean
    )

    large_improvement = (
        selected_large -
        baseline_large
    )

    positive_windows = (
        group["mean_improvement"] > 0
    ).sum()

    total_windows = len(group)

    summary_rows.append(
        {
            "horizon":
                horizon,

            "signals":
                group["signals"].sum(),

            "baseline_mean":
                baseline_mean,

            "selected_mean":
                selected_mean,

            "mean_improvement":
                improvement,

            "baseline_median":
                baseline_median,

            "selected_median":
                selected_median,

            "baseline_large":
                baseline_large,

            "selected_large":
                selected_large,

            "large_improvement":
                large_improvement,

            "positive_windows":
                positive_windows,

            "total_windows":
                total_windows,

            "consistency":
                positive_windows /
                total_windows
        }
    )

summary = pd.DataFrame(
    summary_rows
)

# ---------------------------------------------------------
# Print final results
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL HORIZON COMPARISON")
print("=" * 70)

for _, row in summary.iterrows():

    print(
        f"\n{row['horizon']}"
    )

    print(
        f"  Signals: "
        f"{int(row['signals']):,}"
    )

    print(
        f"  Baseline mean: "
        f"{row['baseline_mean'] * 100:.3f}%"
    )

    print(
        f"  Selected mean: "
        f"{row['selected_mean'] * 100:.3f}%"
    )

    print(
        f"  Mean improvement: "
        f"{row['mean_improvement'] * 100:+.3f} pp"
    )

    print(
        f"  Baseline large-move: "
        f"{row['baseline_large'] * 100:.2f}%"
    )

    print(
        f"  Selected large-move: "
        f"{row['selected_large'] * 100:.2f}%"
    )

    print(
        f"  Large-move improvement: "
        f"{row['large_improvement'] * 100:+.2f} pp"
    )

    print(
        f"  Positive windows: "
        f"{int(row['positive_windows'])}/"
        f"{int(row['total_windows'])}"
    )

# ---------------------------------------------------------
# Identify strongest horizons
# ---------------------------------------------------------

best_mean = summary.loc[
    summary["mean_improvement"].idxmax()
]

best_large = summary.loc[
    summary["large_improvement"].idxmax()
]

best_consistency = summary.loc[
    summary["consistency"].idxmax()
]

print("\n" + "=" * 70)
print("HORIZON WINNERS")
print("=" * 70)

print(
    f"\nBest average movement:"
    f" {best_mean['horizon']} "
    f"({best_mean['mean_improvement'] * 100:+.3f} pp)"
)

print(
    f"Best large-move capture:"
    f" {best_large['horizon']} "
    f"({best_large['large_improvement'] * 100:+.2f} pp)"
)

print(
    f"Most consistent:"
    f" {best_consistency['horizon']} "
    f"({best_consistency['consistency'] * 100:.1f}% positive windows)"
)

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

DETAIL_PATH = (
    RESULTS_DIR +
    "/opportunity_horizon_v1.parquet"
)

SUMMARY_PATH = (
    RESULTS_DIR +
    "/opportunity_horizon_v1_summary.parquet"
)

results_df.to_parquet(
    DETAIL_PATH,
    index=False
)

summary.to_parquet(
    SUMMARY_PATH,
    index=False
)

print("\n" + "=" * 70)
print("STEP 24B COMPLETE")
print("=" * 70)

print(
    "\nDetailed results saved:"
)

print(
    DETAIL_PATH
)

print(
    "\nSummary saved:"
)

print(
    SUMMARY_PATH
)

In [ ]:
# =========================================================
# STEP 24B — OPPORTUNITY HORIZON WALK-FORWARD TEST
# CORRECTED DATA JOIN
# =========================================================

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

HORIZON_PATH = (
    "/content/gdrive/MyDrive/"
    "SilverAgent/database/"
    "silver_h1_features_v2_horizons.parquet"
)

CONTINUOUS_PATH = (
    "/content/gdrive/MyDrive/"
    "SilverAgent/results/"
    "continuous_opportunity_v2_history.parquet"
)

RESULTS_DIR = (
    "/content/gdrive/MyDrive/"
    "SilverAgent/results"
)

# ---------------------------------------------------------
# Load
# ---------------------------------------------------------

horizons = pd.read_parquet(
    HORIZON_PATH
).sort_index()

continuous = pd.read_parquet(
    CONTINUOUS_PATH
).sort_index()

# Make sure both indexes are UTC datetime
horizons.index = pd.to_datetime(
    horizons.index,
    utc=True
)

continuous.index = pd.to_datetime(
    continuous.index,
    utc=True
)

print("=" * 70)
print("STEP 24B — OPPORTUNITY HORIZON WALK-FORWARD TEST")
print("=" * 70)

print(
    f"\nHorizon rows: {len(horizons):,}"
)

print(
    f"Continuous-score rows: {len(continuous):,}"
)

# ---------------------------------------------------------
# Engine scores
# ---------------------------------------------------------

ENGINE_FEATURES = [
    "volatility_score",
    "momentum_energy_score",
    "volume_score",
    "displacement_score"
]

# ---------------------------------------------------------
# Verify score columns exist
# ---------------------------------------------------------

missing_scores = [
    c for c in ENGINE_FEATURES
    if c not in continuous.columns
]

if missing_scores:

    raise RuntimeError(
        "Missing Engine Score columns: "
        + str(missing_scores)
    )

# ---------------------------------------------------------
# Join Engine Scores to horizon targets
# ---------------------------------------------------------

df = horizons.join(
    continuous[
        ENGINE_FEATURES
    ],
    how="inner"
)

print(
    f"\nJoined rows: {len(df):,}"
)

# ---------------------------------------------------------
# Horizon targets
# ---------------------------------------------------------

HORIZONS = {
    "6H": "future_abs_6h",
    "12H": "future_abs_12h",
    "24H": "future_abs_24h",
    "48H": "future_abs_48h"
}

# ---------------------------------------------------------
# Walk-forward settings
# ---------------------------------------------------------

TRAIN_SIZE = 4000
TEST_SIZE = 1000
STEP = 1000

# Fixed top 20%.
# We are NOT tuning this during this experiment.
SELECTION_QUANTILE = 0.80

results = []

# ---------------------------------------------------------
# Run each horizon
# ---------------------------------------------------------

for horizon, target in HORIZONS.items():

    print("\n" + "=" * 70)
    print(f"HORIZON: {horizon}")
    print("=" * 70)

    columns = (
        ENGINE_FEATURES +
        [target]
    )

    work = (
        df[columns]
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .dropna()
    )

    print(
        f"Usable rows: {len(work):,}"
    )

    start = 0
    window = 0

    while (
        start +
        TRAIN_SIZE +
        TEST_SIZE
        <= len(work)
    ):

        train = work.iloc[
            start:
            start + TRAIN_SIZE
        ]

        test = work.iloc[
            start + TRAIN_SIZE:
            start + TRAIN_SIZE + TEST_SIZE
        ]

        # -------------------------------------------------
        # Train
        # -------------------------------------------------

        model = RandomForestRegressor(
            n_estimators=120,
            max_depth=8,
            min_samples_leaf=20,
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            train[ENGINE_FEATURES],
            train[target]
        )

        # -------------------------------------------------
        # Predictions
        # -------------------------------------------------

        train_pred = model.predict(
            train[ENGINE_FEATURES]
        )

        test_pred = model.predict(
            test[ENGINE_FEATURES]
        )

        # -------------------------------------------------
        # Training-only selection threshold
        # -------------------------------------------------

        threshold = np.quantile(
            train_pred,
            SELECTION_QUANTILE
        )

        # Training-only large-move threshold
        large_threshold = np.quantile(
            train[target],
            0.75
        )

        # -------------------------------------------------
        # Select top 20%
        # -------------------------------------------------

        selected = test[
            test_pred >= threshold
        ]

        # -------------------------------------------------
        # Metrics
        # -------------------------------------------------

        baseline_mean = (
            test[target].mean()
        )

        selected_mean = (
            selected[target].mean()
        )

        baseline_median = (
            test[target].median()
        )

        selected_median = (
            selected[target].median()
        )

        baseline_large = (
            test[target] >=
            large_threshold
        ).mean()

        selected_large = (
            selected[target] >=
            large_threshold
        ).mean()

        results.append(
            {
                "horizon":
                    horizon,

                "window":
                    window + 1,

                "test_rows":
                    len(test),

                "signals":
                    len(selected),

                "baseline_mean":
                    baseline_mean,

                "selected_mean":
                    selected_mean,

                "mean_improvement":
                    selected_mean -
                    baseline_mean,

                "baseline_median":
                    baseline_median,

                "selected_median":
                    selected_median,

                "baseline_large":
                    baseline_large,

                "selected_large":
                    selected_large,

                "large_improvement":
                    selected_large -
                    baseline_large
            }
        )

        print(
            f"Window {window + 1:2d} | "
            f"Base "
            f"{baseline_mean * 100:.3f}% | "
            f"Selected "
            f"{selected_mean * 100:.3f}% | "
            f"Gain "
            f"{(selected_mean - baseline_mean) * 100:+.3f} pp | "
            f"Signals "
            f"{len(selected):4d}"
        )

        window += 1
        start += STEP

# ---------------------------------------------------------
# Results
# ---------------------------------------------------------

results_df = pd.DataFrame(
    results
)

# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

summary_rows = []

for horizon in HORIZONS:

    group = results_df[
        results_df["horizon"] ==
        horizon
    ]

    baseline_mean = np.average(
        group["baseline_mean"],
        weights=group["test_rows"]
    )

    selected_mean = np.average(
        group["selected_mean"],
        weights=group["signals"]
    )

    baseline_median = np.average(
        group["baseline_median"],
        weights=group["test_rows"]
    )

    selected_median = np.average(
        group["selected_median"],
        weights=group["signals"]
    )

    baseline_large = np.average(
        group["baseline_large"],
        weights=group["test_rows"]
    )

    selected_large = np.average(
        group["selected_large"],
        weights=group["signals"]
    )

    positive_windows = (
        group["mean_improvement"] > 0
    ).sum()

    total_windows = len(group)

    summary_rows.append(
        {
            "horizon":
                horizon,

            "signals":
                group["signals"].sum(),

            "baseline_mean":
                baseline_mean,

            "selected_mean":
                selected_mean,

            "mean_improvement":
                selected_mean -
                baseline_mean,

            "baseline_median":
                baseline_median,

            "selected_median":
                selected_median,

            "baseline_large":
                baseline_large,

            "selected_large":
                selected_large,

            "large_improvement":
                selected_large -
                baseline_large,

            "positive_windows":
                positive_windows,

            "total_windows":
                total_windows,

            "consistency":
                positive_windows /
                total_windows
        }
    )

summary = pd.DataFrame(
    summary_rows
)

# ---------------------------------------------------------
# Final results
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL HORIZON COMPARISON")
print("=" * 70)

for _, row in summary.iterrows():

    print(
        f"\n{row['horizon']}"
    )

    print(
        f"  Signals: "
        f"{int(row['signals']):,}"
    )

    print(
        f"  Baseline mean: "
        f"{row['baseline_mean'] * 100:.3f}%"
    )

    print(
        f"  Selected mean: "
        f"{row['selected_mean'] * 100:.3f}%"
    )

    print(
        f"  Mean improvement: "
        f"{row['mean_improvement'] * 100:+.3f} pp"
    )

    print(
        f"  Baseline large-move: "
        f"{row['baseline_large'] * 100:.2f}%"
    )

    print(
        f"  Selected large-move: "
        f"{row['selected_large'] * 100:.2f}%"
    )

    print(
        f"  Large-move improvement: "
        f"{row['large_improvement'] * 100:+.2f} pp"
    )

    print(
        f"  Positive windows: "
        f"{int(row['positive_windows'])}/"
        f"{int(row['total_windows'])}"
    )

# ---------------------------------------------------------
# Winners
# ---------------------------------------------------------

best_mean = summary.loc[
    summary["mean_improvement"].idxmax()
]

best_large = summary.loc[
    summary["large_improvement"].idxmax()
]

best_consistency = summary.loc[
    summary["consistency"].idxmax()
]

print("\n" + "=" * 70)
print("HORIZON WINNERS")
print("=" * 70)

print(
    f"\nBest average movement: "
    f"{best_mean['horizon']} "
    f"({best_mean['mean_improvement'] * 100:+.3f} pp)"
)

print(
    f"Best large-move capture: "
    f"{best_large['horizon']} "
    f"({best_large['large_improvement'] * 100:+.2f} pp)"
)

print(
    f"Most consistent: "
    f"{best_consistency['horizon']} "
    f"({best_consistency['consistency'] * 100:.1f}% positive windows)"
)

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

DETAIL_PATH = (
    RESULTS_DIR +
    "/opportunity_horizon_v1.parquet"
)

SUMMARY_PATH = (
    RESULTS_DIR +
    "/opportunity_horizon_v1_summary.parquet"
)

results_df.to_parquet(
    DETAIL_PATH,
    index=False
)

summary.to_parquet(
    SUMMARY_PATH,
    index=False
)

print("\n" + "=" * 70)
print("STEP 24B COMPLETE")
print("=" * 70)

print(
    "\nDetailed results saved:"
)

print(
    DETAIL_PATH
)

print(
    "\nSummary saved:"
)

print(
    SUMMARY_PATH
)

In [ ]:
# =========================================================
# STEP 25 — OPPORTUNITY THRESHOLD ANALYSIS
# =========================================================

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

FEATURE_PATH = (
    "/content/gdrive/MyDrive/"
    "SilverAgent/database/"
    "silver_h1_features_v2_horizons.parquet"
)

CONTINUOUS_PATH = (
    "/content/gdrive/MyDrive/"
    "SilverAgent/results/"
    "continuous_opportunity_v2_history.parquet"
)

RESULTS_DIR = (
    "/content/gdrive/MyDrive/"
    "SilverAgent/results"
)

# ---------------------------------------------------------
# Load
# ---------------------------------------------------------

horizons = pd.read_parquet(
    FEATURE_PATH
).sort_index()

continuous = pd.read_parquet(
    CONTINUOUS_PATH
).sort_index()

horizons.index = pd.to_datetime(
    horizons.index,
    utc=True
)

continuous.index = pd.to_datetime(
    continuous.index,
    utc=True
)

ENGINE_FEATURES = [
    "volatility_score",
    "momentum_energy_score",
    "volume_score",
    "displacement_score"
]

df = horizons.join(
    continuous[ENGINE_FEATURES],
    how="inner"
)

print("=" * 70)
print("STEP 25 — OPPORTUNITY THRESHOLD ANALYSIS")
print("=" * 70)

print(
    f"\nJoined rows: {len(df):,}"
)

# ---------------------------------------------------------
# Horizons
# ---------------------------------------------------------

HORIZONS = {
    "6H": "future_abs_6h",
    "12H": "future_abs_12h",
    "24H": "future_abs_24h"
}

# ---------------------------------------------------------
# Selection levels
# ---------------------------------------------------------
#
# 0.70 = top 30%
# 0.80 = top 20%
# 0.90 = top 10%
# 0.95 = top 5%
# 0.98 = top 2%
# ---------------------------------------------------------

SELECTION_LEVELS = {
    "TOP_30": 0.70,
    "TOP_20": 0.80,
    "TOP_10": 0.90,
    "TOP_5": 0.95,
    "TOP_2": 0.98
}

TRAIN_SIZE = 4000
TEST_SIZE = 1000
STEP = 1000

results = []

# ---------------------------------------------------------
# Test each horizon
# ---------------------------------------------------------

for horizon, target in HORIZONS.items():

    print("\n" + "=" * 70)
    print(f"HORIZON: {horizon}")
    print("=" * 70)

    work = (
        df[
            ENGINE_FEATURES +
            [target]
        ]
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .dropna()
    )

    print(
        f"Usable rows: {len(work):,}"
    )

    start = 0
    window = 0

    while (
        start +
        TRAIN_SIZE +
        TEST_SIZE
        <= len(work)
    ):

        train = work.iloc[
            start:
            start + TRAIN_SIZE
        ]

        test = work.iloc[
            start + TRAIN_SIZE:
            start + TRAIN_SIZE + TEST_SIZE
        ]

        # -------------------------------------------------
        # Train model
        # -------------------------------------------------

        model = RandomForestRegressor(
            n_estimators=120,
            max_depth=8,
            min_samples_leaf=20,
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            train[ENGINE_FEATURES],
            train[target]
        )

        train_pred = model.predict(
            train[ENGINE_FEATURES]
        )

        test_pred = model.predict(
            test[ENGINE_FEATURES]
        )

        # -------------------------------------------------
        # Training-only large move threshold
        # -------------------------------------------------

        large_threshold = np.quantile(
            train[target],
            0.75
        )

        # -------------------------------------------------
        # Every selection level
        # -------------------------------------------------

        for level_name, quantile in (
            SELECTION_LEVELS.items()
        ):

            threshold = np.quantile(
                train_pred,
                quantile
            )

            selected = test[
                test_pred >= threshold
            ]

            baseline_mean = (
                test[target].mean()
            )

            selected_mean = (
                selected[target].mean()
            )

            baseline_large = (
                test[target] >=
                large_threshold
            ).mean()

            selected_large = (
                selected[target] >=
                large_threshold
            ).mean()

            results.append(
                {
                    "horizon":
                        horizon,

                    "threshold":
                        level_name,

                    "window":
                        window + 1,

                    "test_rows":
                        len(test),

                    "signals":
                        len(selected),

                    "baseline_mean":
                        baseline_mean,

                    "selected_mean":
                        selected_mean,

                    "mean_improvement":
                        selected_mean -
                        baseline_mean,

                    "baseline_large":
                        baseline_large,

                    "selected_large":
                        selected_large,

                    "large_improvement":
                        selected_large -
                        baseline_large
                }
            )

        print(
            f"Window {window + 1:2d} complete"
        )

        window += 1
        start += STEP

# ---------------------------------------------------------
# DataFrame
# ---------------------------------------------------------

results_df = pd.DataFrame(
    results
)

# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

summary_rows = []

for horizon in HORIZONS:

    for threshold in SELECTION_LEVELS:

        group = results_df[
            (results_df["horizon"] == horizon) &
            (results_df["threshold"] == threshold)
        ]

        selected_mean = np.average(
            group["selected_mean"],
            weights=group["signals"]
        )

        baseline_mean = np.average(
            group["baseline_mean"],
            weights=group["test_rows"]
        )

        selected_large = np.average(
            group["selected_large"],
            weights=group["signals"]
        )

        baseline_large = np.average(
            group["baseline_large"],
            weights=group["test_rows"]
        )

        positive_windows = (
            group["mean_improvement"] > 0
        ).sum()

        total_windows = len(group)

        summary_rows.append(
            {
                "horizon":
                    horizon,

                "threshold":
                    threshold,

                "signals":
                    group["signals"].sum(),

                "selected_mean":
                    selected_mean,

                "baseline_mean":
                    baseline_mean,

                "mean_improvement":
                    selected_mean -
                    baseline_mean,

                "selected_large":
                    selected_large,

                "baseline_large":
                    baseline_large,

                "large_improvement":
                    selected_large -
                    baseline_large,

                "positive_windows":
                    positive_windows,

                "total_windows":
                    total_windows,

                "consistency":
                    positive_windows /
                    total_windows
            }
        )

summary = pd.DataFrame(
    summary_rows
)

# ---------------------------------------------------------
# Print 24H first
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("24H THRESHOLD RESULTS")
print("=" * 70)

table_24 = summary[
    summary["horizon"] == "24H"
].copy()

for _, row in table_24.iterrows():

    print(
        f"\n{row['threshold']}"
    )

    print(
        f"  Signals: "
        f"{int(row['signals']):,}"
    )

    print(
        f"  Mean movement: "
        f"{row['selected_mean'] * 100:.3f}%"
    )

    print(
        f"  Improvement: "
        f"{row['mean_improvement'] * 100:+.3f} pp"
    )

    print(
        f"  Large-move rate: "
        f"{row['selected_large'] * 100:.2f}%"
    )

    print(
        f"  Large-move gain: "
        f"{row['large_improvement'] * 100:+.2f} pp"
    )

    print(
        f"  Positive windows: "
        f"{int(row['positive_windows'])}/"
        f"{int(row['total_windows'])}"
    )

# ---------------------------------------------------------
# Full comparison
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("ALL HORIZONS — THRESHOLD COMPARISON")
print("=" * 70)

display_table = summary.copy()

display_table[
    "selected_mean"
] *= 100

display_table[
    "mean_improvement"
] *= 100

display_table[
    "selected_large"
] *= 100

display_table[
    "large_improvement"
] *= 100

print(
    display_table[
        [
            "horizon",
            "threshold",
            "signals",
            "selected_mean",
            "mean_improvement",
            "selected_large",
            "large_improvement",
            "positive_windows",
            "total_windows"
        ]
    ].to_string(
        index=False,
        formatters={
            "selected_mean":
                "{:.3f}".format,

            "mean_improvement":
                "{:+.3f}".format,

            "selected_large":
                "{:.2f}".format,

            "large_improvement":
                "{:+.2f}".format
        }
    )
)

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

DETAIL_PATH = (
    RESULTS_DIR +
    "/opportunity_threshold_v1.parquet"
)

SUMMARY_PATH = (
    RESULTS_DIR +
    "/opportunity_threshold_v1_summary.parquet"
)

results_df.to_parquet(
    DETAIL_PATH,
    index=False
)

summary.to_parquet(
    SUMMARY_PATH,
    index=False
)

print("\n" + "=" * 70)
print("STEP 25 COMPLETE")
print("=" * 70)

print(
    "\nSaved detailed results:"
)

print(
    DETAIL_PATH
)

print(
    "\nSaved summary:"
)

print(
    SUMMARY_PATH
)

In [ ]:
from google.colab import drive
import os

print("Mounting Google Drive...")

drive.mount(
    "/content/gdrive",
    force_remount=True
)

BASE = "/content/gdrive/MyDrive/SilverAgent"

print("\n" + "=" * 70)
print("SILVER AGENT DRIVE CHECK")
print("=" * 70)

if os.path.exists(BASE):

    print("\n✓ SilverAgent folder found")

    for folder in [
        "database",
        "models",
        "results",
        "logs"
    ]:

        path = os.path.join(
            BASE,
            folder
        )

        if os.path.exists(path):

            files = os.listdir(path)

            print(
                f"\n{folder}/ "
                f"({len(files)} files)"
            )

            for filename in sorted(files):
                print(
                    f"  {filename}"
                )

        else:

            print(
                f"\n❌ Missing folder: {folder}"
            )

else:

    print(
        "\n❌ SilverAgent folder NOT FOUND"
    )

print("\n" + "=" * 70)
print("DRIVE CHECK COMPLETE")
print("=" * 70)

In [ ]:
# =========================================================
# STEP 26 — EXTREME OPPORTUNITY VALIDATION
# =========================================================

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

HORIZON_PATH = (
    "/content/gdrive/MyDrive/"
    "SilverAgent/database/"
    "silver_h1_features_v2_horizons.parquet"
)

CONTINUOUS_PATH = (
    "/content/gdrive/MyDrive/"
    "SilverAgent/results/"
    "continuous_opportunity_v2_history.parquet"
)

RESULTS_DIR = (
    "/content/gdrive/MyDrive/"
    "SilverAgent/results"
)

# ---------------------------------------------------------
# Load
# ---------------------------------------------------------

horizons = pd.read_parquet(
    HORIZON_PATH
).sort_index()

continuous = pd.read_parquet(
    CONTINUOUS_PATH
).sort_index()

horizons.index = pd.to_datetime(
    horizons.index,
    utc=True
)

continuous.index = pd.to_datetime(
    continuous.index,
    utc=True
)

ENGINE_FEATURES = [
    "volatility_score",
    "momentum_energy_score",
    "volume_score",
    "displacement_score"
]

df = horizons.join(
    continuous[ENGINE_FEATURES],
    how="inner"
)

print("=" * 70)
print("STEP 26 — EXTREME OPPORTUNITY VALIDATION")
print("=" * 70)

# ---------------------------------------------------------
# Focus on 24H first
# ---------------------------------------------------------

target = "future_abs_24h"

work = (
    df[
        ENGINE_FEATURES +
        [target]
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .dropna()
)

print(
    f"\nUsable 24H rows: {len(work):,}"
)

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

TRAIN_SIZE = 4000
TEST_SIZE = 1000
STEP = 1000

SELECTIONS = {
    "TOP_30": 0.70,
    "TOP_20": 0.80,
    "TOP_10": 0.90,
    "TOP_5": 0.95,
    "TOP_2": 0.98
}

results = []

start = 0
window = 0

# ---------------------------------------------------------
# Walk-forward
# ---------------------------------------------------------

while (
    start +
    TRAIN_SIZE +
    TEST_SIZE
    <= len(work)
):

    train = work.iloc[
        start:
        start + TRAIN_SIZE
    ]

    test = work.iloc[
        start + TRAIN_SIZE:
        start + TRAIN_SIZE + TEST_SIZE
    ]

    # -----------------------------------------------------
    # Train model
    # -----------------------------------------------------

    model = RandomForestRegressor(
        n_estimators=120,
        max_depth=8,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        train[ENGINE_FEATURES],
        train[target]
    )

    train_pred = model.predict(
        train[ENGINE_FEATURES]
    )

    test_pred = model.predict(
        test[ENGINE_FEATURES]
    )

    # -----------------------------------------------------
    # Training-only large move threshold
    # -----------------------------------------------------

    large_threshold = np.quantile(
        train[target],
        0.75
    )

    # -----------------------------------------------------
    # Test every selection level
    # -----------------------------------------------------

    for name, quantile in SELECTIONS.items():

        threshold = np.quantile(
            train_pred,
            quantile
        )

        selected_mask = (
            test_pred >= threshold
        )

        selected = test.loc[
            selected_mask
        ]

        if len(selected) == 0:
            continue

        results.append(
            {
                "window":
                    window + 1,

                "threshold":
                    name,

                "start":
                    test.index.min(),

                "end":
                    test.index.max(),

                "signals":
                    len(selected),

                "mean_move":
                    selected[target].mean(),

                "median_move":
                    selected[target].median(),

                "large_move_rate":
                    (
                        selected[target]
                        >= large_threshold
                    ).mean(),

                "max_move":
                    selected[target].max(),

                "p90_move":
                    selected[target].quantile(
                        0.90
                    ),

                "p95_move":
                    selected[target].quantile(
                        0.95
                    )
            }
        )

    print(
        f"Window {window + 1:2d} complete"
    )

    window += 1
    start += STEP

results_df = pd.DataFrame(
    results
)

# ---------------------------------------------------------
# Summary using direct pooled observations
# ---------------------------------------------------------

summary_rows = []

for threshold in SELECTIONS:

    group = results_df[
        results_df["threshold"] ==
        threshold
    ]

    # -----------------------------------------------------
    # Important:
    # We weight by actual signal count.
    # This avoids NaN caused by tiny windows.
    # -----------------------------------------------------

    valid = group[
        group["signals"] > 0
    ]

    total_signals = (
        valid["signals"].sum()
    )

    mean_move = np.average(
        valid["mean_move"],
        weights=valid["signals"]
    )

    median_move = np.average(
        valid["median_move"],
        weights=valid["signals"]
    )

    large_rate = np.average(
        valid["large_move_rate"],
        weights=valid["signals"]
    )

    p90_move = np.average(
        valid["p90_move"],
        weights=valid["signals"]
    )

    p95_move = np.average(
        valid["p95_move"],
        weights=valid["signals"]
    )

    max_move = valid["max_move"].max()

    summary_rows.append(
        {
            "threshold":
                threshold,

            "signals":
                total_signals,

            "mean_move":
                mean_move,

            "median_move":
                median_move,

            "large_move_rate":
                large_rate,

            "p90_move":
                p90_move,

            "p95_move":
                p95_move,

            "max_move":
                max_move,

            "windows":
                len(valid)
        }
    )

summary = pd.DataFrame(
    summary_rows
)

# ---------------------------------------------------------
# Print
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("EXTREME OPPORTUNITY RESULTS — 24H")
print("=" * 70)

for _, row in summary.iterrows():

    print(
        f"\n{row['threshold']}"
    )

    print(
        f"  Signals: "
        f"{int(row['signals']):,}"
    )

    print(
        f"  Mean movement: "
        f"{row['mean_move'] * 100:.3f}%"
    )

    print(
        f"  Median movement: "
        f"{row['median_move'] * 100:.3f}%"
    )

    print(
        f"  Large-move rate: "
        f"{row['large_move_rate'] * 100:.2f}%"
    )

    print(
        f"  90th percentile: "
        f"{row['p90_move'] * 100:.3f}%"
    )

    print(
        f"  95th percentile: "
        f"{row['p95_move'] * 100:.3f}%"
    )

    print(
        f"  Maximum: "
        f"{row['max_move'] * 100:.3f}%"
    )

    print(
        f"  Windows represented: "
        f"{int(row['windows'])}"
    )

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

DETAIL_PATH = (
    RESULTS_DIR +
    "/extreme_opportunity_v1.parquet"
)

SUMMARY_PATH = (
    RESULTS_DIR +
    "/extreme_opportunity_v1_summary.parquet"
)

results_df.to_parquet(
    DETAIL_PATH,
    index=False
)

summary.to_parquet(
    SUMMARY_PATH,
    index=False
)

print("\n" + "=" * 70)
print("STEP 26 COMPLETE")
print("=" * 70)

print(
    "\nDetailed results saved:"
)

print(
    DETAIL_PATH
)

print(
    "\nSummary saved:"
)

print(
    SUMMARY_PATH
)

In [ ]:
# =========================================================
# STEP 27 — OPPORTUNITY ANATOMY
# =========================================================

import numpy as np
import pandas as pd

FEATURE_PATH = (
    "/content/gdrive/MyDrive/"
    "SilverAgent/database/"
    "silver_h1_features_v2_horizons.parquet"
)

CONTINUOUS_PATH = (
    "/content/gdrive/MyDrive/"
    "SilverAgent/results/"
    "continuous_opportunity_v2_history.parquet"
)

RESULTS_DIR = (
    "/content/gdrive/MyDrive/"
    "SilverAgent/results"
)

# ---------------------------------------------------------
# Load
# ---------------------------------------------------------

features = pd.read_parquet(
    FEATURE_PATH
).sort_index()

continuous = pd.read_parquet(
    CONTINUOUS_PATH
).sort_index()

features.index = pd.to_datetime(
    features.index,
    utc=True
)

continuous.index = pd.to_datetime(
    continuous.index,
    utc=True
)

# ---------------------------------------------------------
# Join
# ---------------------------------------------------------

ENGINE_FEATURES = [
    "volatility_score",
    "momentum_energy_score",
    "volume_score",
    "displacement_score"
]

df = features.join(
    continuous[ENGINE_FEATURES],
    how="inner"
)

# ---------------------------------------------------------
# Calculate continuous opportunity score
# ---------------------------------------------------------

df["opportunity_score"] = (
    df[ENGINE_FEATURES]
    .mean(axis=1)
)

# ---------------------------------------------------------
# Additional anatomy features
# ---------------------------------------------------------

ANATOMY_FEATURES = [
    "volatility_score",
    "momentum_energy_score",
    "volume_score",
    "displacement_score",
    "vol10",
    "vol20",
    "atr_percent",
    "relative_volume",
    "distance_high20",
    "distance_low20",
    "price_vs_MA20",
    "mom3",
    "mom6",
    "mom12",
]

# ---------------------------------------------------------
# Keep usable rows
# ---------------------------------------------------------

target = "future_abs_24h"

work = df[
    ANATOMY_FEATURES +
    ["opportunity_score", target]
].replace(
    [np.inf, -np.inf],
    np.nan
).dropna()

print("=" * 70)
print("STEP 27 — OPPORTUNITY ANATOMY")
print("=" * 70)

print(
    f"\nUsable rows: {len(work):,}"
)

# ---------------------------------------------------------
# Opportunity groups
# ---------------------------------------------------------

groups = {
    "BASELINE":
        work,

    "TOP_30":
        work[
            work["opportunity_score"]
            >= work["opportunity_score"].quantile(.70)
        ],

    "TOP_20":
        work[
            work["opportunity_score"]
            >= work["opportunity_score"].quantile(.80)
        ],

    "TOP_10":
        work[
            work["opportunity_score"]
            >= work["opportunity_score"].quantile(.90)
        ],

    "TOP_5":
        work[
            work["opportunity_score"]
            >= work["opportunity_score"].quantile(.95)
        ],

    "TOP_2":
        work[
            work["opportunity_score"]
            >= work["opportunity_score"].quantile(.98)
        ]
}

# ---------------------------------------------------------
# Compare anatomy
# ---------------------------------------------------------

summary = []

for name, group in groups.items():

    row = {
        "group":
            name,

        "samples":
            len(group),

        "mean_24h_move":
            group[target].mean(),

        "median_24h_move":
            group[target].median(),

        "mean_opportunity_score":
            group["opportunity_score"].mean()
    }

    for feature in ANATOMY_FEATURES:

        row[
            feature
        ] = group[feature].mean()

    summary.append(row)

summary_df = pd.DataFrame(
    summary
)

# ---------------------------------------------------------
# Print opportunity progression
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("OPPORTUNITY PROGRESSION")
print("=" * 70)

for _, row in summary_df.iterrows():

    print(
        f"\n{row['group']}"
    )

    print(
        f"  Samples: "
        f"{int(row['samples']):,}"
    )

    print(
        f"  Opportunity score: "
        f"{row['mean_opportunity_score']:.1f}"
    )

    print(
        f"  Mean 24h movement: "
        f"{row['mean_24h_move'] * 100:.3f}%"
    )

    print(
        f"  Median 24h movement: "
        f"{row['median_24h_move'] * 100:.3f}%"
    )

# ---------------------------------------------------------
# Compare each feature against baseline
# ---------------------------------------------------------

baseline = summary_df[
    summary_df["group"] ==
    "BASELINE"
].iloc[0]

print("\n" + "=" * 70)
print("TOP 5% ANATOMY VS BASELINE")
print("=" * 70)

top5 = summary_df[
    summary_df["group"] ==
    "TOP_5"
].iloc[0]

for feature in ANATOMY_FEATURES:

    base_value = baseline[feature]
    top_value = top5[feature]

    if (
        pd.notna(base_value) and
        pd.notna(top_value) and
        base_value != 0
    ):

        difference = (
            top_value -
            base_value
        )

        percentage = (
            difference /
            abs(base_value)
        ) * 100

        print(
            f"\n{feature}"
        )

        print(
            f"  Baseline: "
            f"{base_value:.4f}"
        )

        print(
            f"  Top 5%:   "
            f"{top_value:.4f}"
        )

        print(
            f"  Change:   "
            f"{difference:+.4f} "
            f"({percentage:+.1f}%)"
        )

# ---------------------------------------------------------
# Feature ranking
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("FEATURE SEPARATION — TOP 5%")
print("=" * 70)

separation = []

for feature in ANATOMY_FEATURES:

    base_value = baseline[feature]
    top_value = top5[feature]

    if (
        pd.notna(base_value) and
        pd.notna(top_value) and
        base_value != 0
    ):

        separation.append(
            {
                "feature":
                    feature,

                "baseline":
                    base_value,

                "top5":
                    top_value,

                "absolute_change":
                    abs(
                        top_value -
                        base_value
                    ),

                "relative_change":
                    abs(
                        (
                            top_value -
                            base_value
                        ) /
                        base_value
                    )
            }
        )

separation_df = pd.DataFrame(
    separation
).sort_values(
    "relative_change",
    ascending=False
)

for _, row in separation_df.iterrows():

    print(
        f"{row['feature']:25s} "
        f"change "
        f"{row['relative_change'] * 100:+.1f}%"
    )

# ---------------------------------------------------------
# Correlation with opportunity score
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("FEATURE → OPPORTUNITY SCORE CORRELATION")
print("=" * 70)

correlations = []

for feature in ANATOMY_FEATURES:

    correlation = (
        work[
            [
                feature,
                "opportunity_score"
            ]
        ]
        .corr()
        .iloc[0, 1]
    )

    correlations.append(
        {
            "feature":
                feature,

            "correlation":
                correlation
        }
    )

correlation_df = pd.DataFrame(
    correlations
).sort_values(
    "correlation",
    ascending=False
)

for _, row in correlation_df.iterrows():

    print(
        f"{row['feature']:25s} "
        f"{row['correlation']:+.3f}"
    )

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

summary_path = (
    RESULTS_DIR +
    "/opportunity_anatomy_v1.parquet"
)

separation_path = (
    RESULTS_DIR +
    "/opportunity_anatomy_separation_v1.parquet"
)

correlation_path = (
    RESULTS_DIR +
    "/opportunity_anatomy_correlation_v1.parquet"
)

summary_df.to_parquet(
    summary_path,
    index=False
)

separation_df.to_parquet(
    separation_path,
    index=False
)

correlation_df.to_parquet(
    correlation_path,
    index=False
)

print("\n" + "=" * 70)
print("STEP 27 COMPLETE")
print("=" * 70)

print(
    "\nSaved:"
)

print(summary_path)
print(separation_path)
print(correlation_path)

In [ ]:
# =========================================================
# STEP 28 — OPPORTUNITY ENGINE REDUNDANCY / ABLATION
# =========================================================

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

HORIZON_PATH = (
    "/content/gdrive/MyDrive/"
    "SilverAgent/database/"
    "silver_h1_features_v2_horizons.parquet"
)

CONTINUOUS_PATH = (
    "/content/gdrive/MyDrive/"
    "SilverAgent/results/"
    "continuous_opportunity_v2_history.parquet"
)

RESULTS_DIR = (
    "/content/gdrive/MyDrive/"
    "SilverAgent/results"
)

# ---------------------------------------------------------
# Load
# ---------------------------------------------------------

horizons = pd.read_parquet(
    HORIZON_PATH
).sort_index()

continuous = pd.read_parquet(
    CONTINUOUS_PATH
).sort_index()

horizons.index = pd.to_datetime(
    horizons.index,
    utc=True
)

continuous.index = pd.to_datetime(
    continuous.index,
    utc=True
)

# ---------------------------------------------------------
# Engine components
# ---------------------------------------------------------

ENGINE_GROUPS = {

    "ALL":
    [
        "volatility_score",
        "momentum_energy_score",
        "volume_score",
        "displacement_score"
    ],

    "NO_VOLATILITY":
    [
        "momentum_energy_score",
        "volume_score",
        "displacement_score"
    ],

    "NO_MOMENTUM":
    [
        "volatility_score",
        "volume_score",
        "displacement_score"
    ],

    "NO_VOLUME":
    [
        "volatility_score",
        "momentum_energy_score",
        "displacement_score"
    ],

    "NO_DISPLACEMENT":
    [
        "volatility_score",
        "momentum_energy_score",
        "volume_score"
    ],

    "VOLATILITY_ONLY":
    [
        "volatility_score"
    ],

    "MOMENTUM_ONLY":
    [
        "momentum_energy_score"
    ],

    "VOLUME_ONLY":
    [
        "volume_score"
    ],

    "DISPLACEMENT_ONLY":
    [
        "displacement_score"
    ],

    "VOL_MOM":
    [
        "volatility_score",
        "momentum_energy_score"
    ],

    "VOL_DISP":
    [
        "volatility_score",
        "displacement_score"
    ],

    "MOM_DISP":
    [
        "momentum_energy_score",
        "displacement_score"
    ],

    "MOM_VOL":
    [
        "momentum_energy_score",
        "volume_score"
    ],

    "VOL_VOLUME":
    [
        "volatility_score",
        "volume_score"
    ],

    "DISP_VOLUME":
    [
        "displacement_score",
        "volume_score"
    ]
}

TARGET = "future_abs_24h"

# ---------------------------------------------------------
# Join
# ---------------------------------------------------------

df = horizons.join(
    continuous[
        [
            "volatility_score",
            "momentum_energy_score",
            "volume_score",
            "displacement_score"
        ]
    ],
    how="inner"
)

print("=" * 70)
print("STEP 28 — OPPORTUNITY ENGINE REDUNDANCY / ABLATION")
print("=" * 70)

print(
    f"\nJoined rows: {len(df):,}"
)

# ---------------------------------------------------------
# Walk-forward settings
# ---------------------------------------------------------

TRAIN_SIZE = 4000
TEST_SIZE = 1000
STEP = 1000

# Fixed top 10%.
# We already established this as an important operating
# zone, but we are NOT tuning it in this experiment.
SELECTION_QUANTILE = 0.90

results = []

# ---------------------------------------------------------
# Test every feature configuration
# ---------------------------------------------------------

for group_name, features in ENGINE_GROUPS.items():

    print("\n" + "=" * 70)
    print(
        f"TESTING: {group_name}"
    )
    print(
        f"Features: {features}"
    )
    print("=" * 70)

    work = (
        df[
            features +
            [TARGET]
        ]
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .dropna()
    )

    print(
        f"Usable rows: {len(work):,}"
    )

    start = 0
    window = 0

    while (
        start +
        TRAIN_SIZE +
        TEST_SIZE
        <= len(work)
    ):

        train = work.iloc[
            start:
            start + TRAIN_SIZE
        ]

        test = work.iloc[
            start + TRAIN_SIZE:
            start + TRAIN_SIZE + TEST_SIZE
        ]

        # -------------------------------------------------
        # Model
        # -------------------------------------------------

        model = RandomForestRegressor(
            n_estimators=120,
            max_depth=8,
            min_samples_leaf=20,
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            train[features],
            train[TARGET]
        )

        train_pred = model.predict(
            train[features]
        )

        test_pred = model.predict(
            test[features]
        )

        # -------------------------------------------------
        # Training-only threshold
        # -------------------------------------------------

        threshold = np.quantile(
            train_pred,
            SELECTION_QUANTILE
        )

        selected = test[
            test_pred >= threshold
        ]

        # -------------------------------------------------
        # Metrics
        # -------------------------------------------------

        baseline_mean = (
            test[TARGET].mean()
        )

        selected_mean = (
            selected[TARGET].mean()
        )

        baseline_large_threshold = (
            train[TARGET].quantile(0.75)
        )

        baseline_large = (
            test[TARGET]
            >= baseline_large_threshold
        ).mean()

        selected_large = (
            selected[TARGET]
            >= baseline_large_threshold
        ).mean()

        results.append(
            {
                "configuration":
                    group_name,

                "window":
                    window + 1,

                "signals":
                    len(selected),

                "baseline_mean":
                    baseline_mean,

                "selected_mean":
                    selected_mean,

                "improvement":
                    selected_mean -
                    baseline_mean,

                "baseline_large":
                    baseline_large,

                "selected_large":
                    selected_large,

                "large_improvement":
                    selected_large -
                    baseline_large
            }
        )

        window += 1
        start += STEP

    print(
        f"Completed {window} windows"
    )

# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

results_df = pd.DataFrame(
    results
)

summary_rows = []

for configuration in ENGINE_GROUPS:

    group = results_df[
        results_df["configuration"]
        == configuration
    ]

    signals = group[
        "signals"
    ].sum()

    selected_mean = np.average(
        group["selected_mean"],
        weights=group["signals"]
    )

    baseline_mean = np.average(
        group["baseline_mean"],
        weights=np.ones(len(group))
    )

    selected_large = np.average(
        group["selected_large"],
        weights=group["signals"]
    )

    baseline_large = np.average(
        group["baseline_large"],
        weights=np.ones(len(group))
    )

    positive_windows = (
        group["improvement"] > 0
    ).sum()

    total_windows = len(group)

    summary_rows.append(
        {
            "configuration":
                configuration,

            "signals":
                signals,

            "selected_mean":
                selected_mean,

            "baseline_mean":
                baseline_mean,

            "improvement":
                selected_mean -
                baseline_mean,

            "selected_large":
                selected_large,

            "baseline_large":
                baseline_large,

            "large_improvement":
                selected_large -
                baseline_large,

            "positive_windows":
                positive_windows,

            "total_windows":
                total_windows,

            "consistency":
                positive_windows /
                total_windows
        }
    )

summary = pd.DataFrame(
    summary_rows
)

# ---------------------------------------------------------
# Rank
# ---------------------------------------------------------

summary = summary.sort_values(
    "improvement",
    ascending=False
).reset_index(
    drop=True
)

# ---------------------------------------------------------
# Print ranking
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("ENGINE CONFIGURATION RANKING")
print("=" * 70)

for i, row in summary.iterrows():

    print(
        f"\n#{i + 1} "
        f"{row['configuration']}"
    )

    print(
        f"  Signals: "
        f"{int(row['signals']):,}"
    )

    print(
        f"  Mean movement: "
        f"{row['selected_mean'] * 100:.3f}%"
    )

    print(
        f"  Improvement: "
        f"{row['improvement'] * 100:+.3f} pp"
    )

    print(
        f"  Large-move rate: "
        f"{row['selected_large'] * 100:.2f}%"
    )

    print(
        f"  Large-move gain: "
        f"{row['large_improvement'] * 100:+.2f} pp"
    )

    print(
        f"  Consistency: "
        f"{int(row['positive_windows'])}/"
        f"{int(row['total_windows'])}"
    )

# ---------------------------------------------------------
# Special comparison against ALL
# ---------------------------------------------------------

all_row = summary[
    summary["configuration"] == "ALL"
].iloc[0]

print("\n" + "=" * 70)
print("ABLATION VS CURRENT ENGINE")
print("=" * 70)

for _, row in summary.iterrows():

    if row["configuration"] == "ALL":
        continue

    delta = (
        row["improvement"] -
        all_row["improvement"]
    )

    print(
        f"{row['configuration']:20s} "
        f"vs ALL: "
        f"{delta * 100:+.3f} pp"
    )

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

DETAIL_PATH = (
    RESULTS_DIR +
    "/opportunity_ablation_v2.parquet"
)

SUMMARY_PATH = (
    RESULTS_DIR +
    "/opportunity_ablation_v2_summary.parquet"
)

results_df.to_parquet(
    DETAIL_PATH,
    index=False
)

summary.to_parquet(
    SUMMARY_PATH,
    index=False
)

print("\n" + "=" * 70)
print("STEP 28 COMPLETE")
print("=" * 70)

print(
    "\nDetailed results:"
)

print(
    DETAIL_PATH
)

print(
    "\nSummary:"
)

print(
    SUMMARY_PATH
)

In [ ]:
# ============================================================
# STEP 29 — NESTED WALK-FORWARD ARCHITECTURE TEST
# CORRECTED DATA JOIN
# ============================================================

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor

BASE = "/content/gdrive/MyDrive/SilverAgent"

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

features_path = (
    f"{BASE}/database/"
    "silver_h1_features_v2_horizons.parquet"
)

scores_path = (
    f"{BASE}/results/"
    "continuous_opportunity_v2_history.parquet"
)

features = pd.read_parquet(features_path)
scores = pd.read_parquet(scores_path)

print("Features:", features.shape)
print("Scores:", scores.shape)

# ------------------------------------------------------------
# 2. KEEP ONLY THE ENGINE SCORES FROM SCORES FILE
# ------------------------------------------------------------

engine_features = [
    "volatility_score",
    "momentum_energy_score",
    "volume_score",
    "displacement_score"
]

scores_small = scores[engine_features].copy()

# ------------------------------------------------------------
# 3. ALIGN USING THE INDEX
# ------------------------------------------------------------

df = features.join(
    scores_small,
    how="inner"
)

print("Joined dataset:", df.shape)

# ------------------------------------------------------------
# 4. TARGET
# ------------------------------------------------------------

target = "future_abs_24h"

needed = engine_features + [target]

df = df.dropna(subset=needed).copy()
df = df.sort_index()

print("Usable rows:", len(df))
print("Start:", df.index.min())
print("End:", df.index.max())

# ------------------------------------------------------------
# 5. CANDIDATE ARCHITECTURES
# ------------------------------------------------------------

architectures = {

    "ALL": [
        "volatility_score",
        "momentum_energy_score",
        "volume_score",
        "displacement_score"
    ],

    "VOL_MOM": [
        "volatility_score",
        "momentum_energy_score"
    ],

    "NO_VOLUME": [
        "volatility_score",
        "momentum_energy_score",
        "displacement_score"
    ]
}

thresholds = {
    "TOP20": 0.80,
    "TOP10": 0.90,
    "TOP5": 0.95
}

# ------------------------------------------------------------
# 6. SETTINGS
# ------------------------------------------------------------

OUTER_TRAIN = 6000
OUTER_TEST = 1000

INNER_TRAIN = 4000
INNER_TEST = 1000

STEP = 1000

MIN_SELECTED = 30

results = []
selection_log = []

# ------------------------------------------------------------
# 7. MODEL/EVALUATION FUNCTION
# ------------------------------------------------------------

def evaluate_candidate(
    train_df,
    test_df,
    feature_list,
    threshold
):

    X_train = train_df[feature_list]
    y_train = train_df[target]

    X_test = test_df[feature_list]

    model = RandomForestRegressor(
        n_estimators=120,
        max_depth=8,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    evaluation = test_df.copy()
    evaluation["prediction"] = predictions

    cutoff = np.quantile(
        predictions,
        threshold
    )

    selected = evaluation[
        evaluation["prediction"] >= cutoff
    ]

    if len(selected) < MIN_SELECTED:
        return None

    baseline_mean = evaluation[target].mean()
    selected_mean = selected[target].mean()

    # Large-move threshold is the 75th percentile
    # of THIS test period.
    large_threshold = evaluation[target].quantile(0.75)

    baseline_large = (
        evaluation[target] >= large_threshold
    ).mean()

    selected_large = (
        selected[target] >= large_threshold
    ).mean()

    return {
        "baseline_mean": baseline_mean,
        "selected_mean": selected_mean,
        "improvement": selected_mean - baseline_mean,
        "baseline_large": baseline_large,
        "selected_large": selected_large,
        "large_improvement": (
            selected_large - baseline_large
        ),
        "signals": len(selected)
    }

# ------------------------------------------------------------
# 8. NESTED WALK-FORWARD
# ------------------------------------------------------------

start = OUTER_TRAIN
window_number = 0

while start + OUTER_TEST <= len(df):

    window_number += 1

    outer_train = df.iloc[
        start - OUTER_TRAIN:start
    ].copy()

    outer_test = df.iloc[
        start:start + OUTER_TEST
    ].copy()

    # --------------------------------------------------------
    # INNER VALIDATION
    # --------------------------------------------------------

    inner_train = outer_train.iloc[
        :INNER_TRAIN
    ].copy()

    inner_test = outer_train.iloc[
        INNER_TRAIN:
        INNER_TRAIN + INNER_TEST
    ].copy()

    candidate_scores = []

    for arch_name, feature_list in architectures.items():

        for threshold_name, threshold in thresholds.items():

            score = evaluate_candidate(
                inner_train,
                inner_test,
                feature_list,
                threshold
            )

            if score is None:
                continue

            candidate_scores.append({
                "architecture": arch_name,
                "threshold": threshold_name,
                **score
            })

    if not candidate_scores:
        start += STEP
        continue

    candidate_df = pd.DataFrame(
        candidate_scores
    )

    # Inner validation chooses the architecture.
    candidate_df = candidate_df.sort_values(
        [
            "improvement",
            "large_improvement"
        ],
        ascending=False
    )

    winner = candidate_df.iloc[0]

    selected_arch = winner[
        "architecture"
    ]

    selected_threshold = winner[
        "threshold"
    ]

    selected_features = architectures[
        selected_arch
    ]

    selected_cutoff = thresholds[
        selected_threshold
    ]

    # --------------------------------------------------------
    # OUTER TEST
    # --------------------------------------------------------

    outer_result = evaluate_candidate(
        outer_train,
        outer_test,
        selected_features,
        selected_cutoff
    )

    if outer_result is not None:

        results.append({
            "window": window_number,

            "outer_start":
                outer_test.index.min(),

            "outer_end":
                outer_test.index.max(),

            "selected_architecture":
                selected_arch,

            "selected_threshold":
                selected_threshold,

            **outer_result
        })

        selection_log.append({
            "window": window_number,

            "selected_architecture":
                selected_arch,

            "selected_threshold":
                selected_threshold,

            "inner_improvement":
                winner["improvement"],

            "inner_large_improvement":
                winner["large_improvement"]
        })

        print(
            f"Window {window_number:02d} | "
            f"Selected: {selected_arch:10s} "
            f"{selected_threshold:5s} | "
            f"Outer improvement: "
            f"{outer_result['improvement']:+.3%}"
        )

    else:

        print(
            f"Window {window_number:02d} | "
            "No valid outer result"
        )

    start += STEP

# ------------------------------------------------------------
# 9. RESULTS
# ------------------------------------------------------------

results_df = pd.DataFrame(results)
selection_df = pd.DataFrame(selection_log)

print("\n" + "=" * 70)
print("STEP 29 — NESTED WALK-FORWARD RESULTS")
print("=" * 70)

if len(results_df) == 0:

    print("No valid outer results.")

else:

    weighted_baseline = np.average(
        results_df["baseline_mean"],
        weights=results_df["signals"]
    )

    weighted_selected = np.average(
        results_df["selected_mean"],
        weights=results_df["signals"]
    )

    weighted_large_base = np.average(
        results_df["baseline_large"],
        weights=results_df["signals"]
    )

    weighted_large_selected = np.average(
        results_df["selected_large"],
        weights=results_df["signals"]
    )

    print(
        "\nOuter windows:",
        len(results_df)
    )

    print(
        "Total selected signals:",
        results_df["signals"].sum()
    )

    print(
        f"\nBaseline mean 24h move: "
        f"{weighted_baseline:.3%}"
    )

    print(
        f"Selected mean 24h move: "
        f"{weighted_selected:.3%}"
    )

    print(
        f"Improvement: "
        f"{weighted_selected - weighted_baseline:+.3%}"
    )

    print(
        f"\nBaseline large-move rate: "
        f"{weighted_large_base:.2%}"
    )

    print(
        f"Selected large-move rate: "
        f"{weighted_large_selected:.2%}"
    )

    print(
        f"Large-move improvement: "
        f"{weighted_large_selected - weighted_large_base:+.2%}"
    )

    print(
        "\nPositive outer windows:",
        (results_df["improvement"] > 0).sum(),
        "/",
        len(results_df)
    )

    print("\nArchitecture selections:")
    print(
        selection_df[
            "selected_architecture"
        ].value_counts()
    )

    print("\nThreshold selections:")
    print(
        selection_df[
            "selected_threshold"
        ].value_counts()
    )

    print("\nDetailed results:")
    display(results_df)

# ------------------------------------------------------------
# 10. SAVE
# ------------------------------------------------------------

results_path = (
    f"{BASE}/results/"
    "nested_architecture_v1_results.parquet"
)

selection_path = (
    f"{BASE}/results/"
    "nested_architecture_v1_selection.parquet"
)

results_df.to_parquet(
    results_path,
    index=False
)

selection_df.to_parquet(
    selection_path,
    index=False
)

print("\nSaved:")
print(results_path)
print(selection_path)

print("\nSTEP 29 COMPLETE")

In [ ]:
# ============================================================
# STEP 30 — OPPORTUNITY INTELLIGENCE PROFILE
# ============================================================

import pandas as pd
import numpy as np

BASE = "/content/gdrive/MyDrive/SilverAgent"

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

features_path = (
    f"{BASE}/database/"
    "silver_h1_features_v2_horizons.parquet"
)

scores_path = (
    f"{BASE}/results/"
    "continuous_opportunity_v2_history.parquet"
)

features = pd.read_parquet(features_path)
scores = pd.read_parquet(scores_path)

# Only take the continuous opportunity components
score_columns = [
    "volatility_score",
    "momentum_energy_score",
    "volume_score",
    "displacement_score"
]

scores_small = scores[score_columns].copy()

df = features.join(
    scores_small,
    how="inner"
)

# ------------------------------------------------------------
# 2. BUILD A SIMPLE OPPORTUNITY SCORE
# ------------------------------------------------------------

df["opportunity_score"] = df[
    score_columns
].mean(axis=1)

# ------------------------------------------------------------
# 3. REQUIRED FUTURE TARGETS
# ------------------------------------------------------------

future_targets = [
    "future_abs_6h",
    "future_abs_12h",
    "future_abs_24h"
]

df = df.dropna(
    subset=["opportunity_score"] + future_targets
).copy()

df = df.sort_index()

print("=" * 70)
print("STEP 30 — OPPORTUNITY INTELLIGENCE")
print("=" * 70)

print("\nUsable rows:", len(df))
print("Start:", df.index.min())
print("End:", df.index.max())

# ------------------------------------------------------------
# 4. CREATE SCORE DECILES
# ------------------------------------------------------------

df["opportunity_decile"] = pd.qcut(
    df["opportunity_score"],
    q=10,
    labels=False,
    duplicates="drop"
) + 1

# ------------------------------------------------------------
# 5. DECILE ANALYSIS
# ------------------------------------------------------------

decile_results = []

for decile, group in df.groupby(
    "opportunity_decile"
):

    row = {
        "decile": int(decile),

        "observations": len(group),

        "score_mean":
            group["opportunity_score"].mean()
    }

    for horizon in ["6h", "12h", "24h"]:

        target = f"future_abs_{horizon}"

        values = group[target].dropna()

        row[f"{horizon}_mean"] = values.mean()
        row[f"{horizon}_median"] = values.median()
        row[f"{horizon}_p75"] = values.quantile(0.75)
        row[f"{horizon}_p90"] = values.quantile(0.90)
        row[f"{horizon}_p95"] = values.quantile(0.95)

    decile_results.append(row)

deciles = pd.DataFrame(decile_results)

# ------------------------------------------------------------
# 6. PRINT DECILE TABLE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("OPPORTUNITY DECILES")
print("=" * 70)

display(
    deciles[
        [
            "decile",
            "observations",
            "score_mean",
            "6h_mean",
            "12h_mean",
            "24h_mean",
            "24h_median",
            "24h_p90",
            "24h_p95"
        ]
    ].round(4)
)

# ------------------------------------------------------------
# 7. LARGE-MOVE THRESHOLDS
# ------------------------------------------------------------

large_24h = df[
    "future_abs_24h"
].quantile(0.75)

extreme_24h = df[
    "future_abs_24h"
].quantile(0.90)

very_extreme_24h = df[
    "future_abs_24h"
].quantile(0.95)

print("\n24H movement thresholds:")

print(
    f"75th percentile: "
    f"{large_24h:.3%}"
)

print(
    f"90th percentile: "
    f"{extreme_24h:.3%}"
)

print(
    f"95th percentile: "
    f"{very_extreme_24h:.3%}"
)

# ------------------------------------------------------------
# 8. OPPORTUNITY ZONES
# ------------------------------------------------------------

df["zone"] = pd.cut(
    df["opportunity_score"],
    bins=[-np.inf, 20, 40, 60, 80, np.inf],
    labels=[
        "VERY_LOW",
        "LOW",
        "MEDIUM",
        "HIGH",
        "VERY_HIGH"
    ]
)

zone_results = []

for zone, group in df.groupby(
    "zone",
    observed=True
):

    values = group[
        "future_abs_24h"
    ].dropna()

    zone_results.append({

        "zone": str(zone),

        "observations": len(group),

        "mean_score":
            group["opportunity_score"].mean(),

        "mean_24h":
            values.mean(),

        "median_24h":
            values.median(),

        "p90_24h":
            values.quantile(0.90),

        "p95_24h":
            values.quantile(0.95),

        "large_move_rate":
            (values >= large_24h).mean(),

        "extreme_move_rate":
            (values >= extreme_24h).mean(),

        "very_extreme_rate":
            (values >= very_extreme_24h).mean()
    })

zones = pd.DataFrame(zone_results)

print("\n" + "=" * 70)
print("OPPORTUNITY ZONES")
print("=" * 70)

display(
    zones.round(4)
)

# ------------------------------------------------------------
# 9. TOP OPPORTUNITY PERCENTILES
# ------------------------------------------------------------

percentile_results = []

for name, percentile in [
    ("TOP50", 0.50),
    ("TOP30", 0.70),
    ("TOP20", 0.80),
    ("TOP10", 0.90),
    ("TOP5", 0.95),
    ("TOP2", 0.98)
]:

    cutoff = df[
        "opportunity_score"
    ].quantile(percentile)

    selected = df[
        df["opportunity_score"] >= cutoff
    ]

    values = selected[
        "future_abs_24h"
    ]

    percentile_results.append({

        "group": name,

        "cutoff": cutoff,

        "observations": len(selected),

        "mean_24h":
            values.mean(),

        "median_24h":
            values.median(),

        "p90_24h":
            values.quantile(0.90),

        "p95_24h":
            values.quantile(0.95),

        "large_move_rate":
            (values >= large_24h).mean(),

        "extreme_move_rate":
            (values >= extreme_24h).mean(),

        "very_extreme_rate":
            (values >= very_extreme_24h).mean()
    })

percentiles = pd.DataFrame(
    percentile_results
)

print("\n" + "=" * 70)
print("EXTREME OPPORTUNITY CONCENTRATION")
print("=" * 70)

display(
    percentiles.round(4)
)

# ------------------------------------------------------------
# 10. COMPONENT PROFILE OF EXTREME OPPORTUNITIES
# ------------------------------------------------------------

top5_cutoff = df[
    "opportunity_score"
].quantile(0.95)

top5 = df[
    df["opportunity_score"] >= top5_cutoff
]

baseline = df

component_results = []

for component in score_columns:

    component_results.append({

        "component": component,

        "baseline_mean":
            baseline[component].mean(),

        "top5_mean":
            top5[component].mean(),

        "difference":
            top5[component].mean()
            - baseline[component].mean()
    })

components = pd.DataFrame(
    component_results
)

print("\n" + "=" * 70)
print("TOP 5% COMPONENT PROFILE")
print("=" * 70)

display(
    components.round(3)
)

# ------------------------------------------------------------
# 11. CURRENT MARKET STATE
# ------------------------------------------------------------

latest = df.iloc[-1]

print("\n" + "=" * 70)
print("CURRENT SILVER OPPORTUNITY STATE")
print("=" * 70)

print(
    "\nTimestamp:",
    latest.name
)

print(
    "Price:",
    round(latest["close"], 4)
)

print(
    "Opportunity score:",
    round(
        latest["opportunity_score"],
        2
    )
)

print(
    "Zone:",
    latest["zone"]
)

print(
    "\nComponent scores:"
)

for component in score_columns:

    print(
        f"  {component:25s}: "
        f"{latest[component]:.1f}"
    )

print(
    "\nCurrent raw features:"
)

for column in [
    "vol10",
    "vol20",
    "atr_percent",
    "relative_volume",
    "distance_high20",
    "distance_low20"
]:

    if column in latest.index:

        print(
            f"  {column:25s}: "
            f"{latest[column]:.4f}"
        )

# ------------------------------------------------------------
# 12. SAVE RESULTS
# ------------------------------------------------------------

decile_path = (
    f"{BASE}/results/"
    "opportunity_intelligence_deciles_v1.parquet"
)

zone_path = (
    f"{BASE}/results/"
    "opportunity_intelligence_zones_v1.parquet"
)

percentile_path = (
    f"{BASE}/results/"
    "opportunity_intelligence_percentiles_v1.parquet"
)

component_path = (
    f"{BASE}/results/"
    "opportunity_intelligence_components_v1.parquet"
)

deciles.to_parquet(
    decile_path,
    index=False
)

zones.to_parquet(
    zone_path,
    index=False
)

percentiles.to_parquet(
    percentile_path,
    index=False
)

components.to_parquet(
    component_path,
    index=False
)

print("\nSaved:")
print(decile_path)
print(zone_path)
print(percentile_path)
print(component_path)

print("\nSTEP 30 COMPLETE")

In [ ]:
# ============================================================
# STEP 31 — OPPORTUNITY TYPE ENGINE V2
# ============================================================

import pandas as pd
import numpy as np

BASE = "/content/gdrive/MyDrive/SilverAgent"

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

features_path = (
    f"{BASE}/database/"
    "silver_h1_features_v2_horizons.parquet"
)

scores_path = (
    f"{BASE}/results/"
    "continuous_opportunity_v2_history.parquet"
)

features = pd.read_parquet(features_path)
scores = pd.read_parquet(scores_path)

score_columns = [
    "volatility_score",
    "momentum_energy_score",
    "volume_score",
    "displacement_score"
]

scores_small = scores[score_columns].copy()

df = features.join(
    scores_small,
    how="inner"
)

# ------------------------------------------------------------
# 2. OPPORTUNITY SCORE
# ------------------------------------------------------------

df["opportunity_score"] = df[
    score_columns
].mean(axis=1)

# ------------------------------------------------------------
# 3. ADDITIONAL STATE VARIABLES
# ------------------------------------------------------------

df["volatility_expansion"] = (
    df["vol10"] / df["vol20"]
)

df["momentum_intensity"] = (
    df[
        ["mom3", "mom6", "mom12"]
    ]
    .abs()
    .mean(axis=1)
)

# ------------------------------------------------------------
# 4. CLASSIFICATION FUNCTION
# ------------------------------------------------------------

def classify_opportunity(row):

    score = row["opportunity_score"]

    if pd.isna(score):
        return "UNKNOWN"

    # Only classify energetic situations.
    if score < 60:
        return "OTHER"

    mom6 = row["mom6"]
    mom12 = row["mom12"]

    distance_high = row["distance_high20"]
    distance_low = row["distance_low20"]

    vol_expansion = row["volatility_expansion"]

    displacement = abs(
        row["price_vs_MA20"]
    )

    # --------------------------------------------------------
    # BREAKOUT CONDITIONS
    # --------------------------------------------------------

    if (
        score >= 80
        and distance_high <= 0.005
        and pd.notna(mom6)
        and mom6 > 0
    ):
        return "BREAKOUT_UP"

    if (
        score >= 80
        and distance_low <= 0.005
        and pd.notna(mom6)
        and mom6 < 0
    ):
        return "BREAKOUT_DOWN"

    # --------------------------------------------------------
    # MOMENTUM CONDITIONS
    # --------------------------------------------------------

    if (
        score >= 80
        and pd.notna(mom6)
        and pd.notna(mom12)
        and mom6 > 0
        and mom12 > 0
    ):
        return "MOMENTUM_UP"

    if (
        score >= 80
        and pd.notna(mom6)
        and pd.notna(mom12)
        and mom6 < 0
        and mom12 < 0
    ):
        return "MOMENTUM_DOWN"

    # --------------------------------------------------------
    # VOLATILITY EXPANSION
    # --------------------------------------------------------

    if (
        score >= 80
        and pd.notna(vol_expansion)
        and vol_expansion >= 1.25
    ):
        return "VOLATILITY_EXPANSION"

    # --------------------------------------------------------
    # MEAN REVERSION
    # --------------------------------------------------------

    if (
        score >= 80
        and distance_high > 0.02
        and pd.notna(mom6)
        and mom6 < 0
    ):
        return "MEAN_REVERSION"

    if (
        score >= 80
        and distance_low > 0.02
        and pd.notna(mom6)
        and mom6 > 0
    ):
        return "MEAN_REVERSION"

    # --------------------------------------------------------
    # HIGH ENERGY BUT DIRECTION UNCLEAR
    # --------------------------------------------------------

    if score >= 80:
        return "HIGH_ENERGY_UNCLEAR"

    # --------------------------------------------------------
    # HIGH BUT NOT EXTREME
    # --------------------------------------------------------

    return "HIGH_ENERGY_UNCLEAR"


df["opportunity_type"] = df.apply(
    classify_opportunity,
    axis=1
)

# ------------------------------------------------------------
# 5. FUTURE TARGET
# ------------------------------------------------------------

target = "future_abs_24h"

analysis = df.dropna(
    subset=[
        "opportunity_score",
        target
    ]
).copy()

# ------------------------------------------------------------
# 6. TYPE SUMMARY
# ------------------------------------------------------------

summary = (
    analysis
    .groupby("opportunity_type")
    .agg(
        observations=("opportunity_type", "size"),
        mean_score=("opportunity_score", "mean"),
        mean_24h=(target, "mean"),
        median_24h=(target, "median"),
        p90_24h=(
            target,
            lambda x: x.quantile(0.90)
        ),
        p95_24h=(
            target,
            lambda x: x.quantile(0.95)
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 7. LARGE-MOVE THRESHOLDS
# ------------------------------------------------------------

large_threshold = analysis[
    target
].quantile(0.75)

extreme_threshold = analysis[
    target
].quantile(0.90)

summary["large_move_rate"] = (
    analysis
    .groupby("opportunity_type")[target]
    .apply(
        lambda x:
        (x >= large_threshold).mean()
    )
    .values
)

summary["extreme_move_rate"] = (
    analysis
    .groupby("opportunity_type")[target]
    .apply(
        lambda x:
        (x >= extreme_threshold).mean()
    )
    .values
)

# ------------------------------------------------------------
# 8. PRINT SUMMARY
# ------------------------------------------------------------

print("=" * 70)
print("STEP 31 — OPPORTUNITY TYPE ENGINE V2")
print("=" * 70)

print("\nTYPE DISTRIBUTION:")

print(
    analysis[
        "opportunity_type"
    ].value_counts()
)

print("\n" + "=" * 70)
print("TYPE PERFORMANCE")
print("=" * 70)

display(
    summary
    .sort_values(
        "mean_24h",
        ascending=False
    )
    .round(4)
)

# ------------------------------------------------------------
# 9. HIGH-ENERGY ONLY ANALYSIS
# ------------------------------------------------------------

high_energy = analysis[
    analysis["opportunity_score"] >= 80
].copy()

print("\n" + "=" * 70)
print("HIGH-ENERGY TYPE PERFORMANCE")
print("=" * 70)

if len(high_energy) > 0:

    high_summary = (
        high_energy
        .groupby("opportunity_type")
        .agg(
            observations=(
                "opportunity_type",
                "size"
            ),

            mean_score=(
                "opportunity_score",
                "mean"
            ),

            mean_24h=(
                target,
                "mean"
            ),

            median_24h=(
                target,
                "median"
            ),

            p90_24h=(
                target,
                lambda x:
                x.quantile(0.90)
            ),

            p95_24h=(
                target,
                lambda x:
                x.quantile(0.95)
            )
        )
        .reset_index()
    )

    high_summary["large_move_rate"] = (
        high_energy
        .groupby("opportunity_type")[target]
        .apply(
            lambda x:
            (x >= large_threshold).mean()
        )
        .values
    )

    high_summary["extreme_move_rate"] = (
        high_energy
        .groupby("opportunity_type")[target]
        .apply(
            lambda x:
            (x >= extreme_threshold).mean()
        )
        .values
    )

    display(
        high_summary
        .sort_values(
            "mean_24h",
            ascending=False
        )
        .round(4)
    )

else:

    print("No high-energy observations.")

# ------------------------------------------------------------
# 10. CURRENT STATE
# ------------------------------------------------------------

latest = df.iloc[-1]

print("\n" + "=" * 70)
print("CURRENT OPPORTUNITY INTELLIGENCE")
print("=" * 70)

print(
    "\nTimestamp:",
    latest.name
)

print(
    "Price:",
    round(
        latest["close"],
        4
    )
)

print(
    "Opportunity:",
    round(
        latest["opportunity_score"],
        2
    )
)

print(
    "Type:",
    latest["opportunity_type"]
)

print("\nComponents:")

for column in score_columns:

    print(
        f"  {column:25s}: "
        f"{latest[column]:.1f}"
    )

print("\nMarket state:")

for column in [
    "vol10",
    "vol20",
    "atr_percent",
    "relative_volume",
    "distance_high20",
    "distance_low20",
    "mom3",
    "mom6",
    "mom12",
    "price_vs_MA20"
]:

    if column in latest.index:

        value = latest[column]

        if pd.isna(value):

            print(
                f"  {column:25s}: NaN"
            )

        else:

            print(
                f"  {column:25s}: "
                f"{value:.5f}"
            )

# ------------------------------------------------------------
# 11. HUMAN-READABLE EXPLANATION
# ------------------------------------------------------------

score = latest["opportunity_score"]

vol_score = latest["volatility_score"]
mom_score = latest["momentum_energy_score"]
volume_score = latest["volume_score"]
disp_score = latest["displacement_score"]

explanation = []

if vol_score >= 80:
    explanation.append(
        "volatility is unusually elevated"
    )

if mom_score >= 80:
    explanation.append(
        "momentum energy is unusually elevated"
    )

if volume_score >= 80:
    explanation.append(
        "volume activity is unusually elevated"
    )
elif volume_score < 40:
    explanation.append(
        "volume confirmation is weak"
    )

if disp_score >= 80:
    explanation.append(
        "price displacement is unusually elevated"
    )

print("\n" + "=" * 70)
print("MACHINE EXPLANATION")
print("=" * 70)

if score >= 80:

    print(
        "\nHIGH-ENERGY CONDITION."
    )

elif score >= 60:

    print(
        "\nELEVATED-ENERGY CONDITION."
    )

else:

    print(
        "\nNORMAL / LOW-ENERGY CONDITION."
    )

if explanation:

    print(
        "\nThe machine detects that "
        + ", ".join(explanation)
        + "."
    )

# ------------------------------------------------------------
# 12. SAVE
# ------------------------------------------------------------

history_path = (
    f"{BASE}/results/"
    "opportunity_type_v2_history.parquet"
)

summary_path = (
    f"{BASE}/results/"
    "opportunity_type_v2_summary.parquet"
)

analysis.to_parquet(
    history_path,
    index=True
)

summary.to_parquet(
    summary_path,
    index=False
)

print("\nSaved:")
print(history_path)
print(summary_path)

print("\nSTEP 31 COMPLETE")

In [ ]:
# ============================================================
# STEP 32 — LIVE-SAFE OPPORTUNITY INTELLIGENCE ENGINE
# ============================================================

import pandas as pd
import numpy as np

BASE = "/content/gdrive/MyDrive/SilverAgent"

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

features_path = (
    f"{BASE}/database/"
    "silver_h1_features_v2_horizons.parquet"
)

scores_path = (
    f"{BASE}/results/"
    "continuous_opportunity_v2_history.parquet"
)

features = pd.read_parquet(features_path)
scores = pd.read_parquet(scores_path)

score_columns = [
    "volatility_score",
    "momentum_energy_score",
    "volume_score",
    "displacement_score"
]

scores_small = scores[score_columns].copy()

df = features.join(
    scores_small,
    how="inner"
)

# ------------------------------------------------------------
# 2. BUILD LIVE-SAFE OPPORTUNITY SCORE
# ------------------------------------------------------------
#
# IMPORTANT:
# We do NOT treat missing components as zero.
#
# The score is calculated from available components only,
# and confidence is reduced when components are missing.
# ------------------------------------------------------------

df["available_components"] = df[
    score_columns
].notna().sum(axis=1)

df["opportunity_score_live"] = (
    df[score_columns]
    .mean(axis=1, skipna=True)
)

df["data_completeness"] = (
    df["available_components"]
    / len(score_columns)
)

# ------------------------------------------------------------
# 3. RAW MARKET STATE
# ------------------------------------------------------------

df["volatility_expansion"] = (
    df["vol10"] / df["vol20"]
)

df["momentum_available"] = (
    df[
        ["mom3", "mom6", "mom12"]
    ]
    .notna()
    .sum(axis=1)
)

df["momentum_direction"] = np.select(
    [
        (
            (df["mom6"] > 0)
            & (df["mom12"] > 0)
        ),

        (
            (df["mom6"] < 0)
            & (df["mom12"] < 0)
        )
    ],
    [
        "UP",
        "DOWN"
    ],
    default="UNCLEAR"
)

# ------------------------------------------------------------
# 4. LIVE-SAFE TYPE CLASSIFICATION
# ------------------------------------------------------------

def classify_live(row):

    score = row["opportunity_score_live"]

    if pd.isna(score):
        return "UNKNOWN"

    if score < 60:
        return "OTHER"

    mom6 = row["mom6"]
    mom12 = row["mom12"]

    distance_high = row["distance_high20"]
    distance_low = row["distance_low20"]

    vol_expansion = row["volatility_expansion"]

    # --------------------------------------------------------
    # EXTREME OPPORTUNITY TYPES
    # --------------------------------------------------------

    if score >= 80:

        # Breakout upward
        if (
            pd.notna(mom6)
            and distance_high <= 0.005
            and mom6 > 0
        ):
            return "BREAKOUT_UP"

        # Breakout downward
        if (
            pd.notna(mom6)
            and distance_low <= 0.005
            and mom6 < 0
        ):
            return "BREAKOUT_DOWN"

        # Momentum upward
        if (
            pd.notna(mom6)
            and pd.notna(mom12)
            and mom6 > 0
            and mom12 > 0
        ):
            return "MOMENTUM_UP"

        # Momentum downward
        if (
            pd.notna(mom6)
            and pd.notna(mom12)
            and mom6 < 0
            and mom12 < 0
        ):
            return "MOMENTUM_DOWN"

        # Volatility expansion
        if (
            pd.notna(vol_expansion)
            and vol_expansion >= 1.25
        ):
            return "VOLATILITY_EXPANSION"

        # Mean reversion
        if (
            distance_high > 0.02
            and pd.notna(mom6)
            and mom6 < 0
        ):
            return "MEAN_REVERSION"

        if (
            distance_low > 0.02
            and pd.notna(mom6)
            and mom6 > 0
        ):
            return "MEAN_REVERSION"

        return "HIGH_ENERGY_UNCLEAR"

    # --------------------------------------------------------
    # HIGH BUT NOT EXTREME
    # --------------------------------------------------------

    if score >= 60:
        return "ELEVATED_ENERGY"

    return "OTHER"


df["opportunity_type_live"] = df.apply(
    classify_live,
    axis=1
)

# ------------------------------------------------------------
# 5. CONFIDENCE
# ------------------------------------------------------------
#
# Confidence is deliberately NOT a probability of profit.
#
# It represents confidence that the machine has enough
# information to classify the current market state.
# ------------------------------------------------------------

def calculate_confidence(row):

    score = row["opportunity_score_live"]

    if pd.isna(score):
        return 0

    completeness = row["data_completeness"]

    momentum_complete = (
        row["momentum_available"] / 3
    )

    confidence = (
        0.70 * completeness
        + 0.30 * momentum_complete
    )

    # Convert to percentage
    return round(
        confidence * 100,
        1
    )


df["confidence"] = df.apply(
    calculate_confidence,
    axis=1
)

# ------------------------------------------------------------
# 6. CONFIDENCE LABEL
# ------------------------------------------------------------

def confidence_label(value):

    if value >= 90:
        return "HIGH"

    if value >= 70:
        return "MEDIUM"

    if value >= 50:
        return "LOW"

    return "VERY_LOW"


df["confidence_label"] = (
    df["confidence"]
    .apply(confidence_label)
)

# ------------------------------------------------------------
# 7. MACHINE REASONS
# ------------------------------------------------------------

def build_reasons(row):

    reasons = []

    if pd.notna(
        row["volatility_score"]
    ):

        if row["volatility_score"] >= 80:
            reasons.append(
                "volatility unusually elevated"
            )

        elif row["volatility_score"] >= 60:
            reasons.append(
                "volatility elevated"
            )

    if pd.notna(
        row["momentum_energy_score"]
    ):

        if row["momentum_energy_score"] >= 80:
            reasons.append(
                "momentum energy unusually elevated"
            )

        elif row["momentum_energy_score"] >= 60:
            reasons.append(
                "momentum energy elevated"
            )

    if pd.notna(
        row["volume_score"]
    ):

        if row["volume_score"] >= 80:
            reasons.append(
                "strong volume confirmation"
            )

        elif row["volume_score"] < 40:
            reasons.append(
                "weak volume confirmation"
            )

    if pd.notna(
        row["displacement_score"]
    ):

        if row["displacement_score"] >= 80:
            reasons.append(
                "price displacement unusually high"
            )

        elif row["displacement_score"] >= 60:
            reasons.append(
                "price displacement elevated"
            )

    if row["momentum_direction"] == "UP":
        reasons.append(
            "short-term momentum aligned upward"
        )

    elif row["momentum_direction"] == "DOWN":
        reasons.append(
            "short-term momentum aligned downward"
        )

    if row["available_components"] < 4:
        reasons.append(
            "one or more intelligence components unavailable"
        )

    return reasons


df["machine_reasons"] = df.apply(
    build_reasons,
    axis=1
)

# ------------------------------------------------------------
# 8. CURRENT STATE
# ------------------------------------------------------------

latest = df.iloc[-1]

print("=" * 70)
print("STEP 32 — LIVE-SAFE OPPORTUNITY INTELLIGENCE")
print("=" * 70)

print("\nCURRENT MARKET")

print(
    "\nTimestamp:",
    latest.name
)

print(
    "Price:",
    round(
        latest["close"],
        4
    )
)

print(
    "Opportunity score:",
    round(
        latest["opportunity_score_live"],
        2
    )
)

print(
    "Opportunity type:",
    latest["opportunity_type_live"]
)

print(
    "Confidence:",
    f"{latest['confidence']:.1f}% "
    f"({latest['confidence_label']})"
)

print(
    "Components available:",
    f"{int(latest['available_components'])}/4"
)

print("\nCOMPONENTS")

for column in score_columns:

    value = latest[column]

    if pd.isna(value):

        print(
            f"  {column:25s}: unavailable"
        )

    else:

        print(
            f"  {column:25s}: "
            f"{value:.1f}"
        )

print("\nMARKET STATE")

market_columns = [
    "vol10",
    "vol20",
    "atr_percent",
    "relative_volume",
    "distance_high20",
    "distance_low20",
    "mom3",
    "mom6",
    "mom12",
    "price_vs_MA20",
    "volatility_expansion"
]

for column in market_columns:

    if column not in latest.index:
        continue

    value = latest[column]

    if pd.isna(value):

        print(
            f"  {column:25s}: unavailable"
        )

    else:

        print(
            f"  {column:25s}: "
            f"{value:.5f}"
        )

print("\nMACHINE REASONS")

for reason in latest["machine_reasons"]:

    print(
        "  •",
        reason
    )

# ------------------------------------------------------------
# 9. HUMAN-READABLE ASSESSMENT
# ------------------------------------------------------------

score = latest["opportunity_score_live"]
opportunity_type = latest[
    "opportunity_type_live"
]

confidence = latest["confidence"]

print("\n" + "=" * 70)
print("MACHINE ASSESSMENT")
print("=" * 70)

if score >= 80:

    print(
        "\n⚠️ VERY HIGH OPPORTUNITY / "
        "HIGH-ENERGY MARKET"
    )

elif score >= 60:

    print(
        "\n🟠 ELEVATED OPPORTUNITY"
    )

elif score >= 40:

    print(
        "\n🟡 MEDIUM OPPORTUNITY"
    )

else:

    print(
        "\n🟢 LOW OPPORTUNITY"
    )

print(
    "\nType:",
    opportunity_type
)

print(
    "Classification confidence:",
    f"{confidence:.1f}%"
)

if opportunity_type in [
    "MOMENTUM_UP",
    "BREAKOUT_UP"
]:

    print(
        "\nDirectional state: "
        "UPWARD BIAS"
    )

elif opportunity_type in [
    "MOMENTUM_DOWN",
    "BREAKOUT_DOWN"
]:

    print(
        "\nDirectional state: "
        "DOWNWARD BIAS"
    )

elif opportunity_type == "MEAN_REVERSION":

    print(
        "\nDirectional state: "
        "REVERSION / STRETCHED"
    )

else:

    print(
        "\nDirectional state: "
        "UNCONFIRMED"
    )

# ------------------------------------------------------------
# 10. SAVE LIVE-SAFE HISTORY
# ------------------------------------------------------------

save_columns = [
    "opportunity_score_live",
    "opportunity_type_live",
    "confidence",
    "confidence_label",
    "available_components",
    "data_completeness",
    "momentum_available",
    "momentum_direction",
    "volatility_expansion"
]

live_history = df[
    save_columns
].copy()

live_path = (
    f"{BASE}/results/"
    "opportunity_live_safe_v1_history.parquet"
)

live_history.to_parquet(
    live_path,
    index=True
)

print("\nSaved:")
print(live_path)

print("\nSTEP 32 COMPLETE")

In [ ]:
# ============================================================
# STEP 33 — WALK-FORWARD OPPORTUNITY TYPE VALIDATION
# ============================================================

import pandas as pd
import numpy as np

BASE = "/content/gdrive/MyDrive/SilverAgent"

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

features_path = (
    f"{BASE}/database/"
    "silver_h1_features_v2_horizons.parquet"
)

scores_path = (
    f"{BASE}/results/"
    "continuous_opportunity_v2_history.parquet"
)

features = pd.read_parquet(features_path)
scores = pd.read_parquet(scores_path)

score_columns = [
    "volatility_score",
    "momentum_energy_score",
    "volume_score",
    "displacement_score"
]

scores_small = scores[score_columns].copy()

df = features.join(
    scores_small,
    how="inner"
)

# ------------------------------------------------------------
# 2. OPPORTUNITY SCORE
# ------------------------------------------------------------

df["opportunity_score"] = (
    df[score_columns].mean(axis=1)
)

# ------------------------------------------------------------
# 3. STATE VARIABLES
# ------------------------------------------------------------

df["volatility_expansion"] = (
    df["vol10"] / df["vol20"]
)

# ------------------------------------------------------------
# 4. FIXED TYPE ENGINE
# ------------------------------------------------------------

def classify_opportunity(row):

    score = row["opportunity_score"]

    if pd.isna(score):
        return "UNKNOWN"

    if score < 60:
        return "OTHER"

    mom6 = row["mom6"]
    mom12 = row["mom12"]

    distance_high = row["distance_high20"]
    distance_low = row["distance_low20"]

    vol_expansion = row["volatility_expansion"]

    # --------------------------------------------------------
    # BREAKOUT
    # --------------------------------------------------------

    if (
        score >= 80
        and pd.notna(mom6)
        and distance_high <= 0.005
        and mom6 > 0
    ):
        return "BREAKOUT_UP"

    if (
        score >= 80
        and pd.notna(mom6)
        and distance_low <= 0.005
        and mom6 < 0
    ):
        return "BREAKOUT_DOWN"

    # --------------------------------------------------------
    # MOMENTUM
    # --------------------------------------------------------

    if (
        score >= 80
        and pd.notna(mom6)
        and pd.notna(mom12)
        and mom6 > 0
        and mom12 > 0
    ):
        return "MOMENTUM_UP"

    if (
        score >= 80
        and pd.notna(mom6)
        and pd.notna(mom12)
        and mom6 < 0
        and mom12 < 0
    ):
        return "MOMENTUM_DOWN"

    # --------------------------------------------------------
    # VOLATILITY EXPANSION
    # --------------------------------------------------------

    if (
        score >= 80
        and pd.notna(vol_expansion)
        and vol_expansion >= 1.25
    ):
        return "VOLATILITY_EXPANSION"

    # --------------------------------------------------------
    # MEAN REVERSION
    # --------------------------------------------------------

    if (
        score >= 80
        and distance_high > 0.02
        and pd.notna(mom6)
        and mom6 < 0
    ):
        return "MEAN_REVERSION"

    if (
        score >= 80
        and distance_low > 0.02
        and pd.notna(mom6)
        and mom6 > 0
    ):
        return "MEAN_REVERSION"

    # --------------------------------------------------------
    # HIGH ENERGY BUT UNCLEAR
    # --------------------------------------------------------

    if score >= 80:
        return "HIGH_ENERGY_UNCLEAR"

    return "OTHER"


df["opportunity_type"] = df.apply(
    classify_opportunity,
    axis=1
)

# ------------------------------------------------------------
# 5. REMOVE ROWS WITHOUT FUTURE DATA
# ------------------------------------------------------------

required_targets = [
    "future_abs_6h",
    "future_abs_12h",
    "future_abs_24h"
]

df = df.dropna(
    subset=required_targets
).copy()

df = df.sort_index()

# ------------------------------------------------------------
# 6. WALK-FORWARD SETTINGS
# ------------------------------------------------------------

TRAIN_SIZE = 4000
TEST_SIZE = 1000
STEP = 1000

start = TRAIN_SIZE

window_results = []

type_results = []

window_number = 0

# ------------------------------------------------------------
# 7. WALK FORWARD
# ------------------------------------------------------------

while start + TEST_SIZE <= len(df):

    window_number += 1

    train = df.iloc[
        start - TRAIN_SIZE:start
    ].copy()

    test = df.iloc[
        start:start + TEST_SIZE
    ].copy()

    # --------------------------------------------------------
    # Determine high-opportunity threshold using TRAIN ONLY
    # --------------------------------------------------------

    high_threshold = train[
        "opportunity_score"
    ].quantile(0.80)

    test_high = test[
        test["opportunity_score"] >= high_threshold
    ].copy()

    # --------------------------------------------------------
    # Test every type that occurs in this period
    # --------------------------------------------------------

    for opportunity_type in sorted(
        test["opportunity_type"].unique()
    ):

        group = test[
            test["opportunity_type"]
            == opportunity_type
        ].copy()

        if len(group) < 20:
            continue

        result = {
            "window": window_number,
            "type": opportunity_type,
            "signals": len(group),

            "mean_6h":
                group["future_abs_6h"].mean(),

            "mean_12h":
                group["future_abs_12h"].mean(),

            "mean_24h":
                group["future_abs_24h"].mean(),

            "median_24h":
                group["future_abs_24h"].median(),

            "p90_24h":
                group["future_abs_24h"].quantile(0.90),

            "p95_24h":
                group["future_abs_24h"].quantile(0.95)
        }

        # ----------------------------------------------------
        # Large/extreme thresholds determined from TRAIN ONLY
        # ----------------------------------------------------

        large_threshold = train[
            "future_abs_24h"
        ].quantile(0.75)

        extreme_threshold = train[
            "future_abs_24h"
        ].quantile(0.90)

        result["large_move_rate"] = (
            group["future_abs_24h"]
            >= large_threshold
        ).mean()

        result["extreme_move_rate"] = (
            group["future_abs_24h"]
            >= extreme_threshold
        ).mean()

        type_results.append(result)

    # --------------------------------------------------------
    # Overall test baseline
    # --------------------------------------------------------

    window_results.append({

        "window": window_number,

        "start":
            test.index.min(),

        "end":
            test.index.max(),

        "baseline_6h":
            test["future_abs_6h"].mean(),

        "baseline_12h":
            test["future_abs_12h"].mean(),

        "baseline_24h":
            test["future_abs_24h"].mean(),

        "high_opportunity_signals":
            len(test_high),

        "high_opportunity_24h":
            (
                test_high["future_abs_24h"].mean()
                if len(test_high) > 0
                else np.nan
            )
    })

    print(
        f"Window {window_number:02d} | "
        f"{test.index.min()} → "
        f"{test.index.max()} | "
        f"High-op signals: "
        f"{len(test_high)}"
    )

    start += STEP

# ------------------------------------------------------------
# 8. BUILD DATAFRAMES
# ------------------------------------------------------------

windows = pd.DataFrame(
    window_results
)

types = pd.DataFrame(
    type_results
)

# ------------------------------------------------------------
# 9. TYPE SUMMARY
# ------------------------------------------------------------

type_summary = (
    types
    .groupby("type")
    .agg(
        windows=("window", "nunique"),
        signals=("signals", "sum"),

        mean_6h=("mean_6h", "mean"),
        mean_12h=("mean_12h", "mean"),
        mean_24h=("mean_24h", "mean"),

        median_24h=("median_24h", "mean"),

        mean_p90=("p90_24h", "mean"),
        mean_p95=("p95_24h", "mean"),

        mean_large_rate=(
            "large_move_rate",
            "mean"
        ),

        mean_extreme_rate=(
            "extreme_move_rate",
            "mean"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 10. CONSISTENCY
# ------------------------------------------------------------

consistency = (
    types
    .groupby("type")
    .agg(
        positive_windows=(
            "mean_24h",
            lambda x:
            (x > 0).sum()
        )
    )
    .reset_index()
)

type_summary = type_summary.merge(
    consistency,
    on="type",
    how="left"
)

type_summary["positive_window_rate"] = (
    type_summary["positive_windows"]
    / type_summary["windows"]
)

# ------------------------------------------------------------
# 11. BASELINE SUMMARY
# ------------------------------------------------------------

baseline_24h = (
    windows["baseline_24h"].mean()
)

baseline_12h = (
    windows["baseline_12h"].mean()
)

baseline_6h = (
    windows["baseline_6h"].mean()
)

print("\n" + "=" * 70)
print("STEP 33 — WALK-FORWARD TYPE VALIDATION")
print("=" * 70)

print(
    "\nWindows tested:",
    len(windows)
)

print(
    "\nAverage baseline movement:"
)

print(
    f"  6H  : {baseline_6h:.3%}"
)

print(
    f"  12H : {baseline_12h:.3%}"
)

print(
    f"  24H : {baseline_24h:.3%}"
)

print("\n" + "=" * 70)
print("TYPE RESULTS")
print("=" * 70)

display(
    type_summary
    .sort_values(
        "mean_24h",
        ascending=False
    )
    .round(4)
)

# ------------------------------------------------------------
# 12. HIGH OPPORTUNITY PERFORMANCE
# ------------------------------------------------------------

weighted_high = np.average(
    windows[
        "high_opportunity_24h"
    ].dropna(),
    weights=windows.loc[
        windows["high_opportunity_24h"].notna(),
        "high_opportunity_signals"
    ]
)

print("\n" + "=" * 70)
print("HIGH OPPORTUNITY VALIDATION")
print("=" * 70)

print(
    f"\nBaseline 24h movement: "
    f"{baseline_24h:.3%}"
)

print(
    f"High-opportunity 24h movement: "
    f"{weighted_high:.3%}"
)

print(
    f"Improvement: "
    f"{weighted_high - baseline_24h:+.3%}"
)

print(
    "\nPositive high-opportunity windows:",
    (
        windows[
            "high_opportunity_24h"
        ]
        > windows["baseline_24h"]
    ).sum(),
    "/",
    len(windows)
)

# ------------------------------------------------------------
# 13. SAVE
# ------------------------------------------------------------

windows_path = (
    f"{BASE}/results/"
    "opportunity_type_v2_walk_forward_windows.parquet"
)

types_path = (
    f"{BASE}/results/"
    "opportunity_type_v2_walk_forward_types.parquet"
)

summary_path = (
    f"{BASE}/results/"
    "opportunity_type_v2_walk_forward_summary.parquet"
)

windows.to_parquet(
    windows_path,
    index=False
)

types.to_parquet(
    types_path,
    index=False
)

type_summary.to_parquet(
    summary_path,
    index=False
)

print("\nSaved:")
print(windows_path)
print(types_path)
print(summary_path)

print("\nSTEP 33 COMPLETE")

In [ ]:
# ============================================================
# STEP 34 — OPPORTUNITY EVENT STUDY
# ============================================================

import pandas as pd
import numpy as np

BASE = "/content/gdrive/MyDrive/SilverAgent"

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

features_path = (
    f"{BASE}/database/"
    "silver_h1_features_v2_horizons.parquet"
)

scores_path = (
    f"{BASE}/results/"
    "continuous_opportunity_v2_history.parquet"
)

features = pd.read_parquet(features_path)
scores = pd.read_parquet(scores_path)

score_columns = [
    "volatility_score",
    "momentum_energy_score",
    "volume_score",
    "displacement_score"
]

scores_small = scores[score_columns].copy()

df = features.join(
    scores_small,
    how="inner"
)

df = df.sort_index()

# ------------------------------------------------------------
# 2. OPPORTUNITY SCORE
# ------------------------------------------------------------

df["opportunity_score"] = (
    df[score_columns].mean(axis=1)
)

# ------------------------------------------------------------
# 3. CREATE FUTURE SIGNED RETURNS
# ------------------------------------------------------------
#
# We use the actual future close from the gap-aware future
# columns where available.
#
# Existing future_6h / future_12h / future_24h are already
# created using nearest-real-candle matching.
# ------------------------------------------------------------

for horizon in ["6h", "12h", "24h"]:

    future_column = f"future_{horizon}"

    if future_column in df.columns:

        df[
            f"signed_{horizon}"
        ] = df[future_column]

# ------------------------------------------------------------
# 4. BUILD EVENT GROUPS
# ------------------------------------------------------------

df["opportunity_group"] = np.select(

    [
        df["opportunity_score"] >= 80,
        df["opportunity_score"] >= 60
    ],

    [
        "VERY_HIGH",
        "HIGH"
    ],

    default="NORMAL"
)

# ------------------------------------------------------------
# 5. ANALYSE AVAILABLE FUTURE MOVEMENT
# ------------------------------------------------------------

analysis_columns = [
    "opportunity_score",
    "future_abs_6h",
    "future_abs_12h",
    "future_abs_24h",
    "signed_6h",
    "signed_12h",
    "signed_24h"
]

analysis = df.dropna(
    subset=analysis_columns
).copy()

# ------------------------------------------------------------
# 6. BASIC GROUP SUMMARY
# ------------------------------------------------------------

group_summary = []

for group_name, group in analysis.groupby(
    "opportunity_group"
):

    row = {
        "group": group_name,
        "observations": len(group),
        "mean_score":
            group["opportunity_score"].mean()
    }

    for horizon in ["6h", "12h", "24h"]:

        abs_col = f"future_abs_{horizon}"
        signed_col = f"signed_{horizon}"

        values = group[abs_col]

        signed = group[signed_col]

        row[f"{horizon}_abs_mean"] = values.mean()
        row[f"{horizon}_abs_median"] = values.median()

        row[f"{horizon}_signed_mean"] = signed.mean()

        row[f"{horizon}_up_rate"] = (
            signed > 0
        ).mean()

        row[f"{horizon}_p90"] = (
            values.quantile(0.90)
        )

    group_summary.append(row)

group_summary = pd.DataFrame(
    group_summary
)

print("=" * 70)
print("STEP 34 — OPPORTUNITY EVENT STUDY")
print("=" * 70)

print("\nGROUP SUMMARY")

display(
    group_summary.round(4)
)

# ------------------------------------------------------------
# 7. EVENT MAGNITUDE DISTRIBUTION
# ------------------------------------------------------------

large_threshold = analysis[
    "future_abs_24h"
].quantile(0.75)

extreme_threshold = analysis[
    "future_abs_24h"
].quantile(0.90)

very_extreme_threshold = analysis[
    "future_abs_24h"
].quantile(0.95)

print("\n" + "=" * 70)
print("24H EVENT THRESHOLDS")
print("=" * 70)

print(
    f"\nLarge move:       "
    f"{large_threshold:.3%}"
)

print(
    f"Extreme move:     "
    f"{extreme_threshold:.3%}"
)

print(
    f"Very extreme:     "
    f"{very_extreme_threshold:.3%}"
)

# ------------------------------------------------------------
# 8. EVENT PROBABILITIES BY GROUP
# ------------------------------------------------------------

event_rows = []

for group_name, group in analysis.groupby(
    "opportunity_group"
):

    values = group[
        "future_abs_24h"
    ]

    event_rows.append({

        "group": group_name,

        "observations": len(group),

        "large_move_rate":
            (values >= large_threshold).mean(),

        "extreme_move_rate":
            (values >= extreme_threshold).mean(),

        "very_extreme_rate":
            (values >= very_extreme_threshold).mean(),

        "mean_abs_move":
            values.mean(),

        "median_abs_move":
            values.median(),

        "p90_abs_move":
            values.quantile(0.90),

        "p95_abs_move":
            values.quantile(0.95)
    })

event_summary = pd.DataFrame(
    event_rows
)

print("\n" + "=" * 70)
print("EVENT PROBABILITIES")
print("=" * 70)

display(
    event_summary.round(4)
)

# ------------------------------------------------------------
# 9. UPSIDE / DOWNSIDE BALANCE
# ------------------------------------------------------------

direction_rows = []

for group_name, group in analysis.groupby(
    "opportunity_group"
):

    signed = group["signed_24h"]

    upside = signed[
        signed > 0
    ]

    downside = signed[
        signed < 0
    ]

    direction_rows.append({

        "group": group_name,

        "observations": len(group),

        "up_rate":
            (signed > 0).mean(),

        "down_rate":
            (signed < 0).mean(),

        "mean_signed":
            signed.mean(),

        "median_signed":
            signed.median(),

        "mean_up_move":
            upside.mean()
            if len(upside) > 0
            else np.nan,

        "mean_down_move":
            downside.mean()
            if len(downside) > 0
            else np.nan,

        "largest_up":
            signed.max(),

        "largest_down":
            signed.min()
    })

direction_summary = pd.DataFrame(
    direction_rows
)

print("\n" + "=" * 70)
print("UPSIDE / DOWNSIDE BALANCE")
print("=" * 70)

display(
    direction_summary.round(4)
)

# ------------------------------------------------------------
# 10. TOP 10% EVENT STUDY
# ------------------------------------------------------------

top10_cutoff = analysis[
    "opportunity_score"
].quantile(0.90)

top10 = analysis[
    analysis["opportunity_score"]
    >= top10_cutoff
].copy()

print("\n" + "=" * 70)
print("TOP 10% OPPORTUNITY EVENT STUDY")
print("=" * 70)

print(
    "\nObservations:",
    len(top10)
)

print(
    "Score cutoff:",
    round(top10_cutoff, 2)
)

for horizon in ["6h", "12h", "24h"]:

    values = top10[
        f"future_abs_{horizon}"
    ]

    signed = top10[
        f"signed_{horizon}"
    ]

    print(
        f"\n{horizon.upper()}"
    )

    print(
        f"  Mean absolute move: "
        f"{values.mean():.3%}"
    )

    print(
        f"  Median absolute move: "
        f"{values.median():.3%}"
    )

    print(
        f"  P90: "
        f"{values.quantile(0.90):.3%}"
    )

    print(
        f"  P95: "
        f"{values.quantile(0.95):.3%}"
    )

    print(
        f"  Up rate: "
        f"{(signed > 0).mean():.2%}"
    )

    print(
        f"  Mean signed move: "
        f"{signed.mean():+.3%}"
    )

# ------------------------------------------------------------
# 11. EVENT PERSISTENCE
# ------------------------------------------------------------
#
# How long does elevated opportunity tend to remain elevated?
# ------------------------------------------------------------

score = df[
    "opportunity_score"
]

high_mask = score >= 60
very_high_mask = score >= 80

persistence_rows = []

for name, mask in [
    ("HIGH+", high_mask),
    ("VERY_HIGH", very_high_mask)
]:

    runs = []

    current_run = 0

    for value in mask:

        if value:

            current_run += 1

        else:

            if current_run > 0:
                runs.append(current_run)

            current_run = 0

    if current_run > 0:
        runs.append(current_run)

    if len(runs) > 0:

        persistence_rows.append({

            "group": name,

            "events": len(runs),

            "mean_hours":
                np.mean(runs),

            "median_hours":
                np.median(runs),

            "p75_hours":
                np.percentile(
                    runs,
                    75
                ),

            "max_hours":
                np.max(runs)
        })

persistence = pd.DataFrame(
    persistence_rows
)

print("\n" + "=" * 70)
print("OPPORTUNITY PERSISTENCE")
print("=" * 70)

display(
    persistence.round(2)
)

# ------------------------------------------------------------
# 12. CURRENT STATE
# ------------------------------------------------------------

latest = df.iloc[-1]

print("\n" + "=" * 70)
print("CURRENT EVENT-STATE")
print("=" * 70)

print(
    "\nTimestamp:",
    latest.name
)

print(
    "Price:",
    round(
        latest["close"],
        4
    )
)

print(
    "Opportunity:",
    round(
        latest["opportunity_score"],
        2
    )
)

print(
    "Group:",
    latest["opportunity_group"]
)

# ------------------------------------------------------------
# 13. SAVE
# ------------------------------------------------------------

group_path = (
    f"{BASE}/results/"
    "opportunity_event_study_groups_v1.parquet"
)

event_path = (
    f"{BASE}/results/"
    "opportunity_event_study_events_v1.parquet"
)

direction_path = (
    f"{BASE}/results/"
    "opportunity_event_study_direction_v1.parquet"
)

persistence_path = (
    f"{BASE}/results/"
    "opportunity_event_study_persistence_v1.parquet"
)

group_summary.to_parquet(
    group_path,
    index=False
)

event_summary.to_parquet(
    event_path,
    index=False
)

direction_summary.to_parquet(
    direction_path,
    index=False
)

persistence.to_parquet(
    persistence_path,
    index=False
)

print("\nSaved:")
print(group_path)
print(event_path)
print(direction_path)
print(persistence_path)

print("\nSTEP 34 COMPLETE")

In [ ]:

# ============================================================
# STEP 35 — OPPORTUNITY EPISODE ENGINE
# ============================================================

import pandas as pd
import numpy as np

BASE = "/content/gdrive/MyDrive/SilverAgent"

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

features_path = (
    f"{BASE}/database/"
    "silver_h1_features_v2_horizons.parquet"
)

scores_path = (
    f"{BASE}/results/"
    "continuous_opportunity_v2_history.parquet"
)

features = pd.read_parquet(features_path)
scores = pd.read_parquet(scores_path)

score_columns = [
    "volatility_score",
    "momentum_energy_score",
    "volume_score",
    "displacement_score"
]

df = features.join(
    scores[score_columns],
    how="inner"
)

df = df.sort_index()

# ------------------------------------------------------------
# 2. OPPORTUNITY SCORE
# ------------------------------------------------------------

df["opportunity_score"] = (
    df[score_columns].mean(axis=1)
)

# ------------------------------------------------------------
# 3. DEFINE OPPORTUNITY STATE
# ------------------------------------------------------------

df["state"] = np.select(
    [
        df["opportunity_score"] >= 80,
        df["opportunity_score"] >= 60
    ],
    [
        "VERY_HIGH",
        "HIGH"
    ],
    default="NORMAL"
)

# ------------------------------------------------------------
# 4. IDENTIFY CONTIGUOUS HIGH-OPPORTUNITY EPISODES
# ------------------------------------------------------------
#
# HIGH+ means score >= 60.
#
# A new episode begins when:
#
#   - the previous observation was NORMAL
#   - OR there is a large time gap
#
# We do NOT fabricate missing candles.
# ------------------------------------------------------------

df["high_opportunity"] = (
    df["opportunity_score"] >= 60
)

time_gap = (
    df.index.to_series()
    .diff()
)

new_episode = (
    df["high_opportunity"]
    &
    (
        (~df["high_opportunity"].shift(1).fillna(False))
        |
        (time_gap > pd.Timedelta(minutes=90))
    )
)

df["episode_id"] = (
    new_episode
    .cumsum()
)

# Normal rows should not belong to an episode.

df.loc[
    ~df["high_opportunity"],
    "episode_id"
] = np.nan

# ------------------------------------------------------------
# 5. EXTRACT EPISODES
# ------------------------------------------------------------

episode_rows = []

for episode_id, group in df[
    df["high_opportunity"]
].groupby(
    "episode_id"
):

    if len(group) == 0:
        continue

    start_time = group.index[0]
    end_time = group.index[-1]

    duration_hours = (
        end_time - start_time
    ).total_seconds() / 3600

    peak_idx = (
        group["opportunity_score"]
        .idxmax()
    )

    peak_score = (
        group["opportunity_score"]
        .max()
    )

    start_score = (
        group["opportunity_score"]
        .iloc[0]
    )

    end_score = (
        group["opportunity_score"]
        .iloc[-1]
    )

    episode_rows.append({

        "episode_id":
            int(episode_id),

        "start_time":
            start_time,

        "end_time":
            end_time,

        "duration_hours":
            duration_hours,

        "observations":
            len(group),

        "start_score":
            start_score,

        "peak_score":
            peak_score,

        "end_score":
            end_score,

        "score_change":
            end_score - start_score,

        "peak_time":
            peak_idx,

        "peak_delay_hours":
            (
                peak_idx - start_time
            ).total_seconds() / 3600,

        "state_at_start":
            group["state"].iloc[0],

        "state_at_peak":
            group.loc[
                peak_idx,
                "state"
            ],

        "start_price":
            group["close"].iloc[0]
    })

episodes = pd.DataFrame(
    episode_rows
)

# ------------------------------------------------------------
# 6. EPISODE SUMMARY
# ------------------------------------------------------------

print("=" * 70)
print("STEP 35 — OPPORTUNITY EPISODE ENGINE")
print("=" * 70)

print(
    "\nTotal HIGH+ episodes:",
    len(episodes)
)

print(
    "\nEpisode states:"
)

display(
    episodes[
        "state_at_start"
    ].value_counts()
    .to_frame("episodes")
)

# ------------------------------------------------------------
# 7. DURATION ANALYSIS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EPISODE DURATION")
print("=" * 70)

print(
    "\nMean:",
    round(
        episodes[
            "duration_hours"
        ].mean(),
        2
    ),
    "hours"
)

print(
    "Median:",
    round(
        episodes[
            "duration_hours"
        ].median(),
        2
    ),
    "hours"
)

print(
    "P75:",
    round(
        episodes[
            "duration_hours"
        ].quantile(.75),
        2
    ),
    "hours"
)

print(
    "P90:",
    round(
        episodes[
            "duration_hours"
        ].quantile(.90),
        2
    ),
    "hours"
)

print(
    "Maximum:",
    round(
        episodes[
            "duration_hours"
        ].max(),
        2
    ),
    "hours"
)

# ------------------------------------------------------------
# 8. SCORE BEHAVIOUR
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SCORE BEHAVIOUR")
print("=" * 70)

print(
    "\nMean starting score:",
    round(
        episodes[
            "start_score"
        ].mean(),
        2
    )
)

print(
    "Mean peak score:",
    round(
        episodes[
            "peak_score"
        ].mean(),
        2
    )
)

print(
    "Mean score change:",
    round(
        episodes[
            "score_change"
        ].mean(),
        2
    )
)

print(
    "Mean hours to peak:",
    round(
        episodes[
            "peak_delay_hours"
        ].mean(),
        2
    )
)

print(
    "Median hours to peak:",
    round(
        episodes[
            "peak_delay_hours"
        ].median(),
        2
    )
)

# ------------------------------------------------------------
# 9. RISING VS FALLING EPISODES
# ------------------------------------------------------------

episodes["score_direction"] = np.select(

    [
        episodes["score_change"] > 5,
        episodes["score_change"] < -5
    ],

    [
        "RISING",
        "FALLING"
    ],

    default="FLAT"
)

print("\n" + "=" * 70)
print("EPISODE SCORE DIRECTION")
print("=" * 70)

display(
    episodes[
        "score_direction"
    ].value_counts()
    .to_frame("episodes")
)

# ------------------------------------------------------------
# 10. VERY HIGH EPISODES
# ------------------------------------------------------------

very_high_episodes = episodes[
    episodes["peak_score"] >= 80
].copy()

print("\n" + "=" * 70)
print("VERY HIGH EPISODES")
print("=" * 70)

print(
    "\nEpisodes reaching 80+:",
    len(very_high_episodes)
)

if len(very_high_episodes) > 0:

    print(
        "Mean peak score:",
        round(
            very_high_episodes[
                "peak_score"
            ].mean(),
            2
        )
    )

    print(
        "Median peak score:",
        round(
            very_high_episodes[
                "peak_score"
            ].median(),
            2
        )
    )

    print(
        "Mean duration:",
        round(
            very_high_episodes[
                "duration_hours"
            ].mean(),
            2
        ),
        "hours"
    )

    print(
        "Median duration:",
        round(
            very_high_episodes[
                "duration_hours"
            ].median(),
            2
        ),
        "hours"
    )

# ------------------------------------------------------------
# 11. EPISODE SCORE QUANTILES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EPISODE PEAK SCORE DISTRIBUTION")
print("=" * 70)

display(
    episodes[
        "peak_score"
    ].describe(
        percentiles=[
            .50,
            .75,
            .90,
            .95
        ]
    ).round(2)
)

# ------------------------------------------------------------
# 12. LONGEST EPISODES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("10 LONGEST EPISODES")
print("=" * 70)

display(
    episodes[
        [
            "start_time",
            "end_time",
            "duration_hours",
            "start_score",
            "peak_score",
            "score_change",
            "state_at_peak"
        ]
    ]
    .sort_values(
        "duration_hours",
        ascending=False
    )
    .head(10)
    .round(2)
)

# ------------------------------------------------------------
# 13. STRONGEST EPISODES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("10 STRONGEST EPISODES")
print("=" * 70)

display(
    episodes[
        [
            "start_time",
            "end_time",
            "duration_hours",
            "start_score",
            "peak_score",
            "score_change",
            "peak_delay_hours",
            "state_at_peak"
        ]
    ]
    .sort_values(
        "peak_score",
        ascending=False
    )
    .head(10)
    .round(2)
)

# ------------------------------------------------------------
# 14. SAVE
# ------------------------------------------------------------

episode_path = (
    f"{BASE}/results/"
    "opportunity_episodes_v1.parquet"
)

episodes.to_parquet(
    episode_path,
    index=False
)

print("\nSaved:")
print(episode_path)

print("\nSTEP 35 COMPLETE")

In [ ]:
# ============================================================
# STEP 36 — EPISODE LEAD / LAG EVENT STUDY
# ============================================================

import pandas as pd
import numpy as np

BASE = "/content/gdrive/MyDrive/SilverAgent"

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

features_path = (
    f"{BASE}/database/"
    "silver_h1_features_v2_horizons.parquet"
)

scores_path = (
    f"{BASE}/results/"
    "continuous_opportunity_v2_history.parquet"
)

episodes_path = (
    f"{BASE}/results/"
    "opportunity_episodes_v1.parquet"
)

features = pd.read_parquet(features_path)
scores = pd.read_parquet(scores_path)
episodes = pd.read_parquet(episodes_path)

score_columns = [
    "volatility_score",
    "momentum_energy_score",
    "volume_score",
    "displacement_score"
]

df = features.join(
    scores[score_columns],
    how="inner"
)

df = df.sort_index()

df["opportunity_score"] = (
    df[score_columns].mean(axis=1)
)

# ------------------------------------------------------------
# 2. ENSURE DATETIME INDEX
# ------------------------------------------------------------

df.index = pd.to_datetime(
    df.index,
    utc=True
)

episodes["start_time"] = pd.to_datetime(
    episodes["start_time"],
    utc=True
)

episodes["end_time"] = pd.to_datetime(
    episodes["end_time"],
    utc=True
)

# ------------------------------------------------------------
# 3. FUTURE EVENT MEASUREMENT FUNCTION
# ------------------------------------------------------------
#
# For every episode start:
#
#   entry = close at first alert
#
# Then measure actual future candles inside:
#
#   1h
#   3h
#   6h
#   12h
#   24h
#
# We use REAL timestamps only.
#
# A future candle can be up to 90 minutes away from the
# requested horizon because the database contains market gaps.
# ------------------------------------------------------------

HORIZONS = {
    "1h": 1,
    "3h": 3,
    "6h": 6,
    "12h": 12,
    "24h": 24
}

TOLERANCE = pd.Timedelta(minutes=90)

def nearest_future_row(
    data,
    target_time,
    tolerance=TOLERANCE
):

    future = data[
        data.index >= target_time
    ]

    if len(future) == 0:
        return None

    timestamp = future.index[0]

    if timestamp - target_time > tolerance:
        return None

    return future.iloc[0]


# ------------------------------------------------------------
# 4. BUILD EVENT RESULTS
# ------------------------------------------------------------

event_rows = []

for _, episode in episodes.iterrows():

    start_time = episode["start_time"]

    # Find exact/nearest available starting candle.

    if start_time not in df.index:

        start_candidates = df[
            df.index >= start_time
        ]

        if len(start_candidates) == 0:
            continue

        start_idx = start_candidates.index[0]

        if (
            start_idx - start_time
            > TOLERANCE
        ):
            continue

    else:

        start_idx = start_time

    entry = df.loc[
        start_idx,
        "close"
    ]

    if pd.isna(entry):
        continue

    row = {

        "episode_id":
            episode["episode_id"],

        "start_time":
            start_time,

        "state_at_start":
            episode["state_at_start"],

        "start_score":
            episode["start_score"],

        "peak_score":
            episode["peak_score"],

        "duration_hours":
            episode["duration_hours"],

        "entry_price":
            entry
    }

    # --------------------------------------------------------
    # FUTURE HORIZONS
    # --------------------------------------------------------

    for horizon_name, hours in HORIZONS.items():

        target_time = (
            start_idx
            + pd.Timedelta(
                hours=hours
            )
        )

        future_row = nearest_future_row(
            df,
            target_time
        )

        if future_row is None:

            row[
                f"return_{horizon_name}"
            ] = np.nan

            row[
                f"max_up_{horizon_name}"
            ] = np.nan

            row[
                f"max_down_{horizon_name}"
            ] = np.nan

            row[
                f"max_abs_{horizon_name}"
            ] = np.nan

            row[
                f"time_to_max_{horizon_name}"
            ] = np.nan

            continue

        future_time = future_row.name

        # Actual future close.

        future_close = future_row["close"]

        signed_return = (
            future_close / entry
        ) - 1

        row[
            f"return_{horizon_name}"
        ] = signed_return

        # ----------------------------------------------------
        # LOOK FOR THE MAXIMUM MOVE INSIDE THE HORIZON
        # ----------------------------------------------------

        window_end = (
            start_idx
            + pd.Timedelta(
                hours=hours
            )
        )

        future_window = df[
            (
                df.index > start_idx
            )
            &
            (
                df.index <= window_end
            )
        ]

        if len(future_window) == 0:

            row[
                f"max_up_{horizon_name}"
            ] = np.nan

            row[
                f"max_down_{horizon_name}"
            ] = np.nan

            row[
                f"max_abs_{horizon_name}"
            ] = np.nan

            row[
                f"time_to_max_{horizon_name}"
            ] = np.nan

            continue

        # Maximum upside using future HIGH.

        max_up_idx = (
            future_window["high"]
            .idxmax()
        )

        max_up = (
            future_window.loc[
                max_up_idx,
                "high"
            ] / entry
        ) - 1

        # Maximum downside using future LOW.

        max_down_idx = (
            future_window["low"]
            .idxmin()
        )

        max_down = (
            future_window.loc[
                max_down_idx,
                "low"
            ] / entry
        ) - 1

        # Absolute largest directional move.

        if abs(max_up) >= abs(max_down):

            max_abs = abs(max_up)

            max_abs_idx = max_up_idx

        else:

            max_abs = abs(max_down)

            max_abs_idx = max_down_idx

        time_to_max = (
            max_abs_idx - start_idx
        ).total_seconds() / 3600

        row[
            f"max_up_{horizon_name}"
        ] = max_up

        row[
            f"max_down_{horizon_name}"
        ] = max_down

        row[
            f"max_abs_{horizon_name}"
        ] = max_abs

        row[
            f"time_to_max_{horizon_name}"
        ] = time_to_max

    event_rows.append(row)

events = pd.DataFrame(
    event_rows
)

# ------------------------------------------------------------
# 5. BASIC CHECK
# ------------------------------------------------------------

print("=" * 70)
print("STEP 36 — EPISODE LEAD / LAG EVENT STUDY")
print("=" * 70)

print(
    "\nEpisodes analysed:",
    len(events)
)

# ------------------------------------------------------------
# 6. SUMMARY BY EPISODE STATE
# ------------------------------------------------------------

summary_rows = []

for state, group in events.groupby(
    "state_at_start"
):

    row = {

        "state":
            state,

        "episodes":
            len(group),

        "mean_start_score":
            group["start_score"].mean(),

        "mean_peak_score":
            group["peak_score"].mean()
    }

    for horizon in [
        "1h",
        "3h",
        "6h",
        "12h",
        "24h"
    ]:

        ret = group[
            f"return_{horizon}"
        ]

        max_abs = group[
            f"max_abs_{horizon}"
        ]

        max_up = group[
            f"max_up_{horizon}"
        ]

        max_down = group[
            f"max_down_{horizon}"
        ]

        time_max = group[
            f"time_to_max_{horizon}"
        ]

        row[
            f"{horizon}_mean_return"
        ] = ret.mean()

        row[
            f"{horizon}_up_rate"
        ] = (
            ret > 0
        ).mean()

        row[
            f"{horizon}_mean_max_abs"
        ] = max_abs.mean()

        row[
            f"{horizon}_median_max_abs"
        ] = max_abs.median()

        row[
            f"{horizon}_mean_max_up"
        ] = max_up.mean()

        row[
            f"{horizon}_mean_max_down"
        ] = max_down.mean()

        row[
            f"{horizon}_mean_time_to_max"
        ] = time_max.mean()

    summary_rows.append(row)

summary = pd.DataFrame(
    summary_rows
)

print("\n" + "=" * 70)
print("EPISODE OUTCOME SUMMARY")
print("=" * 70)

display(
    summary.round(4)
)

# ------------------------------------------------------------
# 7. HOW FAST DOES THE MOVE HAPPEN?
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TIME TO MAXIMUM MOVE")
print("=" * 70)

time_rows = []

for state, group in events.groupby(
    "state_at_start"
):

    for horizon in [
        "1h",
        "3h",
        "6h",
        "12h",
        "24h"
    ]:

        values = group[
            f"time_to_max_{horizon}"
        ].dropna()

        if len(values) == 0:
            continue

        time_rows.append({

            "state":
                state,

            "horizon":
                horizon,

            "observations":
                len(values),

            "median_hours":
                values.median(),

            "mean_hours":
                values.mean(),

            "p75_hours":
                values.quantile(.75),

            "p90_hours":
                values.quantile(.90)
        })

time_summary = pd.DataFrame(
    time_rows
)

display(
    time_summary.round(2)
)

# ------------------------------------------------------------
# 8. RISING EPISODES VS FLAT/FALLING
# ------------------------------------------------------------

episodes_with_direction = events.copy()

episodes_with_direction[
    "score_direction"
] = np.select(

    [
        episodes_with_direction[
            "peak_score"
        ]
        >
        episodes_with_direction[
            "start_score"
        ]
        + 5,

        episodes_with_direction[
            "peak_score"
        ]
        <
        episodes_with_direction[
            "start_score"
        ]
        - 5
    ],

    [
        "RISING",
        "FALLING"
    ],

    default="FLAT"
)

print("\n" + "=" * 70)
print("RISING / FLAT / FALLING EPISODES")
print("=" * 70)

direction_rows = []

for direction, group in (
    episodes_with_direction
    .groupby("score_direction")
):

    row = {

        "score_direction":
            direction,

        "episodes":
            len(group)
    }

    for horizon in [
        "1h",
        "3h",
        "6h",
        "12h",
        "24h"
    ]:

        values = group[
            f"max_abs_{horizon}"
        ]

        returns = group[
            f"return_{horizon}"
        ]

        row[
            f"{horizon}_max_abs"
        ] = values.mean()

        row[
            f"{horizon}_return"
        ] = returns.mean()

        row[
            f"{horizon}_up_rate"
        ] = (
            returns > 0
        ).mean()

    direction_rows.append(row)

direction_summary = pd.DataFrame(
    direction_rows
)

display(
    direction_summary.round(4)
)

# ------------------------------------------------------------
# 9. VERY HIGH EPISODES
# ------------------------------------------------------------

very_high = events[
    events["peak_score"] >= 80
].copy()

print("\n" + "=" * 70)
print("VERY HIGH EPISODE OUTCOMES")
print("=" * 70)

print(
    "\nVERY_HIGH episodes:",
    len(very_high)
)

if len(very_high) > 0:

    for horizon in [
        "1h",
        "3h",
        "6h",
        "12h",
        "24h"
    ]:

        ret = very_high[
            f"return_{horizon}"
        ]

        move = very_high[
            f"max_abs_{horizon}"
        ]

        print(
            f"\n{horizon.upper()}"
        )

        print(
            "  Mean return:",
            f"{ret.mean():+.3%}"
        )

        print(
            "  Up rate:",
            f"{(ret > 0).mean():.2%}"
        )

        print(
            "  Mean maximum move:",
            f"{move.mean():.3%}"
        )

# ------------------------------------------------------------
# 10. LEAD / LAG DIAGNOSTIC
# ------------------------------------------------------------
#
# Compare the size of the move in the FIRST 3 HOURS with
# the full 24-hour movement.
#
# If most of the 24h move occurs after 3h, the detector may
# provide useful warning time.
# ------------------------------------------------------------

if len(events) > 0:

    e = events.copy()

    e["early_3h_share"] = (
        e["max_abs_3h"]
        /
        e["max_abs_24h"]
    )

    e["early_6h_share"] = (
        e["max_abs_6h"]
        /
        e["max_abs_24h"]
    )

    e = e.replace(
        [np.inf, -np.inf],
        np.nan
    )

    print("\n" + "=" * 70)
    print("HOW EARLY IS THE SIGNAL?")
    print("=" * 70)

    print(
        "\nMedian 3h share of eventual 24h move:",
        f"{e['early_3h_share'].median():.2%}"
    )

    print(
        "Mean 3h share:",
        f"{e['early_3h_share'].mean():.2%}"
    )

    print(
        "Median 6h share:",
        f"{e['early_6h_share'].median():.2%}"
    )

    print(
        "Mean 6h share:",
        f"{e['early_6h_share'].mean():.2%}"
    )

# ------------------------------------------------------------
# 11. SAVE
# ------------------------------------------------------------

event_path = (
    f"{BASE}/results/"
    "opportunity_episode_leadlag_v1.parquet"
)

summary_path = (
    f"{BASE}/results/"
    "opportunity_episode_leadlag_summary_v1.parquet"
)

time_path = (
    f"{BASE}/results/"
    "opportunity_episode_leadlag_time_v1.parquet"
)

direction_path = (
    f"{BASE}/results/"
    "opportunity_episode_leadlag_direction_v1.parquet"
)

events.to_parquet(
    event_path,
    index=False
)

summary.to_parquet(
    summary_path,
    index=False
)

time_summary.to_parquet(
    time_path,
    index=False
)

direction_summary.to_parquet(
    direction_path,
    index=False
)

print("\nSaved:")
print(event_path)
print(summary_path)
print(time_path)
print(direction_path)

print("\nSTEP 36 COMPLETE")

In [ ]:
# ============================================================
# STEP 37 — DATA-DRIVEN OPPORTUNITY PATTERN DISCOVERY
# ============================================================

import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

BASE = "/content/gdrive/MyDrive/SilverAgent"

EPISODE_PATH = (
    f"{BASE}/results/"
    "opportunity_episodes_v1.parquet"
)

LEADLAG_PATH = (
    f"{BASE}/results/"
    "opportunity_episode_leadlag_v1.parquet"
)

episodes = pd.read_parquet(
    EPISODE_PATH
)

leadlag = pd.read_parquet(
    LEADLAG_PATH
)

# ------------------------------------------------------------
# 1. PREPARE EPISODE DATA
# ------------------------------------------------------------

episodes["start_time"] = pd.to_datetime(
    episodes["start_time"],
    utc=True
)

episodes = episodes.sort_values(
    "start_time"
).reset_index(drop=True)

# ------------------------------------------------------------
# 2. JOIN FUTURE OUTCOMES
# ------------------------------------------------------------

outcome_columns = [
    "episode_id",
    "return_1h",
    "return_3h",
    "return_6h",
    "return_12h",
    "return_24h",
    "max_abs_1h",
    "max_abs_3h",
    "max_abs_6h",
    "max_abs_12h",
    "max_abs_24h"
]

outcomes = leadlag[
    outcome_columns
].copy()

episodes = episodes.merge(
    outcomes,
    on="episode_id",
    how="left"
)

# ------------------------------------------------------------
# 3. CREATE PATTERN FEATURES
# ------------------------------------------------------------
#
# IMPORTANT:
# These features describe the episode itself.
#
# Future price outcomes are NOT used here.
# ------------------------------------------------------------

episodes["peak_rise"] = (
    episodes["peak_score"]
    -
    episodes["start_score"]
)

episodes["peak_rise_per_hour"] = (
    episodes["peak_rise"]
    /
    episodes["peak_delay_hours"].clip(
        lower=1
    )
)

episodes["end_vs_peak"] = (
    episodes["end_score"]
    -
    episodes["peak_score"]
)

episodes["peak_excess"] = (
    episodes["peak_score"]
    -
    60
)

episodes["duration_log"] = np.log1p(
    episodes["duration_hours"]
)

# ------------------------------------------------------------
# 4. PATTERN FEATURES USED FOR CLUSTERING
# ------------------------------------------------------------

pattern_features = [

    "start_score",
    "peak_score",
    "peak_rise",
    "peak_rise_per_hour",
    "end_vs_peak",
    "duration_log",
    "peak_delay_hours"
]

cluster_data = episodes[
    pattern_features
].replace(
    [np.inf, -np.inf],
    np.nan
).dropna()

print("=" * 70)
print("STEP 37 — OPPORTUNITY PATTERN DISCOVERY")
print("=" * 70)

print(
    "\nEpisodes available:",
    len(cluster_data)
)

# ------------------------------------------------------------
# 5. STANDARDIZE FEATURES
# ------------------------------------------------------------

scaler = StandardScaler()

X = scaler.fit_transform(
    cluster_data[
        pattern_features
    ]
)

# ------------------------------------------------------------
# 6. TEST DIFFERENT NUMBERS OF CLUSTERS
# ------------------------------------------------------------

cluster_tests = []

for k in range(2, 8):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(
        X
    )

    score = silhouette_score(
        X,
        labels,
        sample_size=min(
            1000,
            len(X)
        ),
        random_state=42
    )

    cluster_tests.append({

        "clusters": k,

        "silhouette":
            score,

        "inertia":
            model.inertia_
    })

cluster_tests = pd.DataFrame(
    cluster_tests
)

print("\n" + "=" * 70)
print("CLUSTER QUALITY")
print("=" * 70)

display(
    cluster_tests.round(4)
)

# ------------------------------------------------------------
# 7. SELECT BEST K BY SILHOUETTE
# ------------------------------------------------------------

best_k = int(
    cluster_tests.loc[
        cluster_tests["silhouette"].idxmax(),
        "clusters"
    ]
)

best_silhouette = (
    cluster_tests.loc[
        cluster_tests["silhouette"].idxmax(),
        "silhouette"
    ]
)

print(
    "\nSelected clusters:",
    best_k
)

print(
    "Silhouette score:",
    round(
        best_silhouette,
        4
    )
)

# ------------------------------------------------------------
# 8. FIT FINAL CLUSTER MODEL
# ------------------------------------------------------------

final_model = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=20
)

cluster_labels = final_model.fit_predict(
    X
)

cluster_data = cluster_data.copy()

cluster_data[
    "pattern_cluster"
] = cluster_labels

# ------------------------------------------------------------
# 9. COPY LABELS BACK TO EPISODES
# ------------------------------------------------------------

episodes[
    "pattern_cluster"
] = np.nan

episodes.loc[
    cluster_data.index,
    "pattern_cluster"
] = cluster_data[
    "pattern_cluster"
]

episodes[
    "pattern_cluster"
] = episodes[
    "pattern_cluster"
].astype("Int64")

# ------------------------------------------------------------
# 10. CLUSTER CHARACTERISTICS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CLUSTER CHARACTERISTICS")
print("=" * 70)

cluster_summary = (
    episodes
    .dropna(
        subset=[
            "pattern_cluster"
        ]
    )
    .groupby(
        "pattern_cluster"
    )
    .agg(

        episodes=(
            "episode_id",
            "count"
        ),

        mean_start_score=(
            "start_score",
            "mean"
        ),

        mean_peak_score=(
            "peak_score",
            "mean"
        ),

        mean_peak_rise=(
            "peak_rise",
            "mean"
        ),

        mean_peak_rise_per_hour=(
            "peak_rise_per_hour",
            "mean"
        ),

        mean_end_vs_peak=(
            "end_vs_peak",
            "mean"
        ),

        mean_duration_hours=(
            "duration_hours",
            "mean"
        ),

        median_duration_hours=(
            "duration_hours",
            "median"
        ),

        mean_peak_delay_hours=(
            "peak_delay_hours",
            "mean"
        )
    )
    .reset_index()
)

display(
    cluster_summary.round(3)
)

# ------------------------------------------------------------
# 11. STANDARDIZED CLUSTER CENTRES
# ------------------------------------------------------------

centres = pd.DataFrame(
    scaler.inverse_transform(
        final_model.cluster_centers_
    ),
    columns=pattern_features
)

centres[
    "pattern_cluster"
] = range(best_k)

print("\n" + "=" * 70)
print("RAW CLUSTER CENTRES")
print("=" * 70)

display(
    centres[
        [
            "pattern_cluster",
            "start_score",
            "peak_score",
            "peak_rise",
            "peak_rise_per_hour",
            "end_vs_peak",
            "duration_log",
            "peak_delay_hours"
        ]
    ].round(3)
)

# ------------------------------------------------------------
# 12. DESCRIPTIVE FUTURE OUTCOMES BY CLUSTER
# ------------------------------------------------------------
#
# The clusters were created WITHOUT using future data.
#
# Now we ask:
# "Do these different episode shapes lead to different
# outcomes?"
# ------------------------------------------------------------

outcome_rows = []

clustered = episodes.dropna(
    subset=[
        "pattern_cluster",
        "max_abs_24h",
        "return_24h"
    ]
).copy()

for cluster, group in clustered.groupby(
    "pattern_cluster"
):

    abs_move = group[
        "max_abs_24h"
    ]

    signed = group[
        "return_24h"
    ]

    outcome_rows.append({

        "pattern_cluster":
            int(cluster),

        "episodes":
            len(group),

        "mean_24h_max_move":
            abs_move.mean(),

        "median_24h_max_move":
            abs_move.median(),

        "p75_24h_max_move":
            abs_move.quantile(.75),

        "p90_24h_max_move":
            abs_move.quantile(.90),

        "mean_24h_return":
            signed.mean(),

        "median_24h_return":
            signed.median(),

        "up_rate":
            (signed > 0).mean(),

        "large_move_rate":
            (
                abs_move
                >=
                abs_move.quantile(.75)
            ).mean()
    })

outcome_summary = pd.DataFrame(
    outcome_rows
)

print("\n" + "=" * 70)
print("FUTURE OUTCOMES BY DISCOVERED PATTERN")
print("=" * 70)

display(
    outcome_summary.round(4)
)

# ------------------------------------------------------------
# 13. COMBINE CHARACTERISTICS + OUTCOMES
# ------------------------------------------------------------

full_summary = cluster_summary.merge(
    outcome_summary,
    on="pattern_cluster",
    how="left"
)

print("\n" + "=" * 70)
print("FULL PATTERN SUMMARY")
print("=" * 70)

display(
    full_summary.round(4)
)

# ------------------------------------------------------------
# 14. AUTO-GENERATE HUMAN-READABLE LABELS
# ------------------------------------------------------------
#
# These labels are descriptive only.
# They are NOT trading signals.
# ------------------------------------------------------------

labels = {}

for _, row in full_summary.iterrows():

    cluster = int(
        row["pattern_cluster"]
    )

    rise = row[
        "mean_peak_rise"
    ]

    fade = row[
        "mean_end_vs_peak"
    ]

    duration = row[
        "mean_duration_hours"
    ]

    peak_delay = row[
        "mean_peak_delay_hours"
    ]

    if rise > 8 and peak_delay <= 4:

        label = "RAPID_BUILD"

    elif rise > 8 and peak_delay > 4:

        label = "SLOW_BUILD"

    elif fade < -5:

        label = "SPIKE_AND_FADE"

    elif duration >= 8 and abs(rise) <= 5:

        label = "HIGH_STABLE"

    elif duration < 4:

        label = "SHORT_SHOCK"

    else:

        label = "MIXED_PATTERN"

    labels[cluster] = label

episodes[
    "pattern_label"
] = episodes[
    "pattern_cluster"
].map(labels)

full_summary[
    "pattern_label"
] = full_summary[
    "pattern_cluster"
].map(labels)

print("\n" + "=" * 70)
print("DISCOVERED PATTERN LABELS")
print("=" * 70)

display(
    full_summary[
        [
            "pattern_cluster",
            "pattern_label",
            "episodes",
            "mean_start_score",
            "mean_peak_score",
            "mean_peak_rise",
            "mean_duration_hours",
            "mean_peak_delay_hours",
            "mean_24h_max_move",
            "mean_24h_return",
            "up_rate"
        ]
    ].round(4)
)

# ------------------------------------------------------------
# 15. CURRENT / LATEST EPISODES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LATEST DISCOVERED EPISODES")
print("=" * 70)

display(
    episodes[
        [
            "start_time",
            "end_time",
            "start_score",
            "peak_score",
            "duration_hours",
            "peak_delay_hours",
            "pattern_cluster",
            "pattern_label"
        ]
    ]
    .sort_values(
        "start_time",
        ascending=False
    )
    .head(15)
    .round(3)
)

# ------------------------------------------------------------
# 16. SAVE RESULTS
# ------------------------------------------------------------

episode_pattern_path = (
    f"{BASE}/results/"
    "opportunity_episode_patterns_v1.parquet"
)

pattern_summary_path = (
    f"{BASE}/results/"
    "opportunity_episode_pattern_summary_v1.parquet"
)

cluster_quality_path = (
    f"{BASE}/results/"
    "opportunity_episode_cluster_quality_v1.parquet"
)

episodes.to_parquet(
    episode_pattern_path,
    index=False
)

full_summary.to_parquet(
    pattern_summary_path,
    index=False
)

cluster_tests.to_parquet(
    cluster_quality_path,
    index=False
)

print("\nSaved:")
print(episode_pattern_path)
print(pattern_summary_path)
print(cluster_quality_path)

print("\nSTEP 37 COMPLETE")

In [ ]:
# ============================================================
# STEP 37 REPAIR — FINISH PATTERN LABELS + SAVE
# ============================================================

# Use the already-created cluster summaries.
# The duplicate "episodes" columns were renamed by pandas
# during the merge, so we explicitly create one clean column.

full_summary = full_summary.copy()

full_summary["episodes"] = (
    full_summary["episodes_x"]
)

# ------------------------------------------------------------
# CREATE DESCRIPTIVE LABELS
# ------------------------------------------------------------

labels = {}

for _, row in full_summary.iterrows():

    cluster = int(
        row["pattern_cluster"]
    )

    rise = row["mean_peak_rise"]
    fade = row["mean_end_vs_peak"]
    duration = row["mean_duration_hours"]
    peak_delay = row["mean_peak_delay_hours"]

    if rise > 8 and peak_delay <= 4:

        label = "RAPID_BUILD"

    elif rise > 8 and peak_delay > 4:

        label = "SLOW_BUILD"

    elif fade < -5:

        label = "SPIKE_AND_FADE"

    elif duration >= 8 and abs(rise) <= 5:

        label = "HIGH_STABLE"

    elif duration < 4:

        label = "SHORT_SHOCK"

    else:

        label = "MIXED_PATTERN"

    labels[cluster] = label

# Apply labels

episodes["pattern_label"] = (
    episodes["pattern_cluster"]
    .map(labels)
)

full_summary["pattern_label"] = (
    full_summary["pattern_cluster"]
    .map(labels)
)

# ------------------------------------------------------------
# DISCOVERED PATTERNS
# ------------------------------------------------------------

print("=" * 70)
print("STEP 37 — DISCOVERED OPPORTUNITY PATTERNS")
print("=" * 70)

display(
    full_summary[
        [
            "pattern_cluster",
            "pattern_label",
            "episodes",
            "mean_start_score",
            "mean_peak_score",
            "mean_peak_rise",
            "mean_duration_hours",
            "mean_peak_delay_hours",
            "mean_24h_max_move",
            "mean_24h_return",
            "up_rate"
        ]
    ]
    .sort_values(
        "mean_24h_max_move",
        ascending=False
    )
    .round(4)
)

# ------------------------------------------------------------
# LATEST EPISODES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LATEST DISCOVERED EPISODES")
print("=" * 70)

display(
    episodes[
        [
            "start_time",
            "end_time",
            "start_score",
            "peak_score",
            "duration_hours",
            "peak_delay_hours",
            "pattern_cluster",
            "pattern_label"
        ]
    ]
    .sort_values(
        "start_time",
        ascending=False
    )
    .head(15)
    .round(3)
)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

episode_pattern_path = (
    f"{BASE}/results/"
    "opportunity_episode_patterns_v1.parquet"
)

pattern_summary_path = (
    f"{BASE}/results/"
    "opportunity_episode_pattern_summary_v1.parquet"
)

cluster_quality_path = (
    f"{BASE}/results/"
    "opportunity_episode_cluster_quality_v1.parquet"
)

episodes.to_parquet(
    episode_pattern_path,
    index=False
)

full_summary.to_parquet(
    pattern_summary_path,
    index=False
)

cluster_tests.to_parquet(
    cluster_quality_path,
    index=False
)

print("\nSaved:")
print(episode_pattern_path)
print(pattern_summary_path)
print(cluster_quality_path)

print("\nSTEP 37 REPAIR COMPLETE")

In [ ]:

# ============================================================
# STEP 38 — OUT-OF-SAMPLE OPPORTUNITY PATTERN VALIDATION
# ROBUST VERSION
# ============================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

BASE = "/content/gdrive/MyDrive/SilverAgent"

EPISODES_FILE = f"{BASE}/results/opportunity_episode_patterns_v1.parquet"
LEADLAG_FILE = f"{BASE}/results/opportunity_episode_leadlag_v1.parquet"

# ------------------------------------------------------------
# 1. LOAD
# ------------------------------------------------------------

episodes = pd.read_parquet(EPISODES_FILE)
leadlag = pd.read_parquet(LEADLAG_FILE)

print("EPISODE COLUMNS:")
print(list(episodes.columns))

print("\nLEAD/LAG COLUMNS:")
print(list(leadlag.columns))

# ------------------------------------------------------------
# 2. NORMALISE DATETIME COLUMN
# ------------------------------------------------------------

def find_datetime_column(df):
    candidates = [
        "start_time",
        "episode_start",
        "timestamp",
        "time",
        "datetime"
    ]

    for c in candidates:
        if c in df.columns:
            return c

    raise KeyError(
        f"Could not find datetime column. Available columns: {list(df.columns)}"
    )

ep_time_col = find_datetime_column(episodes)
ll_time_col = find_datetime_column(leadlag)

episodes[ep_time_col] = pd.to_datetime(
    episodes[ep_time_col], utc=True
)

leadlag[ll_time_col] = pd.to_datetime(
    leadlag[ll_time_col], utc=True
)

# Standardise name
episodes = episodes.rename(columns={ep_time_col: "start_time"})
leadlag = leadlag.rename(columns={ll_time_col: "start_time"})

# ------------------------------------------------------------
# 3. FIND OUTCOME COLUMNS
# ------------------------------------------------------------

def find_column(df, candidates, description):
    for c in candidates:
        if c in df.columns:
            return c

    raise KeyError(
        f"Could not find {description}.\n"
        f"Tried: {candidates}\n"
        f"Available columns: {list(df.columns)}"
    )

return_col = find_column(
    leadlag,
    [
        "return_24h",
        "future_return_24h",
        "signed_return_24h",
        "mean_return_24h"
    ],
    "24h return"
)

max_abs_col = find_column(
    leadlag,
    [
        "max_abs_24h",
        "max_move_24h",
        "max_abs_move_24h",
        "max_abs_return_24h",
        "max_move"
    ],
    "24h maximum absolute move"
)

print("\nUSING OUTCOME COLUMNS:")
print("24h return:", return_col)
print("24h max move:", max_abs_col)

# ------------------------------------------------------------
# 4. KEEP ONLY THE OUTCOME COLUMNS WE NEED
# ------------------------------------------------------------

outcomes = leadlag[
    [
        "start_time",
        return_col,
        max_abs_col
    ]
].copy()

outcomes = outcomes.rename(
    columns={
        return_col: "return_24h",
        max_abs_col: "max_abs_24h"
    }
)

# ------------------------------------------------------------
# 5. JOIN
# ------------------------------------------------------------

df = episodes.merge(
    outcomes,
    on="start_time",
    how="left",
    suffixes=("", "_outcome")
)

# If duplicate columns somehow appeared, repair them
if "return_24h_outcome" in df.columns:
    df["return_24h"] = df["return_24h"].fillna(
        df["return_24h_outcome"]
    )
    df = df.drop(columns=["return_24h_outcome"])

if "max_abs_24h_outcome" in df.columns:
    df["max_abs_24h"] = df["max_abs_24h"].fillna(
        df["max_abs_24h_outcome"]
    )
    df = df.drop(columns=["max_abs_24h_outcome"])

print("\nAFTER JOIN:")
print("Rows:", len(df))
print("return_24h available:", df["return_24h"].notna().sum())
print("max_abs_24h available:", df["max_abs_24h"].notna().sum())

# ------------------------------------------------------------
# 6. CHECK REQUIRED EPISODE FEATURES
# ------------------------------------------------------------

required_episode_features = [
    "start_score",
    "peak_score",
    "duration_hours"
]

missing = [
    c for c in required_episode_features
    if c not in df.columns
]

if missing:
    raise KeyError(
        f"Missing episode feature columns: {missing}\n"
        f"Available columns: {list(df.columns)}"
    )

# ------------------------------------------------------------
# 7. BUILD CLUSTER FEATURES
# ------------------------------------------------------------

df["peak_rise"] = (
    df["peak_score"] -
    df["start_score"]
)

df["peak_rise_per_hour"] = (
    df["peak_rise"] /
    df["duration_hours"].clip(lower=1)
)

# Some versions may have end_score
if "end_score" in df.columns:
    df["end_vs_peak"] = (
        df["end_score"] -
        df["peak_score"]
    )
else:
    # Fall back to the available episode information
    df["end_vs_peak"] = 0.0

df["duration_log"] = np.log1p(
    df["duration_hours"].clip(lower=0)
)

cluster_features = [
    "start_score",
    "peak_rise",
    "peak_rise_per_hour",
    "end_vs_peak",
    "duration_log"
]

# ------------------------------------------------------------
# 8. CLEAN
# ------------------------------------------------------------

df = df.dropna(
    subset=cluster_features
).copy()

df = df.sort_values(
    "start_time"
).reset_index(drop=True)

print("\nCLEAN DATASET:")
print("Episodes:", len(df))
print(
    "Period:",
    df["start_time"].min(),
    "→",
    df["start_time"].max()
)

# ------------------------------------------------------------
# 9. WALK-FORWARD SETTINGS
# ------------------------------------------------------------

TRAIN_SIZE = 600
TEST_SIZE = 150
STEP = 150

results = []

window_id = 0
train_end = TRAIN_SIZE

# ------------------------------------------------------------
# 10. WALK-FORWARD OOS CLUSTERING
# ------------------------------------------------------------

while train_end + TEST_SIZE <= len(df):

    train = df.iloc[
        train_end - TRAIN_SIZE:
        train_end
    ].copy()

    test = df.iloc[
        train_end:
        train_end + TEST_SIZE
    ].copy()

    # --------------------------------------------------------
    # FIT SCALER ON TRAIN ONLY
    # --------------------------------------------------------

    scaler = StandardScaler()

    X_train = scaler.fit_transform(
        train[cluster_features]
    )

    X_test = scaler.transform(
        test[cluster_features]
    )

    # --------------------------------------------------------
    # FIT KMEANS ON TRAIN ONLY
    # --------------------------------------------------------

    model = KMeans(
        n_clusters=2,
        random_state=42,
        n_init=20
    )

    train_labels = model.fit_predict(X_train)
    test_labels = model.predict(X_test)

    train = train.copy()
    test = test.copy()

    train["cluster"] = train_labels
    test["cluster"] = test_labels

    # --------------------------------------------------------
    # IDENTIFY RAPID BUILD USING TRAIN DATA ONLY
    # --------------------------------------------------------

    train_cluster_rise = (
        train.groupby("cluster")["peak_rise"]
        .mean()
    )

    rapid_cluster = (
        train_cluster_rise.idxmax()
    )

    test["pattern"] = np.where(
        test["cluster"] == rapid_cluster,
        "RAPID_BUILD",
        "SHORT_SHOCK"
    )

    # --------------------------------------------------------
    # EVALUATE
    # --------------------------------------------------------

    for pattern in [
        "RAPID_BUILD",
        "SHORT_SHOCK",
        "ALL"
    ]:

        if pattern == "ALL":
            group = test
        else:
            group = test[
                test["pattern"] == pattern
            ]

        moves = group[
            "max_abs_24h"
        ].dropna()

        returns = group[
            "return_24h"
        ].dropna()

        if len(moves) == 0:
            continue

        results.append({
            "window": window_id,
            "test_start": test["start_time"].min(),
            "test_end": test["start_time"].max(),
            "pattern": pattern,
            "episodes": len(group),
            "outcome_episodes": len(moves),

            "mean_max_abs_24h": moves.mean(),
            "median_max_abs_24h": moves.median(),
            "p75_max_abs_24h": moves.quantile(0.75),
            "p90_max_abs_24h": moves.quantile(0.90),

            "mean_return_24h": returns.mean()
            if len(returns) else np.nan,

            "up_rate_24h": (
                (returns > 0).mean()
                if len(returns)
                else np.nan
            )
        })

    window_id += 1
    train_end += STEP

# ------------------------------------------------------------
# 11. RESULTS
# ------------------------------------------------------------

oos = pd.DataFrame(results)

print("\n============================================================")
print("STEP 38 — OOS OPPORTUNITY PATTERN VALIDATION")
print("============================================================")

print(
    "\nWindows:",
    oos["window"].nunique()
)

print(
    "Total evaluated rows:",
    len(oos)
)

print("\nRESULTS BY PATTERN:")

summary = (
    oos.groupby("pattern")
    .agg(
        windows=("window", "nunique"),
        episodes=("episodes", "sum"),
        outcome_episodes=("outcome_episodes", "sum"),
        mean_max_abs_24h=("mean_max_abs_24h", "mean"),
        median_max_abs_24h=("median_max_abs_24h", "mean"),
        mean_return_24h=("mean_return_24h", "mean"),
        up_rate=("up_rate_24h", "mean")
    )
    .reset_index()
)

print(summary.to_string(index=False))

# ------------------------------------------------------------
# 12. WEIGHTED RESULTS
# ------------------------------------------------------------

weighted_rows = []

for pattern, group in oos.groupby("pattern"):

    total_n = group["outcome_episodes"].sum()

    if total_n == 0:
        continue

    weighted_mean_move = (
        (
            group["mean_max_abs_24h"] *
            group["outcome_episodes"]
        ).sum()
        / total_n
    )

    weighted_mean_return = (
        (
            group["mean_return_24h"] *
            group["outcome_episodes"]
        ).sum()
        / total_n
    )

    weighted_rows.append({
        "pattern": pattern,
        "episodes": total_n,
        "weighted_mean_max_abs_24h":
            weighted_mean_move,
        "weighted_mean_return_24h":
            weighted_mean_return
    })

weighted = pd.DataFrame(weighted_rows)

print("\nWEIGHTED OOS RESULTS:")
print(weighted.to_string(index=False))

# ------------------------------------------------------------
# 13. RAPID BUILD PREMIUM
# ------------------------------------------------------------

if set(["RAPID_BUILD", "ALL"]).issubset(
    set(weighted["pattern"])
):

    rapid = weighted[
        weighted["pattern"] == "RAPID_BUILD"
    ].iloc[0]

    overall = weighted[
        weighted["pattern"] == "ALL"
    ].iloc[0]

    rapid_premium = (
        rapid["weighted_mean_max_abs_24h"]
        -
        overall["weighted_mean_max_abs_24h"]
    )

    print("\nRAPID BUILD PREMIUM VS ALL:")
    print(
        f"{rapid_premium * 100:.3f} percentage points"
    )

# ------------------------------------------------------------
# 14. WINDOW-BY-WINDOW CONSISTENCY
# ------------------------------------------------------------

pivot = oos.pivot(
    index="window",
    columns="pattern",
    values="mean_max_abs_24h"
)

if "RAPID_BUILD" in pivot.columns and "ALL" in pivot.columns:

    valid = pivot.dropna(
        subset=["RAPID_BUILD", "ALL"]
    )

    beats = (
        valid["RAPID_BUILD"] >
        valid["ALL"]
    ).sum()

    total = len(valid)

    print("\nRAPID BUILD CONSISTENCY:")
    print(
        f"{beats}/{total} windows beat ALL"
    )

    if total:
        print(
            f"Consistency: {beats / total * 100:.1f}%"
        )

# ------------------------------------------------------------
# 15. SAVE
# ------------------------------------------------------------

oos_path = (
    f"{BASE}/results/"
    "opportunity_pattern_oos_windows_v1.parquet"
)

summary_path = (
    f"{BASE}/results/"
    "opportunity_pattern_oos_summary_v1.parquet"
)

comparison_path = (
    f"{BASE}/results/"
    "opportunity_pattern_oos_comparison_v1.parquet"
)

oos.to_parquet(
    oos_path,
    index=False
)

summary.to_parquet(
    summary_path,
    index=False
)

weighted.to_parquet(
    comparison_path,
    index=False
)

print("\nSaved:")
print(oos_path)
print(summary_path)
print(comparison_path)

print("\nSTEP 38 COMPLETE")

In [ ]:
# ============================================================
# STEP 39 — PURGED RAPID BUILD VALIDATION
# ============================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

BASE = "/content/gdrive/MyDrive/SilverAgent"

EPISODES_FILE = (
    f"{BASE}/results/opportunity_episode_patterns_v1.parquet"
)

# ------------------------------------------------------------
# 1. LOAD
# ------------------------------------------------------------

df = pd.read_parquet(EPISODES_FILE).copy()

df["start_time"] = pd.to_datetime(
    df["start_time"],
    utc=True
)

df = df.sort_values(
    "start_time"
).reset_index(drop=True)

print("Episodes loaded:", len(df))

# ------------------------------------------------------------
# 2. REQUIRED COLUMNS
# ------------------------------------------------------------

required = [
    "start_time",
    "start_score",
    "peak_score",
    "end_score",
    "duration_hours",
    "return_24h",
    "max_abs_24h"
]

missing = [
    c for c in required
    if c not in df.columns
]

if missing:
    raise KeyError(
        f"Missing columns: {missing}"
    )

# ------------------------------------------------------------
# 3. REBUILD CLUSTER FEATURES
# ------------------------------------------------------------

df["peak_rise"] = (
    df["peak_score"] -
    df["start_score"]
)

df["peak_rise_per_hour"] = (
    df["peak_rise"] /
    df["duration_hours"].clip(lower=1)
)

df["end_vs_peak"] = (
    df["end_score"] -
    df["peak_score"]
)

df["duration_log"] = np.log1p(
    df["duration_hours"].clip(lower=0)
)

cluster_features = [
    "start_score",
    "peak_rise",
    "peak_rise_per_hour",
    "end_vs_peak",
    "duration_log"
]

# Only rows where clustering information exists
df = df.dropna(
    subset=cluster_features
).copy()

print("Usable episodes:", len(df))

# ------------------------------------------------------------
# 4. WALK-FORWARD SETTINGS
# ------------------------------------------------------------

TRAIN_SIZE = 600
TEST_SIZE = 150

# 24-hour purge.
# Episodes ending immediately before the test period can have
# outcomes overlapping the test period.
PURGE_HOURS = 24

results = []

window_id = 0
test_start_index = TRAIN_SIZE + 1

# ------------------------------------------------------------
# 5. WALK FORWARD
# ------------------------------------------------------------

while test_start_index + TEST_SIZE <= len(df):

    # --------------------------------------------------------
    # TEST PERIOD
    # --------------------------------------------------------

    test = df.iloc[
        test_start_index:
        test_start_index + TEST_SIZE
    ].copy()

    test_start_time = test["start_time"].min()

    # --------------------------------------------------------
    # PURGE TRAIN DATA
    # --------------------------------------------------------

    purge_boundary = (
        test_start_time -
        pd.Timedelta(hours=PURGE_HOURS)
    )

    train_candidates = df[
        df["start_time"] < purge_boundary
    ].copy()

    # Keep only most recent TRAIN_SIZE episodes
    train = train_candidates.tail(
        TRAIN_SIZE
    ).copy()

    if len(train) < TRAIN_SIZE:
        test_start_index += TEST_SIZE
        continue

    # --------------------------------------------------------
    # SCALE — TRAIN ONLY
    # --------------------------------------------------------

    scaler = StandardScaler()

    X_train = scaler.fit_transform(
        train[cluster_features]
    )

    X_test = scaler.transform(
        test[cluster_features]
    )

    # --------------------------------------------------------
    # KMEANS — TRAIN ONLY
    # --------------------------------------------------------

    model = KMeans(
        n_clusters=2,
        random_state=42,
        n_init=20
    )

    train_labels = model.fit_predict(
        X_train
    )

    test_labels = model.predict(
        X_test
    )

    train["cluster"] = train_labels
    test["cluster"] = test_labels

    # --------------------------------------------------------
    # IDENTIFY RAPID BUILD FROM TRAIN ONLY
    # --------------------------------------------------------

    train_cluster_rise = (
        train.groupby("cluster")["peak_rise"]
        .mean()
    )

    rapid_cluster = (
        train_cluster_rise.idxmax()
    )

    test["pattern"] = np.where(
        test["cluster"] == rapid_cluster,
        "RAPID_BUILD",
        "SHORT_SHOCK"
    )

    # --------------------------------------------------------
    # EVALUATE PATTERNS
    # --------------------------------------------------------

    for pattern in [
        "ALL",
        "RAPID_BUILD",
        "SHORT_SHOCK"
    ]:

        if pattern == "ALL":
            group = test.copy()
        else:
            group = test[
                test["pattern"] == pattern
            ].copy()

        moves = group[
            "max_abs_24h"
        ].dropna()

        returns = group[
            "return_24h"
        ].dropna()

        if len(moves) == 0:
            continue

        results.append({
            "window": window_id,

            "train_end":
                train["start_time"].max(),

            "test_start":
                test["start_time"].min(),

            "test_end":
                test["start_time"].max(),

            "pattern":
                pattern,

            "episodes":
                len(group),

            "outcome_episodes":
                len(moves),

            "mean_max_abs_24h":
                moves.mean(),

            "median_max_abs_24h":
                moves.median(),

            "p75_max_abs_24h":
                moves.quantile(0.75),

            "p90_max_abs_24h":
                moves.quantile(0.90),

            "mean_return_24h":
                returns.mean(),

            "up_rate":
                (returns > 0).mean()
        })

    window_id += 1
    test_start_index += TEST_SIZE

# ------------------------------------------------------------
# 6. RESULTS
# ------------------------------------------------------------

oos = pd.DataFrame(results)

print("\n============================================================")
print("STEP 39 — PURGED RAPID BUILD VALIDATION")
print("============================================================")

print(
    "\nWindows:",
    oos["window"].nunique()
)

print(
    "Rows:",
    len(oos)
)

# ------------------------------------------------------------
# 7. WEIGHTED SUMMARY
# ------------------------------------------------------------

weighted_rows = []

for pattern, group in oos.groupby("pattern"):

    n = group["outcome_episodes"].sum()

    weighted_move = (
        (
            group["mean_max_abs_24h"] *
            group["outcome_episodes"]
        ).sum()
        / n
    )

    weighted_return = (
        (
            group["mean_return_24h"] *
            group["outcome_episodes"]
        ).sum()
        / n
    )

    weighted_up = (
        (
            group["up_rate"] *
            group["outcome_episodes"]
        ).sum()
        / n
    )

    weighted_rows.append({
        "pattern": pattern,
        "episodes": n,
        "mean_max_abs_24h":
            weighted_move,
        "mean_return_24h":
            weighted_return,
        "up_rate":
            weighted_up
    })

weighted = pd.DataFrame(
    weighted_rows
)

print("\nWEIGHTED OOS RESULTS:")

display(
    weighted.style.format({
        "mean_max_abs_24h": "{:.3%}",
        "mean_return_24h": "{:.3%}",
        "up_rate": "{:.1%}"
    })
)

# ------------------------------------------------------------
# 8. RAPID BUILD PREMIUM
# ------------------------------------------------------------

rapid = weighted[
    weighted["pattern"] == "RAPID_BUILD"
].iloc[0]

overall = weighted[
    weighted["pattern"] == "ALL"
].iloc[0]

shock = weighted[
    weighted["pattern"] == "SHORT_SHOCK"
].iloc[0]

premium_vs_all = (
    rapid["mean_max_abs_24h"]
    -
    overall["mean_max_abs_24h"]
)

premium_vs_shock = (
    rapid["mean_max_abs_24h"]
    -
    shock["mean_max_abs_24h"]
)

print("\nRAPID BUILD PREMIUM:")
print(
    f"vs ALL: "
    f"{premium_vs_all:.3%}"
)

print(
    f"vs SHORT_SHOCK: "
    f"{premium_vs_shock:.3%}"
)

# ------------------------------------------------------------
# 9. WINDOW CONSISTENCY
# ------------------------------------------------------------

pivot = oos.pivot(
    index="window",
    columns="pattern",
    values="mean_max_abs_24h"
)

valid = pivot.dropna(
    subset=[
        "RAPID_BUILD",
        "ALL",
        "SHORT_SHOCK"
    ]
)

rapid_beats_all = (
    valid["RAPID_BUILD"] >
    valid["ALL"]
)

rapid_beats_shock = (
    valid["RAPID_BUILD"] >
    valid["SHORT_SHOCK"]
)

print("\nWINDOW CONSISTENCY:")

print(
    "Rapid Build > ALL:",
    f"{rapid_beats_all.sum()}/{len(valid)}",
    f"({rapid_beats_all.mean():.1%})"
)

print(
    "Rapid Build > Short Shock:",
    f"{rapid_beats_shock.sum()}/{len(valid)}",
    f"({rapid_beats_shock.mean():.1%})"
)

# ------------------------------------------------------------
# 10. SHOW EACH WINDOW
# ------------------------------------------------------------

print("\nWINDOW-BY-WINDOW:")

window_table = valid[
    [
        "ALL",
        "RAPID_BUILD",
        "SHORT_SHOCK"
    ]
].copy()

display(
    window_table.style.format(
        "{:.3%}"
    )
)

# ------------------------------------------------------------
# 11. SAVE
# ------------------------------------------------------------

oos_path = (
    f"{BASE}/results/"
    "opportunity_pattern_purged_oos_windows_v1.parquet"
)

summary_path = (
    f"{BASE}/results/"
    "opportunity_pattern_purged_oos_summary_v1.parquet"
)

oos.to_parquet(
    oos_path,
    index=False
)

weighted.to_parquet(
    summary_path,
    index=False
)

print("\nSaved:")
print(oos_path)
print(summary_path)

print("\nSTEP 39 COMPLETE")

In [ ]:
# ============================================================
# STEP 40 — RAPID BUILD DIRECTION DISCOVERY
# ============================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

BASE = "/content/gdrive/MyDrive/SilverAgent"

EPISODES_FILE = (
    f"{BASE}/results/opportunity_episode_patterns_v1.parquet"
)

# ------------------------------------------------------------
# 1. LOAD
# ------------------------------------------------------------

df = pd.read_parquet(
    EPISODES_FILE
).copy()

df["start_time"] = pd.to_datetime(
    df["start_time"],
    utc=True
)

df = df.sort_values(
    "start_time"
).reset_index(drop=True)

print("Episodes:", len(df))

# ------------------------------------------------------------
# 2. AVAILABLE COLUMNS
# ------------------------------------------------------------

print("\nAvailable columns:")
print(list(df.columns))

# ------------------------------------------------------------
# 3. BUILD PATTERN FEATURES
# ------------------------------------------------------------

df["peak_rise"] = (
    df["peak_score"] -
    df["start_score"]
)

df["peak_rise_per_hour"] = (
    df["peak_rise"] /
    df["duration_hours"].clip(lower=1)
)

df["end_vs_peak"] = (
    df["end_score"] -
    df["peak_score"]
)

df["duration_log"] = np.log1p(
    df["duration_hours"].clip(lower=0)
)

# ------------------------------------------------------------
# 4. PRICE/DIRECTION FEATURES
# ------------------------------------------------------------

# These are only used for describing the episode at entry.
# No future information is used here.

if "start_price" in df.columns:

    # Price direction is already represented by returns where
    # available. We use the episode's own early price movement
    # only if it exists.

    pass

# ------------------------------------------------------------
# 5. CREATE DIRECTION FEATURES FROM AVAILABLE DATA
# ------------------------------------------------------------

direction_features = []

# Episode shape
direction_features += [
    "start_score",
    "peak_rise",
    "peak_rise_per_hour",
    "end_vs_peak",
    "duration_log"
]

# Early observed returns
for col in [
    "return_1h",
    "return_3h",
    "return_6h"
]:
    if col in df.columns:
        direction_features.append(col)

# Remove unavailable columns
direction_features = [
    c for c in direction_features
    if c in df.columns
]

print("\nDirection features:")
print(direction_features)

# ------------------------------------------------------------
# 6. DEFINE DIRECTION OUTCOMES
# ------------------------------------------------------------

# IMPORTANT:
# These are FUTURE outcomes and are NOT used as model features.

direction_horizons = [
    1,
    3,
    6,
    12,
    24
]

for h in direction_horizons:

    col = f"return_{h}h"

    if col in df.columns:

        df[f"up_{h}h"] = np.where(
            df[col].notna(),
            df[col] > 0,
            np.nan
        )

# ------------------------------------------------------------
# 7. KEEP RAPID/SHORT LABELS
# ------------------------------------------------------------

if "pattern_label" in df.columns:

    df["pattern"] = df[
        "pattern_label"
    ]

elif "pattern_cluster" in df.columns:

    # Fall back to the existing cluster labels
    df["pattern"] = np.where(
        df["pattern_cluster"] == 0,
        "RAPID_BUILD",
        "SHORT_SHOCK"
    )

else:
    raise KeyError(
        "No pattern_label or pattern_cluster found."
    )

print("\nPattern distribution:")
print(
    df["pattern"].value_counts()
)

# ------------------------------------------------------------
# 8. DESCRIPTIVE DIRECTION STUDY
# ------------------------------------------------------------

print("\n============================================================")
print("DESCRIPTIVE DIRECTION STUDY")
print("============================================================")

for pattern in [
    "RAPID_BUILD",
    "SHORT_SHOCK"
]:

    group = df[
        df["pattern"] == pattern
    ]

    print(
        f"\n--- {pattern} ---"
    )

    for h in direction_horizons:

        col = f"return_{h}h"

        if col not in group.columns:
            continue

        vals = group[col].dropna()

        if len(vals) == 0:
            continue

        print(
            f"{h:>2}h | "
            f"n={len(vals):>4} | "
            f"mean={vals.mean():+.3%} | "
            f"median={vals.median():+.3%} | "
            f"up={((vals > 0).mean()):.1%}"
        )

# ------------------------------------------------------------
# 9. DIRECTION FEATURE RELATIONSHIPS
# ------------------------------------------------------------

print("\n============================================================")
print("RAPID BUILD FEATURE → 24H DIRECTION")
print("============================================================")

rapid = df[
    df["pattern"] == "RAPID_BUILD"
].copy()

for feature in direction_features:

    if feature not in rapid.columns:
        continue

    valid = rapid[
        [feature, "return_24h"]
    ].dropna()

    if len(valid) < 30:
        continue

    corr = valid[
        feature
    ].corr(
        valid["return_24h"]
    )

    print(
        f"{feature:25s} "
        f"corr={corr:+.4f} "
        f"n={len(valid)}"
    )

# ------------------------------------------------------------
# 10. QUARTILE ANALYSIS
# ------------------------------------------------------------

print("\n============================================================")
print("RAPID BUILD FEATURE QUARTILES")
print("============================================================")

for feature in direction_features:

    if feature not in rapid.columns:
        continue

    valid = rapid[
        [feature, "return_24h"]
    ].dropna()

    if len(valid) < 50:
        continue

    try:
        valid["quartile"] = pd.qcut(
            valid[feature],
            4,
            labels=[
                "Q1",
                "Q2",
                "Q3",
                "Q4"
            ],
            duplicates="drop"
        )
    except:
        continue

    grouped = (
        valid.groupby(
            "quartile",
            observed=False
        )["return_24h"]
        .agg(
            ["count", "mean", "median"]
        )
    )

    print(
        f"\n{feature}"
    )

    print(
        grouped.to_string(
            formatters={
                "mean": "{:.3%}".format,
                "median": "{:.3%}".format
            }
        )
    )

# ------------------------------------------------------------
# 11. SIMPLE DIRECTION RULE DISCOVERY
# ------------------------------------------------------------

print("\n============================================================")
print("SIMPLE DIRECTION RULES")
print("============================================================")

# We are deliberately testing very simple rules.
# No optimisation yet.

rules = {}

if "return_1h" in rapid.columns:
    rules["EARLY_1H_UP"] = (
        rapid["return_1h"] > 0
    )

if "return_3h" in rapid.columns:
    rules["EARLY_3H_UP"] = (
        rapid["return_3h"] > 0
    )

if "return_6h" in rapid.columns:
    rules["EARLY_6H_UP"] = (
        rapid["return_6h"] > 0
    )

if "peak_rise" in rapid.columns:
    rules["RAPID_SCORE_RISE"] = (
        rapid["peak_rise"] > 5
    )

if "peak_rise_per_hour" in rapid.columns:
    rules["FAST_SCORE_RISE"] = (
        rapid["peak_rise_per_hour"] > 2
    )

for name, mask in rules.items():

    subset = rapid[
        mask
    ].copy()

    vals = subset[
        "return_24h"
    ].dropna()

    if len(vals) < 20:
        continue

    print(
        f"{name:25s} "
        f"n={len(vals):4d} | "
        f"mean={vals.mean():+.3%} | "
        f"median={vals.median():+.3%} | "
        f"up={((vals > 0).mean()):.1%}"
    )

# ------------------------------------------------------------
# 12. COMBINED SIMPLE RULES
# ------------------------------------------------------------

print("\n============================================================")
print("COMBINED DIRECTION RULES")
print("============================================================")

combined_rules = {}

if (
    "return_1h" in rapid.columns and
    "peak_rise" in rapid.columns
):

    combined_rules[
        "1H_UP_AND_RAPID_RISE"
    ] = (
        (rapid["return_1h"] > 0) &
        (rapid["peak_rise"] > 5)
    )

if (
    "return_3h" in rapid.columns and
    "peak_rise" in rapid.columns
):

    combined_rules[
        "3H_UP_AND_RAPID_RISE"
    ] = (
        (rapid["return_3h"] > 0) &
        (rapid["peak_rise"] > 5)
    )

if (
    "return_6h" in rapid.columns and
    "peak_rise_per_hour" in rapid.columns
):

    combined_rules[
        "6H_UP_AND_FAST_RISE"
    ] = (
        (rapid["return_6h"] > 0) &
        (rapid["peak_rise_per_hour"] > 2)
    )

for name, mask in combined_rules.items():

    subset = rapid[
        mask
    ]

    vals = subset[
        "return_24h"
    ].dropna()

    if len(vals) < 20:
        continue

    print(
        f"{name:30s} "
        f"n={len(vals):4d} | "
        f"mean={vals.mean():+.3%} | "
        f"median={vals.median():+.3%} | "
        f"up={((vals > 0).mean()):.1%}"
    )

# ------------------------------------------------------------
# 13. SAVE DESCRIPTIVE RESULTS
# ------------------------------------------------------------

output = df[
    [
        "episode_id",
        "start_time",
        "pattern",
        "start_score",
        "peak_score",
        "peak_rise",
        "peak_rise_per_hour",
        "duration_hours",
        "return_1h",
        "return_3h",
        "return_6h",
        "return_12h",
        "return_24h",
        "max_abs_24h"
    ]
].copy()

output_path = (
    f"{BASE}/results/"
    "rapid_build_direction_study_v1.parquet"
)

output.to_parquet(
    output_path,
    index=False
)

print("\nSaved:")
print(output_path)

print("\nSTEP 40 COMPLETE")

In [ ]:

# ============================================================
# STEP 41 — OOS RAPID BUILD DIRECTIONAL CONFIRMATION
# ============================================================

import pandas as pd
import numpy as np

BASE = "/content/gdrive/MyDrive/SilverAgent"

FILE = (
    f"{BASE}/results/opportunity_episode_patterns_v1.parquet"
)

# ------------------------------------------------------------
# 1. LOAD
# ------------------------------------------------------------

df = pd.read_parquet(FILE).copy()

df["start_time"] = pd.to_datetime(
    df["start_time"],
    utc=True
)

df = df.sort_values(
    "start_time"
).reset_index(drop=True)

print("Episodes:", len(df))

# ------------------------------------------------------------
# 2. REBUILD RAPID BUILD FEATURES
# ------------------------------------------------------------

df["peak_rise"] = (
    df["peak_score"] -
    df["start_score"]
)

df["peak_rise_per_hour"] = (
    df["peak_rise"] /
    df["duration_hours"].clip(lower=1)
)

# Existing pattern label
if "pattern_label" in df.columns:

    df["pattern"] = df["pattern_label"]

else:

    raise KeyError(
        "pattern_label not found"
    )

# ------------------------------------------------------------
# 3. VALIDATION SETTINGS
# ------------------------------------------------------------

TRAIN_SIZE = 600
TEST_SIZE = 150

# 24h purge
PURGE_HOURS = 24

results = []

test_start_index = TRAIN_SIZE + 1
window_id = 0

# ------------------------------------------------------------
# 4. WALK FORWARD
# ------------------------------------------------------------

while test_start_index + TEST_SIZE <= len(df):

    test = df.iloc[
        test_start_index:
        test_start_index + TEST_SIZE
    ].copy()

    test_start_time = test[
        "start_time"
    ].min()

    purge_boundary = (
        test_start_time -
        pd.Timedelta(hours=PURGE_HOURS)
    )

    # Training data strictly before purge
    train = df[
        df["start_time"] < purge_boundary
    ].tail(
        TRAIN_SIZE
    ).copy()

    if len(train) < TRAIN_SIZE:
        test_start_index += TEST_SIZE
        continue

    # --------------------------------------------------------
    # RAPID BUILD ONLY
    # --------------------------------------------------------

    rapid_test = test[
        test["pattern"] == "RAPID_BUILD"
    ].copy()

    # --------------------------------------------------------
    # DEFINE FIXED CONFIRMATION RULES
    # --------------------------------------------------------

    rules = {

        # 1-hour confirmation
        "RB_1H_UP": (
            rapid_test["return_1h"] > 0
        ),

        "RB_1H_DOWN": (
            rapid_test["return_1h"] < 0
        ),

        # 3-hour confirmation
        "RB_3H_UP": (
            rapid_test["return_3h"] > 0
        ),

        "RB_3H_DOWN": (
            rapid_test["return_3h"] < 0
        ),

        # 6-hour confirmation
        "RB_6H_UP": (
            rapid_test["return_6h"] > 0
        ),

        "RB_6H_DOWN": (
            rapid_test["return_6h"] < 0
        ),

        # 6h direction + rapid score development
        "RB_6H_UP_FAST": (
            (rapid_test["return_6h"] > 0) &
            (rapid_test["peak_rise_per_hour"] > 2)
        ),

        "RB_6H_DOWN_FAST": (
            (rapid_test["return_6h"] < 0) &
            (rapid_test["peak_rise_per_hour"] > 2)
        )
    }

    # --------------------------------------------------------
    # EVALUATE EACH RULE
    # --------------------------------------------------------

    for rule_name, mask in rules.items():

        group = rapid_test[
            mask
        ].copy()

        returns = group[
            "return_24h"
        ].dropna()

        moves = group[
            "max_abs_24h"
        ].dropna()

        if len(returns) < 10:
            continue

        results.append({

            "window":
                window_id,

            "test_start":
                test["start_time"].min(),

            "test_end":
                test["start_time"].max(),

            "rule":
                rule_name,

            "signals":
                len(group),

            "outcomes":
                len(returns),

            "mean_return_24h":
                returns.mean(),

            "median_return_24h":
                returns.median(),

            "up_rate":
                (returns > 0).mean(),

            "mean_max_abs_24h":
                moves.mean()
                if len(moves)
                else np.nan,

            "p75_max_abs_24h":
                moves.quantile(.75)
                if len(moves)
                else np.nan,

            "p90_max_abs_24h":
                moves.quantile(.90)
                if len(moves)
                else np.nan
        })

    # --------------------------------------------------------
    # ADVANCE
    # --------------------------------------------------------

    window_id += 1
    test_start_index += TEST_SIZE

# ------------------------------------------------------------
# 5. RESULTS
# ------------------------------------------------------------

oos = pd.DataFrame(results)

print("\n============================================================")
print("STEP 41 — OOS DIRECTIONAL CONFIRMATION")
print("============================================================")

print(
    "\nWindows:",
    oos["window"].nunique()
)

print(
    "Rules:",
    oos["rule"].nunique()
)

# ------------------------------------------------------------
# 6. WEIGHTED SUMMARY
# ------------------------------------------------------------

summary_rows = []

for rule, group in oos.groupby("rule"):

    n = group["outcomes"].sum()

    if n == 0:
        continue

    weighted_return = (
        (
            group["mean_return_24h"] *
            group["outcomes"]
        ).sum()
        / n
    )

    weighted_move = (
        (
            group["mean_max_abs_24h"] *
            group["outcomes"]
        ).sum()
        / n
    )

    weighted_up = (
        (
            group["up_rate"] *
            group["outcomes"]
        ).sum()
        / n
    )

    summary_rows.append({

        "rule":
            rule,

        "windows":
            group["window"].nunique(),

        "signals":
            group["signals"].sum(),

        "outcomes":
            n,

        "weighted_mean_return_24h":
            weighted_return,

        "weighted_mean_max_abs_24h":
            weighted_move,

        "weighted_up_rate":
            weighted_up,

        "mean_window_return":
            group["mean_return_24h"].mean(),

        "positive_windows":
            (
                group["mean_return_24h"] > 0
            ).sum(),

        "consistency":
            (
                group["mean_return_24h"] > 0
            ).mean()
    })

summary = pd.DataFrame(
    summary_rows
).sort_values(
    "weighted_mean_return_24h",
    ascending=False
)

print("\nWEIGHTED OOS RESULTS:")

display(
    summary.style.format({
        "weighted_mean_return_24h":
            "{:.3%}",

        "weighted_mean_max_abs_24h":
            "{:.3%}",

        "weighted_up_rate":
            "{:.1%}",

        "mean_window_return":
            "{:.3%}",

        "consistency":
            "{:.1%}"
    })
)

# ------------------------------------------------------------
# 7. WINDOW-BY-WINDOW RESULTS
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "WINDOW-BY-WINDOW MEAN 24H RETURN"
)

print(
    "============================================================"
)

pivot = oos.pivot(
    index="window",
    columns="rule",
    values="mean_return_24h"
)

display(
    pivot.style.format(
        "{:+.3%}"
    )
)

# ------------------------------------------------------------
# 8. CHECK WHETHER UP RULES BEAT DOWN RULES
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "UP vs DOWN CONFIRMATION"
)

print(
    "============================================================"
)

for horizon in [
    "1H",
    "3H",
    "6H"
]:

    up_rule = f"RB_{horizon}_UP"
    down_rule = f"RB_{horizon}_DOWN"

    if (
        up_rule in summary["rule"].values and
        down_rule in summary["rule"].values
    ):

        up = summary[
            summary["rule"] == up_rule
        ].iloc[0]

        down = summary[
            summary["rule"] == down_rule
        ].iloc[0]

        print(
            f"\n{horizon}:"
        )

        print(
            f"UP   return: "
            f"{up['weighted_mean_return_24h']:+.3%}"
        )

        print(
            f"DOWN return: "
            f"{down['weighted_mean_return_24h']:+.3%}"
        )

# ------------------------------------------------------------
# 9. BEST RULE CONSISTENCY
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "RULE CONSISTENCY"
)

print(
    "============================================================"
)

for _, row in summary.iterrows():

    print(
        f"{row['rule']:25s} "
        f"{int(row['positive_windows'])}/"
        f"{int(row['windows'])} positive "
        f"({row['consistency']:.1%})"
    )

# ------------------------------------------------------------
# 10. SAVE
# ------------------------------------------------------------

oos_path = (
    f"{BASE}/results/"
    "rapid_build_direction_oos_v1.parquet"
)

summary_path = (
    f"{BASE}/results/"
    "rapid_build_direction_oos_summary_v1.parquet"
)

oos.to_parquet(
    oos_path,
    index=False
)

summary.to_parquet(
    summary_path,
    index=False
)

print("\nSaved:")
print(oos_path)
print(summary_path)

print("\nSTEP 41 COMPLETE")

In [ ]:

# ============================================================
# STEP 42 — EVENT-DRIVEN RAPID BUILD BACKTEST
# ============================================================

import pandas as pd
import numpy as np

BASE = "/content/gdrive/MyDrive/SilverAgent"

FILE = (
    f"{BASE}/results/opportunity_episode_patterns_v1.parquet"
)

# ------------------------------------------------------------
# 1. LOAD EPISODES
# ------------------------------------------------------------

df = pd.read_parquet(FILE).copy()

df["start_time"] = pd.to_datetime(
    df["start_time"],
    utc=True
)

df = df.sort_values(
    "start_time"
).reset_index(drop=True)

# ------------------------------------------------------------
# 2. REQUIRED DATA
# ------------------------------------------------------------

required = [
    "episode_id",
    "start_time",
    "pattern_label",
    "return_1h",
    "return_3h",
    "return_6h",
    "return_12h",
    "return_24h",
    "max_abs_24h"
]

missing = [
    c for c in required
    if c not in df.columns
]

if missing:
    raise KeyError(
        f"Missing columns: {missing}"
    )

df["pattern"] = df["pattern_label"]

# ------------------------------------------------------------
# 3. STRATEGY DEFINITIONS
# ------------------------------------------------------------

# These are deliberately fixed.
#
# Entry:
#   Rapid Build confirmed UP at 1h / 3h / 6h
#
# Holding period:
#   Maximum 24h AFTER confirmation
#
# Since this episode dataset stores returns from episode START,
# we approximate post-confirmation performance by subtracting
# the return already experienced before confirmation from the
# total return at 24h.
#
# This gives:
#
#   1h confirmation:
#       post-entry return ≈ return_24h - return_1h
#
#   3h confirmation:
#       post-entry return ≈ return_24h - return_3h
#
#   6h confirmation:
#       post-entry return ≈ return_24h - return_6h
#
# This is an event-study approximation, NOT candle-level
# execution. We will build a true candle-level backtester later.

strategies = {

    "RB_1H_UP": {
        "confirmation_col": "return_1h"
    },

    "RB_3H_UP": {
        "confirmation_col": "return_3h"
    },

    "RB_6H_UP": {
        "confirmation_col": "return_6h"
    }
}

# ------------------------------------------------------------
# 4. SIMULATE EACH STRATEGY
# ------------------------------------------------------------

all_results = []

for strategy_name, config in strategies.items():

    confirmation_col = config[
        "confirmation_col"
    ]

    rapid = df[
        df["pattern"] == "RAPID_BUILD"
    ].copy()

    # Confirmed UP only
    signals = rapid[
        rapid[confirmation_col] > 0
    ].copy()

    for _, row in signals.iterrows():

        total_return = row[
            "return_24h"
        ]

        confirmation_return = row[
            confirmation_col
        ]

        if pd.isna(
            total_return
        ) or pd.isna(
            confirmation_return
        ):
            continue

        # Approximate return after confirmation
        post_confirmation_return = (
            total_return -
            confirmation_return
        )

        all_results.append({

            "episode_id":
                row["episode_id"],

            "start_time":
                row["start_time"],

            "strategy":
                strategy_name,

            "confirmation":
                confirmation_col,

            "confirmation_return":
                confirmation_return,

            "post_confirmation_return":
                post_confirmation_return,

            "total_return":
                total_return,

            "max_abs_24h":
                row["max_abs_24h"]
        })

trades = pd.DataFrame(
    all_results
)

# ------------------------------------------------------------
# 5. SUMMARY
# ------------------------------------------------------------

print("\n============================================================")
print("STEP 42 — EVENT-DRIVEN RAPID BUILD BACKTEST")
print("============================================================")

summary_rows = []

for strategy, group in trades.groupby(
    "strategy"
):

    returns = group[
        "post_confirmation_return"
    ].dropna()

    if len(returns) == 0:
        continue

    wins = returns[
        returns > 0
    ]

    losses = returns[
        returns < 0
    ]

    gross_profit = wins.sum()

    gross_loss = abs(
        losses.sum()
    )

    profit_factor = (
        gross_profit / gross_loss
        if gross_loss > 0
        else np.inf
    )

    # Compounded equity
    equity = (
        1000 *
        (1 + returns)
        .cumprod()
    )

    running_max = equity.cummax()

    drawdown = (
        equity /
        running_max
        - 1
    )

    summary_rows.append({

        "strategy":
            strategy,

        "trades":
            len(returns),

        "mean_return":
            returns.mean(),

        "median_return":
            returns.median(),

        "win_rate":
            (returns > 0).mean(),

        "profit_factor":
            profit_factor,

        "final_equity":
            equity.iloc[-1],

        "total_return":
            equity.iloc[-1] / 1000 - 1,

        "max_drawdown":
            drawdown.min(),

        "best_trade":
            returns.max(),

        "worst_trade":
            returns.min()
    })

summary = pd.DataFrame(
    summary_rows
)

print("\nSUMMARY:")

display(
    summary.style.format({

        "mean_return":
            "{:.3%}",

        "median_return":
            "{:.3%}",

        "win_rate":
            "{:.1%}",

        "profit_factor":
            "{:.2f}",

        "final_equity":
            "£{:.2f}",

        "total_return":
            "{:.2%}",

        "max_drawdown":
            "{:.2%}",

        "best_trade":
            "{:.2%}",

        "worst_trade":
            "{:.2%}"
    })
)

# ------------------------------------------------------------
# 6. WINDOWED OOS PERFORMANCE
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "CHRONOLOGICAL WINDOW PERFORMANCE"
)

print(
    "============================================================"
)

# Five chronological windows matching Step 41
df_min = trades["start_time"].min()
df_max = trades["start_time"].max()

trades["time_window"] = pd.qcut(
    trades["start_time"],
    q=5,
    labels=False,
    duplicates="drop"
)

window_results = []

for (strategy, window), group in trades.groupby(
    ["strategy", "time_window"]
):

    r = group[
        "post_confirmation_return"
    ].dropna()

    if len(r) == 0:
        continue

    window_results.append({

        "strategy":
            strategy,

        "window":
            int(window),

        "trades":
            len(r),

        "mean_return":
            r.mean(),

        "median_return":
            r.median(),

        "win_rate":
            (r > 0).mean(),

        "best":
            r.max(),

        "worst":
            r.min()
    })

window_df = pd.DataFrame(
    window_results
)

display(
    window_df.style.format({

        "mean_return":
            "{:+.3%}",

        "median_return":
            "{:+.3%}",

        "win_rate":
            "{:.1%}",

        "best":
            "{:+.2%}",

        "worst":
            "{:+.2%}"
    })
)

# ------------------------------------------------------------
# 7. SIGNAL OVERLAP
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "SIGNAL COUNTS"
)

print(
    "============================================================"
)

print(
    trades.groupby(
        "strategy"
    ).size()
)

# ------------------------------------------------------------
# 8. SAVE
# ------------------------------------------------------------

trades_path = (
    f"{BASE}/results/"
    "rapid_build_event_backtest_v1.parquet"
)

summary_path = (
    f"{BASE}/results/"
    "rapid_build_event_backtest_summary_v1.parquet"
)

window_path = (
    f"{BASE}/results/"
    "rapid_build_event_backtest_windows_v1.parquet"
)

trades.to_parquet(
    trades_path,
    index=False
)

summary.to_parquet(
    summary_path,
    index=False
)

window_df.to_parquet(
    window_path,
    index=False
)

print("\nSaved:")
print(trades_path)
print(summary_path)
print(window_path)

print("\nSTEP 42 COMPLETE")

In [ ]:
# ============================================================
# STEP 43A — INSPECT SILVER CANDLE DATABASE
# ============================================================

import pandas as pd

BASE = "/content/gdrive/MyDrive/SilverAgent"

CANDLES_FILE = (
    f"{BASE}/database/silver_h1.parquet"
)

candles = pd.read_parquet(
    CANDLES_FILE
)

print("Columns:")
print(list(candles.columns))

print("\nIndex:")
print(candles.index)

print("\nIndex name:")
print(candles.index.name)

print("\nFirst 3 rows:")
display(candles.head(3))

print("\nLast 3 rows:")
display(candles.tail(3))

In [ ]:
# ============================================================
# STEP 43 — CANDLE-LEVEL RAPID BUILD BACKTEST
# CORRECTED FOR DATETIMEINDEX DATABASE
# ============================================================

import pandas as pd
import numpy as np

BASE = "/content/gdrive/MyDrive/SilverAgent"

EPISODES_FILE = (
    f"{BASE}/results/opportunity_episode_patterns_v1.parquet"
)

CANDLES_FILE = (
    f"{BASE}/database/silver_h1.parquet"
)

# ------------------------------------------------------------
# 1. LOAD EPISODES
# ------------------------------------------------------------

episodes = pd.read_parquet(
    EPISODES_FILE
).copy()

episodes["start_time"] = pd.to_datetime(
    episodes["start_time"],
    utc=True
)

episodes = episodes.sort_values(
    "start_time"
).reset_index(drop=True)

# ------------------------------------------------------------
# 2. LOAD CANDLES
# ------------------------------------------------------------

candles = pd.read_parquet(
    CANDLES_FILE
).copy()

# The database stores time in the index.
if not isinstance(
    candles.index,
    pd.DatetimeIndex
):
    raise TypeError(
        "Expected silver_h1.parquet to use "
        "a DatetimeIndex."
    )

candles.index = pd.to_datetime(
    candles.index,
    utc=True
)

candles.index.name = "time"

candles = candles.sort_index()

# ------------------------------------------------------------
# 3. BASIC CHECK
# ------------------------------------------------------------

required = [
    "open",
    "high",
    "low",
    "close"
]

missing = [
    c for c in required
    if c not in candles.columns
]

if missing:
    raise KeyError(
        f"Missing candle columns: {missing}"
    )

print("Episodes:", len(episodes))
print("Candles:", len(candles))

print(
    "Candle period:",
    candles.index.min(),
    "→",
    candles.index.max()
)

print(
    "Candle gaps:",
    candles.index.to_series()
    .diff()
    .gt(pd.Timedelta(hours=1))
    .sum()
)

# ------------------------------------------------------------
# 4. ATR14
# ------------------------------------------------------------

prev_close = candles[
    "close"
].shift(1)

tr1 = (
    candles["high"] -
    candles["low"]
)

tr2 = (
    candles["high"] -
    prev_close
).abs()

tr3 = (
    candles["low"] -
    prev_close
).abs()

candles["true_range"] = pd.concat(
    [tr1, tr2, tr3],
    axis=1
).max(axis=1)

candles["ATR14"] = (
    candles["true_range"]
    .rolling(
        14,
        min_periods=14
    )
    .mean()
)

# ------------------------------------------------------------
# 5. NEXT REAL CANDLE
# ------------------------------------------------------------

def get_next_candle(timestamp):

    timestamp = pd.Timestamp(
        timestamp
    )

    if timestamp.tzinfo is None:
        timestamp = timestamp.tz_localize(
            "UTC"
        )
    else:
        timestamp = timestamp.tz_convert(
            "UTC"
        )

    pos = candles.index.searchsorted(
        timestamp,
        side="left"
    )

    if pos >= len(candles):
        return None

    return candles.iloc[pos]

# ------------------------------------------------------------
# 6. SIMULATOR
# ------------------------------------------------------------

def simulate_trade(
    episode_start,
    confirmation_hours,
    stop_atr,
    target_atr,
    cost_pct=0.0002,
    max_hold_hours=24
):

    # --------------------------------------------
    # Confirmation time
    # --------------------------------------------

    confirmation_time = (
        episode_start +
        pd.Timedelta(
            hours=confirmation_hours
        )
    )

    # Enter on NEXT REAL H1 CANDLE
    entry_row = get_next_candle(
        confirmation_time
    )

    if entry_row is None:
        return None

    entry_time = entry_row.name

    entry_price = float(
        entry_row["open"]
    )

    atr = float(
        entry_row["ATR14"]
    )

    if not np.isfinite(atr):
        return None

    if atr <= 0:
        return None

    # --------------------------------------------
    # Stops/targets
    # --------------------------------------------

    stop_price = (
        entry_price -
        stop_atr * atr
    )

    target_price = (
        entry_price +
        target_atr * atr
    )

    # --------------------------------------------
    # Future REAL candles
    # --------------------------------------------

    end_time = (
        entry_time +
        pd.Timedelta(
            hours=max_hold_hours
        )
    )

    future = candles.loc[
        (
            candles.index >
            entry_time
        )
        &
        (
            candles.index <=
            end_time
        )
    ].copy()

    if len(future) == 0:
        return None

    # --------------------------------------------
    # Walk forward
    # --------------------------------------------

    exit_time = None
    exit_price = None
    exit_reason = None

    for t, row in future.iterrows():

        hit_stop = (
            row["low"] <=
            stop_price
        )

        hit_target = (
            row["high"] >=
            target_price
        )

        # Conservative assumption:
        # if both occur inside the same H1 candle,
        # assume STOP happened first.
        if hit_stop:

            exit_time = t
            exit_price = stop_price
            exit_reason = "STOP"

            break

        if hit_target:

            exit_time = t
            exit_price = target_price
            exit_reason = "TARGET"

            break

    # --------------------------------------------
    # Time exit
    # --------------------------------------------

    if exit_time is None:

        exit_time = future.index[-1]

        exit_price = float(
            future.iloc[-1]["close"]
        )

        exit_reason = "TIME"

    # --------------------------------------------
    # Returns
    # --------------------------------------------

    gross_return = (
        exit_price /
        entry_price
        - 1
    )

    net_return = (
        gross_return -
        cost_pct
    )

    hold_hours = (
        exit_time -
        entry_time
    ).total_seconds() / 3600

    return {

        "entry_signal_time":
            episode_start,

        "entry_time":
            entry_time,

        "entry_price":
            entry_price,

        "ATR":
            atr,

        "stop_atr":
            stop_atr,

        "target_atr":
            target_atr,

        "stop_price":
            stop_price,

        "target_price":
            target_price,

        "exit_time":
            exit_time,

        "exit_price":
            exit_price,

        "exit_reason":
            exit_reason,

        "gross_return":
            gross_return,

        "net_return":
            net_return,

        "hold_hours":
            hold_hours
    }

# ------------------------------------------------------------
# 7. RAPID BUILD SIGNALS
# ------------------------------------------------------------

rapid = episodes[
    episodes["pattern_label"] ==
    "RAPID_BUILD"
].copy()

print(
    "\nRapid Build episodes:",
    len(rapid)
)

# ------------------------------------------------------------
# 8. STRATEGIES
# ------------------------------------------------------------

confirmations = {

    "RB_1H_UP": {
        "hours": 1,
        "return_col": "return_1h"
    },

    "RB_3H_UP": {
        "hours": 3,
        "return_col": "return_3h"
    },

    "RB_6H_UP": {
        "hours": 6,
        "return_col": "return_6h"
    }
}

stop_targets = [
    (1.0, 1.5),
    (1.0, 2.0),
    (1.5, 2.0)
]

all_trades = []

# ------------------------------------------------------------
# 9. RUN
# ------------------------------------------------------------

for strategy_name, config in confirmations.items():

    hours = config["hours"]

    return_col = config[
        "return_col"
    ]

    signals = rapid[
        rapid[return_col] > 0
    ].copy()

    print(
        f"\n{strategy_name}: "
        f"{len(signals)} candidate signals"
    )

    for stop_atr, target_atr in stop_targets:

        for _, episode in signals.iterrows():

            result = simulate_trade(
                episode_start=
                    episode["start_time"],

                confirmation_hours=
                    hours,

                stop_atr=
                    stop_atr,

                target_atr=
                    target_atr,

                cost_pct=
                    0.0002,

                max_hold_hours=
                    24
            )

            if result is None:
                continue

            result[
                "episode_id"
            ] = episode[
                "episode_id"
            ]

            result[
                "strategy"
            ] = strategy_name

            result[
                "stop_target"
            ] = (
                f"{stop_atr:.1f}ATR/"
                f"{target_atr:.1f}ATR"
            )

            all_trades.append(
                result
            )

trades = pd.DataFrame(
    all_trades
)

print(
    "\nRaw simulated trades:",
    len(trades)
)

# ------------------------------------------------------------
# 10. ONE POSITION AT A TIME
# ------------------------------------------------------------

non_overlap = []

for (
    strategy,
    stop_target
), group in trades.groupby(
    [
        "strategy",
        "stop_target"
    ]
):

    group = group.sort_values(
        "entry_time"
    )

    last_exit = None

    for _, trade in group.iterrows():

        if (
            last_exit is not None
            and
            trade["entry_time"] <=
            last_exit
        ):
            continue

        non_overlap.append(
            trade
        )

        last_exit = trade[
            "exit_time"
        ]

trades = pd.DataFrame(
    non_overlap
).reset_index(drop=True)

print(
    "After one-position control:",
    len(trades)
)

# ------------------------------------------------------------
# 11. PERFORMANCE
# ------------------------------------------------------------

summary_rows = []

for (
    strategy,
    stop_target
), group in trades.groupby(
    [
        "strategy",
        "stop_target"
    ]
):

    returns = group[
        "net_return"
    ].dropna()

    if len(returns) == 0:
        continue

    wins = returns[
        returns > 0
    ]

    losses = returns[
        returns < 0
    ]

    gross_profit = wins.sum()

    gross_loss = abs(
        losses.sum()
    )

    profit_factor = (
        gross_profit /
        gross_loss
        if gross_loss > 0
        else np.inf
    )

    equity = (
        1000 *
        (1 + returns)
        .cumprod()
    )

    running_max = (
        equity.cummax()
    )

    drawdown = (
        equity /
        running_max
        - 1
    )

    summary_rows.append({

        "strategy":
            strategy,

        "stop_target":
            stop_target,

        "trades":
            len(returns),

        "mean_trade":
            returns.mean(),

        "median_trade":
            returns.median(),

        "win_rate":
            (returns > 0).mean(),

        "profit_factor":
            profit_factor,

        "final_equity":
            equity.iloc[-1],

        "total_return":
            equity.iloc[-1] /
            1000 - 1,

        "max_drawdown":
            drawdown.min(),

        "best_trade":
            returns.max(),

        "worst_trade":
            returns.min(),

        "avg_hold_hours":
            group[
                "hold_hours"
            ].mean()
    })

summary = pd.DataFrame(
    summary_rows
)

print(
    "\n============================================================"
)

print(
    "STEP 43 — CANDLE-LEVEL RESULTS"
)

print(
    "============================================================"
)

display(
    summary.style.format({

        "mean_trade":
            "{:+.3%}",

        "median_trade":
            "{:+.3%}",

        "win_rate":
            "{:.1%}",

        "profit_factor":
            "{:.2f}",

        "final_equity":
            "£{:.2f}",

        "total_return":
            "{:+.2%}",

        "max_drawdown":
            "{:.2%}",

        "best_trade":
            "{:+.2%}",

        "worst_trade":
            "{:+.2%}",

        "avg_hold_hours":
            "{:.1f}"
    })
)

# ------------------------------------------------------------
# 12. EXIT REASONS
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "EXIT REASONS"
)

print(
    "============================================================"
)

exit_summary = (
    trades
    .groupby(
        [
            "strategy",
            "stop_target",
            "exit_reason"
        ]
    )
    .size()
    .reset_index(
        name="trades"
    )
)

display(
    exit_summary
)

# ------------------------------------------------------------
# 13. SAVE
# ------------------------------------------------------------

trades_path = (
    f"{BASE}/results/"
    "rapid_build_candle_backtest_v1.parquet"
)

summary_path = (
    f"{BASE}/results/"
    "rapid_build_candle_backtest_summary_v1.parquet"
)

exit_path = (
    f"{BASE}/results/"
    "rapid_build_candle_backtest_exits_v1.parquet"
)

trades.to_parquet(
    trades_path,
    index=False
)

summary.to_parquet(
    summary_path,
    index=False
)

exit_summary.to_parquet(
    exit_path,
    index=False
)

print("\nSaved:")
print(trades_path)
print(summary_path)
print(exit_path)

print("\nSTEP 43 COMPLETE")

In [ ]:
# ============================================================
# STEP 44 — CAUSAL / LIVE-SAFE RAPID BUILD DETECTOR V1
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

BASE = Path("/content/gdrive/MyDrive/SilverAgent")
DB_PATH = BASE / "database" / "silver_h1_features_v2.parquet"
OPP_PATH = BASE / "results" / "continuous_opportunity_v2_history.parquet"

OUT_PATH = BASE / "results" / "causal_rapid_build_v1.parquet"

# ------------------------------------------------------------
# 2. Load data
# ------------------------------------------------------------

features = pd.read_parquet(DB_PATH)
opp = pd.read_parquet(OPP_PATH)

print("Features:", features.shape)
print("Opportunity:", opp.shape)

# Make sure both use DatetimeIndex
if not isinstance(features.index, pd.DatetimeIndex):
    if "time" in features.columns:
        features["time"] = pd.to_datetime(features["time"], utc=True)
        features = features.set_index("time")

if not isinstance(opp.index, pd.DatetimeIndex):
    if "time" in opp.columns:
        opp["time"] = pd.to_datetime(opp["time"], utc=True)
        opp = opp.set_index("time")

features = features.sort_index()
opp = opp.sort_index()

# ------------------------------------------------------------
# 3. Find opportunity score column
# ------------------------------------------------------------

possible_score_cols = [
    "continuous_opportunity",
    "opportunity_score",
    "score"
]

score_col = None

for c in possible_score_cols:
    if c in opp.columns:
        score_col = c
        break

if score_col is None:
    raise ValueError(
        f"Could not find opportunity score. Available columns:\n{list(opp.columns)}"
    )

print("Using opportunity score:", score_col)

# ------------------------------------------------------------
# 4. Merge only information that exists at that timestamp
# ------------------------------------------------------------

df = features.join(
    opp[[score_col]].rename(columns={score_col: "opportunity_score"}),
    how="inner"
)

df = df.sort_index()

# ------------------------------------------------------------
# 5. Causal features
#
# IMPORTANT:
# Every feature below uses only current or previous candles.
# Nothing looks into the future.
# ------------------------------------------------------------

df["score_prev_1h"] = df["opportunity_score"].shift(1)
df["score_prev_3h"] = df["opportunity_score"].shift(3)
df["score_prev_6h"] = df["opportunity_score"].shift(6)

# Score changes
df["score_rise_1h"] = (
    df["opportunity_score"] - df["score_prev_1h"]
)

df["score_rise_3h"] = (
    df["opportunity_score"] - df["score_prev_3h"]
)

df["score_rise_6h"] = (
    df["opportunity_score"] - df["score_prev_6h"]
)

# Score acceleration
df["score_acceleration"] = (
    df["score_rise_1h"] -
    df["score_rise_1h"].shift(1)
)

# Score rise speed
df["score_speed_3h"] = (
    df["score_rise_3h"] / 3.0
)

df["score_speed_6h"] = (
    df["score_rise_6h"] / 6.0
)

# ------------------------------------------------------------
# 6. Causal price confirmation
# ------------------------------------------------------------

if "close" in df.columns:

    df["return_1h"] = df["close"].pct_change(1)

    df["return_3h"] = df["close"].pct_change(3)

    df["return_6h"] = df["close"].pct_change(6)

else:
    raise ValueError("Close price column not found.")

# ------------------------------------------------------------
# 7. Causal score state
# ------------------------------------------------------------

df["score_above_60"] = (
    df["opportunity_score"] >= 60
)

df["score_above_70"] = (
    df["opportunity_score"] >= 70
)

df["score_above_80"] = (
    df["opportunity_score"] >= 80
)

# Score has been rising recently
df["score_rising_1h"] = (
    df["score_rise_1h"] > 0
)

df["score_rising_3h"] = (
    df["score_rise_3h"] > 0
)

df["score_rising_6h"] = (
    df["score_rise_6h"] > 0
)

# ------------------------------------------------------------
# 8. CAUSAL RAPID BUILD RULES
#
# These are deliberately conservative.
#
# A Rapid Build candidate requires:
#
#   A) Opportunity already elevated
#   B) Score is rising
#   C) Price is confirming upward movement
#
# We create three confirmation speeds:
#
#   1H
#   3H
#   6H
#
# No future episode information is used.
# ------------------------------------------------------------

# Base opportunity requirement
base_opportunity = (
    df["opportunity_score"] >= 60
)

# ------------------------------------------------------------
# 1H confirmation
# ------------------------------------------------------------

df["RB_1H_UP_CAUSAL"] = (
    base_opportunity
    &
    (df["score_rise_1h"] >= 1.0)
    &
    (df["return_1h"] > 0)
)

# ------------------------------------------------------------
# 3H confirmation
# ------------------------------------------------------------

df["RB_3H_UP_CAUSAL"] = (
    base_opportunity
    &
    (df["score_rise_3h"] >= 2.0)
    &
    (df["return_3h"] > 0)
)

# ------------------------------------------------------------
# 6H confirmation
# ------------------------------------------------------------

df["RB_6H_UP_CAUSAL"] = (
    base_opportunity
    &
    (df["score_rise_6h"] >= 3.0)
    &
    (df["return_6h"] > 0)
)

# ------------------------------------------------------------
# 9. Downside confirmation
#
# We calculate these too, but DO NOT assume they are tradable.
# Previous research strongly favoured upside confirmation.
# ------------------------------------------------------------

df["RB_1H_DOWN_CAUSAL"] = (
    base_opportunity
    &
    (df["score_rise_1h"] >= 1.0)
    &
    (df["return_1h"] < 0)
)

df["RB_3H_DOWN_CAUSAL"] = (
    base_opportunity
    &
    (df["score_rise_3h"] >= 2.0)
    &
    (df["return_3h"] < 0)
)

df["RB_6H_DOWN_CAUSAL"] = (
    base_opportunity
    &
    (df["score_rise_6h"] >= 3.0)
    &
    (df["return_6h"] < 0)
)

# ------------------------------------------------------------
# 10. A simple causal strength score
# ------------------------------------------------------------

df["causal_rb_strength"] = 0.0

df.loc[
    df["RB_1H_UP_CAUSAL"],
    "causal_rb_strength"
] += 1

df.loc[
    df["RB_3H_UP_CAUSAL"],
    "causal_rb_strength"
] += 1

df.loc[
    df["RB_6H_UP_CAUSAL"],
    "causal_rb_strength"
] += 1

# Stronger if multiple timeframes agree
df["CAUSAL_RAPID_BUILD"] = (
    df["causal_rb_strength"] >= 1
)

# ------------------------------------------------------------
# 11. Create human-readable state
# ------------------------------------------------------------

def classify(row):

    if row["RB_6H_UP_CAUSAL"]:
        return "RAPID_BUILD_6H_UP"

    if row["RB_3H_UP_CAUSAL"]:
        return "RAPID_BUILD_3H_UP"

    if row["RB_1H_UP_CAUSAL"]:
        return "RAPID_BUILD_1H_UP"

    if row["RB_6H_DOWN_CAUSAL"]:
        return "RAPID_BUILD_6H_DOWN"

    if row["RB_3H_DOWN_CAUSAL"]:
        return "RAPID_BUILD_3H_DOWN"

    if row["RB_1H_DOWN_CAUSAL"]:
        return "RAPID_BUILD_1H_DOWN"

    return "NONE"

df["causal_pattern"] = df.apply(classify, axis=1)

# ------------------------------------------------------------
# 12. Remove impossible/incomplete rows
# ------------------------------------------------------------

required = [
    "opportunity_score",
    "score_rise_1h",
    "score_rise_3h",
    "score_rise_6h",
    "return_1h",
    "return_3h",
    "return_6h"
]

df_clean = df.dropna(subset=required).copy()

# ------------------------------------------------------------
# 13. Summary
# ------------------------------------------------------------

print("\n" + "="*70)
print("STEP 44 — CAUSAL RAPID BUILD DETECTOR")
print("="*70)

print(f"Usable rows: {len(df_clean):,}")
print(
    f"Period: {df_clean.index.min()} → {df_clean.index.max()}"
)

for name in [
    "RB_1H_UP_CAUSAL",
    "RB_3H_UP_CAUSAL",
    "RB_6H_UP_CAUSAL",
    "RB_1H_DOWN_CAUSAL",
    "RB_3H_DOWN_CAUSAL",
    "RB_6H_DOWN_CAUSAL"
]:

    print(
        f"{name:<24} "
        f"{int(df_clean[name].sum()):>6,}"
    )

print("\nCurrent state:")
latest = df_clean.iloc[-1]

print("Timestamp:", df_clean.index[-1])
print("Price:", round(latest["close"], 4))
print("Opportunity:", round(latest["opportunity_score"], 2))
print("1H score rise:", round(latest["score_rise_1h"], 2))
print("3H score rise:", round(latest["score_rise_3h"], 2))
print("6H score rise:", round(latest["score_rise_6h"], 2))
print("1H return:", round(latest["return_1h"] * 100, 3), "%")
print("3H return:", round(latest["return_3h"] * 100, 3), "%")
print("6H return:", round(latest["return_6h"] * 100, 3), "%")
print("Causal pattern:", latest["causal_pattern"])
print("RB strength:", latest["causal_rb_strength"])

# ------------------------------------------------------------
# 14. Save
# ------------------------------------------------------------

df_clean.to_parquet(
    OUT_PATH,
    index=True
)

print("\nSaved:")
print(OUT_PATH)

print("\nSTEP 44 COMPLETE.")

In [ ]:
# ============================================================
# STEP 45 — CAUSAL CANDLE-LEVEL BACKTEST V1
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

BASE = Path("/content/gdrive/MyDrive/SilverAgent")

DB_PATH = BASE / "database" / "silver_h1.parquet"
SIGNAL_PATH = BASE / "results" / "causal_rapid_build_v1.parquet"

OUT_TRADES = BASE / "results" / "causal_candle_backtest_v1.parquet"
OUT_SUMMARY = BASE / "results" / "causal_candle_backtest_summary_v1.parquet"
OUT_WINDOWS = BASE / "results" / "causal_candle_backtest_windows_v1.parquet"

# ------------------------------------------------------------
# 2. Load raw candles
# ------------------------------------------------------------

candles = pd.read_parquet(DB_PATH)
signals = pd.read_parquet(SIGNAL_PATH)

candles = candles.sort_index()
signals = signals.sort_index()

print("Candles:", candles.shape)
print("Signals:", signals.shape)

# ------------------------------------------------------------
# 3. Verify DatetimeIndex
# ------------------------------------------------------------

if not isinstance(candles.index, pd.DatetimeIndex):
    raise ValueError("Candles must use DatetimeIndex.")

if not isinstance(signals.index, pd.DatetimeIndex):
    raise ValueError("Signals must use DatetimeIndex.")

# ------------------------------------------------------------
# 4. ATR14
# ------------------------------------------------------------

prev_close = candles["close"].shift(1)

tr1 = candles["high"] - candles["low"]
tr2 = (candles["high"] - prev_close).abs()
tr3 = (candles["low"] - prev_close).abs()

true_range = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)

candles["ATR14"] = true_range.rolling(
    14,
    min_periods=14
).mean()

# ------------------------------------------------------------
# 5. Align causal signals with raw candles
# ------------------------------------------------------------

common = candles.join(
    signals[
        [
            "RB_1H_UP_CAUSAL",
            "RB_3H_UP_CAUSAL",
            "RB_6H_UP_CAUSAL",
            "RB_1H_DOWN_CAUSAL",
            "RB_3H_DOWN_CAUSAL",
            "RB_6H_DOWN_CAUSAL"
        ]
    ],
    how="inner"
)

common = common.sort_index()

print("Common rows:", len(common))

# ------------------------------------------------------------
# 6. Backtest function
# ------------------------------------------------------------

def backtest_strategy(
    data,
    signal_col,
    stop_atr,
    target_atr,
    max_hold=24,
    cost=0.0002
):

    rows = data.reset_index()
    time_col = rows.columns[0]

    trades = []

    i = 0

    while i < len(rows) - 1:

        signal = rows.iloc[i]

        # ----------------------------------------------------
        # Signal must be true at the completed candle.
        # Entry happens on the NEXT real candle.
        # ----------------------------------------------------

        if not bool(signal[signal_col]):
            i += 1
            continue

        entry_i = i + 1

        if entry_i >= len(rows):
            break

        entry = rows.iloc[entry_i]

        if pd.isna(entry["ATR14"]):
            i += 1
            continue

        entry_price = float(entry["open"])
        atr = float(entry["ATR14"])

        stop_price = entry_price - stop_atr * atr
        target_price = entry_price + target_atr * atr

        exit_i = None
        exit_price = None
        exit_reason = None

        # ----------------------------------------------------
        # Hold for max 24 real candles
        # ----------------------------------------------------

        last_i = min(
            entry_i + max_hold,
            len(rows) - 1
        )

        for j in range(entry_i, last_i + 1):

            candle = rows.iloc[j]

            low = float(candle["low"])
            high = float(candle["high"])

            # Conservative assumption:
            # if both stop and target are touched inside
            # the same candle, assume STOP happened first.
            if low <= stop_price:
                exit_i = j
                exit_price = stop_price
                exit_reason = "STOP"
                break

            if high >= target_price:
                exit_i = j
                exit_price = target_price
                exit_reason = "TARGET"
                break

        # ----------------------------------------------------
        # Time exit
        # ----------------------------------------------------

        if exit_i is None:

            exit_i = last_i

            exit_price = float(
                rows.iloc[exit_i]["close"]
            )

            exit_reason = "TIME"

        # ----------------------------------------------------
        # Return
        # ----------------------------------------------------

        gross_return = (
            exit_price / entry_price
        ) - 1.0

        net_return = gross_return - cost

        hold_hours = (
            rows.iloc[exit_i][time_col]
            - entry[time_col]
        ).total_seconds() / 3600.0

        trades.append({
            "signal_time": signal[time_col],
            "entry_time": entry[time_col],
            "exit_time": rows.iloc[exit_i][time_col],

            "entry_price": entry_price,
            "exit_price": exit_price,

            "ATR14": atr,

            "stop_atr": stop_atr,
            "target_atr": target_atr,

            "stop_price": stop_price,
            "target_price": target_price,

            "gross_return": gross_return,
            "net_return": net_return,

            "hold_hours": hold_hours,

            "exit_reason": exit_reason,

            "signal": signal_col
        })

        # ----------------------------------------------------
        # One position at a time
        # ----------------------------------------------------

        i = exit_i + 1

    return pd.DataFrame(trades)


# ------------------------------------------------------------
# 7. Strategies
# ------------------------------------------------------------

signal_cols = [
    "RB_1H_UP_CAUSAL",
    "RB_3H_UP_CAUSAL",
    "RB_6H_UP_CAUSAL"
]

risk_configs = [
    (1.0, 1.5),
    (1.0, 2.0),
    (1.5, 2.0)
]

all_trades = []

for signal_col in signal_cols:

    for stop_atr, target_atr in risk_configs:

        print(
            f"\nRunning "
            f"{signal_col} | "
            f"SL {stop_atr} ATR | "
            f"TP {target_atr} ATR"
        )

        trades = backtest_strategy(
            common,
            signal_col,
            stop_atr,
            target_atr
        )

        if len(trades) > 0:

            trades["config"] = (
                f"{signal_col}_"
                f"{stop_atr}ATR_"
                f"{target_atr}ATR"
            )

            all_trades.append(trades)

            print(
                "Trades:",
                len(trades)
            )

# ------------------------------------------------------------
# 8. Combine
# ------------------------------------------------------------

if not all_trades:
    raise ValueError("No trades generated.")

trades = pd.concat(
    all_trades,
    ignore_index=True
)

# ------------------------------------------------------------
# 9. Performance calculator
# ------------------------------------------------------------

def calculate_stats(group):

    r = group["net_return"].values

    if len(r) == 0:
        return None

    equity = 1000 * np.cumprod(1 + r)

    running_max = np.maximum.accumulate(equity)

    drawdown = (
        equity / running_max
    ) - 1

    gross_profit = r[r > 0].sum()

    gross_loss = -r[r < 0].sum()

    if gross_loss > 0:
        profit_factor = (
            gross_profit / gross_loss
        )
    else:
        profit_factor = np.inf

    return {
        "trades": len(r),

        "mean_return_pct":
            np.mean(r) * 100,

        "median_return_pct":
            np.median(r) * 100,

        "win_rate_pct":
            np.mean(r > 0) * 100,

        "profit_factor":
            profit_factor,

        "final_equity":
            equity[-1],

        "total_return_pct":
            (equity[-1] / 1000 - 1) * 100,

        "max_drawdown_pct":
            drawdown.min() * 100,

        "best_trade_pct":
            r.max() * 100,

        "worst_trade_pct":
            r.min() * 100,

        "avg_hold_hours":
            group["hold_hours"].mean(),

        "stop_pct":
            np.mean(
                group["exit_reason"] == "STOP"
            ) * 100,

        "target_pct":
            np.mean(
                group["exit_reason"] == "TARGET"
            ) * 100,

        "time_pct":
            np.mean(
                group["exit_reason"] == "TIME"
            ) * 100
    }


# ------------------------------------------------------------
# 10. Summary
# ------------------------------------------------------------

summary_rows = []

for config, group in trades.groupby("config"):

    stats = calculate_stats(group)

    stats["config"] = config

    summary_rows.append(stats)

summary = pd.DataFrame(summary_rows)

summary = summary[
    [
        "config",
        "trades",
        "mean_return_pct",
        "median_return_pct",
        "win_rate_pct",
        "profit_factor",
        "final_equity",
        "total_return_pct",
        "max_drawdown_pct",
        "best_trade_pct",
        "worst_trade_pct",
        "avg_hold_hours",
        "stop_pct",
        "target_pct",
        "time_pct"
    ]
]

# ------------------------------------------------------------
# 11. Chronological 5-window test
# ------------------------------------------------------------

window_rows = []

for config, group in trades.groupby("config"):

    group = group.sort_values("entry_time")

    n = len(group)

    if n < 50:
        continue

    boundaries = np.linspace(
        0,
        n,
        6,
        dtype=int
    )

    for w in range(5):

        part = group.iloc[
            boundaries[w]:
            boundaries[w + 1]
        ]

        if len(part) < 10:
            continue

        stats = calculate_stats(part)

        stats["config"] = config
        stats["window"] = w + 1

        stats["start"] = part["entry_time"].iloc[0]
        stats["end"] = part["entry_time"].iloc[-1]

        window_rows.append(stats)

windows = pd.DataFrame(window_rows)

# ------------------------------------------------------------
# 12. Print results
# ------------------------------------------------------------

print("\n")
print("=" * 100)
print("STEP 45 — CAUSAL CANDLE BACKTEST")
print("=" * 100)

display_cols = [
    "config",
    "trades",
    "mean_return_pct",
    "median_return_pct",
    "win_rate_pct",
    "profit_factor",
    "final_equity",
    "total_return_pct",
    "max_drawdown_pct",
    "avg_hold_hours"
]

print("\nOVERALL RESULTS:\n")

print(
    summary[
        display_cols
    ].round(3).to_string(index=False)
)

print("\n")
print("=" * 100)
print("CHRONOLOGICAL WINDOW RESULTS")
print("=" * 100)

print(
    windows[
        [
            "config",
            "window",
            "trades",
            "mean_return_pct",
            "win_rate_pct",
            "profit_factor",
            "max_drawdown_pct"
        ]
    ]
    .round(3)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 13. Exit breakdown
# ------------------------------------------------------------

print("\n")
print("=" * 100)
print("EXIT BREAKDOWN")
print("=" * 100)

exit_table = (
    trades
    .groupby(
        ["config", "exit_reason"]
    )
    .size()
    .unstack(fill_value=0)
)

print(exit_table)

# ------------------------------------------------------------
# 14. Save everything
# ------------------------------------------------------------

trades.to_parquet(
    OUT_TRADES,
    index=False
)

summary.to_parquet(
    OUT_SUMMARY,
    index=False
)

windows.to_parquet(
    OUT_WINDOWS,
    index=False
)

print("\nSaved:")
print(OUT_TRADES)
print(OUT_SUMMARY)
print(OUT_WINDOWS)

print("\nSTEP 45 COMPLETE.")

In [ ]:
# ============================================================
# STEP 46A — REGIME-AWARE RAPID BUILD FILTER V1
# ============================================================
#
# Goal:
# Determine whether causal Rapid Build signals work better
# under particular MARKET REGIMES.
#
# IMPORTANT:
# - No retrospective RAPID_BUILD labels
# - No future information in regime features
# - Train/test are chronological
# - 24h gap between training and test
# - The filter is learned only from the training period
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

BASE = Path("/content/gdrive/MyDrive/SilverAgent")

DB_PATH = BASE / "database" / "silver_h1.parquet"
FEATURE_PATH = BASE / "database" / "silver_h1_features_v2.parquet"
SIGNAL_PATH = BASE / "results" / "causal_rapid_build_v1.parquet"

OUT_PATH = BASE / "results" / "regime_aware_rapid_build_v1.parquet"
OUT_SUMMARY = BASE / "results" / "regime_aware_rapid_build_summary_v1.parquet"

# ------------------------------------------------------------
# 2. Load
# ------------------------------------------------------------

candles = pd.read_parquet(DB_PATH).sort_index()
features = pd.read_parquet(FEATURE_PATH).sort_index()
signals = pd.read_parquet(SIGNAL_PATH).sort_index()

print("Candles:", candles.shape)
print("Features:", features.shape)
print("Signals:", signals.shape)

# ------------------------------------------------------------
# 3. ATR14
# ------------------------------------------------------------

prev_close = candles["close"].shift(1)

tr1 = candles["high"] - candles["low"]
tr2 = (candles["high"] - prev_close).abs()
tr3 = (candles["low"] - prev_close).abs()

true_range = pd.concat(
    [tr1, tr2, tr3],
    axis=1
).max(axis=1)

candles["ATR14"] = true_range.rolling(
    14,
    min_periods=14
).mean()

# ------------------------------------------------------------
# 4. Join causal information
# ------------------------------------------------------------

needed_signal_cols = [
    "RB_1H_UP_CAUSAL",
    "RB_3H_UP_CAUSAL",
    "RB_6H_UP_CAUSAL",
    "RB_1H_DOWN_CAUSAL",
    "RB_3H_DOWN_CAUSAL",
    "RB_6H_DOWN_CAUSAL"
]

signal_part = signals[needed_signal_cols].copy()

df = candles.join(
    signal_part,
    how="inner"
)

# Add feature columns where available
feature_candidates = [
    "close",
    "atr_percent",
    "vol10",
    "vol20",
    "relative_volume",
    "distance_high20",
    "distance_low20",
    "price_vs_MA20",
    "mom3",
    "mom6",
    "mom12"
]

feature_candidates = [
    c for c in feature_candidates
    if c in features.columns
]

df = df.join(
    features[feature_candidates],
    how="left",
    rsuffix="_feature"
)

df = df.sort_index()

# ------------------------------------------------------------
# 5. Remove duplicate columns if any
# ------------------------------------------------------------

df = df.loc[
    :,
    ~df.columns.duplicated()
]

# ------------------------------------------------------------
# 6. Build purely causal regime features
# ------------------------------------------------------------

# Volatility percentile using ONLY previous observations
for col in [
    "atr_percent",
    "vol10",
    "vol20"
]:

    if col in df.columns:

        df[f"{col}_rank"] = (
            df[col]
            .shift(1)
            .expanding(min_periods=500)
            .rank(pct=True)
        )

# Volume/activity percentile
if "relative_volume" in df.columns:

    df["volume_rank"] = (
        df["relative_volume"]
        .shift(1)
        .expanding(min_periods=500)
        .rank(pct=True)
    )

# ------------------------------------------------------------
# 7. Trend regime
# ------------------------------------------------------------

if "price_vs_MA20" in df.columns:

    df["trend_up"] = (
        df["price_vs_MA20"] > 0
    )

else:

    df["trend_up"] = False

# ------------------------------------------------------------
# 8. Momentum regime
# ------------------------------------------------------------

if "mom6" in df.columns:

    df["momentum_up"] = (
        df["mom6"] > 0
    )

else:

    df["momentum_up"] = False

# ------------------------------------------------------------
# 9. Price position
# ------------------------------------------------------------

if "distance_high20" in df.columns:

    df["near_recent_high"] = (
        df["distance_high20"] <= 0.02
    )

else:

    df["near_recent_high"] = False

if "distance_low20" in df.columns:

    df["near_recent_low"] = (
        df["distance_low20"] <= 0.02
    )

else:

    df["near_recent_low"] = False

# ------------------------------------------------------------
# 10. Volatility regime
# ------------------------------------------------------------

if "atr_percent_rank" in df.columns:

    df["VOL_LOW"] = (
        df["atr_percent_rank"] < 0.33
    )

    df["VOL_MEDIUM"] = (
        (df["atr_percent_rank"] >= 0.33)
        &
        (df["atr_percent_rank"] < 0.67)
    )

    df["VOL_HIGH"] = (
        df["atr_percent_rank"] >= 0.67
    )

else:

    df["VOL_LOW"] = False
    df["VOL_MEDIUM"] = False
    df["VOL_HIGH"] = False

# ------------------------------------------------------------
# 11. Market regime labels
# ------------------------------------------------------------

def regime(row):

    if row["VOL_HIGH"] and row["trend_up"] and row["momentum_up"]:
        return "HIGH_VOL_TREND_UP"

    if row["VOL_HIGH"] and row["trend_up"] and not row["momentum_up"]:
        return "HIGH_VOL_TREND_UP_MOM_WEAK"

    if row["VOL_HIGH"] and not row["trend_up"] and row["momentum_up"]:
        return "HIGH_VOL_TREND_DOWN_MOM_UP"

    if row["VOL_HIGH"] and not row["trend_up"] and not row["momentum_up"]:
        return "HIGH_VOL_TREND_DOWN"

    if row["VOL_MEDIUM"] and row["trend_up"] and row["momentum_up"]:
        return "MED_VOL_TREND_UP"

    if row["VOL_MEDIUM"] and row["trend_up"] and not row["momentum_up"]:
        return "MED_VOL_TREND_UP_MOM_WEAK"

    if row["VOL_MEDIUM"] and not row["trend_up"] and row["momentum_up"]:
        return "MED_VOL_TREND_DOWN_MOM_UP"

    if row["VOL_MEDIUM"] and not row["trend_up"] and not row["momentum_up"]:
        return "MED_VOL_TREND_DOWN"

    if row["VOL_LOW"] and row["trend_up"]:
        return "LOW_VOL_TREND_UP"

    if row["VOL_LOW"] and not row["trend_up"]:
        return "LOW_VOL_TREND_DOWN"

    return "UNKNOWN"

df["market_regime"] = df.apply(
    regime,
    axis=1
)

# ------------------------------------------------------------
# 12. Build future outcome
#
# This is ONLY used for evaluation.
# It is NEVER used to construct the live signal.
# ------------------------------------------------------------

# Next 24 real observations
# We use actual timestamps and don't fabricate candles.

future_close = []

times = df.index

for t in times:

    target = t + pd.Timedelta(hours=24)

    pos = times.searchsorted(
        target
    )

    if pos >= len(times):
        future_close.append(np.nan)
        continue

    future_time = times[pos]

    # Don't accept a candle more than 90 minutes away
    if abs(
        (future_time - target).total_seconds()
    ) > 90 * 60:

        future_close.append(np.nan)

    else:

        future_close.append(
            df.iloc[pos]["close"]
        )

df["future_close_24h"] = future_close

df["future_return_24h"] = (
    df["future_close_24h"] /
    df["close"]
) - 1

# ------------------------------------------------------------
# 13. Candidate signal
# ------------------------------------------------------------

df["CAUSAL_RAPID_ANY_UP"] = (
    df["RB_1H_UP_CAUSAL"]
    |
    df["RB_3H_UP_CAUSAL"]
    |
    df["RB_6H_UP_CAUSAL"]
)

# ------------------------------------------------------------
# 14. Only evaluate actual signal rows
# ------------------------------------------------------------

signal_df = df[
    df["CAUSAL_RAPID_ANY_UP"]
    &
    df["future_return_24h"].notna()
].copy()

print("\nCausal Rapid Build signals with outcomes:",
      len(signal_df))

# ------------------------------------------------------------
# 15. Descriptive regime anatomy
# ------------------------------------------------------------

print("\n")
print("=" * 100)
print("REGIME ANATOMY")
print("=" * 100)

regime_summary = (
    signal_df
    .groupby("market_regime")
    .agg(
        signals=("future_return_24h", "size"),
        mean_return=("future_return_24h", "mean"),
        median_return=("future_return_24h", "median"),
        win_rate=("future_return_24h",
                  lambda x: (x > 0).mean()),
        large_move=("future_return_24h",
                    lambda x: (x.abs() >= 0.02).mean())
    )
    .sort_values(
        "mean_return",
        ascending=False
    )
)

regime_summary["mean_return_pct"] = (
    regime_summary["mean_return"] * 100
)

regime_summary["median_return_pct"] = (
    regime_summary["median_return"] * 100
)

regime_summary["win_rate_pct"] = (
    regime_summary["win_rate"] * 100
)

regime_summary["large_move_pct"] = (
    regime_summary["large_move"] * 100
)

print(
    regime_summary[
        [
            "signals",
            "mean_return_pct",
            "median_return_pct",
            "win_rate_pct",
            "large_move_pct"
        ]
    ]
    .round(3)
    .to_string()
)

# ------------------------------------------------------------
# 16. Walk-forward regime selection
#
# Training:
#   determine which regimes were useful
#
# Testing:
#   ONLY use regimes selected from earlier data
#
# 24h purge:
#   keep a gap before test
# ------------------------------------------------------------

data = signal_df.sort_index()

TRAIN_SIZE = 4000
TEST_SIZE = 1000
PURGE = 24

windows = []

start = 0

while True:

    train_end = start + TRAIN_SIZE
    test_start = train_end + PURGE
    test_end = test_start + TEST_SIZE

    if test_end > len(data):
        break

    train = data.iloc[
        start:train_end
    ].copy()

    test = data.iloc[
        test_start:test_end
    ].copy()

    # --------------------------------------------------------
    # Calculate regime performance on TRAIN only
    # --------------------------------------------------------

    train_regime = (
        train
        .groupby("market_regime")
        .agg(
            n=("future_return_24h", "size"),
            mean_return=("future_return_24h", "mean"),
            win_rate=("future_return_24h",
                      lambda x: (x > 0).mean())
        )
    )

    # Require enough observations
    train_regime = train_regime[
        train_regime["n"] >= 30
    ]

    # Select regimes that beat:
    # 1) zero
    # 2) overall training performance
    #
    # This is deliberately conservative.

    train_baseline = (
        train["future_return_24h"]
        .mean()
    )

    selected_regimes = train_regime[
        (train_regime["mean_return"] > 0)
        &
        (train_regime["mean_return"] > train_baseline)
        &
        (train_regime["win_rate"] >= 0.50)
    ].index.tolist()

    # --------------------------------------------------------
    # Test only the regimes selected using TRAIN
    # --------------------------------------------------------

    selected = test[
        test["market_regime"]
        .isin(selected_regimes)
    ]

    # Baseline = all causal signals
    baseline = test

    if len(selected) > 0:

        selected_mean = (
            selected["future_return_24h"]
            .mean()
        )

        selected_win = (
            selected["future_return_24h"] > 0
        ).mean()

    else:

        selected_mean = np.nan
        selected_win = np.nan

    baseline_mean = (
        baseline["future_return_24h"]
        .mean()
    )

    baseline_win = (
        baseline["future_return_24h"] > 0
    ).mean()

    windows.append({

        "window": len(windows) + 1,

        "train_start":
            train.index[0],

        "train_end":
            train.index[-1],

        "test_start":
            test.index[0],

        "test_end":
            test.index[-1],

        "selected_regimes":
            ",".join(selected_regimes),

        "baseline_n":
            len(baseline),

        "selected_n":
            len(selected),

        "baseline_mean_pct":
            baseline_mean * 100,

        "selected_mean_pct":
            selected_mean * 100
            if not np.isnan(selected_mean)
            else np.nan,

        "baseline_win_pct":
            baseline_win * 100,

        "selected_win_pct":
            selected_win * 100
            if not np.isnan(selected_win)
            else np.nan,

        "premium_pp":
            (
                (selected_mean - baseline_mean) * 100
                if not np.isnan(selected_mean)
                else np.nan
            )
    })

    start += TEST_SIZE

windows_df = pd.DataFrame(windows)

# ------------------------------------------------------------
# 17. Results
# ------------------------------------------------------------

print("\n")
print("=" * 100)
print("WALK-FORWARD REGIME FILTER")
print("=" * 100)

print(
    windows_df[
        [
            "window",
            "baseline_n",
            "selected_n",
            "baseline_mean_pct",
            "selected_mean_pct",
            "baseline_win_pct",
            "selected_win_pct",
            "premium_pp",
            "selected_regimes"
        ]
    ]
    .round(3)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 18. Overall walk-forward result
# ------------------------------------------------------------

valid = windows_df[
    windows_df["selected_n"] > 0
].copy()

baseline_weighted = np.average(
    valid["baseline_mean_pct"],
    weights=valid["baseline_n"]
)

selected_weighted = np.average(
    valid["selected_mean_pct"],
    weights=valid["selected_n"]
)

baseline_win_weighted = np.average(
    valid["baseline_win_pct"],
    weights=valid["baseline_n"]
)

selected_win_weighted = np.average(
    valid["selected_win_pct"],
    weights=valid["selected_n"]
)

premium = (
    selected_weighted -
    baseline_weighted
)

positive_windows = (
    valid["premium_pp"] > 0
).sum()

print("\n")
print("=" * 100)
print("FINAL REGIME FILTER RESULT")
print("=" * 100)

print(
    f"Baseline mean 24h return: "
    f"{baseline_weighted:.3f}%"
)

print(
    f"Filtered mean 24h return: "
    f"{selected_weighted:.3f}%"
)

print(
    f"Regime premium: "
    f"{premium:.3f} percentage points"
)

print(
    f"Baseline win rate: "
    f"{baseline_win_weighted:.2f}%"
)

print(
    f"Filtered win rate: "
    f"{selected_win_weighted:.2f}%"
)

print(
    f"Positive windows: "
    f"{positive_windows}/{len(valid)}"
)

# ------------------------------------------------------------
# 19. Current regime
# ------------------------------------------------------------

latest = df.iloc[-1]

print("\n")
print("=" * 100)
print("CURRENT MARKET REGIME")
print("=" * 100)

print("Timestamp:", df.index[-1])
print("Price:", round(latest["close"], 4))

if "continuous_opportunity" in latest.index:
    print(
        "Opportunity:",
        round(latest["continuous_opportunity"], 2)
    )

print(
    "Market regime:",
    latest["market_regime"]
)

print(
    "Volatility rank:",
    round(
        latest["atr_percent_rank"] * 100, 2
    )
    if pd.notna(latest.get("atr_percent_rank"))
    else "N/A"
)

print(
    "Trend up:",
    bool(latest["trend_up"])
)

print(
    "Momentum up:",
    bool(latest["momentum_up"])
)

print(
    "Rapid Build 1H:",
    bool(latest["RB_1H_UP_CAUSAL"])
)

print(
    "Rapid Build 3H:",
    bool(latest["RB_3H_UP_CAUSAL"])
)

print(
    "Rapid Build 6H:",
    bool(latest["RB_6H_UP_CAUSAL"])
)

# ------------------------------------------------------------
# 20. Save
# ------------------------------------------------------------

signal_df.to_parquet(
    OUT_PATH,
    index=True
)

windows_df.to_parquet(
    OUT_SUMMARY,
    index=False
)

print("\nSaved:")
print(OUT_PATH)
print(OUT_SUMMARY)

print("\nSTEP 46A COMPLETE.")

In [ ]:
# ============================================================
# STEP 46A FIX — WALK-FORWARD REGIME VALIDATION
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Make sure signal dataset exists
# ------------------------------------------------------------

if "signal_df" not in globals():
    raise RuntimeError(
        "signal_df is not in memory. "
        "Please rerun Step 46A first."
    )

data = signal_df.sort_index().copy()

print("Signal rows available:", len(data))

# ------------------------------------------------------------
# 2. Validation settings
# ------------------------------------------------------------

TRAIN_SIZE = 4000
TEST_SIZE = 1000
PURGE = 24

windows = []

start = 0

window_number = 0

# ------------------------------------------------------------
# 3. Walk forward
# ------------------------------------------------------------

while True:

    train_end = start + TRAIN_SIZE
    test_start = train_end + PURGE
    test_end = test_start + TEST_SIZE

    if test_end > len(data):
        break

    train = data.iloc[
        start:train_end
    ].copy()

    test = data.iloc[
        test_start:test_end
    ].copy()

    # --------------------------------------------------------
    # TRAIN ONLY
    # --------------------------------------------------------

    train_regime = (
        train
        .groupby("market_regime")
        .agg(
            n=("future_return_24h", "size"),
            mean_return=("future_return_24h", "mean"),
            win_rate=(
                "future_return_24h",
                lambda x: (x > 0).mean()
            )
        )
    )

    # Require reasonable sample size
    train_regime = train_regime[
        train_regime["n"] >= 30
    ].copy()

    # Overall training performance
    train_baseline = (
        train["future_return_24h"].mean()
    )

    # --------------------------------------------------------
    # Select regimes using TRAIN ONLY
    # --------------------------------------------------------

    selected_regimes = train_regime[
        (train_regime["mean_return"] > 0)
        &
        (train_regime["mean_return"] > train_baseline)
        &
        (train_regime["win_rate"] >= 0.50)
    ].index.tolist()

    # --------------------------------------------------------
    # TEST
    # --------------------------------------------------------

    baseline = test

    selected = test[
        test["market_regime"].isin(
            selected_regimes
        )
    ]

    baseline_mean = (
        baseline["future_return_24h"].mean()
    )

    baseline_win = (
        baseline["future_return_24h"] > 0
    ).mean()

    if len(selected) > 0:

        selected_mean = (
            selected["future_return_24h"].mean()
        )

        selected_win = (
            selected["future_return_24h"] > 0
        ).mean()

    else:

        selected_mean = np.nan
        selected_win = np.nan

    window_number += 1

    windows.append({

        "window":
            window_number,

        "train_start":
            train.index[0],

        "train_end":
            train.index[-1],

        "test_start":
            test.index[0],

        "test_end":
            test.index[-1],

        "baseline_n":
            len(baseline),

        "selected_n":
            len(selected),

        "baseline_mean_pct":
            baseline_mean * 100,

        "selected_mean_pct":
            selected_mean * 100
            if pd.notna(selected_mean)
            else np.nan,

        "baseline_win_pct":
            baseline_win * 100,

        "selected_win_pct":
            selected_win * 100
            if pd.notna(selected_win)
            else np.nan,

        "premium_pp":
            (
                (selected_mean - baseline_mean)
                * 100
                if pd.notna(selected_mean)
                else np.nan
            ),

        "selected_regimes":
            ", ".join(selected_regimes)
    })

    start += TEST_SIZE

# ------------------------------------------------------------
# 4. Build dataframe
# ------------------------------------------------------------

windows_df = pd.DataFrame(windows)

print("\nNumber of walk-forward windows:", len(windows_df))

if len(windows_df) == 0:

    print(
        "\nWARNING: No windows were created."
    )

    print(
        "Data rows:",
        len(data)
    )

    print(
        "Required minimum:",
        TRAIN_SIZE + PURGE + TEST_SIZE
    )

else:

    # --------------------------------------------------------
    # 5. Display windows
    # --------------------------------------------------------

    print("\n")
    print("=" * 110)
    print("WALK-FORWARD REGIME FILTER")
    print("=" * 110)

    print(
        windows_df[
            [
                "window",
                "baseline_n",
                "selected_n",
                "baseline_mean_pct",
                "selected_mean_pct",
                "baseline_win_pct",
                "selected_win_pct",
                "premium_pp",
                "selected_regimes"
            ]
        ]
        .round(3)
        .to_string(index=False)
    )

    # --------------------------------------------------------
    # 6. Weighted result
    # --------------------------------------------------------

    valid = windows_df[
        windows_df["selected_n"] > 0
    ].copy()

    if len(valid) > 0:

        baseline_weighted = np.average(
            valid["baseline_mean_pct"],
            weights=valid["baseline_n"]
        )

        selected_weighted = np.average(
            valid["selected_mean_pct"],
            weights=valid["selected_n"]
        )

        baseline_win_weighted = np.average(
            valid["baseline_win_pct"],
            weights=valid["baseline_n"]
        )

        selected_win_weighted = np.average(
            valid["selected_win_pct"],
            weights=valid["selected_n"]
        )

        premium = (
            selected_weighted -
            baseline_weighted
        )

        positive_windows = (
            valid["premium_pp"] > 0
        ).sum()

        print("\n")
        print("=" * 110)
        print("FINAL REGIME FILTER RESULT")
        print("=" * 110)

        print(
            f"Baseline mean 24h return: "
            f"{baseline_weighted:.3f}%"
        )

        print(
            f"Filtered mean 24h return: "
            f"{selected_weighted:.3f}%"
        )

        print(
            f"Regime premium: "
            f"{premium:.3f} percentage points"
        )

        print(
            f"Baseline win rate: "
            f"{baseline_win_weighted:.2f}%"
        )

        print(
            f"Filtered win rate: "
            f"{selected_win_weighted:.2f}%"
        )

        print(
            f"Positive windows: "
            f"{positive_windows}/{len(valid)}"
        )

        # ----------------------------------------------------
        # 7. Save
        # ----------------------------------------------------

        OUT_SUMMARY = (
            "/content/gdrive/MyDrive/"
            "SilverAgent/results/"
            "regime_aware_rapid_build_v1_walkforward.parquet"
        )

        windows_df.to_parquet(
            OUT_SUMMARY,
            index=False
        )

        print("\nSaved:")
        print(OUT_SUMMARY)

    else:

        print(
            "\nNo test windows contained selected regimes."
        )

print("\nSTEP 46A WALK-FORWARD FIX COMPLETE.")

In [ ]:
# ============================================================
# STEP 46A — REGIME FILTER OOS VALIDATION V2
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. Get signal dataset
# ------------------------------------------------------------

if "signal_df" not in globals():
    raise RuntimeError(
        "signal_df not found. Please rerun Step 46A."
    )

data = signal_df.sort_index().copy()

print("Signal rows:", len(data))

# ------------------------------------------------------------
# 2. Validation design
# ------------------------------------------------------------

TRAIN_SIZE = 2000
TEST_SIZE = 500
PURGE = 24

windows = []

start = 0
window_number = 0

# ------------------------------------------------------------
# 3. Walk-forward validation
# ------------------------------------------------------------

while True:

    train_end = start + TRAIN_SIZE
    test_start = train_end + PURGE
    test_end = test_start + TEST_SIZE

    if test_end > len(data):
        break

    train = data.iloc[
        start:train_end
    ].copy()

    test = data.iloc[
        test_start:test_end
    ].copy()

    # --------------------------------------------------------
    # TRAIN ONLY: determine useful regimes
    # --------------------------------------------------------

    regime_train = (
        train
        .groupby("market_regime")
        .agg(
            n=("future_return_24h", "size"),
            mean_return=("future_return_24h", "mean"),
            win_rate=(
                "future_return_24h",
                lambda x: (x > 0).mean()
            )
        )
    )

    # Require minimum sample size
    regime_train = regime_train[
        regime_train["n"] >= 25
    ].copy()

    overall_train_mean = (
        train["future_return_24h"].mean()
    )

    # --------------------------------------------------------
    # TRAIN-ONLY regime selection
    #
    # A regime qualifies if:
    #
    # 1. Positive mean return
    # 2. Beats overall training baseline
    # 3. Win rate >= 50%
    # --------------------------------------------------------

    selected_regimes = regime_train[
        (regime_train["mean_return"] > 0)
        &
        (
            regime_train["mean_return"]
            > overall_train_mean
        )
        &
        (
            regime_train["win_rate"] >= 0.50
        )
    ].index.tolist()

    # --------------------------------------------------------
    # TEST
    # --------------------------------------------------------

    baseline = test

    filtered = test[
        test["market_regime"].isin(
            selected_regimes
        )
    ]

    baseline_mean = (
        baseline["future_return_24h"].mean()
    )

    baseline_win = (
        baseline["future_return_24h"] > 0
    ).mean()

    if len(filtered) > 0:

        filtered_mean = (
            filtered["future_return_24h"].mean()
        )

        filtered_win = (
            filtered["future_return_24h"] > 0
        ).mean()

        filtered_large = (
            filtered["future_return_24h"].abs()
            >= 0.02
        ).mean()

    else:

        filtered_mean = np.nan
        filtered_win = np.nan
        filtered_large = np.nan

    baseline_large = (
        baseline["future_return_24h"].abs()
        >= 0.02
    ).mean()

    window_number += 1

    windows.append({

        "window": window_number,

        "train_start": train.index[0],
        "train_end": train.index[-1],

        "test_start": test.index[0],
        "test_end": test.index[-1],

        "baseline_n": len(baseline),
        "filtered_n": len(filtered),

        "baseline_mean_pct":
            baseline_mean * 100,

        "filtered_mean_pct":
            filtered_mean * 100
            if pd.notna(filtered_mean)
            else np.nan,

        "baseline_win_pct":
            baseline_win * 100,

        "filtered_win_pct":
            filtered_win * 100
            if pd.notna(filtered_win)
            else np.nan,

        "baseline_large_pct":
            baseline_large * 100,

        "filtered_large_pct":
            filtered_large * 100
            if pd.notna(filtered_large)
            else np.nan,

        "premium_pp":
            (
                (filtered_mean - baseline_mean)
                * 100
                if pd.notna(filtered_mean)
                else np.nan
            ),

        "selected_regimes":
            ", ".join(selected_regimes)
    })

    start += TEST_SIZE

# ------------------------------------------------------------
# 4. Results dataframe
# ------------------------------------------------------------

results = pd.DataFrame(windows)

print("\n")
print("=" * 110)
print("REGIME FILTER — OUT-OF-SAMPLE RESULTS")
print("=" * 110)

print(
    results[
        [
            "window",
            "baseline_n",
            "filtered_n",
            "baseline_mean_pct",
            "filtered_mean_pct",
            "premium_pp",
            "baseline_win_pct",
            "filtered_win_pct",
            "baseline_large_pct",
            "filtered_large_pct",
            "selected_regimes"
        ]
    ]
    .round(3)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 5. Weighted OOS results
# ------------------------------------------------------------

valid = results[
    results["filtered_n"] > 0
].copy()

print("\n")
print("=" * 110)
print("WEIGHTED OOS RESULT")
print("=" * 110)

if len(valid) > 0:

    baseline_mean = np.average(
        valid["baseline_mean_pct"],
        weights=valid["baseline_n"]
    )

    filtered_mean = np.average(
        valid["filtered_mean_pct"],
        weights=valid["filtered_n"]
    )

    baseline_win = np.average(
        valid["baseline_win_pct"],
        weights=valid["baseline_n"]
    )

    filtered_win = np.average(
        valid["filtered_win_pct"],
        weights=valid["filtered_n"]
    )

    baseline_large = np.average(
        valid["baseline_large_pct"],
        weights=valid["baseline_n"]
    )

    filtered_large = np.average(
        valid["filtered_large_pct"],
        weights=valid["filtered_n"]
    )

    premium = (
        filtered_mean -
        baseline_mean
    )

    positive_windows = (
        valid["premium_pp"] > 0
    ).sum()

    print(
        f"Windows tested: {len(valid)}"
    )

    print(
        f"Baseline mean 24h return: "
        f"{baseline_mean:.3f}%"
    )

    print(
        f"Filtered mean 24h return: "
        f"{filtered_mean:.3f}%"
    )

    print(
        f"Regime premium: "
        f"{premium:+.3f} pp"
    )

    print(
        f"Baseline win rate: "
        f"{baseline_win:.2f}%"
    )

    print(
        f"Filtered win rate: "
        f"{filtered_win:.2f}%"
    )

    print(
        f"Baseline large-move rate: "
        f"{baseline_large:.2f}%"
    )

    print(
        f"Filtered large-move rate: "
        f"{filtered_large:.2f}%"
    )

    print(
        f"Positive windows: "
        f"{positive_windows}/{len(valid)}"
    )

else:

    print("No valid filtered windows.")

# ------------------------------------------------------------
# 6. Current regime
# ------------------------------------------------------------

latest = data.iloc[-1]

print("\n")
print("=" * 110)
print("CURRENT STATE")
print("=" * 110)

print("Timestamp:", data.index[-1])
print("Price:", round(latest["close"], 4))

if "continuous_opportunity" in latest.index:
    print(
        "Opportunity:",
        round(
            latest["continuous_opportunity"],
            2
        )
    )

print(
    "Regime:",
    latest["market_regime"]
)

print(
    "Rapid 1H:",
    bool(latest["RB_1H_UP_CAUSAL"])
)

print(
    "Rapid 3H:",
    bool(latest["RB_3H_UP_CAUSAL"])
)

print(
    "Rapid 6H:",
    bool(latest["RB_6H_UP_CAUSAL"])
)

# ------------------------------------------------------------
# 7. Save
# ------------------------------------------------------------

OUT = Path(
    "/content/gdrive/MyDrive/"
    "SilverAgent/results/"
    "regime_filter_oos_v2.parquet"
)

results.to_parquet(
    OUT,
    index=False
)

print("\nSaved:")
print(OUT)

print("\nSTEP 46A V2 COMPLETE.")

In [ ]:
# ============================================================
# STEP 46B — CAUSAL REGIME SIMILARITY ENGINE V1
# ============================================================
#
# Goal:
# Compare the current market state with SIMILAR PAST STATES.
#
# The model learns only from the past.
#
# No:
#   - retrospective Rapid Build labels
#   - future pattern labels
#   - random train/test split
#   - future information in the similarity features
#
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

# ------------------------------------------------------------
# 1. Load existing causal signal dataset
# ------------------------------------------------------------

if "signal_df" not in globals():
    raise RuntimeError(
        "signal_df not found. "
        "Please rerun Step 46A before running this cell."
    )

data = signal_df.sort_index().copy()

print("Signal rows:", len(data))

# ------------------------------------------------------------
# 2. Causal feature construction
# ------------------------------------------------------------
#
# These describe the market state AT THE SIGNAL TIME.
#
# The important point:
# every feature is already known at that timestamp.
#
# ------------------------------------------------------------

candidate_features = [
    "continuous_opportunity",
    "atr_percent",
    "vol10",
    "vol20",
    "relative_volume",
    "distance_high20",
    "distance_low20",
    "price_vs_MA20",
    "mom3",
    "mom6",
    "mom12"
]

available = [
    c for c in candidate_features
    if c in data.columns
]

print("\nAvailable similarity features:")
print(available)

# ------------------------------------------------------------
# 3. Add causal score behaviour
# ------------------------------------------------------------

if "continuous_opportunity" in data.columns:

    data["score_change_1h"] = (
        data["continuous_opportunity"]
        - data["continuous_opportunity"].shift(1)
    )

    data["score_change_3h"] = (
        data["continuous_opportunity"]
        - data["continuous_opportunity"].shift(3)
    )

    data["score_change_6h"] = (
        data["continuous_opportunity"]
        - data["continuous_opportunity"].shift(6)
    )

    data["score_acceleration"] = (
        data["score_change_1h"]
        - data["score_change_1h"].shift(1)
    )

    available += [
        "score_change_1h",
        "score_change_3h",
        "score_change_6h",
        "score_acceleration"
    ]

# Remove duplicates
available = list(dict.fromkeys(available))

# ------------------------------------------------------------
# 4. Keep only complete feature rows
# ------------------------------------------------------------

model_data = data.dropna(
    subset=available + ["future_return_24h"]
).copy()

print(
    "\nComplete model rows:",
    len(model_data)
)

# ------------------------------------------------------------
# 5. Validation settings
# ------------------------------------------------------------

TRAIN_SIZE = 1500
TEST_SIZE = 500
PURGE = 24

# Number of nearest historical states
K_VALUES = [
    25,
    50,
    100
]

# Similarity confidence thresholds
CONFIDENCE_THRESHOLDS = [
    0.55,
    0.60,
    0.65
]

results = []

window_number = 0
start = 0

# ------------------------------------------------------------
# 6. Walk-forward validation
# ------------------------------------------------------------

while True:

    train_end = start + TRAIN_SIZE

    test_start = (
        train_end + PURGE
    )

    test_end = (
        test_start + TEST_SIZE
    )

    if test_end > len(model_data):
        break

    train = model_data.iloc[
        start:train_end
    ].copy()

    test = model_data.iloc[
        test_start:test_end
    ].copy()

    window_number += 1

    print(
        f"\nWindow {window_number}: "
        f"train={len(train)} "
        f"test={len(test)}"
    )

    # --------------------------------------------------------
    # 7. Fit scaler on TRAIN ONLY
    # --------------------------------------------------------

    scaler = StandardScaler()

    X_train = scaler.fit_transform(
        train[available]
    )

    X_test = scaler.transform(
        test[available]
    )

    # --------------------------------------------------------
    # 8. Fit nearest-neighbour model on TRAIN ONLY
    # --------------------------------------------------------

    max_k = max(K_VALUES)

    nn = NearestNeighbors(
        n_neighbors=max_k,
        metric="euclidean"
    )

    nn.fit(X_train)

    distances, indices = nn.kneighbors(
        X_test
    )

    # --------------------------------------------------------
    # 9. Evaluate each K
    # --------------------------------------------------------

    for k in K_VALUES:

        for threshold in CONFIDENCE_THRESHOLDS:

            selected_returns = []

            neighbour_mean_returns = []

            confidence_values = []

            for row_i in range(len(test)):

                # First k historical analogues
                neighbour_idx = indices[
                    row_i,
                    :k
                ]

                neighbour_dist = distances[
                    row_i,
                    :k
                ]

                historical_returns = (
                    train.iloc[
                        neighbour_idx
                    ]["future_return_24h"]
                    .values
                )

                mean_return = (
                    historical_returns.mean()
                )

                positive_fraction = (
                    historical_returns > 0
                ).mean()

                # ------------------------------------------------
                # Similarity confidence
                #
                # Higher when neighbours are close together.
                #
                # This is NOT probability of profit.
                # It is similarity quality.
                # ------------------------------------------------

                mean_distance = (
                    neighbour_dist.mean()
                )

                confidence = (
                    1 /
                    (1 + mean_distance)
                )

                neighbour_mean_returns.append(
                    mean_return
                )

                confidence_values.append(
                    confidence
                )

                # ------------------------------------------------
                # Select only if:
                #
                # 1. historical analogue has positive expected
                #    return
                #
                # 2. majority of analogues were positive
                #
                # 3. similarity is sufficiently strong
                # ------------------------------------------------

                if (
                    mean_return > 0
                    and
                    positive_fraction >= 0.50
                    and
                    confidence >= threshold
                ):

                    selected_returns.append(
                        test.iloc[
                            row_i
                        ]["future_return_24h"]
                    )

            # ----------------------------------------------------
            # Test statistics
            # ----------------------------------------------------

            baseline_returns = (
                test["future_return_24h"]
                .values
            )

            baseline_mean = (
                baseline_returns.mean()
            )

            baseline_win = (
                baseline_returns > 0
            ).mean()

            baseline_large = (
                np.abs(
                    baseline_returns
                ) >= 0.02
            ).mean()

            if len(selected_returns) > 0:

                selected_returns = np.array(
                    selected_returns
                )

                selected_mean = (
                    selected_returns.mean()
                )

                selected_win = (
                    selected_returns > 0
                ).mean()

                selected_large = (
                    np.abs(
                        selected_returns
                    ) >= 0.02
                ).mean()

            else:

                selected_mean = np.nan
                selected_win = np.nan
                selected_large = np.nan

            results.append({

                "window":
                    window_number,

                "k":
                    k,

                "confidence_threshold":
                    threshold,

                "baseline_n":
                    len(baseline_returns),

                "selected_n":
                    len(selected_returns)
                    if isinstance(
                        selected_returns,
                        np.ndarray
                    )
                    else 0,

                "baseline_mean_pct":
                    baseline_mean * 100,

                "selected_mean_pct":
                    selected_mean * 100
                    if pd.notna(selected_mean)
                    else np.nan,

                "baseline_win_pct":
                    baseline_win * 100,

                "selected_win_pct":
                    selected_win * 100
                    if pd.notna(selected_win)
                    else np.nan,

                "baseline_large_pct":
                    baseline_large * 100,

                "selected_large_pct":
                    selected_large * 100
                    if pd.notna(selected_large)
                    else np.nan,

                "premium_pp":
                    (
                        (selected_mean -
                         baseline_mean)
                        * 100
                        if pd.notna(
                            selected_mean
                        )
                        else np.nan
                    ),

                "train_start":
                    train.index[0],

                "train_end":
                    train.index[-1],

                "test_start":
                    test.index[0],

                "test_end":
                    test.index[-1]
            })

# ------------------------------------------------------------
# 10. Results dataframe
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

print("\n")
print("=" * 110)
print("STEP 46B — REGIME SIMILARITY RESULTS")
print("=" * 110)

print(
    "\nWindows generated:",
    results_df["window"].nunique()
)

print(
    "\nDetailed results:"
)

print(
    results_df[
        [
            "window",
            "k",
            "confidence_threshold",
            "baseline_n",
            "selected_n",
            "baseline_mean_pct",
            "selected_mean_pct",
            "premium_pp",
            "baseline_win_pct",
            "selected_win_pct",
            "baseline_large_pct",
            "selected_large_pct"
        ]
    ]
    .round(3)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 11. Aggregate by configuration
# ------------------------------------------------------------

config_summary = (
    results_df[
        results_df["selected_n"] > 0
    ]
    .groupby(
        [
            "k",
            "confidence_threshold"
        ]
    )
    .apply(
        lambda g: pd.Series({

            "windows":
                len(g),

            "baseline_mean_pct":
                np.average(
                    g["baseline_mean_pct"],
                    weights=g["baseline_n"]
                ),

            "selected_mean_pct":
                np.average(
                    g["selected_mean_pct"],
                    weights=g["selected_n"]
                ),

            "premium_pp":
                np.average(
                    g["selected_mean_pct"],
                    weights=g["selected_n"]
                )
                -
                np.average(
                    g["baseline_mean_pct"],
                    weights=g["baseline_n"]
                ),

            "baseline_win_pct":
                np.average(
                    g["baseline_win_pct"],
                    weights=g["baseline_n"]
                ),

            "selected_win_pct":
                np.average(
                    g["selected_win_pct"],
                    weights=g["selected_n"]
                ),

            "baseline_large_pct":
                np.average(
                    g["baseline_large_pct"],
                    weights=g["baseline_n"]
                ),

            "selected_large_pct":
                np.average(
                    g["selected_large_pct"],
                    weights=g["selected_n"]
                ),

            "positive_windows":
                (
                    g["premium_pp"] > 0
                ).sum(),

            "total_selected":
                g["selected_n"].sum()
        })
    )
    .reset_index()
)

print("\n")
print("=" * 110)
print("CONFIGURATION SUMMARY")
print("=" * 110)

print(
    config_summary
    .sort_values(
        "premium_pp",
        ascending=False
    )
    .round(3)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 12. Current-state nearest neighbours
# ------------------------------------------------------------

latest = model_data.iloc[-1]

# Use all data before latest
history = model_data.iloc[:-1]

scaler_final = StandardScaler()

X_hist = scaler_final.fit_transform(
    history[available]
)

X_current = scaler_final.transform(
    latest[available]
    .to_frame()
    .T
)

nn_final = NearestNeighbors(
    n_neighbors=min(
        50,
        len(history)
    ),
    metric="euclidean"
)

nn_final.fit(X_hist)

dist, idx = nn_final.kneighbors(
    X_current
)

neighbour_rows = history.iloc[
    idx[0]
].copy()

neighbour_rows["distance"] = dist[0]

# ------------------------------------------------------------
# Current analogue statistics
# ------------------------------------------------------------

print("\n")
print("=" * 110)
print("CURRENT MARKET — HISTORICAL ANALOGUES")
print("=" * 110)

print(
    "Timestamp:",
    model_data.index[-1]
)

print(
    "Price:",
    round(
        latest["close"],
        4
    )
)

if "continuous_opportunity" in latest.index:

    print(
        "Opportunity:",
        round(
            latest[
                "continuous_opportunity"
            ],
            2
        )
    )

print(
    "\nClosest historical states:"
)

print(
    neighbour_rows[
        [
            "future_return_24h",
            "distance"
        ]
    ]
    .head(10)
    .round(4)
    .to_string()
)

current_mean = (
    neighbour_rows[
        "future_return_24h"
    ]
    .head(50)
    .mean()
)

current_win = (
    neighbour_rows[
        "future_return_24h"
    ]
    .head(50)
    > 0
).mean()

current_large = (
    neighbour_rows[
        "future_return_24h"
    ]
    .head(50)
    .abs()
    >= 0.02
).mean()

print(
    "\n50-neighbour historical analogue:"
)

print(
    f"Mean 24h return: "
    f"{current_mean * 100:.3f}%"
)

print(
    f"Positive outcome rate: "
    f"{current_win * 100:.2f}%"
)

print(
    f"Large-move rate: "
    f"{current_large * 100:.2f}%"
)

# ------------------------------------------------------------
# 13. Save
# ------------------------------------------------------------

OUT_RESULTS = Path(
    "/content/gdrive/MyDrive/"
    "SilverAgent/results/"
    "regime_similarity_oos_v1.parquet"
)

OUT_CONFIG = Path(
    "/content/gdrive/MyDrive/"
    "SilverAgent/results/"
    "regime_similarity_config_summary_v1.parquet"
)

results_df.to_parquet(
    OUT_RESULTS,
    index=False
)

config_summary.to_parquet(
    OUT_CONFIG,
    index=False
)

print("\nSaved:")
print(OUT_RESULTS)
print(OUT_CONFIG)

print("\nSTEP 46B COMPLETE.")

In [ ]:
# ============================================================
# STEP 46B V2 — FAST CAUSAL REGIME SIMILARITY
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

# ------------------------------------------------------------
# 1. Existing data
# ------------------------------------------------------------

if "signal_df" not in globals():
    raise RuntimeError(
        "signal_df not found. Please rerun Step 46A."
    )

data = signal_df.sort_index().copy()

print("Signal rows:", len(data))

# ------------------------------------------------------------
# 2. Features
# ------------------------------------------------------------

candidate_features = [
    "atr_percent",
    "vol10",
    "vol20",
    "relative_volume",
    "distance_high20",
    "distance_low20",
    "price_vs_MA20",
    "mom3",
    "mom6",
    "mom12"
]

available = [
    c for c in candidate_features
    if c in data.columns
]

# Causal opportunity behaviour
if "continuous_opportunity" in data.columns:

    data["score_change_1h"] = (
        data["continuous_opportunity"]
        - data["continuous_opportunity"].shift(1)
    )

    data["score_change_3h"] = (
        data["continuous_opportunity"]
        - data["continuous_opportunity"].shift(3)
    )

    data["score_change_6h"] = (
        data["continuous_opportunity"]
        - data["continuous_opportunity"].shift(6)
    )

    available += [
        "score_change_1h",
        "score_change_3h",
        "score_change_6h"
    ]

available = list(dict.fromkeys(available))

print("Similarity features:", available)

# ------------------------------------------------------------
# 3. Complete data
# ------------------------------------------------------------

model_data = data.dropna(
    subset=available + ["future_return_24h"]
).copy()

print(
    "Complete model rows:",
    len(model_data)
)

# ------------------------------------------------------------
# 4. Validation design
# ------------------------------------------------------------

TRAIN_SIZE = 1500
TEST_SIZE = 500
PURGE = 24

K_VALUES = [25, 50, 100]

CONF_THRESHOLDS = [
    0.55,
    0.60,
    0.65
]

results = []

start = 0
window_number = 0

# ------------------------------------------------------------
# 5. Walk-forward
# ------------------------------------------------------------

while True:

    train_end = (
        start + TRAIN_SIZE
    )

    test_start = (
        train_end + PURGE
    )

    test_end = (
        test_start + TEST_SIZE
    )

    if test_end > len(model_data):
        break

    train = model_data.iloc[
        start:train_end
    ]

    test = model_data.iloc[
        test_start:test_end
    ]

    window_number += 1

    print(
        f"Window {window_number}: "
        f"train {len(train)}, "
        f"test {len(test)}"
    )

    # --------------------------------------------------------
    # TRAIN-ONLY scaling
    # --------------------------------------------------------

    scaler = StandardScaler()

    X_train = scaler.fit_transform(
        train[available].values
    )

    X_test = scaler.transform(
        test[available].values
    )

    y_train = (
        train["future_return_24h"]
        .values
    )

    y_test = (
        test["future_return_24h"]
        .values
    )

    # --------------------------------------------------------
    # One neighbour query for the entire TEST block
    # --------------------------------------------------------

    nn = NearestNeighbors(
        n_neighbors=max(K_VALUES),
        metric="euclidean",
        n_jobs=-1
    )

    nn.fit(X_train)

    distances, indices = nn.kneighbors(
        X_test
    )

    # --------------------------------------------------------
    # Convert neighbour outcomes into matrix
    #
    # shape:
    #   test rows x neighbours
    # --------------------------------------------------------

    neighbour_returns = (
        y_train[indices]
    )

    # --------------------------------------------------------
    # Baseline
    # --------------------------------------------------------

    baseline_mean = (
        y_test.mean()
    )

    baseline_win = (
        y_test > 0
    ).mean()

    baseline_large = (
        np.abs(y_test) >= 0.02
    ).mean()

    # --------------------------------------------------------
    # Evaluate K values
    # --------------------------------------------------------

    for k in K_VALUES:

        # First K neighbours
        kr = neighbour_returns[:, :k]
        kd = distances[:, :k]

        # Historical analogue mean
        predicted_return = (
            kr.mean(axis=1)
        )

        # Historical positive fraction
        positive_fraction = (
            (kr > 0).mean(axis=1)
        )

        # Similarity confidence
        mean_distance = (
            kd.mean(axis=1)
        )

        confidence = (
            1.0 /
            (1.0 + mean_distance)
        )

        # ----------------------------------------------------
        # Test confidence thresholds
        # ----------------------------------------------------

        for threshold in CONF_THRESHOLDS:

            mask = (
                (predicted_return > 0)
                &
                (positive_fraction >= 0.50)
                &
                (confidence >= threshold)
            )

            selected = y_test[mask]

            if len(selected) > 0:

                selected_mean = (
                    selected.mean()
                )

                selected_win = (
                    selected > 0
                ).mean()

                selected_large = (
                    np.abs(selected) >= 0.02
                ).mean()

            else:

                selected_mean = np.nan
                selected_win = np.nan
                selected_large = np.nan

            results.append({

                "window":
                    window_number,

                "k":
                    k,

                "confidence_threshold":
                    threshold,

                "baseline_n":
                    len(y_test),

                "selected_n":
                    len(selected),

                "baseline_mean_pct":
                    baseline_mean * 100,

                "selected_mean_pct":
                    selected_mean * 100
                    if pd.notna(
                        selected_mean
                    )
                    else np.nan,

                "premium_pp":
                    (
                        selected_mean -
                        baseline_mean
                    ) * 100
                    if pd.notna(
                        selected_mean
                    )
                    else np.nan,

                "baseline_win_pct":
                    baseline_win * 100,

                "selected_win_pct":
                    selected_win * 100
                    if pd.notna(
                        selected_win
                    )
                    else np.nan,

                "baseline_large_pct":
                    baseline_large * 100,

                "selected_large_pct":
                    selected_large * 100
                    if pd.notna(
                        selected_large
                    )
                    else np.nan
            })

    start += TEST_SIZE

# ------------------------------------------------------------
# 6. Results
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

print("\n")
print("=" * 110)
print("STEP 46B V2 — CONFIGURATION RESULTS")
print("=" * 110)

print(
    results_df[
        [
            "window",
            "k",
            "confidence_threshold",
            "selected_n",
            "baseline_mean_pct",
            "selected_mean_pct",
            "premium_pp",
            "baseline_win_pct",
            "selected_win_pct",
            "baseline_large_pct",
            "selected_large_pct"
        ]
    ]
    .round(3)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 7. Configuration summary
# ------------------------------------------------------------

summary_rows = []

for (k, threshold), g in results_df.groupby(
    ["k", "confidence_threshold"]
):

    valid = g[
        g["selected_n"] > 0
    ]

    if len(valid) == 0:
        continue

    baseline_mean = np.average(
        valid["baseline_mean_pct"],
        weights=valid["baseline_n"]
    )

    selected_mean = np.average(
        valid["selected_mean_pct"],
        weights=valid["selected_n"]
    )

    baseline_win = np.average(
        valid["baseline_win_pct"],
        weights=valid["baseline_n"]
    )

    selected_win = np.average(
        valid["selected_win_pct"],
        weights=valid["selected_n"]
    )

    baseline_large = np.average(
        valid["baseline_large_pct"],
        weights=valid["baseline_n"]
    )

    selected_large = np.average(
        valid["selected_large_pct"],
        weights=valid["selected_n"]
    )

    summary_rows.append({

        "k": k,

        "confidence_threshold":
            threshold,

        "windows":
            len(valid),

        "total_selected":
            valid["selected_n"].sum(),

        "baseline_mean_pct":
            baseline_mean,

        "selected_mean_pct":
            selected_mean,

        "premium_pp":
            selected_mean -
            baseline_mean,

        "baseline_win_pct":
            baseline_win,

        "selected_win_pct":
            selected_win,

        "baseline_large_pct":
            baseline_large,

        "selected_large_pct":
            selected_large,

        "positive_windows":
            (
                valid["premium_pp"] > 0
            ).sum()
    })

config_summary = pd.DataFrame(
    summary_rows
)

print("\n")
print("=" * 110)
print("CONFIGURATION SUMMARY")
print("=" * 110)

print(
    config_summary
    .sort_values(
        "premium_pp",
        ascending=False
    )
    .round(3)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 8. Current state analogue
# ------------------------------------------------------------

latest = model_data.iloc[-1]

history = model_data.iloc[:-1]

scaler = StandardScaler()

X_history = scaler.fit_transform(
    history[available].values
)

X_current = scaler.transform(
    latest[available]
    .to_numpy()
    .reshape(1, -1)
)

nn = NearestNeighbors(
    n_neighbors=min(
        50,
        len(history)
    ),
    metric="euclidean",
    n_jobs=-1
)

nn.fit(X_history)

distances, indices = nn.kneighbors(
    X_current
)

analogue_returns = (
    history.iloc[
        indices[0]
    ]["future_return_24h"]
    .values
)

print("\n")
print("=" * 110)
print("CURRENT HISTORICAL ANALOGUE")
print("=" * 110)

print(
    "Timestamp:",
    model_data.index[-1]
)

print(
    "Price:",
    round(
        latest["close"],
        4
    )
)

if "continuous_opportunity" in latest.index:

    print(
        "Opportunity:",
        round(
            latest[
                "continuous_opportunity"
            ],
            2
        )
    )

print(
    "\n50 closest historical states:"
)

print(
    "Mean 24h return:",
    round(
        analogue_returns.mean() * 100,
        3
    ),
    "%"
)

print(
    "Positive outcome rate:",
    round(
        (analogue_returns > 0).mean()
        * 100,
        2
    ),
    "%"
)

print(
    "Large-move rate:",
    round(
        (
            np.abs(
                analogue_returns
            ) >= 0.02
        ).mean()
        * 100,
        2
    ),
    "%"
)

# ------------------------------------------------------------
# 9. Save
# ------------------------------------------------------------

OUT_RESULTS = Path(
    "/content/gdrive/MyDrive/"
    "SilverAgent/results/"
    "regime_similarity_oos_v2.parquet"
)

OUT_SUMMARY = Path(
    "/content/gdrive/MyDrive/"
    "SilverAgent/results/"
    "regime_similarity_config_v2.parquet"
)

results_df.to_parquet(
    OUT_RESULTS,
    index=False
)

config_summary.to_parquet(
    OUT_SUMMARY,
    index=False
)

print("\nSaved:")
print(OUT_RESULTS)
print(OUT_SUMMARY)

print("\nSTEP 46B V2 COMPLETE.")

In [ ]:
# ============================================================
# STEP 47 — UNIFIED OPPORTUNITY INTELLIGENCE ENGINE V1
# ============================================================

import os
import numpy as np
import pandas as pd

BASE = "/content/gdrive/MyDrive/SilverAgent"
DB_DIR = os.path.join(BASE, "database")
RESULTS_DIR = os.path.join(BASE, "results")

os.makedirs(RESULTS_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. LOAD OUR EXISTING DATA
# ------------------------------------------------------------

features = pd.read_parquet(
    os.path.join(DB_DIR, "silver_h1_features_v2_horizons.parquet")
)

opp = pd.read_parquet(
    os.path.join(RESULTS_DIR, "continuous_opportunity_v2_history.parquet")
)

opp_type = pd.read_parquet(
    os.path.join(RESULTS_DIR, "opportunity_type_v2_history.parquet")
)

# Make sure timestamps are indexes
for df in [features, opp, opp_type]:
    if not isinstance(df.index, pd.DatetimeIndex):
        if "time" in df.columns:
            df["time"] = pd.to_datetime(df["time"], utc=True)
            df = df.set_index("time")

# ------------------------------------------------------------
# 2. JOIN ONLY INFORMATION THAT EXISTS AT THAT MOMENT
# ------------------------------------------------------------

common_index = features.index.intersection(
    opp.index
).intersection(
    opp_type.index
)

df = features.loc[common_index].copy()

# Add opportunity engine columns
for col in [
    "continuous_opportunity",
    "volatility_score",
    "momentum_energy_score",
    "volume_score",
    "displacement_score"
]:
    if col in opp.columns:
        df[col] = opp.loc[common_index, col]

# Add type
if "opportunity_type" in opp_type.columns:
    df["opportunity_type"] = opp_type.loc[
        common_index, "opportunity_type"
    ]

# ------------------------------------------------------------
# 3. CAUSAL MOMENTUM / SCORE CHANGE
# ------------------------------------------------------------

df["score_change_1h"] = (
    df["continuous_opportunity"]
    - df["continuous_opportunity"].shift(1)
)

df["score_change_3h"] = (
    df["continuous_opportunity"]
    - df["continuous_opportunity"].shift(3)
)

df["score_change_6h"] = (
    df["continuous_opportunity"]
    - df["continuous_opportunity"].shift(6)
)

# Momentum acceleration
df["score_acceleration"] = (
    df["score_change_1h"]
    - df["score_change_3h"] / 3
)

# ------------------------------------------------------------
# 4. OPPORTUNITY ZONES
# ------------------------------------------------------------

def opportunity_zone(score):
    if pd.isna(score):
        return "UNKNOWN"
    if score >= 80:
        return "VERY_HIGH"
    if score >= 60:
        return "HIGH"
    if score >= 40:
        return "MEDIUM"
    return "LOW"

df["opportunity_zone"] = df[
    "continuous_opportunity"
].apply(opportunity_zone)

# ------------------------------------------------------------
# 5. VOLATILITY STATE
# ------------------------------------------------------------

def volatility_state(x):
    if pd.isna(x):
        return "UNKNOWN"
    if x >= 75:
        return "HIGH"
    if x >= 45:
        return "MEDIUM"
    return "LOW"

df["volatility_state"] = df[
    "volatility_score"
].apply(volatility_state)

# ------------------------------------------------------------
# 6. ENERGY DIRECTION
# ------------------------------------------------------------

def energy_direction(row):

    s1 = row["score_change_1h"]
    s3 = row["score_change_3h"]
    s6 = row["score_change_6h"]

    if pd.isna(s1) or pd.isna(s3):
        return "UNKNOWN"

    # Strong recent rise
    if s1 > 2 and s3 > 3:
        return "BUILDING"

    # Strong recent fall
    if s1 < -2 and s3 < -3:
        return "FADING"

    # Otherwise flat/mixed
    return "STABLE"

df["energy_direction"] = df.apply(
    energy_direction,
    axis=1
)

# ------------------------------------------------------------
# 7. DIRECTION EVIDENCE
# ------------------------------------------------------------

# We deliberately DO NOT create a forced BUY/SELL model.
# Instead we count independent pieces of evidence.

direction_cols = [
    "price_vs_MA20",
    "mom3",
    "mom6",
    "mom12"
]

available_direction_cols = [
    c for c in direction_cols if c in df.columns
]

df["long_evidence"] = 0
df["short_evidence"] = 0

if "price_vs_MA20" in df.columns:
    df["long_evidence"] += (
        df["price_vs_MA20"] > 0
    ).astype(int)

    df["short_evidence"] += (
        df["price_vs_MA20"] < 0
    ).astype(int)

if "mom3" in df.columns:
    df["long_evidence"] += (
        df["mom3"] > 0
    ).astype(int)

    df["short_evidence"] += (
        df["mom3"] < 0
    ).astype(int)

if "mom6" in df.columns:
    df["long_evidence"] += (
        df["mom6"] > 0
    ).astype(int)

    df["short_evidence"] += (
        df["mom6"] < 0
    ).astype(int)

if "mom12" in df.columns:
    df["long_evidence"] += (
        df["mom12"] > 0
    ).astype(int)

    df["short_evidence"] += (
        df["mom12"] < 0
    ).astype(int)

df["direction_evidence_total"] = (
    df["long_evidence"]
    + df["short_evidence"]
)

# ------------------------------------------------------------
# 8. DIRECTION STATE
# ------------------------------------------------------------

def direction_state(row):

    long_e = row["long_evidence"]
    short_e = row["short_evidence"]

    if long_e >= 3 and long_e >= short_e + 2:
        return "LONG_BIASED"

    if short_e >= 3 and short_e >= long_e + 2:
        return "SHORT_BIASED"

    if long_e == short_e:
        return "BALANCED"

    return "MIXED"

df["direction_state"] = df.apply(
    direction_state,
    axis=1
)

# ------------------------------------------------------------
# 9. ACTIVITY CONFIRMATION
# ------------------------------------------------------------

def activity_state(x):

    if pd.isna(x):
        return "UNKNOWN"

    if x >= 70:
        return "STRONG"

    if x >= 40:
        return "NORMAL"

    return "WEAK"

df["activity_state"] = df[
    "volume_score"
].apply(activity_state)

# ------------------------------------------------------------
# 10. UNIFIED STATE
# ------------------------------------------------------------

def unified_state(row):

    opp = row["continuous_opportunity"]
    zone = row["opportunity_zone"]
    energy = row["energy_direction"]
    direction = row["direction_state"]
    activity = row["activity_state"]

    if pd.isna(opp):
        return "INSUFFICIENT_DATA"

    # Very high opportunity
    if zone == "VERY_HIGH":

        if energy == "BUILDING":
            if direction in [
                "LONG_BIASED",
                "SHORT_BIASED"
            ]:
                return "HIGH_PRIORITY_OPPORTUNITY"

            return "OPPORTUNITY_WATCH"

        if energy == "FADING":
            return "OPPORTUNITY_FADING"

        return "HIGH_OPPORTUNITY"

    # High opportunity
    if zone == "HIGH":

        if energy == "BUILDING":
            return "OPPORTUNITY"

        if energy == "FADING":
            return "WATCH"

        return "WATCH"

    # Medium
    if zone == "MEDIUM":

        if energy == "BUILDING" and activity == "STRONG":
            return "DEVELOPING"

        return "WAIT"

    return "WAIT"

df["unified_state"] = df.apply(
    unified_state,
    axis=1
)

# ------------------------------------------------------------
# 11. CONFIDENCE
# ------------------------------------------------------------

def confidence(row):

    available = 0
    total = 5

    for c in [
        "continuous_opportunity",
        "volatility_score",
        "momentum_energy_score",
        "volume_score",
        "displacement_score"
    ]:
        if c in row.index and pd.notna(row[c]):
            available += 1

    completeness = available / total

    # Base confidence from data completeness
    score = completeness * 50

    # Strong opportunity increases confidence
    opp = row["continuous_opportunity"]

    if pd.notna(opp):

        if opp >= 80:
            score += 30
        elif opp >= 60:
            score += 20
        elif opp >= 40:
            score += 10

    # Direction agreement
    if row["direction_state"] in [
        "LONG_BIASED",
        "SHORT_BIASED"
    ]:
        score += 10

    # Building energy
    if row["energy_direction"] == "BUILDING":
        score += 10

    return min(score, 100)

df["confidence"] = df.apply(
    confidence,
    axis=1
)

# ------------------------------------------------------------
# 12. FUTURE OUTCOME DATA
# ------------------------------------------------------------

# Keep historical targets only for validation.
# They are NEVER used by the live state engine.

if "future_abs_24h" in df.columns:
    df["future_abs_24h"] = pd.to_numeric(
        df["future_abs_24h"],
        errors="coerce"
    )

if "future_return_24h" in df.columns:
    df["future_return_24h"] = pd.to_numeric(
        df["future_return_24h"],
        errors="coerce"
    )

# ------------------------------------------------------------
# 13. BASIC OUTPUT
# ------------------------------------------------------------

print("=" * 110)
print("STEP 47 — UNIFIED OPPORTUNITY INTELLIGENCE ENGINE V1")
print("=" * 110)

print()
print("Rows:", len(df))
print("Start:", df.index.min())
print("End:", df.index.max())

print()
print("UNIFIED STATE DISTRIBUTION")
print(
    df["unified_state"]
    .value_counts()
    .to_string()
)

print()
print("OPPORTUNITY ZONES")
print(
    df["opportunity_zone"]
    .value_counts()
    .to_string()
)

print()
print("OPPORTUNITY TYPES")
print(
    df["opportunity_type"]
    .value_counts()
    .head(10)
    .to_string()
)

# ------------------------------------------------------------
# 14. HISTORICAL EVENT PERFORMANCE
# ------------------------------------------------------------

if "future_abs_24h" in df.columns:

    print()
    print("=" * 110)
    print("HISTORICAL 24H MOVEMENT BY UNIFIED STATE")
    print("=" * 110)

    states = []

    for state, g in df.groupby("unified_state"):

        g = g.dropna(subset=["future_abs_24h"])

        if len(g) == 0:
            continue

        states.append({
            "state": state,
            "n": len(g),
            "mean_abs_24h_pct":
                g["future_abs_24h"].mean() * 100,
            "median_abs_24h_pct":
                g["future_abs_24h"].median() * 100,
            "p90_abs_24h_pct":
                g["future_abs_24h"].quantile(.90) * 100,
            "positive_24h_pct":
                (
                    g["future_return_24h"] > 0
                ).mean() * 100
                if "future_return_24h" in g.columns
                else np.nan
        })

    state_results = pd.DataFrame(states)

    print(
        state_results
        .sort_values(
            "mean_abs_24h_pct",
            ascending=False
        )
        .to_string(index=False)
    )

else:

    state_results = pd.DataFrame()

# ------------------------------------------------------------
# 15. HIGH-QUALITY OPPORTUNITIES
# ------------------------------------------------------------

high_quality = df[
    (
        df["unified_state"].isin([
            "HIGH_PRIORITY_OPPORTUNITY",
            "OPPORTUNITY",
            "HIGH_OPPORTUNITY",
            "OPPORTUNITY_WATCH"
        ])
    )
    &
    (df["confidence"] >= 60)
].copy()

print()
print("=" * 110)
print("HIGH-QUALITY OPPORTUNITY SAMPLE")
print("=" * 110)

print("Observations:", len(high_quality))

if len(high_quality) > 0:

    print(
        high_quality[
            [
                "continuous_opportunity",
                "opportunity_zone",
                "opportunity_type",
                "energy_direction",
                "direction_state",
                "activity_state",
                "confidence",
                "unified_state"
            ]
        ]
        .tail(10)
        .to_string()
    )

# ------------------------------------------------------------
# 16. CURRENT STATE
# ------------------------------------------------------------

latest = df.iloc[-1]

print()
print("=" * 110)
print("CURRENT HISTORICAL STATE")
print("=" * 110)

print("Timestamp:", df.index[-1])

if "close" in latest.index:
    print("Price:", round(float(latest["close"]), 4))

print(
    "Opportunity:",
    round(float(latest["continuous_opportunity"]), 2)
    if pd.notna(latest["continuous_opportunity"])
    else "N/A"
)

print("Zone:", latest["opportunity_zone"])
print("Type:", latest["opportunity_type"])
print("Volatility:", latest["volatility_state"])
print("Energy:", latest["energy_direction"])
print("Activity:", latest["activity_state"])
print("Direction:", latest["direction_state"])
print("Long evidence:", int(latest["long_evidence"]))
print("Short evidence:", int(latest["short_evidence"]))
print("Confidence:", round(float(latest["confidence"]), 1))
print("FINAL STATE:", latest["unified_state"])

# ------------------------------------------------------------
# 17. SAVE
# ------------------------------------------------------------

out_path = os.path.join(
    RESULTS_DIR,
    "unified_opportunity_intelligence_v1.parquet"
)

df.to_parquet(out_path)

if len(state_results) > 0:

    state_path = os.path.join(
        RESULTS_DIR,
        "unified_opportunity_state_results_v1.parquet"
    )

    state_results.to_parquet(state_path)

print()
print("Saved:")
print(out_path)

if len(state_results) > 0:
    print(state_path)

print()
print("STEP 47 COMPLETE.")

In [ ]:
# ============================================================
# STEP 48 — UNIFIED ENGINE EVENT VALIDATION
# ============================================================

import os
import numpy as np
import pandas as pd

BASE = "/content/gdrive/MyDrive/SilverAgent"
DB_DIR = os.path.join(BASE, "database")
RESULTS_DIR = os.path.join(BASE, "results")

# ------------------------------------------------------------
# 1. LOAD UNIFIED ENGINE
# ------------------------------------------------------------

df = pd.read_parquet(
    os.path.join(
        RESULTS_DIR,
        "unified_opportunity_intelligence_v1.parquet"
    )
)

# ------------------------------------------------------------
# 2. LOAD GAP-AWARE HORIZON DATA
# ------------------------------------------------------------

horizons = pd.read_parquet(
    os.path.join(
        DB_DIR,
        "silver_h1_features_v2_horizons.parquet"
    )
)

# Make sure indexes are DatetimeIndex
for name, obj in [("df", df), ("horizons", horizons)]:

    if not isinstance(obj.index, pd.DatetimeIndex):

        if "time" in obj.columns:
            obj["time"] = pd.to_datetime(
                obj["time"],
                utc=True
            )
            obj.set_index("time", inplace=True)

# ------------------------------------------------------------
# 3. ADD FUTURE TARGETS
# ------------------------------------------------------------

target_cols = [
    "future_abs_6h",
    "future_abs_12h",
    "future_abs_24h",
    "future_return_6h",
    "future_return_12h",
    "future_return_24h"
]

available_targets = [
    c for c in target_cols
    if c in horizons.columns
]

print("Available future targets:")
print(available_targets)

for col in available_targets:

    df[col] = horizons.loc[
        df.index.intersection(horizons.index),
        col
    ]

# ------------------------------------------------------------
# 4. THRESHOLDS
# ------------------------------------------------------------

LARGE_MOVE = 0.02187
EXTREME_MOVE = 0.03642
VERY_EXTREME_MOVE = 0.04628

# ------------------------------------------------------------
# 5. EVENT SUMMARY FUNCTION
# ------------------------------------------------------------

def event_summary(group):

    result = {
        "n": len(group)
    }

    if "future_abs_6h" in group:
        x = group["future_abs_6h"].dropna()

        if len(x):
            result["mean_6h_pct"] = x.mean() * 100
            result["median_6h_pct"] = x.median() * 100

    if "future_abs_12h" in group:
        x = group["future_abs_12h"].dropna()

        if len(x):
            result["mean_12h_pct"] = x.mean() * 100
            result["median_12h_pct"] = x.median() * 100

    if "future_abs_24h" in group:
        x = group["future_abs_24h"].dropna()

        if len(x):

            result["mean_24h_pct"] = x.mean() * 100
            result["median_24h_pct"] = x.median() * 100

            result["p90_24h_pct"] = (
                x.quantile(.90) * 100
            )

            result["p95_24h_pct"] = (
                x.quantile(.95) * 100
            )

            result["large_move_pct"] = (
                x >= LARGE_MOVE
            ).mean() * 100

            result["extreme_move_pct"] = (
                x >= EXTREME_MOVE
            ).mean() * 100

            result["very_extreme_move_pct"] = (
                x >= VERY_EXTREME_MOVE
            ).mean() * 100

    if "future_return_24h" in group:

        x = group["future_return_24h"].dropna()

        if len(x):

            result["signed_mean_24h_pct"] = (
                x.mean() * 100
            )

            result["signed_median_24h_pct"] = (
                x.median() * 100
            )

            result["up_rate_pct"] = (
                x > 0
            ).mean() * 100

            result["down_rate_pct"] = (
                x < 0
            ).mean() * 100

            result["mean_up_move_pct"] = (
                x[x > 0].mean() * 100
                if (x > 0).any()
                else np.nan
            )

            result["mean_down_move_pct"] = (
                x[x < 0].mean() * 100
                if (x < 0).any()
                else np.nan
            )

    return pd.Series(result)

# ------------------------------------------------------------
# 6. STATE PERFORMANCE
# ------------------------------------------------------------

state_results = (
    df
    .groupby("unified_state")
    .apply(event_summary)
    .reset_index()
)

print()
print("=" * 120)
print("STEP 48 — UNIFIED STATE EVENT VALIDATION")
print("=" * 120)

print(
    state_results
    .sort_values(
        "mean_24h_pct",
        ascending=False
    )
    .to_string(index=False)
)

# ------------------------------------------------------------
# 7. OPPORTUNITY ZONE PERFORMANCE
# ------------------------------------------------------------

zone_results = (
    df
    .groupby("opportunity_zone")
    .apply(event_summary)
    .reset_index()
)

print()
print("=" * 120)
print("OPPORTUNITY ZONE PERFORMANCE")
print("=" * 120)

print(
    zone_results
    .sort_values(
        "mean_24h_pct",
        ascending=False
    )
    .to_string(index=False)
)

# ------------------------------------------------------------
# 8. DIRECTION STATE PERFORMANCE
# ------------------------------------------------------------

direction_results = (
    df
    .groupby("direction_state")
    .apply(event_summary)
    .reset_index()
)

print()
print("=" * 120)
print("DIRECTION STATE PERFORMANCE")
print("=" * 120)

print(
    direction_results
    .sort_values(
        "signed_mean_24h_pct",
        ascending=False
    )
    .to_string(index=False)
)

# ------------------------------------------------------------
# 9. OPPORTUNITY TYPE PERFORMANCE
# ------------------------------------------------------------

type_results = (
    df
    .groupby("opportunity_type")
    .apply(event_summary)
    .reset_index()
)

print()
print("=" * 120)
print("OPPORTUNITY TYPE PERFORMANCE")
print("=" * 120)

print(
    type_results
    .sort_values(
        "mean_24h_pct",
        ascending=False
    )
    .to_string(index=False)
)

# ------------------------------------------------------------
# 10. HIGH-CONFIDENCE EVENTS
# ------------------------------------------------------------

high_conf = df[
    df["confidence"] >= 80
].copy()

print()
print("=" * 120)
print("HIGH-CONFIDENCE EVENTS")
print("=" * 120)

print("Observations:", len(high_conf))

if len(high_conf):

    hc = event_summary(high_conf)

    print(
        hc.to_frame("value")
        .to_string()
    )

# ------------------------------------------------------------
# 11. VERY HIGH OPPORTUNITY + DIRECTION
# ------------------------------------------------------------

special = df[
    (df["opportunity_zone"] == "VERY_HIGH")
    &
    (df["confidence"] >= 80)
].copy()

print()
print("=" * 120)
print("VERY HIGH OPPORTUNITY + HIGH CONFIDENCE")
print("=" * 120)

print("Observations:", len(special))

if len(special):

    print(
        event_summary(special)
        .to_frame("value")
        .to_string()
    )

# ------------------------------------------------------------
# 12. SAVE RESULTS
# ------------------------------------------------------------

state_path = os.path.join(
    RESULTS_DIR,
    "unified_state_event_validation_v1.parquet"
)

zone_path = os.path.join(
    RESULTS_DIR,
    "unified_zone_event_validation_v1.parquet"
)

direction_path = os.path.join(
    RESULTS_DIR,
    "unified_direction_event_validation_v1.parquet"
)

type_path = os.path.join(
    RESULTS_DIR,
    "unified_type_event_validation_v1.parquet"
)

state_results.to_parquet(state_path)
zone_results.to_parquet(zone_path)
direction_results.to_parquet(direction_path)
type_results.to_parquet(type_path)

print()
print("Saved:")
print(state_path)
print(zone_path)
print(direction_path)
print(type_path)

print()
print("STEP 48 COMPLETE.")

In [ ]:
# ============================================================
# STEP 49 — CAUSAL DIRECTION VALIDATION V2
# Fixed UTC / DatetimeIndex handling
# ============================================================

import os
import numpy as np
import pandas as pd

BASE = "/content/gdrive/MyDrive/SilverAgent"
DB_DIR = os.path.join(BASE, "database")
RESULTS_DIR = os.path.join(BASE, "results")

# ------------------------------------------------------------
# 1. LOAD UNIFIED ENGINE
# ------------------------------------------------------------

df = pd.read_parquet(
    os.path.join(
        RESULTS_DIR,
        "unified_opportunity_intelligence_v1.parquet"
    )
)

df.index = pd.to_datetime(
    df.index,
    utc=True
)

df = df.sort_index()

# ------------------------------------------------------------
# 2. LOAD RAW SILVER CANDLES
# ------------------------------------------------------------

candles = pd.read_parquet(
    os.path.join(
        DB_DIR,
        "silver_h1.parquet"
    )
)

# The database stores time as the DatetimeIndex
candles.index = pd.to_datetime(
    candles.index,
    utc=True
)

candles = candles.sort_index()

close = candles["close"].astype(float)

print("=" * 120)
print("STEP 49 — CAUSAL DIRECTION VALIDATION V2")
print("=" * 120)

print()
print("Unified rows:", len(df))
print("Candle rows:", len(candles))
print("Candle start:", candles.index.min())
print("Candle end:", candles.index.max())
print("Candle timezone:", candles.index.tz)

# ------------------------------------------------------------
# 3. GAP-AWARE FUTURE PRICE FUNCTION
# ------------------------------------------------------------

def get_future_price(
    prices,
    hours,
    tolerance_minutes=90
):

    source_times = pd.DatetimeIndex(
        prices.index
    ).tz_convert("UTC")

    target_times = (
        source_times
        + pd.Timedelta(hours=hours)
    )

    left = pd.DataFrame({
        "source_time": source_times,
        "target_time": target_times
    })

    right = pd.DataFrame({
        "target_time": source_times,
        "future_price": prices.to_numpy()
    })

    left = left.sort_values(
        "target_time"
    )

    right = right.sort_values(
        "target_time"
    )

    merged = pd.merge_asof(
        left,
        right,
        on="target_time",
        direction="nearest",
        tolerance=pd.Timedelta(
            minutes=tolerance_minutes
        )
    )

    # IMPORTANT:
    # Explicitly restore UTC-aware source timestamps.
    source_index = pd.DatetimeIndex(
        merged["source_time"]
    ).tz_convert("UTC")

    result = pd.Series(
        merged["future_price"].to_numpy(),
        index=source_index,
        dtype=float,
        name=f"future_price_{hours}h"
    )

    return result


# ------------------------------------------------------------
# 4. BUILD FUTURE RETURNS
# ------------------------------------------------------------

for h in [6, 12, 24]:

    future_price = get_future_price(
        close,
        hours=h,
        tolerance_minutes=90
    )

    aligned_future = future_price.reindex(
        df.index
    )

    current_price = close.reindex(
        df.index
    )

    df[f"future_return_{h}h"] = (
        aligned_future / current_price
    ) - 1

    print(
        f"future_return_{h}h valid:",
        int(
            df[f"future_return_{h}h"]
            .notna()
            .sum()
        )
    )

# ------------------------------------------------------------
# 5. DIRECTION SUMMARY
# ------------------------------------------------------------

def direction_summary(group):

    result = {
        "n": len(group)
    }

    for h in [6, 12, 24]:

        col = f"future_return_{h}h"

        x = group[col].dropna()

        if len(x) == 0:
            continue

        result[f"mean_{h}h_pct"] = (
            x.mean() * 100
        )

        result[f"median_{h}h_pct"] = (
            x.median() * 100
        )

        result[f"win_{h}h_pct"] = (
            (x > 0).mean() * 100
        )

        result[f"mean_up_{h}h_pct"] = (
            x[x > 0].mean() * 100
            if (x > 0).any()
            else np.nan
        )

        result[f"mean_down_{h}h_pct"] = (
            x[x < 0].mean() * 100
            if (x < 0).any()
            else np.nan
        )

        result[f"worst_{h}h_pct"] = (
            x.min() * 100
        )

        result[f"best_{h}h_pct"] = (
            x.max() * 100
        )

    return pd.Series(result)

# ------------------------------------------------------------
# 6. DIRECTION STATE
# ------------------------------------------------------------

direction_results = (
    df
    .groupby("direction_state")
    .apply(
        direction_summary,
        include_groups=False
    )
    .reset_index()
)

print()
print("=" * 120)
print("DIRECTION STATE PERFORMANCE")
print("=" * 120)

print(
    direction_results
    .sort_values(
        "mean_24h_pct",
        ascending=False
    )
    .to_string(index=False)
)

# ------------------------------------------------------------
# 7. OPPORTUNITY + DIRECTION
# ------------------------------------------------------------

df["opp_direction_state"] = (
    df["opportunity_zone"].astype(str)
    + "_"
    + df["direction_state"].astype(str)
)

opp_direction_results = (
    df
    .groupby("opp_direction_state")
    .apply(
        direction_summary,
        include_groups=False
    )
    .reset_index()
)

print()
print("=" * 120)
print("OPPORTUNITY + DIRECTION")
print("=" * 120)

print(
    opp_direction_results
    .sort_values(
        "mean_24h_pct",
        ascending=False
    )
    .to_string(index=False)
)

# ------------------------------------------------------------
# 8. HIGH + VERY HIGH OPPORTUNITY
# ------------------------------------------------------------

high = df[
    df["opportunity_zone"].isin([
        "HIGH",
        "VERY_HIGH"
    ])
].copy()

high_direction_results = (
    high
    .groupby("direction_state")
    .apply(
        direction_summary,
        include_groups=False
    )
    .reset_index()
)

print()
print("=" * 120)
print("HIGH + VERY HIGH OPPORTUNITY — DIRECTION")
print("=" * 120)

print(
    high_direction_results
    .sort_values(
        "mean_24h_pct",
        ascending=False
    )
    .to_string(index=False)
)

# ------------------------------------------------------------
# 9. VERY HIGH ONLY
# ------------------------------------------------------------

very_high = df[
    df["opportunity_zone"] == "VERY_HIGH"
].copy()

very_high_results = (
    very_high
    .groupby("direction_state")
    .apply(
        direction_summary,
        include_groups=False
    )
    .reset_index()
)

print()
print("=" * 120)
print("VERY HIGH OPPORTUNITY — DIRECTION")
print("=" * 120)

print(
    very_high_results
    .sort_values(
        "mean_24h_pct",
        ascending=False
    )
    .to_string(index=False)
)

# ------------------------------------------------------------
# 10. OPPORTUNITY TYPE + DIRECTION
# ------------------------------------------------------------

type_direction = (
    high
    .groupby([
        "opportunity_type",
        "direction_state"
    ])
    .apply(
        direction_summary,
        include_groups=False
    )
    .reset_index()
)

print()
print("=" * 120)
print("OPPORTUNITY TYPE + DIRECTION")
print("=" * 120)

print(
    type_direction
    .sort_values(
        ["mean_24h_pct", "n"],
        ascending=[False, False]
    )
    .head(30)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 11. DIRECTION EVIDENCE STRENGTH
# ------------------------------------------------------------

df["direction_edge"] = (
    df["long_evidence"]
    - df["short_evidence"]
)

edge_bins = pd.cut(
    df["direction_edge"],
    bins=[
        -5,
        -3,
        -2,
        -1,
        0,
        1,
        2,
        3,
        5
    ],
    include_lowest=True
)

edge_results = (
    df
    .groupby(
        edge_bins,
        observed=False
    )
    .apply(
        direction_summary,
        include_groups=False
    )
    .reset_index()
)

print()
print("=" * 120)
print("DIRECTION EVIDENCE STRENGTH")
print("=" * 120)

print(
    edge_results.to_string(index=False)
)

# ------------------------------------------------------------
# 12. SAVE
# ------------------------------------------------------------

save_map = {
    "direction":
        "unified_direction_validation_v2.parquet",

    "opp_direction":
        "unified_opportunity_direction_v2.parquet",

    "high_direction":
        "unified_high_opportunity_direction_v2.parquet",

    "very_high":
        "unified_very_high_direction_v2.parquet",

    "type_direction":
        "unified_type_direction_v2.parquet",

    "edge":
        "unified_direction_edge_v2.parquet"
}

direction_results.to_parquet(
    os.path.join(
        RESULTS_DIR,
        save_map["direction"]
    )
)

opp_direction_results.to_parquet(
    os.path.join(
        RESULTS_DIR,
        save_map["opp_direction"]
    )
)

high_direction_results.to_parquet(
    os.path.join(
        RESULTS_DIR,
        save_map["high_direction"]
    )
)

very_high_results.to_parquet(
    os.path.join(
        RESULTS_DIR,
        save_map["very_high"]
    )
)

type_direction.to_parquet(
    os.path.join(
        RESULTS_DIR,
        save_map["type_direction"]
    )
)

edge_results.to_parquet(
    os.path.join(
        RESULTS_DIR,
        save_map["edge"]
    )
)

print()
print("Saved:")

for filename in save_map.values():
    print(
        os.path.join(
            RESULTS_DIR,
            filename
        )
    )

print()
print("STEP 49 V2 COMPLETE.")

In [ ]:
# ============================================================
# STEP 49B — DIRECTION EVIDENCE DIAGNOSTIC
# ============================================================

# df should still exist from Step 49 V2

df["direction_edge"] = (
    df["long_evidence"]
    - df["short_evidence"]
)

print("=" * 100)
print("DIRECTION EVIDENCE STRENGTH")
print("=" * 100)

for edge in sorted(
    df["direction_edge"].dropna().unique()
):

    g = df[
        df["direction_edge"] == edge
    ].copy()

    r = g["future_return_24h"].dropna()

    if len(r) == 0:
        continue

    print(
        f"Evidence edge {int(edge):+d} | "
        f"N={len(r):5d} | "
        f"Mean={r.mean()*100:+.3f}% | "
        f"Median={r.median()*100:+.3f}% | "
        f"Win={(r>0).mean()*100:.1f}%"
    )

print()
print("STEP 49B COMPLETE.")

In [1]:
# ============================================================
# STEP 50 — OPPORTUNITY QUALITY ENGINE V1
# ============================================================

import os
import numpy as np
import pandas as pd

BASE = "/content/gdrive/MyDrive/SilverAgent"
DB_DIR = os.path.join(BASE, "database")
RESULTS_DIR = os.path.join(BASE, "results")

# ------------------------------------------------------------
# 1. LOAD EXISTING DATA
# ------------------------------------------------------------

df = pd.read_parquet(
    os.path.join(
        RESULTS_DIR,
        "unified_opportunity_intelligence_v1.parquet"
    )
)

df.index = pd.to_datetime(df.index, utc=True)
df = df.sort_index()

horizons = pd.read_parquet(
    os.path.join(
        DB_DIR,
        "silver_h1_features_v2_horizons.parquet"
    )
)

horizons.index = pd.to_datetime(
    horizons.index,
    utc=True
)

horizons = horizons.sort_index()

# ------------------------------------------------------------
# 2. ADD GAP-AWARE 24H ABSOLUTE MOVE
# ------------------------------------------------------------

if "future_abs_24h" in horizons.columns:

    df["future_abs_24h"] = horizons[
        "future_abs_24h"
    ].reindex(df.index)

else:

    raise RuntimeError(
        "future_abs_24h not found in horizon database."
    )

# ------------------------------------------------------------
# 3. REQUIRED ENGINE FEATURES
# ------------------------------------------------------------

required = [
    "continuous_opportunity",
    "volatility_score",
    "momentum_energy_score",
    "volume_score",
    "displacement_score",
    "score_change_1h",
    "score_change_3h",
    "score_change_6h",
    "confidence"
]

missing = [
    c for c in required
    if c not in df.columns
]

if missing:

    raise RuntimeError(
        f"Missing required columns: {missing}"
    )

# ------------------------------------------------------------
# 4. BUILD QUALITY COMPONENTS
#
# IMPORTANT:
# These are NORMALISED using historical information only.
# Current observation is excluded via shift(1).
# ------------------------------------------------------------

components = [
    "continuous_opportunity",
    "volatility_score",
    "momentum_energy_score",
    "volume_score",
    "displacement_score"
]

for col in components:

    historical_rank = (
        df[col]
        .shift(1)
        .expanding(min_periods=500)
        .rank(pct=True)
    )

    df[f"{col}_rank"] = (
        historical_rank * 100
    )

# ------------------------------------------------------------
# 5. ENERGY BUILD SCORE
# ------------------------------------------------------------

# Positive score changes indicate energy increasing.
# We don't assume direction — only acceleration of opportunity.

df["energy_build_raw"] = (
    0.50 * df["score_change_1h"].clip(-20, 20)
    +
    0.30 * df["score_change_3h"].clip(-20, 20)
    +
    0.20 * df["score_change_6h"].clip(-20, 20)
)

df["energy_build_rank"] = (
    df["energy_build_raw"]
    .shift(1)
    .expanding(min_periods=500)
    .rank(pct=True)
    * 100
)

# ------------------------------------------------------------
# 6. OPPORTUNITY PERSISTENCE
# ------------------------------------------------------------

# Count consecutive hours where opportunity is >= 60.
# This is causal: only uses current/past information.

high_flag = (
    df["continuous_opportunity"] >= 60
).astype(int)

episode_id = (
    high_flag
    .ne(high_flag.shift())
    .cumsum()
)

df["opportunity_persistence"] = (
    high_flag
    .groupby(episode_id)
    .cumsum()
)

df["persistence_rank"] = (
    df["opportunity_persistence"]
    .shift(1)
    .expanding(min_periods=500)
    .rank(pct=True)
    * 100
)

# ------------------------------------------------------------
# 7. QUALITY SCORE
# ------------------------------------------------------------

# We deliberately give the strongest weight to the things
# already shown to predict future movement magnitude.

df["quality_score"] = (
    0.30 * df["continuous_opportunity_rank"]
    +
    0.20 * df["volatility_score_rank"]
    +
    0.20 * df["momentum_energy_score_rank"]
    +
    0.10 * df["displacement_score_rank"]
    +
    0.05 * df["volume_score_rank"]
    +
    0.10 * df["energy_build_rank"]
    +
    0.05 * df["persistence_rank"]
)

# ------------------------------------------------------------
# 8. QUALITY ZONES
# ------------------------------------------------------------

def quality_zone(x):

    if pd.isna(x):
        return "INSUFFICIENT_DATA"

    if x >= 90:
        return "EXCEPTIONAL"

    if x >= 75:
        return "HIGH"

    if x >= 50:
        return "MEDIUM"

    if x >= 25:
        return "LOW"

    return "VERY_LOW"

df["quality_zone"] = (
    df["quality_score"]
    .apply(quality_zone)
)

# ------------------------------------------------------------
# 9. EVENT THRESHOLDS
# ------------------------------------------------------------

LARGE = 0.02187
EXTREME = 0.03642
VERY_EXTREME = 0.04628

# ------------------------------------------------------------
# 10. QUALITY PERFORMANCE
# ------------------------------------------------------------

def summarize(group):

    x = group[
        "future_abs_24h"
    ].dropna()

    if len(x) == 0:
        return pd.Series({
            "n": 0
        })

    return pd.Series({

        "n": len(x),

        "mean_24h_pct":
            x.mean() * 100,

        "median_24h_pct":
            x.median() * 100,

        "p90_24h_pct":
            x.quantile(.90) * 100,

        "p95_24h_pct":
            x.quantile(.95) * 100,

        "large_move_pct":
            (x >= LARGE).mean() * 100,

        "extreme_move_pct":
            (x >= EXTREME).mean() * 100,

        "very_extreme_move_pct":
            (x >= VERY_EXTREME).mean() * 100
    })

quality_results = (
    df
    .groupby("quality_zone")
    .apply(
        summarize,
        include_groups=False
    )
    .reset_index()
)

print("=" * 120)
print("STEP 50 — OPPORTUNITY QUALITY ENGINE V1")
print("=" * 120)

print()
print("Rows:", len(df))
print("Start:", df.index.min())
print("End:", df.index.max())

print()
print("=" * 120)
print("QUALITY ZONE PERFORMANCE")
print("=" * 120)

print(
    quality_results
    .sort_values(
        "mean_24h_pct",
        ascending=False
    )
    .to_string(index=False)
)

# ------------------------------------------------------------
# 11. SCORE DECILES
# ------------------------------------------------------------

usable = df[
    df["quality_score"].notna()
    &
    df["future_abs_24h"].notna()
].copy()

usable["quality_decile"] = pd.qcut(
    usable["quality_score"],
    10,
    labels=False,
    duplicates="drop"
) + 1

decile_results = (
    usable
    .groupby("quality_decile")
    .apply(
        summarize,
        include_groups=False
    )
    .reset_index()
)

print()
print("=" * 120)
print("QUALITY DECILE PERFORMANCE")
print("=" * 120)

print(
    decile_results
    .sort_values(
        "quality_decile"
    )
    .to_string(index=False)
)

# ------------------------------------------------------------
# 12. TOP OPPORTUNITIES
# ------------------------------------------------------------

for pct in [20, 10, 5, 2]:

    cutoff = usable[
        "quality_score"
    ].quantile(
        1 - pct / 100
    )

    selected = usable[
        usable["quality_score"] >= cutoff
    ]

    result = summarize(selected)

    print()
    print(
        f"TOP {pct}% QUALITY "
        f"(cutoff {cutoff:.2f})"
    )

    print(
        result.to_string()
    )

# ------------------------------------------------------------
# 13. CURRENT STATE
# ------------------------------------------------------------

latest = df.iloc[-1]

print()
print("=" * 120)
print("CURRENT QUALITY STATE")
print("=" * 120)

print("Timestamp:", df.index[-1])

if "close" in latest.index:
    print(
        "Price:",
        round(float(latest["close"]), 4)
    )

print(
    "Opportunity:",
    round(
        float(latest["continuous_opportunity"]),
        2
    )
)

print(
    "Quality score:",
    round(
        float(latest["quality_score"]),
        2
    )
)

print(
    "Quality zone:",
    latest["quality_zone"]
)

print(
    "Volatility:",
    round(
        float(latest["volatility_score"]),
        2
    )
)

print(
    "Momentum energy:",
    round(
        float(latest["momentum_energy_score"]),
        2
    )
    if pd.notna(
        latest["momentum_energy_score"]
    )
    else "N/A"
)

print(
    "Volume:",
    round(
        float(latest["volume_score"]),
        2
    )
)

print(
    "Displacement:",
    round(
        float(latest["displacement_score"]),
        2
    )
)

print(
    "Energy build:",
    round(
        float(latest["energy_build_raw"]),
        2
    )
)

print(
    "Persistence:",
    int(
        latest["opportunity_persistence"]
    )
)

print(
    "Confidence:",
    round(
        float(latest["confidence"]),
        1
    )
)

# ------------------------------------------------------------
# 14. SAVE
# ------------------------------------------------------------

out_path = os.path.join(
    RESULTS_DIR,
    "opportunity_quality_v1_history.parquet"
)

summary_path = os.path.join(
    RESULTS_DIR,
    "opportunity_quality_v1_summary.parquet"
)

df.to_parquet(out_path)
quality_results.to_parquet(summary_path)

print()
print("Saved:")
print(out_path)
print(summary_path)

print()
print("STEP 50 COMPLETE.")

FileNotFoundError: [Errno 2] No such file or directory: '/content/gdrive/MyDrive/SilverAgent/results/unified_opportunity_intelligence_v1.parquet'

In [2]:
# ============================================================
# STEP 51 — STRICT CAUSAL / OOS QUALITY VALIDATION V1
# ============================================================

import os
import numpy as np
import pandas as pd

BASE = "/content/gdrive/MyDrive/SilverAgent"
RESULTS = os.path.join(BASE, "results")

QUALITY_FILE = os.path.join(
    RESULTS, "opportunity_quality_v1_history.parquet"
)

print("=" * 80)
print("STEP 51 — STRICT CAUSAL / OOS QUALITY VALIDATION V1")
print("=" * 80)

# ------------------------------------------------------------
# 1. LOAD QUALITY DATA
# ------------------------------------------------------------

df = pd.read_parquet(QUALITY_FILE)
df = df.sort_index()

print(f"\nLoaded rows: {len(df):,}")
print(f"Start: {df.index.min()}")
print(f"End:   {df.index.max()}")

# ------------------------------------------------------------
# 2. REQUIRED TARGET
# ------------------------------------------------------------

if "future_abs_24h" not in df.columns:
    raise RuntimeError(
        "future_abs_24h is missing from opportunity_quality_v1_history.parquet"
    )

# Convert target to percentage
df["future_abs_24h_pct"] = df["future_abs_24h"] * 100.0

# Remove rows without a valid future target
df = df.dropna(subset=["future_abs_24h_pct"]).copy()

# ------------------------------------------------------------
# 3. QUALITY SCORE
# ------------------------------------------------------------

if "quality_score" not in df.columns:
    raise RuntimeError("quality_score column is missing.")

# Only rows with a valid causal quality score
df = df.dropna(subset=["quality_score"]).copy()

print(f"Usable rows: {len(df):,}")

# ------------------------------------------------------------
# 4. WALK-FORWARD SETTINGS
# ------------------------------------------------------------

TRAIN_SIZE = 4000
TEST_SIZE = 1000

# 24-hour purge.
# We exclude the final 24 observations from training before
# beginning the test period.
PURGE = 24

STEP = 1000

results = []

# ------------------------------------------------------------
# 5. WALK FORWARD
# ------------------------------------------------------------

start = TRAIN_SIZE + PURGE
window_id = 0

while start + TEST_SIZE <= len(df):

    train_end = start - PURGE
    train_start = max(0, train_end - TRAIN_SIZE)

    test_start = start
    test_end = start + TEST_SIZE

    train = df.iloc[train_start:train_end]
    test = df.iloc[test_start:test_end]

    # --------------------------------------------------------
    # CAUSAL THRESHOLDS
    # --------------------------------------------------------
    # Thresholds are determined ONLY from the training period.

    q75 = train["quality_score"].quantile(0.75)
    q90 = train["quality_score"].quantile(0.90)
    q95 = train["quality_score"].quantile(0.95)

    # --------------------------------------------------------
    # TEST BASELINE
    # --------------------------------------------------------

    baseline_mean = test["future_abs_24h_pct"].mean()
    baseline_median = test["future_abs_24h_pct"].median()

    # Use the 75th percentile of the training target
    # to define a "large move".
    large_threshold = train["future_abs_24h_pct"].quantile(0.75)

    baseline_large = (
        test["future_abs_24h_pct"] >= large_threshold
    ).mean() * 100

    # --------------------------------------------------------
    # TEST EACH QUALITY THRESHOLD
    # --------------------------------------------------------

    for label, cutoff in [
        ("TOP25", q75),
        ("TOP10", q90),
        ("TOP5", q95),
    ]:

        selected = test[
            test["quality_score"] >= cutoff
        ]

        if len(selected) == 0:
            continue

        mean_move = selected["future_abs_24h_pct"].mean()
        median_move = selected["future_abs_24h_pct"].median()

        large_move = (
            selected["future_abs_24h_pct"] >= large_threshold
        ).mean() * 100

        p90 = selected["future_abs_24h_pct"].quantile(0.90)
        p95 = selected["future_abs_24h_pct"].quantile(0.95)

        results.append({
            "window": window_id,
            "threshold": label,

            "train_start": train.index.min(),
            "train_end": train.index.max(),

            "test_start": test.index.min(),
            "test_end": test.index.max(),

            "train_rows": len(train),
            "test_rows": len(test),

            "cutoff": cutoff,

            "baseline_mean_24h_pct": baseline_mean,
            "selected_mean_24h_pct": mean_move,
            "premium_pp": mean_move - baseline_mean,

            "baseline_median_24h_pct": baseline_median,
            "selected_median_24h_pct": median_move,

            "baseline_large_move_pct": baseline_large,
            "selected_large_move_pct": large_move,
            "large_move_premium_pp": large_move - baseline_large,

            "selected_p90_24h_pct": p90,
            "selected_p95_24h_pct": p95,

            "selected_n": len(selected),
        })

    print(
        f"Window {window_id:02d} | "
        f"Test {test.index.min()} → {test.index.max()} | "
        f"Baseline {baseline_mean:.3f}%"
    )

    window_id += 1
    start += STEP

# ------------------------------------------------------------
# 6. RESULTS DATAFRAME
# ------------------------------------------------------------

wf = pd.DataFrame(results)

if wf.empty:
    raise RuntimeError("No walk-forward results were generated.")

# ------------------------------------------------------------
# 7. SUMMARY
# ------------------------------------------------------------

summary_rows = []

for threshold in ["TOP25", "TOP10", "TOP5"]:

    x = wf[wf["threshold"] == threshold].copy()

    if x.empty:
        continue

    weighted_selected = np.average(
        x["selected_mean_24h_pct"],
        weights=x["selected_n"]
    )

    weighted_baseline = np.average(
        x["baseline_mean_24h_pct"],
        weights=x["selected_n"]
    )

    weighted_large = np.average(
        x["selected_large_move_pct"],
        weights=x["selected_n"]
    )

    weighted_baseline_large = np.average(
        x["baseline_large_move_pct"],
        weights=x["selected_n"]
    )

    positive_windows = (
        x["premium_pp"] > 0
    ).sum()

    large_positive_windows = (
        x["large_move_premium_pp"] > 0
    ).sum()

    summary_rows.append({
        "threshold": threshold,
        "windows": len(x),
        "signals": int(x["selected_n"].sum()),

        "baseline_mean_24h_pct": weighted_baseline,
        "selected_mean_24h_pct": weighted_selected,
        "premium_pp": weighted_selected - weighted_baseline,

        "baseline_large_move_pct": weighted_baseline_large,
        "selected_large_move_pct": weighted_large,
        "large_move_premium_pp":
            weighted_large - weighted_baseline_large,

        "positive_windows": positive_windows,
        "positive_window_pct":
            positive_windows / len(x) * 100,

        "large_positive_windows":
            large_positive_windows,
        "large_positive_window_pct":
            large_positive_windows / len(x) * 100,

        "worst_window_premium_pp":
            x["premium_pp"].min(),

        "best_window_premium_pp":
            x["premium_pp"].max(),
    })

summary = pd.DataFrame(summary_rows)

# ------------------------------------------------------------
# 8. PRINT RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("STRICT OOS SUMMARY")
print("=" * 80)

print(
    summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)

print("\n" + "=" * 80)
print("WINDOW-BY-WINDOW RESULTS")
print("=" * 80)

print(
    wf[
        [
            "window",
            "threshold",
            "selected_n",
            "baseline_mean_24h_pct",
            "selected_mean_24h_pct",
            "premium_pp",
            "baseline_large_move_pct",
            "selected_large_move_pct",
            "large_move_premium_pp",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)

# ------------------------------------------------------------
# 9. SAVE
# ------------------------------------------------------------

wf_path = os.path.join(
    RESULTS,
    "opportunity_quality_v1_strict_oos.parquet"
)

summary_path = os.path.join(
    RESULTS,
    "opportunity_quality_v1_strict_oos_summary.parquet"
)

wf.to_parquet(wf_path)
summary.to_parquet(summary_path)

print("\nSaved:")
print(wf_path)
print(summary_path)

# ------------------------------------------------------------
# 10. VERDICT
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 51 COMPLETE")
print("=" * 80)

best = summary.sort_values(
    "premium_pp",
    ascending=False
).iloc[0]

print(
    f"\nBest threshold: {best['threshold']}"
)

print(
    f"OOS premium: "
    f"{best['premium_pp']:.3f} percentage points"
)

print(
    f"Large-move premium: "
    f"{best['large_move_premium_pp']:.3f} percentage points"
)

print(
    f"Positive windows: "
    f"{int(best['positive_windows'])}/"
    f"{int(best['windows'])}"
)

print(
    "\nIMPORTANT: This is opportunity detection validation, "
    "NOT a trading strategy."
)

STEP 51 — STRICT CAUSAL / OOS QUALITY VALIDATION V1


FileNotFoundError: [Errno 2] No such file or directory: '/content/gdrive/MyDrive/SilverAgent/results/opportunity_quality_v1_history.parquet'

In [3]:
# ============================================================
# STEP 52 — LIVE CAUSAL OPPORTUNITY SCANNER V1
# ============================================================

import os
import numpy as np
import pandas as pd

BASE = "/content/gdrive/MyDrive/SilverAgent"
RESULTS = os.path.join(BASE, "results")

QUALITY_FILE = os.path.join(
    RESULTS,
    "opportunity_quality_v1_history.parquet"
)

print("=" * 80)
print("STEP 52 — LIVE CAUSAL OPPORTUNITY SCANNER V1")
print("=" * 80)

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

df = pd.read_parquet(QUALITY_FILE)

if not isinstance(df.index, pd.DatetimeIndex):
    df.index = pd.to_datetime(df.index, utc=True)

df = df.sort_index()

print(f"\nRows: {len(df):,}")
print(f"Start: {df.index.min()}")
print(f"End:   {df.index.max()}")

# ------------------------------------------------------------
# 2. IDENTIFY CURRENT STATE
# ------------------------------------------------------------

current = df.iloc[-1]

timestamp = current.name

print("\n" + "=" * 80)
print("CURRENT MARKET STATE")
print("=" * 80)

# ------------------------------------------------------------
# 3. HELPER
# ------------------------------------------------------------

def get_value(row, column):
    if column not in row.index:
        return np.nan
    return row[column]

# ------------------------------------------------------------
# 4. CORE VALUES
# ------------------------------------------------------------

price = get_value(current, "price")

opportunity = get_value(
    current,
    "continuous_opportunity"
)

quality = get_value(
    current,
    "quality_score"
)

volatility = get_value(
    current,
    "volatility_score"
)

momentum_energy = get_value(
    current,
    "momentum_energy_score"
)

volume = get_value(
    current,
    "volume_score"
)

displacement = get_value(
    current,
    "displacement_score"
)

energy_build = get_value(
    current,
    "energy_build_raw"
)

persistence = get_value(
    current,
    "opportunity_persistence"
)

confidence = get_value(
    current,
    "confidence"
)

# ------------------------------------------------------------
# 5. CAUSAL QUALITY RANK
# ------------------------------------------------------------
# IMPORTANT:
# We compare the current score ONLY against previous scores.
# The current observation is excluded from its own percentile.

historical_quality = df["quality_score"].iloc[:-1].dropna()

if len(historical_quality) >= 500 and pd.notna(quality):

    quality_percentile = (
        historical_quality <= quality
    ).mean() * 100

else:

    quality_percentile = np.nan

# ------------------------------------------------------------
# 6. OPPORTUNITY RANK
# ------------------------------------------------------------

historical_opportunity = (
    df["continuous_opportunity"]
    .iloc[:-1]
    .dropna()
)

if len(historical_opportunity) >= 500 and pd.notna(opportunity):

    opportunity_percentile = (
        historical_opportunity <= opportunity
    ).mean() * 100

else:

    opportunity_percentile = np.nan

# ------------------------------------------------------------
# 7. QUALITY ZONE
# ------------------------------------------------------------

if pd.isna(quality):

    quality_zone = "INSUFFICIENT_DATA"

elif quality >= 90:

    quality_zone = "EXCEPTIONAL"

elif quality >= 75:

    quality_zone = "HIGH"

elif quality >= 50:

    quality_zone = "MEDIUM"

elif quality >= 25:

    quality_zone = "LOW"

else:

    quality_zone = "VERY_LOW"

# ------------------------------------------------------------
# 8. OPPORTUNITY ZONE
# ------------------------------------------------------------

if pd.isna(opportunity):

    opportunity_zone = "INSUFFICIENT_DATA"

elif opportunity >= 80:

    opportunity_zone = "VERY_HIGH"

elif opportunity >= 60:

    opportunity_zone = "HIGH"

elif opportunity >= 40:

    opportunity_zone = "MEDIUM"

elif opportunity >= 20:

    opportunity_zone = "LOW"

else:

    opportunity_zone = "VERY_LOW"

# ------------------------------------------------------------
# 9. ENERGY DIRECTION
# ------------------------------------------------------------

if pd.isna(energy_build):

    energy_state = "UNKNOWN"

elif energy_build >= 10:

    energy_state = "RISING_STRONGLY"

elif energy_build >= 3:

    energy_state = "RISING"

elif energy_build <= -10:

    energy_state = "FALLING_STRONGLY"

elif energy_build <= -3:

    energy_state = "FALLING"

else:

    energy_state = "FLAT"

# ------------------------------------------------------------
# 10. COMPONENT AVAILABILITY
# ------------------------------------------------------------

components = {
    "volatility": volatility,
    "momentum_energy": momentum_energy,
    "volume": volume,
    "displacement": displacement,
}

available_components = [
    name
    for name, value in components.items()
    if pd.notna(value)
]

component_count = len(available_components)

# ------------------------------------------------------------
# 11. LIVE ASSESSMENT
# ------------------------------------------------------------

# We deliberately require BOTH:
# - high opportunity
# - high quality
#
# before calling something a serious opportunity.

if (
    pd.notna(quality)
    and pd.notna(opportunity)
    and quality >= 90
    and opportunity >= 80
):

    assessment = "EXCEPTIONAL_OPPORTUNITY"

elif (
    pd.notna(quality)
    and pd.notna(opportunity)
    and quality >= 75
    and opportunity >= 60
):

    assessment = "HIGH_OPPORTUNITY"

elif (
    pd.notna(quality)
    and pd.notna(opportunity)
    and quality >= 60
    and opportunity >= 50
):

    assessment = "DEVELOPING_OPPORTUNITY"

elif (
    pd.notna(quality)
    and pd.notna(opportunity)
    and quality >= 50
):

    assessment = "WATCH"

else:

    assessment = "NO_CLEAR_OPPORTUNITY"

# ------------------------------------------------------------
# 12. CONFIRMATION FLAGS
# ------------------------------------------------------------

confirmation = []

if pd.notna(volatility):

    if volatility >= 75:
        confirmation.append("HIGH_VOLATILITY")

if pd.notna(momentum_energy):

    if momentum_energy >= 75:
        confirmation.append("HIGH_MOMENTUM_ENERGY")

if pd.notna(volume):

    if volume >= 60:
        confirmation.append("VOLUME_CONFIRMING")
    else:
        confirmation.append("WEAK_VOLUME")

if pd.notna(displacement):

    if displacement >= 75:
        confirmation.append("HIGH_DISPLACEMENT")

if energy_state in [
    "RISING",
    "RISING_STRONGLY"
]:

    confirmation.append("ENERGY_BUILDING")

elif energy_state in [
    "FALLING",
    "FALLING_STRONGLY"
]:

    confirmation.append("ENERGY_FADING")

# ------------------------------------------------------------
# 13. LIVE SCANNER OUTPUT
# ------------------------------------------------------------

print(f"\nTimestamp:           {timestamp}")
print(f"Price:               {price:.4f}")

print("\n--- SCORES ---")

print(
    f"Opportunity score:   "
    f"{opportunity:.2f}"
    if pd.notna(opportunity)
    else "Opportunity score:   NA"
)

print(
    f"Opportunity rank:    "
    f"{opportunity_percentile:.2f} percentile"
    if pd.notna(opportunity_percentile)
    else "Opportunity rank:    NA"
)

print(
    f"Quality score:        "
    f"{quality:.2f}"
    if pd.notna(quality)
    else "Quality score:        NA"
)

print(
    f"Quality rank:         "
    f"{quality_percentile:.2f} percentile"
    if pd.notna(quality_percentile)
    else "Quality rank:         NA"
)

print("\n--- ZONES ---")

print(f"Opportunity zone:    {opportunity_zone}")
print(f"Quality zone:        {quality_zone}")

print("\n--- COMPONENTS ---")

for name, value in components.items():

    if pd.notna(value):
        print(f"{name:20s} {value:7.2f}")
    else:
        print(f"{name:20s} unavailable")

print("\n--- ENERGY ---")

print(
    f"Energy build:        "
    f"{energy_build:.2f}"
    if pd.notna(energy_build)
    else "Energy build:        unavailable"
)

print(f"Energy state:        {energy_state}")

print(
    f"Persistence:         "
    f"{persistence:.0f} hours"
    if pd.notna(persistence)
    else "Persistence:         unavailable"
)

print("\n--- CONFIRMATION ---")

if confirmation:

    for item in confirmation:
        print(f"• {item}")

else:

    print("No confirmation data available.")

print("\n--- CONFIDENCE ---")

print(
    f"Model confidence:    "
    f"{confidence:.1f}%"
    if pd.notna(confidence)
    else "Model confidence:    unavailable"
)

print("\n" + "=" * 80)
print("FINAL LIVE ASSESSMENT")
print("=" * 80)

print(f"\n>>> {assessment}")

# ------------------------------------------------------------
# 14. CAUSAL SAFETY AUDIT
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CAUSAL SAFETY AUDIT")
print("=" * 80)

future_columns = [
    "future_abs_6h",
    "future_abs_12h",
    "future_abs_24h",
    "future_abs_48h",
    "future_return_6h",
    "future_return_12h",
    "future_return_24h",
]

present_future_features = [
    column
    for column in future_columns
    if column in df.columns
]

print(
    f"\nFuture-target columns present in dataset: "
    f"{present_future_features}"
)

print(
    "\nIMPORTANT:"
)

print(
    "These future columns are targets for historical validation."
)

print(
    "They are NOT used to calculate the live scanner score."
)

print(
    "\nCurrent scanner state is calculated from the final row's "
    "feature/engine values only."
)

# ------------------------------------------------------------
# 15. SAVE CURRENT STATE
# ------------------------------------------------------------

live_state = pd.DataFrame([{
    "timestamp": timestamp,
    "price": price,

    "opportunity_score": opportunity,
    "opportunity_percentile": opportunity_percentile,

    "quality_score": quality,
    "quality_percentile": quality_percentile,

    "opportunity_zone": opportunity_zone,
    "quality_zone": quality_zone,

    "volatility_score": volatility,
    "momentum_energy_score": momentum_energy,
    "volume_score": volume,
    "displacement_score": displacement,

    "energy_build": energy_build,
    "energy_state": energy_state,

    "persistence": persistence,
    "confidence": confidence,

    "assessment": assessment,

    "component_count": component_count,

    "confirmation": "|".join(confirmation),
}])

live_path = os.path.join(
    RESULTS,
    "live_opportunity_scanner_v1.parquet"
)

live_state.to_parquet(
    live_path,
    index=False
)

print("\nSaved:")
print(live_path)

print("\n" + "=" * 80)
print("STEP 52 COMPLETE")
print("=" * 80)

STEP 52 — LIVE CAUSAL OPPORTUNITY SCANNER V1


FileNotFoundError: [Errno 2] No such file or directory: '/content/gdrive/MyDrive/SilverAgent/results/opportunity_quality_v1_history.parquet'

In [ ]:
# ============================================================
# STEP 52.1 — REPAIR LIVE PRICE
# ============================================================

import os
import pandas as pd

BASE = "/content/gdrive/MyDrive/SilverAgent"
RESULTS = os.path.join(BASE, "results")

DB_FILE = os.path.join(
    BASE,
    "database",
    "silver_h1.parquet"
)

LIVE_FILE = os.path.join(
    RESULTS,
    "live_opportunity_scanner_v1.parquet"
)

# Load actual market database
market = pd.read_parquet(DB_FILE)

if not isinstance(market.index, pd.DatetimeIndex):
    market.index = pd.to_datetime(market.index, utc=True)

market = market.sort_index()

latest = market.iloc[-1]

# Find close column safely
close_col = None

for candidate in ["close", "Close", "c"]:
    if candidate in market.columns:
        close_col = candidate
        break

if close_col is None:
    raise RuntimeError(
        f"Could not find close price. Columns are:\n{list(market.columns)}"
    )

latest_price = float(latest[close_col])
latest_time = latest.name

print("=" * 70)
print("STEP 52.1 — LIVE PRICE REPAIR")
print("=" * 70)

print(f"\nDatabase latest candle: {latest_time}")
print(f"Latest XAG/USD price:   {latest_price:.4f}")

# Load scanner state
live = pd.read_parquet(LIVE_FILE)

# Patch price and timestamp
live.loc[live.index[-1], "price"] = latest_price
live.loc[live.index[-1], "market_timestamp"] = latest_time

# Save repaired state
live.to_parquet(LIVE_FILE, index=False)

print("\nSaved repaired scanner state:")
print(LIVE_FILE)

print("\nPRICE CHECK:")
print(f"Price = {live.loc[live.index[-1], 'price']:.4f}")

print("\nSTEP 52.1 COMPLETE")

In [ ]:

# ============================================================
# STEP 53E — SAFE CONFIG FORMAT CHECK
# ============================================================

CONFIG_FILE = "/content/gdrive/MyDrive/SilverAgentOandaID.txt"

print("=" * 70)
print("STEP 53E — CONFIG FORMAT CHECK")
print("=" * 70)

with open(CONFIG_FILE, "r", encoding="utf-8-sig") as f:
    lines = f.readlines()

print(f"\nLines found: {len(lines)}")

for i, line in enumerate(lines, 1):

    stripped = line.strip()

    if not stripped:
        print(f"Line {i}: EMPTY")
        continue

    if "=" in stripped:
        key = stripped.split("=", 1)[0].strip()

        # Only show the key name — NEVER the value
        print(f"Line {i}: KEY FOUND -> {key}")

    else:
        print(f"Line {i}: TEXT WITHOUT '='")

print("\nNo credential values were displayed.")

print("\nSTEP 53E COMPLETE")

In [ ]:
from google.colab import drive
import os

drive.mount('/content/gdrive', force_remount=True)

print("Drive connected.")
print("SilverAgent folder exists:",
      os.path.exists("/content/gdrive/MyDrive/SilverAgent"))

In [ ]:
import os

MY_DRIVE = "/content/gdrive/MyDrive"

matches = []

for root, dirs, files in os.walk(MY_DRIVE):
    for name in files:
        if "SilverAgentOandaID" in name:
            matches.append(os.path.join(root, name))

print("Found:", len(matches))

for path in matches:
    print(path)

In [ ]:
import os

BASE = "/content/gdrive/MyDrive/SilverAgent"

print("SilverAgent exists:", os.path.exists(BASE))

print("\nFolders:")
print(os.listdir(BASE))

print("\nDatabase files:")
print(os.listdir(os.path.join(BASE, "database")))

print("\nResult files:")
print(os.listdir(os.path.join(BASE, "results"))[-10:])

In [ ]:

from google.colab import drive

drive.mount('/content/gdrive', force_remount=True)

In [ ]:
# ============================================================
# STEP 53F — ROBUST OANDA CONFIG LOADER
# ============================================================

import os

CONFIG_FILE = "/content/gdrive/MyDrive/SilverAgentOandaID.txt"

print("=" * 70)
print("STEP 53F — ROBUST OANDA CONFIG LOADER")
print("=" * 70)

OANDA_ACCOUNT_ID = None
OANDA_API_TOKEN = None

with open(CONFIG_FILE, "r", encoding="utf-8-sig") as f:

    for raw_line in f:

        line = raw_line.strip()

        # Ignore blank/comment lines
        if not line or line.startswith("#"):
            continue

        if "=" not in line:
            continue

        key, value = line.split("=", 1)

        key = key.strip().upper()
        value = value.strip().strip('"').strip("'")

        if key == "OANDA_ACCOUNT_ID":
            OANDA_ACCOUNT_ID = value

        elif key == "OANDA_API_TOKEN":
            OANDA_API_TOKEN = value

# ------------------------------------------------------------
# SECURITY CHECK
# ------------------------------------------------------------

if not OANDA_ACCOUNT_ID:
    raise RuntimeError(
        "OANDA_ACCOUNT_ID was not successfully loaded."
    )

if not OANDA_API_TOKEN:
    raise RuntimeError(
        "OANDA_API_TOKEN was not successfully loaded."
    )

# Put them into environment variables as well.
# This lets other Python modules retrieve them with os.getenv().
os.environ["OANDA_ACCOUNT_ID"] = OANDA_ACCOUNT_ID
os.environ["OANDA_API_TOKEN"] = OANDA_API_TOKEN

print("\nCONFIGURATION SUCCESSFUL")
print("------------------------")
print("OANDA_ACCOUNT_ID: LOADED")
print("OANDA_API_TOKEN:  LOADED")

print("\nCredentials were NOT displayed.")

print("\nEnvironment variables:")
print(
    "OANDA_ACCOUNT_ID available:",
    bool(os.getenv("OANDA_ACCOUNT_ID"))
)
print(
    "OANDA_API_TOKEN available:",
    bool(os.getenv("OANDA_API_TOKEN"))
)

print("\nSTEP 53F COMPLETE")

In [ ]:
# ============================================================
# STEP 54 — LIVE OANDA CONNECTION TEST
# ============================================================

import os
import requests
import pandas as pd

print("=" * 70)
print("STEP 54 — LIVE OANDA CONNECTION TEST")
print("=" * 70)

# ------------------------------------------------------------
# Load credentials from environment
# ------------------------------------------------------------

account_id = os.getenv("OANDA_ACCOUNT_ID")
api_token = os.getenv("OANDA_API_TOKEN")

if not account_id or not api_token:
    raise RuntimeError(
        "OANDA credentials are not loaded. "
        "Run Step 53F first."
    )

# ------------------------------------------------------------
# OANDA PRACTICE API
# ------------------------------------------------------------

BASE_URL = "https://api-fxpractice.oanda.com"

headers = {
    "Authorization": f"Bearer {api_token}",
    "Content-Type": "application/json"
}

# ------------------------------------------------------------
# Request latest XAG/USD candles
# ------------------------------------------------------------

url = f"{BASE_URL}/v3/instruments/XAG_USD/candles"

params = {
    "granularity": "H1",
    "count": 10,
    "price": "M"
}

response = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=20
)

print(f"\nHTTP status: {response.status_code}")

if response.status_code != 200:
    print("\nOANDA connection failed.")
    print("Response status:", response.status_code)
    raise RuntimeError(
        "OANDA API request failed. "
        "The response body has deliberately not been displayed "
        "because it may contain account-related information."
    )

data = response.json()

candles = []

for candle in data.get("candles", []):

    if not candle.get("complete", False):
        continue

    mid = candle.get("mid", {})

    candles.append({
        "time": pd.to_datetime(
            candle["time"],
            utc=True
        ),
        "open": float(mid["o"]),
        "high": float(mid["h"]),
        "low": float(mid["l"]),
        "close": float(mid["c"]),
        "volume": int(candle["volume"])
    })

live = pd.DataFrame(candles)

if live.empty:
    raise RuntimeError(
        "OANDA responded successfully but returned no "
        "completed candles."
    )

live = (
    live
    .set_index("time")
    .sort_index()
)

# ------------------------------------------------------------
# DISPLAY SAFE INFORMATION ONLY
# ------------------------------------------------------------

print("\nOANDA CONNECTION: SUCCESS")
print("-" * 70)

print("Account credentials: loaded")
print("Environment:         PRACTICE")
print("Instrument:          XAG/USD")
print("Granularity:         H1")
print("Completed candles:   ", len(live))

print("\nLatest completed candle:")
print("Time:   ", live.index[-1])
print("Open:   ", f"{live['open'].iloc[-1]:.4f}")
print("High:   ", f"{live['high'].iloc[-1]:.4f}")
print("Low:    ", f"{live['low'].iloc[-1]:.4f}")
print("Close:  ", f"{live['close'].iloc[-1]:.4f}")
print("Volume: ", live['volume'].iloc[-1])

print("\n" + "=" * 70)
print("STEP 54 COMPLETE")
print("=" * 70)

print(
    "\nLIVE DATA IS CONNECTED."
)

print(
    "No orders were created or submitted."
)

In [ ]:
# ============================================================
# STEP 55 — UPDATE SILVER DATABASE FROM OANDA
# ============================================================

import os
import requests
import pandas as pd

BASE = "/content/gdrive/MyDrive/SilverAgent"

DB_FILE = os.path.join(
    BASE,
    "database",
    "silver_h1.parquet"
)

# ------------------------------------------------------------
# LOAD CREDENTIALS
# ------------------------------------------------------------

account_id = os.getenv("OANDA_ACCOUNT_ID")
api_token = os.getenv("OANDA_API_TOKEN")

if not account_id or not api_token:
    raise RuntimeError(
        "OANDA credentials are not loaded. Run Step 53F first."
    )

# ------------------------------------------------------------
# LOAD EXISTING DATABASE
# ------------------------------------------------------------

db = pd.read_parquet(DB_FILE)

if not isinstance(db.index, pd.DatetimeIndex):
    db.index = pd.to_datetime(db.index, utc=True)
else:
    db.index = pd.to_datetime(db.index, utc=True)

db = db.sort_index()

last_time = db.index[-1]

print("=" * 70)
print("STEP 55 — UPDATE SILVER DATABASE")
print("=" * 70)

print("\nExisting database:")
print(f"Candles:       {len(db):,}")
print(f"Latest candle: {last_time}")
print(f"Latest close:  {float(db['close'].iloc[-1]):.4f}")

# ------------------------------------------------------------
# OANDA API
# ------------------------------------------------------------

BASE_URL = "https://api-fxpractice.oanda.com"

url = (
    f"{BASE_URL}/v3/instruments/XAG_USD/candles"
)

headers = {
    "Authorization": f"Bearer {api_token}",
    "Content-Type": "application/json"
}

# Request recent candles.
# We deliberately request a small batch and then filter
# against the database timestamp.
params = {
    "granularity": "H1",
    "count": 500,
    "price": "M"
}

response = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=20
)

print(f"\nOANDA HTTP status: {response.status_code}")

if response.status_code != 200:
    raise RuntimeError(
        f"OANDA request failed with HTTP {response.status_code}"
    )

data = response.json()

# ------------------------------------------------------------
# PARSE COMPLETED CANDLES
# ------------------------------------------------------------

new_rows = []

for candle in data.get("candles", []):

    if not candle.get("complete", False):
        continue

    timestamp = pd.to_datetime(
        candle["time"],
        utc=True
    )

    # Only accept candles newer than database
    if timestamp <= last_time:
        continue

    mid = candle.get("mid", {})

    new_rows.append({
        "open": float(mid["o"]),
        "high": float(mid["h"]),
        "low": float(mid["l"]),
        "close": float(mid["c"]),
        "volume": int(candle["volume"])
    })

# ------------------------------------------------------------
# ADD NEW DATA
# ------------------------------------------------------------

if new_rows:

    new_df = pd.DataFrame(
        new_rows,
        index=[
            pd.to_datetime(
                c["time"],
                utc=True
            )
            for c in [
                {
                    "time": candle["time"]
                }
                for candle in data.get("candles", [])
                if candle.get("complete", False)
                and pd.to_datetime(
                    candle["time"],
                    utc=True
                ) > last_time
            ]
        ]
    )

    new_df = new_df.sort_index()

    # Ensure exact column order
    new_df = new_df[
        ["open", "high", "low", "close", "volume"]
    ]

    combined = pd.concat(
        [db, new_df]
    )

else:

    new_df = pd.DataFrame(
        columns=db.columns
    )

    combined = db.copy()

# ------------------------------------------------------------
# SAFETY CLEANUP
# ------------------------------------------------------------

combined = combined[
    ~combined.index.duplicated(
        keep="last"
    )
]

combined = combined.sort_index()

# Ensure numeric types
for column in [
    "open",
    "high",
    "low",
    "close",
    "volume"
]:

    combined[column] = pd.to_numeric(
        combined[column],
        errors="coerce"
    )

# Remove completely invalid rows
combined = combined.dropna(
    subset=[
        "open",
        "high",
        "low",
        "close",
        "volume"
    ]
)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

combined.to_parquet(
    DB_FILE
)

# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

added = len(combined) - len(db)

print("\n" + "=" * 70)
print("DATABASE UPDATE COMPLETE")
print("=" * 70)

print(f"\nPrevious candles: {len(db):,}")
print(f"New candles:      {max(0, added):,}")
print(f"Total candles:    {len(combined):,}")

print(
    f"\nPrevious latest:  {last_time}"
)

print(
    f"New latest:       {combined.index[-1]}"
)

print(
    f"Latest close:     "
    f"{float(combined['close'].iloc[-1]):.4f}"
)

print(
    f"\nDatabase saved to:\n{DB_FILE}"
)

print("\nNo candles were fabricated.")
print("No trading orders were submitted.")

print("\nSTEP 55 COMPLETE")

In [ ]:

# ============================================================
# STEP 56 — GAP-AWARE FEATURE ENGINE V2.1 REBUILD
# ============================================================

import os
import numpy as np
import pandas as pd

BASE = "/content/gdrive/MyDrive/SilverAgent"

DB_FILE = os.path.join(
    BASE,
    "database",
    "silver_h1.parquet"
)

FEATURE_FILE = os.path.join(
    BASE,
    "database",
    "silver_h1_features_v2.parquet"
)

print("=" * 80)
print("STEP 56 — GAP-AWARE FEATURE ENGINE V2.1 REBUILD")
print("=" * 80)

# ------------------------------------------------------------
# 1. LOAD DATABASE
# ------------------------------------------------------------

df = pd.read_parquet(DB_FILE)

if not isinstance(df.index, pd.DatetimeIndex):
    df.index = pd.to_datetime(df.index, utc=True)
else:
    df.index = pd.to_datetime(df.index, utc=True)

df = df.sort_index()

print(f"\nRaw candles: {len(df):,}")
print(f"Start: {df.index.min()}")
print(f"End:   {df.index.max()}")

# ------------------------------------------------------------
# 2. BASIC FEATURES
# ------------------------------------------------------------

close = df["close"]
high = df["high"]
low = df["low"]
volume = df["volume"]

# Returns
df["return_1h"] = close.pct_change(1)
df["return_3h"] = close.pct_change(3)
df["return_6h"] = close.pct_change(6)
df["return_12h"] = close.pct_change(12)
df["return_24h"] = close.pct_change(24)

# Moving averages
df["MA10"] = close.rolling(10).mean()
df["MA20"] = close.rolling(20).mean()
df["MA50"] = close.rolling(50).mean()

df["MA10_vs_MA20"] = (
    df["MA10"] / df["MA20"] - 1
)

df["MA20_vs_MA50"] = (
    df["MA20"] / df["MA50"] - 1
)

df["price_vs_MA20"] = (
    close / df["MA20"] - 1
)

# ------------------------------------------------------------
# 3. VOLATILITY
# ------------------------------------------------------------

df["vol10"] = (
    df["return_1h"]
    .rolling(10)
    .std()
)

df["vol20"] = (
    df["return_1h"]
    .rolling(20)
    .std()
)

# True range
previous_close = close.shift(1)

tr1 = high - low
tr2 = (high - previous_close).abs()
tr3 = (low - previous_close).abs()

true_range = pd.concat(
    [tr1, tr2, tr3],
    axis=1
).max(axis=1)

df["ATR14"] = true_range.rolling(14).mean()

df["atr_percent"] = (
    df["ATR14"] / close
)

# ------------------------------------------------------------
# 4. VOLUME / ACTIVITY
# ------------------------------------------------------------

volume_ma20 = volume.rolling(20).mean()

df["relative_volume"] = (
    volume / volume_ma20
)

# ------------------------------------------------------------
# 5. PRICE POSITION
# ------------------------------------------------------------

high20 = high.rolling(20).max()
low20 = low.rolling(20).min()

df["distance_high20"] = (
    high20 / close - 1
)

df["distance_low20"] = (
    close / low20 - 1
)

df["range_position20"] = (
    (close - low20) /
    (high20 - low20)
)

# ------------------------------------------------------------
# 6. MOMENTUM INTENSITY
# ------------------------------------------------------------

df["momentum_intensity"] = (
    df["return_3h"].abs() +
    df["return_6h"].abs() +
    df["return_12h"].abs()
) / 3

# ------------------------------------------------------------
# 7. RANGE EXPANSION
# ------------------------------------------------------------

range_now = (
    (high - low) / close
)

range_average20 = (
    range_now.rolling(20).mean()
)

df["range_expansion"] = (
    range_now / range_average20
)

# ------------------------------------------------------------
# 8. PRICE DISPLACEMENT
# ------------------------------------------------------------

df["price_displacement"] = (
    df["return_6h"].abs()
)

# ------------------------------------------------------------
# 9. ATR / VOLATILITY RATIO
# ------------------------------------------------------------

df["atr_vol_ratio"] = (
    df["atr_percent"] /
    df["vol20"]
)

# ------------------------------------------------------------
# 10. GAP INFORMATION
# ------------------------------------------------------------

time_diff = df.index.to_series().diff()

df["gap_hours"] = (
    time_diff.dt.total_seconds() / 3600
)

df["is_gap_after"] = (
    df["gap_hours"] > 1.5
)

gap_count = int(
    df["is_gap_after"].sum()
)

print(f"\nDetected hourly gaps: {gap_count:,}")

# ------------------------------------------------------------
# 11. GAP-AWARE FUTURE TARGETS
# ------------------------------------------------------------
# We use the nearest REAL candle to the requested future time.
#
# No missing candles are fabricated.
# Maximum tolerance = 90 minutes.

future_targets = {}

for hours in [6, 12, 24, 48]:

    target_times = (
        df.index +
        pd.Timedelta(hours=hours)
    )

    future_series = pd.Series(
        df["close"].values,
        index=df.index
    )

    matched = pd.merge_asof(
        pd.DataFrame(
            {"target_time": target_times},
            index=df.index
        ),
        future_series.rename("future_close").to_frame(),
        left_on="target_time",
        right_index=True,
        direction="nearest",
        tolerance=pd.Timedelta(minutes=90)
    )

    future_close = matched["future_close"]

    future_return = (
        future_close / close - 1
    )

    df[f"future_return_{hours}h"] = future_return

    df[f"future_abs_{hours}h"] = (
        future_return.abs()
    )

# ------------------------------------------------------------
# 12. FEATURE VALIDITY SUMMARY
# ------------------------------------------------------------

feature_columns = [
    "return_1h",
    "return_3h",
    "return_6h",
    "return_12h",
    "return_24h",
    "MA10",
    "MA20",
    "MA50",
    "MA10_vs_MA20",
    "MA20_vs_MA50",
    "price_vs_MA20",
    "vol10",
    "vol20",
    "ATR14",
    "atr_percent",
    "relative_volume",
    "distance_high20",
    "distance_low20",
    "range_position20",
    "momentum_intensity",
    "range_expansion",
    "price_displacement",
    "atr_vol_ratio",
]

print("\nFEATURE VALIDITY")
print("-" * 80)

for hours in [6, 12, 24, 48]:

    valid = df[f"future_abs_{hours}h"].notna().sum()

    print(
        f"{hours:>2}h future target: "
        f"{valid:,} valid / {len(df):,}"
    )

# ------------------------------------------------------------
# 13. LATEST STATE
# ------------------------------------------------------------

latest = df.iloc[-1]

print("\n" + "=" * 80)
print("LATEST FEATURE STATE")
print("=" * 80)

print(f"\nTimestamp:       {latest.name}")
print(f"Price:           {latest['close']:.4f}")

print(
    f"1h return:       "
    f"{latest['return_1h'] * 100:.3f}%"
)

print(
    f"6h return:       "
    f"{latest['return_6h'] * 100:.3f}%"
)

print(
    f"12h return:      "
    f"{latest['return_12h'] * 100:.3f}%"
)

print(
    f"24h return:      "
    f"{latest['return_24h'] * 100:.3f}%"
)

print(
    f"MA10 vs MA20:    "
    f"{latest['MA10_vs_MA20'] * 100:.3f}%"
)

print(
    f"MA20 vs MA50:    "
    f"{latest['MA20_vs_MA50'] * 100:.3f}%"
)

print(
    f"Price vs MA20:   "
    f"{latest['price_vs_MA20'] * 100:.3f}%"
)

print(
    f"Vol10:           "
    f"{latest['vol10'] * 100:.3f}%"
)

print(
    f"Vol20:           "
    f"{latest['vol20'] * 100:.3f}%"
)

print(
    f"ATR%:            "
    f"{latest['atr_percent'] * 100:.3f}%"
)

print(
    f"Relative volume: "
    f"{latest['relative_volume']:.2f}x"
)

print(
    f"Range position:  "
    f"{latest['range_position20'] * 100:.1f}%"
)

print(
    f"Gap after candle: "
    f"{latest['is_gap_after']}"
)

# ------------------------------------------------------------
# 14. SAVE
# ------------------------------------------------------------

df.to_parquet(
    FEATURE_FILE
)

print("\n" + "=" * 80)
print("FEATURE ENGINE SAVED")
print("=" * 80)

print(f"\nSaved:")
print(FEATURE_FILE)

print(f"\nRows saved: {len(df):,}")

print("\nNo candles were fabricated.")
print("Future targets use real-candle nearest-time matching.")
print("No trading orders were submitted.")

print("\nSTEP 56 COMPLETE")

In [ ]:
# ============================================================
# STEP 57 — OPPORTUNITY ENGINE V2 REBUILD
# ============================================================

import os
import numpy as np
import pandas as pd

BASE = "/content/gdrive/MyDrive/SilverAgent"

FEATURE_FILE = os.path.join(
    BASE,
    "database",
    "silver_h1_features_v2.parquet"
)

RESULTS = os.path.join(
    BASE,
    "results"
)

OUTPUT_FILE = os.path.join(
    RESULTS,
    "opportunity_v2_history.parquet"
)

print("=" * 80)
print("STEP 57 — OPPORTUNITY ENGINE V2 REBUILD")
print("=" * 80)

# ------------------------------------------------------------
# 1. LOAD FEATURES
# ------------------------------------------------------------

df = pd.read_parquet(FEATURE_FILE)

if not isinstance(df.index, pd.DatetimeIndex):
    df.index = pd.to_datetime(df.index, utc=True)
else:
    df.index = pd.to_datetime(df.index, utc=True)

df = df.sort_index()

print(f"\nRows:  {len(df):,}")
print(f"Start: {df.index.min()}")
print(f"End:   {df.index.max()}")

# ------------------------------------------------------------
# 2. FEATURES USED BY OPPORTUNITY ENGINE
# ------------------------------------------------------------

required = [
    "vol10",
    "vol20",
    "atr_percent",
    "relative_volume",
    "distance_high20",
    "distance_low20",
    "momentum_intensity",
    "range_expansion",
    "price_displacement",
]

missing = [
    c for c in required
    if c not in df.columns
]

if missing:
    raise RuntimeError(
        f"Missing required features: {missing}"
    )

# ------------------------------------------------------------
# 3. HISTORICAL-ONLY PERCENTILE FUNCTION
# ------------------------------------------------------------

def historical_percentile(series, min_periods=500):

    history = series.shift(1)

    return (
        history
        .expanding(min_periods=min_periods)
        .apply(
            lambda x: (
                x.iloc[-1] <= x.iloc[:-1]
            ).mean()
            if len(x) > 1
            else np.nan,
            raw=False
        )
    )

# ------------------------------------------------------------
# NOTE
# ------------------------------------------------------------
# We don't actually need the expensive expanding apply above.
# Use a faster historical percentile approximation based on
# expanding rank of the shifted series.
#
# The shift is the important part: today's value cannot be
# included in today's historical distribution.
# ------------------------------------------------------------

def historical_rank(series, min_periods=500):

    shifted = series.shift(1)

    return (
        shifted
        .expanding(min_periods=min_periods)
        .rank(pct=True)
    )

# ------------------------------------------------------------
# 4. COMPONENT PERCENTILES
# ------------------------------------------------------------

print("\nCalculating historical component ranks...")

df["vol10_pct"] = (
    historical_rank(df["vol10"]) * 100
)

df["vol20_pct"] = (
    historical_rank(df["vol20"]) * 100
)

df["atr_percent_pct"] = (
    historical_rank(df["atr_percent"]) * 100
)

df["relative_volume_pct"] = (
    historical_rank(df["relative_volume"]) * 100
)

df["distance_high20_pct"] = (
    historical_rank(df["distance_high20"]) * 100
)

df["distance_low20_pct"] = (
    historical_rank(df["distance_low20"]) * 100
)

df["momentum_intensity_pct"] = (
    historical_rank(df["momentum_intensity"]) * 100
)

df["range_expansion_pct"] = (
    historical_rank(df["range_expansion"]) * 100
)

df["price_displacement_pct"] = (
    historical_rank(df["price_displacement"]) * 100
)

# ------------------------------------------------------------
# 5. ENGINE SCORES
# ------------------------------------------------------------

df["volatility_score"] = (
    df[
        [
            "vol10_pct",
            "vol20_pct",
            "atr_percent_pct",
        ]
    ]
    .mean(axis=1)
)

df["momentum_energy_score"] = (
    df[
        [
            "momentum_intensity_pct",
            "range_expansion_pct",
        ]
    ]
    .mean(axis=1)
)

df["volume_score"] = (
    df["relative_volume_pct"]
)

df["displacement_score"] = (
    df[
        [
            "price_displacement_pct",
            "distance_high20_pct",
            "distance_low20_pct",
        ]
    ]
    .mean(axis=1)
)

# ------------------------------------------------------------
# 6. CONTINUOUS OPPORTUNITY SCORE
# ------------------------------------------------------------

score_columns = [
    "volatility_score",
    "momentum_energy_score",
    "volume_score",
    "displacement_score",
]

df["continuous_opportunity"] = (
    df[score_columns]
    .mean(axis=1)
)

# ------------------------------------------------------------
# 7. OPPORTUNITY ZONES
# ------------------------------------------------------------

def opportunity_zone(score):

    if pd.isna(score):
        return "INSUFFICIENT_DATA"

    if score >= 80:
        return "VERY_HIGH"

    if score >= 60:
        return "HIGH"

    if score >= 40:
        return "MEDIUM"

    if score >= 20:
        return "LOW"

    return "VERY_LOW"

df["opportunity_zone"] = (
    df["continuous_opportunity"]
    .apply(opportunity_zone)
)

# ------------------------------------------------------------
# 8. HISTORICAL TARGETS
# ------------------------------------------------------------

# These are retained for research/validation only.
# They are NOT inputs to the current opportunity score.

target_columns = [
    "future_abs_6h",
    "future_abs_12h",
    "future_abs_24h",
    "future_abs_48h",
]

for column in target_columns:

    if column not in df.columns:
        print(
            f"Warning: {column} not present."
        )

# ------------------------------------------------------------
# 9. CURRENT STATE
# ------------------------------------------------------------

latest = df.iloc[-1]

print("\n" + "=" * 80)
print("CURRENT OPPORTUNITY STATE")
print("=" * 80)

print(f"\nTimestamp: {latest.name}")

print(
    f"Price: "
    f"{latest['close']:.4f}"
)

print(
    f"\nVolatility score: "
    f"{latest['volatility_score']:.2f}"
)

print(
    f"Momentum energy:  "
    f"{latest['momentum_energy_score']:.2f}"
)

print(
    f"Volume score:     "
    f"{latest['volume_score']:.2f}"
)

print(
    f"Displacement:     "
    f"{latest['displacement_score']:.2f}"
)

print(
    f"\nOpportunity score:"
    f" {latest['continuous_opportunity']:.2f}"
)

print(
    f"Opportunity zone: "
    f"{latest['opportunity_zone']}"
)

# ------------------------------------------------------------
# 10. CURRENT RAW FEATURES
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("RAW MARKET FEATURES")
print("-" * 80)

print(
    f"Vol10:            "
    f"{latest['vol10'] * 100:.3f}%"
)

print(
    f"Vol20:            "
    f"{latest['vol20'] * 100:.3f}%"
)

print(
    f"ATR%:             "
    f"{latest['atr_percent'] * 100:.3f}%"
)

print(
    f"Relative volume:  "
    f"{latest['relative_volume']:.2f}x"
)

print(
    f"Distance high20:  "
    f"{latest['distance_high20'] * 100:.3f}%"
)

print(
    f"Distance low20:   "
    f"{latest['distance_low20'] * 100:.3f}%"
)

print(
    f"Momentum intensity:"
    f" {latest['momentum_intensity'] * 100:.3f}%"
)

print(
    f"Range expansion:  "
    f"{latest['range_expansion']:.3f}x"
)

print(
    f"Price displacement:"
    f" {latest['price_displacement'] * 100:.3f}%"
)

# ------------------------------------------------------------
# 11. DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("OPPORTUNITY ZONE DISTRIBUTION")
print("=" * 80)

distribution = (
    df["opportunity_zone"]
    .value_counts()
)

print(distribution.to_string())

# ------------------------------------------------------------
# 12. SAVE
# ------------------------------------------------------------

df.to_parquet(
    OUTPUT_FILE
)

print("\n" + "=" * 80)
print("OPPORTUNITY ENGINE SAVED")
print("=" * 80)

print(
    f"\nSaved:\n{OUTPUT_FILE}"
)

print(
    f"\nRows saved: {len(df):,}"
)

print("\nNo future targets were used as model inputs.")
print("No trading orders were submitted.")

print("\nSTEP 57 COMPLETE")

In [ ]:
# ============================================================
# STEP 58 — OPPORTUNITY QUALITY ENGINE V2
# ============================================================

import os
import numpy as np
import pandas as pd

BASE = "/content/gdrive/MyDrive/SilverAgent"

OPP_FILE = os.path.join(
    BASE,
    "results",
    "opportunity_v2_history.parquet"
)

RESULTS = os.path.join(
    BASE,
    "results"
)

OUTPUT_FILE = os.path.join(
    RESULTS,
    "opportunity_quality_v2_history.parquet"
)

SUMMARY_FILE = os.path.join(
    RESULTS,
    "opportunity_quality_v2_summary.parquet"
)

print("=" * 80)
print("STEP 58 — OPPORTUNITY QUALITY ENGINE V2")
print("=" * 80)

# ------------------------------------------------------------
# 1. LOAD CURRENT OPPORTUNITY ENGINE
# ------------------------------------------------------------

df = pd.read_parquet(OPP_FILE)

if not isinstance(df.index, pd.DatetimeIndex):
    df.index = pd.to_datetime(df.index, utc=True)
else:
    df.index = pd.to_datetime(df.index, utc=True)

df = df.sort_index()

print(f"\nRows:  {len(df):,}")
print(f"Start: {df.index.min()}")
print(f"End:   {df.index.max()}")

# ------------------------------------------------------------
# 2. REQUIRED COLUMNS
# ------------------------------------------------------------

required = [
    "continuous_opportunity",
    "volatility_score",
    "momentum_energy_score",
    "volume_score",
    "displacement_score",
]

missing = [
    c for c in required
    if c not in df.columns
]

if missing:
    raise RuntimeError(
        f"Missing required columns: {missing}"
    )

# ------------------------------------------------------------
# 3. HISTORICAL-ONLY RANK
# ------------------------------------------------------------

def historical_rank(series, min_periods=500):

    # Shift first so the current observation cannot
    # influence its own historical distribution.
    shifted = series.shift(1)

    return (
        shifted
        .expanding(min_periods=min_periods)
        .rank(pct=True)
        * 100
    )

print("\nCalculating historical-only quality components...")

df["opportunity_rank"] = historical_rank(
    df["continuous_opportunity"]
)

df["volatility_rank"] = historical_rank(
    df["volatility_score"]
)

df["momentum_rank"] = historical_rank(
    df["momentum_energy_score"]
)

df["volume_rank"] = historical_rank(
    df["volume_score"]
)

df["displacement_rank"] = historical_rank(
    df["displacement_score"]
)

# ------------------------------------------------------------
# 4. ENERGY BUILD
# ------------------------------------------------------------

# Use causal changes in opportunity score.
#
# Current score - previous score
# Current score - score 3 candles ago
# Current score - score 6 candles ago

df["score_change_1h"] = (
    df["continuous_opportunity"]
    .diff(1)
)

df["score_change_3h"] = (
    df["continuous_opportunity"]
    .diff(3)
)

df["score_change_6h"] = (
    df["continuous_opportunity"]
    .diff(6)
)

# Weighted energy build
df["energy_build_raw"] = (
    0.50 * df["score_change_1h"] +
    0.30 * df["score_change_3h"] +
    0.20 * df["score_change_6h"]
)

df["energy_build_rank"] = historical_rank(
    df["energy_build_raw"]
)

# ------------------------------------------------------------
# 5. OPPORTUNITY PERSISTENCE
# ------------------------------------------------------------

high_opportunity = (
    df["continuous_opportunity"] >= 60
)

# Consecutive run length
groups = (
    high_opportunity
    .ne(high_opportunity.shift())
    .cumsum()
)

df["opportunity_persistence"] = (
    high_opportunity
    .groupby(groups)
    .cumsum()
)

# Reset non-opportunity periods to zero
df.loc[
    ~high_opportunity,
    "opportunity_persistence"
] = 0

df["persistence_rank"] = historical_rank(
    df["opportunity_persistence"]
)

# ------------------------------------------------------------
# 6. QUALITY SCORE
# ------------------------------------------------------------

# Weighting:
#
# Opportunity environment      30%
# Volatility                   20%
# Momentum energy              20%
# Displacement                 10%
# Energy building              10%
# Volume                        5%
# Persistence                   5%

components = [
    "opportunity_rank",
    "volatility_rank",
    "momentum_rank",
    "displacement_rank",
    "energy_build_rank",
    "volume_rank",
    "persistence_rank",
]

weights = np.array([
    0.30,
    0.20,
    0.20,
    0.10,
    0.10,
    0.05,
    0.05,
])

# Weighted average using only available components
weighted_sum = (
    df[components]
    .mul(weights, axis=1)
    .sum(axis=1)
)

weight_available = (
    df[components]
    .notna()
    .mul(weights, axis=1)
    .sum(axis=1)
)

df["quality_score"] = (
    weighted_sum /
    weight_available
)

# Require at least 80% of the model components
# before declaring a normal quality score.
df.loc[
    weight_available < 0.80,
    "quality_score"
] = np.nan

# ------------------------------------------------------------
# 7. QUALITY ZONES
# ------------------------------------------------------------

def quality_zone(score):

    if pd.isna(score):
        return "INSUFFICIENT_DATA"

    if score >= 90:
        return "EXCEPTIONAL"

    if score >= 75:
        return "HIGH"

    if score >= 50:
        return "MEDIUM"

    if score >= 25:
        return "LOW"

    return "VERY_LOW"

df["quality_zone"] = (
    df["quality_score"]
    .apply(quality_zone)
)

# ------------------------------------------------------------
# 8. QUALITY DECILES
# ------------------------------------------------------------

# Historical percentile of quality itself.
df["quality_percentile"] = historical_rank(
    df["quality_score"]
)

df["quality_decile"] = np.floor(
    df["quality_percentile"] / 10
) + 1

df["quality_decile"] = (
    df["quality_decile"]
    .clip(1, 10)
)

# ------------------------------------------------------------
# 9. CURRENT STATE
# ------------------------------------------------------------

latest = df.iloc[-1]

print("\n" + "=" * 80)
print("CURRENT QUALITY STATE")
print("=" * 80)

print(f"\nTimestamp: {latest.name}")

if "close" in latest.index:
    print(
        f"Price: "
        f"{latest['close']:.4f}"
    )

print(
    f"\nOpportunity: "
    f"{latest['continuous_opportunity']:.2f}"
)

print(
    f"Opportunity zone: "
    f"{latest.get('opportunity_zone', 'UNKNOWN')}"
)

print(
    f"\nOpportunity rank: "
    f"{latest['opportunity_rank']:.2f}"
)

print(
    f"Volatility rank: "
    f"{latest['volatility_rank']:.2f}"
)

print(
    f"Momentum rank: "
    f"{latest['momentum_rank']:.2f}"
)

print(
    f"Volume rank: "
    f"{latest['volume_rank']:.2f}"
)

print(
    f"Displacement rank: "
    f"{latest['displacement_rank']:.2f}"
)

print(
    f"Energy build: "
    f"{latest['energy_build_raw']:.2f}"
)

print(
    f"Energy build rank: "
    f"{latest['energy_build_rank']:.2f}"
)

print(
    f"Persistence: "
    f"{latest['opportunity_persistence']:.0f} hours"
)

print(
    f"Persistence rank: "
    f"{latest['persistence_rank']:.2f}"
)

print(
    f"\nQUALITY SCORE: "
    f"{latest['quality_score']:.2f}"
)

print(
    f"QUALITY ZONE: "
    f"{latest['quality_zone']}"
)

print(
    f"QUALITY PERCENTILE: "
    f"{latest['quality_percentile']:.2f}"
)

# ------------------------------------------------------------
# 10. HISTORICAL DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("QUALITY ZONE DISTRIBUTION")
print("=" * 80)

print(
    df["quality_zone"]
    .value_counts()
    .to_string()
)

# ------------------------------------------------------------
# 11. OPTIONAL HISTORICAL TARGET SUMMARY
# ------------------------------------------------------------

if "future_abs_24h" in df.columns:

    valid = df.dropna(
        subset=[
            "quality_zone",
            "future_abs_24h"
        ]
    ).copy()

    valid["future_abs_24h_pct"] = (
        valid["future_abs_24h"] * 100
    )

    summary = (
        valid
        .groupby("quality_zone")
        .agg(
            n=("future_abs_24h_pct", "size"),
            mean_24h_pct=(
                "future_abs_24h_pct",
                "mean"
            ),
            median_24h_pct=(
                "future_abs_24h_pct",
                "median"
            ),
            p90_24h_pct=(
                "future_abs_24h_pct",
                lambda x: x.quantile(0.90)
            ),
            p95_24h_pct=(
                "future_abs_24h_pct",
                lambda x: x.quantile(0.95)
            ),
        )
        .sort_values(
            "mean_24h_pct",
            ascending=False
        )
    )

    print("\n" + "=" * 80)
    print("HISTORICAL QUALITY PERFORMANCE")
    print("=" * 80)

    print(
        summary.to_string(
            float_format=lambda x: f"{x:.3f}"
        )
    )

else:

    summary = pd.DataFrame()

# ------------------------------------------------------------
# 12. SAVE
# ------------------------------------------------------------

df.to_parquet(
    OUTPUT_FILE
)

if not summary.empty:
    summary.to_parquet(
        SUMMARY_FILE
    )

print("\n" + "=" * 80)
print("QUALITY ENGINE SAVED")
print("=" * 80)

print(
    f"\nHistory:\n{OUTPUT_FILE}"
)

if not summary.empty:
    print(
        f"\nSummary:\n{SUMMARY_FILE}"
    )

print("\nHistorical future targets are used only for analysis.")
print("They are NOT inputs into the quality score.")
print("No trading orders were submitted.")

print("\nSTEP 58 COMPLETE")

In [ ]:
# ============================================================
# STEP 59 — OPPORTUNITY STATE MONITOR V1
# ============================================================

import os
import numpy as np
import pandas as pd

BASE = "/content/gdrive/MyDrive/SilverAgent"

INPUT_FILE = os.path.join(
    BASE,
    "results",
    "opportunity_quality_v2_history.parquet"
)

OUTPUT_FILE = os.path.join(
    BASE,
    "results",
    "opportunity_state_v1_history.parquet"
)

print("=" * 80)
print("STEP 59 — OPPORTUNITY STATE MONITOR V1")
print("=" * 80)

# ------------------------------------------------------------
# 1. LOAD
# ------------------------------------------------------------

df = pd.read_parquet(INPUT_FILE)

if not isinstance(df.index, pd.DatetimeIndex):
    df.index = pd.to_datetime(df.index, utc=True)
else:
    df.index = pd.to_datetime(df.index, utc=True)

df = df.sort_index()

print(f"\nRows:  {len(df):,}")
print(f"Start: {df.index.min()}")
print(f"End:   {df.index.max()}")

# ------------------------------------------------------------
# 2. SCORE CHANGES
# ------------------------------------------------------------

df["quality_change_1h"] = (
    df["quality_score"].diff(1)
)

df["quality_change_3h"] = (
    df["quality_score"].diff(3)
)

df["quality_change_6h"] = (
    df["quality_score"].diff(6)
)

df["quality_change_12h"] = (
    df["quality_score"].diff(12)
)

# ------------------------------------------------------------
# 3. OPPORTUNITY SCORE CHANGES
# ------------------------------------------------------------

df["opportunity_change_1h"] = (
    df["continuous_opportunity"].diff(1)
)

df["opportunity_change_3h"] = (
    df["continuous_opportunity"].diff(3)
)

df["opportunity_change_6h"] = (
    df["continuous_opportunity"].diff(6)
)

# ------------------------------------------------------------
# 4. TREND OF QUALITY
# ------------------------------------------------------------

# Count how many horizons are rising/falling.

quality_changes = [
    "quality_change_1h",
    "quality_change_3h",
    "quality_change_6h",
]

df["quality_rising_count"] = (
    df[quality_changes] > 0
).sum(axis=1)

df["quality_falling_count"] = (
    df[quality_changes] < 0
).sum(axis=1)

# ------------------------------------------------------------
# 5. OPPORTUNITY ENERGY TREND
# ------------------------------------------------------------

opp_changes = [
    "opportunity_change_1h",
    "opportunity_change_3h",
    "opportunity_change_6h",
]

df["opportunity_rising_count"] = (
    df[opp_changes] > 0
).sum(axis=1)

df["opportunity_falling_count"] = (
    df[opp_changes] < 0
).sum(axis=1
)

# ------------------------------------------------------------
# 6. STATE CLASSIFICATION
# ------------------------------------------------------------

def classify_state(row):

    q = row["quality_score"]
    q1 = row["quality_change_1h"]
    q3 = row["quality_change_3h"]
    q6 = row["quality_change_6h"]

    opp = row["continuous_opportunity"]
    opp1 = row["opportunity_change_1h"]
    opp3 = row["opportunity_change_3h"]

    if pd.isna(q):
        return "INSUFFICIENT_DATA"

    # --------------------------------------------------------
    # FADING
    # --------------------------------------------------------

    if (
        q >= 70
        and q1 < -2
        and q3 < -3
        and q6 < -3
    ):
        return "FADING"

    # --------------------------------------------------------
    # EXCEPTIONAL
    # --------------------------------------------------------

    if q >= 90:
        if q1 >= 0:
            return "EXCEPTIONAL_BUILDING"
        else:
            return "EXCEPTIONAL_FADING"

    # --------------------------------------------------------
    # HIGH OPPORTUNITY
    # --------------------------------------------------------

    if q >= 75:

        if (
            q1 > 1
            and q3 > 0
            and q6 > 0
        ):
            return "HIGH_OPPORTUNITY_BUILDING"

        if (
            q1 < -1
            and q3 < 0
        ):
            return "HIGH_OPPORTUNITY_FADING"

        return "HIGH_OPPORTUNITY"

    # --------------------------------------------------------
    # DEVELOPING
    # --------------------------------------------------------

    if (
        q >= 60
        and (
            q1 > 1
            or q3 > 2
            or opp1 > 2
        )
    ):
        return "DEVELOPING"

    # --------------------------------------------------------
    # WATCH
    # --------------------------------------------------------

    if q >= 50:
        return "WATCH"

    # --------------------------------------------------------
    # WAIT
    # --------------------------------------------------------

    return "WAIT"


df["opportunity_state"] = (
    df.apply(
        classify_state,
        axis=1
    )
)

# ------------------------------------------------------------
# 7. STATE NUMERIC PRIORITY
# ------------------------------------------------------------

priority_map = {
    "WAIT": 0,
    "WATCH": 1,
    "DEVELOPING": 2,
    "HIGH_OPPORTUNITY": 3,
    "HIGH_OPPORTUNITY_BUILDING": 4,
    "HIGH_OPPORTUNITY_FADING": 2,
    "EXCEPTIONAL_BUILDING": 5,
    "EXCEPTIONAL_FADING": 3,
    "FADING": 1,
    "INSUFFICIENT_DATA": -1,
}

df["state_priority"] = (
    df["opportunity_state"]
    .map(priority_map)
)

# ------------------------------------------------------------
# 8. STATE TRANSITIONS
# ------------------------------------------------------------

df["previous_state"] = (
    df["opportunity_state"]
    .shift(1)
)

df["state_changed"] = (
    df["opportunity_state"]
    != df["previous_state"]
)

# ------------------------------------------------------------
# 9. QUALITY ACCELERATION
# ------------------------------------------------------------

df["quality_acceleration"] = (
    df["quality_change_1h"]
    - df["quality_change_3h"] / 3
)

# ------------------------------------------------------------
# 10. CURRENT STATE
# ------------------------------------------------------------

latest = df.iloc[-1]

print("\n" + "=" * 80)
print("CURRENT OPPORTUNITY STATE")
print("=" * 80)

print(f"\nTimestamp: {latest.name}")

if "close" in latest.index:
    print(
        f"Price: "
        f"{latest['close']:.4f}"
    )

print(
    f"\nOpportunity: "
    f"{latest['continuous_opportunity']:.2f}"
)

print(
    f"Quality: "
    f"{latest['quality_score']:.2f}"
)

print(
    f"Quality percentile: "
    f"{latest['quality_percentile']:.2f}"
)

print(
    f"\n1h quality change: "
    f"{latest['quality_change_1h']:+.2f}"
)

print(
    f"3h quality change: "
    f"{latest['quality_change_3h']:+.2f}"
)

print(
    f"6h quality change: "
    f"{latest['quality_change_6h']:+.2f}"
)

print(
    f"\n1h opportunity change: "
    f"{latest['opportunity_change_1h']:+.2f}"
)

print(
    f"3h opportunity change: "
    f"{latest['opportunity_change_3h']:+.2f}"
)

print(
    f"6h opportunity change: "
    f"{latest['opportunity_change_6h']:+.2f}"
)

print(
    f"\nQuality rising horizons: "
    f"{latest['quality_rising_count']}/3"
)

print(
    f"Quality falling horizons: "
    f"{latest['quality_falling_count']}/3"
)

print(
    f"\nState: "
    f"{latest['opportunity_state']}"
)

print(
    f"Priority: "
    f"{latest['state_priority']}"
)

print(
    f"State changed this candle: "
    f"{latest['state_changed']}"
)

# ------------------------------------------------------------
# 11. STATE DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("STATE DISTRIBUTION")
print("=" * 80)

print(
    df["opportunity_state"]
    .value_counts()
    .to_string()
)

# ------------------------------------------------------------
# 12. STATE PERFORMANCE
# ------------------------------------------------------------

if "future_abs_24h" in df.columns:

    valid = df.dropna(
        subset=[
            "opportunity_state",
            "future_abs_24h"
        ]
    ).copy()

    valid["future_abs_24h_pct"] = (
        valid["future_abs_24h"] * 100
    )

    performance = (
        valid
        .groupby("opportunity_state")
        .agg(
            n=("future_abs_24h_pct", "size"),
            mean_24h=(
                "future_abs_24h_pct",
                "mean"
            ),
            median_24h=(
                "future_abs_24h_pct",
                "median"
            ),
            p90=(
                "future_abs_24h_pct",
                lambda x: x.quantile(.90)
            ),
            large_move=(
                "future_abs_24h_pct",
                lambda x: (x >= 2.0).mean() * 100
            ),
        )
        .sort_values(
            "mean_24h",
            ascending=False
        )
    )

    print("\n" + "=" * 80)
    print("STATE HISTORICAL PERFORMANCE")
    print("=" * 80)

    print(
        performance.to_string(
            float_format=lambda x: f"{x:.3f}"
        )
    )

# ------------------------------------------------------------
# 13. SAVE
# ------------------------------------------------------------

df.to_parquet(
    OUTPUT_FILE
)

print("\n" + "=" * 80)
print("STATE MONITOR SAVED")
print("=" * 80)

print(
    f"\n{OUTPUT_FILE}"
)

print(
    "\nNo future target was used to classify the current state."
)

print(
    "No trading orders were submitted."
)

print("\nSTEP 59 COMPLETE")

In [ ]:
# ============================================================
# STEP 60A — DEMO ACCOUNT + RISK CHECK
# NO ORDERS ARE PLACED
# ============================================================

import os
import requests
from pathlib import Path

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

BASE_URL = "https://api-fxpractice.oanda.com"
INSTRUMENT = "XAG_USD"

# Maximum risk we will allow on a single demo trade
RISK_PERCENT = 0.25

# Maximum number of simultaneous positions
MAX_OPEN_POSITIONS = 1

CONFIG_FILE = Path(
    "/content/gdrive/MyDrive/SilverAgentOandaID.txt"
)

# ------------------------------------------------------------
# Load credentials from existing Drive config
# ------------------------------------------------------------

if not CONFIG_FILE.exists():
    raise FileNotFoundError(
        f"Config file not found:\n{CONFIG_FILE}"
    )

for line in CONFIG_FILE.read_text().splitlines():

    line = line.strip()

    if not line or "=" not in line:
        continue

    key, value = line.split("=", 1)

    os.environ[key.strip()] = value.strip()

ACCOUNT_ID = os.getenv("OANDA_ACCOUNT_ID")
API_TOKEN = os.getenv("OANDA_API_TOKEN")

if not ACCOUNT_ID or not API_TOKEN:
    raise RuntimeError(
        "OANDA credentials were not loaded."
    )

HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}

# ------------------------------------------------------------
# ACCOUNT CHECK
# ------------------------------------------------------------

print("=" * 65)
print("STEP 60A — OANDA PRACTICE ACCOUNT CHECK")
print("=" * 65)

account_url = f"{BASE_URL}/v3/accounts/{ACCOUNT_ID}"

response = requests.get(
    account_url,
    headers=HEADERS,
    timeout=15
)

print("HTTP status:", response.status_code)

if response.status_code != 200:
    print(response.text)
    raise RuntimeError("OANDA account request failed.")

account = response.json()["account"]

balance = float(account["balance"])
nav = float(account["NAV"])
margin_available = float(account["marginAvailable"])

print("\nConnection:      OK")
print("Account:         PRACTICE")
print(f"Balance:         {balance:.2f}")
print(f"NAV:             {nav:.2f}")
print(f"Margin available:{margin_available:.2f}")

# ------------------------------------------------------------
# OPEN POSITIONS
# ------------------------------------------------------------

positions_url = (
    f"{BASE_URL}/v3/accounts/{ACCOUNT_ID}/openPositions"
)

response = requests.get(
    positions_url,
    headers=HEADERS,
    timeout=15
)

if response.status_code != 200:
    print(response.text)
    raise RuntimeError(
        "Could not retrieve open positions."
    )

positions = response.json().get("positions", [])

print("\nOpen positions:", len(positions))

for position in positions:

    instrument = position["instrument"]
    long_units = position["long"]["units"]
    short_units = position["short"]["units"]

    print(
        f"  {instrument}: "
        f"LONG={long_units}, "
        f"SHORT={short_units}"
    )

# ------------------------------------------------------------
# XAG/USD CURRENT MARKET DATA
# ------------------------------------------------------------

candles_url = (
    f"{BASE_URL}/v3/instruments/{INSTRUMENT}/candles"
)

params = {
    "granularity": "H1",
    "count": 3,
    "price": "M"
}

response = requests.get(
    candles_url,
    headers=HEADERS,
    params=params,
    timeout=15
)

if response.status_code != 200:
    print(response.text)
    raise RuntimeError(
        "Could not retrieve XAG/USD candles."
    )

candles = response.json()["candles"]

completed = [
    candle
    for candle in candles
    if candle.get("complete", False)
]

if not completed:
    raise RuntimeError(
        "No completed XAG/USD H1 candle returned."
    )

latest = completed[-1]

close_price = float(latest["mid"]["c"])

print("\nLatest completed XAG/USD H1 candle")
print("-" * 65)
print("Time :", latest["time"])
print("Open :", latest["mid"]["o"])
print("High :", latest["mid"]["h"])
print("Low  :", latest["mid"]["l"])
print("Close:", latest["mid"]["c"])

# ------------------------------------------------------------
# RISK CALCULATION
# ------------------------------------------------------------

risk_amount = nav * (RISK_PERCENT / 100)

print("\nRisk configuration")
print("-" * 65)
print(f"Risk percentage: {RISK_PERCENT:.2f}%")
print(f"Maximum risk:    {risk_amount:.2f}")

# ------------------------------------------------------------
# SAFETY GATE
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("SAFETY GATE")
print("=" * 65)

if len(positions) >= MAX_OPEN_POSITIONS:

    print("STATUS: BLOCKED")
    print(
        f"Existing positions: {len(positions)}"
    )
    print(
        f"Maximum allowed: {MAX_OPEN_POSITIONS}"
    )

else:

    print("STATUS: READY")
    print("No existing position limit reached.")

print("\nORDER SUBMITTED: NO")
print("THIS CELL CANNOT PLACE AN ORDER.")

In [ ]:
# ============================================================
# STEP 60 — LOAD OANDA CREDENTIALS FROM DRIVE
# ============================================================

import os
from pathlib import Path

CONFIG_FILE = Path(
    "/content/gdrive/MyDrive/SilverAgentOandaID.txt"
)

if not CONFIG_FILE.exists():
    raise FileNotFoundError(
        f"Could not find:\n{CONFIG_FILE}"
    )

# Load the two values from the existing config file
for line in CONFIG_FILE.read_text().splitlines():

    line = line.strip()

    if not line or "=" not in line:
        continue

    key, value = line.split("=", 1)

    os.environ[key.strip()] = value.strip()

# Retrieve them without displaying the actual credentials
account_id = os.getenv("OANDA_ACCOUNT_ID")
api_token = os.getenv("OANDA_API_TOKEN")

print("=" * 55)
print("OANDA CREDENTIAL CHECK")
print("=" * 55)

print(
    "OANDA_ACCOUNT_ID:",
    "LOADED" if account_id else "MISSING"
)

print(
    "OANDA_API_TOKEN:",
    "LOADED" if api_token else "MISSING"
)

if not account_id or not api_token:
    raise RuntimeError(
        "One or more OANDA credentials could not be loaded."
    )

print("\nCredentials successfully loaded.")
print("Credentials themselves were NOT displayed.")

In [ ]:

# ============================================================
# STEP 60 — LOAD OANDA CREDENTIALS FROM DRIVE
# ============================================================

import os
from pathlib import Path

CONFIG_FILE = Path(
    "/content/gdrive/MyDrive/SilverAgentOandaID.txt"
)

if not CONFIG_FILE.exists():
    raise FileNotFoundError(
        f"Could not find:\n{CONFIG_FILE}"
    )

# Load the two values from the existing config file
for line in CONFIG_FILE.read_text().splitlines():

    line = line.strip()

    if not line or "=" not in line:
        continue

    key, value = line.split("=", 1)

    os.environ[key.strip()] = value.strip()

# Retrieve them without displaying the actual credentials
account_id = os.getenv("OANDA_ACCOUNT_ID")
api_token = os.getenv("OANDA_API_TOKEN")

print("=" * 55)
print("OANDA CREDENTIAL CHECK")
print("=" * 55)

print(
    "OANDA_ACCOUNT_ID:",
    "LOADED" if account_id else "MISSING"
)

print(
    "OANDA_API_TOKEN:",
    "LOADED" if api_token else "MISSING"
)

if not account_id or not api_token:
    raise RuntimeError(
        "One or more OANDA credentials could not be loaded."
    )

print("\nCredentials successfully loaded.")
print("Credentials themselves were NOT displayed.")

In [ ]:
# ============================================================
# STEP 60 — FIND OANDA CONFIG FILE
# ============================================================

from pathlib import Path

print("Checking SilverAgent locations...\n")

base = Path("/content/gdrive/MyDrive")

# Check expected location
expected = base / "SilverAgentOandaID.txt"

print("Expected path:")
print(expected)
print("Exists:", expected.exists())

print("\nFiles in My Drive containing 'Oanda' or 'SilverAgent':")

matches = []

for p in base.rglob("*"):
    if p.is_file():
        name = p.name.lower()

        if (
            "oanda" in name
            or "silveragent" in name
        ):
            matches.append(p)

if matches:
    for p in matches:
        print("FOUND:", p)
else:
    print("No matching files found.")

print("\nSilverAgent folder exists:",
      (base / "SilverAgent").exists())

In [ ]:
# ============================================================
# STEP 60 — CHECK CREDENTIAL FILE FORMAT
# ============================================================

from pathlib import Path

config_file = Path(
    "/content/gdrive/MyDrive/SilverAgentOandaID.txt"
)

lines = config_file.read_text().splitlines()

print("=" * 60)
print("OANDA CONFIG FILE CHECK")
print("=" * 60)

print("File exists: YES")
print("Number of lines:", len(lines))

for i, line in enumerate(lines, start=1):

    line = line.strip()

    if "=" in line:
        key, value = line.split("=", 1)

        print(
            f"Line {i}: "
            f"KEY={key.strip()} | "
            f"VALUE={'PRESENT' if value.strip() else 'EMPTY'}"
        )
    else:
        print(
            f"Line {i}: "
            f"NO KEY/VALUE SEPARATOR"
        )

print("\nExpected keys:")
print("OANDA_ACCOUNT_ID")
print("OANDA_API_TOKEN")

print("\nActual keys found:")

for line in lines:

    line = line.strip()

    if "=" in line:
        key = line.split("=", 1)[0].strip()
        print("-", key)

print("\nIMPORTANT:")
print("Your actual Account ID and API token were NOT displayed.")

In [ ]:
# ============================================================
# STEP 60 — REPAIR + LOAD OANDA CREDENTIALS
# ============================================================

import os
from pathlib import Path

CONFIG_FILE = Path(
    "/content/gdrive/MyDrive/SilverAgentOandaID.txt"
)

raw_lines = CONFIG_FILE.read_text(
    encoding="utf-8-sig"
).splitlines()

loaded = {}

for line in raw_lines:

    line = line.strip()

    if not line or "=" not in line:
        continue

    key, value = line.split("=", 1)

    # Remove invisible characters
    key = key.replace("\ufeff", "").strip()

    # Correct common zero/O typo in these specific keys
    if key in ["0ANDA_ACCOUNT_ID", "OANDA_ACCOUNT_ID"]:
        key = "OANDA_ACCOUNT_ID"

    elif key in ["0ANDA_API_TOKEN", "OANDA_API_TOKEN"]:
        key = "OANDA_API_TOKEN"

    loaded[key] = value.strip()

# Put corrected values into environment
if "OANDA_ACCOUNT_ID" in loaded:
    os.environ["OANDA_ACCOUNT_ID"] = loaded["OANDA_ACCOUNT_ID"]

if "OANDA_API_TOKEN" in loaded:
    os.environ["OANDA_API_TOKEN"] = loaded["OANDA_API_TOKEN"]

# ------------------------------------------------------------
# SAFE CHECK
# ------------------------------------------------------------

account_loaded = bool(os.getenv("OANDA_ACCOUNT_ID"))
token_loaded = bool(os.getenv("OANDA_API_TOKEN"))

print("=" * 60)
print("OANDA CREDENTIAL LOAD")
print("=" * 60)

print(
    "OANDA_ACCOUNT_ID:",
    "LOADED" if account_loaded else "MISSING"
)

print(
    "OANDA_API_TOKEN:",
    "LOADED" if token_loaded else "MISSING"
)

if not account_loaded or not token_loaded:
    raise RuntimeError(
        "Credentials could not be loaded."
    )

print("\nSUCCESS — credentials loaded.")
print("Actual credentials were NOT displayed.")

In [ ]:
# ============================================================
# STEP 61 — OANDA PRACTICE ACCOUNT SAFETY CHECK
# NO ORDER WILL BE PLACED
# ============================================================

import os
import requests

BASE_URL = "https://api-fxpractice.oanda.com"
INSTRUMENT = "XAG_USD"

ACCOUNT_ID = os.getenv("OANDA_ACCOUNT_ID")
API_TOKEN = os.getenv("OANDA_API_TOKEN")

if not ACCOUNT_ID or not API_TOKEN:
    raise RuntimeError(
        "OANDA credentials are not loaded. "
        "Run Step 60 again."
    )

HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}

print("=" * 65)
print("STEP 61 — OANDA PRACTICE ACCOUNT CHECK")
print("=" * 65)

# ------------------------------------------------------------
# Account
# ------------------------------------------------------------

url = f"{BASE_URL}/v3/accounts/{ACCOUNT_ID}"

r = requests.get(
    url,
    headers=HEADERS,
    timeout=15
)

print("Account API status:", r.status_code)

if r.status_code != 200:
    print(r.text)
    raise RuntimeError("Account verification failed.")

account = r.json()["account"]

balance = float(account["balance"])
nav = float(account["NAV"])
margin_available = float(account["marginAvailable"])

print("\nACCOUNT")
print("-" * 65)
print("Environment:      OANDA PRACTICE")
print(f"Balance:          {balance:.2f}")
print(f"NAV:              {nav:.2f}")
print(f"Margin available: {margin_available:.2f}")

# ------------------------------------------------------------
# Open positions
# ------------------------------------------------------------

url = f"{BASE_URL}/v3/accounts/{ACCOUNT_ID}/openPositions"

r = requests.get(
    url,
    headers=HEADERS,
    timeout=15
)

if r.status_code != 200:
    print(r.text)
    raise RuntimeError("Open-position check failed.")

positions = r.json().get("positions", [])

print("\nOPEN POSITIONS")
print("-" * 65)
print("Number of open positions:", len(positions))

for p in positions:
    print(
        f"{p['instrument']} | "
        f"LONG={p['long']['units']} | "
        f"SHORT={p['short']['units']}"
    )

# ------------------------------------------------------------
# Current XAG/USD
# ------------------------------------------------------------

url = f"{BASE_URL}/v3/instruments/{INSTRUMENT}/candles"

params = {
    "granularity": "H1",
    "count": 3,
    "price": "M"
}

r = requests.get(
    url,
    headers=HEADERS,
    params=params,
    timeout=15
)

print("\nMARKET DATA STATUS:", r.status_code)

if r.status_code != 200:
    print(r.text)
    raise RuntimeError("XAG/USD data request failed.")

candles = r.json()["candles"]

completed = [
    c for c in candles
    if c.get("complete") is True
]

if not completed:
    raise RuntimeError(
        "No completed XAG/USD candle returned."
    )

candle = completed[-1]

print("\nLATEST COMPLETED XAG/USD H1")
print("-" * 65)
print("Time :", candle["time"])
print("Open :", candle["mid"]["o"])
print("High :", candle["mid"]["h"])
print("Low  :", candle["mid"]["l"])
print("Close:", candle["mid"]["c"])

# ------------------------------------------------------------
# Risk calculation
# ------------------------------------------------------------

RISK_PERCENT = 0.25
risk_amount = nav * RISK_PERCENT / 100

print("\nRISK LIMIT")
print("-" * 65)
print(f"Risk per trade: {RISK_PERCENT:.2f}%")
print(f"Maximum planned risk: {risk_amount:.2f}")

# ------------------------------------------------------------
# FINAL SAFETY STATUS
# ------------------------------------------------------------

print("\n" + "=" * 65)

if len(positions) == 0:
    print("STATUS: READY")
    print("No existing position detected.")
else:
    print("STATUS: POSITION ALREADY OPEN")
    print("We will NOT open another position.")

print("ORDER SUBMITTED: NO")
print("=" * 65)

In [ ]:
# ============================================================
# STEP 62 — DEMO TRADING DECISION ENGINE
# CONNECTS TO STEP 59
# NO ORDER PLACED
# ============================================================

import os
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("/content/gdrive/MyDrive/SilverAgent")
RESULTS = BASE / "results"

# ------------------------------------------------------------
# Load latest Opportunity State Monitor
# ------------------------------------------------------------

state_file = RESULTS / "opportunity_state_v1_history.parquet"

if not state_file.exists():
    raise FileNotFoundError(
        f"Could not find:\n{state_file}"
    )

state = pd.read_parquet(state_file)

if state.empty:
    raise RuntimeError("Opportunity state database is empty.")

state = state.sort_index()

latest = state.iloc[-1]

# ------------------------------------------------------------
# Extract current state safely
# ------------------------------------------------------------

def get_value(row, possible_names, default=np.nan):

    for name in possible_names:
        if name in row.index:
            value = row[name]

            if pd.notna(value):
                return value

    return default


timestamp = state.index[-1]

price = get_value(
    latest,
    ["price", "close"]
)

opportunity = get_value(
    latest,
    ["opportunity_score", "continuous_opportunity"]
)

quality = get_value(
    latest,
    ["quality_score"]
)

state_name = get_value(
    latest,
    ["opportunity_state", "state"],
    "UNKNOWN"
)

quality_change_1h = get_value(
    latest,
    ["quality_change_1h"],
    0
)

quality_change_3h = get_value(
    latest,
    ["quality_change_3h"],
    0
)

quality_change_6h = get_value(
    latest,
    ["quality_change_6h"],
    0
)

# ------------------------------------------------------------
# Display current intelligence
# ------------------------------------------------------------

print("=" * 70)
print("STEP 62 — DEMO TRADING DECISION ENGINE")
print("=" * 70)

print("\nLATEST MACHINE STATE")
print("-" * 70)

print("Timestamp:", timestamp)
print(f"Price:     {price:.5f}" if pd.notna(price) else "Price:     unavailable")
print(
    f"Opportunity: {opportunity:.2f}"
    if pd.notna(opportunity)
    else "Opportunity: unavailable"
)
print(
    f"Quality:     {quality:.2f}"
    if pd.notna(quality)
    else "Quality:     unavailable"
)
print("State:", state_name)

print("\nQUALITY MOMENTUM")
print("-" * 70)

print(f"1H: {quality_change_1h:+.2f}")
print(f"3H: {quality_change_3h:+.2f}")
print(f"6H: {quality_change_6h:+.2f}")

# ------------------------------------------------------------
# DEMO TRADING RULES
# ------------------------------------------------------------
#
# IMPORTANT:
# We are deliberately NOT using the rejected directional
# prediction engine.
#
# For the first automated demo trial we require:
#
# 1. High opportunity
# 2. High/Exceptional quality
# 3. State must represent a meaningful opportunity
# 4. Quality must not be rapidly collapsing
#
# Direction is deliberately left neutral for now.
#
# Therefore this engine can currently return NO TRADE.
# That is intentional.
# ------------------------------------------------------------

opportunity_ok = (
    pd.notna(opportunity)
    and opportunity >= 75
)

quality_ok = (
    pd.notna(quality)
    and quality >= 75
)

state_ok = str(state_name) in [
    "HIGH_OPPORTUNITY",
    "HIGH_OPPORTUNITY_BUILDING",
    "EXCEPTIONAL_BUILDING",
    "HIGH_OPPORTUNITY_FADING",
    "EXCEPTIONAL_FADING",
    "FADING"
]

quality_not_collapsing = (
    quality_change_1h > -15
    and quality_change_3h > -20
)

# ------------------------------------------------------------
# Decision
# ------------------------------------------------------------

checks = {
    "Opportunity >= 75": opportunity_ok,
    "Quality >= 75": quality_ok,
    "Valid opportunity state": state_ok,
    "Quality not collapsing": quality_not_collapsing
}

print("\nTRADING CHECKS")
print("-" * 70)

for name, result in checks.items():
    print(
        f"{'PASS' if result else 'FAIL':<5} | {name}"
    )

all_pass = all(checks.values())

if all_pass:
    decision = "CANDIDATE"
else:
    decision = "NO TRADE"

print("\n" + "=" * 70)
print("DEMO DECISION:", decision)
print("=" * 70)

if decision == "CANDIDATE":
    print(
        "\nThe market currently meets the opportunity/quality "
        "requirements."
    )
    print(
        "However, NO ORDER WILL BE PLACED by this cell."
    )
    print(
        "The next step must determine trade direction and "
        "risk parameters."
    )
else:
    print(
        "\nThe current market does not satisfy all demo "
        "trading requirements."
    )
    print(
        "The bot would remain OUT of the market."
    )

print("\nORDER SUBMITTED: NO")

In [ ]:
# ============================================================
# STEP 63 — LIVE PIPELINE SYNCHRONISATION
# OANDA → DATABASE → FEATURES → OPPORTUNITY → QUALITY → STATE
#
# NO ORDERS ARE PLACED
# ============================================================

import os
import requests
import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

BASE_URL = "https://api-fxpractice.oanda.com"
INSTRUMENT = "XAG_USD"

BASE = Path("/content/gdrive/MyDrive/SilverAgent")
DB_DIR = BASE / "database"
RESULTS = BASE / "results"

DB_FILE = DB_DIR / "silver_h1.parquet"
FEATURE_FILE = DB_DIR / "silver_h1_features_v2.parquet"
OPP_FILE = RESULTS / "opportunity_v2_history.parquet"
QUALITY_FILE = RESULTS / "opportunity_quality_v2_history.parquet"
STATE_FILE = RESULTS / "opportunity_state_v1_history.parquet"

ACCOUNT_ID = os.getenv("OANDA_ACCOUNT_ID")
API_TOKEN = os.getenv("OANDA_API_TOKEN")

if not ACCOUNT_ID or not API_TOKEN:
    raise RuntimeError(
        "OANDA credentials are not loaded. Run Step 60 first."
    )

HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}

DB_DIR.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. LOAD EXISTING DATABASE
# ------------------------------------------------------------

if DB_FILE.exists():
    db = pd.read_parquet(DB_FILE)
    db.index = pd.to_datetime(db.index, utc=True)

    print("Existing database:", len(db), "candles")
    print("Database latest:", db.index[-1])
else:
    db = pd.DataFrame()

# ------------------------------------------------------------
# 2. GET RECENT COMPLETED OANDA CANDLES
# ------------------------------------------------------------

url = f"{BASE_URL}/v3/instruments/{INSTRUMENT}/candles"

params = {
    "granularity": "H1",
    "count": 10,
    "price": "M"
}

r = requests.get(
    url,
    headers=HEADERS,
    params=params,
    timeout=20
)

print("\nOANDA HTTP status:", r.status_code)

if r.status_code != 200:
    print(r.text)
    raise RuntimeError("OANDA request failed.")

raw_candles = r.json()["candles"]

completed = [
    c for c in raw_candles
    if c.get("complete") is True
]

if not completed:
    raise RuntimeError("No completed H1 candles returned.")

rows = []

for c in completed:

    rows.append({
        "time": pd.to_datetime(c["time"], utc=True),
        "open": float(c["mid"]["o"]),
        "high": float(c["mid"]["h"]),
        "low": float(c["mid"]["l"]),
        "close": float(c["mid"]["c"]),
        "volume": int(c["volume"])
    })

new_data = pd.DataFrame(rows).set_index("time")

# ------------------------------------------------------------
# 3. UPDATE DATABASE
# ------------------------------------------------------------

if len(db) > 0:
    combined = pd.concat([db, new_data])
else:
    combined = new_data.copy()

combined = combined[~combined.index.duplicated(keep="last")]
combined = combined.sort_index()

# Keep database at existing maximum size + new data.
# Do NOT fabricate missing candles.
combined.to_parquet(DB_FILE)

added = len(combined) - len(db)

print("\nDATABASE UPDATE")
print("-" * 65)
print("New OANDA completed candles:", len(new_data))
print("New candles added:", max(0, added))
print("Total candles:", len(combined))
print("Latest candle:", combined.index[-1])
print("Latest close:", combined["close"].iloc[-1])

# ------------------------------------------------------------
# 4. REBUILD BASIC GAP-AWARE FEATURES
# ------------------------------------------------------------

df = combined.copy()

close = df["close"]
high = df["high"]
low = df["low"]
volume = df["volume"]

df["ma10"] = close.rolling(10, min_periods=10).mean()
df["ma20"] = close.rolling(20, min_periods=20).mean()
df["ma50"] = close.rolling(50, min_periods=50).mean()

df["ma10_vs_ma20"] = (
    df["ma10"] / df["ma20"] - 1
) * 100

df["ma20_vs_ma50"] = (
    df["ma20"] / df["ma50"] - 1
) * 100

df["price_vs_ma20"] = (
    close / df["ma20"] - 1
) * 100

df["return_1h"] = close.pct_change(1) * 100
df["return_3h"] = close.pct_change(3) * 100
df["return_6h"] = close.pct_change(6) * 100
df["return_12h"] = close.pct_change(12) * 100
df["return_24h"] = close.pct_change(24) * 100

df["vol10"] = (
    df["return_1h"]
    .rolling(10, min_periods=10)
    .std()
)

df["vol20"] = (
    df["return_1h"]
    .rolling(20, min_periods=20)
    .std()
)

# True Range
prev_close = close.shift(1)

tr1 = high - low
tr2 = (high - prev_close).abs()
tr3 = (low - prev_close).abs()

true_range = pd.concat(
    [tr1, tr2, tr3],
    axis=1
).max(axis=1)

df["atr14"] = true_range.rolling(
    14,
    min_periods=14
).mean()

df["atr_percent"] = (
    df["atr14"] / close
) * 100

df["relative_volume"] = (
    volume /
    volume.rolling(20, min_periods=20).mean()
)

# Position within recent range
rolling_high = high.rolling(
    20,
    min_periods=20
).max()

rolling_low = low.rolling(
    20,
    min_periods=20
).min()

df["distance_high20"] = (
    rolling_high / close - 1
) * 100

df["distance_low20"] = (
    close / rolling_low - 1
) * 100

df["range_position"] = (
    (close - rolling_low) /
    (rolling_high - rolling_low)
) * 100

# Momentum intensity
df["momentum_intensity"] = (
    df["return_3h"].abs()
    + df["return_6h"].abs()
) / 2

# Range expansion
df["range_expansion"] = (
    (high - low) /
    (high - low).rolling(20, min_periods=20).mean()
)

# Price displacement
df["price_displacement"] = (
    df["return_6h"].abs()
)

# ------------------------------------------------------------
# 5. CAUSAL HISTORICAL PERCENTILE FUNCTION
# ------------------------------------------------------------

def causal_percentile(series, min_periods=500):

    previous = series.shift(1)

    return (
        previous
        .expanding(min_periods=min_periods)
        .rank(pct=True)
        * 100
    )

# Component percentiles
df["vol10_pct"] = causal_percentile(df["vol10"])
df["vol20_pct"] = causal_percentile(df["vol20"])
df["atr_pct_rank"] = causal_percentile(df["atr_percent"])

df["momentum_intensity_pct"] = causal_percentile(
    df["momentum_intensity"]
)

df["range_expansion_pct"] = causal_percentile(
    df["range_expansion"]
)

df["relative_volume_pct"] = causal_percentile(
    df["relative_volume"]
)

df["price_displacement_pct"] = causal_percentile(
    df["price_displacement"]
)

df["distance_high20_pct"] = causal_percentile(
    df["distance_high20"]
)

df["distance_low20_pct"] = causal_percentile(
    df["distance_low20"]
)

# ------------------------------------------------------------
# 6. OPPORTUNITY ENGINE
# ------------------------------------------------------------

df["volatility_score"] = df[
    ["vol10_pct", "vol20_pct", "atr_pct_rank"]
].mean(axis=1)

df["momentum_energy_score"] = df[
    ["momentum_intensity_pct", "range_expansion_pct"]
].mean(axis=1)

df["volume_score"] = df["relative_volume_pct"]

df["displacement_score"] = df[
    [
        "price_displacement_pct",
        "distance_high20_pct",
        "distance_low20_pct"
    ]
].mean(axis=1)

df["opportunity_score"] = df[
    [
        "volatility_score",
        "momentum_energy_score",
        "volume_score",
        "displacement_score"
    ]
].mean(axis=1)

def opportunity_zone(x):

    if pd.isna(x):
        return "INSUFFICIENT_DATA"

    if x < 20:
        return "VERY_LOW"
    elif x < 40:
        return "LOW"
    elif x < 60:
        return "MEDIUM"
    elif x < 80:
        return "HIGH"
    else:
        return "VERY_HIGH"

df["opportunity_zone"] = df[
    "opportunity_score"
].apply(opportunity_zone)

# ------------------------------------------------------------
# 7. QUALITY ENGINE
# ------------------------------------------------------------

df["energy_build"] = (
    df["momentum_energy_score"].diff(3)
)

df["persistence"] = (
    df["opportunity_score"]
    .ge(60)
    .astype(int)
    .groupby(
        (~df["opportunity_score"].ge(60)).cumsum()
    )
    .cumsum()
)

df["quality_score"] = (
    df["opportunity_score"] * 0.65
    + df["energy_build"].rank(pct=True) * 100 * 0.15
    + df["persistence"].rank(pct=True) * 100 * 0.20
)

# ------------------------------------------------------------
# 8. QUALITY ZONE
# ------------------------------------------------------------

def quality_zone(x):

    if pd.isna(x):
        return "INSUFFICIENT_DATA"

    if x >= 90:
        return "EXCEPTIONAL"
    elif x >= 75:
        return "HIGH"
    elif x >= 50:
        return "MEDIUM"
    elif x >= 25:
        return "LOW"
    else:
        return "VERY_LOW"

df["quality_zone"] = df[
    "quality_score"
].apply(quality_zone)

# ------------------------------------------------------------
# 9. OPPORTUNITY STATE
# ------------------------------------------------------------

opp = df["opportunity_score"]
qual = df["quality_score"]

opp_change_1h = opp.diff(1)
opp_change_3h = opp.diff(3)
opp_change_6h = opp.diff(6)

qual_change_1h = qual.diff(1)
qual_change_3h = qual.diff(3)
qual_change_6h = qual.diff(6)

def determine_state(row):

    o = row["opportunity_score"]
    q = row["quality_score"]

    if pd.isna(o) or pd.isna(q):
        return "INSUFFICIENT_DATA"

    if q >= 90:
        if (
            row["quality_change_1h"] < -5
            and row["quality_change_3h"] < -5
        ):
            return "EXCEPTIONAL_FADING"

        return "EXCEPTIONAL_BUILDING"

    if q >= 75:
        if (
            row["quality_change_1h"] < -5
            and row["quality_change_3h"] < -5
        ):
            return "HIGH_OPPORTUNITY_FADING"

        if (
            row["quality_change_1h"] > 0
            and row["quality_change_3h"] > 0
        ):
            return "HIGH_OPPORTUNITY_BUILDING"

        return "HIGH_OPPORTUNITY"

    if q >= 60:
        return "DEVELOPING"

    if q >= 40:
        return "WATCH"

    if q < 25:
        return "WAIT"

    return "WATCH"

df["opportunity_change_1h"] = opp_change_1h
df["opportunity_change_3h"] = opp_change_3h
df["opportunity_change_6h"] = opp_change_6h

df["quality_change_1h"] = qual_change_1h
df["quality_change_3h"] = qual_change_3h
df["quality_change_6h"] = qual_change_6h

df["opportunity_state"] = df.apply(
    determine_state,
    axis=1
)

# ------------------------------------------------------------
# 10. SAVE LIVE PIPELINE OUTPUTS
# ------------------------------------------------------------

df.to_parquet(FEATURE_FILE)

df[
    [
        "close",
        "volatility_score",
        "momentum_energy_score",
        "volume_score",
        "displacement_score",
        "opportunity_score",
        "opportunity_zone"
    ]
].to_parquet(OPP_FILE)

df[
    [
        "close",
        "opportunity_score",
        "quality_score",
        "quality_zone",
        "energy_build",
        "persistence"
    ]
].to_parquet(QUALITY_FILE)

df[
    [
        "close",
        "opportunity_score",
        "quality_score",
        "quality_change_1h",
        "quality_change_3h",
        "quality_change_6h",
        "opportunity_change_1h",
        "opportunity_change_3h",
        "opportunity_change_6h",
        "opportunity_state"
    ]
].to_parquet(STATE_FILE)

# ------------------------------------------------------------
# 11. CURRENT LIVE STATE
# ------------------------------------------------------------

latest = df.iloc[-1]

print("\n" + "=" * 70)
print("CURRENT SYNCHRONISED XAG/USD STATE")
print("=" * 70)

print("Timestamp:", df.index[-1])
print(f"Price: {latest['close']:.5f}")

print("\nOPPORTUNITY")
print("-" * 70)
print(
    f"Score:       {latest['opportunity_score']:.2f}"
)
print(
    f"Zone:        {latest['opportunity_zone']}"
)

print("\nCOMPONENTS")
print("-" * 70)
print(
    f"Volatility:  {latest['volatility_score']:.2f}"
)
print(
    f"Momentum:    {latest['momentum_energy_score']:.2f}"
)
print(
    f"Volume:      {latest['volume_score']:.2f}"
)
print(
    f"Displacement:{latest['displacement_score']:.2f}"
)

print("\nQUALITY")
print("-" * 70)
print(
    f"Score:       {latest['quality_score']:.2f}"
)
print(
    f"Zone:        {latest['quality_zone']}"
)
print(
    f"1H change:   {latest['quality_change_1h']:+.2f}"
)
print(
    f"3H change:   {latest['quality_change_3h']:+.2f}"
)
print(
    f"6H change:   {latest['quality_change_6h']:+.2f}"
)

print("\nSTATE")
print("-" * 70)
print(
    "Opportunity state:",
    latest["opportunity_state"]
)

print("\n" + "=" * 70)
print("PIPELINE STATUS: SYNCHRONISED")
print("ORDER SUBMITTED: NO")
print("=" * 70)

In [ ]:
# ============================================================
# STEP 64 — DEMO TRADE PROPOSAL + RISK ENGINE
# NO ORDER WILL BE PLACED
# ============================================================

import os
import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

BASE = Path("/content/gdrive/MyDrive/SilverAgent")
DB_FILE = BASE / "database" / "silver_h1_features_v2.parquet"
STATE_FILE = BASE / "results" / "opportunity_state_v1_history.parquet"

RISK_PERCENT = 0.25

# Opportunity/quality thresholds
MIN_OPPORTUNITY = 75
MIN_QUALITY = 75

# ATR risk model
STOP_ATR = 1.5
TARGET_ATR = 2.0

# ------------------------------------------------------------
# LOAD CURRENT DATA
# ------------------------------------------------------------

if not DB_FILE.exists():
    raise FileNotFoundError(DB_FILE)

if not STATE_FILE.exists():
    raise FileNotFoundError(STATE_FILE)

features = pd.read_parquet(DB_FILE)
features.index = pd.to_datetime(features.index, utc=True)
features = features.sort_index()

state = pd.read_parquet(STATE_FILE)
state.index = pd.to_datetime(state.index, utc=True)
state = state.sort_index()

latest = features.iloc[-1]

# ------------------------------------------------------------
# CURRENT MARKET
# ------------------------------------------------------------

price = float(latest["close"])

atr = latest.get("atr14", np.nan)

if pd.isna(atr):
    raise RuntimeError(
        "ATR14 is unavailable. Cannot calculate risk safely."
    )

atr = float(atr)

# ------------------------------------------------------------
# FIND CURRENT STATE
# ------------------------------------------------------------

current_state = state.iloc[-1]

opportunity = float(
    current_state.get(
        "opportunity_score",
        latest.get("opportunity_score", np.nan)
    )
)

quality = float(
    current_state.get(
        "quality_score",
        latest.get("quality_score", np.nan)
    )
)

state_name = str(
    current_state.get(
        "opportunity_state",
        "UNKNOWN"
    )
)

quality_change_1h = float(
    current_state.get(
        "quality_change_1h",
        0
    )
)

quality_change_3h = float(
    current_state.get(
        "quality_change_3h",
        0
    )
)

quality_change_6h = float(
    current_state.get(
        "quality_change_6h",
        0
    )
)

# ------------------------------------------------------------
# CURRENT COMPONENTS
# ------------------------------------------------------------

volatility_score = float(
    latest.get("volatility_score", np.nan)
)

momentum_score = float(
    latest.get("momentum_energy_score", np.nan)
)

volume_score = float(
    latest.get("volume_score", np.nan)
)

displacement_score = float(
    latest.get("displacement_score", np.nan)
)

# ------------------------------------------------------------
# OPPORTUNITY CHECKS
# ------------------------------------------------------------

opportunity_ok = (
    pd.notna(opportunity)
    and opportunity >= MIN_OPPORTUNITY
)

quality_ok = (
    pd.notna(quality)
    and quality >= MIN_QUALITY
)

state_ok = state_name in [
    "HIGH_OPPORTUNITY",
    "HIGH_OPPORTUNITY_BUILDING",
    "EXCEPTIONAL_BUILDING",
    "HIGH_OPPORTUNITY_FADING",
    "EXCEPTIONAL_FADING",
    "FADING"
]

quality_building = (
    quality_change_1h > 0
    or quality_change_3h > 0
    or quality_change_6h > 0
)

quality_not_collapsing = (
    quality_change_1h > -15
    and quality_change_3h > -20
    and quality_change_6h > -25
)

# ------------------------------------------------------------
# DIRECTION
# ------------------------------------------------------------
#
# IMPORTANT:
# We deliberately do NOT use the old directional engine.
#
# The historical research did not establish a robust
# directional edge.
#
# Therefore direction remains UNCONFIRMED.
# ------------------------------------------------------------

direction = "UNCONFIRMED"

# ------------------------------------------------------------
# RISK CALCULATION
# ------------------------------------------------------------

# Account balance is retrieved from environment if available.
# Otherwise we use the last verified demo balance only for
# display, and DO NOT place an order.

account_balance = 100000.0

risk_amount = account_balance * (
    RISK_PERCENT / 100
)

stop_distance = atr * STOP_ATR
target_distance = atr * TARGET_ATR

# XAG/USD units are handled separately when we build the
# actual OANDA order engine. This cell only calculates
# price distances and account risk.

stop_distance_pct = (
    stop_distance / price
) * 100

target_distance_pct = (
    target_distance / price
) * 100

reward_risk = (
    target_distance / stop_distance
)

# ------------------------------------------------------------
# PROPOSAL STATUS
# ------------------------------------------------------------

all_opportunity_checks = all([
    opportunity_ok,
    quality_ok,
    state_ok,
    quality_not_collapsing
])

# We explicitly refuse to turn this into a trade without
# directional confirmation.

if all_opportunity_checks:

    proposal = "OPPORTUNITY DETECTED"
    trade_permission = "WAIT FOR DIRECTION"

else:

    proposal = "NO QUALIFYING OPPORTUNITY"
    trade_permission = "NO TRADE"

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

print("=" * 70)
print("STEP 64 — DEMO TRADE PROPOSAL")
print("=" * 70)

print("\nMARKET")
print("-" * 70)
print("Timestamp:", features.index[-1])
print(f"XAG/USD:   {price:.5f}")
print(f"ATR14:     {atr:.5f}")

print("\nOPPORTUNITY")
print("-" * 70)
print(f"Score:       {opportunity:.2f}")
print(f"Quality:     {quality:.2f}")
print(f"State:       {state_name}")

print("\nCOMPONENTS")
print("-" * 70)
print(f"Volatility:   {volatility_score:.2f}")
print(f"Momentum:     {momentum_score:.2f}")
print(f"Volume:       {volume_score:.2f}")
print(f"Displacement: {displacement_score:.2f}")

print("\nQUALITY MOMENTUM")
print("-" * 70)
print(f"1H: {quality_change_1h:+.2f}")
print(f"3H: {quality_change_3h:+.2f}")
print(f"6H: {quality_change_6h:+.2f}")

print("\nCHECKS")
print("-" * 70)
print(
    f"{'PASS' if opportunity_ok else 'FAIL'} | "
    f"Opportunity >= {MIN_OPPORTUNITY}"
)
print(
    f"{'PASS' if quality_ok else 'FAIL'} | "
    f"Quality >= {MIN_QUALITY}"
)
print(
    f"{'PASS' if state_ok else 'FAIL'} | "
    "Valid opportunity state"
)
print(
    f"{'PASS' if quality_not_collapsing else 'FAIL'} | "
    "Quality not collapsing"
)

print("\nRISK MODEL")
print("-" * 70)
print(f"Demo balance:       £{account_balance:,.2f}")
print(f"Risk percentage:    {RISK_PERCENT:.2f}%")
print(f"Maximum risk:       £{risk_amount:,.2f}")
print(f"Stop distance:      {stop_distance:.5f}")
print(f"Target distance:    {target_distance:.5f}")
print(f"Stop distance:      {stop_distance_pct:.3f}%")
print(f"Target distance:    {target_distance_pct:.3f}%")
print(f"Reward / risk:      {reward_risk:.2f}:1")

print("\nDIRECTION")
print("-" * 70)
print("Direction:", direction)

print("\n" + "=" * 70)
print("PROPOSAL:", proposal)
print("TRADE PERMISSION:", trade_permission)
print("=" * 70)

print("\nORDER SUBMITTED: NO")

In [ ]:
# ============================================================
# STEP 65 — FIRST AUTOMATED OANDA PRACTICE TRADE
# ============================================================

import os
import requests
import pandas as pd
import numpy as np

BASE_URL = "https://api-fxpractice.oanda.com"
INSTRUMENT = "XAG_USD"

RISK_PERCENT = 0.25
STOP_ATR = 1.5
TARGET_ATR = 2.0
MAX_OPEN_POSITIONS = 1

ACCOUNT_ID = os.getenv("OANDA_ACCOUNT_ID")
API_TOKEN = os.getenv("OANDA_API_TOKEN")

if not ACCOUNT_ID or not API_TOKEN:
    raise RuntimeError("OANDA credentials are not loaded.")

HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}

# ------------------------------------------------------------
# Load latest synchronised feature data
# ------------------------------------------------------------

from pathlib import Path

FEATURE_FILE = Path(
    "/content/gdrive/MyDrive/SilverAgent/database/silver_h1_features_v2.parquet"
)

df = pd.read_parquet(FEATURE_FILE)
df.index = pd.to_datetime(df.index, utc=True)
df = df.sort_index()

x = df.iloc[-1]

price = float(x["close"])
atr = float(x["atr14"])
ma20 = float(x["ma20"])
ma50 = float(x["ma50"])

mom6 = float(x["return_6h"])
mom12 = float(x["return_12h"])

# ------------------------------------------------------------
# Current opportunity state
# ------------------------------------------------------------

opp = float(x["opportunity_score"])
quality = float(x["quality_score"])

state_file = Path(
    "/content/gdrive/MyDrive/SilverAgent/results/opportunity_state_v1_history.parquet"
)

state = pd.read_parquet(state_file)
state.index = pd.to_datetime(state.index, utc=True)
state = state.sort_index()

current_state = state.iloc[-1]["opportunity_state"]

# ------------------------------------------------------------
# Check existing positions directly with OANDA
# ------------------------------------------------------------

positions_url = (
    f"{BASE_URL}/v3/accounts/{ACCOUNT_ID}/openPositions"
)

r = requests.get(
    positions_url,
    headers=HEADERS,
    timeout=15
)

if r.status_code != 200:
    raise RuntimeError(
        f"Position check failed: {r.text}"
    )

positions = r.json().get("positions", [])

if len(positions) >= MAX_OPEN_POSITIONS:
    print("NO TRADE — existing position already open.")
    raise SystemExit

# ------------------------------------------------------------
# SIMPLE EXPERIMENTAL DIRECTION MODEL
# ------------------------------------------------------------

long_score = 0
short_score = 0

# Trend
if price > ma20:
    long_score += 1
else:
    short_score += 1

if ma20 > ma50:
    long_score += 1
else:
    short_score += 1

# Momentum
if mom6 > 0:
    long_score += 1
else:
    short_score += 1

if mom12 > 0:
    long_score += 1
else:
    short_score += 1

# Opportunity must be strong
opportunity_ok = opp >= 75
quality_ok = quality >= 75

# Direction requires at least 3/4
if long_score >= 3 and long_score > short_score:
    direction = "LONG"
    confidence = long_score / 4 * 100

elif short_score >= 3 and short_score > long_score:
    direction = "SHORT"
    confidence = short_score / 4 * 100

else:
    direction = "WAIT"
    confidence = 50

# ------------------------------------------------------------
# Display decision
# ------------------------------------------------------------

print("=" * 70)
print("STEP 65 — FIRST PRACTICE TRADE")
print("=" * 70)

print(f"\nPrice:       {price:.5f}")
print(f"Opportunity: {opp:.2f}")
print(f"Quality:     {quality:.2f}")
print(f"State:       {current_state}")

print("\nDirection model")
print("-" * 70)
print(f"LONG score:  {long_score}/4")
print(f"SHORT score: {short_score}/4")
print(f"Direction:   {direction}")
print(f"Confidence:  {confidence:.0f}%")

print("\nRisk model")
print("-" * 70)

# ------------------------------------------------------------
# Only trade when opportunity + quality + direction agree
# ------------------------------------------------------------

if not opportunity_ok:
    print("NO TRADE — opportunity below 75.")
    raise SystemExit

if not quality_ok:
    print("NO TRADE — quality below 75.")
    raise SystemExit

if direction == "WAIT":
    print("NO TRADE — direction not strong enough.")
    raise SystemExit

# ------------------------------------------------------------
# Account balance
# ------------------------------------------------------------

account_url = f"{BASE_URL}/v3/accounts/{ACCOUNT_ID}"

r = requests.get(
    account_url,
    headers=HEADERS,
    timeout=15
)

if r.status_code != 200:
    raise RuntimeError(
        f"Account check failed: {r.text}"
    )

account = r.json()["account"]

balance = float(account["NAV"])

risk_amount = balance * (
    RISK_PERCENT / 100
)

# Price distances
stop_distance = atr * STOP_ATR
target_distance = atr * TARGET_ATR

if direction == "LONG":
    stop_price = price - stop_distance
    target_price = price + target_distance
else:
    stop_price = price + stop_distance
    target_price = price - target_distance

# ------------------------------------------------------------
# XAG/USD unit sizing
#
# OANDA XAG/USD uses units of the instrument.
# We calculate units so the price-distance risk is capped.
# ------------------------------------------------------------

units_float = risk_amount / stop_distance

units = int(max(1, units_float))

print(f"Balance:       £{balance:,.2f}")
print(f"Risk limit:    £{risk_amount:,.2f}")
print(f"ATR:           {atr:.5f}")
print(f"Stop:          {stop_price:.5f}")
print(f"Target:        {target_price:.5f}")
print(f"Units:         {units}")

# ------------------------------------------------------------
# FINAL SAFETY CHECK
# ------------------------------------------------------------

print("\nFINAL SAFETY CHECK")
print("-" * 70)

print("Environment: OANDA PRACTICE")
print("Instrument:", INSTRUMENT)
print("Direction:", direction)
print("Risk:", f"{RISK_PERCENT}%")
print("Order type: MARKET")
print("Stop-loss: ATTACHED")
print("Take-profit: ATTACHED")

# ------------------------------------------------------------
# SUBMIT PRACTICE ORDER
# ------------------------------------------------------------

order_units = (
    str(units)
    if direction == "LONG"
    else str(-units)
)

order = {
    "order": {
        "units": order_units,
        "instrument": INSTRUMENT,
        "timeInForce": "FOK",
        "type": "MARKET",
        "positionFill": "DEFAULT",
        "stopLossOnFill": {
            "price": f"{stop_price:.3f}"
        },
        "takeProfitOnFill": {
            "price": f"{target_price:.3f}"
        }
    }
}

orders_url = (
    f"{BASE_URL}/v3/accounts/{ACCOUNT_ID}/orders"
)

print("\nSubmitting Practice order...")

response = requests.post(
    orders_url,
    headers=HEADERS,
    json=order,
    timeout=15
)

print("OANDA HTTP status:", response.status_code)

if response.status_code not in [200, 201]:
    print("\nORDER REJECTED")
    print(response.text)
    raise RuntimeError("Practice order was rejected.")

result = response.json()

print("\n" + "=" * 70)
print("🎯 FIRST PRACTICE ORDER SUBMITTED")
print("=" * 70)

print("Instrument:", INSTRUMENT)
print("Direction:", direction)
print("Units:", units)
print("Entry price:", price)
print("Stop:", stop_price)
print("Target:", target_price)

if "orderFillTransaction" in result:
    fill = result["orderFillTransaction"]

    print(
        "Fill price:",
        fill.get("price", "N/A")
    )

    print(
        "Order ID:",
        fill.get("orderID", "N/A")
    )

print("\nTHIS WAS AN OANDA PRACTICE/DEMO ORDER.")
print("NO LIVE FUNDS WERE USED.")

In [ ]:
# ============================================================
# STEP 65 — FIRST EXPERIMENTAL PRACTICE TRADE
# ============================================================
# IMPORTANT:
# This is an EXPERIMENTAL demo trade.
# It is NOT a proven trading strategy.
#
# OANDA PRACTICE ONLY
# Risk: 0.10% maximum
# Maximum open positions: 1
# ============================================================

import os
import requests
import pandas as pd
import numpy as np
from pathlib import Path

BASE_URL = "https://api-fxpractice.oanda.com"
INSTRUMENT = "XAG_USD"

RISK_PERCENT = 0.10
STOP_ATR = 1.5
TARGET_ATR = 2.0
MAX_OPEN_POSITIONS = 1

ACCOUNT_ID = os.getenv("OANDA_ACCOUNT_ID")
API_TOKEN = os.getenv("OANDA_API_TOKEN")

if not ACCOUNT_ID or not API_TOKEN:
    raise RuntimeError(
        "OANDA credentials are not loaded."
    )

HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}

# ------------------------------------------------------------
# LOAD SYNCHRONISED FEATURES
# ------------------------------------------------------------

FEATURE_FILE = Path(
    "/content/gdrive/MyDrive/SilverAgent/database/silver_h1_features_v2.parquet"
)

if not FEATURE_FILE.exists():
    raise FileNotFoundError(FEATURE_FILE)

df = pd.read_parquet(FEATURE_FILE)
df.index = pd.to_datetime(df.index, utc=True)
df = df.sort_index()

x = df.iloc[-1]
previous = df.iloc[-2]

price = float(x["close"])
previous_close = float(previous["close"])

atr = float(x["atr14"])
ma20 = float(x["ma20"])
ma50 = float(x["ma50"])
mom6 = float(x["return_6h"])
mom12 = float(x["return_12h"])

opportunity = float(x["opportunity_score"])
quality = float(x["quality_score"])

# ------------------------------------------------------------
# CURRENT STATE
# ------------------------------------------------------------

STATE_FILE = Path(
    "/content/gdrive/MyDrive/SilverAgent/results/opportunity_state_v1_history.parquet"
)

state = pd.read_parquet(STATE_FILE)
state.index = pd.to_datetime(state.index, utc=True)
state = state.sort_index()

current_state = str(
    state.iloc[-1]["opportunity_state"]
)

# ------------------------------------------------------------
# OPEN POSITION CHECK
# ------------------------------------------------------------

positions_url = (
    f"{BASE_URL}/v3/accounts/{ACCOUNT_ID}/openPositions"
)

r = requests.get(
    positions_url,
    headers=HEADERS,
    timeout=15
)

if r.status_code != 200:
    raise RuntimeError(
        f"Position check failed: {r.text}"
    )

positions = r.json().get("positions", [])

if len(positions) >= MAX_OPEN_POSITIONS:
    print("NO TRADE — an existing position is already open.")
    raise SystemExit

# ------------------------------------------------------------
# OPPORTUNITY / QUALITY GATE
# ------------------------------------------------------------

if opportunity < 75:
    print(
        f"NO TRADE — opportunity {opportunity:.2f} "
        "is below 75."
    )
    raise SystemExit

if quality < 75:
    print(
        f"NO TRADE — quality {quality:.2f} "
        "is below 75."
    )
    raise SystemExit

# ------------------------------------------------------------
# DIRECTION SCORE
# ------------------------------------------------------------

long_score = 0
short_score = 0

# Price vs MA20
if price > ma20:
    long_score += 1
else:
    short_score += 1

# MA20 vs MA50
if ma20 > ma50:
    long_score += 1
else:
    short_score += 1

# 6H momentum
if mom6 > 0:
    long_score += 1
else:
    short_score += 1

# 12H momentum
if mom12 > 0:
    long_score += 1
else:
    short_score += 1

# ------------------------------------------------------------
# TIE BREAKER
# ------------------------------------------------------------
# If the four-factor direction score is tied, use the latest
# completed H1 candle direction.
#
# This is explicitly experimental, not predictive evidence.

if long_score > short_score:

    direction = "LONG"
    direction_basis = "4-factor score"

elif short_score > long_score:

    direction = "SHORT"
    direction_basis = "4-factor score"

else:

    if price > previous_close:
        direction = "LONG"
    elif price < previous_close:
        direction = "SHORT"
    else:
        print(
            "NO TRADE — direction tied and latest candle "
            "was unchanged."
        )
        raise SystemExit

    direction_basis = "H1 candle tie-breaker"

# ------------------------------------------------------------
# ACCOUNT / RISK
# ------------------------------------------------------------

account_url = (
    f"{BASE_URL}/v3/accounts/{ACCOUNT_ID}"
)

r = requests.get(
    account_url,
    headers=HEADERS,
    timeout=15
)

if r.status_code != 200:
    raise RuntimeError(
        f"Account check failed: {r.text}"
    )

account = r.json()["account"]

balance = float(account["NAV"])

risk_amount = balance * (
    RISK_PERCENT / 100
)

# ------------------------------------------------------------
# STOP / TARGET
# ------------------------------------------------------------

stop_distance = atr * STOP_ATR
target_distance = atr * TARGET_ATR

if stop_distance <= 0:
    raise RuntimeError(
        "Invalid ATR/stop distance."
    )

if direction == "LONG":

    stop_price = price - stop_distance
    target_price = price + target_distance

else:

    stop_price = price + stop_distance
    target_price = price - target_distance

# ------------------------------------------------------------
# POSITION SIZE
# ------------------------------------------------------------

units_float = risk_amount / stop_distance

units = int(units_float)

if units < 1:
    raise RuntimeError(
        "Calculated position size is below 1 unit."
    )

# ------------------------------------------------------------
# DISPLAY FINAL PROPOSAL
# ------------------------------------------------------------

print("=" * 70)
print("STEP 65 — FIRST EXPERIMENTAL PRACTICE TRADE")
print("=" * 70)

print("\nMARKET")
print("-" * 70)
print("Timestamp:", df.index[-1])
print(f"Price:       {price:.5f}")
print(f"ATR14:       {atr:.5f}")

print("\nINTELLIGENCE")
print("-" * 70)
print(f"Opportunity: {opportunity:.2f}")
print(f"Quality:     {quality:.2f}")
print(f"State:       {current_state}")

print("\nDIRECTION")
print("-" * 70)
print(f"LONG score:  {long_score}/4")
print(f"SHORT score: {short_score}/4")
print(f"Direction:   {direction}")
print(f"Basis:       {direction_basis}")

print("\nRISK")
print("-" * 70)
print(f"Demo NAV:    £{balance:,.2f}")
print(f"Risk:        {RISK_PERCENT:.2f}%")
print(f"Risk amount: £{risk_amount:,.2f}")
print(f"Units:       {units}")
print(f"Stop:        {stop_price:.5f}")
print(f"Target:      {target_price:.5f}")
print(
    f"Reward/Risk: "
    f"{target_distance / stop_distance:.2f}:1"
)

print("\n" + "=" * 70)
print("PRACTICE ORDER — SUBMITTING")
print("=" * 70)

# ------------------------------------------------------------
# OANDA MARKET ORDER
# ------------------------------------------------------------

order_units = (
    str(units)
    if direction == "LONG"
    else str(-units)
)

order_payload = {
    "order": {
        "units": order_units,
        "instrument": INSTRUMENT,
        "timeInForce": "FOK",
        "type": "MARKET",
        "positionFill": "DEFAULT",

        "stopLossOnFill": {
            "price": f"{stop_price:.3f}"
        },

        "takeProfitOnFill": {
            "price": f"{target_price:.3f}"
        }
    }
}

orders_url = (
    f"{BASE_URL}/v3/accounts/{ACCOUNT_ID}/orders"
)

response = requests.post(
    orders_url,
    headers=HEADERS,
    json=order_payload,
    timeout=15
)

print("OANDA HTTP status:", response.status_code)

if response.status_code not in (200, 201):

    print("\nORDER REJECTED")
    print(response.text)

    raise RuntimeError(
        "Practice order was rejected."
    )

result = response.json()

# ------------------------------------------------------------
# RESULT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("🎯 PRACTICE ORDER SUCCESSFULLY SUBMITTED")
print("=" * 70)

print("Instrument:", INSTRUMENT)
print("Direction:", direction)
print("Units:", units)
print("Stop:", stop_price)
print("Target:", target_price)

fill = result.get(
    "orderFillTransaction",
    {}
)

print(
    "Fill price:",
    fill.get("price", "N/A")
)

print(
    "Order ID:",
    fill.get("orderID", "N/A")
)

print(
    "Trade ID:",
    fill.get("tradeOpened", {}).get(
        "tradeID",
        "N/A"
    )
)

print("\nOANDA ENVIRONMENT: PRACTICE")
print("LIVE MONEY USED: NO")
print("MAX PLANNED RISK: 0.10%")

In [ ]:
# STEP 66 — LIVE DEMO TRADE MONITOR
import os
import requests
from datetime import datetime, timezone

ACCOUNT_ID = os.environ["OANDA_ACCOUNT_ID"]
API_TOKEN = os.environ["OANDA_API_TOKEN"]

BASE_URL = "https://api-fxpractice.oanda.com"
TRADE_ID = "5"

headers = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}

# Get trade details
url = f"{BASE_URL}/v3/accounts/{ACCOUNT_ID}/trades/{TRADE_ID}"
r = requests.get(url, headers=headers, timeout=20)

print("HTTP:", r.status_code)

if r.status_code != 200:
    print(r.text)
else:
    data = r.json()
    trade = data.get("trade", {})

    print("\n==============================")
    print("     SILVER DEMO TRADE")
    print("==============================")
    print("Trade ID:", trade.get("id"))
    print("Instrument:", trade.get("instrument"))
    print("Units:", trade.get("currentUnits"))
    print("Entry price:", trade.get("price"))
    print("Open time:", trade.get("openTime"))

    print("\n--- PROTECTION ---")

    sl = trade.get("stopLossOrder", {})
    tp = trade.get("takeProfitOrder", {})

    print("Stop loss:", sl.get("price"))
    print("Take profit:", tp.get("price"))

    print("\n--- P/L ---")
    print("Unrealized P/L:", trade.get("unrealizedPL"))
    print("Realized P/L:", trade.get("realizedPL"))

    print("\n--- STATUS ---")
    print("State:", trade.get("state"))
    print("Last transaction:", trade.get("lastTransactionID"))

    # Current account summary
    account_url = f"{BASE_URL}/v3/accounts/{ACCOUNT_ID}/summary"
    ar = requests.get(account_url, headers=headers, timeout=20)

    if ar.status_code == 200:
        account = ar.json()["account"]

        print("\n--- ACCOUNT ---")
        print("Balance:", account.get("balance"))
        print("NAV:", account.get("NAV"))
        print("Margin used:", account.get("marginUsed"))
        print("Margin available:", account.get("marginAvailable"))

    print("\nChecked:", datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC"))
    print("==============================")

In [ ]:
# STEP 67 — DEMO TRADE JOURNAL + LIVE MONITOR

import os
import requests
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

ACCOUNT_ID = os.environ["OANDA_ACCOUNT_ID"]
API_TOKEN = os.environ["OANDA_API_TOKEN"]

BASE_URL = "https://api-fxpractice.oanda.com"
TRADE_ID = "5"

headers = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}

# ---------------------------------------------------------
# 1. Get trade
# ---------------------------------------------------------

trade_url = f"{BASE_URL}/v3/accounts/{ACCOUNT_ID}/trades/{TRADE_ID}"

r = requests.get(
    trade_url,
    headers=headers,
    timeout=20
)

if r.status_code != 200:
    print("Trade request failed:", r.status_code)
    print(r.text)
    raise RuntimeError("Could not retrieve Trade 5.")

trade = r.json()["trade"]

instrument = trade["instrument"]
units = float(trade["currentUnits"])
entry = float(trade["price"])
unrealized_pl = float(trade.get("unrealizedPL", 0))
state = trade["state"]

stop_price = float(
    trade["stopLossOrder"]["price"]
)

target_price = float(
    trade["takeProfitOrder"]["price"]
)

# ---------------------------------------------------------
# 2. Get current market price
# ---------------------------------------------------------

price_url = (
    f"{BASE_URL}/v3/accounts/"
    f"{ACCOUNT_ID}/pricing"
)

pr = requests.get(
    price_url,
    headers=headers,
    params={"instruments": instrument},
    timeout=20
)

if pr.status_code != 200:
    print("Price request failed:", pr.status_code)
    print(pr.text)
    raise RuntimeError("Could not retrieve market price.")

price_data = pr.json()["prices"][0]

bid = float(price_data["bids"][0]["price"])
ask = float(price_data["asks"][0]["price"])

current_price = (
    (bid + ask) / 2
)

# ---------------------------------------------------------
# 3. Calculate trade metrics
# ---------------------------------------------------------

if units < 0:
    direction = "SHORT"

    distance_to_stop = stop_price - current_price
    distance_to_target = current_price - target_price

    initial_risk_distance = stop_price - entry

else:
    direction = "LONG"

    distance_to_stop = current_price - stop_price
    distance_to_target = target_price - current_price

    initial_risk_distance = entry - stop_price


risk_progress = (
    (entry - current_price) / initial_risk_distance
    if direction == "SHORT"
    else
    (current_price - entry) / initial_risk_distance
)

risk_multiple = risk_progress

# ---------------------------------------------------------
# 4. Account information
# ---------------------------------------------------------

account_url = (
    f"{BASE_URL}/v3/accounts/"
    f"{ACCOUNT_ID}/summary"
)

ar = requests.get(
    account_url,
    headers=headers,
    timeout=20
)

if ar.status_code == 200:

    account = ar.json()["account"]

    balance = float(account["balance"])
    nav = float(account["NAV"])
    margin_used = float(account["marginUsed"])
    margin_available = float(account["marginAvailable"])

else:

    balance = None
    nav = None
    margin_used = None
    margin_available = None


# ---------------------------------------------------------
# 5. Create observation
# ---------------------------------------------------------

timestamp = datetime.now(timezone.utc)

observation = {
    "timestamp_utc": timestamp.isoformat(),
    "trade_id": TRADE_ID,
    "instrument": instrument,
    "direction": direction,
    "units": units,
    "entry_price": entry,
    "current_price": current_price,
    "bid": bid,
    "ask": ask,
    "stop_price": stop_price,
    "target_price": target_price,
    "unrealized_pl": unrealized_pl,
    "distance_to_stop": distance_to_stop,
    "distance_to_target": distance_to_target,
    "risk_multiple": risk_multiple,
    "state": state,
    "balance": balance,
    "nav": nav,
    "margin_used": margin_used,
    "margin_available": margin_available
}

# ---------------------------------------------------------
# 6. Save journal to Drive
# ---------------------------------------------------------

LOG_DIR = Path(
    "/content/gdrive/MyDrive/SilverAgent/logs"
)

LOG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

JOURNAL_FILE = LOG_DIR / "demo_trade_journal.csv"

new_row = pd.DataFrame([observation])

if JOURNAL_FILE.exists():

    old = pd.read_csv(JOURNAL_FILE)

    journal = pd.concat(
        [old, new_row],
        ignore_index=True
    )

else:

    journal = new_row

journal.to_csv(
    JOURNAL_FILE,
    index=False
)

# ---------------------------------------------------------
# 7. Display
# ---------------------------------------------------------

print()
print("=" * 55)
print("       SILVER AGENT — TRADE 5 MONITOR")
print("=" * 55)

print(f"Trade:              {TRADE_ID}")
print(f"Instrument:         {instrument}")
print(f"Direction:          {direction}")
print(f"Units:              {abs(units):.0f}")

print()
print("--- MARKET ---")
print(f"Bid:                {bid:.5f}")
print(f"Ask:                {ask:.5f}")
print(f"Mid:                {current_price:.5f}")

print()
print("--- TRADE ---")
print(f"Entry:              {entry:.5f}")
print(f"Stop:               {stop_price:.5f}")
print(f"Target:             {target_price:.5f}")

print()
print("--- PERFORMANCE ---")
print(f"Unrealised P/L:     £{unrealized_pl:.4f}")
print(f"Risk multiple:      {risk_multiple:.3f}R")
print(f"Distance to stop:   {distance_to_stop:.5f}")
print(f"Distance to target: {distance_to_target:.5f}")

print()
print("--- ACCOUNT ---")
print(f"Balance:            £{balance:,.2f}")
print(f"NAV:                £{nav:,.2f}")
print(f"Margin used:        £{margin_used:,.2f}")
print(f"Margin available:   £{margin_available:,.2f}")

print()
print(f"State:              {state}")
print(f"Journal entries:    {len(journal)}")
print(f"Logged to:          {JOURNAL_FILE}")

print()
print("IMPORTANT: This cell only monitors and records.")
print("It does NOT modify, close, or cancel the trade.")

print("=" * 55)

In [ ]:
from google.colab import drive

drive.mount('/content/gdrive', force_remount=True)